# MICA benchmark, Phase 3b and 3c: choose the test entries

This notebook does the rest of the entry selection in one run:

1. installs Phenix from your download link (you type the user name and password when asked),
2. computes the map-model correlation (CC_mask, `phenix.map_model_cc`) for the entries that the selection rule asks for,
3. applies the selection rule of `docs/SPEC.md` (decisions 4 and 5) and repeats until the list of entries is final,
4. checks how close each chosen protein is to proteins in the PDB before 2026 (decision 9, MMseqs2),
5. packs everything into one zip file and downloads it.

**What you do**

1. Click **Runtime -> Run all**. If Colab warns that the notebook was not authored by Google, click **Run anyway**.
2. In step 3 two boxes appear. Paste the Phenix **user name** from the Phenix email, press Enter, then the **password**
   (the second box hides what you type). The password is not saved in the notebook or in the zip.
3. Keep this tab open and the computer awake. It takes about 1.5 to 3 hours. Do not edit any code.
   If Colab disconnects, Colab erases its files: run all again from the top.
4. At the end the zip `selection_result_....zip` downloads by itself. Send it to me.
   If no download starts: click the folder icon on the left, open the folder `work`, click the three dots next to the zip, then **Download**.

**If something goes wrong.** A step that cannot continue prints `STOPPED` and a plain explanation; the later steps are skipped
and the last step still makes a zip with the log. Send me that zip. If a step shows a red error instead, copy all of its text and send it to me.
To stop early, press the stop button, then run only the last cell: it packs what exists.

**Phenix.** Phenix is licensed to you personally. Do not share the installer, the password or this Colab session. When you are done,
use **Runtime -> Disconnect and delete runtime** to remove the Phenix copy from Colab.


## Step 1. Look at the machine

In [ ]:
import datetime, getpass, glob, hashlib, json, os, platform, re, shutil, subprocess, sys, time, zipfile
from collections import deque

IN_COLAB = os.path.isdir("/content") and not os.environ.get("MICA_WORK")      # MICA_WORK is only used by the tests
WORK = os.environ.get("MICA_WORK") or ("/content/work" if IN_COLAB else os.path.abspath("work"))
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
LOG = os.path.join(WORK, "run.log")
STATE = {"ok": True, "problem": "", "final": False, "rounds": 0, "t0": time.time(), "started": datetime.datetime.now(datetime.timezone.utc).isoformat(),
         "phenix_env": None, "phenix_version": "", "installer_sha256": "", "mmseqs_version": "", "novelty": "not run"}
SECRETS = []          # kept in memory only, to make sure they never end up in the zip


def log(msg=""):
    print(msg, flush=True)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")


def fail(msg):
    STATE["ok"] = False
    STATE["problem"] = msg
    log("\n" + "!" * 72 + "\nSTOPPED: " + msg + "\nThe steps below are skipped. The last step still packs a zip with the log: send it to me.\n" + "!" * 72)


def run(cmd, env=None, every=1, cwd=None):
    """Runs a command and prints its output as it comes. Returns (exit code, last lines). every=N prints one line in N."""
    e = dict(os.environ)
    e.update(env or {})
    e["PYTHONUNBUFFERED"] = "1"
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e, cwd=cwd, errors="replace")
    last, n = deque(maxlen=40), 0
    for line in p.stdout:
        line = line.rstrip("\n")
        last.append(line)
        n += 1
        if every == 1 or n <= 5 or n % every == 0:
            log(line)
    p.wait()
    return p.returncode, list(last)


def secret(name, prompt, hidden=True):
    """Colab Secrets, Kaggle Secrets, an environment variable, or a prompt, in this order."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    return (getpass.getpass(prompt) if hidden else input(prompt)).strip()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def meminfo_gb():
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    return round(int(line.split()[1]) / 1048576, 1)
    except OSError:
        pass
    return None


free_gb = round(shutil.disk_usage(WORK).free / 1e9)
log("python %s | %s | %s CPU cores | %s GB RAM | %d GB free disk" % (platform.python_version(), platform.platform(), os.cpu_count(), meminfo_gb(), free_gb))
log("This is Colab." if IN_COLAB else "This is not Colab (a test run).")
if free_gb < 45:
    log("WARNING: only %d GB of disk are free; Phenix needs about 40 GB while it installs. If step 3 fails with 'No space left', open Runtime -> "
        "Disconnect and delete runtime, then run all again." % free_gb)


## Step 2. Write the scripts and the frozen pool files

The scripts and the data below are exact copies of the files in the repository. The last cell of this step checks their hashes.

In [ ]:
import os
for d in ("data/pool", "data/selection", "sel_out"):
    os.makedirs(d, exist_ok=True)
print("folders ready")


In [ ]:
%%writefile select_entries.py
#!/usr/bin/env python3
"""Phase 3b of the MICA 2026 benchmark: cluster the candidate pool and apply the selection rule.

The rule (docs/SPEC.md decision 5, plus the operational rules written in LOG.md):
  * One entry per MMseqs2 cluster at 30% identity. Each entry is represented by its longest chain.
  * Two strata, processed in this order with one shared list of used clusters:
    up to 3.0 A, then above 3.0 to 4.0 A.
  * In each stratum the candidates, sorted by PDB ID, are shuffled with random.Random(2026).
    The walk takes the first 10 entries that pass every filter.
  * An entry is skipped if its cluster is already used or if its EMDB map is already used.
  * An entry whose CC_mask is below 0.70 (or could not be computed) is rejected and does not use
    up its cluster. CC_mask comes from compute_cc_mask.py (Phenix map_model_cc).

Subcommands
  cluster   run MMseqs2 on the longest chain of every candidate
              -> candidate_clusters.csv, mmseqs_info.json
  select    walk the shuffled order and apply the rule
              -> selection_log.csv (every candidate, with its decision), selection_info.json,
                 cc_todo.csv (entries whose CC_mask is still needed), and, when the walk
                 finishes without a missing CC_mask, entries_2026.csv

Exit codes of select: 0 walk complete (the list is written when --entries is given); 2 input missing or
unusable; 3 CC_mask values are missing (see cc_todo.csv).

CC_mask is computed in rounds. Every round starts the walk from the beginning with all CC_mask values known so
far, so only a round that ends with exit code 0 defines the list. Earlier rounds only decide which entries to
compute next; their decisions for the second stratum are provisional, because the first stratum may still use
clusters that the second stratum has not seen yet. The result does not depend on the number of rounds.

Examples
  python select_entries.py cluster --pool data/pool --out data/selection
  python select_entries.py select  --pool data/pool --out data/selection
  python select_entries.py select  --pool data/pool --out data/selection --cc data/selection/cc_mask.csv \\
         --entries data/entries_2026.csv
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import tempfile

SCRIPT_VERSION = "1.0"

SEED = 2026
QUOTA = 10          # entries per stratum
CC_MIN = 0.70       # decision 4
BATCH = 25          # entries per stratum whose CC_mask is requested in one round
MIN_SEQ_ID = 0.30   # decision 5
COVERAGE = 0.8      # operational choice, see LOG.md: every member must be covered 80% by its alignment
COV_MODE = 1        # to the representative, so that fragments join the cluster of a longer homologue
STRATA = ("le3.0", "gt3.0_le4.0")
CC_STATUSES = ("ok", "failed")

ENTRY_COLUMNS = [
    "emdb_id", "pdb_id", "emdb_release_date", "pdb_release_date", "resolution_A", "stratum",
    "modeled_residues", "unique_chain_seqs", "unk_fraction", "cc_mask", "cluster_id", "seed_rank",
    "max_id_to_pre2026_pdb", "cluster_size", "longest_seq_len", "n_chains", "map_size_mb", "cif_url", "map_url",
]
LOG_COLUMNS = ["stratum", "seed_rank", "pdb_id", "emdb_id", "cluster_id", "resolution_A", "modeled_residues",
               "cc_mask", "decision", "reason"]
TODO_COLUMNS = ["pdb_id", "emdb_id", "stratum", "seed_rank", "resolution_A", "cif_url", "map_url", "map_size_mb"]


# --------------------------------------------------------------------------------------
# Input and output
# --------------------------------------------------------------------------------------
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_csv(path, columns, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore", lineterminator="\n")
        w.writeheader()
        for r in rows:
            w.writerow({k: ("" if r.get(k) is None else r.get(k)) for k in columns})


def read_candidates(pool_dir):
    rows = read_csv(os.path.join(pool_dir, "pool_candidates.csv"))
    out = []
    for r in rows:
        if r.get("stratum") not in STRATA:
            raise ValueError("%s has stratum %r; expected one of %s" % (r.get("pdb_id"), r.get("stratum"), STRATA))
        out.append(r)
    ids = [r["pdb_id"] for r in out]
    if len(ids) != len(set(ids)):
        raise ValueError("pool_candidates.csv lists a PDB ID twice")
    return out


def read_longest_sequences(fasta_path, candidates):
    """The sequence of each candidate's longest entity (flag longest=1 in the FASTA header)."""
    wanted = {r["pdb_id"] for r in candidates}
    seqs, name, buf = {}, None, []

    def flush():
        if name is not None:
            head, _, rest = name.partition(" ")
            pid = head.split("_E")[0]
            if pid in wanted and "longest=1" in rest.split():
                if pid in seqs:
                    raise ValueError("two longest chains for %s" % pid)
                seqs[pid] = "".join(buf)

    with open(fasta_path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(">"):
                flush()
                name, buf = line[1:], []
            else:
                buf.append(line.strip())
    flush()
    missing = sorted(wanted - set(seqs))
    if missing:
        raise ValueError("no longest-chain sequence for %d candidates, e.g. %s" % (len(missing), ", ".join(missing[:5])))
    return seqs


# --------------------------------------------------------------------------------------
# Clustering
# --------------------------------------------------------------------------------------
def mmseqs_version(binary):
    try:
        r = subprocess.run([binary, "version"], capture_output=True, text=True, timeout=60)
        return (r.stdout or r.stderr).strip()
    except (OSError, subprocess.SubprocessError) as e:
        return "unknown (%s)" % e


def run_mmseqs(seqs, binary, threads=1):
    """Cluster with MMseqs2; returns {pdb_id: representative pdb_id} and the command that was run."""
    tmp = tempfile.mkdtemp(prefix="select_mmseqs_")
    try:
        fasta = os.path.join(tmp, "longest_chains.fasta")
        with open(fasta, "w", encoding="utf-8") as f:
            for pid in sorted(seqs):  # fixed input order
                f.write(">%s\n%s\n" % (pid, seqs[pid]))
        prefix = os.path.join(tmp, "clu")
        cmd = [binary, "easy-cluster", fasta, prefix, os.path.join(tmp, "work"),
               "--min-seq-id", str(MIN_SEQ_ID), "-c", str(COVERAGE), "--cov-mode", str(COV_MODE),
               "--threads", str(threads), "-v", "1"]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("mmseqs failed (exit %d):\n%s\n%s" % (r.returncode, r.stdout[-1500:], r.stderr[-1500:]))
        rep_of = {}
        with open(prefix + "_cluster.tsv", encoding="utf-8") as f:
            for line in f:
                rep, member = line.rstrip("\n").split("\t")[:2]
                rep_of[member] = rep
        missing = set(seqs) - set(rep_of)
        if missing:
            raise RuntimeError("mmseqs left %d sequences without a cluster" % len(missing))
        return rep_of, cmd
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def cmd_cluster(args):
    candidates = read_candidates(args.pool)
    fasta = os.path.join(args.pool, "pool_candidate_sequences.fasta")
    seqs = read_longest_sequences(fasta, candidates)
    binary = args.mmseqs
    if shutil.which(binary) is None and not os.path.exists(binary):
        print("mmseqs not found (%s). Install MMseqs2 or pass --mmseqs PATH." % binary, file=sys.stderr)
        return 2
    try:
        rep_of, cmd = run_mmseqs(seqs, binary, args.threads)
    except RuntimeError as e:
        print("Clustering failed: %s" % e, file=sys.stderr)
        return 2
    sizes = {}
    for rep in rep_of.values():
        sizes[rep] = sizes.get(rep, 0) + 1
    rows = [{"pdb_id": pid, "cluster_id": rep_of[pid], "cluster_size": sizes[rep_of[pid]],
             "longest_seq_len": len(seqs[pid])} for pid in sorted(seqs)]
    os.makedirs(args.out, exist_ok=True)
    write_csv(os.path.join(args.out, "candidate_clusters.csv"), ["pdb_id", "cluster_id", "cluster_size", "longest_seq_len"], rows)
    info = {"script_version": SCRIPT_VERSION, "mmseqs_version": mmseqs_version(binary),
            "command_template": [c if not c.startswith(tempfile.gettempdir()) else "<tmp>/" + os.path.basename(c) for c in cmd],
            "min_seq_id": MIN_SEQ_ID, "coverage": COVERAGE, "cov_mode": COV_MODE, "threads": args.threads,
            "n_sequences": len(seqs), "n_clusters": len(sizes),
            "largest_clusters": sorted(sizes.values(), reverse=True)[:10],
            "pool_candidates_sha256": sha256_file(os.path.join(args.pool, "pool_candidates.csv")),
            "fasta_sha256": sha256_file(fasta), "python": platform.python_version()}
    with open(os.path.join(args.out, "mmseqs_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)
    print("%d candidates in %d clusters at %d%% identity (largest clusters: %s)"
          % (len(seqs), len(sizes), round(MIN_SEQ_ID * 100), info["largest_clusters"]))
    return 0


# --------------------------------------------------------------------------------------
# The walk
# --------------------------------------------------------------------------------------
def shuffled(ids, seed=SEED):
    ids = sorted(ids)
    random.Random(seed).shuffle(ids)
    return ids


def read_cc(path):
    """{pdb_id: (status, cc_mask or None, note)}; the last row of a PDB ID wins.

    status is "ok" (cc_mask is a finite number) or "failed" (Phenix ran, but no CC_mask came out).
    A transient problem (download error, timeout) is not a result and must not be in this file: the entry
    stays unknown and is computed again. Anything else in the file is an error, not a silent rejection."""
    cc = {}
    if not path:
        return cc
    for line_no, r in enumerate(read_csv(path), 2):
        pid = (r.get("pdb_id") or "").strip()
        status = (r.get("status") or "").strip().lower()
        if not pid:
            raise ValueError("%s line %d: no pdb_id" % (path, line_no))
        if status not in CC_STATUSES:
            raise ValueError("%s line %d (%s): status %r; the allowed values are %s"
                             % (path, line_no, pid, status, " and ".join(CC_STATUSES)))
        val = None
        if status == "ok":
            try:
                val = float((r.get("cc_mask") or "").strip())
            except ValueError:
                val = None
            if val is None or not math.isfinite(val):
                raise ValueError("%s line %d (%s): status ok, but cc_mask %r is not a number"
                                 % (path, line_no, pid, r.get("cc_mask")))
        cc[pid] = (status, val, r.get("note") or "")
    return cc


def walk(cands, rep_of, cc, quota=QUOTA, cc_min=CC_MIN, seed=SEED, batch=BATCH):
    """Apply the rule. Returns (log rows, selected rows, todo rows).

    A candidate whose CC_mask is unknown stops the exact walk of its stratum, because what follows depends
    on the result. The entries to evaluate next are found by a second, optimistic walk that assumes the
    unknown ones pass."""
    by_id = {r["pdb_id"]: r for r in cands}
    used_cluster, used_map = {}, {}
    log_rows, selected, todo = [], [], []
    for stratum in STRATA:
        order = shuffled([r["pdb_id"] for r in cands if r["stratum"] == stratum], seed)
        n_sel, stop_rank = 0, None
        for rank, pid in enumerate(order, 1):
            r = by_id[pid]
            row = {"stratum": stratum, "seed_rank": rank, "pdb_id": pid, "emdb_id": r["emdb_id"],
                   "cluster_id": rep_of[pid], "resolution_A": r["resolution_A"],
                   "modeled_residues": r["n_modeled_residues"], "cc_mask": "", "decision": "", "reason": ""}
            log_rows.append(row)
            if stop_rank is not None:
                row["decision"], row["reason"] = "not_evaluated", "an earlier entry still needs its CC_mask"
                continue
            if n_sel >= quota:
                row["decision"], row["reason"] = "not_reached", "%d entries already selected in this stratum" % quota
                continue
            if rep_of[pid] in used_cluster:
                row["decision"], row["reason"] = "skip_cluster_used", "cluster %s used by %s" % (rep_of[pid], used_cluster[rep_of[pid]])
                continue
            if r["emdb_id"] in used_map:
                row["decision"], row["reason"] = "skip_map_used", "%s used by %s" % (r["emdb_id"], used_map[r["emdb_id"]])
                continue
            status, val, note = cc.get(pid, ("", None, ""))
            if not status:
                row["decision"], row["reason"] = "needs_cc", "CC_mask not computed yet"
                stop_rank = rank
                continue
            if status != "ok":
                row["decision"], row["reason"] = "reject_cc_failed", "CC_mask could not be computed: %s" % note
                continue
            row["cc_mask"] = "%.4f" % val
            if val < cc_min:
                row["decision"], row["reason"] = "reject_cc_mask", "CC_mask %.4f below %.2f" % (val, cc_min)
                continue
            row["decision"] = "selected"
            n_sel += 1
            row["reason"] = "number %d of %d in the stratum" % (n_sel, quota)
            used_cluster[rep_of[pid]] = pid
            used_map[r["emdb_id"]] = pid
            selected.append({"row": r, "rank": rank, "cc": val, "cluster": rep_of[pid]})
        if stop_rank is not None:
            todo.extend(optimistic_todo(stratum, order, by_id, rep_of, cc, used_cluster, used_map,
                                        quota - n_sel, stop_rank, batch, cc_min))
    return log_rows, selected, todo


def optimistic_todo(stratum, order, by_id, rep_of, cc, used_cluster, used_map, need, start_rank, batch, cc_min):
    """Entries to evaluate next. From the first unknown entry on, assume that every unknown CC_mask passes and
    that every known one keeps its result, and collect the unknown entries until enough selections are in
    sight: two and a half times what is still needed, at least 5 and at most `batch`."""
    want = min(batch, max(5, math.ceil(2.5 * need)))
    used_c, used_m = dict(used_cluster), dict(used_map)
    out, n_assumed = [], 0
    for rank, pid in enumerate(order, 1):
        if rank < start_rank:
            continue
        if n_assumed >= want:
            break
        r = by_id[pid]
        if rep_of[pid] in used_c or r["emdb_id"] in used_m:
            continue
        known = cc.get(pid)
        if known is not None:
            status, val, _ = known
            if status != "ok" or val < cc_min:
                continue
        else:
            out.append({"pdb_id": pid, "emdb_id": r["emdb_id"], "stratum": stratum, "seed_rank": rank,
                        "resolution_A": r["resolution_A"], "cif_url": r["cif_url"], "map_url": r["map_url"],
                        "map_size_mb": mb(r.get("map_size_kb"))})
        n_assumed += 1
        used_c[rep_of[pid]] = pid
        used_m[r["emdb_id"]] = pid
    return out


def mb(kb):
    try:
        return round(float(kb) / 1024.0, 1)
    except (TypeError, ValueError):
        return ""


def entry_rows(selected, sizes):
    rows = []
    for s in selected:
        r = s["row"]
        rows.append({
            "emdb_id": r["emdb_id"], "pdb_id": r["pdb_id"], "emdb_release_date": r.get("emdb_map_release"),
            "pdb_release_date": r.get("release_date_pdb"), "resolution_A": r["resolution_A"], "stratum": r["stratum"],
            "modeled_residues": r["n_modeled_residues"], "unique_chain_seqs": r["n_unique_seq"],
            "unk_fraction": r["unk_fraction"], "cc_mask": "%.4f" % s["cc"], "cluster_id": s["cluster"],
            "seed_rank": s["rank"], "max_id_to_pre2026_pdb": "", "cluster_size": sizes.get(s["cluster"], ""),
            "longest_seq_len": r["longest_seq_len"], "n_chains": r["n_chains"], "map_size_mb": mb(r.get("map_size_kb")),
            "cif_url": r["cif_url"], "map_url": r["map_url"]})
    return rows


def cmd_select(args):
    cands = read_candidates(args.pool)
    clusters_path = args.clusters or os.path.join(args.out, "candidate_clusters.csv")
    if not os.path.exists(clusters_path):
        print("%s not found: run the cluster subcommand first." % clusters_path, file=sys.stderr)
        return 2
    cl_rows = read_csv(clusters_path)
    rep_of = {r["pdb_id"]: r["cluster_id"] for r in cl_rows}
    sizes = {}
    for rep in rep_of.values():
        sizes[rep] = sizes.get(rep, 0) + 1
    missing = [r["pdb_id"] for r in cands if r["pdb_id"] not in rep_of]
    if missing:
        print("%d candidates have no cluster (e.g. %s). Run the cluster subcommand again." % (len(missing), missing[:3]), file=sys.stderr)
        return 2
    cc = read_cc(args.cc)
    log_rows, selected, todo = walk(cands, rep_of, cc, args.quota, args.cc_min, args.seed, args.batch)
    os.makedirs(args.out, exist_ok=True)
    write_csv(os.path.join(args.out, "selection_log.csv"), LOG_COLUMNS, log_rows)
    write_csv(os.path.join(args.out, "cc_todo.csv"), TODO_COLUMNS, todo)
    final = not todo
    counts = {}
    for r in log_rows:
        counts[r["decision"]] = counts.get(r["decision"], 0) + 1
    per_stratum = {s: sum(1 for x in selected if x["row"]["stratum"] == s) for s in STRATA}
    info = {"script_version": SCRIPT_VERSION, "python": platform.python_version(), "seed": args.seed,
            "quota_per_stratum": args.quota, "cc_min": args.cc_min, "batch": args.batch, "final": final,
            "decisions": counts, "selected_per_stratum": per_stratum, "n_cc_requested": len(todo),
            "pool_candidates_sha256": sha256_file(os.path.join(args.pool, "pool_candidates.csv")),
            "clusters_sha256": sha256_file(clusters_path),
            "cc_file_sha256": sha256_file(args.cc) if args.cc else None,
            "first_in_walk": {s: [r["pdb_id"] for r in log_rows if r["stratum"] == s][:5] for s in STRATA}}
    with open(os.path.join(args.out, "selection_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)
    print("decisions: %s" % counts)
    if not final:
        print("CC_mask is needed for %d entries (cc_todo.csv). Compute it, add it to the cc_mask file, and run select again." % len(todo))
        return 3
    total = len(selected)
    if args.entries:
        write_csv(args.entries, ENTRY_COLUMNS, entry_rows(selected, sizes))
        print("wrote %s" % args.entries)
    print("FINAL: %d entries selected (%s)" % (total, ", ".join("%s: %d" % kv for kv in per_stratum.items())))
    if total < 12:
        print("WARNING: below the floor of 12 entries (decision 5); the scope-cut order in decision 12 applies.")
    elif total < 20:
        print("Note: below the target of 20 entries.")
    return 0


def parse_args(argv):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    c = sub.add_parser("cluster", help="run MMseqs2 on the longest chains")
    c.add_argument("--pool", required=True, help="folder with pool_candidates.csv and pool_candidate_sequences.fasta")
    c.add_argument("--out", required=True)
    c.add_argument("--mmseqs", default="mmseqs")
    c.add_argument("--threads", type=int, default=1, help="keep 1 for a reproducible clustering")
    s = sub.add_parser("select", help="apply the selection rule")
    s.add_argument("--pool", required=True)
    s.add_argument("--out", required=True)
    s.add_argument("--clusters", default=None, help="default: OUT/candidate_clusters.csv")
    s.add_argument("--cc", default=None, help="cc_mask.csv from compute_cc_mask.py")
    s.add_argument("--entries", default=None, help="where to write the final list (only if the walk is complete)")
    s.add_argument("--seed", type=int, default=SEED)
    s.add_argument("--quota", type=int, default=QUOTA)
    s.add_argument("--cc-min", type=float, default=CC_MIN)
    s.add_argument("--batch", type=int, default=BATCH, help="entries per stratum whose CC_mask is requested in one round")
    return p.parse_args(argv)


def main(argv=None):
    args = parse_args(argv)
    try:
        return cmd_cluster(args) if args.cmd == "cluster" else cmd_select(args)
    except (ValueError, OSError) as e:
        print("ERROR: %s" % e, file=sys.stderr)
        return 2


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile compute_cc_mask.py
#!/usr/bin/env python3
"""Phase 3b: map-model correlation (CC_mask) of the deposited model against the deposited map, with Phenix.

Decision 4 of docs/SPEC.md: an entry needs CC_mask >= 0.70 from phenix.map_model_cc, with identical settings for
every entry. The command is

    phenix.map_model_cc <model.cif> <map.map> resolution=<resolution_A of the entry> ignore_symmetry_conflicts=True --force

Phenix 2.2.1 has retired that program: without --force it stops with "phenix.map_model_cc and phenix.model_map_cc are
replaced by phenix.map_correlations". So the script knows a few ways to call Phenix (VARIANTS below). For the first
entries it tries them one after the other until one gives a CC_mask; from then on that one variant is used for every entry
(it is written in the column "variant" of cc_mask.csv, and a later run reads it from there). For every entry listed in
cc_todo.csv the script downloads the model (RCSB) and the map (EMDB, .map.gz), runs Phenix, reads CC_mask from its
output, deletes the big files and appends one row to cc_mask.csv. The files of the next entries are downloaded while
Phenix works on the current one. Entries that already have a row are skipped, so the script can be stopped and
started again at any time.

A row has status "ok" (Phenix finished and printed a CC_mask) or "failed" (Phenix ran but gave no usable CC_mask,
timed out or was killed, or the file does not exist on the server, HTTP 404). select_entries.py counts "failed" as a
rejection. A temporary problem (download error, server busy) is not a result: no row is written and the entry is tried
again next time.

Exit codes: 0 every entry of the todo list has a row; 2 some entries have none (temporary problems);
3 Phenix cannot be run at all or the disk is full (nothing is written for the entry that was being processed).

    python compute_cc_mask.py --todo cc_todo.csv --out cc_mask.csv --phenix-env /path/to/phenix_env.sh
    python compute_cc_mask.py --diagnose --out cc_mask.csv --phenix-env /path/to/phenix_env.sh    (writes diagnostics.txt)
"""
from __future__ import annotations

import argparse
import concurrent.futures
import csv
import gzip
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import zlib

import requests

SCRIPT_VERSION = "1.1"
TIMEOUT_S = 1800
AHEAD = 2                      # entries whose files are downloaded while Phenix works on the current one
COLUMNS = ["pdb_id", "emdb_id", "status", "cc_mask", "resolution_used", "seconds", "variant", "phenix_version", "note"]
CC_RE = re.compile(r"\bCC_mask\b[\s:=|,()\[\]]*(-?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)", re.I)
UA = "mica-benchmark-2026 (academic benchmark; contact via GitHub Zohaib-Bioinfo)"
MIRROR = "https://files.wwpdb.org/pub/pdb/data/structures/all/mmCIF/%s.cif.gz"
STOP = threading.Event()       # set when the script ends early: running downloads give up

# (name, command with {model} {map} {res} filled in, only tried after Phenix complained about ignore_symmetry_conflicts)
VARIANTS = (
    ("map_model_cc --force",
     ["phenix.map_model_cc", "{model}", "{map}", "resolution={res}", "ignore_symmetry_conflicts=True", "--force"], False),
    ("map_model_cc --force first",
     ["phenix.map_model_cc", "--force", "{model}", "{map}", "resolution={res}", "ignore_symmetry_conflicts=True"], False),
    ("map_correlations",
     ["phenix.map_correlations", "{model}", "{map}", "resolution={res}", "ignore_symmetry_conflicts=True"], False),
    ("map_correlations full names",
     ["phenix.map_correlations", "input_files.model={model}", "input_files.map_in_1={map}", "map_model_cc.resolution={res}",
      "map_model_cc.ignore_symmetry_conflicts=True"], False),
    ("map_model_cc --force, no symmetry option",
     ["phenix.map_model_cc", "{model}", "{map}", "resolution={res}", "--force"], True),
    ("map_correlations, no symmetry option",
     ["phenix.map_correlations", "{model}", "{map}", "resolution={res}"], True),
)
TEXT_SUFFIXES = (".log", ".txt", ".out", ".json", ".csv", ".eff")


class Transient(Exception):
    """A problem that says nothing about the entry: try again later."""


class Permanent(Exception):
    """The file does not exist on the server."""


class Infrastructure(Exception):
    """Phenix cannot run or the disk is full: stop everything."""


# --------------------------------------------------------------------------------------
# Reading the output of Phenix
# --------------------------------------------------------------------------------------
def parse_cc_mask(text):
    """The first CC_mask value in the output of Phenix, and every line that carries one."""
    values, lines = [], []
    for line in text.splitlines():
        m = CC_RE.search(line)
        if m:
            values.append(float(m.group(1)))
            lines.append(line.strip())
    return (values[0] if values else None), lines


def short_error(text):
    """One line that says what went wrong."""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    for ln in lines:
        if ln.startswith("Sorry") or "Traceback" in ln:
            idx = lines.index(ln)
            return " | ".join(lines[idx: idx + 2])[:300]
    return (lines[-1] if lines else "no output")[:300]


def complains_about_option(text):
    """True if Phenix says it does not know the parameter ignore_symmetry_conflicts."""
    words = ("unknown", "unrecognized", "not recognized", "ambiguous", "invalid", "no such", "unexpected", "sorry")
    return any("ignore_symmetry_conflicts" in ln.lower() and any(w in ln.lower() for w in words) for ln in text.splitlines())


# --------------------------------------------------------------------------------------
# Files
# --------------------------------------------------------------------------------------
def download(url, dest, retries=3, timeout=180):
    """Downloads url to dest. Raises Permanent for HTTP 404 and Transient for everything else."""
    last = "unknown error"
    for attempt in range(1, retries + 1):
        part = dest + ".part"
        try:
            with requests.get(url, stream=True, timeout=timeout, headers={"User-Agent": UA}) as r:
                if r.status_code == 404:
                    raise Permanent("HTTP 404 for %s" % url)
                if r.status_code != 200:
                    raise IOError("HTTP %d" % r.status_code)
                with open(part, "wb") as f:
                    for block in r.iter_content(1 << 20):
                        if STOP.is_set():
                            raise Transient("stopped")
                        f.write(block)
            os.replace(part, dest)
            return
        except (Permanent, Transient):
            raise
        except OSError as e:
            if "No space left" in str(e):
                raise Infrastructure("disk full while downloading %s" % url)
            last = str(e)[:200]
        except requests.RequestException as e:
            last = "%s: %s" % (type(e).__name__, str(e)[:150])
        finally:
            if os.path.exists(part):
                os.remove(part)
        if attempt < retries:
            time.sleep(2 * attempt)
    raise Transient("download of %s failed after %d attempts: %s" % (url, retries, last))


def gunzip_if_needed(path):
    with open(path, "rb") as f:
        magic = f.read(2)
    if magic != b"\x1f\x8b":
        return
    tmp = path + ".raw"
    try:
        with gzip.open(path, "rb") as src, open(tmp, "wb") as dst:
            shutil.copyfileobj(src, dst, 1 << 20)
    except (OSError, EOFError, zlib.error) as e:
        if os.path.exists(tmp):
            os.remove(tmp)
        if "No space left" in str(e):
            raise Infrastructure("disk full while unpacking %s" % path)
        raise Transient("cannot unpack %s: %s" % (path, e))
    os.replace(tmp, path)


def fetch_model(row, workdir):
    dest = os.path.join(workdir, "%s.cif" % row["pdb_id"])
    try:
        download(row["cif_url"], dest)
    except Permanent:
        mirror = MIRROR % row["pdb_id"].lower()
        download(mirror, dest + ".gz")                      # the same file from the wwPDB archive
        os.replace(dest + ".gz", dest)
    gunzip_if_needed(dest)
    with open(dest, "rb") as f:
        head = f.read(200)
    if not head.lstrip().startswith(b"data_"):
        raise Transient("%s is not an mmCIF file (starts with %r)" % (row["cif_url"], head[:40]))
    return dest


def fetch_map(row, workdir):
    dest = os.path.join(workdir, "%s.map" % row["emdb_id"].replace("EMD-", "emd_"))
    download(row["map_url"], dest)
    gunzip_if_needed(dest)
    with open(dest, "rb") as f:
        head = f.read(1024)
    if len(head) < 1024 or head[208:212] != b"MAP ":
        raise Transient("%s is not an MRC/CCP4 map (header %r)" % (row["map_url"], head[:16]))
    return dest


def fetch_files(row, workdir):
    os.makedirs(workdir, exist_ok=True)
    return fetch_model(row, workdir), fetch_map(row, workdir)


# --------------------------------------------------------------------------------------
# Phenix
# --------------------------------------------------------------------------------------
def bash_phenix(env_script, args, timeout, cwd):
    """Runs a Phenix command with its environment loaded. Returns (return code, output) or (None, output) on timeout."""
    cmd = ["bash", "-c", 'source "$0" && exec "$@"', env_script] + list(args)
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, cwd=cwd, errors="replace")
    except subprocess.TimeoutExpired as e:
        out = (e.stdout or b"")
        out = out.decode("utf-8", "replace") if isinstance(out, bytes) else out
        return None, out
    return r.returncode, (r.stdout or "") + (r.stderr or "")


def phenix_version(env_script, workdir):
    rc, out = bash_phenix(env_script, ["phenix.version"], 180, workdir)
    lines = [ln.strip() for ln in out.splitlines() if ln.strip()]
    hit = [ln for ln in lines if re.search(r"\d+\.\d+", ln)]
    return (hit[0] if hit else (lines[0] if lines else "unknown"))[:80], rc


def new_text_files(workdir, before, limit=2000000):
    """Names and text of the small text files that Phenix wrote into workdir (some versions print nothing)."""
    names, chunks = [], []
    for name in sorted(set(os.listdir(workdir)) - before):
        path = os.path.join(workdir, name)
        if os.path.isfile(path) and name.lower().endswith(TEXT_SUFFIXES) and os.path.getsize(path) <= limit:
            with open(path, "r", errors="replace") as f:
                chunks.append(f.read())
            names.append(name)
    return names, "\n".join(chunks)


def run_phenix(env_script, model, mapfile, resolution, workdir, timeout, locked=None):
    """Returns (status, cc, variant, note, log text).

    Without `locked` the variants are tried one after the other until one gives a CC_mask. `locked` is the name of the
    variant that has already worked for an earlier entry: it is then the only one used, so that every entry gets the same
    settings and a failure is a result of the entry, not of the way Phenix was called."""
    log, reasons, complained, not_found = [], [], False, 0
    variants = [v for v in VARIANTS if v[0] == locked] if locked else list(VARIANTS)
    tried = []
    for name, template, after_complaint in variants:
        if after_complaint and not (complained or locked):
            continue
        tried.append(name)
        args = [a.format(model=os.path.basename(model), map=os.path.basename(mapfile), res=resolution) for a in template]
        before = set(os.listdir(workdir))
        rc, out = bash_phenix(env_script, args, timeout, workdir)
        entry = "$ %s\n%s\n[exit code %s]\n" % (" ".join(args), out[-200000:], "timeout" if rc is None else rc)
        if "No space left on device" in out:
            raise Infrastructure("disk full while running Phenix")
        if rc is None:
            log.append(entry + "=> %s: timeout after %d s\n" % (name, timeout))
            return "failed", None, name, "timeout after %d s" % timeout, "".join(log)     # another variant would take as long again
        if rc < 0 or rc == 137:
            note = "killed by signal %d (probably out of memory)" % (-rc if rc < 0 else 9)
            log.append(entry + "=> %s: %s\n" % (name, note))
            return "failed", None, name, note, "".join(log)
        value, lines = parse_cc_mask(out)
        if value is None and rc == 0:
            names, text = new_text_files(workdir, before)
            value, lines = parse_cc_mask(text)
            if names:
                entry += "[files written by Phenix: %s]\n" % ", ".join(names)
        if rc == 0 and value is not None and -1.0 <= value <= 1.0:
            note = "" if len(lines) == 1 else "%d CC_mask lines in the output, the first one is used" % len(lines)
            log.append(entry + "=> %s: CC_mask %s\n" % (name, value))
            return "ok", value, name, note, "".join(log)
        if rc == 127:
            not_found += 1
            reason = "program not found: %s" % short_error(out)
        elif rc != 0:
            reason = "phenix exited with %d: %s%s" % (rc, short_error(out), "" if value is None else " (its output does have CC_mask %s)" % value)
            if complains_about_option(out):
                complained = True
        elif value is None:
            reason = "no CC_mask in the output"
        else:
            reason = "CC_mask %s is outside -1..1" % value
        log.append(entry + "=> %s: %s\n" % (name, reason))
        reasons.append(reason)
    if tried and not_found == len(tried):
        raise Infrastructure("none of the Phenix programs could be started (%s): %s" % (", ".join(tried), reasons[-1]))
    last = tried[-1] if tried else (locked or VARIANTS[0][0])
    if len(reasons) == 1:
        note = reasons[0]
    else:
        note = "; ".join("%s: %s" % (n, r[:110]) for n, r in zip(tried, reasons))[:600]
    return "failed", None, last, note, "".join(log)


def diagnose(env_script, out_path, workdir):
    """Writes what Phenix says about its own programs, for the case that no variant works."""
    os.makedirs(workdir, exist_ok=True)
    commands = [["phenix.version"], ["phenix.map_correlations", "--help"], ["phenix.map_correlations"],
                ["phenix.map_model_cc", "--force", "--help"], ["phenix.map_model_cc", "--force"],
                ["bash", "-c", 'ls "$PHENIX/phenix_bin" "$PHENIX/bin" 2>/dev/null | grep -i -E "map_|_cc|validation_cryo|mtriage" | sort -u']]
    with open(out_path, "w", encoding="utf-8") as f:
        for args in commands:
            rc, out = bash_phenix(env_script, args, 180, workdir)
            f.write("$ %s\n[exit code %s]\n%s\n\n" % (" ".join(args), "timeout" if rc is None else rc, "\n".join(out.splitlines()[:300])[:30000]))
    print("diagnostics written to %s" % out_path, flush=True)
    return 0


# --------------------------------------------------------------------------------------
# The loop
# --------------------------------------------------------------------------------------
def read_csv(path):
    if not os.path.exists(path):
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def append_row(path, row):
    new = not os.path.exists(path) or os.path.getsize(path) == 0
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUMNS, extrasaction="ignore", lineterminator="\n")
        if new:
            w.writeheader()
        w.writerow({k: ("" if row.get(k) is None else row.get(k)) for k in COLUMNS})
        f.flush()
        os.fsync(f.fileno())


def locked_variant(rows):
    """The variant that gave the first CC_mask of an earlier run, if there was one."""
    known = {v[0] for v in VARIANTS}
    for r in rows:
        if r.get("status") == "ok" and r.get("variant") in known:
            return r["variant"]
    return None


def process(row, args, workdir, version, files, locked):
    t0 = time.time()
    try:
        model, mapfile = files()
        waited = time.time() - t0
        status, cc, variant, note, log = run_phenix(args.phenix_env, model, mapfile, row["resolution_A"], workdir, args.timeout, locked)
    except Permanent as e:
        waited = time.time() - t0
        status, cc, variant, note, log = "failed", None, locked or "", "download: %s" % e, "%s\n" % e
    seconds = round(time.time() - t0)
    return {"pdb_id": row["pdb_id"], "emdb_id": row["emdb_id"], "status": status,
            "cc_mask": "" if cc is None else "%.4f" % cc, "resolution_used": row["resolution_A"], "seconds": seconds,
            "variant": variant, "phenix_version": version, "note": note}, log, round(waited)


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--todo", default=None, help="cc_todo.csv from select_entries.py")
    p.add_argument("--out", required=True, help="cc_mask.csv (rows are appended)")
    p.add_argument("--phenix-env", required=True, help="path of phenix_env.sh")
    p.add_argument("--logs", default=None, help="folder for one log per entry (default: cc_logs next to --out)")
    p.add_argument("--work", default=None, help="scratch folder (default: cc_work next to --out)")
    p.add_argument("--limit", type=int, default=0, help="only the first N entries that still need a result")
    p.add_argument("--timeout", type=int, default=TIMEOUT_S, help="seconds allowed for one Phenix run")
    p.add_argument("--show-output", action="store_true", help="print the commands, exit codes and CC lines of the Phenix output")
    p.add_argument("--diagnose", action="store_true", help="write diagnostics.txt next to --out (Phenix help texts) and stop")
    args = p.parse_args(argv)
    base = os.path.dirname(os.path.abspath(args.out))
    logs = args.logs or os.path.join(base, "cc_logs")
    work = args.work or os.path.join(base, "cc_work")
    if args.diagnose:
        if not os.path.exists(args.phenix_env):
            print("ERROR: %s does not exist" % args.phenix_env, file=sys.stderr)
            return 3
        return diagnose(args.phenix_env, os.path.join(base, "diagnostics.txt"), os.path.join(work, "diagnose"))
    if not args.todo:
        p.error("--todo is required")
    os.makedirs(logs, exist_ok=True)
    todo = read_csv(args.todo)
    have = {r["pdb_id"] for r in read_csv(args.out)}
    pending = [r for r in todo if r["pdb_id"] not in have]
    if args.limit:
        pending = pending[: args.limit]
    print("compute_cc_mask %s: %d entries in the list, %d already done, %d to do now" % (SCRIPT_VERSION, len(todo), len(have & {r["pdb_id"] for r in todo}), len(pending)), flush=True)
    if not pending:
        return 0
    os.makedirs(work, exist_ok=True)
    version, rc = phenix_version(args.phenix_env, work)
    if rc == 127 or not os.path.exists(args.phenix_env):
        print("ERROR: Phenix does not start (exit code %s). Is %s the right file?" % (rc, args.phenix_env), file=sys.stderr)
        return 3
    print("Phenix: %s" % version, flush=True)
    locked = locked_variant(read_csv(args.out))
    if locked:
        print("variant of the earlier results (used for every entry): %s" % locked, flush=True)
    n_ok = n_failed = n_skipped = 0
    STOP.clear()
    pool = concurrent.futures.ThreadPoolExecutor(max_workers=max(1, AHEAD))
    futures = {}
    try:
        for i, row in enumerate(pending, 1):
            for nxt in pending[i - 1: i + AHEAD]:           # this entry and the next ones: their files are downloaded in the background
                if nxt["pdb_id"] not in futures:
                    futures[nxt["pdb_id"]] = pool.submit(fetch_files, nxt, os.path.join(work, nxt["pdb_id"]))
            files = futures.pop(row["pdb_id"]).result
            entry_dir = os.path.join(work, row["pdb_id"])
            try:
                result, log, waited = process(row, args, entry_dir, version, files, locked)
            except Transient as e:
                n_skipped += 1
                print("[%d/%d] %s  NOT DONE (temporary problem, will be tried again): %s" % (i, len(pending), row["pdb_id"], e), flush=True)
                continue
            except Infrastructure as e:
                print("ERROR: %s" % e, file=sys.stderr)
                return 3
            finally:
                shutil.rmtree(entry_dir, ignore_errors=True)
            with open(os.path.join(logs, "%s.log" % row["pdb_id"]), "w", encoding="utf-8") as f:
                f.write("entry %s (%s), stratum %s, resolution %s A\nscript compute_cc_mask %s, %s\nresult: %s\n\n%s"
                        % (row["pdb_id"], row["emdb_id"], row.get("stratum", ""), row["resolution_A"], SCRIPT_VERSION, version,
                           {k: result[k] for k in ("status", "cc_mask", "variant", "note")}, log))
            append_row(args.out, result)
            if result["status"] == "ok":
                n_ok += 1
                locked = locked or result["variant"]
            else:
                n_failed += 1
            print("[%d/%d] %s  %s  %s A  map %s MB: %s  (%d s, of which %d s waiting for the files)%s" % (
                i, len(pending), row["pdb_id"], row.get("stratum", ""), row["resolution_A"], row.get("map_size_mb", "?"),
                ("CC_mask %s [%s]" % (result["cc_mask"], result["variant"])) if result["status"] == "ok" else "FAILED: %s" % result["note"],
                result["seconds"], waited, "" if result["status"] != "ok" or not result["note"] else "  [" + result["note"] + "]"), flush=True)
            if args.show_output:
                for ln in log.splitlines():
                    if "CC_mask" in ln or ln.startswith("$") or ln.startswith("[exit") or ln.startswith("=>"):
                        print("    | " + ln[:200], flush=True)
    finally:
        STOP.set()                                              # running downloads give up, waiting ones are cancelled
        pool.shutdown(wait=True, cancel_futures=True)
        for pdb_id in futures:                                  # files of entries that were downloaded ahead but not used
            shutil.rmtree(os.path.join(work, pdb_id), ignore_errors=True)
    print("done: %d ok, %d failed (rejected), %d not done" % (n_ok, n_failed, n_skipped), flush=True)
    have = {r["pdb_id"] for r in read_csv(args.out)}
    return 0 if all(r["pdb_id"] in have for r in pending) else 2


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile novelty_check.py
#!/usr/bin/env python3
"""Phase 3c: how close is each selected entry to proteins that were in the PDB before 2026? (docs/SPEC.md decision 9)

Every unique chain sequence of every selected entry is searched with MMseqs2 against all protein chains of PDB entries
released before 2026-01-01. The entry's value is the highest sequence identity found over its chains. Two numbers are
kept, because identity over a short fragment says little about the whole chain:

  max_id_to_pre2026_pdb   highest identity among alignments that cover at least 50% of the query chain (0 if none).
                          This is the number of the spec; the sensitivity analysis keeps the entries at or below 0.30.
  max_id_any_coverage     highest identity among all alignments, whatever they cover.

Identity is the fraction of identical residues in the alignment (0 to 1). The reference is built from the wwPDB file
pdb_seqres.txt.gz minus every entry that the RCSB search lists as released on or after 2026-01-01 (all methods),
duplicates removed. Checks that stop the run: the pool's cryo-EM entries and the selected entries must all be in the
excluded list, and the reference must not be small.

    python novelty_check.py --entries data/entries_2026.csv --fasta data/pool/pool_candidate_sequences.fasta \\
        --cryoem-ids data/pool/pool_ids_all_cryoem.txt --out novelty_out
"""
from __future__ import annotations

import argparse
import csv
import datetime as dt
import gzip
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import time

import requests

SCRIPT_VERSION = "1.0"
SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
SEQRES_URL = "https://files.wwpdb.org/pub/pdb/derived_data/pdb_seqres.txt.gz"
CUTOFF = dt.date(2026, 1, 1)
MIN_QCOV = 0.5
MIN_REFERENCE_CHAINS = 100_000
M8_FIELDS = "query,target,fident,qcov,tcov,alnlen,evalue,bits"
OUT_COLUMNS = ["pdb_id", "n_chain_seqs", "max_id_to_pre2026_pdb", "best_hit", "best_hit_query", "best_hit_qcov",
               "max_id_any_coverage", "best_any_hit", "best_any_hit_query", "best_any_hit_qcov"]
UA = "mica-benchmark-2026 (academic benchmark; contact via GitHub Zohaib-Bioinfo)"


# --------------------------------------------------------------------------------------
# RCSB: entries released from 2026 on
# --------------------------------------------------------------------------------------
def terminal(attribute, operator, value):
    return {"type": "terminal", "service": "text", "parameters": {"attribute": attribute, "operator": operator, "value": value}}


def post_window(url, a, b, retries=4):
    body = {"query": {"type": "group", "logical_operator": "and", "nodes": [
        terminal("rcsb_accession_info.initial_release_date", "greater_or_equal", a.isoformat()),
        terminal("rcsb_accession_info.initial_release_date", "less", b.isoformat())]},
        "return_type": "entry", "request_options": {"paginate": {"start": 0, "rows": 1000}, "results_content_type": ["experimental"]}}
    last = ""
    for attempt in range(retries):
        try:
            r = requests.post(url, json=body, timeout=(15, 120), headers={"User-Agent": UA})
            if r.status_code == 204:
                return [], 0
            if r.status_code == 400:
                raise RuntimeError("the RCSB search did not accept the query: %s" % r.text[:300])
            r.raise_for_status()
            js = r.json()
            ids = [x["identifier"] if isinstance(x, dict) else str(x) for x in js.get("result_set", [])]
            return ids, js.get("total_count", len(ids))
        except requests.RequestException as e:
            last = str(e)[:200]
            time.sleep(2 * (attempt + 1))
    raise RuntimeError("the RCSB search failed after %d attempts: %s" % (retries, last))


def window_ids(url, a, b):
    ids, total = post_window(url, a, b)
    if total > len(ids):
        if (b - a).days <= 1:
            raise RuntimeError("search truncated for the single day %s: %d of %d hits" % (a, len(ids), total))
        mid = a + (b - a) // 2
        return window_ids(url, a, mid) + window_ids(url, mid, b)
    return ids


def released_since(url, start, through):
    """Every PDB entry (all methods) with an initial release date in [start, through)."""
    out, cur = set(), start
    while cur < through:
        nxt = min(cur + dt.timedelta(days=7), through)
        out.update(i.upper() for i in window_ids(url, cur, nxt))
        cur = nxt
    return out


# --------------------------------------------------------------------------------------
# The reference: pdb_seqres without the 2026 entries
# --------------------------------------------------------------------------------------
def download(url, dest, retries=3):
    last = ""
    for attempt in range(retries):
        part = dest + ".part"
        try:
            with requests.get(url, stream=True, timeout=(15, 120), headers={"User-Agent": UA}) as r:
                r.raise_for_status()
                with open(part, "wb") as f:
                    for block in r.iter_content(1 << 20):
                        f.write(block)
            os.replace(part, dest)
            return
        except (requests.RequestException, OSError) as e:
            last = str(e)[:200]
            time.sleep(3 * (attempt + 1))
        finally:
            if os.path.exists(part):
                os.remove(part)
    raise RuntimeError("download of %s failed after %d attempts: %s" % (url, retries, last))


def build_reference(seqres_gz, exclude, out_fasta):
    """Writes the protein chains of entries not in `exclude`, each distinct sequence once. Returns the counts."""
    stats = {"protein_chains_in_seqres": 0, "chains_of_excluded_entries": 0, "reference_unique_sequences": 0}
    seen = set()
    header = None
    with gzip.open(seqres_gz, "rt", encoding="utf-8", errors="replace") as f, open(out_fasta, "w", encoding="utf-8") as out:
        for line in f:
            if line.startswith(">"):
                header = line[1:].split()
                continue
            if header is None or len(header) < 2 or header[1] != "mol:protein":
                header = None
                continue
            seq = line.strip()
            name, header = header[0], None
            pid = name.split("_")[0].upper()
            stats["protein_chains_in_seqres"] += 1
            if pid in exclude:
                stats["chains_of_excluded_entries"] += 1
                continue
            if seq in seen:
                continue
            seen.add(seq)
            out.write(">%s\n%s\n" % (name, seq))
    stats["reference_unique_sequences"] = len(seen)
    return stats


# --------------------------------------------------------------------------------------
# Queries and search
# --------------------------------------------------------------------------------------
def read_fasta(path):
    seqs, name, buf = {}, None, []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    seqs[name] = "".join(buf)
                name, buf = line[1:].split()[0], []
            else:
                buf.append(line.strip())
    if name is not None:
        seqs[name] = "".join(buf)
    return seqs


def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def query_sequences(all_seqs, pdb_ids):
    """{entity name: sequence} of every entity of the given entries; each distinct sequence of an entry once."""
    out, missing = {}, []
    for pid in pdb_ids:
        mine = {n: s for n, s in all_seqs.items() if n.split("_E")[0] == pid}
        if not mine:
            missing.append(pid)
            continue
        seen = set()
        for n in sorted(mine):
            if mine[n] not in seen:
                seen.add(mine[n])
                out[n] = mine[n]
    if missing:
        raise ValueError("no sequences for %s in the FASTA file" % ", ".join(missing))
    return out


def run_search(queries, ref_fasta, binary, threads, workdir):
    qfa = os.path.join(workdir, "queries.fasta")
    with open(qfa, "w", encoding="utf-8") as f:
        for n in sorted(queries):
            f.write(">%s\n%s\n" % (n, queries[n]))
    m8 = os.path.join(workdir, "hits.m8")
    tmp = os.path.join(workdir, "mmseqs_tmp")
    cmd = [binary, "easy-search", qfa, ref_fasta, m8, tmp, "-s", "7.5", "-e", "1e-3", "--max-seqs", "2000",
           "--format-output", M8_FIELDS, "--threads", str(threads), "-v", "1"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("mmseqs failed (exit %d):\n%s\n%s" % (r.returncode, r.stdout[-1500:], r.stderr[-1500:]))
    hits = {}
    with open(m8, encoding="utf-8") as f:
        for line in f:
            q, t, fid, qcov, tcov, alnlen, ev, bits = line.rstrip("\n").split("\t")[:8]
            if "_" in t:
                t = t.split("_", 1)[0].upper() + "_" + t.split("_", 1)[1]          # 101m_A -> 101M_A
            hits.setdefault(q, []).append({"target": t, "id": float(fid), "qcov": float(qcov), "alnlen": int(alnlen), "evalue": float(ev)})
    return cmd, hits


def entry_values(pdb_ids, queries, hits):
    rows = []
    for pid in pdb_ids:
        names = [n for n in queries if n.split("_E")[0] == pid]
        best, best_any = (0.0, "", "", 0.0), (0.0, "", "", 0.0)
        for n in names:
            for h in hits.get(n, []):
                cand = (h["id"], h["target"], n, h["qcov"])
                if cand[0] > best_any[0]:
                    best_any = cand
                if h["qcov"] >= MIN_QCOV and cand[0] > best[0]:
                    best = cand
        rows.append({"pdb_id": pid, "n_chain_seqs": len(names),
                     "max_id_to_pre2026_pdb": "%.3f" % best[0], "best_hit": best[1], "best_hit_query": best[2],
                     "best_hit_qcov": "%.2f" % best[3] if best[1] else "",
                     "max_id_any_coverage": "%.3f" % best_any[0], "best_any_hit": best_any[1], "best_any_hit_query": best_any[2],
                     "best_any_hit_qcov": "%.2f" % best_any[3] if best_any[1] else ""})
    return rows


def write_csv(path, columns, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore", lineterminator="\n")
        w.writeheader()
        w.writerows(rows)


def update_entries(path, values):
    rows = read_csv(path)
    with open(path, newline="", encoding="utf-8") as f:
        columns = next(csv.reader(f))
    if "max_id_to_pre2026_pdb" not in columns:
        columns.append("max_id_to_pre2026_pdb")
    for r in rows:
        r["max_id_to_pre2026_pdb"] = values[r["pdb_id"]]
    tmp = path + ".tmp"
    write_csv(tmp, columns, rows)
    os.replace(tmp, path)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--entries", required=True, help="entries_2026.csv (its max_id_to_pre2026_pdb column is filled in)")
    p.add_argument("--fasta", required=True, help="pool_candidate_sequences.fasta")
    p.add_argument("--out", required=True)
    p.add_argument("--cryoem-ids", default=None, help="pool_ids_all_cryoem.txt: all of them must be in the excluded list")
    p.add_argument("--mmseqs", default="mmseqs")
    p.add_argument("--threads", type=int, default=2)
    p.add_argument("--through", default=None, help="last release date to exclude, default: a week from today")
    p.add_argument("--seqres", default=None, help="a local pdb_seqres.txt.gz (default: download)")
    p.add_argument("--search-url", default=SEARCH_URL)
    p.add_argument("--seqres-url", default=SEQRES_URL)
    p.add_argument("--min-reference-chains", type=int, default=MIN_REFERENCE_CHAINS)
    args = p.parse_args(argv)
    try:
        return run(args)
    except (ValueError, RuntimeError, OSError) as e:
        print("ERROR: %s" % e, file=sys.stderr)
        return 2


def run(args):
    if shutil.which(args.mmseqs) is None and not os.path.exists(args.mmseqs):
        raise RuntimeError("mmseqs not found (%s)" % args.mmseqs)
    os.makedirs(args.out, exist_ok=True)
    entries = read_csv(args.entries)
    pdb_ids = [r["pdb_id"] for r in entries]
    if not pdb_ids:
        raise ValueError("%s has no entries" % args.entries)
    queries = query_sequences(read_fasta(args.fasta), pdb_ids)
    print("%d entries, %d distinct chain sequences to search" % (len(pdb_ids), len(queries)), flush=True)

    through = dt.date.fromisoformat(args.through) if args.through else dt.date.today() + dt.timedelta(days=7)
    exclude = released_since(args.search_url, CUTOFF, through)
    print("RCSB: %d entries released from %s to %s (all methods): they are left out of the reference" % (len(exclude), CUTOFF, through), flush=True)
    if args.cryoem_ids:
        with open(args.cryoem_ids, encoding="utf-8") as f:
            cryo = {ln.strip().upper() for ln in f if ln.strip()}
        lost = sorted(cryo - exclude)
        if lost:
            raise RuntimeError("%d cryo-EM entries of the pool are missing from the excluded list (e.g. %s); the all-methods search "
                               "is incomplete" % (len(lost), ", ".join(lost[:5])))
    still = sorted(set(pdb_ids) - exclude)
    if still:
        raise RuntimeError("selected entries are not in the excluded list: %s" % ", ".join(still[:5]))

    seqres = args.seqres
    if not seqres:
        seqres = os.path.join(args.out, "pdb_seqres.txt.gz")
        print("downloading pdb_seqres.txt.gz ...", flush=True)
        download(args.seqres_url, seqres)
    ref = os.path.join(args.out, "reference_pre2026.fasta")
    stats = build_reference(seqres, exclude, ref)
    print("reference: %(reference_unique_sequences)d distinct protein sequences (%(protein_chains_in_seqres)d protein chains in "
          "pdb_seqres, %(chains_of_excluded_entries)d of them from 2026 entries)" % stats, flush=True)
    if stats["reference_unique_sequences"] < args.min_reference_chains:
        raise RuntimeError("the reference has only %d distinct sequences (expected at least %d); is pdb_seqres complete?"
                           % (stats["reference_unique_sequences"], args.min_reference_chains))

    work = tempfile.mkdtemp(prefix="novelty_")
    try:
        cmd, hits = run_search(queries, ref, args.mmseqs, args.threads, work)
        r = subprocess.run([args.mmseqs, "version"], capture_output=True, text=True)
        version = (r.stdout or r.stderr).strip()
    finally:
        shutil.rmtree(work, ignore_errors=True)
    rows = entry_values(pdb_ids, queries, hits)
    write_csv(os.path.join(args.out, "novelty.csv"), OUT_COLUMNS, rows)
    update_entries(args.entries, {r["pdb_id"]: r["max_id_to_pre2026_pdb"] for r in rows})
    info = {"script_version": SCRIPT_VERSION, "mmseqs_version": version, "mmseqs_command": [c if not c.startswith(tempfile.gettempdir()) else "<tmp>/" + os.path.basename(c) for c in cmd],
            "min_query_coverage": MIN_QCOV, "cutoff_date": CUTOFF.isoformat(), "excluded_through": through.isoformat(),
            "n_excluded_entries": len(exclude), "excluded_ids_sha256": hashlib.sha256("\n".join(sorted(exclude)).encode()).hexdigest(),
            "seqres_sha256": sha256_file(seqres), "reference_fasta_sha256": sha256_file(ref), "n_queries": len(queries),
            "n_entries": len(pdb_ids), "entries_at_or_below_0.30": sum(1 for r in rows if float(r["max_id_to_pre2026_pdb"]) <= 0.30),
            "python": platform.python_version(), "finished_utc": dt.datetime.now(dt.timezone.utc).isoformat(), **stats}
    with open(os.path.join(args.out, "novelty_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)
    if not args.seqres:
        os.remove(seqres)                  # 270 MB; its hash is in novelty_info.json
    os.remove(ref)                         # large; rebuilt from pdb_seqres and the excluded list
    vals = sorted(float(r["max_id_to_pre2026_pdb"]) for r in rows)
    print("novelty: %d entries; highest identity to a pre-2026 PDB chain (>=50%% of the chain covered): min %.2f, median %.2f, max %.2f; "
          "%d entries at or below 0.30" % (len(rows), vals[0], vals[len(vals) // 2], vals[-1], info["entries_at_or_below_0.30"]), flush=True)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile data/pool/pool_candidates.csv
pdb_id,emdb_id,stratum,release_date_pdb,emdb_map_release,emdb_status,emdb_release_verified,map_size_kb,resolution_A,recon_method,n_polymer_entities,poly_types,n_modeled_residues,n_unk,unk_fraction,n_chains,n_unique_seq,longest_entity,longest_seq_len,seqres_total_len,entities,n_emdb_links,emdb_ids_all,multi_recon_rows,first_fail,passes_cheap_filters,cif_url,map_url,note,error
10ED,EMD-75104,le3.0,2026-08-26,2026-08-26,REL,yes,131073,1.95,SINGLE PARTICLE,2,polypeptide(L),647,0,0.0,2,2,2,606,664,E1:len=58:copies=1:modeled=51|E2:len=606:copies=1:modeled=596,1,EMD-75104,False,,True,https://files.rcsb.org/download/10ED.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75104/map/emd_75104.map.gz,,
10FJ,EMD-75133,gt3.0_le4.0,2026-07-01,2026-07-01,REL,yes,87809,3.5,SINGLE PARTICLE,3,polypeptide(L),612,0,0.0,3,3,2,238,661,E1:len=185:copies=1:modeled=172|E2:len=238:copies=1:modeled=220|E3:len=238:copies=1:modeled=220,1,EMD-75133,False,,True,https://files.rcsb.org/download/10FJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75133/map/emd_75133.map.gz,,
10OY,EMD-75365,le3.0,2026-10-07,2026-10-07,REL,yes,55297,2.62,SINGLE PARTICLE,2,polypeptide(L),538,0,0.0,3,2,1,344,554,E1:len=344:copies=2:modeled=505|E2:len=210:copies=1:modeled=33,1,EMD-75365,False,,True,https://files.rcsb.org/download/10OY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75365/map/emd_75365.map.gz,,
10TL,EMD-71869,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,108001,3.5,SINGLE PARTICLE,2,polypeptide(L),574,0,0.0,4,2,1,388,393,E1:len=388:copies=2:modeled=564|E2:len=5:copies=2:modeled=10,1,EMD-71869,False,,True,https://files.rcsb.org/download/10TL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71869/map/emd_71869.map.gz,,
10TM,EMD-71871,le3.0,2026-05-27,2026-05-27,REL,yes,67109,3.0,SINGLE PARTICLE,2,polypeptide(L),574,0,0.0,4,2,1,388,393,E1:len=388:copies=2:modeled=564|E2:len=5:copies=2:modeled=10,1,EMD-71871,False,,True,https://files.rcsb.org/download/10TM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71871/map/emd_71871.map.gz,,
10VM,EMD-75491,gt3.0_le4.0,2026-09-30,2026-09-30,REL,yes,226493,3.08,SINGLE PARTICLE,3,polypeptide(L),680,0,0.0,3,3,3,274,728,E1:len=250:copies=1:modeled=238|E2:len=204:copies=1:modeled=199|E3:len=274:copies=1:modeled=243,1,EMD-75491,False,,True,https://files.rcsb.org/download/10VM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75491/map/emd_75491.map.gz,,
10XW,EMD-75516,gt3.0_le4.0,2026-07-15,2026-07-15,REL,yes,256001,3.5,SINGLE PARTICLE,1,polypeptide(L),240,0,0.0,1,1,1,241,241,E1:len=241:copies=1:modeled=240,1,EMD-75516,False,,True,https://files.rcsb.org/download/10XW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75516/map/emd_75516.map.gz,,
11CV,EMD-75626,le3.0,2026-05-20,2026-05-20,REL,yes,151733,2.66,SINGLE PARTICLE,1,polypeptide(L),692,0,0.0,4,1,1,326,326,E1:len=326:copies=4:modeled=692,1,EMD-75626,False,,True,https://files.rcsb.org/download/11CV.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75626/map/emd_75626.map.gz,,
11HL,EMD-75695,le3.0,2026-05-13,2026-05-13,REL,yes,1048577,2.8,SINGLE PARTICLE,1,polypeptide(L),675,0,0.0,3,1,1,605,605,E1:len=605:copies=3:modeled=675,1,EMD-75695,False,,True,https://files.rcsb.org/download/11HL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75695/map/emd_75695.map.gz,,
11HM,EMD-75696,gt3.0_le4.0,2026-08-26,2026-08-26,REL,yes,864001,3.56,SINGLE PARTICLE,3,polypeptide(L),413,0,0.0,3,3,1,1258,1732,E1:len=1258:copies=1:modeled=184|E2:len=240:copies=1:modeled=120|E3:len=234:copies=1:modeled=109,1,EMD-75696,False,,True,https://files.rcsb.org/download/11HM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75696/map/emd_75696.map.gz,,
11IT,EMD-75726,gt3.0_le4.0,2026-08-26,2026-08-26,REL,yes,536871,3.62,SINGLE PARTICLE,3,polypeptide(L),419,0,0.0,3,3,1,1258,1733,E1:len=1258:copies=1:modeled=184|E2:len=240:copies=1:modeled=125|E3:len=235:copies=1:modeled=110,1,EMD-75726,False,,True,https://files.rcsb.org/download/11IT.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75726/map/emd_75726.map.gz,,
11JF,EMD-75735,le3.0,2026-05-20,2026-05-20,REL,yes,67109,2.85,SINGLE PARTICLE,2,polypeptide(L),683,0,0.0,3,2,2,1724,2617,E1:len=893:copies=2:modeled=534|E2:len=1724:copies=1:modeled=149,1,EMD-75735,False,,True,https://files.rcsb.org/download/11JF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75735/map/emd_75735.map.gz,,
11KM,EMD-75776,le3.0,2026-07-22,2026-07-22,REL,yes,151733,2.97,SINGLE PARTICLE,1,polypeptide(L),646,0,0.0,1,1,1,676,676,E1:len=676:copies=1:modeled=646,1,EMD-75776,False,,True,https://files.rcsb.org/download/11KM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75776/map/emd_75776.map.gz,,
11OR,EMD-75892,gt3.0_le4.0,2026-04-08,2026-04-08,REL,yes,226493,3.2,SINGLE PARTICLE,3,polypeptide(L),545,0,0.0,3,3,1,217,548,E1:len=217:copies=1:modeled=214|E2:len=211:copies=1:modeled=211|E3:len=120:copies=1:modeled=120,1,EMD-75892,False,,True,https://files.rcsb.org/download/11OR.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75892/map/emd_75892.map.gz,,
11OU,EMD-75893,gt3.0_le4.0,2026-04-08,2026-04-08,REL,yes,226493,3.2,SINGLE PARTICLE,3,polypeptide(L),549,0,0.0,3,3,3,219,553,E1:len=120:copies=1:modeled=120|E2:len=214:copies=1:modeled=214|E3:len=219:copies=1:modeled=215,1,EMD-75893,False,,True,https://files.rcsb.org/download/11OU.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-75893/map/emd_75893.map.gz,,
11TD,EMD-76031,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,186625,3.7,SINGLE PARTICLE,3,polypeptide(L),627,0,0.0,3,3,1,475,918,E1:len=475:copies=1:modeled=396|E2:len=229:copies=1:modeled=123|E3:len=214:copies=1:modeled=108,1,EMD-76031,False,,True,https://files.rcsb.org/download/11TD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76031/map/emd_76031.map.gz,,
11TF,EMD-76032,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,186625,3.4,SINGLE PARTICLE,3,polypeptide(L),627,0,0.0,3,3,1,475,918,E1:len=475:copies=1:modeled=396|E2:len=229:copies=1:modeled=123|E3:len=214:copies=1:modeled=108,1,EMD-76032,False,,True,https://files.rcsb.org/download/11TF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76032/map/emd_76032.map.gz,,
11TH,EMD-76035,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,186625,3.5,SINGLE PARTICLE,3,polypeptide(L),627,0,0.0,3,3,3,475,918,E1:len=229:copies=1:modeled=123|E2:len=214:copies=1:modeled=108|E3:len=475:copies=1:modeled=396,1,EMD-76035,False,,True,https://files.rcsb.org/download/11TH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76035/map/emd_76035.map.gz,,
11TK,EMD-76037,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,186625,3.54,SINGLE PARTICLE,3,polypeptide(L),620,0,0.0,3,3,1,475,918,E1:len=475:copies=1:modeled=389|E2:len=229:copies=1:modeled=123|E3:len=214:copies=1:modeled=108,1,EMD-76037,False,,True,https://files.rcsb.org/download/11TK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76037/map/emd_76037.map.gz,,
11TL,EMD-76038,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,186625,3.9,SINGLE PARTICLE,3,polypeptide(L),620,0,0.0,3,3,1,475,918,E1:len=475:copies=1:modeled=389|E2:len=229:copies=1:modeled=123|E3:len=214:copies=1:modeled=108,1,EMD-76038,False,,True,https://files.rcsb.org/download/11TL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76038/map/emd_76038.map.gz,,
11WN,EMD-76136,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,151733,3.2,SINGLE PARTICLE,1,polypeptide(L),649,0,0.0,1,1,1,676,676,E1:len=676:copies=1:modeled=649,1,EMD-76136,False,,True,https://files.rcsb.org/download/11WN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76136/map/emd_76136.map.gz,,
11WO,EMD-76137,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,151733,3.3,SINGLE PARTICLE,1,polypeptide(L),657,0,0.0,1,1,1,676,676,E1:len=676:copies=1:modeled=657,1,EMD-76137,False,,True,https://files.rcsb.org/download/11WO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76137/map/emd_76137.map.gz,,
11WP,EMD-76138,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,151733,3.2,SINGLE PARTICLE,1,polypeptide(L),638,0,0.0,1,1,1,676,676,E1:len=676:copies=1:modeled=638,1,EMD-76138,False,,True,https://files.rcsb.org/download/11WP.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76138/map/emd_76138.map.gz,,
11WQ,EMD-76139,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,151733,3.2,SINGLE PARTICLE,1,polypeptide(L),638,0,0.0,1,1,1,676,676,E1:len=676:copies=1:modeled=638,1,EMD-76139,False,,True,https://files.rcsb.org/download/11WQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76139/map/emd_76139.map.gz,,
11XX,EMD-76175,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,442369,3.6,SINGLE PARTICLE,1,polypeptide(L),553,0,0.0,1,1,1,553,553,E1:len=553:copies=1:modeled=553,1,EMD-76175,False,,True,https://files.rcsb.org/download/11XX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76175/map/emd_76175.map.gz,,
11ZV,EMD-76232,le3.0,2026-05-13,2026-05-13,REL,yes,95552,2.84,SINGLE PARTICLE,2,polypeptide(L),240,0,0.0,2,2,2,176,351,E1:len=175:copies=1:modeled=113|E2:len=176:copies=1:modeled=127,1,EMD-76232,False,,True,https://files.rcsb.org/download/11ZV.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76232/map/emd_76232.map.gz,,
12IY,EMD-76463,le3.0,2026-07-15,2026-07-15,REL,yes,67109,2.94,SINGLE PARTICLE,1,polypeptide(L),277,0,0.0,1,1,1,374,374,E1:len=374:copies=1:modeled=277,1,EMD-76463,False,,True,https://files.rcsb.org/download/12IY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76463/map/emd_76463.map.gz,,
12OY,EMD-76655,gt3.0_le4.0,2026-05-13,2026-05-13,REL,yes,629857,3.6,SINGLE PARTICLE,2,polypeptide(L),410,0,0.0,2,2,1,304,431,E1:len=304:copies=1:modeled=285|E2:len=127:copies=1:modeled=125,1,EMD-76655,False,,True,https://files.rcsb.org/download/12OY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76655/map/emd_76655.map.gz,,
12OZ,EMD-76656,gt3.0_le4.0,2026-05-13,2026-05-13,REL,yes,629857,3.3,SINGLE PARTICLE,2,polypeptide(L),410,0,0.0,2,2,1,304,431,E1:len=304:copies=1:modeled=285|E2:len=127:copies=1:modeled=125,1,EMD-76656,False,,True,https://files.rcsb.org/download/12OZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76656/map/emd_76656.map.gz,,
12PB,EMD-76659,gt3.0_le4.0,2026-05-13,2026-05-13,REL,yes,629857,3.1,SINGLE PARTICLE,2,polypeptide(L),410,0,0.0,2,2,2,304,431,E1:len=127:copies=1:modeled=125|E2:len=304:copies=1:modeled=285,1,EMD-76659,False,,True,https://files.rcsb.org/download/12PB.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76659/map/emd_76659.map.gz,,
12QJ,EMD-76684,gt3.0_le4.0,2026-06-03,2026-06-03,REL,yes,186625,3.35,SINGLE PARTICLE,3,polypeptide(L),630,0,0.0,3,3,1,394,630,E1:len=394:copies=1:modeled=394|E2:len=125:copies=1:modeled=125|E3:len=111:copies=1:modeled=111,1,EMD-76684,False,,True,https://files.rcsb.org/download/12QJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76684/map/emd_76684.map.gz,,
12RP,EMD-76713,le3.0,2026-06-17,2026-06-17,REL,yes,287966,2.78,SINGLE PARTICLE,1,polypeptide(L),595,0,0.0,3,1,1,1271,1271,E1:len=1271:copies=3:modeled=595,1,EMD-76713,False,,True,https://files.rcsb.org/download/12RP.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76713/map/emd_76713.map.gz,,
13BD,EMD-76936,le3.0,2026-06-17,2026-06-17,REL,yes,287966,2.65,SINGLE PARTICLE,1,polypeptide(L),461,0,0.0,2,1,1,1271,1271,E1:len=1271:copies=2:modeled=461,1,EMD-76936,False,,True,https://files.rcsb.org/download/13BD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76936/map/emd_76936.map.gz,,
13BF,EMD-76938,le3.0,2026-07-15,2026-07-15,REL,yes,67109,3.0,SINGLE PARTICLE,3,polypeptide(L),487,0,0.0,3,3,1,221,519,E1:len=221:copies=1:modeled=221|E2:len=215:copies=1:modeled=215|E3:len=83:copies=1:modeled=51,1,EMD-76938,False,,True,https://files.rcsb.org/download/13BF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76938/map/emd_76938.map.gz,,
13BR,EMD-76947,gt3.0_le4.0,2026-07-15,2026-07-15,REL,yes,67109,3.1,SINGLE PARTICLE,3,polypeptide(L),384,0,0.0,3,3,3,284,741,E1:len=215:copies=1:modeled=103|E2:len=242:copies=1:modeled=121|E3:len=284:copies=1:modeled=160,1,EMD-76947,False,,True,https://files.rcsb.org/download/13BR.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76947/map/emd_76947.map.gz,,
13BS,EMD-76948,gt3.0_le4.0,2026-07-15,2026-07-15,REL,yes,131073,3.1,SINGLE PARTICLE,3,polypeptide(L),581,0,0.0,3,3,2,226,594,E1:len=154:copies=1:modeled=145|E2:len=226:copies=1:modeled=222|E3:len=214:copies=1:modeled=214,1,EMD-76948,False,,True,https://files.rcsb.org/download/13BS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76948/map/emd_76948.map.gz,,
13KK,EMD-77112,gt3.0_le4.0,2026-07-15,2026-07-15,REL,yes,2726,3.76,SINGLE PARTICLE,1,polypeptide(L),462,0,0.0,14,1,1,102,102,E1:len=102:copies=14:modeled=462,1,EMD-77112,False,,True,https://files.rcsb.org/download/13KK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-77112/map/emd_77112.map.gz,,
13KW,EMD-77126,gt3.0_le4.0,2026-07-15,2026-07-15,REL,yes,171501,4.0,SINGLE PARTICLE,1,polypeptide(L),511,0,0.0,7,1,1,102,102,E1:len=102:copies=7:modeled=511,1,EMD-77126,False,,True,https://files.rcsb.org/download/13KW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-77126/map/emd_77126.map.gz,,
20ZN,EMD-67436,le3.0,2026-08-05,2026-08-05,REL,yes,226493,2.76,SINGLE PARTICLE,1,polypeptide(L),608,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=608,1,EMD-67436,False,,True,https://files.rcsb.org/download/20ZN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67436/map/emd_67436.map.gz,,
21AH,EMD-67447,gt3.0_le4.0,2026-04-08,2026-04-08,REL,yes,174457,3.18,SINGLE PARTICLE,1,polypeptide(L),268,0,0.0,1,1,1,366,366,E1:len=366:copies=1:modeled=268,1,EMD-67447,False,,True,https://files.rcsb.org/download/21AH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67447/map/emd_67447.map.gz,,
21AO,EMD-67523,le3.0,2026-05-06,2026-05-06,REL,yes,108001,2.85,SINGLE PARTICLE,3,polypeptide(L),546,0,0.0,3,3,3,321,553,E1:len=125:copies=1:modeled=125|E2:len=107:copies=1:modeled=107|E3:len=321:copies=1:modeled=314,1,EMD-67523,False,,True,https://files.rcsb.org/download/21AO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67523/map/emd_67523.map.gz,,
21AQ,EMD-67524,le3.0,2026-04-29,2026-04-29,REL,yes,131073,2.97,SINGLE PARTICLE,3,polypeptide(L),546,0,0.0,3,3,3,321,552,E1:len=123:copies=1:modeled=123|E2:len=108:copies=1:modeled=108|E3:len=321:copies=1:modeled=315,1,EMD-67524,False,,True,https://files.rcsb.org/download/21AQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67524/map/emd_67524.map.gz,,
21DI,EMD-67586,gt3.0_le4.0,2026-04-22,2026-04-22,REL,yes,28312,3.5,SINGLE PARTICLE,1,polypeptide(L),477,0,0.0,9,1,1,134,134,E1:len=134:copies=9:modeled=477,1,EMD-67586,False,,True,https://files.rcsb.org/download/21DI.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67586/map/emd_67586.map.gz,,
21FH,EMD-67625,le3.0,2026-05-20,2026-05-20,REL,yes,108001,2.66,SINGLE PARTICLE,1,polypeptide(L),500,0,0.0,1,1,1,527,527,E1:len=527:copies=1:modeled=500,1,EMD-67625,False,,True,https://files.rcsb.org/download/21FH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67625/map/emd_67625.map.gz,,
21FI,EMD-67626,le3.0,2026-05-20,2026-05-20,REL,yes,108001,2.52,SINGLE PARTICLE,1,polypeptide(L),502,0,0.0,1,1,1,502,502,E1:len=502:copies=1:modeled=502,1,EMD-67626,False,,True,https://files.rcsb.org/download/21FI.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67626/map/emd_67626.map.gz,,
21FJ,EMD-67627,gt3.0_le4.0,2026-05-20,2026-05-20,REL,yes,28312,3.18,SINGLE PARTICLE,1,polypeptide(L),500,0,0.0,1,1,1,527,527,E1:len=527:copies=1:modeled=500,1,EMD-67627,False,,True,https://files.rcsb.org/download/21FJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67627/map/emd_67627.map.gz,,
21FK,EMD-67628,gt3.0_le4.0,2026-05-20,2026-05-20,REL,yes,28312,3.29,SINGLE PARTICLE,1,polypeptide(L),499,0,0.0,1,1,1,504,504,E1:len=504:copies=1:modeled=499,1,EMD-67628,False,,True,https://files.rcsb.org/download/21FK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67628/map/emd_67628.map.gz,,
21NP,EMD-67846,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,131073,3.3,SINGLE PARTICLE,1,polypeptide(L),422,0,0.0,1,1,1,429,429,E1:len=429:copies=1:modeled=422,1,EMD-67846,False,,True,https://files.rcsb.org/download/21NP.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67846/map/emd_67846.map.gz,,
21NQ,EMD-67847,le3.0,2026-05-27,2026-05-27,REL,yes,186625,2.8,SINGLE PARTICLE,1,polypeptide(L),414,0,0.0,1,1,1,429,429,E1:len=429:copies=1:modeled=414,1,EMD-67847,False,,True,https://files.rcsb.org/download/21NQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67847/map/emd_67847.map.gz,,
21NX,EMD-67855,le3.0,2026-08-26,2026-08-26,REL,yes,87809,2.99,SINGLE PARTICLE,2,polypeptide(L),595,0,0.0,3,2,2,263,488,E1:len=225:copies=1:modeled=171|E2:len=263:copies=2:modeled=424,1,EMD-67855,False,,True,https://files.rcsb.org/download/21NX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67855/map/emd_67855.map.gz,,
21OR,EMD-67869,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,186625,3.2,SINGLE PARTICLE,1,polypeptide(L),414,0,0.0,1,1,1,429,429,E1:len=429:copies=1:modeled=414,1,EMD-67869,False,,True,https://files.rcsb.org/download/21OR.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67869/map/emd_67869.map.gz,,
21TW,EMD-67994,le3.0,2026-04-29,2026-04-29,REL,yes,226493,2.43,SINGLE PARTICLE,2,polypeptide(L),638,0,0.0,4,2,1,363,521,E1:len=363:copies=1:modeled=224|E2:len=158:copies=3:modeled=414,1,EMD-67994,False,,True,https://files.rcsb.org/download/21TW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67994/map/emd_67994.map.gz,,
21TY,EMD-67995,le3.0,2026-09-09,2026-09-09,REL,yes,33959,2.93,SINGLE PARTICLE,1,polypeptide(L),662,0,0.0,1,1,1,748,748,E1:len=748:copies=1:modeled=662,1,EMD-67995,False,,True,https://files.rcsb.org/download/21TY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67995/map/emd_67995.map.gz,,
21VL,EMD-68036,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,256001,3.3,SINGLE PARTICLE,2,polypeptide(L),601,0,0.0,2,2,1,522,704,E1:len=522:copies=1:modeled=424|E2:len=182:copies=1:modeled=177,1,EMD-68036,False,,True,https://files.rcsb.org/download/21VL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68036/map/emd_68036.map.gz,,
21WL,EMD-68052,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,186625,3.7,SINGLE PARTICLE,1,polypeptide(L),452,0,0.0,1,1,1,468,468,E1:len=468:copies=1:modeled=452,1,EMD-68052,False,,True,https://files.rcsb.org/download/21WL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68052/map/emd_68052.map.gz,,
22FX,EMD-68251,le3.0,2026-03-04,2026-03-04,REL,yes,76996,1.24,SINGLE PARTICLE,1,polypeptide(L),173,0,0.0,1,1,1,182,182,E1:len=182:copies=1:modeled=173,1,EMD-68251,False,,True,https://files.rcsb.org/download/22FX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68251/map/emd_68251.map.gz,,
22JY,EMD-68389,le3.0,2026-02-04,2026-02-04,REL,yes,256001,2.2,SINGLE PARTICLE,1,polypeptide(L),255,0,0.0,5,1,1,51,51,E1:len=51:copies=5:modeled=255,1,EMD-68389,False,,True,https://files.rcsb.org/download/22JY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68389/map/emd_68389.map.gz,,
22OF,EMD-68536,gt3.0_le4.0,2026-07-01,2026-07-01,REL,yes,108001,3.6,SINGLE PARTICLE,2,polypeptide(L),515,0,0.0,2,2,1,309,550,E1:len=309:copies=1:modeled=306|E2:len=241:copies=1:modeled=209,1,EMD-68536,False,,True,https://files.rcsb.org/download/22OF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68536/map/emd_68536.map.gz,,
22PD,EMD-68583,le3.0,2026-09-16,2026-09-16,REL,yes,28312,2.95,SINGLE PARTICLE,3,polypeptide(L),436,0,0.0,3,3,3,200,440,E1:len=128:copies=1:modeled=128|E2:len=112:copies=1:modeled=112|E3:len=200:copies=1:modeled=196,1,EMD-68583,False,,True,https://files.rcsb.org/download/22PD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68583/map/emd_68583.map.gz,,
22PL,EMD-68584,gt3.0_le4.0,2026-08-26,2026-08-26,REL,yes,95552,3.3,SINGLE PARTICLE,3,polypeptide(L),585,0,0.0,3,3,3,311,664,E1:len=141:copies=1:modeled=127|E2:len=212:copies=1:modeled=206|E3:len=311:copies=1:modeled=252,1,EMD-68584,False,,True,https://files.rcsb.org/download/22PL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68584/map/emd_68584.map.gz,,
22RD,EMD-68616,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,171501,3.38,SINGLE PARTICLE,1,polypeptide(L),269,0,0.0,1,1,1,333,333,E1:len=333:copies=1:modeled=269,1,EMD-68616,False,,True,https://files.rcsb.org/download/22RD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68616/map/emd_68616.map.gz,,
22WP,EMD-68735,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,67109,3.33,SINGLE PARTICLE,1,polypeptide(L),257,0,0.0,1,1,1,336,336,E1:len=336:copies=1:modeled=257,1,EMD-68735,False,,True,https://files.rcsb.org/download/22WP.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68735/map/emd_68735.map.gz,,
22WR,EMD-68737,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,67109,3.2,SINGLE PARTICLE,1,polypeptide(L),257,0,0.0,1,1,1,336,336,E1:len=336:copies=1:modeled=257,1,EMD-68737,False,,True,https://files.rcsb.org/download/22WR.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68737/map/emd_68737.map.gz,,
22WT,EMD-68739,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,70305,3.21,SINGLE PARTICLE,1,polypeptide(L),252,0,0.0,1,1,1,280,280,E1:len=280:copies=1:modeled=252,1,EMD-68739,False,,True,https://files.rcsb.org/download/22WT.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68739/map/emd_68739.map.gz,,
23IC,EMD-68988,gt3.0_le4.0,2026-08-12,2026-08-12,REL,yes,186625,3.5,SINGLE PARTICLE,1,polypeptide(L),425,0,0.0,1,1,1,499,499,E1:len=499:copies=1:modeled=425,1,EMD-68988,False,,True,https://files.rcsb.org/download/23IC.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68988/map/emd_68988.map.gz,,
23IF,EMD-68990,gt3.0_le4.0,2026-08-12,2026-08-12,REL,yes,186625,3.3,SINGLE PARTICLE,1,polypeptide(L),427,0,0.0,1,1,1,499,499,E1:len=499:copies=1:modeled=427,1,EMD-68990,False,,True,https://files.rcsb.org/download/23IF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68990/map/emd_68990.map.gz,,
23IG,EMD-68992,gt3.0_le4.0,2026-08-12,2026-08-12,REL,yes,186625,3.7,SINGLE PARTICLE,1,polypeptide(L),401,0,0.0,1,1,1,499,499,E1:len=499:copies=1:modeled=401,1,EMD-68992,False,,True,https://files.rcsb.org/download/23IG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68992/map/emd_68992.map.gz,,
23WM,EMD-69321,gt3.0_le4.0,2026-08-05,2026-08-05,REL,yes,322487,3.13,SINGLE PARTICLE,2,polypeptide(L),265,0,0.0,2,2,1,1552,1594,E1:len=1552:copies=1:modeled=257|E2:len=42:copies=1:modeled=8,1,EMD-69321,False,,True,https://files.rcsb.org/download/23WM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-69321/map/emd_69321.map.gz,,
23WN,EMD-69322,le3.0,2026-08-05,2026-08-05,REL,yes,322487,2.98,SINGLE PARTICLE,2,polypeptide(L),233,0,0.0,2,2,1,1513,1555,E1:len=1513:copies=1:modeled=225|E2:len=42:copies=1:modeled=8,1,EMD-69322,False,,True,https://files.rcsb.org/download/23WN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-69322/map/emd_69322.map.gz,,
23WO,EMD-69323,gt3.0_le4.0,2026-08-05,2026-08-05,REL,yes,322487,3.5,SINGLE PARTICLE,2,polypeptide(L),221,0,0.0,2,2,1,1503,1545,E1:len=1503:copies=1:modeled=210|E2:len=42:copies=1:modeled=11,1,EMD-69323,False,,True,https://files.rcsb.org/download/23WO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-69323/map/emd_69323.map.gz,,
25IF,EMD-80132,gt3.0_le4.0,2026-07-01,2026-07-01,REL,yes,67109,3.42,SINGLE PARTICLE,2,polypeptide(L),361,0,0.0,2,2,1,400,476,E1:len=400:copies=1:modeled=287|E2:len=76:copies=1:modeled=74,1,EMD-80132,False,,True,https://files.rcsb.org/download/25IF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80132/map/emd_80132.map.gz,,
25PV,EMD-80292,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,186625,3.03,SINGLE PARTICLE,3,polypeptide(L),685,0,0.0,3,3,1,427,861,E1:len=427:copies=1:modeled=251|E2:len=220:copies=1:modeled=220|E3:len=214:copies=1:modeled=214,1,EMD-80292,False,,True,https://files.rcsb.org/download/25PV.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80292/map/emd_80292.map.gz,,
25RV,EMD-80331,le3.0,2026-09-30,2026-09-30,REL,yes,67109,2.46,SINGLE PARTICLE,1,polypeptide(L),698,0,0.0,2,1,1,724,724,E1:len=724:copies=2:modeled=698,1,EMD-80331,False,,True,https://files.rcsb.org/download/25RV.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80331/map/emd_80331.map.gz,,
25RW,EMD-80332,le3.0,2026-09-30,2026-09-30,REL,yes,67109,2.35,SINGLE PARTICLE,1,polypeptide(L),698,0,0.0,2,1,1,724,724,E1:len=724:copies=2:modeled=698,1,EMD-80332,False,,True,https://files.rcsb.org/download/25RW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80332/map/emd_80332.map.gz,,
26CW,EMD-66424,le3.0,2026-09-30,2026-09-30,REL,yes,67109,2.46,SINGLE PARTICLE,1,polypeptide(L),476,0,0.0,4,1,1,119,119,E1:len=119:copies=4:modeled=476,1,EMD-66424,False,,True,https://files.rcsb.org/download/26CW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66424/map/emd_66424.map.gz,,
26DE,EMD-80553,le3.0,2026-08-26,2026-08-26,REL,yes,256001,2.73,SINGLE PARTICLE,2,polypeptide(L),504,0,0.0,2,2,2,376,519,E1:len=143:copies=1:modeled=139|E2:len=376:copies=1:modeled=365,1,EMD-80553,False,,True,https://files.rcsb.org/download/26DE.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80553/map/emd_80553.map.gz,,
26DK,EMD-80558,gt3.0_le4.0,2026-08-26,2026-08-26,REL,yes,16385,3.5,SINGLE PARTICLE,1,polypeptide(L),191,0,0.0,1,1,1,197,197,E1:len=197:copies=1:modeled=191,1,EMD-80558,False,,True,https://files.rcsb.org/download/26DK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80558/map/emd_80558.map.gz,,
26GP,EMD-80643,gt3.0_le4.0,2026-08-26,2026-08-26,REL,yes,16385,3.8,SINGLE PARTICLE,1,polypeptide(L),247,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=247,1,EMD-80643,False,,True,https://files.rcsb.org/download/26GP.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80643/map/emd_80643.map.gz,,
26PF,EMD-80806,gt3.0_le4.0,2026-08-12,2026-08-12,REL,yes,219489,3.1,SINGLE PARTICLE,3,polypeptide(L),636,0,0.0,3,3,3,864,1093,E1:len=120:copies=1:modeled=118|E2:len=109:copies=1:modeled=108|E3:len=864:copies=1:modeled=410,1,EMD-80806,False,,True,https://files.rcsb.org/download/26PF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80806/map/emd_80806.map.gz,,
26QF,EMD-80823,gt3.0_le4.0,2026-09-02,2026-09-02,REL,yes,67109,3.22,SINGLE PARTICLE,1,polypeptide(L),378,0,0.0,1,1,1,394,394,E1:len=394:copies=1:modeled=378,1,EMD-80823,False,,True,https://files.rcsb.org/download/26QF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80823/map/emd_80823.map.gz,,
26QG,EMD-80824,gt3.0_le4.0,2026-09-02,2026-09-02,REL,yes,95552,3.2,SINGLE PARTICLE,1,polypeptide(L),381,0,0.0,1,1,1,394,394,E1:len=394:copies=1:modeled=381,1,EMD-80824,False,,True,https://files.rcsb.org/download/26QG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80824/map/emd_80824.map.gz,,
26QH,EMD-80825,gt3.0_le4.0,2026-09-02,2026-09-02,REL,yes,95552,3.23,SINGLE PARTICLE,1,polypeptide(L),390,0,0.0,1,1,1,394,394,E1:len=394:copies=1:modeled=390,1,EMD-80825,False,,True,https://files.rcsb.org/download/26QH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80825/map/emd_80825.map.gz,,
26QI,EMD-80826,gt3.0_le4.0,2026-09-02,2026-09-02,REL,yes,67109,3.43,SINGLE PARTICLE,1,polypeptide(L),388,0,0.0,1,1,1,394,394,E1:len=394:copies=1:modeled=388,1,EMD-80826,False,,True,https://files.rcsb.org/download/26QI.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80826/map/emd_80826.map.gz,,
26QJ,EMD-80827,gt3.0_le4.0,2026-09-02,2026-09-02,REL,yes,67109,3.34,SINGLE PARTICLE,1,polypeptide(L),378,0,0.0,1,1,1,394,394,E1:len=394:copies=1:modeled=378,1,EMD-80827,False,,True,https://files.rcsb.org/download/26QJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80827/map/emd_80827.map.gz,,
26QM,EMD-80829,gt3.0_le4.0,2026-09-02,2026-09-02,REL,yes,95552,3.07,SINGLE PARTICLE,1,polypeptide(L),380,0,0.0,1,1,1,394,394,E1:len=394:copies=1:modeled=380,1,EMD-80829,False,,True,https://files.rcsb.org/download/26QM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80829/map/emd_80829.map.gz,,
26QS,EMD-80832,le3.0,2026-09-02,2026-09-02,REL,yes,95552,2.88,SINGLE PARTICLE,1,polypeptide(L),391,0,0.0,1,1,1,391,391,E1:len=391:copies=1:modeled=391,1,EMD-80832,False,,True,https://files.rcsb.org/download/26QS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80832/map/emd_80832.map.gz,,
26QT,EMD-80833,gt3.0_le4.0,2026-09-02,2026-09-02,REL,yes,95552,3.06,SINGLE PARTICLE,1,polypeptide(L),381,0,0.0,1,1,1,394,394,E1:len=394:copies=1:modeled=381,1,EMD-80833,False,,True,https://files.rcsb.org/download/26QT.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80833/map/emd_80833.map.gz,,
26VA,EMD-80903,gt3.0_le4.0,2026-07-08,2026-07-08,REL,yes,67109,3.4,SINGLE PARTICLE,1,polypeptide(L),500,0,0.0,1,1,1,624,624,E1:len=624:copies=1:modeled=500,1,EMD-80903,False,,True,https://files.rcsb.org/download/26VA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80903/map/emd_80903.map.gz,,
26VB,EMD-80904,gt3.0_le4.0,2026-07-08,2026-07-08,REL,yes,67109,3.7,SINGLE PARTICLE,1,polypeptide(L),500,0,0.0,1,1,1,624,624,E1:len=624:copies=1:modeled=500,1,EMD-80904,False,,True,https://files.rcsb.org/download/26VB.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80904/map/emd_80904.map.gz,,
26VE,EMD-80907,gt3.0_le4.0,2026-07-08,2026-07-08,REL,yes,67109,3.7,SINGLE PARTICLE,1,polypeptide(L),484,0,0.0,1,1,1,624,624,E1:len=624:copies=1:modeled=484,1,EMD-80907,False,,True,https://files.rcsb.org/download/26VE.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80907/map/emd_80907.map.gz,,
26WC,EMD-80928,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,108001,3.63,SINGLE PARTICLE,3,polypeptide(L),610,0,0.0,6,3,1,187,410,E1:len=187:copies=2:modeled=166|E2:len=116:copies=2:modeled=230|E3:len=107:copies=2:modeled=214,1,EMD-80928,False,,True,https://files.rcsb.org/download/26WC.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80928/map/emd_80928.map.gz,,
28IZ,EMD-56538,le3.0,2026-08-19,2026-08-19,REL,yes,67109,2.83,SINGLE PARTICLE,1,polypeptide(L),520,0,0.0,2,1,1,267,267,E1:len=267:copies=2:modeled=520,1,EMD-56538,False,,True,https://files.rcsb.org/download/28IZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56538/map/emd_56538.map.gz,,
28QL,EMD-56749,le3.0,2026-03-04,2026-03-04,REL,yes,287966,2.72,SINGLE PARTICLE,2,polypeptide(L),388,0,0.0,2,2,2,227,390,E1:len=163:copies=1:modeled=161|E2:len=227:copies=1:modeled=227,1,EMD-56749,False,,True,https://files.rcsb.org/download/28QL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56749/map/emd_56749.map.gz,,
28QM,EMD-56750,gt3.0_le4.0,2026-02-25,2026-02-25,REL,yes,359662,3.4,SINGLE PARTICLE,2,polypeptide(L),532,0,0.0,2,2,1,373,532,E1:len=373:copies=1:modeled=373|E2:len=159:copies=1:modeled=159,1,EMD-56750,False,,True,https://files.rcsb.org/download/28QM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56750/map/emd_56750.map.gz,,
28QN,EMD-56751,gt3.0_le4.0,2026-02-25,2026-02-25,REL,yes,359662,3.3,SINGLE PARTICLE,2,polypeptide(L),534,0,0.0,2,2,2,374,534,E1:len=160:copies=1:modeled=160|E2:len=374:copies=1:modeled=374,1,EMD-56751,False,,True,https://files.rcsb.org/download/28QN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56751/map/emd_56751.map.gz,,
28VJ,EMD-56885,le3.0,2026-04-08,2026-04-08,REL,yes,108001,2.79,SINGLE PARTICLE,2,polypeptide(L),413,0,0.0,2,2,2,489,559,E1:len=70:copies=1:modeled=55|E2:len=489:copies=1:modeled=358,1,EMD-56885,False,,True,https://files.rcsb.org/download/28VJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56885/map/emd_56885.map.gz,,
28WZ,EMD-56931,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,67109,3.5,SINGLE PARTICLE,3,polypeptide(L),644,0,0.0,3,3,1,687,1213,E1:len=687:copies=1:modeled=223|E2:len=456:copies=1:modeled=371|E3:len=70:copies=1:modeled=50,1,EMD-56931,False,,True,https://files.rcsb.org/download/28WZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56931/map/emd_56931.map.gz,,
31CJ,EMD-58290,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,233645,3.3,SINGLE PARTICLE,1,polypeptide(L),496,0,0.0,2,1,1,534,534,E1:len=534:copies=2:modeled=496,1,EMD-58290,False,,True,https://files.rcsb.org/download/31CJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-58290/map/emd_58290.map.gz,,
31CK,EMD-58291,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,55297,3.5,SINGLE PARTICLE,1,polypeptide(L),496,0,0.0,2,1,1,534,534,E1:len=534:copies=2:modeled=496,1,EMD-58291,False,,True,https://files.rcsb.org/download/31CK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-58291/map/emd_58291.map.gz,,
31ZN,EMD-58734,le3.0,2026-08-19,2026-08-19,REL,yes,44958,2.9,SINGLE PARTICLE,3,polypeptide(L),649,0,0.0,3,3,2,442,1075,E1:len=311:copies=1:modeled=104|E2:len=442:copies=1:modeled=309|E3:len=322:copies=1:modeled=236,1,EMD-58734,False,,True,https://files.rcsb.org/download/31ZN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-58734/map/emd_58734.map.gz,,
32PZ,EMD-59064,le3.0,2026-08-19,2026-08-19,REL,yes,108001,2.83,SINGLE PARTICLE,2,polypeptide(L),627,0,0.0,2,2,2,549,695,E1:len=146:copies=1:modeled=135|E2:len=549:copies=1:modeled=492,1,EMD-59064,False,,True,https://files.rcsb.org/download/32PZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-59064/map/emd_59064.map.gz,,
32QB,EMD-59066,le3.0,2026-08-19,2026-08-19,REL,yes,108001,2.85,SINGLE PARTICLE,2,polypeptide(L),628,0,0.0,2,2,2,549,695,E1:len=146:copies=1:modeled=136|E2:len=549:copies=1:modeled=492,1,EMD-59066,False,,True,https://files.rcsb.org/download/32QB.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-59066/map/emd_59066.map.gz,,
33AS,EMD-59308,gt3.0_le4.0,2026-09-30,2026-09-30,REL,yes,287966,3.61,SINGLE PARTICLE,1,polypeptide(L),473,0,0.0,1,1,1,488,488,E1:len=488:copies=1:modeled=473,1,EMD-59308,False,,True,https://files.rcsb.org/download/33AS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-59308/map/emd_59308.map.gz,,
33AU,EMD-59316,le3.0,2026-09-30,2026-09-30,REL,yes,287966,3.0,SINGLE PARTICLE,1,polypeptide(L),473,0,0.0,1,1,1,488,488,E1:len=488:copies=1:modeled=473,1,EMD-59316,False,,True,https://files.rcsb.org/download/33AU.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-59316/map/emd_59316.map.gz,,
33AV,EMD-59317,gt3.0_le4.0,2026-09-30,2026-09-30,REL,yes,131073,3.9,SINGLE PARTICLE,1,polypeptide(L),480,0,0.0,1,1,1,488,488,E1:len=488:copies=1:modeled=480,1,EMD-59317,False,,True,https://files.rcsb.org/download/33AV.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-59317/map/emd_59317.map.gz,,
33AW,EMD-59318,gt3.0_le4.0,2026-09-30,2026-09-30,REL,yes,67109,3.84,SINGLE PARTICLE,1,polypeptide(L),465,0,0.0,1,1,1,482,482,E1:len=482:copies=1:modeled=465,1,EMD-59318,False,,True,https://files.rcsb.org/download/33AW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-59318/map/emd_59318.map.gz,,
35TL,EMD-77181,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,95552,3.93,SINGLE PARTICLE,3,polypeptide(L),334,0,0.0,3,3,1,290,727,E1:len=290:copies=1:modeled=114|E2:len=223:copies=1:modeled=115|E3:len=214:copies=1:modeled=105,1,EMD-77181,False,,True,https://files.rcsb.org/download/35TL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-77181/map/emd_77181.map.gz,,
36BB,EMD-77348,gt3.0_le4.0,2026-07-15,2026-07-15,REL,yes,536871,3.6,SINGLE PARTICLE,3,polypeptide(L),418,0,0.0,3,3,1,472,905,E1:len=472:copies=1:modeled=122|E2:len=236:copies=1:modeled=108|E3:len=197:copies=1:modeled=188,1,EMD-77348,False,,True,https://files.rcsb.org/download/36BB.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-77348/map/emd_77348.map.gz,,
36CQ,EMD-77390,le3.0,2026-06-17,2026-06-17,REL,yes,536871,2.76,SINGLE PARTICLE,3,polypeptide(L),632,0,0.0,3,3,2,427,640,E1:len=38:copies=1:modeled=31|E2:len=427:copies=1:modeled=426|E3:len=175:copies=1:modeled=175,1,EMD-77390,False,,True,https://files.rcsb.org/download/36CQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-77390/map/emd_77390.map.gz,,
43JF,EMD-67166,gt3.0_le4.0,2026-09-23,2026-05-13,REL,yes,171501,3.62,SINGLE PARTICLE,1,polypeptide(L),45,0,0.0,1,1,1,750,750,E1:len=750:copies=1:modeled=45,1,EMD-67166,False,,True,https://files.rcsb.org/download/43JF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67166/map/emd_67166.map.gz,,
43JG,EMD-67166,gt3.0_le4.0,2026-09-23,2026-05-13,REL,yes,171501,3.62,SINGLE PARTICLE,1,polypeptide(L),153,0,0.0,1,1,1,164,164,E1:len=164:copies=1:modeled=153,1,EMD-67166,False,,True,https://files.rcsb.org/download/43JG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67166/map/emd_67166.map.gz,,
43JH,EMD-67166,gt3.0_le4.0,2026-09-23,2026-05-13,REL,yes,171501,3.62,SINGLE PARTICLE,1,polypeptide(L),169,0,0.0,1,1,1,171,171,E1:len=171:copies=1:modeled=169,1,EMD-67166,False,,True,https://files.rcsb.org/download/43JH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67166/map/emd_67166.map.gz,,
43MU,EMD-81997,le3.0,2026-09-16,2026-09-16,REL,yes,464858,2.76,SINGLE PARTICLE,3,polypeptide(L),626,0,0.0,3,3,1,299,808,E1:len=299:copies=1:modeled=198|E2:len=271:copies=1:modeled=208|E3:len=238:copies=1:modeled=220,1,EMD-81997,False,,True,https://files.rcsb.org/download/43MU.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-81997/map/emd_81997.map.gz,,
43YY,EMD-82402,le3.0,2026-08-05,2026-08-05,REL,yes,364501,2.3,SINGLE PARTICLE,3,polypeptide(L),633,0,0.0,3,3,1,487,760,E1:len=487:copies=1:modeled=398|E2:len=144:copies=1:modeled=125|E3:len=129:copies=1:modeled=110,1,EMD-82402,False,,True,https://files.rcsb.org/download/43YY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-82402/map/emd_82402.map.gz,,
46KE,EMD-83847,gt3.0_le4.0,2026-10-07,2026-10-07,REL,yes,536871,3.1,SINGLE PARTICLE,1,polypeptide(L),191,0,0.0,1,1,1,197,197,E1:len=197:copies=1:modeled=191,1,EMD-83847,False,,True,https://files.rcsb.org/download/46KE.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-83847/map/emd_83847.map.gz,,
9CWP,EMD-45969,gt3.0_le4.0,2026-01-21,2026-01-21,REL,yes,536871,3.2,SINGLE PARTICLE,2,polypeptide(L),263,0,0.0,2,2,1,193,341,E1:len=193:copies=1:modeled=193|E2:len=148:copies=1:modeled=70,1,EMD-45969,False,,True,https://files.rcsb.org/download/9CWP.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-45969/map/emd_45969.map.gz,,
9CWQ,EMD-45971,le3.0,2026-01-21,2026-01-21,REL,yes,536871,2.8,SINGLE PARTICLE,1,polypeptide(L),248,0,0.0,1,1,1,304,304,E1:len=304:copies=1:modeled=248,1,EMD-45971,False,,True,https://files.rcsb.org/download/9CWQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-45971/map/emd_45971.map.gz,,
9E94,EMD-47768,gt3.0_le4.0,2026-02-11,2026-02-11,REL,yes,67109,3.67,SINGLE PARTICLE,1,polypeptide(L),440,0,0.0,2,1,1,313,313,E1:len=313:copies=2:modeled=440,1,EMD-47768,False,,True,https://files.rcsb.org/download/9E94.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-47768/map/emd_47768.map.gz,,
9FTE,EMD-50743,gt3.0_le4.0,2026-01-14,2026-01-14,REL,yes,108001,3.82,SINGLE PARTICLE,2,polypeptide(L),362,0,0.0,2,2,1,426,573,E1:len=426:copies=1:modeled=255|E2:len=147:copies=1:modeled=107,1,EMD-50743,False,,True,https://files.rcsb.org/download/9FTE.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-50743/map/emd_50743.map.gz,,
9H2Q,EMD-51814,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,67109,3.8,SINGLE PARTICLE,2,polypeptide(L),542,0,0.0,2,2,1,674,983,E1:len=674:copies=1:modeled=242|E2:len=309:copies=1:modeled=300,1,EMD-51814,False,,True,https://files.rcsb.org/download/9H2Q.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-51814/map/emd_51814.map.gz,,
9HJQ,EMD-52224,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,32001,3.8,SINGLE PARTICLE,1,polypeptide(L),658,0,0.0,2,1,1,401,401,E1:len=401:copies=2:modeled=658,1,EMD-52224,False,,True,https://files.rcsb.org/download/9HJQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-52224/map/emd_52224.map.gz,,
9I3T,EMD-52615,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,536871,3.7,SINGLE PARTICLE,3,polypeptide(L),629,0,0.0,3,3,1,492,938,E1:len=492:copies=1:modeled=395|E2:len=231:copies=1:modeled=125|E3:len=215:copies=1:modeled=109,1,EMD-52615,False,,True,https://files.rcsb.org/download/9I3T.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-52615/map/emd_52615.map.gz,,
9I5N,EMD-52634,gt3.0_le4.0,2026-02-25,2026-02-25,REL,yes,108001,3.4,SINGLE PARTICLE,3,polypeptide(L),368,0,0.0,3,3,3,143,368,E1:len=118:copies=1:modeled=118|E2:len=107:copies=1:modeled=107|E3:len=143:copies=1:modeled=143,1,EMD-52634,False,,True,https://files.rcsb.org/download/9I5N.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-52634/map/emd_52634.map.gz,,
9I8C,EMD-52713,le3.0,2026-02-18,2026-02-18,REL,yes,67109,2.7,SINGLE PARTICLE,1,polypeptide(L),338,0,0.0,1,1,1,338,338,E1:len=338:copies=1:modeled=338,1,EMD-52713,False,,True,https://files.rcsb.org/download/9I8C.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-52713/map/emd_52713.map.gz,,
9IB1,EMD-52789,gt3.0_le4.0,2026-03-04,2026-03-04,REL,yes,536871,3.2,SINGLE PARTICLE,3,polypeptide(L),421,0,0.0,3,3,1,1288,1950,E1:len=1288:copies=1:modeled=198|E2:len=214:copies=1:modeled=106|E3:len=448:copies=1:modeled=117,1,EMD-52789,False,,True,https://files.rcsb.org/download/9IB1.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-52789/map/emd_52789.map.gz,,
9IB3,EMD-52791,gt3.0_le4.0,2026-03-04,2026-03-04,REL,yes,536871,3.3,SINGLE PARTICLE,3,polypeptide(L),421,0,0.0,3,3,1,1288,1950,E1:len=1288:copies=1:modeled=198|E2:len=214:copies=1:modeled=106|E3:len=448:copies=1:modeled=117,1,EMD-52791,False,,True,https://files.rcsb.org/download/9IB3.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-52791/map/emd_52791.map.gz,,
9IBW,EMD-52814,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,442369,3.22,SINGLE PARTICLE,3,polypeptide(L),629,0,0.0,3,3,1,488,925,E1:len=488:copies=1:modeled=403|E2:len=215:copies=1:modeled=105|E3:len=222:copies=1:modeled=121,1,EMD-52814,False,,True,https://files.rcsb.org/download/9IBW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-52814/map/emd_52814.map.gz,,
9J6R,EMD-61180,gt3.0_le4.0,2026-03-04,2026-03-04,REL,yes,67109,3.2,SINGLE PARTICLE,1,polypeptide(L),545,0,0.0,1,1,1,562,562,E1:len=562:copies=1:modeled=545,1,EMD-61180,False,,True,https://files.rcsb.org/download/9J6R.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61180/map/emd_61180.map.gz,,
9J6S,EMD-61181,le3.0,2026-03-04,2026-03-04,REL,yes,67109,2.8,SINGLE PARTICLE,1,polypeptide(L),545,0,0.0,1,1,1,562,562,E1:len=562:copies=1:modeled=545,1,EMD-61181,False,,True,https://files.rcsb.org/download/9J6S.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61181/map/emd_61181.map.gz,,
9J6T,EMD-61182,gt3.0_le4.0,2026-03-04,2026-03-04,REL,yes,67109,3.4,SINGLE PARTICLE,1,polypeptide(L),528,0,0.0,1,1,1,555,555,E1:len=555:copies=1:modeled=528,1,EMD-61182,False,,True,https://files.rcsb.org/download/9J6T.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61182/map/emd_61182.map.gz,,
9J6U,EMD-61184,le3.0,2026-03-04,2026-03-04,REL,yes,67109,2.9,SINGLE PARTICLE,1,polypeptide(L),531,0,0.0,1,1,1,555,555,E1:len=555:copies=1:modeled=531,1,EMD-61184,False,,True,https://files.rcsb.org/download/9J6U.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61184/map/emd_61184.map.gz,,
9J6V,EMD-61185,gt3.0_le4.0,2026-03-04,2026-03-04,REL,yes,67109,3.5,SINGLE PARTICLE,1,polypeptide(L),528,0,0.0,1,1,1,555,555,E1:len=555:copies=1:modeled=528,1,EMD-61185,False,,True,https://files.rcsb.org/download/9J6V.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61185/map/emd_61185.map.gz,,
9JBS,EMD-61324,le3.0,2026-03-11,2026-03-11,REL,yes,108001,2.88,SINGLE PARTICLE,1,polypeptide(L),415,0,0.0,1,1,1,569,569,E1:len=569:copies=1:modeled=415,1,EMD-61324,False,,True,https://files.rcsb.org/download/9JBS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61324/map/emd_61324.map.gz,,
9JBT,EMD-61325,gt3.0_le4.0,2026-03-11,2026-03-11,REL,yes,108001,3.2,SINGLE PARTICLE,1,polypeptide(L),409,0,0.0,1,1,1,569,569,E1:len=569:copies=1:modeled=409,1,EMD-61325,False,,True,https://files.rcsb.org/download/9JBT.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61325/map/emd_61325.map.gz,,
9JBU,EMD-61326,gt3.0_le4.0,2026-03-11,2026-03-11,REL,yes,108001,3.8,SINGLE PARTICLE,1,polypeptide(L),415,0,0.0,1,1,1,569,569,E1:len=569:copies=1:modeled=415,1,EMD-61326,False,,True,https://files.rcsb.org/download/9JBU.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61326/map/emd_61326.map.gz,,
9JKH,EMD-61561,gt3.0_le4.0,2026-04-08,2026-04-08,REL,yes,55297,3.2,SINGLE PARTICLE,1,polypeptide(L),529,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=529,1,EMD-61561,False,,True,https://files.rcsb.org/download/9JKH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61561/map/emd_61561.map.gz,,
9JKI,EMD-61562,le3.0,2026-04-08,2026-04-08,REL,yes,55297,2.69,SINGLE PARTICLE,1,polypeptide(L),529,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=529,1,EMD-61562,False,,True,https://files.rcsb.org/download/9JKI.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61562/map/emd_61562.map.gz,,
9JKJ,EMD-61563,le3.0,2026-04-08,2026-04-08,REL,yes,67109,2.69,SINGLE PARTICLE,1,polypeptide(L),530,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=530,1,EMD-61563,False,,True,https://files.rcsb.org/download/9JKJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61563/map/emd_61563.map.gz,,
9JKK,EMD-61564,le3.0,2026-04-08,2026-04-08,REL,yes,67109,3.0,SINGLE PARTICLE,1,polypeptide(L),530,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=530,1,EMD-61564,False,,True,https://files.rcsb.org/download/9JKK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61564/map/emd_61564.map.gz,,
9JKM,EMD-61565,gt3.0_le4.0,2026-04-08,2026-04-08,REL,yes,32001,3.44,SINGLE PARTICLE,1,polypeptide(L),541,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=541,1,EMD-61565,False,,True,https://files.rcsb.org/download/9JKM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61565/map/emd_61565.map.gz,,
9JZI,EMD-61914,gt3.0_le4.0,2026-03-11,2026-03-11,REL,yes,4245,3.09,SINGLE PARTICLE,1,polypeptide(L),337,0,0.0,1,1,1,393,393,E1:len=393:copies=1:modeled=337,1,EMD-61914,False,,True,https://files.rcsb.org/download/9JZI.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61914/map/emd_61914.map.gz,,
9JZJ,EMD-61915,le3.0,2026-03-11,2026-03-11,REL,yes,4245,2.78,SINGLE PARTICLE,1,polypeptide(L),337,0,0.0,1,1,1,393,393,E1:len=393:copies=1:modeled=337,1,EMD-61915,False,,True,https://files.rcsb.org/download/9JZJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61915/map/emd_61915.map.gz,,
9JZN,EMD-61929,gt3.0_le4.0,2026-03-11,2026-03-11,REL,yes,2545,3.15,SINGLE PARTICLE,1,polypeptide(L),336,0,0.0,1,1,1,393,393,E1:len=393:copies=1:modeled=336,1,EMD-61929,False,,True,https://files.rcsb.org/download/9JZN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61929/map/emd_61929.map.gz,,
9K7G,EMD-62149,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,30119,3.57,SINGLE PARTICLE,1,polypeptide(L),371,0,0.0,1,1,1,413,413,E1:len=413:copies=1:modeled=371,1,EMD-62149,False,,True,https://files.rcsb.org/download/9K7G.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62149/map/emd_62149.map.gz,,
9K7S,EMD-62152,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,67109,3.69,SINGLE PARTICLE,1,polypeptide(L),371,0,0.0,1,1,1,384,384,E1:len=384:copies=1:modeled=371,1,EMD-62152,False,,True,https://files.rcsb.org/download/9K7S.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62152/map/emd_62152.map.gz,,
9K88,EMD-62164,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,67109,3.93,SINGLE PARTICLE,1,polypeptide(L),371,0,0.0,1,1,1,384,384,E1:len=384:copies=1:modeled=371,1,EMD-62164,False,,True,https://files.rcsb.org/download/9K88.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62164/map/emd_62164.map.gz,,
9K8Y,EMD-62182,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,67109,3.76,SINGLE PARTICLE,1,polypeptide(L),361,0,0.0,1,1,1,376,376,E1:len=376:copies=1:modeled=361,1,EMD-62182,False,,True,https://files.rcsb.org/download/9K8Y.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62182/map/emd_62182.map.gz,,
9K90,EMD-62183,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,67109,3.82,SINGLE PARTICLE,1,polypeptide(L),371,0,0.0,1,1,1,384,384,E1:len=384:copies=1:modeled=371,1,EMD-62183,False,,True,https://files.rcsb.org/download/9K90.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62183/map/emd_62183.map.gz,,
9KKN,EMD-62393,le3.0,2026-05-27,2026-05-27,REL,yes,67109,2.72,SINGLE PARTICLE,1,polypeptide(L),404,0,0.0,1,1,1,637,637,E1:len=637:copies=1:modeled=404,1,EMD-62393,False,,True,https://files.rcsb.org/download/9KKN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62393/map/emd_62393.map.gz,,
9KKO,EMD-62394,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,67109,3.25,SINGLE PARTICLE,1,polypeptide(L),404,0,0.0,1,1,1,637,637,E1:len=637:copies=1:modeled=404,1,EMD-62394,False,,True,https://files.rcsb.org/download/9KKO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62394/map/emd_62394.map.gz,,
9KQ5,EMD-62496,gt3.0_le4.0,2026-06-03,2026-06-03,REL,yes,67109,3.21,SINGLE PARTICLE,1,polypeptide(L),404,0,0.0,1,1,1,637,637,E1:len=637:copies=1:modeled=404,1,EMD-62496,False,,True,https://files.rcsb.org/download/9KQ5.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62496/map/emd_62496.map.gz,,
9KQM,EMD-62507,gt3.0_le4.0,2026-06-03,2026-06-03,REL,yes,67109,3.5,SINGLE PARTICLE,1,polypeptide(L),403,0,0.0,1,1,1,637,637,E1:len=637:copies=1:modeled=403,1,EMD-62507,False,,True,https://files.rcsb.org/download/9KQM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62507/map/emd_62507.map.gz,,
9KSL,EMD-62552,le3.0,2026-06-03,2026-06-03,REL,yes,500001,1.81,SINGLE PARTICLE,1,polypeptide(L),300,0,0.0,1,1,1,315,315,E1:len=315:copies=1:modeled=300,1,EMD-62552,False,,True,https://files.rcsb.org/download/9KSL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62552/map/emd_62552.map.gz,,
9KVS,EMD-62600,le3.0,2026-06-10,2026-06-10,REL,yes,42593,2.81,SINGLE PARTICLE,1,polypeptide(L),274,0,0.0,1,1,1,538,538,E1:len=538:copies=1:modeled=274,1,EMD-62600,False,,True,https://files.rcsb.org/download/9KVS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62600/map/emd_62600.map.gz,,
9L19,EMD-62738,gt3.0_le4.0,2026-04-15,2026-04-15,REL,yes,67109,3.33,SINGLE PARTICLE,1,polypeptide(L),543,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=543,1,EMD-62738,False,,True,https://files.rcsb.org/download/9L19.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62738/map/emd_62738.map.gz,,
9L1A,EMD-62739,le3.0,2026-04-15,2026-04-15,REL,yes,67109,2.75,SINGLE PARTICLE,1,polypeptide(L),542,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=542,1,EMD-62739,False,,True,https://files.rcsb.org/download/9L1A.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62739/map/emd_62739.map.gz,,
9L1B,EMD-62740,le3.0,2026-04-15,2026-04-15,REL,yes,67109,2.9,SINGLE PARTICLE,1,polypeptide(L),553,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=553,1,EMD-62740,False,,True,https://files.rcsb.org/download/9L1B.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62740/map/emd_62740.map.gz,,
9L1C,EMD-62741,le3.0,2026-04-15,2026-04-15,REL,yes,67109,2.69,SINGLE PARTICLE,1,polypeptide(L),553,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=553,1,EMD-62741,False,,True,https://files.rcsb.org/download/9L1C.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62741/map/emd_62741.map.gz,,
9L1D,EMD-62742,gt3.0_le4.0,2026-04-15,2026-04-15,REL,yes,67109,3.24,SINGLE PARTICLE,1,polypeptide(L),540,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=540,1,EMD-62742,False,,True,https://files.rcsb.org/download/9L1D.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62742/map/emd_62742.map.gz,,
9L1E,EMD-62743,le3.0,2026-04-15,2026-04-15,REL,yes,67109,2.82,SINGLE PARTICLE,1,polypeptide(L),553,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=553,1,EMD-62743,False,,True,https://files.rcsb.org/download/9L1E.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62743/map/emd_62743.map.gz,,
9L38,EMD-62783,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,108001,3.2,SINGLE PARTICLE,1,polypeptide(L),412,0,0.0,1,1,1,491,491,E1:len=491:copies=1:modeled=412,1,EMD-62783,False,,True,https://files.rcsb.org/download/9L38.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62783/map/emd_62783.map.gz,,
9L8N,EMD-62887,gt3.0_le4.0,2026-01-21,2026-01-21,REL,yes,67109,3.48,SINGLE PARTICLE,1,polypeptide(L),627,0,0.0,1,1,1,761,761,E1:len=761:copies=1:modeled=627,1,EMD-62887,False,,True,https://files.rcsb.org/download/9L8N.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62887/map/emd_62887.map.gz,,
9LBW,EMD-62957,gt3.0_le4.0,2026-01-07,2026-01-07,REL,yes,32001,3.44,SINGLE PARTICLE,3,polypeptide(L),519,0,0.0,4,3,2,473,890,E1:len=183:copies=2:modeled=285|E2:len=473:copies=1:modeled=125|E3:len=234:copies=1:modeled=109,1,EMD-62957,False,,True,https://files.rcsb.org/download/9LBW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62957/map/emd_62957.map.gz,,
9LFA,EMD-63043,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,67109,3.5,SINGLE PARTICLE,2,polypeptide(L),390,0,0.0,2,2,1,340,511,E1:len=340:copies=1:modeled=299|E2:len=171:copies=1:modeled=91,1,EMD-63043,False,,True,https://files.rcsb.org/download/9LFA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63043/map/emd_63043.map.gz,,
9LFC,EMD-63044,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,67109,3.6,SINGLE PARTICLE,3,polypeptide(L),408,0,0.0,3,3,1,340,521,E1:len=340:copies=1:modeled=301|E2:len=171:copies=1:modeled=97|E3:len=10:copies=1:modeled=10,1,EMD-63044,False,,True,https://files.rcsb.org/download/9LFC.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63044/map/emd_63044.map.gz,,
9LFD,EMD-63045,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,67109,3.6,SINGLE PARTICLE,2,polypeptide(L),365,0,0.0,2,2,1,365,498,E1:len=365:copies=1:modeled=298|E2:len=133:copies=1:modeled=67,1,EMD-63045,False,,True,https://files.rcsb.org/download/9LFD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63045/map/emd_63045.map.gz,,
9LKD,EMD-63175,gt3.0_le4.0,2026-01-21,2026-01-21,REL,yes,67109,3.6,SINGLE PARTICLE,1,polypeptide(L),259,0,0.0,1,1,1,317,317,E1:len=317:copies=1:modeled=259,1,EMD-63175,False,,True,https://files.rcsb.org/download/9LKD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63175/map/emd_63175.map.gz,,
9LLG,EMD-63201,le3.0,2026-03-04,2026-03-04,REL,yes,44958,2.77,SINGLE PARTICLE,2,polypeptide(L),435,0,0.0,2,2,1,516,611,E1:len=516:copies=1:modeled=365|E2:len=95:copies=1:modeled=70,1,EMD-63201,False,,True,https://files.rcsb.org/download/9LLG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63201/map/emd_63201.map.gz,,
9LM0,EMD-63211,gt3.0_le4.0,2026-03-18,2026-03-18,REL,yes,322487,3.82,SINGLE PARTICLE,1,polypeptide(L),673,0,0.0,2,1,1,877,877,E1:len=877:copies=2:modeled=673,1,EMD-63211,False,,True,https://files.rcsb.org/download/9LM0.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63211/map/emd_63211.map.gz,,
9LNM,EMD-63236,le3.0,2026-01-28,2026-01-28,REL,yes,67109,2.67,SINGLE PARTICLE,1,polypeptide(L),519,0,0.0,1,1,1,614,614,E1:len=614:copies=1:modeled=519,1,EMD-63236,False,,True,https://files.rcsb.org/download/9LNM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63236/map/emd_63236.map.gz,,
9LNN,EMD-63237,le3.0,2026-01-28,2026-01-28,REL,yes,67109,2.6,SINGLE PARTICLE,1,polypeptide(L),519,0,0.0,1,1,1,614,614,E1:len=614:copies=1:modeled=519,1,EMD-63237,False,,True,https://files.rcsb.org/download/9LNN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63237/map/emd_63237.map.gz,,
9LNO,EMD-63238,gt3.0_le4.0,2026-01-28,2026-01-28,REL,yes,67109,3.15,SINGLE PARTICLE,1,polypeptide(L),520,0,0.0,1,1,1,614,614,E1:len=614:copies=1:modeled=520,1,EMD-63238,False,,True,https://files.rcsb.org/download/9LNO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63238/map/emd_63238.map.gz,,
9LOZ,EMD-63264,gt3.0_le4.0,2026-01-28,2026-01-28,REL,yes,256001,3.55,SINGLE PARTICLE,3,polypeptide(L),411,0,0.0,3,3,1,194,418,E1:len=194:copies=1:modeled=194|E2:len=117:copies=1:modeled=117|E3:len=107:copies=1:modeled=100,1,EMD-63264,False,,True,https://files.rcsb.org/download/9LOZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63264/map/emd_63264.map.gz,,
9LP1,EMD-63266,gt3.0_le4.0,2026-01-28,2026-01-28,REL,yes,256001,3.47,SINGLE PARTICLE,3,polypeptide(L),412,0,0.0,3,3,3,194,418,E1:len=117:copies=1:modeled=117|E2:len=107:copies=1:modeled=101|E3:len=194:copies=1:modeled=194,1,EMD-63266,False,,True,https://files.rcsb.org/download/9LP1.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63266/map/emd_63266.map.gz,,
9LQT,EMD-63304,le3.0,2026-02-04,2026-02-04,REL,yes,28312,2.6,SINGLE PARTICLE,1,polypeptide(L),566,0,0.0,2,1,1,380,380,E1:len=380:copies=2:modeled=566,1,EMD-63304,False,,True,https://files.rcsb.org/download/9LQT.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63304/map/emd_63304.map.gz,,
9LQY,EMD-63309,le3.0,2026-02-04,2026-02-04,REL,yes,44958,2.4,SINGLE PARTICLE,1,polypeptide(L),584,0,0.0,2,1,1,380,380,E1:len=380:copies=2:modeled=584,1,EMD-63309,False,,True,https://files.rcsb.org/download/9LQY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63309/map/emd_63309.map.gz,,
9LQZ,EMD-63310,le3.0,2026-02-04,2026-02-04,REL,yes,44958,2.6,SINGLE PARTICLE,1,polypeptide(L),586,0,0.0,2,1,1,380,380,E1:len=380:copies=2:modeled=586,1,EMD-63310,False,,True,https://files.rcsb.org/download/9LQZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63310/map/emd_63310.map.gz,,
9LV4,EMD-63412,le3.0,2026-02-18,2026-02-18,REL,yes,226493,2.67,SINGLE PARTICLE,2,polypeptide(L),451,0,0.0,2,2,1,430,555,E1:len=430:copies=1:modeled=368|E2:len=125:copies=1:modeled=83,1,EMD-63412,False,,True,https://files.rcsb.org/download/9LV4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63412/map/emd_63412.map.gz,,
9LW1,EMD-63426,le3.0,2026-01-07,2026-01-07,REL,yes,536871,2.5,SINGLE PARTICLE,1,polypeptide(L),516,0,0.0,2,1,1,297,297,E1:len=297:copies=2:modeled=516,1,EMD-63426,False,,True,https://files.rcsb.org/download/9LW1.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63426/map/emd_63426.map.gz,,
9LW3,EMD-63427,le3.0,2026-05-27,2026-05-27,REL,yes,536871,3.0,SINGLE PARTICLE,1,polypeptide(L),516,0,0.0,2,1,1,297,297,E1:len=297:copies=2:modeled=516,1,EMD-63427,False,,True,https://files.rcsb.org/download/9LW3.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63427/map/emd_63427.map.gz,,
9LY5,EMD-63503,le3.0,2026-03-04,2026-03-04,REL,yes,340737,2.98,SINGLE PARTICLE,3,polypeptide(L),612,0,0.0,3,3,3,448,884,E1:len=223:copies=1:modeled=218|E2:len=213:copies=1:modeled=213|E3:len=448:copies=1:modeled=181,1,EMD-63503,False,,True,https://files.rcsb.org/download/9LY5.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63503/map/emd_63503.map.gz,,
9M9M,EMD-63734,le3.0,2026-03-18,2026-03-18,REL,yes,67109,2.52,SINGLE PARTICLE,1,polypeptide(L),504,0,0.0,2,1,1,424,424,E1:len=424:copies=2:modeled=504,1,EMD-63734,False,,True,https://files.rcsb.org/download/9M9M.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63734/map/emd_63734.map.gz,,
9MA0,EMD-63745,gt3.0_le4.0,2026-03-18,2026-03-18,REL,yes,67109,3.33,SINGLE PARTICLE,3,polypeptide(L),650,0,0.0,3,3,1,478,963,E1:len=478:copies=1:modeled=414|E2:len=248:copies=1:modeled=124|E3:len=237:copies=1:modeled=112,1,EMD-63745,False,,True,https://files.rcsb.org/download/9MA0.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63745/map/emd_63745.map.gz,,
9MAN,EMD-63756,gt3.0_le4.0,2026-03-18,2026-03-18,REL,yes,702465,3.78,SINGLE PARTICLE,2,polypeptide(L),377,0,0.0,4,2,2,501,1001,E1:len=500:copies=2:modeled=197|E2:len=501:copies=2:modeled=180,1,EMD-63756,False,,True,https://files.rcsb.org/download/9MAN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63756/map/emd_63756.map.gz,,
9MKG,EMD-48326,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,67109,3.45,SINGLE PARTICLE,3,polypeptide(L),553,0,0.0,3,3,3,277,718,E1:len=222:copies=1:modeled=213|E2:len=219:copies=1:modeled=217|E3:len=277:copies=1:modeled=123,1,EMD-48326,False,,True,https://files.rcsb.org/download/9MKG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-48326/map/emd_48326.map.gz,,
9MM5,EMD-48385,gt3.0_le4.0,2026-01-28,2026-01-28,REL,yes,67109,3.26,SINGLE PARTICLE,3,polypeptide(L),534,0,0.0,3,3,3,490,708,E1:len=69:copies=1:modeled=65|E2:len=149:copies=1:modeled=111|E3:len=490:copies=1:modeled=358,1,EMD-48385,False,,True,https://files.rcsb.org/download/9MM5.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-48385/map/emd_48385.map.gz,,
9MNM,EMD-48430,gt3.0_le4.0,2026-01-14,2026-01-14,REL,yes,186625,3.5,SINGLE PARTICLE,1,polypeptide(L),608,0,0.0,4,1,1,231,231,E1:len=231:copies=4:modeled=608,1,EMD-48430,False,,True,https://files.rcsb.org/download/9MNM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-48430/map/emd_48430.map.gz,,
9MUH,EMD-48632,gt3.0_le4.0,2026-01-28,2026-01-28,REL,yes,256001,3.1,SINGLE PARTICLE,1,polypeptide(L),498,0,0.0,2,1,1,265,265,E1:len=265:copies=2:modeled=498,1,EMD-48632,False,,True,https://files.rcsb.org/download/9MUH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-48632/map/emd_48632.map.gz,,
9MUN,EMD-48638,gt3.0_le4.0,2026-02-25,2026-02-25,REL,yes,226493,3.26,SINGLE PARTICLE,1,polypeptide(L),471,0,0.0,1,1,1,561,561,E1:len=561:copies=1:modeled=471,1,EMD-48638,False,,True,https://files.rcsb.org/download/9MUN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-48638/map/emd_48638.map.gz,,
9NEN,EMD-49328,gt3.0_le4.0,2026-02-04,2026-02-04,REL,yes,226493,3.5,SINGLE PARTICLE,1,polypeptide(L),501,0,0.0,1,1,1,538,538,E1:len=538:copies=1:modeled=501,1,EMD-49328,False,,True,https://files.rcsb.org/download/9NEN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-49328/map/emd_49328.map.gz,,
9NNZ,EMD-49589,gt3.0_le4.0,2026-02-18,2026-02-18,REL,yes,108001,3.94,SINGLE PARTICLE,2,polypeptide(L),374,0,0.0,2,2,1,514,862,E1:len=514:copies=1:modeled=83|E2:len=348:copies=1:modeled=291,1,EMD-49589,False,,True,https://files.rcsb.org/download/9NNZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-49589/map/emd_49589.map.gz,,
9NP0,EMD-49625,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,42593,3.32,SINGLE PARTICLE,3,polypeptide(L),535,0,0.0,3,3,1,351,788,E1:len=351:copies=1:modeled=304|E2:len=225:copies=1:modeled=124|E3:len=212:copies=1:modeled=107,1,EMD-49625,False,,True,https://files.rcsb.org/download/9NP0.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-49625/map/emd_49625.map.gz,,
9NU4,EMD-49796,gt3.0_le4.0,2026-02-11,2026-02-11,REL,yes,536871,3.6,SINGLE PARTICLE,2,polypeptide(L),631,0,0.0,2,2,1,614,680,E1:len=614:copies=1:modeled=602|E2:len=66:copies=1:modeled=29,1,EMD-49796,False,,True,https://files.rcsb.org/download/9NU4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-49796/map/emd_49796.map.gz,,
9O7G,EMD-70198,gt3.0_le4.0,2026-05-13,2026-05-13,REL,yes,186625,3.1,SINGLE PARTICLE,3,polypeptide(L),303,0,0.0,3,3,1,121,308,E1:len=121:copies=1:modeled=121|E2:len=77:copies=1:modeled=72|E3:len=110:copies=1:modeled=110,1,EMD-70198,False,,True,https://files.rcsb.org/download/9O7G.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70198/map/emd_70198.map.gz,,
9O7I,EMD-70200,gt3.0_le4.0,2026-05-13,2026-05-13,REL,yes,186625,3.4,SINGLE PARTICLE,3,polypeptide(L),562,0,0.0,4,3,3,224,510,E1:len=72:copies=2:modeled=132|E2:len=214:copies=1:modeled=214|E3:len=224:copies=1:modeled=216,1,EMD-70200,False,,True,https://files.rcsb.org/download/9O7I.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70200/map/emd_70200.map.gz,,
9OBT,EMD-70298,gt3.0_le4.0,2026-05-06,2026-05-06,REL,yes,340737,3.01,SINGLE PARTICLE,2,polypeptide(L),510,0,0.0,4,2,2,862,1709,E1:len=847:copies=2:modeled=254|E2:len=862:copies=2:modeled=256,1,EMD-70298,False,,True,https://files.rcsb.org/download/9OBT.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70298/map/emd_70298.map.gz,,
9OBV,EMD-70300,gt3.0_le4.0,2026-05-06,2026-05-06,REL,yes,256001,3.21,SINGLE PARTICLE,2,polypeptide(L),508,0,0.0,4,2,2,862,1709,E1:len=847:copies=2:modeled=248|E2:len=862:copies=2:modeled=260,1,EMD-70300,False,,True,https://files.rcsb.org/download/9OBV.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70300/map/emd_70300.map.gz,,
9OBW,EMD-70301,gt3.0_le4.0,2026-05-06,2026-05-06,REL,yes,256001,3.15,SINGLE PARTICLE,2,polypeptide(L),500,0,0.0,4,2,2,862,1709,E1:len=847:copies=2:modeled=242|E2:len=862:copies=2:modeled=258,1,EMD-70301,False,,True,https://files.rcsb.org/download/9OBW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70301/map/emd_70301.map.gz,,
9OBY,EMD-70303,gt3.0_le4.0,2026-05-06,2026-05-06,REL,yes,340737,3.6,SINGLE PARTICLE,2,polypeptide(L),512,0,0.0,4,2,2,862,1709,E1:len=847:copies=2:modeled=252|E2:len=862:copies=2:modeled=260,1,EMD-70303,False,,True,https://files.rcsb.org/download/9OBY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70303/map/emd_70303.map.gz,,
9OBZ,EMD-70304,le3.0,2026-05-06,2026-05-06,REL,yes,340737,2.81,SINGLE PARTICLE,2,polypeptide(L),520,0,0.0,4,2,2,862,1709,E1:len=847:copies=2:modeled=252|E2:len=862:copies=2:modeled=268,1,EMD-70304,False,,True,https://files.rcsb.org/download/9OBZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70304/map/emd_70304.map.gz,,
9OC0,EMD-70305,le3.0,2026-05-06,2026-05-06,REL,yes,340737,2.97,SINGLE PARTICLE,2,polypeptide(L),512,0,0.0,4,2,2,862,1709,E1:len=847:copies=2:modeled=248|E2:len=862:copies=2:modeled=264,1,EMD-70305,False,,True,https://files.rcsb.org/download/9OC0.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70305/map/emd_70305.map.gz,,
9OC1,EMD-70306,le3.0,2026-05-06,2026-05-06,REL,yes,340737,2.76,SINGLE PARTICLE,2,polypeptide(L),512,0,0.0,4,2,2,862,1709,E1:len=847:copies=2:modeled=250|E2:len=862:copies=2:modeled=262,1,EMD-70306,False,,True,https://files.rcsb.org/download/9OC1.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70306/map/emd_70306.map.gz,,
9OC2,EMD-70307,le3.0,2026-05-06,2026-05-06,REL,yes,340737,2.69,SINGLE PARTICLE,2,polypeptide(L),510,0,0.0,4,2,2,862,1709,E1:len=847:copies=2:modeled=252|E2:len=862:copies=2:modeled=258,1,EMD-70307,False,,True,https://files.rcsb.org/download/9OC2.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70307/map/emd_70307.map.gz,,
9ODH,EMD-70359,gt3.0_le4.0,2026-03-04,2026-03-04,REL,yes,87809,3.8,SINGLE PARTICLE,3,polypeptide(L),694,0,0.0,3,3,3,354,769,E1:len=344:copies=1:modeled=332|E2:len=71:copies=1:modeled=53|E3:len=354:copies=1:modeled=309,1,EMD-70359,False,,True,https://files.rcsb.org/download/9ODH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70359/map/emd_70359.map.gz,,
9ODI,EMD-70360,gt3.0_le4.0,2026-03-04,2026-03-04,REL,yes,87809,3.7,SINGLE PARTICLE,3,polypeptide(L),698,0,0.0,3,3,3,354,769,E1:len=344:copies=1:modeled=333|E2:len=71:copies=1:modeled=53|E3:len=354:copies=1:modeled=312,1,EMD-70360,False,,True,https://files.rcsb.org/download/9ODI.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70360/map/emd_70360.map.gz,,
9OK7,EMD-70555,gt3.0_le4.0,2026-08-12,2026-08-12,REL,yes,108001,3.79,SINGLE PARTICLE,1,polypeptide(L),484,0,0.0,1,1,1,892,892,E1:len=892:copies=1:modeled=484,1,EMD-70555,False,,True,https://files.rcsb.org/download/9OK7.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70555/map/emd_70555.map.gz,,
9OK8,EMD-70556,gt3.0_le4.0,2026-08-12,2026-08-12,REL,yes,108001,3.69,SINGLE PARTICLE,1,polypeptide(L),583,0,0.0,1,1,1,892,892,E1:len=892:copies=1:modeled=583,1,EMD-70556,False,,True,https://files.rcsb.org/download/9OK8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70556/map/emd_70556.map.gz,,
9OPQ,EMD-70721,gt3.0_le4.0,2026-05-13,2026-05-13,REL,yes,340737,3.3,SINGLE PARTICLE,2,polypeptide(L),634,0,0.0,2,2,2,506,882,E1:len=376:copies=1:modeled=292|E2:len=506:copies=1:modeled=342,1,EMD-70721,False,,True,https://files.rcsb.org/download/9OPQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70721/map/emd_70721.map.gz,,
9OTA,EMD-70828,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,70305,3.2,SINGLE PARTICLE,1,polypeptide(L),534,0,0.0,2,1,1,270,270,E1:len=270:copies=2:modeled=534,1,EMD-70828,False,,True,https://files.rcsb.org/download/9OTA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70828/map/emd_70828.map.gz,,
9OVK,EMD-70900,le3.0,2026-05-27,2026-05-27,REL,yes,629857,1.7,SINGLE PARTICLE,1,polypeptide(L),279,0,0.0,1,1,1,1344,1344,E1:len=1344:copies=1:modeled=279,1,EMD-70900,False,,True,https://files.rcsb.org/download/9OVK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70900/map/emd_70900.map.gz,,
9OVM,EMD-70902,le3.0,2026-05-27,2026-05-27,REL,yes,256001,2.05,SINGLE PARTICLE,1,polypeptide(L),284,0,0.0,1,1,1,1326,1326,E1:len=1326:copies=1:modeled=284,1,EMD-70902,False,,True,https://files.rcsb.org/download/9OVM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70902/map/emd_70902.map.gz,,
9OVO,EMD-70904,le3.0,2026-05-27,2026-05-27,REL,yes,256001,1.98,SINGLE PARTICLE,1,polypeptide(L),285,0,0.0,1,1,1,1326,1326,E1:len=1326:copies=1:modeled=285,1,EMD-70904,False,,True,https://files.rcsb.org/download/9OVO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70904/map/emd_70904.map.gz,,
9P1V,EMD-71135,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,87809,3.27,SINGLE PARTICLE,3,polypeptide(L),532,0,0.0,3,3,1,838,1547,E1:len=838:copies=1:modeled=93|E2:len=235:copies=1:modeled=214|E3:len=474:copies=1:modeled=225,1,EMD-71135,False,,True,https://files.rcsb.org/download/9P1V.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71135/map/emd_71135.map.gz,,
9P4L,EMD-71273,le3.0,2026-04-15,2026-04-15,REL,yes,536871,2.22,SINGLE PARTICLE,1,polypeptide(L),522,0,0.0,1,1,1,743,743,E1:len=743:copies=1:modeled=522,1,EMD-71273,False,,True,https://files.rcsb.org/download/9P4L.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71273/map/emd_71273.map.gz,,
9P4M,EMD-71274,gt3.0_le4.0,2026-04-15,2026-04-15,REL,yes,536871,3.05,SINGLE PARTICLE,1,polypeptide(L),518,0,0.0,1,1,1,736,736,E1:len=736:copies=1:modeled=518,1,EMD-71274,False,,True,https://files.rcsb.org/download/9P4M.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71274/map/emd_71274.map.gz,,
9P4N,EMD-71275,le3.0,2026-04-15,2026-04-15,REL,yes,536871,2.18,SINGLE PARTICLE,2,polypeptide(L),617,0,0.0,2,2,1,743,867,E1:len=743:copies=1:modeled=522|E2:len=124:copies=1:modeled=95,1,EMD-71275,False,,True,https://files.rcsb.org/download/9P4N.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71275/map/emd_71275.map.gz,,
9P4O,EMD-71276,le3.0,2026-04-15,2026-04-15,REL,yes,813190,2.05,SINGLE PARTICLE,1,polypeptide(L),522,0,0.0,1,1,1,743,743,E1:len=743:copies=1:modeled=522,1,EMD-71276,False,,True,https://files.rcsb.org/download/9P4O.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71276/map/emd_71276.map.gz,,
9P4P,EMD-71277,gt3.0_le4.0,2026-04-15,2026-04-15,REL,yes,864001,3.09,SINGLE PARTICLE,1,polypeptide(L),525,0,0.0,1,1,1,743,743,E1:len=743:copies=1:modeled=525,1,EMD-71277,False,,True,https://files.rcsb.org/download/9P4P.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71277/map/emd_71277.map.gz,,
9P4Q,EMD-71278,le3.0,2026-04-15,2026-04-15,REL,yes,340737,2.2,SINGLE PARTICLE,2,polypeptide(L),617,0,0.0,2,2,1,743,867,E1:len=743:copies=1:modeled=525|E2:len=124:copies=1:modeled=92,1,EMD-71278,False,,True,https://files.rcsb.org/download/9P4Q.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71278/map/emd_71278.map.gz,,
9P4R,EMD-71279,le3.0,2026-04-15,2026-04-15,REL,yes,629857,2.51,SINGLE PARTICLE,1,polypeptide(L),525,0,0.0,1,1,1,743,743,E1:len=743:copies=1:modeled=525,1,EMD-71279,False,,True,https://files.rcsb.org/download/9P4R.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71279/map/emd_71279.map.gz,,
9PGD,EMD-71626,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,226493,3.2,SINGLE PARTICLE,3,polypeptide(L),519,0,0.0,3,3,1,396,685,E1:len=396:copies=1:modeled=396|E2:len=154:copies=1:modeled=105|E3:len=135:copies=1:modeled=18,1,EMD-71626,False,,True,https://files.rcsb.org/download/9PGD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71626/map/emd_71626.map.gz,,
9PPQ,EMD-71775,gt3.0_le4.0,2026-03-04,2026-03-04,REL,yes,186625,3.34,SINGLE PARTICLE,1,polypeptide(L),286,0,0.0,1,1,1,334,334,E1:len=334:copies=1:modeled=286,1,EMD-71775,False,,True,https://files.rcsb.org/download/9PPQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71775/map/emd_71775.map.gz,,
9PRZ,EMD-71817,le3.0,2026-02-25,2026-02-25,REL,yes,116873,2.95,SINGLE PARTICLE,1,polypeptide(L),206,0,0.0,1,1,1,231,231,E1:len=231:copies=1:modeled=206,1,EMD-71817,False,,True,https://files.rcsb.org/download/9PRZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71817/map/emd_71817.map.gz,,
9PV0,EMD-71880,le3.0,2026-08-05,2026-08-05,REL,yes,131073,2.52,SINGLE PARTICLE,1,polypeptide(L),361,0,0.0,1,1,1,495,495,E1:len=495:copies=1:modeled=361,1,EMD-71880,False,,True,https://files.rcsb.org/download/9PV0.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71880/map/emd_71880.map.gz,,
9PVM,EMD-71893,gt3.0_le4.0,2026-08-26,2026-08-26,REL,yes,67109,3.27,SINGLE PARTICLE,1,polypeptide(L),441,0,0.0,1,1,1,580,580,E1:len=580:copies=1:modeled=441,1,EMD-71893,False,,True,https://files.rcsb.org/download/9PVM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71893/map/emd_71893.map.gz,,
9PX8,EMD-71961,gt3.0_le4.0,2026-08-12,2026-08-12,REL,yes,108001,3.3,SINGLE PARTICLE,1,polypeptide(L),465,0,0.0,15,1,1,31,31,E1:len=31:copies=15:modeled=465,1,EMD-71961,False,,True,https://files.rcsb.org/download/9PX8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71961/map/emd_71961.map.gz,,
9PXJ,EMD-71971,gt3.0_le4.0,2026-06-10,2026-06-10,REL,yes,131073,3.18,SINGLE PARTICLE,2,polypeptide(L),696,0,0.0,2,2,1,795,1051,E1:len=795:copies=1:modeled=441|E2:len=256:copies=1:modeled=255,1,EMD-71971,False,,True,https://files.rcsb.org/download/9PXJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-71971/map/emd_71971.map.gz,,
9PZD,EMD-72067,gt3.0_le4.0,2026-08-12,2026-08-12,REL,yes,108001,3.3,SINGLE PARTICLE,1,polypeptide(L),310,0,0.0,10,1,1,31,31,E1:len=31:copies=10:modeled=310,1,EMD-72067,False,,True,https://files.rcsb.org/download/9PZD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72067/map/emd_72067.map.gz,,
9Q33,EMD-72178,gt3.0_le4.0,2026-01-14,2026-01-14,REL,yes,67109,3.2,SINGLE PARTICLE,2,polypeptide(L),536,0,0.0,2,2,2,442,630,E1:len=188:copies=1:modeled=160|E2:len=442:copies=1:modeled=376,1,EMD-72178,False,,True,https://files.rcsb.org/download/9Q33.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72178/map/emd_72178.map.gz,,
9Q37,EMD-72185,gt3.0_le4.0,2026-08-19,2026-08-19,REL,yes,778,3.45,SINGLE PARTICLE,1,polypeptide(L),405,0,0.0,15,1,1,31,31,E1:len=31:copies=15:modeled=405,1,EMD-72185,False,,True,https://files.rcsb.org/download/9Q37.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72185/map/emd_72185.map.gz,,
9Q3L,EMD-72202,le3.0,2026-08-19,2026-08-19,REL,yes,186625,2.5,SINGLE PARTICLE,1,polypeptide(L),574,0,0.0,2,1,1,460,460,E1:len=460:copies=2:modeled=574,1,EMD-72202,False,,True,https://files.rcsb.org/download/9Q3L.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72202/map/emd_72202.map.gz,,
9Q7J,EMD-72301,gt3.0_le4.0,2026-09-09,2026-09-09,REL,yes,108001,3.75,SINGLE PARTICLE,1,polypeptide(L),279,0,0.0,9,1,1,31,31,E1:len=31:copies=9:modeled=279,1,EMD-72301,False,,True,https://files.rcsb.org/download/9Q7J.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72301/map/emd_72301.map.gz,,
9QJJ,EMD-53204,le3.0,2026-03-18,2026-03-18,REL,yes,28312,2.5,SINGLE PARTICLE,2,polypeptide(L),546,0,0.0,2,2,1,435,736,E1:len=435:copies=1:modeled=258|E2:len=301:copies=1:modeled=288,1,EMD-53204,False,,True,https://files.rcsb.org/download/9QJJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53204/map/emd_53204.map.gz,,
9QJN,EMD-53205,le3.0,2026-03-18,2026-03-18,REL,yes,28312,2.4,SINGLE PARTICLE,3,polypeptide(L),631,0,0.0,3,3,3,349,766,E1:len=93:copies=1:modeled=65|E2:len=324:copies=1:modeled=280|E3:len=349:copies=1:modeled=286,1,EMD-53205,False,,True,https://files.rcsb.org/download/9QJN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53205/map/emd_53205.map.gz,,
9QL1,EMD-53221,le3.0,2026-03-18,2026-03-18,REL,yes,28312,2.4,SINGLE PARTICLE,3,polypeptide(L),681,0,0.0,3,3,3,442,1075,E1:len=311:copies=1:modeled=118|E2:len=322:copies=1:modeled=236|E3:len=442:copies=1:modeled=327,1,EMD-53221,False,,True,https://files.rcsb.org/download/9QL1.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53221/map/emd_53221.map.gz,,
9QO8,EMD-53259,le3.0,2026-02-18,2026-02-18,REL,yes,131073,2.9,SINGLE PARTICLE,1,polypeptide(L),535,0,0.0,1,1,1,632,632,E1:len=632:copies=1:modeled=535,1,EMD-53259,False,,True,https://files.rcsb.org/download/9QO8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53259/map/emd_53259.map.gz,,
9QO9,EMD-53260,gt3.0_le4.0,2026-03-04,2026-03-04,REL,yes,186625,3.2,SINGLE PARTICLE,1,polypeptide(L),527,0,0.0,1,1,1,632,632,E1:len=632:copies=1:modeled=527,1,EMD-53260,False,,True,https://files.rcsb.org/download/9QO9.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53260/map/emd_53260.map.gz,,
9QP4,EMD-53268,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,442369,3.06,SINGLE PARTICLE,3,polypeptide(L),620,0,0.0,3,3,1,488,925,E1:len=488:copies=1:modeled=396|E2:len=215:copies=1:modeled=104|E3:len=222:copies=1:modeled=120,1,EMD-53268,False,,True,https://files.rcsb.org/download/9QP4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53268/map/emd_53268.map.gz,,
9QQS,EMD-53313,le3.0,2026-05-27,2026-05-27,REL,yes,442369,2.65,SINGLE PARTICLE,3,polypeptide(L),491,0,0.0,3,3,2,477,817,E1:len=211:copies=1:modeled=163|E2:len=477:copies=1:modeled=209|E3:len=129:copies=1:modeled=119,1,EMD-53313,False,,True,https://files.rcsb.org/download/9QQS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53313/map/emd_53313.map.gz,,
9QTY,EMD-53367,le3.0,2026-05-27,2026-05-27,REL,yes,186625,2.98,SINGLE PARTICLE,3,polypeptide(L),486,0,0.0,3,3,3,344,684,E1:len=211:copies=1:modeled=163|E2:len=129:copies=1:modeled=119|E3:len=344:copies=1:modeled=204,1,EMD-53367,False,,True,https://files.rcsb.org/download/9QTY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53367/map/emd_53367.map.gz,,
9QZO,EMD-53472,le3.0,2026-02-11,2026-02-11,REL,yes,240946,2.9,SINGLE PARTICLE,1,polypeptide(L),550,0,0.0,1,1,1,663,663,E1:len=663:copies=1:modeled=550,1,EMD-53472,False,,True,https://files.rcsb.org/download/9QZO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53472/map/emd_53472.map.gz,,
9R07,EMD-53481,gt3.0_le4.0,2026-02-11,2026-02-11,REL,yes,108001,3.2,SINGLE PARTICLE,1,polypeptide(L),565,0,0.0,1,1,1,663,663,E1:len=663:copies=1:modeled=565,1,EMD-53481,False,,True,https://files.rcsb.org/download/9R07.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53481/map/emd_53481.map.gz,,
9R0A,EMD-53482,gt3.0_le4.0,2026-02-11,2026-02-11,REL,yes,108001,3.1,SINGLE PARTICLE,1,polypeptide(L),561,0,0.0,1,1,1,663,663,E1:len=663:copies=1:modeled=561,1,EMD-53482,False,,True,https://files.rcsb.org/download/9R0A.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53482/map/emd_53482.map.gz,,
9R0I,EMD-53483,gt3.0_le4.0,2026-02-11,2026-02-11,REL,yes,108001,3.1,SINGLE PARTICLE,1,polypeptide(L),550,0,0.0,1,1,1,663,663,E1:len=663:copies=1:modeled=550,1,EMD-53483,False,,True,https://files.rcsb.org/download/9R0I.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53483/map/emd_53483.map.gz,,
9R0M,EMD-53484,gt3.0_le4.0,2026-02-11,2026-02-11,REL,yes,240946,3.2,SINGLE PARTICLE,1,polypeptide(L),550,0,0.0,1,1,1,663,663,E1:len=663:copies=1:modeled=550,1,EMD-53484,False,,True,https://files.rcsb.org/download/9R0M.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53484/map/emd_53484.map.gz,,
9R0N,EMD-53485,gt3.0_le4.0,2026-02-11,2026-02-11,REL,yes,240946,3.4,SINGLE PARTICLE,1,polypeptide(L),549,0,0.0,1,1,1,663,663,E1:len=663:copies=1:modeled=549,1,EMD-53485,False,,True,https://files.rcsb.org/download/9R0N.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53485/map/emd_53485.map.gz,,
9R0O,EMD-53486,gt3.0_le4.0,2026-02-11,2026-02-11,REL,yes,240946,3.1,SINGLE PARTICLE,1,polypeptide(L),558,0,0.0,1,1,1,663,663,E1:len=663:copies=1:modeled=558,1,EMD-53486,False,,True,https://files.rcsb.org/download/9R0O.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53486/map/emd_53486.map.gz,,
9R1H,EMD-53509,gt3.0_le4.0,2026-02-25,2026-02-25,REL,yes,108001,3.02,SINGLE PARTICLE,1,polypeptide(L),547,0,0.0,1,1,1,613,613,E1:len=613:copies=1:modeled=547,1,EMD-53509,False,,True,https://files.rcsb.org/download/9R1H.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53509/map/emd_53509.map.gz,,
9R8P,EMD-53835,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,11944,4.0,SINGLE PARTICLE,1,polypeptide(L),582,0,0.0,1,1,1,590,590,E1:len=590:copies=1:modeled=582,1,EMD-53835,False,,True,https://files.rcsb.org/download/9R8P.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53835/map/emd_53835.map.gz,,
9RH2,EMD-53955,le3.0,2026-06-17,2026-06-17,REL,yes,61012,2.7,SINGLE PARTICLE,3,polypeptide(L),612,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=388|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53955,False,,True,https://files.rcsb.org/download/9RH2.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53955/map/emd_53955.map.gz,,
9RH3,EMD-53956,le3.0,2026-06-17,2026-06-17,REL,yes,61012,2.7,SINGLE PARTICLE,3,polypeptide(L),612,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=388|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53956,False,,True,https://files.rcsb.org/download/9RH3.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53956/map/emd_53956.map.gz,,
9RH4,EMD-53957,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,61012,3.1,SINGLE PARTICLE,3,polypeptide(L),602,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=378|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53957,False,,True,https://files.rcsb.org/download/9RH4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53957/map/emd_53957.map.gz,,
9RH5,EMD-53958,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,61012,3.3,SINGLE PARTICLE,3,polypeptide(L),610,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=386|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53958,False,,True,https://files.rcsb.org/download/9RH5.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53958/map/emd_53958.map.gz,,
9RH6,EMD-53959,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,61012,3.4,SINGLE PARTICLE,3,polypeptide(L),611,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=387|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53959,False,,True,https://files.rcsb.org/download/9RH6.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53959/map/emd_53959.map.gz,,
9RH7,EMD-53960,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,61012,3.7,SINGLE PARTICLE,3,polypeptide(L),602,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=378|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53960,False,,True,https://files.rcsb.org/download/9RH7.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53960/map/emd_53960.map.gz,,
9RH8,EMD-53961,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,61012,3.1,SINGLE PARTICLE,3,polypeptide(L),610,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=386|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53961,False,,True,https://files.rcsb.org/download/9RH8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53961/map/emd_53961.map.gz,,
9RH9,EMD-53962,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,61012,3.1,SINGLE PARTICLE,3,polypeptide(L),611,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=387|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53962,False,,True,https://files.rcsb.org/download/9RH9.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53962/map/emd_53962.map.gz,,
9RHA,EMD-53963,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,61012,3.5,SINGLE PARTICLE,3,polypeptide(L),602,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=378|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53963,False,,True,https://files.rcsb.org/download/9RHA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53963/map/emd_53963.map.gz,,
9RHB,EMD-53964,le3.0,2026-06-17,2026-06-17,REL,yes,61012,2.9,SINGLE PARTICLE,3,polypeptide(L),612,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=388|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53964,False,,True,https://files.rcsb.org/download/9RHB.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53964/map/emd_53964.map.gz,,
9RHC,EMD-53965,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,61012,3.2,SINGLE PARTICLE,3,polypeptide(L),612,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=388|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53965,False,,True,https://files.rcsb.org/download/9RHC.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53965/map/emd_53965.map.gz,,
9RHD,EMD-53966,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,61012,3.2,SINGLE PARTICLE,3,polypeptide(L),606,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=382|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53966,False,,True,https://files.rcsb.org/download/9RHD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53966/map/emd_53966.map.gz,,
9RHE,EMD-53967,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,61012,3.5,SINGLE PARTICLE,3,polypeptide(L),602,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=378|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53967,False,,True,https://files.rcsb.org/download/9RHE.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53967/map/emd_53967.map.gz,,
9RHF,EMD-53968,le3.0,2026-06-17,2026-06-17,REL,yes,61012,3.0,SINGLE PARTICLE,3,polypeptide(L),612,0,0.0,3,3,1,407,653,E1:len=407:copies=1:modeled=388|E2:len=127:copies=1:modeled=118|E3:len=119:copies=1:modeled=106,1,EMD-53968,False,,True,https://files.rcsb.org/download/9RHF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-53968/map/emd_53968.map.gz,,
9RQL,EMD-54178,le3.0,2026-07-08,2026-07-08,REL,yes,131073,2.65,SINGLE PARTICLE,1,polypeptide(L),676,0,0.0,1,1,1,729,729,E1:len=729:copies=1:modeled=676,1,EMD-54178,False,,True,https://files.rcsb.org/download/9RQL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54178/map/emd_54178.map.gz,,
9RQN,EMD-54180,le3.0,2026-07-08,2026-07-08,REL,yes,131073,2.94,SINGLE PARTICLE,1,polypeptide(L),676,0,0.0,1,1,1,729,729,E1:len=729:copies=1:modeled=676,1,EMD-54180,False,,True,https://files.rcsb.org/download/9RQN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54180/map/emd_54180.map.gz,,
9RRS,EMD-54195,le3.0,2026-02-04,2026-02-04,REL,yes,629857,1.88,SINGLE PARTICLE,2,polypeptide(L),548,0,0.0,2,2,2,738,1135,E1:len=397:copies=1:modeled=35|E2:len=738:copies=1:modeled=513,1,EMD-54195,False,,True,https://files.rcsb.org/download/9RRS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54195/map/emd_54195.map.gz,,
9RWS,EMD-54346,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,131073,3.28,SINGLE PARTICLE,1,polypeptide(L),512,0,0.0,1,1,1,551,551,E1:len=551:copies=1:modeled=512,1,EMD-54346,False,,True,https://files.rcsb.org/download/9RWS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54346/map/emd_54346.map.gz,,
9RY4,EMD-54372,le3.0,2026-07-22,2026-07-22,REL,yes,536871,2.0,SINGLE PARTICLE,1,polypeptide(L),263,0,0.0,1,1,1,280,280,E1:len=280:copies=1:modeled=263,1,EMD-54372,False,,True,https://files.rcsb.org/download/9RY4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54372/map/emd_54372.map.gz,,
9RZM,EMD-54404,le3.0,2026-07-29,2026-07-29,REL,yes,764412,2.41,SINGLE PARTICLE,1,polypeptide(L),250,0,0.0,1,1,1,280,280,E1:len=280:copies=1:modeled=250,1,EMD-54404,False,,True,https://files.rcsb.org/download/9RZM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54404/map/emd_54404.map.gz,,
9S15,EMD-54442,le3.0,2026-10-07,2026-10-07,REL,yes,174457,2.8,SINGLE PARTICLE,1,polypeptide(L),184,0,0.0,1,1,1,184,184,E1:len=184:copies=1:modeled=184,1,EMD-54442,False,,True,https://files.rcsb.org/download/9S15.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54442/map/emd_54442.map.gz,,
9S5J,EMD-54599,gt3.0_le4.0,2026-07-15,2026-07-15,REL,yes,287966,3.5,SINGLE PARTICLE,2,polypeptide(L),642,0,0.0,2,2,1,510,694,E1:len=510:copies=1:modeled=458|E2:len=184:copies=1:modeled=184,1,EMD-54599,False,,True,https://files.rcsb.org/download/9S5J.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54599/map/emd_54599.map.gz,,
9S6R,EMD-54629,gt3.0_le4.0,2026-03-18,2026-03-18,REL,yes,322487,3.08,SINGLE PARTICLE,3,polypeptide(L),618,0,0.0,3,3,3,467,922,E1:len=240:copies=1:modeled=128|E2:len=215:copies=1:modeled=105|E3:len=467:copies=1:modeled=385,1,EMD-54629,False,,True,https://files.rcsb.org/download/9S6R.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54629/map/emd_54629.map.gz,,
9SAS,EMD-54696,gt3.0_le4.0,2026-08-26,2026-08-26,REL,yes,536871,3.4,SINGLE PARTICLE,2,polypeptide(L),670,0,0.0,4,2,2,295,335,E1:len=40:copies=2:modeled=80|E2:len=295:copies=2:modeled=590,1,EMD-54696,False,,True,https://files.rcsb.org/download/9SAS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54696/map/emd_54696.map.gz,,
9SAU,EMD-54697,gt3.0_le4.0,2026-08-19,2026-08-19,REL,yes,536871,3.6,SINGLE PARTICLE,2,polypeptide(L),670,0,0.0,4,2,2,294,335,E1:len=41:copies=2:modeled=82|E2:len=294:copies=2:modeled=588,1,EMD-54697,False,,True,https://files.rcsb.org/download/9SAU.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54697/map/emd_54697.map.gz,,
9SF5,EMD-54814,gt3.0_le4.0,2026-09-02,2026-09-02,REL,yes,702465,3.7,SINGLE PARTICLE,3,polypeptide(L),430,0,0.0,3,3,1,1284,1514,E1:len=1284:copies=1:modeled=200|E2:len=124:copies=1:modeled=124|E3:len=106:copies=1:modeled=106,1,EMD-54814,False,,True,https://files.rcsb.org/download/9SF5.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54814/map/emd_54814.map.gz,,
9SF8,EMD-54816,le3.0,2026-09-02,2026-09-02,REL,yes,32001,2.6,SINGLE PARTICLE,1,polypeptide(L),562,0,0.0,1,1,1,562,562,E1:len=562:copies=1:modeled=562,1,EMD-54816,False,,True,https://files.rcsb.org/download/9SF8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54816/map/emd_54816.map.gz,,
9SP8,EMD-55065,gt3.0_le4.0,2026-09-30,2026-09-30,REL,yes,95552,3.32,SINGLE PARTICLE,2,polypeptide(L),632,0,0.0,2,2,2,600,714,E1:len=114:copies=1:modeled=114|E2:len=600:copies=1:modeled=518,1,EMD-55065,False,,True,https://files.rcsb.org/download/9SP8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55065/map/emd_55065.map.gz,,
9SR9,EMD-55134,le3.0,2026-08-12,2026-08-12,REL,yes,186625,2.5,SINGLE PARTICLE,1,polypeptide(L),386,0,0.0,1,1,1,392,392,E1:len=392:copies=1:modeled=386,1,EMD-55134,False,,True,https://files.rcsb.org/download/9SR9.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55134/map/emd_55134.map.gz,,
9SWM,EMD-55320,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,23329,3.38,SINGLE PARTICLE,1,polypeptide(L),642,0,0.0,2,1,1,379,379,E1:len=379:copies=2:modeled=642,1,EMD-55320,False,,True,https://files.rcsb.org/download/9SWM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55320/map/emd_55320.map.gz,,
9SWQ,EMD-55324,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,16385,3.31,SINGLE PARTICLE,1,polypeptide(L),642,0,0.0,2,1,1,379,379,E1:len=379:copies=2:modeled=642,1,EMD-55324,False,,True,https://files.rcsb.org/download/9SWQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55324/map/emd_55324.map.gz,,
9SZB,EMD-55364,le3.0,2026-08-26,2026-08-26,REL,yes,359662,2.78,SINGLE PARTICLE,2,polypeptide(L),354,0,0.0,2,2,1,314,410,E1:len=314:copies=1:modeled=268|E2:len=96:copies=1:modeled=86,1,EMD-55364,False,,True,https://files.rcsb.org/download/9SZB.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55364/map/emd_55364.map.gz,,
9T3R,EMD-55517,gt3.0_le4.0,2026-03-11,2026-03-11,REL,yes,226493,3.5,SINGLE PARTICLE,3,polypeptide(L),663,0,0.0,3,3,3,636,1119,E1:len=216:copies=1:modeled=108|E2:len=267:copies=1:modeled=118|E3:len=636:copies=1:modeled=437,1,EMD-55517,False,,True,https://files.rcsb.org/download/9T3R.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55517/map/emd_55517.map.gz,,
9T3S,EMD-55518,le3.0,2026-03-11,2026-03-11,REL,yes,186625,3.0,SINGLE PARTICLE,3,polypeptide(L),663,0,0.0,3,3,3,636,1119,E1:len=267:copies=1:modeled=118|E2:len=216:copies=1:modeled=108|E3:len=636:copies=1:modeled=437,1,EMD-55518,False,,True,https://files.rcsb.org/download/9T3S.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55518/map/emd_55518.map.gz,,
9T5J,EMD-55585,le3.0,2026-07-15,2026-07-15,REL,yes,131073,2.8,SINGLE PARTICLE,3,polypeptide(L),684,0,0.0,3,3,1,614,1274,E1:len=614:copies=1:modeled=190|E2:len=376:copies=1:modeled=254|E3:len=284:copies=1:modeled=240,1,EMD-55585,False,,True,https://files.rcsb.org/download/9T5J.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55585/map/emd_55585.map.gz,,
9T5K,EMD-55586,le3.0,2026-04-22,2026-04-22,REL,yes,256001,2.6,SINGLE PARTICLE,1,polypeptide(L),381,0,0.0,1,1,1,453,453,E1:len=453:copies=1:modeled=381,1,EMD-55586,False,,True,https://files.rcsb.org/download/9T5K.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55586/map/emd_55586.map.gz,,
9T9M,EMD-55722,gt3.0_le4.0,2026-01-28,2026-01-28,REL,yes,442369,3.09,SINGLE PARTICLE,3,polypeptide(L),544,0,0.0,3,3,3,220,561,E1:len=218:copies=1:modeled=217|E2:len=123:copies=1:modeled=116|E3:len=220:copies=1:modeled=211,1,EMD-55722,False,,True,https://files.rcsb.org/download/9T9M.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55722/map/emd_55722.map.gz,,
9THK,EMD-55929,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,186625,3.6,SINGLE PARTICLE,1,polypeptide(L),494,0,0.0,2,1,1,293,293,E1:len=293:copies=2:modeled=494,1,EMD-55929,False,,True,https://files.rcsb.org/download/9THK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55929/map/emd_55929.map.gz,,
9TIU,EMD-55970,gt3.0_le4.0,2026-09-09,2026-09-09,REL,yes,186625,3.68,SINGLE PARTICLE,1,polypeptide(L),575,0,0.0,5,1,1,136,136,E1:len=136:copies=5:modeled=575,1,EMD-55970,False,,True,https://files.rcsb.org/download/9TIU.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55970/map/emd_55970.map.gz,,
9TIV,EMD-55971,gt3.0_le4.0,2026-09-09,2026-09-09,REL,yes,186625,3.46,SINGLE PARTICLE,1,polypeptide(L),630,0,0.0,5,1,1,136,136,E1:len=136:copies=5:modeled=630,1,EMD-55971,False,,True,https://files.rcsb.org/download/9TIV.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55971/map/emd_55971.map.gz,,
9TK9,EMD-56021,le3.0,2026-07-29,2026-07-29,REL,yes,186625,2.73,SINGLE PARTICLE,1,polypeptide(L),247,0,0.0,1,1,1,293,293,E1:len=293:copies=1:modeled=247,1,EMD-56021,False,,True,https://files.rcsb.org/download/9TK9.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56021/map/emd_56021.map.gz,,
9TKA,EMD-56022,le3.0,2026-07-29,2026-07-29,REL,yes,186625,2.7,SINGLE PARTICLE,1,polypeptide(L),247,0,0.0,1,1,1,293,293,E1:len=293:copies=1:modeled=247,1,EMD-56022,False,,True,https://files.rcsb.org/download/9TKA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56022/map/emd_56022.map.gz,,
9TKM,EMD-56033,le3.0,2026-04-08,2026-04-08,REL,yes,364501,2.7,SINGLE PARTICLE,3,polypeptide(L),675,0,0.0,3,3,1,278,779,E1:len=278:copies=1:modeled=214|E2:len=263:copies=1:modeled=237|E3:len=238:copies=1:modeled=224,1,EMD-56033,False,,True,https://files.rcsb.org/download/9TKM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56033/map/emd_56033.map.gz,,
9TUZ,EMD-56281,gt3.0_le4.0,2026-07-08,2026-07-08,REL,yes,108001,3.4,SINGLE PARTICLE,3,polypeptide(L),482,0,0.0,3,3,1,1109,1598,E1:len=1109:copies=1:modeled=48|E2:len=256:copies=1:modeled=220|E3:len=233:copies=1:modeled=214,1,EMD-56281,False,,True,https://files.rcsb.org/download/9TUZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56281/map/emd_56281.map.gz,,
9TZW,EMD-56477,gt3.0_le4.0,2026-06-03,2026-06-03,REL,yes,256001,3.3,SINGLE PARTICLE,1,polypeptide(L),552,0,0.0,4,1,1,142,142,E1:len=142:copies=4:modeled=552,1,EMD-56477,False,,True,https://files.rcsb.org/download/9TZW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56477/map/emd_56477.map.gz,,
9TZY,EMD-56479,le3.0,2026-06-03,2026-06-03,REL,yes,23329,2.94,SINGLE PARTICLE,1,polypeptide(L),552,0,0.0,4,1,1,142,142,E1:len=142:copies=4:modeled=552,1,EMD-56479,False,,True,https://files.rcsb.org/download/9TZY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56479/map/emd_56479.map.gz,,
9U4J,EMD-63848,gt3.0_le4.0,2026-03-25,2026-03-25,REL,yes,256001,3.4,SINGLE PARTICLE,1,polypeptide(L),273,0,0.0,1,1,1,288,288,E1:len=288:copies=1:modeled=273,1,EMD-63848,False,,True,https://files.rcsb.org/download/9U4J.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63848/map/emd_63848.map.gz,,
9U7H,EMD-63936,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,131073,3.08,SINGLE PARTICLE,2,polypeptide(L),538,0,0.0,2,2,1,613,622,E1:len=613:copies=1:modeled=534|E2:len=9:copies=1:modeled=4,1,EMD-63936,False,,True,https://files.rcsb.org/download/9U7H.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-63936/map/emd_63936.map.gz,,
9UBC,EMD-64004,gt3.0_le4.0,2026-02-25,2026-02-25,REL,yes,67109,3.43,SINGLE PARTICLE,2,polypeptide(L),204,0,0.0,2,2,1,196,211,E1:len=196:copies=1:modeled=189|E2:len=15:copies=1:modeled=15,1,EMD-64004,False,,True,https://files.rcsb.org/download/9UBC.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64004/map/emd_64004.map.gz,,
9UDC,EMD-64067,gt3.0_le4.0,2026-04-15,2026-04-15,REL,yes,55297,3.38,SINGLE PARTICLE,1,polypeptide(L),657,0,0.0,1,1,1,729,729,E1:len=729:copies=1:modeled=657,1,EMD-64067,False,,True,https://files.rcsb.org/download/9UDC.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64067/map/emd_64067.map.gz,,
9UGL,EMD-64140,le3.0,2026-04-15,2026-04-15,REL,yes,256001,2.65,SINGLE PARTICLE,1,polypeptide(L),268,0,0.0,1,1,1,401,401,E1:len=401:copies=1:modeled=268,1,EMD-64140,False,,True,https://files.rcsb.org/download/9UGL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64140/map/emd_64140.map.gz,,
9UGU,EMD-64144,gt3.0_le4.0,2026-01-07,2026-01-07,REL,yes,32001,3.07,SINGLE PARTICLE,1,polypeptide(L),416,0,0.0,1,1,1,429,429,E1:len=429:copies=1:modeled=416,1,EMD-64144,False,,True,https://files.rcsb.org/download/9UGU.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64144/map/emd_64144.map.gz,,
9UGX,EMD-64147,gt3.0_le4.0,2026-01-07,2026-01-07,REL,yes,32001,3.16,SINGLE PARTICLE,1,polypeptide(L),414,0,0.0,1,1,1,429,429,E1:len=429:copies=1:modeled=414,1,EMD-64147,False,,True,https://files.rcsb.org/download/9UGX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64147/map/emd_64147.map.gz,,
9UGY,EMD-64148,gt3.0_le4.0,2026-01-07,2026-01-07,REL,yes,55297,3.5,SINGLE PARTICLE,1,polypeptide(L),415,0,0.0,1,1,1,429,429,E1:len=429:copies=1:modeled=415,1,EMD-64148,False,,True,https://files.rcsb.org/download/9UGY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64148/map/emd_64148.map.gz,,
9UHG,EMD-64163,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,67109,3.91,SINGLE PARTICLE,3,polypeptide(L),424,0,0.0,3,3,1,198,424,E1:len=198:copies=1:modeled=198|E2:len=119:copies=1:modeled=119|E3:len=107:copies=1:modeled=107,1,EMD-64163,False,,True,https://files.rcsb.org/download/9UHG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64163/map/emd_64163.map.gz,,
9UI1,EMD-64181,gt3.0_le4.0,2026-04-22,2026-04-22,REL,yes,131073,3.54,SINGLE PARTICLE,1,polypeptide(L),539,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=539,1,EMD-64181,False,,True,https://files.rcsb.org/download/9UI1.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64181/map/emd_64181.map.gz,,
9UI6,EMD-64185,gt3.0_le4.0,2026-04-22,2026-04-22,REL,yes,131073,3.22,SINGLE PARTICLE,1,polypeptide(L),547,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=547,1,EMD-64185,False,,True,https://files.rcsb.org/download/9UI6.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64185/map/emd_64185.map.gz,,
9UIF,EMD-64193,gt3.0_le4.0,2026-04-22,2026-04-22,REL,yes,131073,3.27,SINGLE PARTICLE,1,polypeptide(L),547,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=547,1,EMD-64193,False,,True,https://files.rcsb.org/download/9UIF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64193/map/emd_64193.map.gz,,
9UIT,EMD-64204,gt3.0_le4.0,2026-04-22,2026-04-22,REL,yes,67109,3.14,SINGLE PARTICLE,1,polypeptide(L),548,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=548,1,EMD-64204,False,,True,https://files.rcsb.org/download/9UIT.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64204/map/emd_64204.map.gz,,
9UJM,EMD-64217,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,256001,3.07,SINGLE PARTICLE,1,polypeptide(L),267,0,0.0,1,1,1,389,389,E1:len=389:copies=1:modeled=267,1,EMD-64217,False,,True,https://files.rcsb.org/download/9UJM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64217/map/emd_64217.map.gz,,
9UPM,EMD-64397,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,15186,3.21,SINGLE PARTICLE,1,polypeptide(L),294,0,0.0,1,1,1,452,452,E1:len=452:copies=1:modeled=294,1,EMD-64397,False,,True,https://files.rcsb.org/download/9UPM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64397/map/emd_64397.map.gz,,
9UPN,EMD-64398,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,171501,3.37,SINGLE PARTICLE,1,polypeptide(L),286,0,0.0,1,1,1,434,434,E1:len=434:copies=1:modeled=286,1,EMD-64398,False,,True,https://files.rcsb.org/download/9UPN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64398/map/emd_64398.map.gz,,
9UPO,EMD-64399,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,18967,3.71,SINGLE PARTICLE,1,polypeptide(L),286,0,0.0,1,1,1,434,434,E1:len=434:copies=1:modeled=286,1,EMD-64399,False,,True,https://files.rcsb.org/download/9UPO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64399/map/emd_64399.map.gz,,
9US8,EMD-64463,gt3.0_le4.0,2026-05-06,2026-05-06,REL,yes,186625,3.32,SINGLE PARTICLE,1,polypeptide(L),414,0,0.0,1,1,1,553,553,E1:len=553:copies=1:modeled=414,1,EMD-64463,False,,True,https://files.rcsb.org/download/9US8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64463/map/emd_64463.map.gz,,
9UUR,EMD-64516,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,28312,3.36,SINGLE PARTICLE,2,polypeptide(L),657,0,0.0,2,2,1,403,791,E1:len=403:copies=1:modeled=308|E2:len=388:copies=1:modeled=349,1,EMD-64516,False,,True,https://files.rcsb.org/download/9UUR.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64516/map/emd_64516.map.gz,,
9UUX,EMD-64519,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,28312,3.12,SINGLE PARTICLE,2,polypeptide(L),657,0,0.0,2,2,1,403,791,E1:len=403:copies=1:modeled=308|E2:len=388:copies=1:modeled=349,1,EMD-64519,False,,True,https://files.rcsb.org/download/9UUX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64519/map/emd_64519.map.gz,,
9UVU,EMD-64537,gt3.0_le4.0,2026-05-06,2026-05-06,REL,yes,32001,3.69,SINGLE PARTICLE,2,polypeptide(L),274,0,0.0,2,2,2,487,497,E1:len=10:copies=1:modeled=10|E2:len=487:copies=1:modeled=264,1,EMD-64537,False,,True,https://files.rcsb.org/download/9UVU.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64537/map/emd_64537.map.gz,,
9UVW,EMD-64539,le3.0,2026-05-06,2026-05-06,REL,yes,24115,2.89,SINGLE PARTICLE,1,polypeptide(L),266,0,0.0,1,1,1,486,486,E1:len=486:copies=1:modeled=266,1,EMD-64539,False,,True,https://files.rcsb.org/download/9UVW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64539/map/emd_64539.map.gz,,
9UVX,EMD-64540,gt3.0_le4.0,2026-05-06,2026-05-06,REL,yes,20354,3.31,SINGLE PARTICLE,1,polypeptide(L),263,0,0.0,1,1,1,486,486,E1:len=486:copies=1:modeled=263,1,EMD-64540,False,,True,https://files.rcsb.org/download/9UVX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64540/map/emd_64540.map.gz,,
9UVY,EMD-64541,le3.0,2026-05-06,2026-05-06,REL,yes,108001,2.98,SINGLE PARTICLE,1,polypeptide(L),653,0,0.0,1,1,1,765,765,E1:len=765:copies=1:modeled=653,1,EMD-64541,False,,True,https://files.rcsb.org/download/9UVY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64541/map/emd_64541.map.gz,,
9UVZ,EMD-64542,le3.0,2026-05-06,2026-05-06,REL,yes,171501,2.99,SINGLE PARTICLE,1,polypeptide(L),682,0,0.0,1,1,1,759,759,E1:len=759:copies=1:modeled=682,1,EMD-64542,False,,True,https://files.rcsb.org/download/9UVZ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64542/map/emd_64542.map.gz,,
9UW3,EMD-64544,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,28312,3.36,SINGLE PARTICLE,2,polypeptide(L),643,0,0.0,2,2,1,403,791,E1:len=403:copies=1:modeled=295|E2:len=388:copies=1:modeled=348,1,EMD-64544,False,,True,https://files.rcsb.org/download/9UW3.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64544/map/emd_64544.map.gz,,
9UW4,EMD-64545,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,28312,3.2,SINGLE PARTICLE,2,polypeptide(L),652,0,0.0,2,2,2,403,791,E1:len=388:copies=1:modeled=346|E2:len=403:copies=1:modeled=306,1,EMD-64545,False,,True,https://files.rcsb.org/download/9UW4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64545/map/emd_64545.map.gz,,
9UXN,EMD-64587,le3.0,2026-05-27,2026-05-27,REL,yes,67109,2.7,SINGLE PARTICLE,1,polypeptide(L),286,0,0.0,1,1,1,337,337,E1:len=337:copies=1:modeled=286,1,EMD-64587,False,,True,https://files.rcsb.org/download/9UXN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64587/map/emd_64587.map.gz,,
9UXO,EMD-64588,le3.0,2026-05-27,2026-05-27,REL,yes,67109,2.97,SINGLE PARTICLE,1,polypeptide(L),292,0,0.0,1,1,1,337,337,E1:len=337:copies=1:modeled=292,1,EMD-64588,False,,True,https://files.rcsb.org/download/9UXO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64588/map/emd_64588.map.gz,,
9UXP,EMD-64589,le3.0,2026-05-27,2026-05-27,REL,yes,67109,2.9,SINGLE PARTICLE,1,polypeptide(L),286,0,0.0,1,1,1,337,337,E1:len=337:copies=1:modeled=286,1,EMD-64589,False,,True,https://files.rcsb.org/download/9UXP.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64589/map/emd_64589.map.gz,,
9UXQ,EMD-64590,le3.0,2026-05-27,2026-05-27,REL,yes,67109,2.89,SINGLE PARTICLE,1,polypeptide(L),286,0,0.0,1,1,1,337,337,E1:len=337:copies=1:modeled=286,1,EMD-64590,False,,True,https://files.rcsb.org/download/9UXQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64590/map/emd_64590.map.gz,,
9UXS,EMD-64597,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,536871,3.4,SINGLE PARTICLE,2,polypeptide(L),316,0,0.0,2,2,2,1245,1382,E1:len=137:copies=1:modeled=116|E2:len=1245:copies=1:modeled=200,1,EMD-64597,False,,True,https://files.rcsb.org/download/9UXS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64597/map/emd_64597.map.gz,,
9V2U,EMD-64740,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,95552,3.8,SINGLE PARTICLE,3,polypeptide(L),577,0,0.0,3,3,2,249,680,E1:len=211:copies=1:modeled=155|E2:len=249:copies=1:modeled=204|E3:len=220:copies=1:modeled=218,1,EMD-64740,False,,True,https://files.rcsb.org/download/9V2U.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64740/map/emd_64740.map.gz,,
9V35,EMD-64749,le3.0,2026-07-01,2026-07-01,REL,yes,87809,2.97,SINGLE PARTICLE,1,polypeptide(L),266,0,0.0,1,1,1,393,393,E1:len=393:copies=1:modeled=266,1,EMD-64749,False,,True,https://files.rcsb.org/download/9V35.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64749/map/emd_64749.map.gz,,
9V38,EMD-64751,gt3.0_le4.0,2026-07-01,2026-07-01,REL,yes,67109,3.25,SINGLE PARTICLE,2,polypeptide(L),259,0,0.0,2,2,1,393,403,E1:len=393:copies=1:modeled=253|E2:len=10:copies=1:modeled=6,1,EMD-64751,False,,True,https://files.rcsb.org/download/9V38.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64751/map/emd_64751.map.gz,,
9V3C,EMD-64752,gt3.0_le4.0,2026-07-01,2026-07-01,REL,yes,32001,3.82,SINGLE PARTICLE,2,polypeptide(L),360,0,0.0,2,2,1,393,467,E1:len=393:copies=1:modeled=286|E2:len=74:copies=1:modeled=74,1,EMD-64752,False,,True,https://files.rcsb.org/download/9V3C.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64752/map/emd_64752.map.gz,,
9V3T,EMD-64756,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,67109,3.73,SINGLE PARTICLE,1,polypeptide(L),424,0,0.0,1,1,1,476,476,E1:len=476:copies=1:modeled=424,1,EMD-64756,False,,True,https://files.rcsb.org/download/9V3T.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64756/map/emd_64756.map.gz,,
9V3V,EMD-64757,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,67109,3.37,SINGLE PARTICLE,1,polypeptide(L),424,0,0.0,1,1,1,476,476,E1:len=476:copies=1:modeled=424,1,EMD-64757,False,,True,https://files.rcsb.org/download/9V3V.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64757/map/emd_64757.map.gz,,
9V3X,EMD-64759,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,67109,3.4,SINGLE PARTICLE,1,polypeptide(L),424,0,0.0,1,1,1,476,476,E1:len=476:copies=1:modeled=424,1,EMD-64759,False,,True,https://files.rcsb.org/download/9V3X.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64759/map/emd_64759.map.gz,,
9V3Y,EMD-64761,gt3.0_le4.0,2026-07-01,2026-07-01,REL,yes,55297,3.06,SINGLE PARTICLE,2,polypeptide(L),260,0,0.0,2,2,2,393,399,E1:len=6:copies=1:modeled=6|E2:len=393:copies=1:modeled=254,1,EMD-64761,False,,True,https://files.rcsb.org/download/9V3Y.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64761/map/emd_64761.map.gz,,
9V3Z,EMD-64762,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,67109,3.4,SINGLE PARTICLE,1,polypeptide(L),424,0,0.0,1,1,1,476,476,E1:len=476:copies=1:modeled=424,1,EMD-64762,False,,True,https://files.rcsb.org/download/9V3Z.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64762/map/emd_64762.map.gz,,
9V4D,EMD-64777,gt3.0_le4.0,2026-07-01,2026-07-01,REL,yes,55297,3.07,SINGLE PARTICLE,2,polypeptide(L),276,0,0.0,2,2,1,393,402,E1:len=393:copies=1:modeled=267|E2:len=9:copies=1:modeled=9,1,EMD-64777,False,,True,https://files.rcsb.org/download/9V4D.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64777/map/emd_64777.map.gz,,
9V82,EMD-64830,le3.0,2026-05-06,2026-05-06,REL,yes,174457,2.56,SINGLE PARTICLE,1,polypeptide(L),244,0,0.0,1,1,1,472,472,E1:len=472:copies=1:modeled=244,1,EMD-64830,False,,True,https://files.rcsb.org/download/9V82.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64830/map/emd_64830.map.gz,,
9V8W,EMD-64853,le3.0,2026-04-15,2026-04-15,REL,yes,67109,2.75,SINGLE PARTICLE,1,polypeptide(L),505,0,0.0,1,1,1,635,635,E1:len=635:copies=1:modeled=505,1,EMD-64853,False,,True,https://files.rcsb.org/download/9V8W.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64853/map/emd_64853.map.gz,,
9V8X,EMD-64854,le3.0,2026-04-15,2026-04-15,REL,yes,67109,2.85,SINGLE PARTICLE,1,polypeptide(L),504,0,0.0,1,1,1,635,635,E1:len=635:copies=1:modeled=504,1,EMD-64854,False,,True,https://files.rcsb.org/download/9V8X.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64854/map/emd_64854.map.gz,,
9V8Y,EMD-64855,gt3.0_le4.0,2026-04-15,2026-04-15,REL,yes,55297,3.28,SINGLE PARTICLE,1,polypeptide(L),476,0,0.0,1,1,1,635,635,E1:len=635:copies=1:modeled=476,1,EMD-64855,False,,True,https://files.rcsb.org/download/9V8Y.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64855/map/emd_64855.map.gz,,
9VBD,EMD-64920,gt3.0_le4.0,2026-04-08,2026-04-08,REL,yes,131073,3.26,SINGLE PARTICLE,1,polypeptide(L),521,0,0.0,1,1,1,893,893,E1:len=893:copies=1:modeled=521,1,EMD-64920,False,,True,https://files.rcsb.org/download/9VBD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64920/map/emd_64920.map.gz,,
9VBV,EMD-64934,gt3.0_le4.0,2026-04-08,2026-04-08,REL,yes,186625,3.61,SINGLE PARTICLE,1,polypeptide(L),521,0,0.0,1,1,1,893,893,E1:len=893:copies=1:modeled=521,1,EMD-64934,False,,True,https://files.rcsb.org/download/9VBV.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64934/map/emd_64934.map.gz,,
9VCT,EMD-64967,gt3.0_le4.0,2026-05-20,2026-05-20,REL,yes,67109,3.47,SINGLE PARTICLE,1,polypeptide(L),603,0,0.0,1,1,1,760,760,E1:len=760:copies=1:modeled=603,1,EMD-64967,False,,True,https://files.rcsb.org/download/9VCT.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64967/map/emd_64967.map.gz,,
9VH3,EMD-65054,le3.0,2026-09-23,2026-09-23,REL,yes,131073,2.72,SINGLE PARTICLE,1,polypeptide(L),574,0,0.0,1,1,1,835,835,E1:len=835:copies=1:modeled=574,1,EMD-65054,False,,True,https://files.rcsb.org/download/9VH3.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65054/map/emd_65054.map.gz,,
9VH9,EMD-65060,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,55297,3.4,SINGLE PARTICLE,1,polypeptide(L),553,0,0.0,1,1,1,643,643,E1:len=643:copies=1:modeled=553,1,EMD-65060,False,,True,https://files.rcsb.org/download/9VH9.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65060/map/emd_65060.map.gz,,
9VI4,EMD-65077,le3.0,2026-03-11,2026-03-11,REL,yes,1048577,1.8,SINGLE PARTICLE,1,polypeptide(L),516,0,0.0,1,1,1,516,516,E1:len=516:copies=1:modeled=516,1,EMD-65077,False,,True,https://files.rcsb.org/download/9VI4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65077/map/emd_65077.map.gz,,
9VJ3,EMD-65102,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,95552,3.4,SINGLE PARTICLE,2,polypeptide(L),445,0,0.0,2,2,1,401,622,E1:len=401:copies=1:modeled=280|E2:len=221:copies=1:modeled=165,1,EMD-65102,False,,True,https://files.rcsb.org/download/9VJ3.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65102/map/emd_65102.map.gz,,
9VJ4,EMD-65103,gt3.0_le4.0,2026-03-11,2026-03-11,REL,yes,35996,3.5,SINGLE PARTICLE,2,polypeptide(L),445,0,0.0,2,2,1,401,622,E1:len=401:copies=1:modeled=280|E2:len=221:copies=1:modeled=165,1,EMD-65103,False,,True,https://files.rcsb.org/download/9VJ4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65103/map/emd_65103.map.gz,,
9VND,EMD-65208,gt3.0_le4.0,2026-01-21,2026-01-21,REL,yes,32001,3.3,SINGLE PARTICLE,3,polypeptide(L),457,0,0.0,7,3,3,288,565,E1:len=135:copies=2:modeled=143|E2:len=142:copies=4:modeled=235|E3:len=288:copies=1:modeled=79,1,EMD-65208,False,,True,https://files.rcsb.org/download/9VND.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65208/map/emd_65208.map.gz,,
9VNE,EMD-65209,gt3.0_le4.0,2026-01-21,2026-01-21,REL,yes,23329,3.11,SINGLE PARTICLE,2,polypeptide(L),526,0,0.0,6,2,1,142,277,E1:len=142:copies=4:modeled=339|E2:len=135:copies=2:modeled=187,1,EMD-65209,False,,True,https://files.rcsb.org/download/9VNE.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65209/map/emd_65209.map.gz,,
9VPG,EMD-65253,gt3.0_le4.0,2026-05-13,2026-05-13,REL,yes,28312,3.14,SINGLE PARTICLE,1,polypeptide(L),465,0,0.0,1,1,1,582,582,E1:len=582:copies=1:modeled=465,1,EMD-65253,False,,True,https://files.rcsb.org/download/9VPG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65253/map/emd_65253.map.gz,,
9VUH,EMD-65363,le3.0,2026-07-15,2026-07-15,REL,yes,131073,2.7,SINGLE PARTICLE,3,polypeptide(L),634,0,0.0,3,3,2,365,1015,E1:len=332:copies=1:modeled=298|E2:len=365:copies=1:modeled=217|E3:len=318:copies=1:modeled=119,1,EMD-65363,False,,True,https://files.rcsb.org/download/9VUH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65363/map/emd_65363.map.gz,,
9VVM,EMD-64067,gt3.0_le4.0,2026-07-22,2026-04-15,REL,yes,55297,3.38,SINGLE PARTICLE,1,polypeptide(L),657,0,0.0,1,1,1,729,729,E1:len=729:copies=1:modeled=657,1,EMD-64067,False,,True,https://files.rcsb.org/download/9VVM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64067/map/emd_64067.map.gz,,
9VWR,EMD-65400,le3.0,2026-07-22,2026-07-22,REL,yes,67109,2.9,SINGLE PARTICLE,1,polypeptide(L),551,0,0.0,1,1,1,566,566,E1:len=566:copies=1:modeled=551,1,EMD-65400,False,,True,https://files.rcsb.org/download/9VWR.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65400/map/emd_65400.map.gz,,
9VWS,EMD-65401,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,67109,3.2,SINGLE PARTICLE,1,polypeptide(L),539,0,0.0,1,1,1,630,630,E1:len=630:copies=1:modeled=539,1,EMD-65401,False,,True,https://files.rcsb.org/download/9VWS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65401/map/emd_65401.map.gz,,
9VXT,EMD-65440,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,11944,3.9,SINGLE PARTICLE,1,polypeptide(L),584,0,0.0,2,1,1,379,379,E1:len=379:copies=2:modeled=584,1,EMD-65440,False,,True,https://files.rcsb.org/download/9VXT.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65440/map/emd_65440.map.gz,,
9VZP,EMD-65482,le3.0,2026-07-29,2026-07-29,REL,yes,442369,2.93,SINGLE PARTICLE,1,polypeptide(L),611,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=611,1,EMD-65482,False,,True,https://files.rcsb.org/download/9VZP.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65482/map/emd_65482.map.gz,,
9W14,EMD-65523,le3.0,2026-03-18,2026-03-18,REL,yes,1088392,2.33,SINGLE PARTICLE,3,polypeptide(L),412,0,0.0,3,3,1,223,449,E1:len=223:copies=1:modeled=186|E2:len=107:copies=1:modeled=107|E3:len=119:copies=1:modeled=119,1,EMD-65523,False,,True,https://files.rcsb.org/download/9W14.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65523/map/emd_65523.map.gz,,
9W22,EMD-65563,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,256001,3.12,SINGLE PARTICLE,1,polypeptide(L),692,0,0.0,2,1,1,358,358,E1:len=358:copies=2:modeled=692,1,EMD-65563,False,,True,https://files.rcsb.org/download/9W22.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65563/map/emd_65563.map.gz,,
9W3F,EMD-65599,gt3.0_le4.0,2026-06-10,2026-06-10,REL,yes,47410,3.04,SINGLE PARTICLE,2,polypeptide(L),393,0,0.0,2,2,1,517,642,E1:len=517:copies=1:modeled=275|E2:len=125:copies=1:modeled=118,1,EMD-65599,False,,True,https://files.rcsb.org/download/9W3F.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65599/map/emd_65599.map.gz,,
9W43,EMD-65617,le3.0,2026-07-01,2026-07-01,REL,yes,108001,2.96,SINGLE PARTICLE,3,polypeptide(L),507,0,0.0,3,3,1,213,539,E1:len=213:copies=1:modeled=213|E2:len=211:copies=1:modeled=211|E3:len=115:copies=1:modeled=83,1,EMD-65617,False,,True,https://files.rcsb.org/download/9W43.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65617/map/emd_65617.map.gz,,
9W4I,EMD-65634,le3.0,2026-07-01,2026-07-01,REL,yes,256001,2.57,SINGLE PARTICLE,3,polypeptide(L),620,0,0.0,3,3,2,317,861,E1:len=297:copies=1:modeled=190|E2:len=317:copies=1:modeled=211|E3:len=247:copies=1:modeled=219,1,EMD-65634,False,,True,https://files.rcsb.org/download/9W4I.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65634/map/emd_65634.map.gz,,
9W62,EMD-65674,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,108001,3.28,SINGLE PARTICLE,1,polypeptide(L),574,0,0.0,1,1,1,914,914,E1:len=914:copies=1:modeled=574,1,EMD-65674,False,,True,https://files.rcsb.org/download/9W62.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65674/map/emd_65674.map.gz,,
9W63,EMD-65675,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,108001,3.29,SINGLE PARTICLE,1,polypeptide(L),573,0,0.0,1,1,1,914,914,E1:len=914:copies=1:modeled=573,1,EMD-65675,False,,True,https://files.rcsb.org/download/9W63.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65675/map/emd_65675.map.gz,,
9W64,EMD-65676,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,108001,3.81,SINGLE PARTICLE,1,polypeptide(L),575,0,0.0,1,1,1,914,914,E1:len=914:copies=1:modeled=575,1,EMD-65676,False,,True,https://files.rcsb.org/download/9W64.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65676/map/emd_65676.map.gz,,
9W65,EMD-65677,le3.0,2026-05-27,2026-05-27,REL,yes,108001,2.94,SINGLE PARTICLE,1,polypeptide(L),575,0,0.0,1,1,1,914,914,E1:len=914:copies=1:modeled=575,1,EMD-65677,False,,True,https://files.rcsb.org/download/9W65.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65677/map/emd_65677.map.gz,,
9W66,EMD-65678,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,23329,3.3,SINGLE PARTICLE,1,polypeptide(L),581,0,0.0,1,1,1,914,914,E1:len=914:copies=1:modeled=581,1,EMD-65678,False,,True,https://files.rcsb.org/download/9W66.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65678/map/emd_65678.map.gz,,
9W8B,EMD-65742,gt3.0_le4.0,2026-10-07,2026-10-07,REL,yes,131073,3.03,SINGLE PARTICLE,1,polypeptide(L),385,0,0.0,1,1,1,514,514,E1:len=514:copies=1:modeled=385,1,EMD-65742,False,,True,https://files.rcsb.org/download/9W8B.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65742/map/emd_65742.map.gz,,
9W8C,EMD-65743,le3.0,2026-10-07,2026-10-07,REL,yes,131073,2.9,SINGLE PARTICLE,1,polypeptide(L),387,0,0.0,1,1,1,514,514,E1:len=514:copies=1:modeled=387,1,EMD-65743,False,,True,https://files.rcsb.org/download/9W8C.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65743/map/emd_65743.map.gz,,
9W8D,EMD-65744,le3.0,2026-10-07,2026-10-07,REL,yes,131073,2.89,SINGLE PARTICLE,1,polypeptide(L),385,0,0.0,1,1,1,514,514,E1:len=514:copies=1:modeled=385,1,EMD-65744,False,,True,https://files.rcsb.org/download/9W8D.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65744/map/emd_65744.map.gz,,
9W8E,EMD-65745,le3.0,2026-10-07,2026-10-07,REL,yes,131073,2.88,SINGLE PARTICLE,1,polypeptide(L),376,0,0.0,1,1,1,474,474,E1:len=474:copies=1:modeled=376,1,EMD-65745,False,,True,https://files.rcsb.org/download/9W8E.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65745/map/emd_65745.map.gz,,
9W8F,EMD-65746,le3.0,2026-10-07,2026-10-07,REL,yes,131073,2.97,SINGLE PARTICLE,1,polypeptide(L),387,0,0.0,1,1,1,474,474,E1:len=474:copies=1:modeled=387,1,EMD-65746,False,,True,https://files.rcsb.org/download/9W8F.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65746/map/emd_65746.map.gz,,
9W8J,EMD-65747,le3.0,2026-06-24,2026-06-24,REL,yes,67109,2.73,SINGLE PARTICLE,1,polypeptide(L),557,0,0.0,1,1,1,595,595,E1:len=595:copies=1:modeled=557,1,EMD-65747,False,,True,https://files.rcsb.org/download/9W8J.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65747/map/emd_65747.map.gz,,
9W8Q,EMD-65748,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,67109,3.2,SINGLE PARTICLE,1,polypeptide(L),554,0,0.0,1,1,1,593,593,E1:len=593:copies=1:modeled=554,1,EMD-65748,False,,True,https://files.rcsb.org/download/9W8Q.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65748/map/emd_65748.map.gz,,
9W97,EMD-65765,le3.0,2026-05-27,2026-05-27,REL,yes,67109,2.84,SINGLE PARTICLE,1,polypeptide(L),540,0,0.0,1,1,1,1291,1291,E1:len=1291:copies=1:modeled=540,1,EMD-65765,False,,True,https://files.rcsb.org/download/9W97.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65765/map/emd_65765.map.gz,,
9W98,EMD-65766,le3.0,2026-05-27,2026-05-27,REL,yes,67109,2.71,SINGLE PARTICLE,1,polypeptide(L),536,0,0.0,1,1,1,1291,1291,E1:len=1291:copies=1:modeled=536,1,EMD-65766,False,,True,https://files.rcsb.org/download/9W98.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65766/map/emd_65766.map.gz,,
9W99,EMD-65767,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,67109,3.04,SINGLE PARTICLE,1,polypeptide(L),541,0,0.0,1,1,1,1291,1291,E1:len=1291:copies=1:modeled=541,1,EMD-65767,False,,True,https://files.rcsb.org/download/9W99.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65767/map/emd_65767.map.gz,,
9W9A,EMD-65768,le3.0,2026-05-27,2026-05-27,REL,yes,67109,2.98,SINGLE PARTICLE,1,polypeptide(L),542,0,0.0,1,1,1,1291,1291,E1:len=1291:copies=1:modeled=542,1,EMD-65768,False,,True,https://files.rcsb.org/download/9W9A.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65768/map/emd_65768.map.gz,,
9W9B,EMD-65769,le3.0,2026-05-27,2026-05-27,REL,yes,67109,2.87,SINGLE PARTICLE,1,polypeptide(L),536,0,0.0,1,1,1,1291,1291,E1:len=1291:copies=1:modeled=536,1,EMD-65769,False,,True,https://files.rcsb.org/download/9W9B.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65769/map/emd_65769.map.gz,,
9W9C,EMD-65770,gt3.0_le4.0,2026-05-27,2026-05-27,REL,yes,67109,3.02,SINGLE PARTICLE,1,polypeptide(L),534,0,0.0,1,1,1,1291,1291,E1:len=1291:copies=1:modeled=534,1,EMD-65770,False,,True,https://files.rcsb.org/download/9W9C.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65770/map/emd_65770.map.gz,,
9WAG,EMD-65817,le3.0,2026-03-04,2026-03-04,REL,yes,464858,2.95,SINGLE PARTICLE,3,polypeptide(L),625,0,0.0,3,3,1,302,812,E1:len=302:copies=1:modeled=191|E2:len=272:copies=1:modeled=209|E3:len=238:copies=1:modeled=225,1,EMD-65817,False,,True,https://files.rcsb.org/download/9WAG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65817/map/emd_65817.map.gz,,
9WBR,EMD-65847,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,67109,3.13,SINGLE PARTICLE,1,polypeptide(L),557,0,0.0,1,1,1,797,797,E1:len=797:copies=1:modeled=557,1,EMD-65847,False,,True,https://files.rcsb.org/download/9WBR.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65847/map/emd_65847.map.gz,,
9WC6,EMD-65856,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,67109,3.1,SINGLE PARTICLE,1,polypeptide(L),557,0,0.0,1,1,1,797,797,E1:len=797:copies=1:modeled=557,1,EMD-65856,False,,True,https://files.rcsb.org/download/9WC6.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65856/map/emd_65856.map.gz,,
9WF3,EMD-65922,le3.0,2026-04-01,2026-04-01,REL,yes,67109,2.65,SINGLE PARTICLE,2,polypeptide(L),680,0,0.0,2,2,1,697,725,E1:len=697:copies=1:modeled=652|E2:len=28:copies=1:modeled=28,1,EMD-65922,False,,True,https://files.rcsb.org/download/9WF3.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65922/map/emd_65922.map.gz,,
9WF9,EMD-65923,le3.0,2026-04-01,2026-04-01,REL,yes,67109,2.83,SINGLE PARTICLE,2,polypeptide(L),680,0,0.0,2,2,2,697,725,E1:len=28:copies=1:modeled=28|E2:len=697:copies=1:modeled=652,1,EMD-65923,False,,True,https://files.rcsb.org/download/9WF9.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65923/map/emd_65923.map.gz,,
9WFA,EMD-65924,le3.0,2026-03-18,2026-03-18,REL,yes,40311,2.64,SINGLE PARTICLE,1,polypeptide(L),245,0,0.0,1,1,1,291,291,E1:len=291:copies=1:modeled=245,1,EMD-65924,False,,True,https://files.rcsb.org/download/9WFA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65924/map/emd_65924.map.gz,,
9WFC,EMD-65925,gt3.0_le4.0,2026-04-01,2026-04-01,REL,yes,67109,3.33,SINGLE PARTICLE,2,polypeptide(L),681,0,0.0,2,2,1,697,725,E1:len=697:copies=1:modeled=653|E2:len=28:copies=1:modeled=28,1,EMD-65925,False,,True,https://files.rcsb.org/download/9WFC.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65925/map/emd_65925.map.gz,,
9WFL,EMD-65930,le3.0,2026-04-01,2026-04-01,REL,yes,67109,2.75,SINGLE PARTICLE,2,polypeptide(L),682,0,0.0,2,2,1,697,726,E1:len=697:copies=1:modeled=653|E2:len=29:copies=1:modeled=29,1,EMD-65930,False,,True,https://files.rcsb.org/download/9WFL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65930/map/emd_65930.map.gz,,
9WFN,EMD-65932,le3.0,2026-04-01,2026-04-01,REL,yes,67109,2.94,SINGLE PARTICLE,2,polypeptide(L),668,0,0.0,2,2,2,696,712,E1:len=16:copies=1:modeled=16|E2:len=696:copies=1:modeled=652,1,EMD-65932,False,,True,https://files.rcsb.org/download/9WFN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65932/map/emd_65932.map.gz,,
9WFO,EMD-65933,gt3.0_le4.0,2026-07-15,2026-07-15,REL,yes,67109,3.31,SINGLE PARTICLE,2,polypeptide(L),662,0,0.0,2,2,2,696,713,E1:len=17:copies=1:modeled=17|E2:len=696:copies=1:modeled=645,1,EMD-65933,False,,True,https://files.rcsb.org/download/9WFO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65933/map/emd_65933.map.gz,,
9WIJ,EMD-65995,gt3.0_le4.0,2026-07-01,2026-07-01,REL,yes,131073,3.05,SINGLE PARTICLE,1,polypeptide(L),290,0,0.0,1,1,1,447,447,E1:len=447:copies=1:modeled=290,1,EMD-65995,False,,True,https://files.rcsb.org/download/9WIJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65995/map/emd_65995.map.gz,,
9WIX,EMD-66010,gt3.0_le4.0,2026-09-02,2026-09-02,REL,yes,70305,3.3,SINGLE PARTICLE,1,polypeptide(L),444,0,0.0,1,1,1,546,546,E1:len=546:copies=1:modeled=444,1,EMD-66010,False,,True,https://files.rcsb.org/download/9WIX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66010/map/emd_66010.map.gz,,
9WME,EMD-66088,gt3.0_le4.0,2026-09-23,2026-09-23,REL,yes,23329,3.2,SINGLE PARTICLE,2,polypeptide(L),446,0,0.0,2,2,2,657,1110,E1:len=453:copies=1:modeled=439|E2:len=657:copies=1:modeled=7,1,EMD-66088,False,,True,https://files.rcsb.org/download/9WME.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66088/map/emd_66088.map.gz,,
9WML,EMD-66096,le3.0,2026-03-25,2026-03-25,REL,yes,67109,2.79,SINGLE PARTICLE,1,polypeptide(L),541,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=541,1,EMD-66096,False,,True,https://files.rcsb.org/download/9WML.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66096/map/emd_66096.map.gz,,
9WMM,EMD-66097,le3.0,2026-03-25,2026-03-25,REL,yes,67109,2.8,SINGLE PARTICLE,1,polypeptide(L),540,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=540,1,EMD-66097,False,,True,https://files.rcsb.org/download/9WMM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66097/map/emd_66097.map.gz,,
9WMN,EMD-66098,le3.0,2026-03-25,2026-03-25,REL,yes,67109,2.91,SINGLE PARTICLE,1,polypeptide(L),540,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=540,1,EMD-66098,False,,True,https://files.rcsb.org/download/9WMN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66098/map/emd_66098.map.gz,,
9WMO,EMD-66099,gt3.0_le4.0,2026-03-25,2026-03-25,REL,yes,67109,3.01,SINGLE PARTICLE,1,polypeptide(L),540,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=540,1,EMD-66099,False,,True,https://files.rcsb.org/download/9WMO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66099/map/emd_66099.map.gz,,
9WMP,EMD-66100,le3.0,2026-03-25,2026-03-25,REL,yes,67109,2.91,SINGLE PARTICLE,1,polypeptide(L),540,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=540,1,EMD-66100,False,,True,https://files.rcsb.org/download/9WMP.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66100/map/emd_66100.map.gz,,
9WMQ,EMD-66101,le3.0,2026-07-01,2026-07-01,REL,yes,1048577,1.73,SINGLE PARTICLE,3,polypeptide(L),210,0,0.0,3,3,1,182,406,E1:len=182:copies=1:modeled=182|E2:len=120:copies=1:modeled=22|E3:len=104:copies=1:modeled=6,1,EMD-66101,False,,True,https://files.rcsb.org/download/9WMQ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66101/map/emd_66101.map.gz,,
9WMR,EMD-66102,le3.0,2026-07-01,2026-07-01,REL,yes,108001,2.48,SINGLE PARTICLE,1,polypeptide(L),182,0,0.0,1,1,1,214,214,E1:len=214:copies=1:modeled=182,1,EMD-66102,False,,True,https://files.rcsb.org/download/9WMR.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66102/map/emd_66102.map.gz,,
9WNG,EMD-66117,le3.0,2026-09-09,2026-09-09,REL,yes,70305,2.98,SINGLE PARTICLE,3,polypeptide(L),685,0,0.0,3,3,3,307,813,E1:len=236:copies=1:modeled=211|E2:len=270:copies=1:modeled=222|E3:len=307:copies=1:modeled=252,1,EMD-66117,False,,True,https://files.rcsb.org/download/9WNG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66117/map/emd_66117.map.gz,,
9WQ6,EMD-66160,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,131073,3.4,SINGLE PARTICLE,1,polypeptide(L),261,0,0.0,1,1,1,339,339,E1:len=339:copies=1:modeled=261,1,EMD-66160,False,,True,https://files.rcsb.org/download/9WQ6.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66160/map/emd_66160.map.gz,,
9WQA,EMD-66164,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,70305,3.28,SINGLE PARTICLE,3,polypeptide(L),687,0,0.0,3,3,1,318,778,E1:len=318:copies=1:modeled=259|E2:len=213:copies=1:modeled=211|E3:len=247:copies=1:modeled=217,1,EMD-66164,False,,True,https://files.rcsb.org/download/9WQA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66164/map/emd_66164.map.gz,,
9WQB,EMD-66165,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,70305,3.18,SINGLE PARTICLE,3,polypeptide(L),692,0,0.0,3,3,1,318,778,E1:len=318:copies=1:modeled=259|E2:len=213:copies=1:modeled=211|E3:len=247:copies=1:modeled=222,1,EMD-66165,False,,True,https://files.rcsb.org/download/9WQB.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66165/map/emd_66165.map.gz,,
9WS8,EMD-66194,gt3.0_le4.0,2026-01-07,2026-01-07,REL,yes,32001,3.47,SINGLE PARTICLE,1,polypeptide(L),416,0,0.0,1,1,1,429,429,E1:len=429:copies=1:modeled=416,1,EMD-66194,False,,True,https://files.rcsb.org/download/9WS8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66194/map/emd_66194.map.gz,,
9WTX,EMD-66226,gt3.0_le4.0,2026-09-23,2026-09-23,REL,yes,131073,3.37,SINGLE PARTICLE,1,polypeptide(L),500,0,0.0,1,1,1,551,551,E1:len=551:copies=1:modeled=500,1,EMD-66226,False,,True,https://files.rcsb.org/download/9WTX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66226/map/emd_66226.map.gz,,
9WTY,EMD-66227,gt3.0_le4.0,2026-09-23,2026-09-23,REL,yes,131073,3.11,SINGLE PARTICLE,1,polypeptide(L),510,0,0.0,1,1,1,551,551,E1:len=551:copies=1:modeled=510,1,EMD-66227,False,,True,https://files.rcsb.org/download/9WTY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66227/map/emd_66227.map.gz,,
9WWG,EMD-66311,le3.0,2026-09-30,2026-09-30,REL,yes,95552,2.64,SINGLE PARTICLE,1,polypeptide(L),515,0,0.0,1,1,1,545,545,E1:len=545:copies=1:modeled=515,1,EMD-66311,False,,True,https://files.rcsb.org/download/9WWG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66311/map/emd_66311.map.gz,,
9WWI,EMD-66312,gt3.0_le4.0,2026-09-30,2026-09-30,REL,yes,95552,3.22,SINGLE PARTICLE,1,polypeptide(L),515,0,0.0,1,1,1,545,545,E1:len=545:copies=1:modeled=515,1,EMD-66312,False,,True,https://files.rcsb.org/download/9WWI.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66312/map/emd_66312.map.gz,,
9WX7,EMD-66332,le3.0,2026-03-25,2026-03-25,REL,yes,67109,2.57,SINGLE PARTICLE,1,polypeptide(L),540,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=540,1,EMD-66332,False,,True,https://files.rcsb.org/download/9WX7.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66332/map/emd_66332.map.gz,,
9WX9,EMD-66339,le3.0,2026-03-25,2026-03-25,REL,yes,67109,2.84,SINGLE PARTICLE,1,polypeptide(L),540,0,0.0,1,1,1,636,636,E1:len=636:copies=1:modeled=540,1,EMD-66339,False,,True,https://files.rcsb.org/download/9WX9.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66339/map/emd_66339.map.gz,,
9X2N,EMD-66480,gt3.0_le4.0,2026-09-23,2026-09-23,REL,yes,67109,3.2,SINGLE PARTICLE,2,polypeptide(L),541,0,0.0,2,2,1,534,599,E1:len=534:copies=1:modeled=511|E2:len=65:copies=1:modeled=30,1,EMD-66480,False,,True,https://files.rcsb.org/download/9X2N.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66480/map/emd_66480.map.gz,,
9X35,EMD-66494,le3.0,2026-08-19,2026-08-19,REL,yes,67109,2.81,SINGLE PARTICLE,3,polypeptide(L),588,0,0.0,3,3,3,215,598,E1:len=170:copies=1:modeled=166|E2:len=213:copies=1:modeled=210|E3:len=215:copies=1:modeled=212,1,EMD-66494,False,,True,https://files.rcsb.org/download/9X35.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66494/map/emd_66494.map.gz,,
9X3S,EMD-66508,gt3.0_le4.0,2026-07-22,2026-07-22,REL,yes,55297,3.7,SINGLE PARTICLE,3,polypeptide(L),681,0,0.0,4,3,3,364,848,E1:len=163:copies=2:modeled=324|E2:len=321:copies=1:modeled=164|E3:len=364:copies=1:modeled=193,1,EMD-66508,False,,True,https://files.rcsb.org/download/9X3S.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66508/map/emd_66508.map.gz,,
9X3Y,EMD-66514,le3.0,2026-04-15,2026-04-15,REL,yes,67109,2.77,SINGLE PARTICLE,2,polypeptide(L),247,0,0.0,2,2,1,325,333,E1:len=325:copies=1:modeled=241|E2:len=8:copies=1:modeled=6,1,EMD-66514,False,,True,https://files.rcsb.org/download/9X3Y.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66514/map/emd_66514.map.gz,,
9X41,EMD-66517,gt3.0_le4.0,2026-05-20,2026-05-20,REL,yes,67109,3.31,SINGLE PARTICLE,1,polypeptide(L),242,0,0.0,1,1,1,325,325,E1:len=325:copies=1:modeled=242,1,EMD-66517,False,,True,https://files.rcsb.org/download/9X41.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66517/map/emd_66517.map.gz,,
9XBS,EMD-66710,gt3.0_le4.0,2026-08-26,2026-08-26,REL,yes,131073,3.16,SINGLE PARTICLE,1,polypeptide(L),558,0,0.0,1,1,1,577,577,E1:len=577:copies=1:modeled=558,1,EMD-66710,False,,True,https://files.rcsb.org/download/9XBS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66710/map/emd_66710.map.gz,,
9XCB,EMD-66728,gt3.0_le4.0,2026-03-18,2026-03-18,REL,yes,131073,3.08,SINGLE PARTICLE,1,polypeptide(L),405,0,0.0,1,1,1,452,452,E1:len=452:copies=1:modeled=405,1,EMD-66728,False,,True,https://files.rcsb.org/download/9XCB.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66728/map/emd_66728.map.gz,,
9XCH,EMD-66732,le3.0,2026-09-09,2026-09-09,REL,yes,136049,2.93,SINGLE PARTICLE,2,polypeptide(L),680,0,0.0,2,2,1,666,935,E1:len=666:copies=1:modeled=443|E2:len=269:copies=1:modeled=237,1,EMD-66732,False,,True,https://files.rcsb.org/download/9XCH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66732/map/emd_66732.map.gz,,
9XD0,EMD-66750,gt3.0_le4.0,2026-03-11,2026-03-11,REL,yes,35996,3.53,SINGLE PARTICLE,3,polypeptide(L),454,0,0.0,3,3,1,401,631,E1:len=401:copies=1:modeled=280|E2:len=221:copies=1:modeled=165|E3:len=9:copies=1:modeled=9,1,EMD-66750,False,,True,https://files.rcsb.org/download/9XD0.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66750/map/emd_66750.map.gz,,
9XFJ,EMD-66824,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,67109,3.25,SINGLE PARTICLE,2,polypeptide(L),518,0,0.0,2,2,1,361,715,E1:len=361:copies=1:modeled=307|E2:len=354:copies=1:modeled=211,1,EMD-66824,False,,True,https://files.rcsb.org/download/9XFJ.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66824/map/emd_66824.map.gz,,
9XGW,EMD-66860,gt3.0_le4.0,2026-09-09,2026-09-09,REL,yes,55297,3.2,SINGLE PARTICLE,3,polypeptide(L),416,0,0.0,3,3,3,184,420,E1:len=127:copies=1:modeled=127|E2:len=109:copies=1:modeled=107|E3:len=184:copies=1:modeled=182,1,EMD-66860,False,,True,https://files.rcsb.org/download/9XGW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66860/map/emd_66860.map.gz,,
9XGX,EMD-66861,le3.0,2026-09-09,2026-09-09,REL,yes,131073,2.8,SINGLE PARTICLE,3,polypeptide(L),417,0,0.0,3,3,3,199,425,E1:len=117:copies=1:modeled=117|E2:len=109:copies=1:modeled=109|E3:len=199:copies=1:modeled=191,1,EMD-66861,False,,True,https://files.rcsb.org/download/9XGX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66861/map/emd_66861.map.gz,,
9XI8,EMD-66900,gt3.0_le4.0,2026-08-12,2026-08-12,REL,yes,28312,3.45,SINGLE PARTICLE,3,polypeptide(L),612,0,0.0,3,3,3,1256,1701,E1:len=228:copies=1:modeled=222|E2:len=217:copies=1:modeled=213|E3:len=1256:copies=1:modeled=177,1,EMD-66900,False,,True,https://files.rcsb.org/download/9XI8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66900/map/emd_66900.map.gz,,
9XJK,EMD-66938,gt3.0_le4.0,2026-09-02,2026-09-02,REL,yes,67109,3.46,SINGLE PARTICLE,3,polypeptide(L),617,0,0.0,3,3,2,225,631,E1:len=188:copies=1:modeled=174|E2:len=225:copies=1:modeled=225|E3:len=218:copies=1:modeled=218,1,EMD-66938,False,,True,https://files.rcsb.org/download/9XJK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66938/map/emd_66938.map.gz,,
9XKO,EMD-66973,le3.0,2026-04-29,2026-04-29,REL,yes,108001,2.35,SINGLE PARTICLE,1,polypeptide(L),368,0,0.0,1,1,1,397,397,E1:len=397:copies=1:modeled=368,1,EMD-66973,False,,True,https://files.rcsb.org/download/9XKO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66973/map/emd_66973.map.gz,,
9XMM,EMD-67027,le3.0,2026-04-15,2026-04-15,REL,yes,186625,2.88,SINGLE PARTICLE,3,polypeptide(L),690,0,0.0,3,3,1,595,1078,E1:len=595:copies=1:modeled=440|E2:len=474:copies=1:modeled=241|E3:len=9:copies=1:modeled=9,1,EMD-67027,False,,True,https://files.rcsb.org/download/9XMM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67027/map/emd_67027.map.gz,,
9XQB,EMD-67107,gt3.0_le4.0,2026-01-07,2026-01-07,REL,yes,186625,3.45,SINGLE PARTICLE,3,polypeptide(L),453,0,0.0,3,3,2,431,694,E1:len=127:copies=1:modeled=107|E2:len=431:copies=1:modeled=222|E3:len=136:copies=1:modeled=124,1,EMD-67107,False,,True,https://files.rcsb.org/download/9XQB.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67107/map/emd_67107.map.gz,,
9XQN,EMD-67119,gt3.0_le4.0,2026-02-04,2026-02-04,REL,yes,186625,3.91,SINGLE PARTICLE,3,polypeptide(L),592,0,0.0,3,3,1,422,650,E1:len=422:copies=1:modeled=365|E2:len=107:copies=1:modeled=106|E3:len=121:copies=1:modeled=121,1,EMD-67119,False,,True,https://files.rcsb.org/download/9XQN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67119/map/emd_67119.map.gz,,
9XR3,EMD-67133,gt3.0_le4.0,2026-09-30,2026-09-30,REL,yes,42593,3.15,SINGLE PARTICLE,1,polypeptide(L),560,0,0.0,2,1,1,485,485,E1:len=485:copies=2:modeled=560,1,EMD-67133,False,,True,https://files.rcsb.org/download/9XR3.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67133/map/emd_67133.map.gz,,
9XR4,EMD-67134,gt3.0_le4.0,2026-09-30,2026-09-30,REL,yes,67109,3.27,SINGLE PARTICLE,1,polypeptide(L),563,0,0.0,2,1,1,485,485,E1:len=485:copies=2:modeled=563,1,EMD-67134,False,,True,https://files.rcsb.org/download/9XR4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-67134/map/emd_67134.map.gz,,
9XZG,EMD-72353,le3.0,2026-08-05,2026-08-05,REL,yes,702465,2.7,SINGLE PARTICLE,2,polypeptide(L),523,0,0.0,2,2,1,443,766,E1:len=443:copies=1:modeled=324|E2:len=323:copies=1:modeled=199,1,EMD-72353,False,,True,https://files.rcsb.org/download/9XZG.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72353/map/emd_72353.map.gz,,
9XZK,EMD-72359,gt3.0_le4.0,2026-01-28,2026-01-28,REL,yes,629857,3.91,SINGLE PARTICLE,2,polypeptide(L),492,0,0.0,2,2,1,776,884,E1:len=776:copies=1:modeled=476|E2:len=108:copies=1:modeled=16,1,EMD-72359,False,,True,https://files.rcsb.org/download/9XZK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72359/map/emd_72359.map.gz,,
9XZN,EMD-72367,gt3.0_le4.0,2026-04-29,2026-04-29,REL,yes,226493,3.63,SINGLE PARTICLE,1,polypeptide(L),580,0,0.0,1,1,1,594,594,E1:len=594:copies=1:modeled=580,1,EMD-72367,False,,True,https://files.rcsb.org/download/9XZN.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72367/map/emd_72367.map.gz,,
9XZX,EMD-72377,gt3.0_le4.0,2026-05-06,2026-05-06,REL,yes,32001,3.1,SINGLE PARTICLE,3,polypeptide(L),474,0,0.0,3,3,1,236,476,E1:len=236:copies=1:modeled=236|E2:len=116:copies=1:modeled=115|E3:len=124:copies=1:modeled=123,1,EMD-72377,False,,True,https://files.rcsb.org/download/9XZX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72377/map/emd_72377.map.gz,,
9Y1V,EMD-72404,gt3.0_le4.0,2026-09-09,2026-09-09,REL,yes,55297,3.4,SINGLE PARTICLE,1,polypeptide(L),93,0,0.0,3,1,1,31,31,E1:len=31:copies=3:modeled=93,1,EMD-72404,False,,True,https://files.rcsb.org/download/9Y1V.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72404/map/emd_72404.map.gz,,
9Y61,EMD-72526,gt3.0_le4.0,2026-07-15,2026-07-15,REL,yes,67109,3.1,SINGLE PARTICLE,2,polypeptide(L),433,0,0.0,2,2,2,467,641,E1:len=174:copies=1:modeled=146|E2:len=467:copies=1:modeled=287,1,EMD-72526,False,,True,https://files.rcsb.org/download/9Y61.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72526/map/emd_72526.map.gz,,
9Y63,EMD-72539,le3.0,2026-07-22,2026-07-22,REL,yes,174457,2.3,SINGLE PARTICLE,1,polypeptide(L),687,0,0.0,3,1,1,296,296,E1:len=296:copies=3:modeled=687,1,EMD-72539,False,,True,https://files.rcsb.org/download/9Y63.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72539/map/emd_72539.map.gz,,
9Y7L,EMD-72658,le3.0,2026-05-27,2026-05-27,REL,yes,186625,2.7,SINGLE PARTICLE,3,polypeptide(L),585,0,0.0,3,3,1,263,716,E1:len=263:copies=1:modeled=142|E2:len=233:copies=1:modeled=223|E3:len=220:copies=1:modeled=220,1,EMD-72658,False,,True,https://files.rcsb.org/download/9Y7L.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72658/map/emd_72658.map.gz,,
9Y7M,EMD-72659,le3.0,2026-05-27,2026-05-27,REL,yes,186625,2.57,SINGLE PARTICLE,3,polypeptide(L),582,0,0.0,3,3,2,263,716,E1:len=233:copies=1:modeled=223|E2:len=263:copies=1:modeled=139|E3:len=220:copies=1:modeled=220,1,EMD-72659,False,,True,https://files.rcsb.org/download/9Y7M.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72659/map/emd_72659.map.gz,,
9Y7N,EMD-72660,le3.0,2026-05-27,2026-05-27,REL,yes,186625,2.69,SINGLE PARTICLE,3,polypeptide(L),482,0,0.0,3,3,1,263,716,E1:len=263:copies=1:modeled=39|E2:len=233:copies=1:modeled=223|E3:len=220:copies=1:modeled=220,1,EMD-72660,False,,True,https://files.rcsb.org/download/9Y7N.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72660/map/emd_72660.map.gz,,
9Y9X,EMD-72714,gt3.0_le4.0,2026-09-23,2026-09-23,REL,yes,55297,3.35,SINGLE PARTICLE,1,polypeptide(L),93,0,0.0,3,1,1,31,31,E1:len=31:copies=3:modeled=93,1,EMD-72714,False,,True,https://files.rcsb.org/download/9Y9X.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72714/map/emd_72714.map.gz,,
9YA9,EMD-72725,gt3.0_le4.0,2026-02-18,2026-02-18,REL,yes,87809,3.09,SINGLE PARTICLE,2,polypeptide(L),570,0,0.0,3,2,2,467,610,E1:len=143:copies=2:modeled=237|E2:len=467:copies=1:modeled=333,1,EMD-72725,False,,True,https://files.rcsb.org/download/9YA9.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72725/map/emd_72725.map.gz,,
9YC5,EMD-72760,le3.0,2026-02-04,2026-02-04,REL,yes,131073,2.94,SINGLE PARTICLE,3,polypeptide(L),606,0,0.0,3,3,1,283,723,E1:len=283:copies=1:modeled=180|E2:len=221:copies=1:modeled=214|E3:len=219:copies=1:modeled=212,1,EMD-72760,False,,True,https://files.rcsb.org/download/9YC5.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72760/map/emd_72760.map.gz,,
9YFW,EMD-72909,gt3.0_le4.0,2026-06-10,2026-06-10,REL,yes,1213858,3.21,SINGLE PARTICLE,1,polypeptide(L),564,0,0.0,1,1,1,656,656,E1:len=656:copies=1:modeled=564,1,EMD-72909,False,,True,https://files.rcsb.org/download/9YFW.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72909/map/emd_72909.map.gz,,
9YGR,EMD-72939,gt3.0_le4.0,2026-07-01,2026-07-01,REL,yes,442369,3.8,SINGLE PARTICLE,3,polypeptide(L),432,0,0.0,3,3,1,1334,1776,E1:len=1334:copies=1:modeled=204|E2:len=228:copies=1:modeled=122|E3:len=214:copies=1:modeled=106,1,EMD-72939,False,,True,https://files.rcsb.org/download/9YGR.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72939/map/emd_72939.map.gz,,
9YI2,EMD-72976,gt3.0_le4.0,2026-04-15,2026-04-15,REL,yes,108001,3.6,SINGLE PARTICLE,1,polypeptide(L),546,0,0.0,2,1,1,590,590,E1:len=590:copies=2:modeled=546,1,EMD-72976,False,,True,https://files.rcsb.org/download/9YI2.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-72976/map/emd_72976.map.gz,,
9YQL,EMD-73339,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,131073,3.57,SINGLE PARTICLE,1,polypeptide(L),476,0,0.0,1,1,1,608,608,E1:len=608:copies=1:modeled=476,1,EMD-73339,False,,True,https://files.rcsb.org/download/9YQL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73339/map/emd_73339.map.gz,,
9YR4,EMD-73359,gt3.0_le4.0,2026-06-24,2026-06-24,REL,yes,87809,3.64,SINGLE PARTICLE,2,polypeptide(L),479,0,0.0,2,2,1,607,617,E1:len=607:copies=1:modeled=475|E2:len=10:copies=1:modeled=4,1,EMD-73359,False,,True,https://files.rcsb.org/download/9YR4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73359/map/emd_73359.map.gz,,
9YW2,EMD-73536,gt3.0_le4.0,2026-04-22,2026-04-22,REL,yes,67109,3.27,SINGLE PARTICLE,3,polypeptide(L),323,0,0.0,3,3,2,880,2027,E1:len=821:copies=1:modeled=191|E2:len=880:copies=1:modeled=122|E3:len=326:copies=1:modeled=10,1,EMD-73536,False,,True,https://files.rcsb.org/download/9YW2.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73536/map/emd_73536.map.gz,,
9YWU,EMD-73561,gt3.0_le4.0,2026-10-07,2026-10-07,REL,yes,131073,3.5,SINGLE PARTICLE,1,polypeptide(L),391,0,0.0,1,1,1,456,456,E1:len=456:copies=1:modeled=391,1,EMD-73561,False,,True,https://files.rcsb.org/download/9YWU.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73561/map/emd_73561.map.gz,,
9YWV,EMD-73562,gt3.0_le4.0,2026-10-07,2026-10-07,REL,yes,131073,3.6,SINGLE PARTICLE,1,polypeptide(L),391,0,0.0,1,1,1,456,456,E1:len=456:copies=1:modeled=391,1,EMD-73562,False,,True,https://files.rcsb.org/download/9YWV.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73562/map/emd_73562.map.gz,,
9YX8,EMD-73599,le3.0,2026-06-17,2026-06-17,REL,yes,287966,2.85,SINGLE PARTICLE,1,polypeptide(L),585,0,0.0,3,1,1,1250,1250,E1:len=1250:copies=3:modeled=585,1,EMD-73599,False,,True,https://files.rcsb.org/download/9YX8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73599/map/emd_73599.map.gz,,
9YZ1,EMD-73664,le3.0,2026-10-07,2026-10-07,REL,yes,67109,2.26,SINGLE PARTICLE,1,polypeptide(L),227,0,0.0,1,1,1,268,268,E1:len=268:copies=1:modeled=227,1,EMD-73664,False,,True,https://files.rcsb.org/download/9YZ1.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73664/map/emd_73664.map.gz,,
9Z1N,EMD-73750,gt3.0_le4.0,2026-05-13,2026-05-13,REL,yes,442369,3.44,SINGLE PARTICLE,2,polypeptide(L),395,0,0.0,2,2,1,1243,1393,E1:len=1243:copies=1:modeled=269|E2:len=150:copies=1:modeled=126,1,EMD-73750,False,,True,https://files.rcsb.org/download/9Z1N.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73750/map/emd_73750.map.gz,,
9Z1X,EMD-73755,le3.0,2026-06-17,2026-06-17,REL,yes,226493,3.0,SINGLE PARTICLE,1,polypeptide(L),364,0,0.0,1,1,1,1214,1214,E1:len=1214:copies=1:modeled=364,1,EMD-73755,False,,True,https://files.rcsb.org/download/9Z1X.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73755/map/emd_73755.map.gz,,
9Z1Y,EMD-73756,gt3.0_le4.0,2026-06-17,2026-06-17,REL,yes,226493,3.26,SINGLE PARTICLE,1,polypeptide(L),364,0,0.0,1,1,1,1214,1214,E1:len=1214:copies=1:modeled=364,1,EMD-73756,False,,True,https://files.rcsb.org/download/9Z1Y.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73756/map/emd_73756.map.gz,,
9Z24,EMD-73762,gt3.0_le4.0,2026-06-10,2026-06-10,REL,yes,226493,3.1,SINGLE PARTICLE,1,polypeptide(L),338,0,0.0,1,1,1,1214,1214,E1:len=1214:copies=1:modeled=338,1,EMD-73762,False,,True,https://files.rcsb.org/download/9Z24.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73762/map/emd_73762.map.gz,,
9Z26,EMD-73764,le3.0,2026-06-10,2026-06-10,REL,yes,226493,2.9,SINGLE PARTICLE,1,polypeptide(L),364,0,0.0,1,1,1,1214,1214,E1:len=1214:copies=1:modeled=364,1,EMD-73764,False,,True,https://files.rcsb.org/download/9Z26.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73764/map/emd_73764.map.gz,,
9Z3Y,EMD-73795,le3.0,2026-06-17,2026-06-17,REL,yes,287966,2.99,SINGLE PARTICLE,1,polypeptide(L),225,0,0.0,1,1,1,1271,1271,E1:len=1271:copies=1:modeled=225,1,EMD-73795,False,,True,https://files.rcsb.org/download/9Z3Y.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73795/map/emd_73795.map.gz,,
9Z4I,EMD-73804,gt3.0_le4.0,2026-07-15,2026-07-15,REL,yes,95552,3.62,SINGLE PARTICLE,1,polypeptide(L),487,0,0.0,1,1,1,620,620,E1:len=620:copies=1:modeled=487,1,EMD-73804,False,,True,https://files.rcsb.org/download/9Z4I.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73804/map/emd_73804.map.gz,,
9Z4O,EMD-73810,le3.0,2026-06-10,2026-06-10,REL,yes,131073,2.63,SINGLE PARTICLE,1,polypeptide(L),481,0,0.0,1,1,1,575,575,E1:len=575:copies=1:modeled=481,1,EMD-73810,False,,True,https://files.rcsb.org/download/9Z4O.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73810/map/emd_73810.map.gz,,
9Z5I,EMD-73816,gt3.0_le4.0,2026-08-26,2026-08-26,REL,yes,67109,3.34,SINGLE PARTICLE,1,polypeptide(L),445,0,0.0,1,1,1,580,580,E1:len=580:copies=1:modeled=445,1,EMD-73816,False,,True,https://files.rcsb.org/download/9Z5I.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73816/map/emd_73816.map.gz,,
9Z5U,EMD-73824,gt3.0_le4.0,2026-07-01,2026-07-01,REL,yes,108001,3.06,SINGLE PARTICLE,3,polypeptide(L),423,0,0.0,3,3,1,589,1032,E1:len=589:copies=1:modeled=203|E2:len=229:copies=1:modeled=115|E3:len=214:copies=1:modeled=105,1,EMD-73824,False,,True,https://files.rcsb.org/download/9Z5U.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73824/map/emd_73824.map.gz,,
9Z60,EMD-73829,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,10047,3.53,SINGLE PARTICLE,3,polypeptide(L),694,0,0.0,3,3,1,646,1092,E1:len=646:copies=1:modeled=462|E2:len=228:copies=1:modeled=121|E3:len=218:copies=1:modeled=111,1,EMD-73829,False,,True,https://files.rcsb.org/download/9Z60.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73829/map/emd_73829.map.gz,,
9Z62,EMD-73831,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,7607,3.1,SINGLE PARTICLE,3,polypeptide(L),696,0,0.0,3,3,1,646,1092,E1:len=646:copies=1:modeled=464|E2:len=228:copies=1:modeled=121|E3:len=218:copies=1:modeled=111,1,EMD-73831,False,,True,https://files.rcsb.org/download/9Z62.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73831/map/emd_73831.map.gz,,
9Z64,EMD-73833,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,14508,3.11,SINGLE PARTICLE,3,polypeptide(L),694,0,0.0,3,3,1,646,1092,E1:len=646:copies=1:modeled=462|E2:len=228:copies=1:modeled=121|E3:len=218:copies=1:modeled=111,1,EMD-73833,False,,True,https://files.rcsb.org/download/9Z64.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73833/map/emd_73833.map.gz,,
9Z6N,EMD-73843,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,67109,3.7,SINGLE PARTICLE,1,polypeptide(L),605,0,0.0,2,1,1,392,392,E1:len=392:copies=2:modeled=605,1,EMD-73843,False,,True,https://files.rcsb.org/download/9Z6N.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73843/map/emd_73843.map.gz,,
9Z7R,EMD-73878,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,67109,3.5,SINGLE PARTICLE,1,polypeptide(L),610,0,0.0,2,1,1,392,392,E1:len=392:copies=2:modeled=610,1,EMD-73878,False,,True,https://files.rcsb.org/download/9Z7R.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73878/map/emd_73878.map.gz,,
9Z7S,EMD-73879,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,67109,3.5,SINGLE PARTICLE,1,polypeptide(L),584,0,0.0,2,1,1,392,392,E1:len=392:copies=2:modeled=584,1,EMD-73879,False,,True,https://files.rcsb.org/download/9Z7S.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73879/map/emd_73879.map.gz,,
9Z7T,EMD-73880,gt3.0_le4.0,2026-07-29,2026-07-29,REL,yes,67109,3.9,SINGLE PARTICLE,1,polypeptide(L),595,0,0.0,2,1,1,392,392,E1:len=392:copies=2:modeled=595,1,EMD-73880,False,,True,https://files.rcsb.org/download/9Z7T.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73880/map/emd_73880.map.gz,,
9ZBV,EMD-74011,gt3.0_le4.0,2026-01-21,2026-01-21,REL,yes,67109,3.2,SINGLE PARTICLE,1,polypeptide(L),463,0,0.0,4,1,1,134,134,E1:len=134:copies=4:modeled=463,1,EMD-74011,False,,True,https://files.rcsb.org/download/9ZBV.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74011/map/emd_74011.map.gz,,
9ZDY,EMD-74078,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,67109,3.9,SINGLE PARTICLE,1,polypeptide(L),463,0,0.0,4,1,1,134,134,E1:len=134:copies=4:modeled=463,1,EMD-74078,False,,True,https://files.rcsb.org/download/9ZDY.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74078/map/emd_74078.map.gz,,
9ZK8,EMD-74353,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,67109,3.18,SINGLE PARTICLE,1,polypeptide(L),208,0,0.0,1,1,1,208,208,E1:len=208:copies=1:modeled=208,1,EMD-74353,False,,True,https://files.rcsb.org/download/9ZK8.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74353/map/emd_74353.map.gz,,
9ZKC,EMD-74362,le3.0,2026-09-16,2026-09-16,REL,yes,67109,2.66,SINGLE PARTICLE,1,polypeptide(L),208,0,0.0,1,1,1,208,208,E1:len=208:copies=1:modeled=208,1,EMD-74362,False,,True,https://files.rcsb.org/download/9ZKC.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74362/map/emd_74362.map.gz,,
9ZKD,EMD-74364,le3.0,2026-09-16,2026-09-16,REL,yes,67109,2.32,SINGLE PARTICLE,1,polypeptide(L),208,0,0.0,1,1,1,208,208,E1:len=208:copies=1:modeled=208,1,EMD-74364,False,,True,https://files.rcsb.org/download/9ZKD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74364/map/emd_74364.map.gz,,
9ZL7,EMD-74394,le3.0,2026-09-16,2026-09-16,REL,yes,67109,2.77,SINGLE PARTICLE,1,polypeptide(L),208,0,0.0,1,1,1,208,208,E1:len=208:copies=1:modeled=208,1,EMD-74394,False,,True,https://files.rcsb.org/download/9ZL7.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74394/map/emd_74394.map.gz,,
9ZLC,EMD-74398,le3.0,2026-09-16,2026-09-16,REL,yes,67109,2.52,SINGLE PARTICLE,1,polypeptide(L),208,0,0.0,1,1,1,208,208,E1:len=208:copies=1:modeled=208,1,EMD-74398,False,,True,https://files.rcsb.org/download/9ZLC.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74398/map/emd_74398.map.gz,,
9ZLF,EMD-74402,le3.0,2026-09-16,2026-09-16,REL,yes,67109,2.71,SINGLE PARTICLE,1,polypeptide(L),206,0,0.0,1,1,1,206,206,E1:len=206:copies=1:modeled=206,1,EMD-74402,False,,True,https://files.rcsb.org/download/9ZLF.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74402/map/emd_74402.map.gz,,
9ZLH,EMD-74403,le3.0,2026-09-16,2026-09-16,REL,yes,67109,2.14,SINGLE PARTICLE,1,polypeptide(L),206,0,0.0,1,1,1,206,206,E1:len=206:copies=1:modeled=206,1,EMD-74403,False,,True,https://files.rcsb.org/download/9ZLH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74403/map/emd_74403.map.gz,,
9ZNO,EMD-74452,le3.0,2026-02-04,2026-02-04,REL,yes,442369,2.91,SINGLE PARTICLE,1,polypeptide(L),662,0,0.0,2,1,1,365,365,E1:len=365:copies=2:modeled=662,1,EMD-74452,False,,True,https://files.rcsb.org/download/9ZNO.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74452/map/emd_74452.map.gz,,
9ZR7,EMD-74594,gt3.0_le4.0,2026-06-10,2026-06-10,REL,yes,119165,3.28,SINGLE PARTICLE,3,polypeptide(L),494,0,0.0,3,3,1,304,745,E1:len=304:copies=1:modeled=261|E2:len=228:copies=1:modeled=125|E3:len=213:copies=1:modeled=108,1,EMD-74594,False,,True,https://files.rcsb.org/download/9ZR7.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74594/map/emd_74594.map.gz,,
9ZRT,EMD-74630,gt3.0_le4.0,2026-09-16,2026-09-16,REL,yes,296353,3.3,SINGLE PARTICLE,3,polypeptide(L),443,0,0.0,3,3,1,357,803,E1:len=357:copies=1:modeled=221|E2:len=232:copies=1:modeled=126|E3:len=214:copies=1:modeled=96,1,EMD-74630,False,,True,https://files.rcsb.org/download/9ZRT.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74630/map/emd_74630.map.gz,,


In [ ]:
%%writefile data/pool/pool_candidate_sequences.fasta
>10ED_E1 len=58 copies=1 longest=0
MYIYRLVLLLVVGIYLFSPAIMDWWIEPTGAWYRPYLLWLILIVVTFILQSQRDADEL
>10ED_E2 len=606 copies=1 longest=1
MPMSFSLTQMILISAGYLMVLFGVAWISERGLIPRSIIRHPLTYTLSLGVYASAWAFYGSVGLAYQYGYGFLACYLGVSGAFLLAPVLLYPILKITRTYQLSSLADLLAFRFRSTWAGALTTIIMLIGVLPLLALQIQAVADSISILTGEPVKARVAFAFCTLIILFTIFFGSRHIATREKHEGLVFAIAFESVIKLLALGGIGLYALYGVFGGPHELEVWLLQNQTALAALHTPLQEGPWRTLLLVFFASAIVMPHMYHMAFTENLNPRSLVSASWGLPLFLLLMSLAVPLVLWAGLRLGASTNPEYFTLGLGIAANNQALALLAYIGGLSAASGLIIVTTLALSGMALNHLVLPLYQPPAEGNIYRWLKWTRRALIVAIITAGFMFYLSQNYHQSLANLGIVAFVATLQFLPGVLSVLYWPTANRRGFIAGLLAGTLVWMVTMLLPLLGNLQGFYIPLLDMIYVLDDTSWHMAAIASLAANVLLFTLISLFTNASTEEVSAAEACAVDNVRRPQRRELHAASPQEFATQLAKPLGAKAAQKEVEQALRDLYLPFDERRPYALRRLRDRIEANLSGLMGPSVAQDMVETFLPYKSGNHHHHHHHH
>10FJ_E1 len=185 copies=1 longest=0
QAAAPPKAVLKLEPPWINVLQEDSVTLTCQGARSPESDSIQWFHNGNLIPTHTQPSYRFKANNNDSGEYTCQTGQTSLSDPVHLTVLSEWLVLQTPHLEFQEGETIMLRCHSWKDKPLVKVTFFQNGKSQKFSHLDPTFSIPQANHSHSGDYHCTGNIGYTLFSSKPVTITVQVPSMGSSSPMGI
>10FJ_E2 len=238 copies=1 longest=1
MGWSCIILFLVATATGAHSQIQLVQSGPELKKPGETVKISCKASGYTFTNYGMNWVKQAPGKGLKWMGWLNTYTGESIYPDDFKGRFAFSSETSASTAYLQINNLKNEDMATYFCARGDYGYDDPLDYWGQGTSVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVV
>10FJ_E3 len=238 copies=1 longest=0
MGWSCIILFLVATATGVHSDIVMTQAAPSVPVTPGESVSISCRSSKSLLHTNGNTYLHWFLQRPGQSPQLLIYRMSVLASGVPDRFSGSGSGTAFTLSISRVEAEDVGVFYCMQHLEYPLTFGAGTKLELKRTVDAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>10OY_E1 len=344 copies=2 longest=1
MVDRGPLLTSAIIFYLAIGAAIFEVLEEPHWKEAKKNYYTQKLHLLKEFPCLSQEGLDKILQVVSDAADQGVAITGNQTFNNWNWPNAMIFAATVITTIGYGNVAPKTPAGRLFCVFYGLFGVPLCLTWISALGKFFGGRAKRLGQFLTRRGVSLRKAQITCTAIFIVWGVLVHLVIPPFVFMVTEEWNYIEGLYYSFITISTIGFGDFVAGVNPSANYHALYRYFVELWIYLGLAWLSLFVNWKVSMFVEVHKAIKKRRRRRKESFESSPHSRKALQMAGSTASKDVNIFSFLSKKEETYNDLIKQIGKKAMKTSGGGERVPGPGHGLGPQGDRSNSLEVLFQ
>10OY_E2 len=210 copies=1 longest=0
HHHHHHHDYDIPTTENLYFQGSTFSKLREQLGPVTQEFWDNLEKETEGLRQEMSKDLEEVKAKVQPYLDDFQKKWQEEMELYRQKVEPLRAELQEGARQKLHELQEKLSPLGEEMRDRARAHVDALRTHLAPYSDELRQRLAARLEALKENGGARLAEYHAKATEHLSTLSEKAKPALEDLRQGLLPVLESFKVSFLSALEEYTKKLNTQ
>10TL_E1 len=388 copies=2 longest=1
MDSSAAPTNASNCTDALAYSSCSPAPSPGSWVNLSHLDGNLSDPCGPNRTDLGGRDSLCPPTGSPSMITAITIMALYSIVCVVGLFGNFLVMYVIVRYTKMKTATNIYIFNLALADALATSTLPFQSVNYLMGTWPFGTILCKIVISIDYYNMFTSIWTLCTMSVDRYIAVCHPVKALDFRTPRNAKIINVCNWILSSAIGLPVMFMATTKYRQGSIDCTLTFSHPTWYWENLLKICVFIFAFIMPVLIITVCYGLMILRLKSVRMLSGSKEKDRNLRRITRMVLVVVAVFIVCWTPIHIYVIIKALVTIPETTFQTVSWHFCIALGYTNSCLNPVLYAFLDENFKRCFREFCIPTSSNIEQQNSTRIRQNTRDHPSTANTVDRTNHQ
>10TL_E2 len=5 copies=2 longest=0
YAGFX
>10TM_E1 len=388 copies=2 longest=1
MDSSAAPTNASNCTDALAYSSCSPAPSPGSWVNLSHLDGNLSDPCGPNRTDLGGRDSLCPPTGSPSMITAITIMALYSIVCVVGLFGNFLVMYVIVRYTKMKTATNIYIFNLALADALATSTLPFQSVNYLMGTWPFGTILCKIVISIDYYNMFTSIWTLCTMSVDRYIAVCHPVKALDFRTPRNAKIINVCNWILSSAIGLPVMFMATTKYRQGSIDCTLTFSHPTWYWENLLKICVFIFAFIMPVLIITVCYGLMILRLKSVRMLSGSKEKDRNLRRITRMVLVVVAVFIVCWTPIHIYVIIKALVTIPETTFQTVSWHFCIALGYTNSCLNPVLYAFLDENFKRCFREFCIPTSSNIEQQNSTRIRQNTRDHPSTANTVDRTNHQ
>10TM_E2 len=5 copies=2 longest=0
YAGFX
>10VM_E1 len=250 copies=1 longest=0
MSGDREDVERLLRSVEWAIKAGDPYSARILVELAREDAEKIGDERLRREVEELLRELEELEEELARLPKLPPDRTHSLRYFRLGVSDPIHGVPEFISVGYVDSHPITTYDSVTRQKEPRAPWMAENLAPDHWERYTQLLRGWQQMFKVELKRLQRHYNHSGSHTYQRMIGCELLEDGSTTGFLQYAYDGQDFLIFNKDTLSWLAVDNVAHTIKQAWEANQHELLYQKNWLEEECIAWLKRFLEYGKDTLQ
>10VM_E2 len=204 copies=1 longest=0
MGQNIDQPTEMTATEGAIVQINCTYQTSGFNGLFWYQQHAGEAPTFLSYNVLDGLEEKGRFSSFLSRSKGYSYLLLKELQMKDSASYLCAVKDSNYQLIWGAGTKLIIKPDIQNPDPAVYQLRDSKSSDKSVCLFTDFDSQTNVSQSKDSDVYITDKCVLDMRSMDFKSNSAVAWSNKSDFACANAFNNSIIPEDTFFPSPESS
>10VM_E3 len=274 copies=1 longest=1
MAGVTQTPKFQVLKTGQSMTLQCAQDMNHNSMYWYRQDPGMGLRLIYYSASEGTTDKGEVPNGYNVSRLNKREFSLRLESAAPSQTSVYFCASSVWTGEGSGELFFGEGSRLTVLEDLKNVFPPEVAVFEPSEAEISHTQKATLVCLATGFYPDHVELSWWVNGKEVHSGVCTDPQPLKEQPALNDSRYALSSRLRVSATFWQNPRNHFRCQVQFYGLSENDEWTQDRAKPVTQIVSAEAWGRADLEGGLEVLFQGPGGGLNDIFEAQKIEWHE
>10XW_E1 len=241 copies=1 longest=1
MGYQLPNGSSVQIGSVLGSGIAVTAATNAAASISDLTQGCVITCAASHGLVVGDVVMFTKTPWVRALNRAFIVGKVDTNDVTLARFDTEDVTKYPTGAFSVGTPGEVVKVSGFIDFPFITNVAVSGGDQQTTTFQPLQVNTAISLNTTKNPLVQTYTFTHDEEDPIRPILEDLDDTQKTTVIKFTNPAAASGKGEIRIYPAKVSFQKIPSAEVNNVETVQSTLTMQSDMVIYRKDLVEALS
>11CV_E1 len=326 copies=4 longest=1
SVDKRYQVFVSSTFTDLQEERSNVIQSLMEMDCIPAGMELFPSIDEEQWEFIKKIIDDSDYYLLIIGGRYGTVADDGLSFTEKAFDYAVSKGLKVVVLVHENPGALPFDKSEADPLLREKLQKFIEKASTGRLRKTWSSAKDLPGLVALSLNKTIKAYPAIGWVRADKVSSEATLFELNELRKRNEELSEKVRQLENRPDYAQPENLASLNDIYELPYEISTVGWYDEDREEHKSLNIKWAELFSLLSPYLTTEAIDNEIYDILKNHIEKVAPHSYNHTTIDFQELKTIYLQFKALGLLEHNNVNWKLSPKGMELMMQLRTVKSKS
>11HL_E1 len=605 copies=3 longest=1
MTRLTVLALLAGLLASSRASMSDSEVNQEAKPEVKPEVKPETHINLKVSDGSSEIFFKIKKTTPLRRLMEAFAKRQGKEMDSLRFLYDGIRIQADQTPEDLDMEDNDIIEAHRENDDDDKTGTGSQSNAESVQNHTFEVLNNTIRALELILRKLEILYEMILQLHEEVEAIQKAINVLYENQKLIANQFNSAIGKIQDSLSSTASALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLAKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGTNSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFTIDKEKWKEWRKEMENLTKEIKETLEEARKTLKQAEETFKTPSSGGTDDDDKTGSLNDIFEAQKIEWHEQGSHHHHHHHH
>11HM_E1 len=1258 copies=1 longest=1
MGWSCIILFLVATATGVHSAYTNFTRGVYYPDKVFRSSVLHLTQDLFLPFFSNVTWFHAISGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVFIKVCEFQFCNDPFLDVYHKNNKSWMESESGVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPIIGRDFPQGFSALEPLVDLPIGINITRFQTLLALNRSYLTPGDSSSGWTAGAADYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNVTNLCPFHEVFNATRFASVYAWNRTRISNCVADYSVLYNFAPFFAFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDSKHSGNYDYWYRSLRKSKLKPFERDISTEIYQAGNKPCKGKGPNCYFPLESYGFRPTYGVGHQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTKSNKKFLPFQQFGRDIVDTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVSVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEYVNNSYECDIPIGAGICASYQTQTKSRGSAGSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKYFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLFSTPSALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFLTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQLELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIASSGYIPEAPRDGQAYVRKDGEWVLLSTFLEGTKHHHHHHHHGGSGGLNDIFEAQKIEWHE
>11HM_E2 len=240 copies=1 longest=0
MGWSCIILFLVATATGVHSQVQLVQSGAEVKKPGESLRISCKTSGYIFTNFWIGWARQMPGKGLEWMGIIYPDDSDTRYSPSFQGQVTISADKSISTAYLQWSTLKASDTAMYYCVRPDGVNDWFGVDPWGQGTLVTVSSASTKGPSVFPLAPCSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEP
>11HM_E3 len=234 copies=1 longest=0
MGWSCIILFLVATATGVHSETTLTQSPATLSVSPGERATLSCRASQSVTTNLAWYQQKPGRAPRLLIYGASTRATGIPARFSGSGSGTEFTLTISSLQSEDFALYYCQQYNHWPPYTFGQGTKLEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>11IT_E1 len=1258 copies=1 longest=1
MGWSCIILFLVATATGVHSAYTNFTRGVYYPDKVFRSSVLHLTQDLFLPFFSNVTWFHAISGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVFIKVCEFQFCNDPFLDVYHKNNKSWMESESGVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPIIGRDFPQGFSALEPLVDLPIGINITRFQTLLALNRSYLTPGDSSSGWTAGAADYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNVTNLCPFHEVFNATRFASVYAWNRTRISNCVADYSVLYNFAPFFAFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDSKHSGNYDYWYRSLRKSKLKPFERDISTEIYQAGNKPCKGKGPNCYFPLESYGFRPTYGVGHQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTKSNKKFLPFQQFGRDIVDTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVSVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEYVNNSYECDIPIGAGICASYQTQTKSRGSAGSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKYFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLFSTPSALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFLTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQLELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIASSGYIPEAPRDGQAYVRKDGEWVLLSTFLEGTKHHHHHHHHGGSGGLNDIFEAQKIEWHE
>11IT_E2 len=240 copies=1 longest=0
MGWSCIILFLVATATGVHSQVQLQQSGPGLVKPSETLSLTCTVSGGSISSYYWSWIRQPPGKGLEWIGYIYYSGLTNYNPSLKSRITLSVDTSKNQFSLKLNSVTAADTAVYYCATYVRLGGFWSGEADAFDIWGPGTMVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDK
>11IT_E3 len=235 copies=1 longest=0
MGWSCIILFLVATATGVHSQSVLTQPASVSGSPGQSITISCTGTSSDVGNYHLVSWYQQHPGKAPKLIIYEGSKRPSGVSNRFSGSKSGNTASLTISGLQAEDEADYYCCLYADSTTYVFGIGTKVTVLGQPKAAPSVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADSSPVKAGVETTTPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTVAPTECS
>11JF_E1 len=893 copies=2 longest=0
MATEEFIIRIPPYHYIHVLDQNSNVSRVEVGPKTYIRQDNERVLFAPMRMVTVPPRHYCTVANPVSRDAQGLVLFDVTGQVRLRHADLEIRLAQDPFPLYPGEVLEKDITPLQVVLPNTALHLKALLDFEDKDGDKVVAGDEWLFEGPGTYIPRKEVEVVEIIQATIIRQNQALRLRARKECWDRDGKERVTGEEWLVTTVGAYLPAVFEEVLDLVDAVILTEKTALHLRARRNFRDFRGVSRRTGEEWLVTVQDTEAHVPDVHEEVLGVVPITTLGPHNYCVILDPVGPDGKNQLGQKRVVKGEKSFFLQPGEQLEQGIQDVYVLSEQQGLLLRALQPLEEGEDEEKVSHQAGDHWLIRGPLEYVPSAKVEVVEERQAIPLDENEGIYVQDVKTGKVRAVIGSTYMLTQDEVLWEKELPPGVEELLNKGQDPLADRGEKDTAKSLQPLAPRNKTRVVSYRVPHNAAVQVYDYREKRARVVFGPELVSLGPEEQFTVLSLSAGRPKRPHARRALCLLLGPDFFTDVITIETADHARLQLQLAYNWHFEVNDRKDPQETAKLFSVPDFVGDACKAIASRVRGAVASVTFDDFHKNSARIIRTAVFGFETSEAKGPDGMALPRPRDQAVFPQNGLVVSSVDVQSVEPVDQRTRDALQRSVQLAIEITTNSQEAAAKHEAQRLEQEARGRLERQKILDQSEAEKARKELLELEALSMAVESTGTAKAEAESRAEAARIEGEGSVLQAKLKAQALAIETEAELQRVQKVRELELVYARAQLELEVSKAQQLAEVEVKKFKQMTEAIGPSTIRDLAVAGPEMQVKLLQSLGLKSTLITDGSTPINLFNTAFGLLGMGPEGQPLGRRVASGPSPGEGISPQSAQAPQAPGDNHVVPVLR
>11JF_E2 len=1724 copies=1 longest=1
MVMGIFANCIFCLKVKYLPQQQKKKLQTDIKENGGKFSFSLNPQCTHIILDNADVLSQYQLNSIQKNHVHIANPDFIWKSIREKRLLDVKNYDPYKPLDITPPPDQKASSSEVKTEGLCPDSATEEEDTVELTEFGMQNVEIPHLPQDFEVAKYNTLEKVGMEGGQEAVVVELQCSRDSRDCPFLISSHFLLDDGMETRRQFAIKKTSEDASEYFENYIEELKKQGFLLREHFTPEATQLASEQLQALLLEEVMNSSTLSQEVSDLVEMIWAEALGHLEHMLLKPVNRISLNDVSKAEGILLLVKAALKNGETAEQLQKMMTEFYRLIPHKGTMPKEVNLGLLAKKADLCQLIRDMVNVCETNLSKPNPPSLAKYRALRCKIEHVEQNTEEFLRVRKEVLQNHHSKSPVDVLQIFRVGRVNETTEFLSKLGNVRPLLHGSPVQNIVGILCRGLLLPKVVEDRGVQRTDVGNLGSGIYFSDSLSTSIKYSHPGETDGTRLLLICDVALGKCMDLHEKDFSLTEAPPGYDSVHGVSQTASVTTDFEDDEFVVYKTNQVKMKYIIKFSMPGDQIKDFHPSDHTELEEYRPEFSNFSKVEDYQLPDAKTSSSTKAGLQDASGNLVPLEDVHIKGRIIDTVAQVIVFQTYTNKSHVPIEAKYIFPLDDKAAVCGFEAFINGKHIVGEIKEKEEAQQEYLEAVTQGHGAYLMSQDAPDVFTVSVGNLPPKAKVLIKITYITELSILGTVGVFFMPATVAPWQQDKALNENLQDTVEKICIKEIGTKQSFSLTMSIEMPYVIEFIFSDTHELKQKRTDCKAVISTMEGSSLDSSGFSLHIGLSAAYLPRMWVEKHPEKESEACMLVFQPDLDVDLPDLASESEVIICLDCSSSMEGVTFLQAKQIALHALSLVGEKQKVNIIQFGTGYKELFSYPKHITSNTMAAEFIMSATPTMGNTDFWKTLRYLSLLYPARGSRNILLVSDGHLQDESLTLQLVKRSRPHTRLFACGIGSTANRHVLRILSQCGAGVFEYFNAKSKHSWRKQIEDQMTRLCSPSCHSVSVKWQQLNPDVPEALQAPAQVPSLFLNDRLLVYGFIPHCTQATLCALIQEKEFRTMVSTTELQKTTGTMIHKLAARALIRDYEDGILHENETSHEMKKQTLKSLIIKLSKENSLITQFTSFVAVEKRDENESPFPDIPKVSELIAKEDVDFLPYMSWQGEPQEAVRNQSLLASSEWPELRLSKRKHRKIPFSKRKMELSQPEVSEDFEEDGLGVLPAFTSNLERGGVEKLLDLSWTESCKPTATEPLFKKVSPWETSTSSFFPILAPAVGSYLPPTARAHSPASLSFASYRQVASFGSAAPPRQFDASQFSQGPVPGTCADWIPQSASCPTGPPQNPPSSPYCGIVFSGSSLSSAQSAPLQHPGGFTTRPSAGTFPELDSPQLHFSLPTDPDPIRGFGSYHPSASSPFHFQPSAASLTANLRLPMASALPEALCSQSRTTPVDLCLLEESVGSLEGSRCPVFAFQSSDTESDELSEVLQDSCFLQIKCDTKDDSILCFLEVKEEDEIVCIQHWQDAVPWTELLSLQTEDGFWKLTPELGLILNLNTNGLHSFLKQKGIQSLGVKGRECLLDLIATMLVLQFIRTRLEKEGIVFKSLMKMDDASISRNIPWAFEAIKQASEWVRRTEGQYPSICPRLELGNDWDSATKQLLGLQPISTVSPLHRVLHYSQG
>11KM_E1 len=676 copies=1 longest=1
MHHHHHHENLYFQSMTKNPVDHTLIIEPPVRLHGDPTVPKPNIASLDEYKRMYEESINDPSTFWGNMARDMMTWDKQFSTVVQGSIDKADSAWFADGAISPCYNLVDRHAIARPDAVALIYEADEPNQGRYITYRELLASVSQCAGALQSMGVGMGDRVAIYMPMIPETIIAMLAIVRLGAIHSVIFAGFSAESVADRVNDSECKVIITADESHRGGKRIPLKGVVNKALTECPTIKKVLVFQRSAEPTASMVEGRDVWWHDIIPKFPRYCPPAVVNPEHPLFLLYTSGSTGKPKGVVHCTGGYLLGAAATCKYVFDLHPTDRMGCAGDVGWITGHTYIVYGPLMLGAATLVFESTPAYPDYSRYWSVVERHRLTQWYIAPTAIRLLQRAGNEFVKHDRSSLRVLGSVGEPIAPESFMWYYEVVGEKRCAVADTYWQTETGSHIVTSLGPVTPMKPGSATLPFFGIDAVIIDPLTGKIIEGNDVEGVLAIRSPWPSAARTVWRGHDRYIDTYLKPYPGFYFTGDGATRDKDGYIWIRGRVDDVVNISGHRLSTAEIEAALLSHDAVAESAVVGVHDELTGQAVNAFILLKPGYEATVELEKELIMAVRSTIGPFASPRKLIFSDLPKTRSGKIMRRILRKILAGEVDQIGDLSTLADPKVVEHIIHAVHYAHQKKP
>11OR_E1 len=217 copies=1 longest=1
EVQLVESGGGVVQPGRSLRLSCAASGFTFSNYGMHWVRQAPGKGLEWVAVYWYDGGNKFYADSVKGRFTISRDNSKNTLYLQMNSLRVEDTAVYYCARDTAPPDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPK
>11OR_E2 len=211 copies=1 longest=0
MTQSPGTLSLSPGERATLSCRASQSISSSFLAWYQQKPGQAPRLLIYGASSRATGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQQYGTSPRLTFGGGTKVDIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRG
>11OR_E3 len=120 copies=1 longest=0
EVQLQESGGGLVQPGGSLRLSCAASGRTISRYAMSWFRQAPGKEREFVATARRSGDGAFYADSVQGRFTVSRDDAKNTVYLQMNSLKPEDTAVYYCAIDSDTFYSGSYDYWGQGTQVTVS
>11OU_E1 len=120 copies=1 longest=0
EVQLQESGGGLVQPGGSLRLSCAASGRTISRYAMSWFRQAPGKEREFVATARRSGDGAFYADSVQGRFTVSRDDAKNTVYLQMNSLKPEDTAVYYCAIDSDTFYSGSYDYWGQGTQVTVS
>11OU_E2 len=214 copies=1 longest=0
IVLTQSPGTLSLSPGERATLSCRASQSVSSSYLAWYQQKPGQAPRLLIYGASSRATGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQQYGSSPRLTFGGGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGE
>11OU_E3 len=219 copies=1 longest=1
QVQLVESGGGVVQPGRSLRLSCAASGFTFSSYGMHWVRQAPGKGLEWVAVIWYDGSNKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDTAPPDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSC
>11TD_E1 len=475 copies=1 longest=1
MNPTDIADTTLDESIYSNYYLYESIPKPCTKEGIKAFGELFLPPLYSLVFVFGLLGNSVVVLVLFKYKRLRSMTDVYLLNLAISDLLFVFSLPFWGYYAADQWVFGLGLCKMISWMYLVGFYSGIFFVMLMSIDRYLAIVHAVFSLRARTLTYGVITSLATWSVAVFASLPGFLFSTCYTERNHTYCKTKYSLNSTTWKVLSSLEINILGLVIPLGIMLFCYSMIIRTLQHARRQLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLERARSTLNEKNKAVKMIFAVVVLFLGFWTPYNIVLFLETLVELEVLQDCTFERYLDYAIQATETLAFVHCCLNPIIYFFLGEKFRKYILQLFKTCRGLFVLCQYCGLLQIYSADTPSSSYTQSTMDHDLHDAL
>11TD_E2 len=229 copies=1 longest=0
EISEVQLVESGGGLVQPGGSLRLSCAASGFNVVDFSLHWVRQAPGKGLEWVAYISSSSGSTSYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARWGYWPGEPWWKAFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKS
>11TD_E3 len=214 copies=1 longest=0
SDIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQYLYYSLVTFGQGTKVEIKRTVAAPSVFIFPPSDSQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRG
>11TF_E1 len=475 copies=1 longest=1
MNPTDIADTTLDESIYSNYYLYESIPKPCTKEGIKAFGELFLPPLYSLVFVFGLLGNSVVVLVLFKYKRLRSMTDVYLLNLAISDLLFVFSLPFWGYYAADQWVFGLGLCKMISWMYLVGFYSGIFFVMLMSIDRYLAIVHAVFSLRARTLTYGVITSLATWSVAVFASLPGFLFSTCYTERNHTYCKTKYSLNSTTWKVLSSLEINILGLVIPLGIMLFCYSMIIRTLQHARRQLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLERARSTLNEKNKAVKMIFAVVVLFLGFWTPYNIVLFLETLVELEVLQDCTFERYLDYAIQATETLAFVHCCLNPIIYFFLGEKFRKYILQLFKTCRGLFVLCQYCGLLQIYSADTPSSSYTQSTMDHDLHDAL
>11TF_E2 len=229 copies=1 longest=0
EISEVQLVESGGGLVQPGGSLRLSCAASGFNVVDFSLHWVRQAPGKGLEWVAYISSSSGSTSYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARWGYWPGEPWWKAFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKS
>11TF_E3 len=214 copies=1 longest=0
SDIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQYLYYSLVTFGQGTKVEIKRTVAAPSVFIFPPSDSQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRG
>11TH_E1 len=229 copies=1 longest=0
EISEVQLVESGGGLVQPGGSLRLSCAASGFNVVDFSLHWVRQAPGKGLEWVAYISSSSGSTSYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARWGYWPGEPWWKAFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKS
>11TH_E2 len=214 copies=1 longest=0
SDIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQYLYYSLVTFGQGTKVEIKRTVAAPSVFIFPPSDSQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRG
>11TH_E3 len=475 copies=1 longest=1
MNPTDIADTTLDESIYSNYYLYESIPKPCTKEGIKAFGELFLPPLYSLVFVFGLLGNSVVVLVLFKYKRLRSMTDVYLLNLAISDLLFVFSLPFWGYYAADQWVFGLGLCKMISWMYLVGFYSGIFFVMLMSIDRYLAIVHAVFSLRARTLTYGVITSLATWSVAVFASLPGFLFSTCYTERNHTYCKTKYSLNSTTWKVLSSLEINILGLVIPLGIMLFCYSMIIRTLQHARRQLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLERARSTLNEKNKAVKMIFAVVVLFLGFWTPYNIVLFLETLVELEVLQDCTFERYLDYAIQATETLAFVHCCLNPIIYFFLGEKFRKYILQLFKTCRGLFVLCQYCGLLQIYSADTPSSSYTQSTMDHDLHDAL
>11TK_E1 len=475 copies=1 longest=1
MNPTDIADTTLDESIYSNYYLYESIPKPCTKEGIKAFGELFLPPLYSLVFVFGLLGNSVVVLVLFKYKRLRSMTDVYLLNLAISDLLFVFSLPFWGYYAADQWVFGLGLCKMISWMYLVGFYSGIFFVMLMSIDRYLAIVHAVFSLRARTLTYGVITSLATWSVAVFASLPGFLFSTCYTERNHTYCKTKYSLNSTTWKVLSSLEINILGLVIPLGIMLFCYSMIIRTLQHARRQLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLERARSTLNEKNKAVKMIFAVVVLFLGFWTPYNIVLFLETLVELEVLQDCTFERYLDYAIQATETLAFVHCCLNPIIYFFLGEKFRKYILQLFKTCRGLFVLCQYCGLLQIYSADTPSSSYTQSTMDHDLHDAL
>11TK_E2 len=229 copies=1 longest=0
EISEVQLVESGGGLVQPGGSLRLSCAASGFNVVDFSLHWVRQAPGKGLEWVAYISSSSGSTSYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARWGYWPGEPWWKAFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKS
>11TK_E3 len=214 copies=1 longest=0
SDIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQYLYYSLVTFGQGTKVEIKRTVAAPSVFIFPPSDSQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRG
>11TL_E1 len=475 copies=1 longest=1
MNPTDIADTTLDESIYSNYYLYESIPKPCTKEGIKAFGELFLPPLYSLVFVFGLLGNSVVVLVLFKYKRLRSMTDVYLLNLAISDLLFVFSLPFWGYYAADQWVFGLGLCKMISWMYLVGFYSGIFFVMLMSIDRYLAIVHAVFSLRARTLTYGVITSLATWSVAVFASLPGFLFSTCYTERNHTYCKTKYSLNSTTWKVLSSLEINILGLVIPLGIMLFCYSMIIRTLQHARRQLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLERARSTLNEKNKAVKMIFAVVVLFLGFWTPYNIVLFLETLVELEVLQDCTFERYLDYAIQATETLAFVHCCLNPIIYFFLGEKFRKYILQLFKTCRGLFVLCQYCGLLQIYSADTPSSSYTQSTMDHDLHDAL
>11TL_E2 len=229 copies=1 longest=0
EISEVQLVESGGGLVQPGGSLRLSCAASGFNVVDFSLHWVRQAPGKGLEWVAYISSSSGSTSYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARWGYWPGEPWWKAFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKS
>11TL_E3 len=214 copies=1 longest=0
SDIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQYLYYSLVTFGQGTKVEIKRTVAAPSVFIFPPSDSQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRG
>11WN_E1 len=676 copies=1 longest=1
MHHHHHHENLYFQSMTKNPVDHTLIIEPPVRLHGDPTVPKPNIASLDEYKRMYEESINDPSTFWGNMARDMMTWDKQFSTVVQGSIDKADSAWFADGAISPCYNLVDRHAIARPDAVALIYEADEPNQGRYITYRELLASVSQCAGALQSMGVGMGDRVAIYMPMIPETIIAMLAIVRLGAIHSVIFAGFSAESVADRVNDSECKVIITADESHRGGKRIPLKGVVNKALTECPTIKKVLVFQRSAEPTASMVEGRDVWWHDIIPKFPRYCPPAVVNPEHPLFLLYTSGSTGKPKGVVHCTGGYLLGAAATCKYVFDLHPTDRMGCAGDVGWITGHTYIVYGPLMLGAATLVFESTPAYPDYSRYWSVVERHRLTQWYIAPTAIRLLQRAGNEFVKHDRSSLRVLGSVGEPIAPESFMWYYEVVGEKRCAVADTYWQTETGSHIVTSLGPVTPMKPGSATLPFFGIDAVIIDPLTGKIIEGNDVEGVLAIRSPWPSAARTVWRGHDRYIDTYLKPYPGFYFTGDGATRDKDGYIWIRGRVDDVVNISGHRLSTAEIEAALLSHDAVAESAVVGVHDELTGQAVNAFILLKPGYEATVELEKELIMAVRSTIGPFASPRKLIFSDLPKTRSGKIMRRILRKILAGEVDQIGDLSTLADPKVVEHIIHAVHYAHQKKP
>11WO_E1 len=676 copies=1 longest=1
MHHHHHHENLYFQSMTKNPVDHTLIIEPPVRLHGDPTVPKPNIASLDEYKRMYEESINDPSTFWGNMARDMMTWDKQFSTVVQGSIDKADSAWFADGAISPCYNLVDRHAIARPDAVALIYEADEPNQGRYITYRELLASVSQCAGALQSMGVGMGDRVAIYMPMIPETIIAMLAIVRLGAIHSVIFAGFSAESVADRVNDSECKVIITADESHRGGKRIPLKGVVNKALTECPTIKKVLVFQRSAEPTASMVEGRDVWWHDIIPKFPRYCPPAVVNPEHPLFLLYTSGSTGKPKGVVHCTGGYLLGAAATCKYVFDLHPTDRMGCAGDVGWITGHTYIVYGPLMLGAATLVFESTPAYPDYSRYWSVVERHRLTQWYIAPTAIRLLQRAGNEFVKHDRSSLRVLGSVGEPIAPESFMWYYEVVGEKRCAVADTYWQTETGSHIVTSLGPVTPMKPGSATLPFFGIDAVIIDPLTGKIIEGNDVEGVLAIRSPWPSAARTVWRGHDRYIDTYLKPYPGFYFTGDGATRDKDGYIWIRGRVDDVVNISGHRLSTAEIEAALLSHDAVAESAVVGVHDELTGQAVNAFILLKPGYEATVELEKELIMAVRSTIGPFASPRKLIFSDLPKTRSGKIMRRILRKILAGEVDQIGDLSTLADPKVVEHIIHAVHYAHQKKP
>11WP_E1 len=676 copies=1 longest=1
MHHHHHHENLYFQSMTKNPVDHTLIIEPPVRLHGDPTVPKPNIASLDEYKRMYEESINDPSTFWGNMARDMMTWDKQFSTVVQGSIDKADSAWFADGAISPCYNLVDRHAIARPDAVALIYEADEPNQGRYITYRELLASVSQCAGALQSMGVGMGDRVAIYMPMIPETIIAMLAIVRLGAIHSVIFAGFSAESVADRVNDSECKVIITADESHRGGKRIPLKGVVNKALTECPTIKKVLVFQRSAEPTASMVEGRDVWWHDIIPKFPRYCPPAVVNPEHPLFLLYTSGSTGKPKGVVHCTGGYLLGAAATCKYVFDLHPTDRMGCAGDVGWITGHTYIVYGPLMLGAATLVFESTPAYPDYSRYWSVVERHRLTQWYIAPTAIRLLQRAGNEFVKHDRSSLRVLGSVGEPIAPESFMWYYEVVGEKRCAVADTYWQTETGSHIVTSLGPVTPMKPGSATLPFFGIDAVIIDPLTGKIIEGNDVEGVLAIRSPWPSAARTVWRGHDRYIDTYLKPYPGFYFTGDGATRDKDGYIWIRGRVDDVVNISGHRLSTAEIEAALLSHDAVAESAVVGVHDELTGQAVNAFILLKPGYEATVELEKELIMAVRSTIGPFASPRKLIFSDLPKTRSGKIMRRILRKILAGEVDQIGDLSTLADPKVVEHIIHAVHYAHQKKP
>11WQ_E1 len=676 copies=1 longest=1
MHHHHHHENLYFQSMTKNPVDHTLIIEPPVRLHGDPTVPKPNIASLDEYKRMYEESINDPSTFWGNMARDMMTWDKQFSTVVQGSIDKADSAWFADGAISPCYNLVDRHAIARPDAVALIYEADEPNQGRYITYRELLASVSQCAGALQSMGVGMGDRVAIYMPMIPETIIAMLAIVRLGAIHSVIFAGFSAESVADRVNDSECKVIITADESHRGGKRIPLKGVVNKALTECPTIKKVLVFQRSAEPTASMVEGRDVWWHDIIPKFPRYCPPAVVNPEHPLFLLYTSGSTGKPKGVVHCTGGYLLGAAATCKYVFDLHPTDRMGCAGDVGWITGHTYIVYGPLMLGAATLVFESTPAYPDYSRYWSVVERHRLTQWYIAPTAIRLLQRAGNEFVKHDRSSLRVLGSVGEPIAPESFMWYYEVVGEKRCAVADTYWQTETGSHIVTSLGPVTPMKPGSATLPFFGIDAVIIDPLTGKIIEGNDVEGVLAIRSPWPSAARTVWRGHDRYIDTYLKPYPGFYFTGDGATRDKDGYIWIRGRVDDVVNISGHRLSTAEIEAALLSHDAVAESAVVGVHDELTGQAVNAFILLKPGYEATVELEKELIMAVRSTIGPFASPRKLIFSDLPKTRSGKIMRRILRKILAGEVDQIGDLSTLADPKVVEHIIHAVHYAHQKKP
>11XX_E1 len=553 copies=1 longest=1
MSESKEMVRGTFLITISILITKVLGVLFIIPFNYLIGGQENMAPFTYAYAPYNIAIAVATAGVPLAASKYVAKYNAIGAYKVSQKFYKSSFIVMSITGVLGFLVLYFLAPYISELTLARNIHDKNGWSVDDITWIIRIISMVVIFIPVLATWRGIFQGYKSMGPTAVSEVTEQIARVIFILIGSYLVLNVFDGSILLANGIATFAAAVGAIIGIFTLWYYWRKRKHNIDRMVESDYTDIDVSYGKMYKEIIAYSIPFVIVSLNYPLFNLVDQFTHNGALSLVGIPSQLQDIFFNMLNMSTNKIVMIPTSLSAGFAVSLIPYITKTFAEGRLHEMHHQIRTSIGVLMFITVPASIGIMALAQPLFTVFYGYDPIVLGHDPNHDGSRLLFYYAPVAILISLLSVTASMLQGIDKQKLTVYVILASVVIKLALNYPLIMLFHTPGAILSTSIALLFAIGCNFYILKKYAKFKFSYSWIHFAKIFLYSFIMMLGVELVFFLANLFLEPTKLGYLIIIILGVTVGILIYGTITIKTRLADEFLGEIPEKLRRRVRFLR
>11ZV_E1 len=175 copies=1 longest=0
MGILPSPGMPALLSLVSLLSVLLMGCVAETGTPELEVVQLNISAHMDFGEARLDSVTINGNTSYCVTKPYFRLETNFMCTGCTMNLRTDTCSFDLSAVNNGMSFSQFCLSTESGACEMKIIVTYVWNYLLRQRLYVTAVEGQTHTGGGSGLNDIFEAQKIEWHEGGSHHHHHHHH
>11ZV_E2 len=176 copies=1 longest=1
MHHHHHHHHGGSLVPRGSGGSGLNDIFEAQKIEWHEGGSHMLTREEVQAIIDGHVIEVKYDSVEEFLEKYKGKISEETIELIEKLIELTSKDPKLSRLPLIFMQMLDMFIEIMKDLKDRGITSEEDRKLLYDVFKERFEQWFDSFWPGEEEIKEIMLKILDLLYNEDYEKIYEAEK
>12IY_E1 len=374 copies=1 longest=1
DYKDDDDENLYFQGMEECWVTEIANGSKDGLDSNPMKDYMILSGPQKTAVAVLCTLLGLLSALENVAVLYLILSSHQLRRKPSYLFIGSLAGADFLASVVFACSFVNFHVFHGVDSKAVFLLKIGSVTMTFTASVGSLLLTAIDRYLCLRYPPSYKALLTRGRALVTLGIMWVLSALVSYLPLMGWTCCPRPCSELFPLIPNDYLLSWLLFIAFLFSGIIYTYGHVLWKAHQHVASLSGHQDRQVPGMARMRLDVRLAKTLGLVLAVLLICWFPVLALMAHSLATTLSDQVKKAFAFCSMLCLINSMVNPVIYALRSGEIRSSAHHCLAHWKKCVRGLGSEAKEEAPRSSVTETEADGKITPWPDSRDLDLSDC
>12OY_E1 len=304 copies=1 longest=1
MADAASQVLLGSGLTILSQPLMYVKVLIQVGYEPLPPTIGRNIFGRQVCQLPGLFSYAQHIASIDGRRGLFTGLTPRLCSGVLGTVVHGKVLQHYQESDKGEELGPGNVQKEVSSSFDHVIKETTREMIARSAATLITHPFHVITLRSMVQFIGRESKYCGLCDSIITIYREEGILGFFAGLVPRLLGDILSLWLCNSVYYLMKEAFVKNNILADDVPCHLVSQAVAGFFASMLTYPFVLVSNLMAVNNCGLAGGCPPYSPIYTSWIDCWCMLQKEGNMSRGNSLFFRKVPFGKTYCCDLKMLI
>12OY_E2 len=127 copies=1 longest=0
GQRQLVESGGGLVQPGGSLRLSCAASGRTSSTYTMGWFRQAPGKEREFVAAISWTGTPYYADSVKGRFTISRDNAKNTVYLQMNSLEPEDTAVYYCAAARPGLFIFVSDYARTAKYDYWGKGTPVTV
>12OZ_E1 len=304 copies=1 longest=1
MADAASQVLLGSGLTILSQPLMYVKVLIQVGYEPLPPTIGRNIFGRQVCQLPGLFSYAQHIASIDGRRGLFTGLTPRLCSGVLGTVVHGKVLQHYQESDKGEELGPGNVQKEVSSSFDHVIKETTREMIARSAATLITHPFHVITLRSMVQFIGRESKYCGLCDSIITIYREEGILGFFAGLVPRLLGDILSLWLCNSVYYLMKEAFVKNNILADDVPCHLVSQAVAGFFASMLTYPFVLVSNLMAVNNCGLAGGCPPYSPIYTSWIDCWCMLQKEGNMSRGNSLNNRKVPFGKTYCCDLKMLI
>12OZ_E2 len=127 copies=1 longest=0
GQRQLVESGGGLVQPGGSLRLSCAASGRTSSTYTMGWFRQAPGKEREFVAAISWTGTPYYADSVKGRFTISRDNAKNTVYLQMNSLEPEDTAVYYCAAARPGLFIFVSDYARTAKYDYWGKGTPVTV
>12PB_E1 len=127 copies=1 longest=0
GQRQLVESGGGLVQPGGSLRLSCAASGRTSSTYTMGWFRQAPGKEREFVAAISWTGTPYYADSVKGRFTISRDNAKNTVYLQMNSLEPEDTAVYYCAAARPGLFIFVSDYARTAKYDYWGKGTPVTV
>12PB_E2 len=304 copies=1 longest=1
MADAASQVLLGSGLTILSQPLMYVEVLIQVGYEPLPPTIGRNIFGRQVCQLPGLFSYAQHIASIDGRRGLFTGLTPRLCSGVLGTVVHGKVLQHYQESDKGEELGPGNVQKEVSSSFDHVIKETTREMIARSAATLITHPFHVITLRSMVQFIGRESKYCGLCDSIITIYREEGILGFFAGLVPRLLGDILSLWLCNSVYYLMKEAFVKNNILADDVPCHLVSQAVAGFFASMLTAPFDLVSNLMAVNNCGLAGGCPPYSPIYTSWIDCWCMLQKEGNMSRGNSLFFRKVPFGKTYCCDLKMLI
>12QJ_E1 len=394 copies=1 longest=1
SSGNKIHFINVQEGGSDAIILESNGHFAMVDTGEDYDFPDGSDSRYPWREGIETSYKHVLTDRVFRRLKELGVQKLDFILVTHTHSDHIGNVDELLSTYPVDRVYLKKYSDNRITNSERLWDNLYGYDKVLQTASEKGVSVIQNITQGDAHFQFGDMDIQLYNYENETDSSGELKKIWDDNSNSLISVVKVNGKKIYLGGDLDNVHGAEDKYGPLIGKVDLMKFNHHHDTNKSNTKDFIKNLSPSLIVQTSDSLPWKNGVDSEYVNWLKERGIERINAASKDYDATVFDIRQDGLVNISTSYKPIPSFQAGWHKSSYGNWWYQAPDSTGEYAVGWNEIKGEWYYFNQTGILLQNQWKKWNNRWFYLTDSGVAAKNWKKIAGIWYYFNKENQMEI
>12QJ_E2 len=125 copies=1 longest=0
QVQLQESGPRLVKVSETLSLTCSLSGHSISSGHHWAWIRQSPGRGLEWIATVDHNGRTYYNPSLRGRVILSKDTTKNSFSVQLTSVTAADTAVYYCARREAPSPGTGWYPPFGQWGQGALVTVST
>12QJ_E3 len=111 copies=1 longest=0
QSVLTQPPSLSGAPGQTVTISCTGSSSNIGAGYDVHWYRQLPGTAPTLLISVFDNRPLGVPGRISGSKSGTSASLAITGLQAEDEADYYCQSHDNMLSGSVFGGGTKLTVL
>12RP_E1 len=1271 copies=3 longest=1
MFVFLVLLLLVSSQCVNLTTTTQLPLAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHVISGTNGTKRFDNPVLPFNDGVYFASIEKSNITRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFNNKSWMESESRVYSSAKNCTFEYVFQPFLMDLEGKQGNFTNLREFVFKNIDGYFKIYSKHTPIIVRDLPQGFSALEPLVDLPIGINITRFQTLLHRSYLTSGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESVVRFPNITNLCPFYEVFNATRFPSVYAWNRKRISNCVADYSVLYNFAPFFTFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVISWNSNRLDSKADGNYNYWYRLFRKSKLKPFERDISTEIYQAGNNPCNGVKGFNCYFPLQSYSFRPTYGVGYQPYRVVVLSFELLHAPATVCGPKNSTNLVKNKCVNFNFNGLTGTGVLTDSNKKFLPFQQFGRDIADTTDAVRDPQTLDILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVPVAIRADQLTPTWRVYSTGSKGFQTRAGCLIGAKYVNNSYECDIPIGAGICASYQTQTRSRGSASSVDSQSIIAYTMSLGAENLVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPITYFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLFSTPSALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSRDVDLGDISGINASVVNIQKEIERLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGRSLEVLFQGPGHHHHHHHHSAWSHPQFEKGGGSGGGGSGGSAWSHPQFEK
>13BD_E1 len=1271 copies=2 longest=1
MFVFLVLLLLVSSQCVNLTTTTQLPLAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHVISGTNGTKRFDNPVLPFNDGVYFASIEKSNITRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFNNKSWMESESRVYSSAKNCTFEYVFQPFLMDLEGKQGNFTNLREFVFKNIDGYFKIYSKHTPIIVRDLPQGFSALEPLVDLPIGINITRFQTLLHRSYLTSGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESVVRFPNITNLCPFYEVFNATRFPSVYAWNRKRISNCVADYSVLYNFAPFFTFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVISWNSNRLDSKADGNYNYWYRLFRKSKLKPFERDISTEIYQAGNNPCNGVKGFNCYFPLQSYSFRPTYGVGYQPYRVVVLSFELLHAPATVCGPKQSTNLVKNKCVNFNFNGLTGTGVLTDSNKKFLPFQQFGRDIADTTDAVRDPQTLDILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVPVAIRADQLTPTWRVYSTGSKGFQTRAGCLIGAKYVNNSYECDIPIGAGICASYQTQTRSRGSASSVDSQSIIAYTMSLGAENLVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPITYFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICKQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLFSTPSALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSRDVDLGDISGINASVVNIQKEIERLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGRSLEVLFQGPGHHHHHHHHSAWSHPQFEKGGGSGGGGSGGSAWSHPQFEK
>13BF_E1 len=221 copies=1 longest=1
QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYGISWVRQAPGQGLEWMGWISAYNGNTNYAQKFQGRVTMTTDTSTTTGYMELRRLRSDDTAVYYCARDYDLAFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCD
>13BF_E2 len=215 copies=1 longest=0
EIVLTQSPGTLSLSPGERATLSCRASQSVSSYLAWYQQKPGQAPRLLIYDASSRATGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQQYGSSFRYTFGGGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>13BF_E3 len=83 copies=1 longest=0
HHHHHHTEPLALTETDICKLPKDEGTCRDFILKWYYDPNTKSCARFWYGGCGGNENKFGSQKECEKVCAPVLAKPGVISVMGT
>13BR_E1 len=215 copies=1 longest=0
DIVMTQSPLSLPVTPGEPASISCRSSQSIPYNLLDWYLQKSGQSPQLLIYLASNRASGVPDRFSGSGSGTDFTLKISRVEAEDVGFYYCMQGLETPLTFGQGTKLEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>13BR_E2 len=242 copies=1 longest=0
EVQLVESGGGLEQPGGSLRLSCAGSGFTFSSYGMTWVRQAPGKGLEWVSSISSSGYYTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARDLGTLLGSLGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDKTHTGSGSDYKDDDDK
>13BR_E3 len=284 copies=1 longest=1
QENQDGRYSLTYIYTGLSKHVEDVPAFQALGSLNDLQFFRYNSKDRKSQPMGLWRQVEGMEDWKQDSQLQKAREDIFMETLKDIVEYYNDSNGSHVLQGRFGCEIENNRSSGAFWKYYYDGKDYIEFNKEIPAWVPFDPAAQITKQKWEAEPVYVQRAKAYLEEECPATLRKYLKYSKNILDRQDPPSVVVTSHQAPGEKKKLKCLAYDFYPGKIDVHWTRAGEVQEPELRGDVLHNGNGTYQSWVVVAVPPQDTAPYSCHVQHSSLAQPLVVPWEASHHHHHH
>13BS_E1 len=154 copies=1 longest=0
SLSGALCFRMKDSALKVLYLHNNQLLAGGLHAGKVIKGEEISVVPNRWLDASLSPVILGVQGGSQCLSCGVGQEPTLTLEPVNIMELYLGAKESKSFTFYRRDMGLTSSFESAAYPGWFLCTVPEADQPVRLTQLPENGGWNAPITDFYFQQCD
>13BS_E2 len=226 copies=1 longest=1
QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYGISWVRQAPGQGLEWMGWISAYSGNTNYAQKFQGRVTMTTDTSTTTGYMELRRLRSDDTAVYYCARDTLSGAFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDKTHT
>13BS_E3 len=214 copies=1 longest=0
EIVLTQSPGTLSLSPGERATLSCRASQSVGSFLAWYQQKPGQAPRLLIYGASNRATGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQQGSGSPLTFGGGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>13KK_E1 len=102 copies=14 longest=1
MAGQAFRKFLPLFDRVLVERSAAETVTKGGIMLPEKSQGKVLQATVVAVGSGSKGKGGEIQPVSVKVGDKVLLPEYGGTKVVLDDKDYFLFRDGDILGKYVD
>13KW_E1 len=102 copies=7 longest=1
MAGQAFRKFLPLFDRVLVERSAAETVTKGGIMLPEKSQGKVLQATVVAVGSGSKGKGGEIQPVSVKVGDKVLLPEYGGTKVVLDDKDYFLFRDGDILGKYVD
>20ZN_E1 len=620 copies=1 longest=1
MLSAIYTVLAGLLFLPLLVNLCCPYFFQDIGYFLKVAAVGRRVRSYGKRRPARTILRAFLEKARQTPHKPFLLFRDETLTYAQVDRRSNQVARALHDHLGLRQGDCVALLMGNEPAYVWLWLGLVKLGCAMACLNYNIRAKSLLHCFQCCGAKVLLVSPELQAAVEEILPSLKKDDVSIYYVSRTSNTDGIDSFLDKVDEVSTEPIPESWRSEVTFSTPALYIYTSGTTGLPKAAMITHQRIWYGTGLTFVSGLKADDVIYITLPFYHSAALLIGIHGCIVAGATLALRTKFSASQFWDDCRKYNVTVIQYIGELLRYLCNSPQKPNDRDHKVRLALGNGLRGDVWRQFVKRFGDICIYEFYAATEGNIGFMNYARKVGAVGRVNYLQKKIITYDLIKYDVEKDEPVRDENGYCVRVPKGEVGLLVCKITQLTPFNGYAGAKAQTEKKKLRDVFKKGDLYFNSGDLLMVDHENFIYFHDRVGDTFRWKGENVATTEVADTVGLVDFVQEVNVYGVHVPDHEGRIGMASIKMKENHEFDGKKLFQHIADYLPSYARPRFLRIQDTIEITGTFAHRKMTLVEEGFNPAVIKDALYFLDDTAKMYVPMTEDIYNAISAKTLKL
>21AH_E1 len=366 copies=1 longest=1
MDFLNSSDQNLTSEELLNRMPSKILVSLTLSGLALMTTTINSLVIAAIIVTRKLHHPANYLICSLAVTDFLVAVLVMPFSIVYIVRESWIMGQVVCDIWLSVDITCCTCSILHLSAIALDRYRAITDAVEYARKRTPKHAGIMITIVWIISVFISMPPLFWRHQGTSRDDECIIKHDHIVSTIYSTFGAFYIPLALILILYYKIYRAAKTLYHKRQASRIAKEEVNGQVLLESGEKSTKSVSTSYVLEKSLSDPSTDFDKIHSTVRSLRSEFKHEKSWRRQKISGTRERKAATTLGLILGAFVICWLPFFVKELVVNVCDKCKISEEMSNFLAWLGYLNSLINPLIYTIFNEDFKKAFQKLVRCRC
>21AO_E1 len=125 copies=1 longest=0
EVQLVQSGAEMKKPGESLKISCQGSGYDFSIFWIGWVRQVPGKGLEWIGIIFPGDSNTRYNPSFQGQVTISADESINTAYLQWSSLKSSDTAIYYCARHEDNGDYLTPYYVLDVWGQGTTVTVSS
>21AO_E2 len=107 copies=1 longest=0
DIQMTQSPSSQSASIGDTVTITCRASDNINNFLNWYQQKPGKAPKLMIYFASHLQTGVPSRFSGSGSGTDFTLTISSLQPEDLATYYCQQSFRTPSTFGQGTKIEIK
>21AO_E3 len=321 copies=1 longest=1
DSGPIICAGPIHSNKSAGIPHLLGYSEKICQIDRLIHVSSWLRNHSQFQGYVGQRGGRSQVSYYPAENSYSRWSGLLSPCDADWLGMLVVKKAKESDMIVPGPSYKGKVFFERPTFDGYVGWGCGSGKSRTESGELCSSDSGTSSGLLPSDRVLWIGDVACQPMTPIPEETFLELKSFSQSEFPDICKIDGIVFNQCEGESLPQPFDVAWMDVGHSHKIIMREHKTKWVQESSSKDFVCYKEGTGPCSESEEKACKTSGSCRGDMQFCKVAGCEHGEEASEAKCRCSLVHKPGEVVVSYGGTRVRPKCYGFSRMMATLEVN
>21AQ_E1 len=123 copies=1 longest=0
EVQLVESGGGLVKPGGSLRLSCAASGFTFSSYSMNWVRQAPGKGLEWVASISGSSAYIYYADSMKGRFTISRDNAKNSVYLQMNSLGVEDTAVYYCARDPASMLGDIGELNHWGRGIRVTVSS
>21AQ_E2 len=108 copies=1 longest=0
SYVLTQPPSVSVAPGNTARITCGGDNIGSSAVHWYQQKPGQAPVVVLYDDSVRPSRIPERFSGSNSGNTATLSISRVEAGDEADYYCQIWDTITSHRVFGGGTKLTVL
>21AQ_E3 len=321 copies=1 longest=1
DSGPIICAGPIHSNKSAGIPHLLGYSEKICQIDRLIHVSSWLRNHSQFQGYVGQRGGRSQVSYYPAENSYSRWSGLLSPCDADWLGMLVVKKAKESDMIVPGPSYKGKVFFERPTFDGYVGWGCGSGKSRTESGELCSSDSGTSSGLLPSDRVLWIGDVACQPMTPIPEETFLELKSFSQSEFPDICKIDGIVFNQCEGESLPQPFDVAWMDVGHSHKIIMREHKTKWVQESSSKDFVCYKEGTGPCSESEEKACKTSGSCRGDMQFCKVAGCEHGEEASEAKCRCSLVHKPGEVVVSYGGTRVRPKCYGFSRMMATLEVN
>21DI_E1 len=134 copies=9 longest=1
MPALANPRHERFAQELAKGKSASEAYVAAGYEESRSAASRLSTNVNVQARVTELQERAATNVSISREWVLEQLVENARLAKEAGDISPSNQALNLIGKELGMFVERTENVNIEHVVSDDLPTAEEWEAEHGTAH
>21FH_E1 len=527 copies=1 longest=1
LKENKIDWFIFISTFVLIVAICLPLVLYPEKGGELVSQANSFVTSNFGMVFLWAGMGGFAFLTYLFFSEHGKIKFGTKDEKPEFSNFSWGAMVFCAGVASDLVYWGTIEWAYYFTAPPYGIEPKSTDAAEIAAAYGIFHWGPSAWAFFVLPSLPIAYSYYILKVPVLRLSEICRIVIGKHSDGVLGKVIDIIFMFGILGAAGTSLGLGTPLVAAGISKITGIESSTSMTVFVLVLCTALFTWSAYSGLKRGLKLLSDLAIICSVILLTYVLLVGPTEFILKMGTNSVGLLFTNFVRWNLYTDPVSGSGFVESWTVFYWAWWIVYTPFMGLFIAKISRGRTIRQVILGGIAWGTLGCAAYFSILGNYAMHLELTKELSVTGLLEQIGAPATIMEVIGTLPLGSVVVIVFSFIALIFLATTFDSAAYMMASSTTPTLGLNEEPAKWNRLFWALTLFILPGTLMVLGGDLKTLQTASILTAIPFIFVIMLLVVALMKMLKTGEHFNQDLQDDPIKLAEVDSLEERQKPAA
>21FI_E1 len=502 copies=1 longest=1
NKIDWFIFISTFVLIVAICLPLVLYPEKGGELVSQANSFVTSNFGMVFLWAGMGGFAFLTYLFFSEHGKIKFGTKDEKPEFSNFSWGAMVFCAGVASGLVYWGTIEWAYYFTAPPYGIEPKSTDAAEIAAAYGIFHWGPSAWAFFVLPSLPIAYSYYILKVPVLRLSEICRIVIGKHSDGVLGKVIDIIFMFGILGAAGTSLGLGTPLVAAGISKITGIESSTSMTVFVLVLCTALFTWSAYSGLKRGLKLLSDLAIICSVILLTYVLLVGPTEFILKMGTNSVGLLFTNFVRWNLYTDPVSGSGFVESWTVFYWAWWIVYTPFMGLFIAKISRGRTIRQVILGGIAWGTLGCAAYFSILGNYAMHLELTKELSVTGLLEQIGAPATIMEVIGTLPLGSVVVIVFSFIALIFLATTFDSAAYMMASSTTPTLGLNEEPAKWNRLFWALTLFILPGTLMVLGGDLKTLQTASILTAIPFIFVIMLLVVALMKMLKTGEHFNQD
>21FJ_E1 len=527 copies=1 longest=1
LKENKIDWFIFISTFVLIVAICLPLVLYPEKGGELVSQANSFVTSNFGMVFLWAGMGGFAFLTYLFFSEHGKIKFGTKDEKPEFSNFSWGAMVFCAGVASGLVYWGTIEWAYYFTAPPYGIEPKSTDAAEIAAAYGIFHWGPSAWAFFVLPSLPIAYSYYILKVPVLRLSEICRIVIGKHSDGVLGKVIDIIFMFGILGAAGTSLGLGTPLVAAGISKITGIESSTSMTVFVLVLCTALFTWSAYSGLKRGLKLLSDLAIICSVILLTYVLLVGPTEFILKMGTNSVGLLFTNFVRWNLYTDPVSGSGFVESWTVFYWAWWIVYTPFMGLFIAKISRGRTIRQVILGGIAWGTLGCAAYFSILGNYAMHLELTKELSVTGLLEQIGAPATIMEVIGTLPLGSVVVIVFSFIALIFLATTFDSAAYMMASSTTPTLGLNEEPAKWNRLFWALTLFILPGTLMVLGGDLKTLQTASILTAIPFIFVIMLLVVALMKMLKTGEHFNQDLQDDPIKLAEVDSLEERQKPAA
>21FK_E1 len=504 copies=1 longest=1
LKENKIDWFIFISTFVLIVAICLPLVLYPEKGGELVSQANSFVTSNFGMVFLWAGMGGFAFLTYLFFSEHGKIKFGTKDEKPEFSNFSWGAMVFCAGVASDLVYWGTIEWAYYFTAPPYGIEPKSTDAAEIAAAYGIFHWGPSAWAFFVLPSLPIAYSYYILKVPVLRLSEICRIVIGKHSDGVLGKVIDIIFMFGILGAAGTSLGLGTPLVAAGISKITGIESSTSMTVFVLVLCTALFTWSAYSGLKRGLKLLSDLAIICSVILLTYVLLVGPTEFILKMGTNSVGLLFTNFVRWNLYTDPVSGSGFVESWTVFYWAWWIVYTPFMGLFIAKISRGRTIRQVILGGIAWGTLGCAAYFSILGNYAMHLELTKELSVTGLLEQIGAPATIMEVIGTLPLGSVVVIVFSFIALIFLATTFDSAAYMMASSTTPTLGLNEEPAKWNRLFWALTLFILPGTLMVLGGDLKTLQTASILTAIPFIFVIMLLVVALMKMLKTGEHFNQ
>21NP_E1 len=429 copies=1 longest=1
MAAQGYGYYRTVIFSAMFGGYSLYYFNRKTFSFVMPSLVEEIPLDKDDLGFITSSQSAAYAISKFVSGVLSDQMSARWLFSSGLLLVGLVNIFFAWSSTVPVFAALWFLNGLAQGLGWPPCGKVLRKWFEPSQFGTWWAILSTSMNLAGGLGPILATILAQSYSWRSTLALSGALCVVVSFLCLLLIHNEPADVGLRNLDPMPSEGKKGSLKEESTLQELLLSPYLWVLSTGYLVVFGVKTCCTDWGQFFLIQEKGQSALVGSSYMSALEVGGLVGSIAAGYLSDRAMAKAGLSNYGNPRHGLLLFMMAGMTVSMYLFRVTVTSDSPKLWILVLGAVFGFSSYGPIALFGVIANESAPPNLCGTSHAIVGLMANVGGFLAGLPFSTIAKHYSWSTAFWVAEVICAASTAAFFLLRNIRTKMGRVSKKAE
>21NQ_E1 len=429 copies=1 longest=1
MAAQGYGYYRTVIFSAMFGGYSLYYFNRKTFSFVMPSLVEEIPLDKDDLGFITSSQSAAYAISKFVSGVLSDQMSARWLFSSGLLLVGLVNIFFAWSSTVPVFAALWFLNGLAQGLGWPPCGKVLRKWFEPSQFGTWWAILSTSMNLAGGLGPILATILAQSYSWRSTLALSGALCVVVSFLCLLLIHNEPADVGLRNLDPMPSEGKKGSLKEESTLQELLLSPYLWVLSTGYLVVFGVKTCCTDWGQFFLIQEKGQSALVGSSYMSALEVGGLVGSIAAGYLSDRAMAKAGLSNYGNPRHGLLLFMMAGMTVSMYLFRVTVTSDSPKLWILVLGAVFGFSSYGPIALFGVIANESAPPNLCGTSHAIVGLMANVGGFLAGLPFSTIAKHYSWSTAFWVAEVICAASTAAFFLLRNIRTKMGRVSKKAE
>21NX_E1 len=225 copies=1 longest=0
MWQLLLPTALVLTAFSGIQAGLQKAVVNLDPKWVRVLEEDSVTLRCQGTFSPEDNSIKWFHNESLIPHQDANYVIQSARVKDSGMYRCQTALSTISDPVQLEVHMGWLLLQTTKWLFQEGDPIHLRCHSWQNRPVRKVTYLQNGKGKKYFHENSELLIPKATHNDSGSYFCRGLIGHNNKSSASFRISLGDPGSPSMFPPWHQGGSHHHHHHHHSRVDYKDDDDK
>21NX_E2 len=263 copies=2 longest=1
MYRMQLLSCIALSLALVTNSARIHEPSGPISTINPCPPCKECHKCPAPNLEGGPSVFIFPPNIKDVLMISLTPKVTCVVVDVSEDDPDVQISWFVNNVEVHTAQTQTHREDYNSTIRVVSTLPIQHQDWMSGKEFKCKVNNKDLPSPIERTISKIKGLVRAPQVYILPPPAEQLSRKDVSLTCLVVGFNPGDISVEWTSNGHTEENYKDTAPVLDSDGSYFIYSKLNMKTSKWEKTDSFSCNVRHEGLKNYYLKKTISRSPGK
>21OR_E1 len=429 copies=1 longest=1
MAAQGYGYYRTVIFSAMFGGYSLYYFNRKTFSFVMPSLVEEIPLDKDDLGFITSSQSAAYAISKFVSGVLSDQMSARWLFSSGLLLVGLVNIFFAWSSTVPVFAALWFLNGLAQGLGWPPCGKVLRKWFEPSQFGTWWAILSTSMNLAGGLGPILATILAQSYSWRSTLALSGALCVVVSFLCLLLIHNEPADVGLRNLDPMPSEGKKGSLKEESTLQELLLSPYLWVLSTGYLVVFGVKTCCTDWGQFFLIQEKGQSALVGSSYMSALEVGGLVGSIAAGYLSDRAMAKAGLSNYGNPRHGLLLFMMAGMTVSMYLFRVTVTSDSPKLWILVLGAVFGFSSYGPIALFGVIANESAPPNLCGTSHAIVGLMANVGGFLAGLPFSTIAKHYSWSTAFWVAEVICAASTAAFFLLRNIRTKMGRVSKKAE
>21TW_E1 len=363 copies=1 longest=1
EVQLVESGGGLVQPGGSLRLSCAASGFTFSDYWMYWVRQAPGKGLEWVSEINTNGLITKYPDSVKGRFTISRDNAKNTLYLQMNSLRPEDTAVYYCARSPSGFNRGQGTLVTVSSGGGGSGGGSEVQLVESGGGLVQPGNSLRLSCAASGFTFSSFGMSWVRQAPGKGLEWVSSISGSGSDTLYADSVKGRFTISRDNAKTTLYLQMNSLRPEDTAVYYCTIGGSLSRSSQGTLVTVSSGGGGSGGGSEVQLVESGGGLVQPGGSLRLSCAASGFTFSDYWMYWVRQAPGKGLEWVSEINTNGLITKYPDSVKGRFTISRDNAKNTLYLQMNSLRPEDTAVYYCARSPSGFNRGQGTLVTVSS
>21TW_E2 len=158 copies=3 longest=0
GVRSSSRTPSDKPVAHVVANPQAEGQLQWLNRRANALLANGVELRDNQLVVPSEGLYLIYSQVLFKGQGCPSTHVLLTHTISRIAVSYQTKVNLLSAIKSPCQRETPEGAEAKPWYEPIYLGGVFQLEKGDRLSAEINRPDYLDFAESGQVYFGIIAL
>21TY_E1 len=748 copies=1 longest=1
MKTIIALSYIFCLVFADYKDDDDAENLYFQGMLPDWKSSLILMAYIIIFLTGLPANLLALRAFVGRIRQPQPAPVHILLLSLTLADLLLLLLLPFKIIEAASNFRWYLPKVVCALTSFGFYSSIYCSTWLLAGISIERYLGVAFPVQYKLSRRPLYGVIAALVAWVMSFGHCTIVIIVQYLNTTEQVRSGNEITCYENFTDNQLDVVLPVRLELCLVLFFIPMAVTIFCYWRFVWIMLSHLELRRLLREQGEKLLKLAEESEKKIKEAIEKGEDVVKVATEETHKLIKAIWESSFEVARLLGVPEEVLEKFRKELEKLLEESKKRIEEAIKKGEDVLKVVEEEAKKLREHSEKFLETARSLGVDPRPLERLARIVRVVEESLVRIVKAIKEGEDVEKVLLSESTLIKSSIILSSGLDPREAYEELLATYEETRDSELVKEFVELVRELLRLRGEPEELVRQLDKLIEAIDRMEKAVAEGEDTLPLFLELAREILKALSDVARILAEREGLDVELIDEIVDTASRLLEEVIKKAKEGEDLDEEEEKVKKELEELKKRTEERARALGQDVELVRTIVDSLSTLLSEIVEGIKRVKEGEDPLEVLARVVLTATLQILELVRTAERRRRERRRRRAVGLAVVTLLNFLVCFGPYNVSHLVGYHQRKSPWWRSIAVVFSSLNASLDPLLFYFSSSVVRRAFGRGLQVLRNQGSSLLGRRGKDTAEGTNEDRGVGQGEGMPSSDFTTELEVLFQ
>21VL_E1 len=522 copies=1 longest=1
MSTGKRKYTKRSDYWNKGSTEKAALSPTQSATKEKNLVLSPEIGTIGLNSIKAFTNFMQPYETRFPENIRTYKEMGEDPDVATALDATYIFVDRAFFDFKIKYNVSSAKSRRAAKFVDYTLRNMNAPLRQYVRSLLTYKQFGFAFAEKVYELDEDPKSPYFGYYRLVKLAFRPQDTIDLAQPFTYSDDGRTILTVNQNITNGMVSPGTNATLIGRKEIPMEKVIYVGSNITENNPLGVSPLLAVYRSWREKSLIQEYEVVGVSKDLGGMPVLMVPSDILNRASLNPSGDEAQSLRVLQANIANLHAGEQSYMVLPSDVYEGTVMRQYDLVFQGVEGSGKQFDTQALIKQRKLDIYNRFGAGVLIMGDGEGGSYSLSDNKQTLLSHFIERDVDIITEALNTQVIPQLLRLNGIFLSQEDMPKFVSDDIGDPDIEVNAKAIQQIVAAGAIPLTPEVINEFFERLGFNYRIPDDIVADPDKFQEFLETFMPDKTSRSGDGLAAGAGNGTSTSPAALDTSAANLAN
>21VL_E2 len=182 copies=1 longest=0
MLIASDLVPMIRILVFNPSQETLPDAMIEQIIQTWIDVLGNDDANKCAVLWNSLISVLEYLWNTDVLNHNTQTGGALSRKEKVGEVQVEVTFGTGQTEYISPWENIYKGYLDGDMMIPGCTSGRGVTSKVLVGGVDAREIDRVNSDPNSVNGLGGVGSVDRHTRNIKYARNYGPIGYYRRDK
>21WL_E1 len=468 copies=1 longest=1
MATFRDKVVSVTLTYGATSISETQFDIPLILTGHNVTGNLVDYFTSSDALLQAGFGTADPAYKMAKLLFDGLFAPEQVIVGKRDVSKTTLTPVVEDSATYVITIKANSKSKDFKFVADDTATAQEIVVGLTEMINADVVYKEFFTVSNDGSVITVTPVAGKYATMDSSGFTTKVEYANDILEDIAKIADYENSWFWLLSDSHAEQDIIDLAGYVEEHDKVYFFSTSQPGVLTKEEDNILERLGDMGYNNTCMALWMTNADTVFPEAAVVGSICSATPGTTTLHGKTLVGIEIEKLGQTAENFIVQQNGNIYRKEHGVLFYRDGFMVSGFYVDYVVHALWFKARVEESLFALFKQQSMLGSGVRATSAGLALIRQAVTANPIQVGINNGSIANEVVTSEETGLLVSLKPTIYIPSRADMTDAQINARLVDGMVIEYVYAGFFHYVKVQVNVLTNRTANSQNSNSSTVSS
>22FX_E1 len=182 copies=1 longest=1
MTTASPSQVRQNYHQDAEAAINRQINLELYASYVYLSMSCYFDRDDVALKNFAKYFLHQSHEEREHAEKLMKLQNQRGGRIFLQDIKKPDRDDWESGLNAMECALHLEKSVNQSLLELHKLATDKNDPHLCDFIETYYLSEQVKSIKELGDHVTNLRKMGAPEAGMAEYLFDKHTLGHGDES
>22JY_E1 len=51 copies=5 longest=1
NKKLDLSNVQSKCGSKDNIKHVLGGGSVQIVYKPVDLSKVTFKCGSLGNIH
>22OF_E1 len=309 copies=1 longest=1
MMSGILEVYDAKRTDNLIITLEGISGSETIKAIKKRIAQKKLKLTEERQALRVEPKGKPLADDQKLSDLGLSSQKAVLYVRDLGPQIAWKTVFMAEYAGPLFVYPLFYLRPTFIYGQAAVNATMHPAVQIAFFAWSFHYAKRLFETQFIHRFGNSTMPQFNLVKNCSYYWGFAAFVAYFVNHPLFTPPAFGDLQVYFGLAGFVISEFGNLSIHILLRNLRPAGTRERRIPKPDGNPLSLLFNYVSCPNYTYEVASWIFFSIMVQSLPAIIFTTAGFAQMAIWAQGKHRNYLKEFPDYPKNRKAIVPFVL
>22OF_E2 len=241 copies=1 longest=0
MHHHHHHHHHHSGDEVDAGSGHMMSVQTYLVAYNVLQILGWSAILVKTVLGLANGLTWPQLYESVEFELKIFQTAAILEVIHAIVGLVRSPVGTTAMQVTSRVVLVWPILHLCSTARFSIGVPLLLVAWSVTEVIRYSFYALSVLKQPIPYFLLYLRYTLFYVLYPMGVSGELLTLFASLNEVDEKKILTLEMPNRLNMGISFWWVLIIAALSYIPGFPQLYFYMIGQRKKILGGGSKKKQ
>22PD_E1 len=128 copies=1 longest=0
EVQLVESGGGLIQPGGSLRLSCAASGFSVSTNYMSWVRQAPGKGLEWVSVIYGGDTTYYADSVRGRFTISRDNSKNTLYLEMNSLRTDDTAVYYCARDEGRYFDIWTGHVGPNCFDYWGQGTLVTVSS
>22PD_E2 len=112 copies=1 longest=0
DIQLTQSPSSLSASVGDRVTITCRASQSISTYLNWYQQKPGKAPKLLIYGASNLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQSYITLVTFGQGTRLEIKRTVAA
>22PD_E3 len=200 copies=1 longest=1
NLCPFDEVFNATRFASVYAWNRKRISNCVADYSVLYNFAPFFAFKCYGVSPTKLNDLCFTNVYADSFVIRGNEVSQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDSKVGGNYNYRYRLFRKSNLKPFERDISTEIYQAGNKPCNGVAGVNCYFPLQSYGFRPTYGVGHQPYRVVVLSFELLHAPATVCGPKKSTNL
>22PL_E1 len=141 copies=1 longest=0
SHTLPVRLLRPSDDVQKIVEELQSLSKMLLKDVEEEKGVLVSQNYTLPCLSPDAQPPNNIHSPAIRAYLKTIRQLDNKSVIDEIIEHLDKLIFQDAPETNISVPTDTHECKRFILTISQQFSECMDLALKSLTSGAQQATT
>22PL_E2 len=212 copies=1 longest=0
ALPAKPENISCVYYYRKNLTCTWSPGKETSYTQYTVKRTYAFGEKHDNCTTNSSTSENRASCSFFLPRITIPDNYTIEVEAENGDGVIKSHMTYWRLENIAKTEPPKIFRVKPVLGIKRMIQIEWIKPELAPVSSDLKYTLRFRTVNSTSWMEVNFAKNRKDKNQTYNLTGLQPFTEYVIALRCAVKESKFWSDWSQEKMGMTEEEHHHHHH
>22PL_E3 len=311 copies=1 longest=1
ERLPLTPVSLKVSTNSTRQSLHLQWTVHNLPYHQELKMVFQIQISRIETSNVIWVGNYSTTVKWNQVLHWSWESELPLECATHFVRIKSLVDDAKFPEPNFWSNWSSWEEVSVQDSTGQDILFVFPKDKLVEEGTNVTICYVSRNIQNNVSCYLEGKQIHGEQLDPHVTAFNLNSVPFIRNKGTNIYCEASQGNVSEGMKGIVLFVSKVLEEPKDFSCETEDFKTLHCTWDPGTDTALGWSKQPSQSYTLFESFSGEKKLCTHKNWCNWQITQDSQETYNFTLIAENYLRKRSVNILFNLTHRVYHHHHHH
>22RD_E1 len=333 copies=1 longest=1
APADPGKAGVPGVAAPGAPAAAPPAKEIPEVLVDPRSRRRYVRGRFLGKGGFAKCFEISDADTKEVFAGKIVPKSLLLKPHQREKMSMEISIHRSLAHQHVVGFHGFFEDNDFVFVVLELCRRRSLLELHKRRKALTEPEARYYLRQIVLGCQYLHRNRVIHRDLKLGNLFLNEDLEVKIGDFGLATKVEYDGERKKVLCGTPNYIAPEVLSKKGHSFEVDVWSIGCIMYTLLVGKPPFETSCLKETYLRIKKNEYSIPKHINPVAASLIQKMLQTDPTARPTINELLNDEFFTSGYIPARLPITCLTIPPRFSIAPSSLDPSNRKPLTVLNK
>22WP_E1 len=336 copies=1 longest=1
GPGSGSGLSLLHLQEKNWSALLTAVVIILTIAGNILVIMAVSLEKKLQNATNYFLMSLAIADMLLGFLVMPVSMLTILYGYRWPLPSKLCAVWIYLDVLFSTASIMHLCAISLDRYVAIQNPIHHSRFNSRTKAFLKIIAVWTISVGISMPIPVFGLQDDSKVFKEGSCLLADDNFVLIGSFVSFFIPLTIMVITYFLTIKSLQKEATLCVSDLGTRAKLASFSFLPQSSLSSEKLFQRSIHREPGSYTGRRTMQSISNEQKACKVLGIVFFLFVVMWCPFFITNIMAVICKESCNEDVIGALLNVFVWIGYLSSAVNPLVYTLFNKTYRSAFSRY
>22WR_E1 len=336 copies=1 longest=1
GPGSGSGLSLLHLQEKNWSALLTAVVIILTIAGNILVIMAVSLEKKLQNATNYFLMSLAIADMLLGFLVMPVSMLTILYGYRWPLPSKLCAVWIYLDVLFSTASIMHLCAISLDRYVAIQNPIHHSRFNSRTKAFLKIIAVWTISVGISMPIPVFGLQDDSKVFKEGSCLLADDNFVLIGSFVSFFIPLTIMVITYFLTIKSLQKEATLCVSDLGTRAKLASFSFLPQSSLSSEKLFQRSIHREPGSYTGRRTMQSISNEQKACKVLGIVFFLFVVMWCPFFITNIMAVICKESCNEDVIGALLNVFVWIGYLSSAVNPLVYTLFNKTYRSAFSRY
>22WT_E1 len=280 copies=1 longest=1
WAALLILMVIIPTIGGNTLVILAVSLEKKLQYATNYFLMSLAVADLLVGLFVMPIALLTIMFEAMWPLPLVLCPAWLFLDVLFSTASIWHLCAISVDRYIAIKKPIQANQYNSRATAFIKITVVWLISIGIAIPVPIKGIETDVDNPNNITCVLTKERFGDFMLFGSLAAFFTPLAIMIVTYFLTIHALQKKRLLSGSRQTISNEQRASKVLGIVFFLFLLMWCPFFITNITLVLCDSCNQTTLQMLLEIFVWIGYVSSGVNPLVYTLFNKTFRDAFGRY
>23IC_E1 len=499 copies=1 longest=1
MAHSHEMTSRKPPGIRLFGGITVLRTYQTLVLVLTFVAYTCFHMTRKIPSIVKSVLDPQTKLGSSPWGRLHTKNTLNIGWLPFNTIDGSALLGEIDVAFLAVYSVGMFFAGHLGDRMDLRIFLTIGMFGTAVFTALFGAGYWLNIHNFYYFLVIQMIAGLFQAIGWPSVVAIVGNWFGKSKRGLIMGIWNAHTSVGNISGSLLAAFLLKFGWGWSFAIPSLIMVAVGLLVFVFLPVSPEVMEIDIDDGEISSVKDTTKEPLLEPGQEVKHNAVGFLEAWKIPGVAPFALCLFFSKLVAYTFLYWLPFYISHTPIGGEYLSDALAGSLSTIFDVGGVLGGVLAGHISDRLNARAVTAASFMYCAIPALFLYRTYGSMSIMWNICLMFITGMFVNGPYALITTAVSADLGTHSSLNGNSRALATVTAIIDGTGSVGAAIGPLLTGYISSSSWSAVFTMLMAAALLAGLLLTQLVCSELKGKATSNASKDVADAQGTYSDEV
>23IF_E1 len=499 copies=1 longest=1
MAHSHEMTSRKPPGIRLFGGITVLRTYQTLVLVLTFVAYTCFHMTRKIPSIVKSVLDPQTKLGSSPWGRLHTKNTLNIGWLPFNTIDGSALLGEIDVAFLAVYSVGMFFAGHLGDRMDLRIFLTIGMFGTAVFTALFGAGYWLNIHNFYYFLVIQMIAGLFQAIGWPSVVAIVGNWFGKSKRGLIMGIWNAHTSVGNISGSLLAAFLLKFGWGWSFAIPSLIMVAVGLLVFVFLPVSPEVMEIDIDDGEISSVKDTTKEPLLEPGQEVKHNAVGFLEAWKIPGVAPFALCLFFSKLVAYTFLYWLPFYISHTPIGGEYLSDALAGSLSTIFDVGGVLGGVLAGHISDRLNARAVTAASFMYCAIPALFLYRTYGSMSIMWNICLMFITGMFVNGPYALITTAVSADLGTHSSLNGNSRALATVTAIIDGTGSVGAAIGPLLTGYISSSSWSAVFTMLMAAALLAGLLLTQLVCSELKGKATSNASKDVADAQGTYSDEV
>23IG_E1 len=499 copies=1 longest=1
MAHSHEMTSRKPPGIRLFGGITVLRTYQTLVLVLTFVAYTCFHMTRKIPSIVKSVLDPQTKLGSSPWGRLHTKNTLNIGWLPFNTIDGSALLGEIDVAFLAVYSVGMFFAGHLGDRMDLRIFLTIGMFGTAVFTALFGAGYWLNIHNFYYFLVIQMIAGLFQAIGWPSVVAIVGNWFGKSKRGLIMGIWNAHTSVGNISGSLLAAFLLKFGWGWSFAIPSLIMVAVGLLVFVFLPVSPEVMEIDIDDGEISSVKDTTKEPLLEPGQEVKHNAVGFLEAWKIPGVAPFALCLFFSKLVAYTFLYWLPFYISHTPIGGEYLSDALAGSLSTIFDVGGVLGGVLAGHISDRLNARAVTAASFMYCAIPALFLYRTYGSMSIMWNICLMFITGMFVNGPYALITTAVSADLGTHSSLNGNSRALATVTAIIDGTGSVGAAIGPLLTGYISSSSWSAVFTMLMAAALLAGLLLTQLVCSELKGKATSNASKDVADAQGTYSDEV
>23WM_E1 len=1552 copies=1 longest=1
MGHHHHHHHHSSGLEVLFQGPGGTRQQNLDVLDKIVVYPEGEYDREKADAIVDALAWIPLKYLQALLDAGVKIVLANSEELLELLKKYPRLRDLIRALIEVAKMTVGVGDGGDKRAYINIEGSVEGVRLLALHQTAHVIDDVVLNSISLSPIFLKYFEYLKKLLLEALKKKPEALKNYELNKKILDKLENEELTPEEIRELVRKMKENIVGLSAAYLGTYPEEAFAWGFAAYALRRGGDEEGLERFRRYLMRTSGLDPESVEKVLDALIEMGEFAAELGELEELLKKAKEKAKKEKTEAAKEALKIAETLSLAAEIALELEVLLLEFPDDLEVHRLALRATRLLLEVVEEILRKAEKLVDEAPELVAMVAALAARVLALLARNLLRLPEELAEEAREAGRLLGEVALKLLELLKKLLEKAEKEGNKEIHELIAEAIAALIGVADALRRLGLEEEAEELKKKIAELVNKLVESYPEQLFKLIEKTLDHPETLAEVLSQVKLTKELAKELFEKALEAIKEKNYKLAYLYLTAAATNPEMLPKIKELLEKEEDIEVIKVIAKAAITAANNENLTKEQAELLAELLLLALEKLIPDEELLELIIKILKALAKLFKKLDPKTAIEYLKKVTELVLELLEREKDPERIIKLGEALAKLLEGVGENAAKTKDKELVKTAIELAKEVIEKILKRLLEDKSLSPEEVAALVEAIVKIAEGTIKALILAGLLEEALEFLKEVLEKIVETIEQLPDELRPEAIAALLESLGESLVEIAKTALENLPEEEAEKFLKEVLRLILEFLEKLFEIALPLADDKEAVEKIMKGLAKLVGGLLVVLLLVPGVEEEVRRIIRLITEFIEEVFRRMVEKAETEEEREEVREFIAELLLEFLKIIGEKVVKGTEEAVRKGLLKPEGLRAALRLLGELVEALLRLAAELLPEELAREVAREMLEIAFETLAEVLKVVSPDPLAAVEAAEAAFEVLLRLIRVAVELHGLDEEELVELAVRKLREARREIEKESREERELERRLLEIRAEIDPEEVAKELEKEIEELKKNIEEALKKLEEFFEKVQERIKELPERERELVEKLATAFLELMLTVIRNSVERLLELALLLAKVLVKLGKYEEAFEVIKEAVRLAKEVVRKLEELAERSESEELREHALETAAEILARAIKALAEAIKAFADKDPEIAEELFELMLELLEELERLVRRAVERGFPRVVRHLGPAVKALAEAALHHPNREEALERAVERLRRLLEELERLAERVEREEEDPVRRLALLAALAEAQVAVLEALARLSEHLPEERRKEVEEWLKKRVLELLERLLELLREAEELYRRTGDERLREIARELRRILERLAEVTEGSLELAEAAWELLAPHPEAFGARAEVVVNLAKLDPERGLRALEELEKQVSTEEQREALAYALGRIVGYVDEERAKELIDRAIELGQYDAFAEGLAEALLEAPEERIPKLLKAIEEFLKKLEEELKEIEKKINEPEFQKSKDEEEKKKIIEKLKKIKRAARIVLEAALKVLLKAPELASRVAKLALEIISKVERILKKI
>23WM_E2 len=42 copies=1 longest=0
DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIA
>23WN_E1 len=1513 copies=1 longest=1
MGHHHHHHHHSSGLEVLFQGPGGTRNLELARAADVTVTVADTPEEMYEAAKVAVETVRELAAGDPRRDEYVALAERLFRTGIERGGIAGIAIYADGRRRVFVVAPSDASDEALIYALAHQLAHLIIAEDLERRGLPLSAVPPGVVEGLADVFGATAYAAYLELKGEKVTLEKWREMQLRLAEETERIGREAGLEAHVEGGRIAAEIARRTNEEEAQKLIEEVKPLVEFILGLLRVARTAHRAVPGGRATVTLKPDGLAEVELKGADPKNPAIAIIKVAAMLQGLLLRFPEDEEVHKLVVENLRILIENLEKLVEKAEKDLNTNPEFIRQLAAALELVAEVLRTILRFGKEELKEEAKELGRLLGEVALKLLELLKKLLEKAEKEGNKEIHELIAEAIAALIGVADALRRLGLEEEAEELKKKIAELVNKLVESYPEQLFKLIEKTLDHPETLAEVLSQVKLTKELAKELFEKALEAIKEKNYKLAYLYLTAAATNPEMLPKIKELLEKEEDIEVIKVIAKAAITAANNENLTKEQAELLAELLLLALEKLIPDEELLELIIKILKALAKLFKKLDPKTAIEYLKKVTELVLELLEREKDPERIIKLGEALAKLLEGVGENAAKTKDKELVKTAIELAKEVIEKILKRLLEDKSLSPEEVAALVEAIVKIAEGTIKALILAGLLEEALEFLKEVLEKIVETIEQLPDELRPEAIAALLESLGESLVEIAKTALENLPEEEAEKFLKEVLRLILEFLEKLFEIALPLADDKEAVEKIMKGLAKLVGGLLVVLLLVPGVEEEVRRIIRLITEFIEEVFRRMVEKAETEEEREEVREFIAELLLEFLKIIGEKVVKGTEEAVRKGLLKPEGLRAALRLLGELVEALLRLAAELLPEELAREVAREMLEIAFETLAEVLKVVSPDPLAAVEAAEAAFEVLLRLIRVAVELHGLDEEELVELAVRKLREARREIEKESREERELERRLLEIRAEIDPEEVAKELEKEIEELKKNIEEALKKLEEFFEKVQERIKELPERERELVEKLATAFLELMLTVIRNSVERLLELALLLAKVLVKLGKYEEAFEVIKEAVRLAKEVVRKLEELAERSESEELREHALETAAEILARAIKALAEAIKAFADKDPEIAEELFELMLELLEELERLVRRAVERGFPRVVRHLGPAVKALAEAALHHPNREEALERAVERLRRLLEELERLAERVEREEEDPVRRLALLAALAEAQVAVLEALARLSEHLPEERRKEVEEWLKKRVLELLERLLELLREAEELYRRTGDERLREIARELRRILERLAEVTEGSLELAEAAWELLAPHPEAFGARAEVVVNLAKLDPERGLRALEELEKQVSTEEQREALAYALGRIVGYVDEERAKELIDRAIELGQYDAFAEGLAEALLEAPEERIPKLLKAIEEFLKKLEEELKEIEKKINEPEFQKSKDEEELEKIKKKIESIKRAAEKILKAALKKILEEPELALKVAELALKVIAKANKILEKL
>23WN_E2 len=42 copies=1 longest=0
DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIA
>23WO_E1 len=1503 copies=1 longest=1
MGHHHHHHHHSSGLEVLFQGPGGTEVTVLTASSPDEIPAIAERVLELTGDREVAERVREAMAELWRALAAMPDVVIGGLALLDYPDVVVLVEGPRPASEEARLLTLAHQIAHGIFARLAREAGVDLSLPEVRAANEGVATFAGLAEMLGVASVEEAARAAFDWYLGDPRARERAARLPVTLEQLVRMVREVVKLAEIYRRAREQGLLPAEFTDAHLGGTAVAMDILLFVGEARLEEEAERVERDEDLSEEEKKALRVILTAKALILLALGLLALLARHPDNKEVKELVLRLANLLIEQVERLLEEAKKIEDKVPELAAEAIRLLARALLVLAELLLRLSPYFPEEVKENGRLLGEVALKLLELLKKLLEKAEKEGNKEIHELIAEAIAALIGVADALRRLGLEEEAEELKKKIAELVNKLVESYPEQLFKLIEKTLDHPETLAEVLSQVKLTKELAKELFEKALEAIKEKNYKLAYLYLTAAATNPEMLPKIKELLEKEEDIEVIKVIAKAAITAANNENLTKEQAELLAELLLLALEKLIPDEELLELIIKILKALAKLFKKLDPKTAIEYLKKVTELVLELLEREKDPERIIKLGEALAKLLEGVGENAAKTKDKELVKTAIELAKEVIEKILKRLLEDKSLSPEEVAALVEAIVKIAEGTIKALILAGLLEEALEFLKEVLEKIVETIEQLPDELRPEAIAALLESLGESLVEIAKTALENLPEEEAEKFLKEVLRLILEFLEKLFEIALPLADDKEAVEKIMKGLAKLVGGLLVVLLLVPGVEEEVRRIIRLITEFIEEVFRRMVEKAETEEEREEVREFIAELLLEFLKIIGEKVVKGTEEAVRKGLLKPEGLRAALRLLGELVEALLRLAAELLPEELAREVAREMLEIAFETLAEVLKVVSPDPLAAVEAAEAAFEVLLRLIRVAVELHGLDEEELVELAVRKLREARREIEKESREERELERRLLEIRAEIDPEEVAKELEKEIEELKKNIEEALKKLEEFFEKVQERIKELPERERELVEKLATAFLELMLTVIRNSVERLLELALLLAKVLVKLGKYEEAFEVIKEAVRLAKEVVRKLEELAERSESEELREHALETAAEILARAIKALAEAIKAFADKDPEIAEELFELMLELLEELERLVRRAVERGFPRVVRHLGPAVKALAEAALHHPNREEALERAVERLRRLLEELERLAERVEREEEDPVRRLALLAALAEAQVAVLEALARLSEHLPEERRKEVEEWLKKRVLELLERLLELLREAEELYRRTGDERLREIARELRRILERLAEVTEGSLELAEAAWELLAPHPEAFGARAEVVVNLAKLDPERGLRALEELEKQVSTEEQREALAYALGRIVGYVDEERAKELIDRAIELGQYDAFAEGLAEALLEAPEERIPKLLKAIEEFLKKLEEELKEIEKKINEPEFQKSKDEEEKKKIKERLKKIFRAASIIAETSAKLLDKAPLLALRVAKLALRNIEKASELLEKI
>23WO_E2 len=42 copies=1 longest=0
DAEFRHDSGYEVHHQKLVFFAEDVGSNKGAIIGLMVGGVVIA
>25IF_E1 len=400 copies=1 longest=1
MGKTIIALSYIFCLVFADYKDDDDAANFTPVNGSSGNQSVRLVTSSSLEVLFQGPGSMNHTTSEYYDYEYDHEHYSDLPDVPVDCPAGTCFTSDVYLIVLLVLYAAVFLVGVPGNTLVAWVTWKESRHRLGASWFLHLTMADLLCCVSLPFLAVPIAQKGHWPYGAAGCWLLSSITILSMYASVLLLTGLSGDLFLLAFRPSWKGADHRTFGVRVVQASSWMLGLLLTVPSAVYRRLLQEHYPPRLVCGIDYGGSVSAEVAITTVRFLFGFLGPLVFMAGCHGILQRQMARRHWPLGTAVVVGFFICWTPYHVLRVIIAAAPPHSLLLARVLEAEPLFNGLALAHSALNPIMFLYFGRKQLCKSLQAACHWALRDPQDEESAVTKVSISTSHEMVSEMPV
>25IF_E2 len=76 copies=1 longest=0
NLHLLRQKIEEQAAKYKHSVPKKCCYDGARVNFYETCEERVARVTIGPLCIRAFNECCTIANKIRKESPHKPVQLG
>25PV_E1 len=427 copies=1 longest=1
AGPTVPDRDNDGIPDSLEVEGYTVDVKNKRTSLSPWISNIHEKKGLTKYKSSPEKWSTASDPYSDFEKVTGRIDKNVSPEARHPLVAAYPIVHVDMENIILSKNEDQSTQNTDSQTRTISKNTSTSRTHTSEVHGNAEVHASFFDIGGSVSAGFSNSNSSTVAIDHSLSLAGERTWAETMGLNTADTARLNANIRYVNTGTAPIYNVLPTTSLVLGKNQTLATIKAKENQLSQILAPNNYYPSKNLAPIALNAQDDFSSTPITMNYNQFLELEKTKQLRLDTDQVYGNIATYNFENGRVRVDTGSNWSEVLPQIQETTARIIFNGKDLNLVERRIAAVNPSDPLETTKPDMTLKEALKIAFGFNEPNGNLQYQGKDITEFDFNFDQQTSQNIKNQLAELNATNIYTVLDKIKLNAKMNILIRDKRFH
>25PV_E2 len=220 copies=1 longest=0
QVQLQESGPGLVKPSQTLSLTCTVSGGSLSSGGSYWSWIRQHPGKGLEYLGYIFYFGRPYYNPSLQSRITISVDTSKNQFSLKLNSVTAADTAVYYCARASLRGILWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSC
>25PV_E3 len=214 copies=1 longest=0
EIVLTQSPATLSLSPGERATLSCRASQSVGNYLAWYQQKLGQAPRLLIYDASNRATGIPARFSGSGSGTDFTLTISSLEPEDFAVYYCQQRSNWPLTFGGGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>25RV_E1 len=724 copies=2 longest=1
MDKLIITGGNRLDGEIRISGAKNSALPILAATLLADTPVTVCNLPHLHDITTMIELFGRMGVQPIIDEKLNVEVDASSIKTLVAPYELVKTMRASILVLGPMLARFGEAEVALPGGCAIGSRPVDLHIRGLEAMGAQIEVEGGYIKAKAPAGGLRGGHFFFDTVSVTGTENLMMAAALANGRTVLQNAAREPEVVDLANCLNAMGANVQGAGSDTIVIEGVKRLGGARYDVLPDRIETGTYLVAAAATGGRVKLKDTDPTILEAVLQKLEEAGAHISTGSNWIELDMKGNRPKAVNVRTAPYPAFPTDMQAQFISMNAVAEGTGAVIETVFENRFMHVYEMNRMGAQILVEGNTAIVTGVPKLKGAPVMATDLRASASLVIAGLVAEGDTLIDRIYHIDRGYECIEEKLQLLGAKIRRVPGMIKQRTLKNIIRATGVGLHSGEKVYLTLKPAPVDTGIVFCRTDLDPVVEIPARAENVGETTMSTTLVKGDVKVDTVEHLLSAMAGLGIDNAYVELSASEVPIMDGSAGPFVFLIQSAGLQEQEAAKKFIRIKREVSVEEGDKRAVFVPFDGFKVSFEIDFDHPVFRGRTQQASVDFSSTSFVKEVSRARTFGFMRDIEYLRSQNLALGGSVENAIVVDENRVLNEDGLRYEDEFVKHKILDAIGDLYLLGNSLIGEFRGFKSGHALNNQLLRTLIADKDAWEVVTFEDARTAPISYMRPAAAV
>25RW_E1 len=724 copies=2 longest=1
MDKLIITGGNRLDGEIRISGAKNSALPILAATLLADTPVTVCNLPHLHDITTMIELFGRMGVQPIIDEKLNVEVDASSIKTLVAPYELVKTMRASILVLGPMLARFGEAEVALPGGCAIGSRPVDLHIRGLEAMGAQIEVEGGYIKAKAPAGGLRGGHFFFDTVSVTGTENLMMAAALANGRTVLQNAAREPEVVDLANCLNAMGANVQGAGSDTIVIEGVKRLGGARYDVLPDRIETGTYLVAAAATGGRVKLKDTDPTILEAVLQKLEEAGAHISTGSNWIELDMKGNRPKAVNVRTAPYPAFPTDMQAQFISMNAVAEGTGAVIETVFENRFMHVYEMNRMGAQILVEGNTAIVTGVPKLKGAPVMATDLRASASLVIAGLVAEGDTLIDRIYHIDRGYECIEEKLQLLGAKIRRVPGMIKQRTLKNIIRATGVGLHSGEKVYLTLKPAPVDTGIVFCRTDLDPVVEIPARAENVGETTMSTTLVKGDVKVDTVEHLLSAMAGLGIDNAYVELSASEVPIMDGSAGPFVFLIQSAGLQEQEAAKKFIRIKREVSVEEGDKRAVFVPFDGFKVSFEIDFDHPVFRGRTQQASVDFSSTSFVKEVSRARTFGFMRDIEYLRSQNLALGGSVENAIVVDENRVLNEDGLRYEDEFVKHKILDAIGDLYLLGNSLIGEFRGFKSGHALNNQLLRTLIADKDAWEVVTFEDARTAPISYMRPAAAV
>26CW_E1 len=119 copies=4 longest=1
GITGTWYNQLGSTFIVTAGADGALTGTYESAVGNAESRYVLTGRYDSAPATDGSGTALGWTVAWKNNYRNAHSATTWSGQYVGGAEARINTQWLLTSGTTEANAWKSTLVGHDTFTKVK
>26DE_E1 len=143 copies=1 longest=0
MATLENTARQYPLFIEQTLDFSDLVDGQAATLTLELQNGAILVPEISKLLVTTASNAAGAATLDIGVIEPGLAADPDGIAADLDLKTTGAKVLAPAKLTYPNGAVITFTATASTGGTAGKFKFLLAYIVEGRGNETYGSPAYN
>26DE_E2 len=376 copies=1 longest=1
MPLPVQGNYTNFARLTNEQKTVWSLQFWRQARNAAFINMFLGTDANSMIQQITELRRDEKGARAVITLIADMVGDGVVGDNQLEGNEEALTAFDTVIQLDQMRAANVHEGRMADQRSIVNFRTTSRDMLAYWLADRMDQLAFLSLAGVSYAYRTNGALRGSSPFPNLTFAADVTPPSANRRLRWDGTNKVLVPNAATSDVTAADTPSYALLVNLKAYAKTKYIRGLRGDGGEEMYHVFLDPLAMAKLKLDPDYIANLRSGYTRGNVNPLFKGGIVTVDGLVIHEFRHVYNTRGMAPGAKWGASGNVDGCSMLFCGAQALGFADIGNPRWVEKEFDYDNKHGISVAKILGFLKPQFPSIYEDGNTEDFGVINVYVAA
>26DK_E1 len=197 copies=1 longest=1
MTCTIDLQILQGKTFEFAFRWADPIKLYKPITGATATAPVVLTVPGHGLSDGWPFEVQSVKAPPELNTQPGQFHLAKVLTPDSLELNDVNGIGFKAFGGAGTIVFYTPADLTGLAARFAIRRTPGDADALLTGNTTDGRVVIDIPTSTISMLIGADVTATLDWNKALYDLELYDPTDVSVVYPVGSGRVTVTAEVAK
>26GP_E1 len=620 copies=1 longest=1
MANGLRKTGQTNTFPERPPVPGRPAYCYVERRIVPNVNGGQPIGDDYFGSHNANSATNSGSYTVGVPQYTDKVPQYTSMWKVVDVTICVPAVPSEPGIPGKTIVDPKTGWNAGGRSVTAVEGDAILKFTVPDKPTGVLVGFSDGRFDNSFGHPTHAFAFYPDRFRIYDRGVIIFEDFFRFTNRYAMSRVGGRVTFYEVNPNDDSSLRPIVNGITPSGGTIYADATLYSVGDFVNDPSFGGNSSAQLNALVPRLRPTIYEATYAGVDSELPRMVLVADAVEPQVINGIVPMPLGNISNYSGATLVRTLPRLVLQAGTSDFVAPWSYVSAHIPRPFGSTTILTGEVLGLDNTIPTPQRLISDRPYAEAGGKWPSRYIIDVWENLWPPGTLMVPEPMPVYSTAMLDAPVLLMVTDGMEISGTIDLIMLVSLDAYDYLGLNTTFTLESTIELMMRDGLAVYSNISTARETALQYAVNVLTGALTTYQNFGFTGFARVGNDTFGIKNDGLYRLEGETDDGELLRALIDFGADDYGVKNTKRLSSAYAGITTDGRVYFRITGDDGEERVYTLAGESPEFRTPDGVAKGVAARHWRLRLEVEDASFADLDNIEVEVGVSSRRIHRRN
>26PF_E1 len=120 copies=1 longest=0
VQLVQSGAEVKKPGASVKVSCKASGYTFTTYYLHWMRQAPGEGLEWVGLINPSVGSASSAQKFKGRVTMTSDTSTSIAYLEVNSLTSEDTAVYYCARVGPSRYSTSSPYWGQGTLVTVSS
>26PF_E2 len=109 copies=1 longest=0
QPVLTQSPSASGTPGQRVTISCSGSSSNIGSNTVNWYQQLPGTAPKLLIYGNNQRPSGVPDRFSGSKSGTSASLAISGLQSDDEADYYCSSYTSSSSLVFGGGTKLTVL
>26PF_E3 len=864 copies=1 longest=1
MDAMKRGLCCVLLLCGAVFVSPSQEIHARFRRGARGSHHHHHHHHHHEAALLVCQYTIQSLIHLTGEDPGFFNVEIPEFPFYPTCNVCTADVNVTINFDVGGKKHQLDLDFGQLTPHTKAVYQPRGAFGGSENATNLFLLELLGAGELALTMRSKKLPINVTTGEEQQVSLESVDVYFQDVFGTMWCHHAEMQNPVYLIPETVPYIKWDNCNSTNITAVVRAQGLDVTLPLSLPTSAQDSNFSVKTQMLGNEIDIECIMEDGEISQVLPGDNKFNITCSGYESHVPSGGILTSTSPVATPIPGTGYAYSLRLTPRPVSRFLGNNSILYVFYSGNGPKASGGDYCIQSNIVFSDEIPASQDMPTNTTDITYVGDNATYSVPMVTSEDANSPNVTVTAFWAWPNNTETDFKCKWTLTSGTPSGCENISGAFASNRTFDITVSGLGTAPKTLIITRTATNATTTTHKVIFSKAPESTTTSPTLNTTGFADPNTTTGLPSSTHVPTNLTAPASTGPTVSTADVTSPTPAGTTSGASPVTPSPSPRDNGTESKAPDMTSPTSAVTTPTPNGTSPTPAMTTPTPNATSPTLGKTSPTSAVTTPTPNATSPTPAVTTPTPNATSPTVGETSPQANATNHTLGGTSPTPVVTSPPKNATSDVTTGQHNRTSSSTSSMSLRPSSIPETTSHMPLLTSAHPTGGENITQVTPASISTHHVSTSSPAPRPGTTSQASGPGNSSTSTKPGEVNVTKGTPPKNATSPQAPSGQKTAVPTVTSTGGKANSTTGGKHTTGHGARTSTEPTTDYGDDSTTPRPRYNATTYLPPSTSSKLRPRWTFTSPPVTTAQATVPVPPTSQPRFSNLHHHHHHHHHH
>26QF_E1 len=394 copies=1 longest=1
MALLDLALEGMAVFGFVLFLVLWLMHFMAIIYTRLHLNKKATDKQPYSKLPGVSLLKPLKGVDPNLINNLETFFELDYPKYEVLLCVQDHDDPAIDVCKKLLGKYPNVDARLFIGGKKVGINPKINNLMPGYEVAKYDLIWICDSGIRVIPDTLTDMVNQMTEKVGLVHGLPYVADRQGFAATLEQVYFGTSHPRYYISANVTGFKCVTGMSCLMRKDVLDQAGGLIAFAQYIAEDYFMAKAIADRGWRFAMSTQVAMQNSGSYSISQFQSRMIRWTKLRINMLPATIICEPISECFVASLIIGWAAHHVFRWDIMVFFMCHCLAWFIFDYIQLRGVQGGTLCFSKLDYAVAWFIRESMTIYIFLSALWDPTISWRTGRYRLRCGGTAEEILDV
>26QG_E1 len=394 copies=1 longest=1
MALLDLALEGMAVFGFVLFLVLWLMHFMAIIYTRLHLNKKATDKQPYSKLPGVSLLKPLKGVDPNLINNLETFFELDYPKYEVLLCVQDHDDPAIDVCKKLLGKYPNVDARLFIGGKKVGINPKINNLMPGYEVAKYDLIWICDSGIRVIPDTLTDMVNQMTEKVGLVHGLPYVADRQGFAATLEQVYFGTSHPRYYISANVTGFKCVTGMSCLMRKDVLDQAGGLIAFAQYIAEDYFMAKAIADRGWRFAMSTQVAMQNSGSYSISQFQSRMIRWTKLRINMLPATIICEPISECFVASLIIGWAAHHVFRWDIMVFFMCHCLAWFIFDYIQLRGVQGGTLCFSKLDYAVAWFIRESMTIYIFLSALWDPTISWRTGRYRLRCGGTAEEILDV
>26QH_E1 len=394 copies=1 longest=1
MALLDLALEGMAVFGFVLFLVLWLMHFMAIIYTRLHLNKKATDKQPYSKLPGVSLLKPLKGVDPNLINNLETFFELDYPKYEVLLCVQDHDDPAIDVCKKLLGKYPNVDARLFIGGKKVGINPKINNLMPGYEVAKYDLIWICDSGIRVIPDTLTDMVNQMTEKVGLVHGLPYVADRQGFAATLEQVYFGTSHPRYYISANVTGFKCVTGMSCLMRKDVLDQAGGLIAFAQYIAEDYFMAKAIADRGWRFAMSTQVAMQNSGSYSISQFQSRMIRWTKLRINMLPATIICEPISECFVASLIIGWAAHHVFRWDIMVFFMCHCLAWFIFDYIQLRGVQGGTLCFSKLDYAVAWFIRESMTIYIFLSALWDPTISWRTGRYRLRCGGTAEEILDV
>26QI_E1 len=394 copies=1 longest=1
MALLDLALEGMAVFGFVLFLVLWLMHFMAIIYTRLHLNKKATDKQPYSKLPGVSLLKPLKGVDPNLINNLETFFELDYPKYEVLLCVQDHDDPAIDVCKKLLGKYPNVDARLFIGGKKVGINPKINNLMPGYEVAKYDLIWICDSGIRVIPDTLTDMVNQMTEKVGLVHGLPYVADRQGFAATLEQVYFGTSHPRYYISANVTGFKCVTGMSCLMRKDVLDQAGGLIAFAQYIAEDYFMAKAIADRGWRFAMSTQVAMQNSGSYSISQFQSRMIRWTKLRINMLPATIICEPISECFVASLIIGWAAHHVFRWDIMVFFMCHCLAWFIFDYIQLRGVQGGTLCFSKLDYAVAWFIRESMTIYIFLSALWDPTISWRTGRYRLRCGGTAEEILDV
>26QJ_E1 len=394 copies=1 longest=1
MALLDLALEGMAVFGFVLFLVLWLMHFMAIIYTRLHLNKKATDKQPYSKLPGVSLLKPLKGVDPNLINNLETFFELDYPKYEVLLCVQDHDDPAIDVCKKLLGKYPNVDARLFIGGKKVGINPKINNLMPGYEVAKYDLIWICDSGIRVIPDTLTDMVNQMTEKVGLVHGLPYVADRQGFAATLEQVYFGTSHPRYYISANVTGFKCVTGMSCLMRKDVLDQAGGLIAFAQYIAEDYFMAKAIADRGWRFAMSTQVAMQNSGSYSISQFQSRMIRWTKLRINMLPATIICEPISECFVASLIIGWAAHHVFRWDIMVFFMCHCLAWFIFDYIQLRGVQGGTLCFSKLDYAVAWFIRESMTIYIFLSALWDPTISWRTGRYRLRCGGTAEEILDV
>26QM_E1 len=394 copies=1 longest=1
MALLDLALEGMAVFGFVLFLVLWLMHFMAIIYTRLHLNKKATDKQPYSKLPGVSLLKPLKGVDPNLINNLETFFELDYPKYEVLLCVQDHDDPAIDVCKKLLGKYPNVDARLFIGGKKVGINPKINNLMPGYEVAKYDLIWICDSGIRVIPDTLTDMVNQMTEKVGLVHGLPYVADRQGFAATLEQVYFGTSHPRYYISANVTGFKCVTGMSCLMRKDVLDQAGGLIAFAQYIAEDYFMAKAIADRGWRFAMSTQVAMQNSGSYSISQFQSRMIRWTKLRINMLPATIICEPISECFVASLIIGWAAHHVFRWDIMVFFMCHCLAWFIFDYIQLRGVQGGTLCFSKLDYAVAWFIRESMTIYIFLSALWDPTISWRTGRYRLRCGGTAEEILDV
>26QS_E1 len=391 copies=1 longest=1
MALLDLALEGMAVFGFVLFLVLWLMHFMAIIYTRLHLNKKATDKQPYSKLPGVSLLKPLKGVDPNLINNLETFFELDYPKYEVLLCVQDHDDPAIDVCKKLLGKYPNVDARLFIGGKKVGINPKINNLMPGYEVAKYDLIWICDSGIRVIPDTLTDMVNQMTEKVGLVHGLPYVADRQGFAATLEQVYFGTSHPRYYISANVTGFKCVTGMSCLMRKDVLDQAGGLIAFAQYIAEDYFMAKAIADRGWRFAMSTQVAMQNSGSYSISQFQSRMIRWTKLRINMLPATIICEPISECFVASLIIGWAAHHVFRWDIMVFFMCHCLAWFIFDYIQLRGVQGGTLCFSKLDYAVAWFIRESMTIYIFLSALWDPTISWRTGRYRLRCGGTAEEI
>26QT_E1 len=394 copies=1 longest=1
MALLDLALEGMAVFGFVLFLVLWLMHFMAIIYTRLHLNKKATDKQPYSKLPGVSLLKPLKGVDPNLINNLETFFELDYPKYEVLLCVQDHDDPAIDVCKKLLGKYPNVDARLFIGGKKVGINPKINNLMPGYEVAKYDLIWICDSGIRVIPDTLTDMVNQMTEKVGLVHGLPYVADRQGFAATLEQVYFGTSHPRYYISANVTGFKCVTGMSCLMRKDVLDQAGGLIAFAQYIAEDYFMAKAIADRGWRFAMSTQVAMQNSGSYSISQFQSRMIRWTKLRINMLPATIICEPISECFVASLIIGWAAHHVFRWDIMVFFMCHCLAWFIFDYIQLRGVQGGTLCFSKLDYAVAWFIRESMTIYIFLSALWDPTISWRTGRYRLRCGGTAEEILDV
>26VA_E1 len=624 copies=1 longest=1
MSVGVSTSAPLSPTSGTSVGMSTFSIMDYVVFVLLLVLSLAIGLYHACRGWGRHTVGELLMADRKMGCLPVALSLLATFQSAVAILGVPSEIYRFGTQYWFLGCCYFLGLLIPAHIFIPVFYRLHLTSAYEYLELRFNKTVRVCGTVTFIFQMVIYMGVVLYAPSLALNAVTGFDLWLSVLALGIVCTVYTALGGLKAVIWTDVFQTLVMFLGQLAVIIVGSAKVGGLGRVWAVASQHGRISGFELDPDPFVRHTFWTLAFGGVFMMLSLYGVNQAQVQRYLSSRTEKAAVLSCYAVFPFQQVSLCVGCLIGLVMFAYYQEYPMSIQQAQAAPDQFVLYFVMDLLKGLPGLPGLFIACLFSGSLSTISSAFNSLATVTMEDLIRPWFPEFSEARAIMLSRGLAFGYGLLCLGMAYISSQMGPVLQAAISIFGMVGGPLLGLFCLGMFFPCANPPGAVVGLLAGLVMAFWIGIGSIVTSMGSSMSRLEEELRRRLTETGLQRFYSLSYLWYSAHNSTTVIVVGLIVSLLTGRMRGRSLNPATIYPVLPKLLSLLPLSCQKRLHCRSYGQDHLDTGLFPEKPRNGVLGDSRDKEAMALDGTAYQGSSSTCILQETSLSNSLEVLFQ
>26VB_E1 len=624 copies=1 longest=1
MSVGVSTSAPLSPTSGTSVGMSTFSIMDYVVFVLLLVLSLAIGLYHACRGWGRHTVGELLMADRKMGCLPVALSLLATFQSAVAILGVPSEIYRFGTQYWFLGCCYFLGLLIPAHIFIPVFYRLHLTSAYEYLELRFNKTVRVCGTVTFIFQMVIYMGVVLYAPSLALNAVTGFDLWLSVLALGIVCTVYTALGGLKAVIWTDVFQTLVMFLGQLAVIIVGSAKVGGLGRVWAVASQHGRISGFELDPDPFVRHTFWTLAFGGVFMMLSLYGVNQAQVQRYLSSRTEKAAVLSCYAVFPFQQVSLCVGCLIGLVMFAYYQEYPMSIQQAQAAPDQFVLYFVMDLLKGLPGLPGLFIACLFSGSLSTISSAFNSLATVTMEDLIRPWFPEFSEARAIMLSRGLAFGYGLLCLGMAYISSQMGPVLQAAISIFGMVGGPLLGLFCLGMFFPCANPPGAVVGLLAGLVMAFWIGIGSIVTSMGSSMSRLEEELRRRLTETGLQRFYSLSYLWYSAHNSTTVIVVGLIVSLLTGRMRGRSLNPATIYPVLPKLLSLLPLSCQKRLHCRSYGQDHLDTGLFPEKPRNGVLGDSRDKEAMALDGTAYQGSSSTCILQETSLSNSLEVLFQ
>26VE_E1 len=624 copies=1 longest=1
MSVGVSTSAPLSPTSGTSVGMSTFSIMDYVVFVLLLVLSLAIGLYHACRGWGRHTVGELLMADRKMGCLPVALSLLATFQSAVAILGVPSEIYRFGTQYWFLGCCYFLGLLIPAHIFIPVFYRLHLTSAYEYLELRFNKTVRVCGTVTFIFQMVIYMGVVLYAPSLALNAVTGFDLWLSVLALGIVCTVYTALGGLKAVIWTDVFQTLVMFLGQLAVIIVGSAKVGGLGRVWAVASQHGRISGFELDPDPFVRHTFWTLAFGGVFMMLSLYGVNQAQVQRYLSSRTEKAAVLSCYAVFPFQQVSLCVGCLIGLVMFAYYQEYPMSIQQAQAAPDQFVLYFVMDLLKGLPGLPGLFIACLFSGSLSTISSAFNSLATVTMEDLIRPWFPEFSEARAIMLSRGLAFGYGLLCLGMAYISSQMGPVLQAAISIFGMVGGPLLGLFCLGMFFPCANPPGAVVGLLAGLVMAFWIGIGSIVTSMGSSMSRLEEELRRRLTETGLQRFYSLSYLWYSAHNSTTVIVVGLIVSLLTGRMRGRSLNPATIYPVLPKLLSLLPLSCQKRLHCRSYGQDHLDTGLFPEKPRNGVLGDSRDKEAMALDGTAYQGSSSTCILQETSLSNSLEVLFQ
>26WC_E1 len=187 copies=2 longest=1
MMTPENDEEQTSVFSATVYGDKIQGKNKRKRVIGLCIRISMVISLLSMITMSAFLIVRLNQCMSANKAAITDSAVAVAAASSTHRKVVSSTTQYDHKESCNGLYYQGSCYILHSDYKSFEDAKANCAAESSTLPNKSDVLTTWLIDYVEDTWGSDGNPITKTTSDYQDSDVSQEVRKYFCTHHHHHH
>26WC_E2 len=116 copies=2 longest=0
QVQLKQSGPELVKPGASVKVSCKASGYAFTNYNMYWVKQSHGKSLEWIGYIDPYNGGTSYNQKFKGKVTLTVDKSSSTAYMHLNSLTSEDSAVYYCARRASMDYWGQGTTLTVSSA
>26WC_E3 len=107 copies=2 longest=0
DIQMTQSPSSMYASLGESVTITCKASQDINSYLSWFQQKPGKSPKTLIYRANRLVDGVPSRFSGSGSGQDYSLTISSLEYEDMGIYYCLQYDEFPYTFGGGTKLEIK
>28IZ_E1 len=267 copies=2 longest=1
MKRQNVRTLSLIVCTFTYLLVGAAVFDALESDHEMREEEKLKAEEIRIKGKYNISSEDYRQLELVILQSEPHRAGVQWKFAGSFYFAITVITTIGYGHAAPGTDAGKAFCMFYAVLGIPLTVVMFQSLGERMNTFVRYLLKRIKKCCGMRNTDVSMENMVTVGFFSCMGTLCIGAAAFSQCEEWSFFHAYYYCFITLTTIGFGDYVALQTKGALQKKPLYVAFSFMYILVGLTVIGAFLNLVVLRFLTMNSEDERRDAEAELEVLFQ
>28QL_E1 len=163 copies=1 longest=0
MSDLGKKLLEAARAGQDDEVRILMANGADVNAADDVGVTPLHLAAQRGHLEIVEVLLKYGADVNAADLWGQTPLHLAATAGHLEIVEVLLKNGADVNARDNIGHTPLHLAAWAGHLEIVEVLLKYGADVNAQDKFGKTPKDLARDNGNEWIRELLEKAERKLK
>28QL_E2 len=227 copies=1 longest=1
KGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATNGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGTYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIKANFKIRHNVEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGI
>28QM_E1 len=373 copies=1 longest=1
KIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPAAAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSAVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELAKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDAALAAAQTNGSHHAH
>28QM_E2 len=159 copies=1 longest=0
DLGRKLLEAARAGQDDEVRILMANGADVNAADNTGTTPLHLAAYSGHLEIVEVLLKHGADVDASDVFGYTPLHLAAYWGHLEIVEVLLKNGADVNAMDSDGMTPLHLAAKWGYLEIVEVLLKHGADVNAQDKFGKTPKDLARDNGNQWIYELLEAAGRA
>28QN_E1 len=160 copies=1 longest=0
ADLGRKLLEAARAGQDDEVRILMANGADVNAADNTGTTPLHLAAYSGHLEIVEVLLKHGADVDASDVFGYTPLHLAAYWGHLEIVEVLLKNGADVNAMDSDGMTPLHLAAKWGYLEIVEVLLKHGADVNAQDKFGKTPKDLARDNGNQWIYELLEAAGRA
>28QN_E2 len=374 copies=1 longest=1
MKIQEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELAKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTNGSHHHH
>28VJ_E1 len=70 copies=1 longest=0
GGMYEVIVRNAPRSFVKEVREETGAKVSRTYINLNRISAVFTTFTHERKEDAEAIAERARARGLEVFLVE
>28VJ_E2 len=489 copies=1 longest=1
MNKQIFVLYFNIFLIFLGIGLVIPVLPVYLKDLGLTGSDLGLLVAAFALSQMIISPFGGTLADKLGKKLIICIGLILFSVSEFMFAVGHNFSVLMLSRVIGGMSAGMVMPGVTGLIADISPSHQKAKNFGYMSAIINSGFILGPGIGGFMAEVSHRMPFYFAGALGILAFIMSIVLIHDPKKSTTSGFQKLEPQLLTKINWKVFITPVILTLVLSFGLSAFETLYSLYTADKVNYSPKDISIAITGGGIFGALFQIYFFDKFMKYFSELTFIAWSLLYSVVVLILLVFANDYWSIMLISFVVFIGFDMIRPAITNYFSNIAGERQGFAGGLNSTFTSMGNFIGPLIAGALFDVHIEAPIYMAIGVSLAGVVIVLIEKQHRAAAADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYL
>28WZ_E1 len=687 copies=1 longest=1
KIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELAKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTSSGLEVLFQGPSRKKMAALECEDPERELKKQKRAARFQHGHSRRLRLEPLVLQMSSLESSGADPDWQELQIVGTCPDITKHYLRLTCAPDPSTVRPVAVLKKSLCMVKCHWKEKQDYAFACEQMKSIRQDLTVQGIRTEFTVEVYETHARIALEKGDHEEFNQCQTQLKSLYAENLPGNVGEFTAYRILYYIFTKNSGDITTELAYLTRELKADPCVAHALALRTAWALGNYHRFFRLYCHAPCMSGYLVDKFADRERKVALKAMIKTFRPALPVSYLQAELAFEGEAACRAFLEPLGLAYTGPDNSSIDCRLSLAQLSAF
>28WZ_E2 len=456 copies=1 longest=0
GKPIPNPLLGLDSTGSGKPIPNPLLGLDSTGSGKPIPNPLLGLDSTSSGLEVLFQGPMAHITINQYLQQVYEAIDSRDGASCAELVSFKHPHVANPRLQMASPEEKCQQVLEPPYDEMFAAHLRCTYAVGNHDFIEAYKCQTVIVQSFLRAFQAHKEENWALPVMYAVALDLRVFANNADQQLVKKGKSKVGDMLEKAAELLMSCFRVCASDTRAGIEDSKKWGMLFLVNQLFKIYFKINKLHLCKPLIRAIDSSNLKDDYSTAQRVTYKYYVGRKAMFDSDFKQAEEYLSFAFEHCHRSSQKNKRMILIYLLPVKMLLGHMPTVELLKKYHLMQFAEVTRAVSEGNLLLLHEALAKHEAFFIRCGIFLILEKLKIITYRNLFKKVYLLLKTHQLSLDAFLVALKFMQVEDVDIDEVQCILANLIYMGHVKGYISHQHQKLVVSKQNPFPPLSTVC
>28WZ_E3 len=70 copies=1 longest=0
MSEKKQPVDLGLLEEDDEFEEFPAEDWAGLDEDEDAHVWEDNWDDDNVEDDFSNQLRAELEKHGYKMETS
>31CJ_E1 len=534 copies=2 longest=1
WSHPQFEKGGGSGGGSGGSAWSHPQFEKMSSHPYVTQQNTPLADDTTLMSTTDLQSYITHANDTFVQVSGYTLQELQGQPHNMVRHPDMPKAAFADMWFTLKKGEPWSGIVKNRRKNGDHYWVRANAVPMVREGKISGYMSIRTRATDEEIAAVEPLYKALNAGRTSKRIHKGLVVRKGWLGKLPSLPLRWRARGVMTLMFILLAAMLWFVAAPVVTYILCALVVLLASACFEWQIVRPIENVAHQALKVATGERNSVEHLNRSDELGLTLRAVGQLGLMCRWLINDVSSQVSSVRNGSETLAKGTDELNEHTQQTVDNVQQTVATMNQMAASVKQNSATASAADKLSITASNAAVQGGEAMTTVIKTMDDIADSTQRIGTITSLINDIAFQTNILALNAAVEAARAGEQGKGFAVVAGEVRHLASRSANAANDIRKLIDASADKVQSGSQQVHAAGRTMEDIVAQVKNVTQLIAQISHSTLEQADGLSSLTRAVDELNLITQKNAELVEESAQVSAMVKHRASRLEDAVTVLH
>31CK_E1 len=534 copies=2 longest=1
WSHPQFEKGGGSGGGSGGSAWSHPQFEKMSSHPYVTQQNTPLADDTTLMSTTDLQSYITHANDTFVQVSGYTLQELQGQPHNMVRHPDMPKAAFADMWFTLKKGEPWSGIVKNRRKNGDHYWVRANAVPMVREGKISGYMSIRTRATDEEIAAVEPLYKALNAGRTSKRIHKGLVVRKGWLGKLPSLPLRWRARGVMTLMFILLAAMLWFVAAPVVTYILCALVVLLASACFEWQIVRPIENVAHQALKVATGERNSVEHLNRSDELGLTLRAVGQLGLMCRWLINDVSSQVSSVRNGSETLAKGTDELNEHTQQTVDNVQQTVATMNQMAASVKQNSATASAADKLSITASNAAVQGGEAMTTVIKTMDDIADSTQRIGTITSLINDIAFQTNILALNAAVEAARAGEQGKGFAVVAGEVRHLASRSANAANDIRKLIDASADKVQSGSQQVHAAGRTMEDIVAQVKNVTQLIAQISHSTLEQADGLSSLTRAVDELNLITQKNAELVEESAQVSAMVKHRASRLEDAVTVLH
>31ZN_E1 len=311 copies=1 longest=0
SNAMAGKKNVLSSLAVYAEDSEPESDGEAGIEAVGSAAEEKGGLVSDAYGEDDFSRLGGDEDGYEEEEDENSRQSEDDDSETEKPEADDPKDNTEAEKRDPQELVASFSERVRNMSPDEIKIPPEPPGRCSNHLQDKIQKLYERKIKEGMDMNYIIQRKKEFRNPSIYEKLIQFCAIDELGTNYPKDMFDPHGWSEDSYYEALAKAQKIEMDKLEKAKKERTKIEFVTGTKKGTTTNATSTTTTTASTAVADAQKRKSKWDSAIPVTTIAQPTILTTTATLPAVVTVTTSASGSKTTVISAVGTIVKKAKQ
>31ZN_E2 len=442 copies=1 longest=1
SNAMSEDEERENENHLLVVPESRFDRDSGESEEAEEEVGEGTPQSSALTEGDYVPDSPALSPIELKQELPKYLPALQGCRSVEEFQCLNRIEEGTYGVVYRAKDKKTDEIVALKRLKMEKEKEGFPITSLREINTILKAQHPNIVTVREIVVGSNMDKIYIVMNYVEHDLKSLMETMKQPFLPGEVKTLMIQLLRGVKHLHDNWILHRDLKTSNLLLSHAGILKVGDFGLAREYGSPLKAYTPVVVTLWYRAPELLLGAKEYSTAVDMWSVGCIFGELLTQKPLFPGKSEIDQINKVFKDLGTPSEKIWPGYSELPAVKKMTFSEHPYNNLRKRFGALLSDQGFDLMNKFLTYFPGRRISAEDGLKHEYFRETPLPIDPSMFPTWPAKSEQQRVKRGTSPRPPEGGLGYSQLGDDDLKETGFHLTTTNQGASAAGPGFSLKF
>31ZN_E3 len=322 copies=1 longest=0
SNAMAAAAAAAGAAGSAAPAAAAGAPGSGGAPSGSQGVLIGDRLYSGVLITLENCLLPDDKLRFTPSMSSGLDTDTETDLRVVGCELIQAAGILLRLPQVAMATGQVLFQRFFYTKSFVKHSMEHVSMACVHLASKIEEAPRRIRDVINVFHRLRQLRDKKKPVPLLLDQDYVNLKNQIIKAERRVLKELGFCVHVKHPHKIIVMYLQVLECERNQHLVQTSWNYMNDSLRTDVFVRFQPESIACACIYLAARTLEIPLPNRPHWFLLFGATEEEIQEICLKILQLYARKKVDLTHLEGEVEKRKHAIEEAKAQARGLLPGG
>32PZ_E1 len=146 copies=1 longest=0
MGAWLFILAVVVNCINLFGQVHFTILYADLEADYINPIELCSKVNKLITPEAALHGALSLLFLLNGYWFVFLLNLPVLAYNLNKIYNKVQLLDATEIFRTLGKHKRESFLKLGFHLLMFFFYLYRMIMALIAESGDDFDYKDDDDK
>32PZ_E2 len=549 copies=1 longest=1
MAGATSSIIRENDFEDELAESMQSYNRETADKLALTRTESVKPEPEITAPPHSRFSRSFKTVLIAQCAFTGFFSTIAGAIYYPVLSVIERKFDIDEELVNVTVVVYFVFQGLAPTFMGGFADSLGRRPVVLVAIVIYFGACIGLACAQTYAQIIVLRCLQAAGISPVIAINSGIMGDVTTRAERGGYVGYVAGFQVLGSAFGALIGAGLSSRWGWRAIFWFLAIGSGICFLASFLILPETKRNISGNGSVTPKSYLNRAPILVLPTVRKSLHLDNPDYETLELPTQLNLLAPFKILKAYEICILMLVAGLQFAMYTTHLTALSTALSKQYHLTVAKVGLCYLPSGICTLCSIVIAGRYLNWNYRRRLKYYQNWLGKKRSKLLEEHDNDLNLVQRIIENDPKYTFNIFKARLQPAFVTLLLSSSGFCAYGWCITVKAPLAAVLCMSGFASLFSNCILTFSTTLIVDLFPTKTSTATGCLNLFRCILSAVFIAALSKMVEKMKFGGVFTFLGALTSSSSILLFILLRKGKELAFKRKKQELGVNGENLYFQ
>32QB_E1 len=146 copies=1 longest=0
MGAWLFILAVVVNCINLFGQVHFTILYADLEADYINPIELCSKVNKLITPEAALHGALSLLFLLNGYWFVFLLNLPVLAYNLNKIYNKVQLLDATEIFRTLGKHKRESFLKLGFHLLMFFFYLYRMIMALIAESGDDFDYKDDDDK
>32QB_E2 len=549 copies=1 longest=1
MAGATSSIIRENDFEDELAESMQSYNRETADKLALTRTESVKPEPEITAPPHSRFSRSFKTVLIAQCAFTGFFSTIAGAIYYPVLSVIERKFDIDEELVNVTVVVYFVFQGLAPTFMGGFADSLGRRPVVLVAIVIYFGACIGLACAQTYAQIIVLRCLQAAGISPVIAINSGIMGDVTTRAERGGYVGYVAGFQVLGSAFGALIGAGLSSRWGWRAIFWFLAIGSGICFLASFLILPETKRNISGNGSVTPKSYLNRAPILVLPTVRKSLHLDNPDYETLELPTQLNLLAPFKILKAYEICILMLVAGLQFAMYTTHLTALSTALSKQYHLTVAKVGLCYLPSGICTLCSIVIAGRYLNWNYRRRLKYYQNWLGKKRSKLLEEHDNDLNLVQRIIENDPKYTFNIFKARLQPAFVTLLLSSSGFCAYGWCITVKAPLAAVLCMSGFASLFSNCILTFSTTLIVDLFPTKTSTATGCLNLFRCILSAVFIAALSKMVEKMKFGGVFTFLGALTSSSSILLFILLRKGKELAFKRKKQELGVNGENLYFQ
>33AS_E1 len=488 copies=1 longest=1
MDPQDLYTWEPKGLAVVDMALAQESAGLVMLYHFDGYIDAGETGDQIVDQVLDSLPHQVVARFDHDRLVDYRARRPLLTFKRDTWSDYEEPTIEVRLVQDATGAPFLFLSGPEPDVEWERFAAAVGQIVERLGVRLSVSFHGIPMGVPHTRPVGITPHGSRTDLVPGHRSPFEEAQVPGSAEALVEYRLAQAGHDVLGVAAHVPHYVARSAYPDAALTVLEAITAATGLVLPGIAHSLRTDAHRTQTEIDRQIQEGDEELIALVRGLERRADGERIDAQLKALKVIPVIAIDNAEDIIPLGKVLAENGLPAAEITFRSDAAVEAIRLLRQAQPEMLIGAGTILNGEQALAAKEAGATFVVSPGFNPNTVRACDEIGIPIVPGVNNPSTVEAALEMGITTLKFFPAEASGGISMVKSLVGPYGDIRLMPTGGITPSNIDNYLAIPQVLACGGTWMVDKKLVTNGEWDEIARLTREIVEQVNPGHHHHHH
>33AU_E1 len=488 copies=1 longest=1
MDPQDLYTWEPKGLAVVDMALAQESAGLVMLYHFDGYIDAGETGDQIVDQVLDSLPHQVVARFDHDRLVDYRARRPLLTFKRDTWSDYEEPTIEVRLVQDATGAPFLFLSGPEPDVEWERFAAAVGQIVERLGVRLSVSFHGIPMGVPHTRPVGITPHGSRTDLVPGHRSPFEEAQVPGSAEALVEYRLAQAGHDVLGVAAHVPHYVARSAYPDAALTVLEAITAATGLVLPGIAHSLRTDAHRTQTEIDRQIQEGDEELIALVRGLERRADGERIDAQLKALKVIPVIAIDNAEDIIPLGKVLAENGLPAAEITFRSDAAVEAIRLLRQAQPEMLIGAGTILNGEQALAAKEAGATFVVSPGFNPNTVRACDEIGIPIVPGVNNPSTVEAALEMGITTLKFFPAEASGGISMVKSLVGPYGDIRLMPTGGITPSNIDNYLAIPQVLACGGTWMVDKKLVTNGEWDEIARLTREIVEQVNPGHHHHHH
>33AV_E1 len=488 copies=1 longest=1
MDPQDLYTWEPKGLAVVDMALAQESAGLVMLYHFDGYIDAGETGDQIVDQVLDSLPHQVVARFDHDRLVDYRARRPLLTFKRDTWSDYEEPTIEVRLVQDATGAPFLFLSGPEPDVEWERFAAAVGQIVERLGVRLSVSFHGIPMGVPHTRPVGITPHGSRTDLVPGHRSPFEEAQVPGSAEALVEYRLAQAGHDVLGVAAHVPHYVARSAYPDAALTVLEAITAATGLVLPGIAHSLRTDAHRTQTEIDRQIQEGDEELIALVRGLEARYDGAVIDAQLKALKVIPVIAIDNAEDIIPLGKVLAENGLPAAEITFRSDAAVEAIRLLRQAQPEMLIGAGTILNGEQALAAKEAGATFVVSPGFNPNTVRACREIGIPIVPGVNNPSTVEAALEMGLTTLKFFPAEASGGISMVKSLVGPYGDIRLMPTGGITPSNIDNYLAIPQVLACGGTWMVDKKLVTNGEWDEIARLTREIVEQVNPGHHHHHH
>33AW_E1 len=482 copies=1 longest=1
MREYEPGQPGMYELEFPAPQLSSSDGRGPVLVHALEGFSDAGHAIRLAAAHLKAALDTELVASFAIDELLDYRSRRPLMTFKTDHFTHSDDPELSLYALRDSIGTPFLLLAGLEPDLKWERFITAVRLLAERLGVRQTIGLGTVPMAVPHTRPITMTAHSNNRELISDFTPSISEIQVPGSASNLLEYRMAQHGHEVVGFTVHVPHYLTQTDYPAAAQALLEQVAKTGSLQLPLAVLAEAAAEVQAKIDEQVQASAEVAQVVAALERQYAAAQAKSAVARRLGKVTASRVADVMTKTKSGYAASRQNYMAELIAQRLTGTQEIRFSNAAMQRGTELEPHARARYIIETGEIVTEVGLIDHPTIAGFGASPDGLVGDTGLIEIKCPNTWTHIETIKTGKPKPEYIKQMQTQMACTGRQWCDFVSYDDRLPDDMQYFRTRIERDDALIAEIETEVSAFLAELEAEIEYLKRKAAKLAGHHHHHH
>35TL_E1 len=290 copies=1 longest=1
MRIFAVFIFMTYWHLLNAFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYNKINQRILVVDPVTSEHELTCQAEGYPKAEVIWTSSDHQVLSGKTTTTNSKREEKLFNVTSTLRINTTTNEIFYCTFRRLDPEENHTAELVIPELPLAHPPNERTHLVILGAILLCLGVALTFIFRLRKGRMMDVKKCGIQDTNSKKQSDTHLEET
>35TL_E2 len=223 copies=1 longest=0
EVQLVESGGGLVQPGGSLRLSCAASSIDLEMYKIHWVRQAPGKGLEWVARINVGFDVTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCWPYSIVMLPFHGYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSC
>35TL_E3 len=214 copies=1 longest=0
DIQMTQSPSSLSASVGDRVTITCRASQDVGKDVAWYQQKPGKAPKLLIYYYKFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCLNANTGGITFGQGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>36BB_E1 len=472 copies=1 longest=1
MDWTWRILFLVAAATGTHAQMQLVQSGPEVKKPGTSVKVSCKASGFTFWSSAVQWVRQARGQRLEWIGWIVIASGETNYAQKFQERVTITRDMSTSTAYMELSSLRSEDTAVYYCAAPYCSSRECNNGFDIWGQGTMVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDKTHTCPPCPAPELLGGPSVFLFPPKPKDTLMISRTPEVTCVVVDVSHEDPEVKFNWYVDGVEVHNAKTKPREEQYNSTYRVVSVLTVLHQDWLNGKEYKCKVSNKALPAPIEKTISKAKGQPREPQVYTLPPSRDELTKNQVSLTCLVKGFYPSDIAVEWESNGQPENNYKTTPPVLDSDGSFFLYSKLTVDKSRWQQGNVFSCSVMHEALHNHYTQKSLSLSPGK
>36BB_E2 len=236 copies=1 longest=0
MVLQTQVFISLLLWISGAYGEIVLTQSPGTLSLSPGERATLSCRASQSVSSMDLAWYQQKPGQAPRLLIYYASSRATGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQHYGGMRGWTFGQGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSNTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>36BB_E3 len=197 copies=1 longest=0
PNITNLCPFDEVFNATRFASVYAWNRKRISNCVADYSVLYNFAPFFAFKCYGVSPTKLNDLCFTNVYADSFVIRGNEVSQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDGKVSGNYNYRYRLFRKSNLKPFERDISTEIYQAGNKPCNGVAGVNCYFPLQSYSFRPTYGVGHQPYRVVVLSFELLHAPATVCG
>36CQ_E1 len=38 copies=1 longest=0
MSKGKKRSGARPGRPQPLRGTKGKRKGARLWYVGGQQF
>36CQ_E2 len=427 copies=1 longest=1
MSNIQTGAERMPHDLSHLGFLAGQIGRLITISTTPVIAGDSFEMDAVGALRLSPLRRGLAIDSTVDIFTFYVPHRHVYGEQWIKFMKDGVNATPLPTVNTTGYIDHAAFLGTINPDTNKIPKHLFQGYLNIYNNYFKAPWMPDRTEANPNELNQDDARYGFRCCHLKNIWTAPLPPETELSRQMTTSTTSIDIMGLQAAYANLHTDQERDYFMQRYHDVISSFGGKTSYDADNRPLLVMRSNLWASGYDVDGTDQTSLGQFSGRVQQTYKHSVPRFFVPEHGTMFTLALVRFPPTATKEIQYLNAKGALTYTDIAGDPVLYGNLPPREISMKDVFRSGDSSKKFKIAEGQWYRYAPSYVSPAYHLLEGFPFIQEPPSGDLQERVLIRHHDYDQCFQSVQLLQWNSQVKFNVTVYRNLPTTRDSIMTS
>36CQ_E3 len=175 copies=1 longest=0
MFQTFISRHNSNFFSDKLVLTSVTPASSAPVLQTPKATSSTLYFDSLTVNAGNGGFLHCIQMDTSVNAANQVVSVGADIAFDADPKFFACLVRFESSSVPTTLPTAYDVYPLDGRHDGGYYTVKDCVTIDVLPRTPGNNVYVGFMVWSNFTATKCRGLVSLNQVIKEIICLQPLK
>43JF_E1 len=750 copies=1 longest=1
MSDVLRSTTFQLEFKGQDGITGIKQFTRAVSDADKTVEELSATLGDNVEVTYKNVQSKQELTAEARALVSQMERTNARVKEMTSLYQHQASMIGKTAQEQEVLNAVYKLGANATQEQKQQVTQLVQNYQLLRDGSNQTEGSFRNLRGVSQQLGWQLQDVAVQAQLGTSAFVIFSQQGSQLAAAFGPTGALVGAVIAVAGAIGGALVNSMGVAGEEIDKLIGKVDSLGKATKELAAINITKEIKEQEKELKNLNAQIAFSNKYQEENGKLTGNGRASYLKMVSASEQLSDELKKNKEALDTINKSFTQNIKDTKEAEEATKKLLESYGLKTELLGKTDREQAKLNATTELGSAATEDQIAAVMAGIDVYYNEYDALKAREKAIKDAEKAEKDSAAESRRISQQRERAFQAETLSLIKQTETTQEEYNRRKKIIDEYVKYEGTNQRTDEAYAALEQWRTQKLAEEYKKREAVRKQIEEAQRKQGGRDDPAGFENDTYARNLQLLTDQKKALGEAELTERQRIDTLIEAETERHVARLNEISNTQLEGQLQNYATFTGSMGKVFGQLQQFAEEGSKEAAALFYINQAIALAETIVNTELAATKAMGQLGVFGIPASTLIRATGYASAGIIAGQTIAGAYDKGGNIPSGQLGIVSEYGDELVNGVLVEGPARVTSREDTAKMMNGGGGSVSIVIENRIDGASYREERIDENTVRIIAERVFNDNIDSGVSSVLGNRNSRSTKQMKNSFNVKGKV
>43JG_E1 len=164 copies=1 longest=1
MATKGDISDLNVLLYGGQPMVPLVEGFTRSRQSGVVRSDVAGGASRQRKKYFGTTHIASATFYLRSPAQMDYIQLFINNNEGKRWICHLAADRPLVEPYVVQALTDWNHVEVNALRGTVTVQLEIFSARDECLDGLLNILYPCIGDDICEYLELFGTMTEGWPE
>43JH_E1 len=171 copies=1 longest=1
MTDEEIREIYASAPVSKTVIEVVELSAPWFSKTYYLQRQMTDSIEVTLETGETVVAEYVPMSIDQSSSNADLNYERNIVIQQVNDIIASEQTNYDPDVYGDQLPQFTSRGYILYRDGSVSQIKQSPIRLPIRKMRRDERGALFNVTTKPANQWATGEVATTTRVPMLKGFL
>43MU_E1 len=299 copies=1 longest=1
IEDLITEVAQGALTLSLPKQQDSLPDTKASGPAHSKEVPALTAVETGATNPLVPSDTVQTRHVIQRRSRSESTIESFFARGACVAIIEVDNEEPTTRAQKLFATWRITYKDTVQLRRKLEFFTYSRFDMEFTFVVTANFTNTNNGHALNQVYQIMYIPPGAPTPKSWDDYTWQTSSNPSIFYTYGAAPARISVPYVGLANAYSHFYDGFAKVPLKTDANDQIGDSLYSAMTVDDFGVLAIRVVNDHNPTKVTSKVRIYMKPKHVRVWCPRPPRAVPYYGPGVDYKDNLNPLSEKGLTTY
>43MU_E2 len=271 copies=1 longest=0
SPNVEACGYSDRVLQLTLGNSTITTQEAANSVVAYGRWPEFIRDDEANPVDQPTEPDVATCRFYTLDTVMWGKESKGWWWKLPDALRDMGLFGQNMYYHYLGRSGYTVHVQCNASKFHQGALGVFAIPEYCLAGDSDKQRYTSYANANPGEKGGKFYSQFNRDTAVTSPKREFCPVDYLLGCGVLLGNAFVYPHQIINLRTNNSATIVLPYVNALAIDSMVKHNNWGIAILPLSPLDFAQDSSVEIPITVTIAPMCSEFNGLRNVTAPKFQ
>43MU_E3 len=238 copies=1 longest=0
GLPVLNTPGSNQYLTSDNHQSPCAIPEFDVTPPIDIPGEVKNMMELAEIDTMIPLNLENTKRNTMDMYRVTLSDSADLSQPILCLSLSPASDPRLSHTMLGEVLNYYTHWAGSLKFTFLFCGSMMATGKILVAYAPPGAQPPTSRKEAMLGTHVIWDLGLQSSCTMVVPWISNVTYRQTTQDSFTEGGYISMFYQTRIVVPLSTPKSMSMLGFVSACNDFSVRLLRDTTHISQSALPQ
>43YY_E1 len=487 copies=1 longest=1
MKSLNYFWLSLPVLSTLSLPQDVTRCSANTNFRRFFSKFNVQAPAVVVLGGYLPIGENQGVNSTWHCAGQHPTASGVHGIFVSHIRGGHGFEIGISQEPFDPSGYQLYLHKATNGNTNATARLRICQFPSIKTLGPTANNDVTTGRNCLFNKAIPAHMSEHSVVGITWDNDRVTVFSDKIYYFYFKNDWSRVATKCYNSGGCAMQYVYEPTYYMLNVTSAGEDGISYQPCTANCIGYSANVFATEPNGHIPEGFSFNNWFLLSNDSTLVHGKVVSNQPLLVNCLLAIPKIYGLGQFFSFNQTIDGVCNGAAAQRAPEALRFNINDTSVILAEGSIVLHTALGTNLSFVCSNSSDPYLATFATPLGATQVPYYCFLKVDTYNSTVYKFLAVLPPTVREIVITKYGDVYVNGFGYLHLGLLDAVTINFTGHGTDDDVSGFWTIASTNFVDALIEVQGTAIQRILYCDDPVSQLKCSQVAFDLDDGFYPI
>43YY_E2 len=144 copies=1 longest=0
MEFRLNWVVLFALLQGVQGEEKLVESGGGLVQPGGSVRLSCVGSGFTFSGHEIIWVRQAPGKGLEWLAGSYSSGSSTYYADSVNGRFTMSRDNSQNTAYLQMNELRTGDAAHYYCARGGVYDVNYLGDDADLWGPGVEVVVSSA
>43YY_E3 len=129 copies=1 longest=0
MAWTVLLIGLLAVGSGVDSQTVIQEPAMSVSPGGTVTLTCAFSSGSVTGSNYPSWFQQTPGQPPRLLMYRTNNRPTGVPGRFSGGISGNKAALLITGAQANDEAVYFCKLHKTSTNGIFGGGTHLTVLG
>46KE_E1 len=197 copies=1 longest=1
MTCTIDLQILQGKTFEFAFRWADPIKLYKPITGATATAPVVLTVPGHGLSDGWPFEVQSVKAPPELNTQPGQFHLAKVLTPDSLELNDVNGIGFKAFGGAGTIVFYTPADLTGLAARFAIRRTPGDADALLTGNTTDGRVVIDIPTSTISMLIGADVTATLDWNKALYDLELYDPTDVSVVYPVGSGRVTVTAEVAK
>9CWP_E1 len=193 copies=1 longest=1
NLCPFHEVFNATRFASVYAWNRTRISNCVADYSVLYNFAPFFAFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDSKHSGNYDYWYRLFRKSKLKPFERDISTEIYQAGNKPCKGKGPNCYFPLQSYGFRPTYGVGHQPYRVVVLSFELLHAPATVCGP
>9CWP_E2 len=148 copies=1 longest=0
ELEEQVMHVLDQVSELAHELLHKLTGEELERAAYFNWWATEMMLELIKSDDEREIREIEEEARRILEHLEELARKGGSEALEELEKALRELKKSTDELERSTEELEKNPSEDALVENNRLIVENNKIIVEVLRIIAKVLKLEHHHHHH
>9CWQ_E1 len=304 copies=1 longest=1
MFVFLVLLPLVSSQCVMPLFNLITTTQSYTNSFTRGVYYPDKVFRSSVLHLTQDLFLPFFSNVTWFHAISGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVFIKVCEFQFCNDPFLDVYHKNNKSWMESESGVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPIIGRDFPQGFSALEPLVDLPIGINITRFQTLLALNRSYLTPGDSSSGWTAGAADYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFT
>9E94_E1 len=313 copies=2 longest=1
MRRGALLAGALAAYAAYLVLGALLVARLEGPHEARLRAELETLRAQLLQRSPCVAAPALDAFVERVLAAGRLGRVVLANASGSANASDPAWDFASALFFASTLITTVGYGYTTPLTDAGKAFSIAFALLGVPTTMLLLTASAQRLSLLLTHVPLSWLSMRWGWDPRRAACWHLVALLGVVVTVCFLVPAVIFAHLEEAWSFLDAFYFCFISLSTIGLGDYVPGEAPGQPYRALYKVLVTVYLFLGLVAMVLVLQTFRHVSDLHGLTELILLPPPCPASFNADEDDRVDILGPQPESHQQLSASSHTDYASIPR
>9FTE_E1 len=426 copies=1 longest=1
SVYAVYAQTDNSSSYNEAFFSAGLICISGLCLAVVSHMFCARHSMFAAKLLTHMMVASLGTQILFLASAYASPHLSEESCSAVAAVAHYLYLCQFSWMLIQSVNFWYVLVVSDEHTERRCLLFCLLSWGLPSFVVILLILILRGIYHRSMPQIYGLIHGDLCFIPNIYAALFTAALVPLMCLVVVFVVFIHAYQLKPQWKGYDDVFRGRTNAAEIPLILYLFALISMTWLWGGLHMAYRHFWMLVLFVIFNSLQGLYVFVVYFILHNQTCCPMKASYTVEMNGHPGPSTAFFTPGSGIPPAGEINKSTQNLINAMEEVPSDWERSSFQQTSQASPDLKTSPQNGASFPSSGGYGPGSLIADEESQEFDDLIFALKTGAGLSVSDNESGQGSQEGGTLTDSQIVELRRIPIADTHLSGRHHHHHHHH
>9FTE_E2 len=147 copies=1 longest=0
MAEVQLVESGGGLVQPGGSLRLSCAASGSSFSINAMGWYRQAPGKQRELVAAISRGGSTNYASSVKGRFTISRDNGKNTVYLQMNSPKPEDTAVYYCNVGSRSTLPIWYDYWGQGTQVTVSSAAAEQKLISEEDLNGAAHHHHHHGS
>9H2Q_E1 len=674 copies=1 longest=1
HHHHHHKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELAKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTRITSLLLDDVDNEMAAIAMQGFRSMIEQFNVNNPATAKELQAMEAQLTAMSDQLVGADGELPAEIQAIKDALAQALKQPSADGLATAMGQVAFAAAKVGGGSAGTAGTVQMNVKQLYKTAFSSTSSSSYAAALSDGYSAYKTLNDLYSESRSGVQDAISQTALPALSRSVSRSGIESQGRSADASQRAAETIVRDSQTLGDVYSRLQVLESLMSIIVSNPQVNQEEIKQKLTAAISKAPQFGYPAVQNSNDSLQKFAAKLEREFVDGERRLAEARELAFEKQPQFIQQVLVNIASLFSGYLS
>9H2Q_E2 len=309 copies=1 longest=0
GGGRGSGKEVRPPATALQVFDAHDGEVNAVQFSPDGRLLATGGNDRRVKLWEVFGGSCEFKGSLTGSNGGITSIEFDPDGKYLLAASNDKACRIWTVDDYRLRHTLTGHSGKVYSAKFSPDGRRIVSGSHDRTLKLWDLRKGTCIKTVFAGSSANDIVCTEQSIMSGHHDKKIRFWDIRTGSIVQEIELQGKITSLDLNPDRTELLSCSRDNTLKIIDLRTNSIKQTFSAPGFKSGSDWTRVVFSPDGSYVAAGSADGSLYIWSVRTGKVEKVLSKHHSSSINAVAWSPSGSHVVSVDKGGKAVLWAQP
>9HJQ_E1 len=401 copies=2 longest=1
MPGDGAGASRQRKLEALIRDPRSPINVESLLDGLNSLVLDLDFPALRKNKNIDNFLNRYEKIVKKIRGLQMKAEDYDVVKVIGRGAFGEVQLVRHKASQKVYAMKLLSKFEMIKRSDSAFFWEERDIMAFANSPWVVQLFYAFQDDRYLYMVMEYMPGGDLVNLMSNYDVPEKWAKFYTAEVVLALDAIHSMGLIHRDVKPDNMLLDKHGHLKLADFGTCMKMDETGMVHCDTAVGTPDYISPEVLKSQGGDGFYGRECDWWSVGVFLYEMLVGDTPFYADSLVGTYSKIMDHKNSLCFPEDAEISKHAKNLICAFLTDREVRLGRNGVEEIRQHPFFKNDQWHWDNIRETAAPVVPELSSDIDSSNFDDIEDDKGDVETFPIPKAFVGNQLPFIGFTYYR
>9I3T_E1 len=492 copies=1 longest=1
MAQQWSLQRLAGRHPQDSYEDSTQSSIFTYTNSNSTRGPFEGPNYHIAPRWVYHLTSVWMIFVVIASVFTNGLVLAATMKFKKLRHPLNWILVNLAVADLAETVIASTISVVNQVYGYFVLGHPMCVLEGYTVSLCGITGLWSLAIISWERWMVVCKPFGNVRFDAKLAIVGIAFSWIWAAVWTAPPIFGWSRYWPHGLKTSCGPDVFSGSSYPGVQSYMIVLMVTCCITPLSIIVLCYLQVWLAIADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLKESESTQKAEKEVTRMVVVMVLAFCFCWGPYAFFACFAAANPGYPFHPLMAALPAFFAKSATIYNPVIYVFMNRQFRNCILQLFGKKVDDGSELSSASKTEVSSVSSVSPALEVLFQGPDYKDDDDKGAGATETSQVAPA
>9I3T_E2 len=231 copies=1 longest=0
EISEVQLVESGGGLVQPGGSLRLSCAASGFNVVDFSLHWVRQAPGKGLEWVAYISSSSGSTSYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARWGYWPGEPWWKAFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCD
>9I3T_E3 len=215 copies=1 longest=0
DIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQYLYYSLVTFGQGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9I5N_E1 len=118 copies=1 longest=0
QVQLKQSGPGLVQPSQSLSITCTVSGFSLTGYGVHWVRQSPGKGLEWLGMIWSGGSTDYNAAFISRLSISKDSSKSQVFFKMNSLQANDTAIYYCARSSGNYGEDAMDYWGQGTSVTV
>9I5N_E2 len=107 copies=1 longest=0
ETTVTQSPASLSMAIGEKVTIRCITSTDIDDDMNWYQQKPGEPPELLISEDNTLRPGVPSRFSSSGYGTDFVFAIENMLPEDVADYFCLQSDNLPFTFGSGTKLEIK
>9I5N_E3 len=143 copies=1 longest=1
NPQIAAHVISEASSKTTSVLQWAEKGYYTMSNNLVTLENGKQLTVKRQGLYYIYAQVTFCSNREASSQAPFIASLCLKSPGRFERILLRAANTHSSAKPCGQQSIHLGGVFELQPGASVFVNVTDPSQVSHGTGFTSFGLLKL
>9I8C_E1 len=338 copies=1 longest=1
RLSDNPEVRMAVHKIADLISSMTIYLMQNTEDGDIRIRNELSRKIDITPYSLMTRKSWMYNIVYTMLLDGEGNSVVFPKYTADGLIDELVPLTPSKVNFLDTPDGYQVLYGGQTFNYDEVLHFIYNPDPERPYIGRGYRVVLKDIADNLKQATATKKSFMSGKYMPSLIVKVDAATAELSSEEGRNAVFKKYLQATEAGQPWIIPAELLEVEQVKPLSLKDIAINEAVELDKRTVAGMFGVPAFLLGIGEFNRDEYNNFINSTILPIAKGIEQELTRKLLISPDLYFKFNPRSLYAYDLKELAEVGSNMYVRGIMEGNEVRDWLGLSPKEGLSELVIL
>9IB1_E1 len=1288 copies=1 longest=1
MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPGSASSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTPSALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGTSLEVLFQGPGHHHHHHHHSAWSHPQFEKGGGSGGGGSGGSAWSHPQFEK
>9IB1_E2 len=214 copies=1 longest=0
DIQMTQSPSSLSASVGDRVTITCRASQSISSYLNWYQQKPGKAPKLLIYAASSLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQGYSTLYTFGQGTKLEIKRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKDINVKWKIDGSERQNGVLNSWTDQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC
>9IB1_E3 len=448 copies=1 longest=0
EVQLLESGGGLVQPGGSLRLSCAASGFTFSYYYMGWVRQAPGKGLEWVSYISGGGGYTYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARGDPSRSTFDYWGQGTLVTVSSAKTTAPSVYPLAPVCGDTTGSSVTLGCLVKGYFPEPVTLTWNSGSLSSGVHTFPAVLQSDLYTLSSSVTVTSSTWPSQSITCNVAHPASSTKVDKKIEPRGPTIKPCPPCKCPAPNLLGGPSVFIFPPKIKDVLMISLSPIVTCVVVDVSEDDPDVQISWFVNNVEVHTAQTQTHREDYNSTLRVVSALPIQHQDWMSGKEFKCKVNNKDLPAPIERTISKPKGSVRAPQVYVLPPPEEEMTKKQVTLTCMVTDFMPEDIYVEWTNNGKTELNYKNTEPVLDSDGSYFMYSKLRVEKKNWVERNSYSCSVVHEGLHNHHTTKSFSRTPGK
>9IB3_E1 len=1288 copies=1 longest=1
MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHAIHVSGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLGVYYHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPINLVRDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQDVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEHVNNSYECDIPIGAGICASYQTQTNSPGSASSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLNRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKDFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTPSALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGTSLEVLFQGPGHHHHHHHHSAWSHPQFEKGGGSGGGGSGGSAWSHPQFEK
>9IB3_E2 len=214 copies=1 longest=0
DIQMTQSPSSLSASVGDRVTITCRASQSISSYLNWYQQKPGKAPKLLIYAASSLQSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCQQGYSTLYTFGQGTKLEIKRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKDINVKWKIDGSERQNGVLNSWTDQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC
>9IB3_E3 len=448 copies=1 longest=0
EVQLLESGGGLVQPGGSLRLSCAASGFTFSYYYMGWVRQAPGKGLEWVSYISGGGGYTYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCARGDPSRSTFDYWGQGTLVTVSSAKTTAPSVYPLAPVCGDTTGSSVTLGCLVKGYFPEPVTLTWNSGSLSSGVHTFPAVLQSDLYTLSSSVTVTSSTWPSQSITCNVAHPASSTKVDKKIEPRGPTIKPCPPCKCPAPNLLGGPSVFIFPPKIKDVLMISLSPIVTCVVVDVSEDDPDVQISWFVNNVEVHTAQTQTHREDYNSTLRVVSALPIQHQDWMSGKEFKCKVNNKDLPAPIERTISKPKGSVRAPQVYVLPPPEEEMTKKQVTLTCMVTDFMPEDIYVEWTNNGKTELNYKNTEPVLDSDGSYFMYSKLRVEKKNWVERNSYSCSVVHEGLHNHHTTKSFSRTPGK
>9IBW_E1 len=488 copies=1 longest=1
MSEEEFYLFKNISSVGPWDGPQYHIAPVWAFYLQAAFMGTVFLIGFPLNAMVLVATLRYKKLRQPLNYILVNVSFGGFLLCIFSVFPVFVASCNGYFVFGRHVCALEGFLGTVAGLVTGWSLAFLAFERYIVICKPFGNFRFSSKHALTVVLATWTIGIGVSIPPFFGWSRFIPEGLQCSCGPDWYTVGTKYRSESYTWFLFIFCFIVPLSLICFSYTQLLRALKAVAAQARRQLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLRSTLQESATTQKAEREVSRMVVVMVGSFCVCYVPYAAFAMYMVNNRNHGLDLRLVTIPSFFSKSACIYNPIIYCFMNKQFQACIMKMVCGKAMTDESDTCSSQKTEVSTVSSTQVGPNLEVLFQGPDYKDDDDKGAGATETSQVAPA
>9IBW_E2 len=215 copies=1 longest=0
MDIVLTQSPATLSVTPGDRVSLSCRASQSVSNYLHWYQQKSHESPRLLIKYASQSISGIPSRFSGSGSGTDFTLSINSVETEDFGMYFCQQSNSWPLTFGAGTKLELRRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKDINVKWKIDGSERQNGVLNSWTDQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC
>9IBW_E3 len=222 copies=1 longest=0
MQIQLQQSGPELVKPGASVKISCKASGYTFTDFYINWMKQRPGQGLEWIGWIFPGSGNTHYNEKFKGKATLIVDTSSSTAFMQLNSLTSEDSAVYFCTRPVSYYYDFDYWGQGTTLTVSSAKTTAPSVYPLAPVCGDTSGSSVTLGCLVKGYFPEPVTLTWNSGSLSSGVHTFPAVLQSDLYTLSSSVTVTSSTWPSQSITCNVAHPASSTKVDKKIEPRGC
>9J6R_E1 len=562 copies=1 longest=1
DRETWGKKIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV
>9J6S_E1 len=562 copies=1 longest=1
DRETWGKKIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV
>9J6T_E1 len=555 copies=1 longest=1
KIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV
>9J6U_E1 len=555 copies=1 longest=1
KIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV
>9J6V_E1 len=555 copies=1 longest=1
KIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV
>9JBS_E1 len=569 copies=1 longest=1
MAPTLATAHRRRWWMACTAVLENLLFSAVLLGWGSLLIMLKSEGFYSYLCTEPENVTNGTVGGTAEPGHEEVSWMNGWLSCQAQDEMLNLAFTVGSFLLSAITLPLGIVMDKYGPRKLRLLGSACFAVSCLLIAYGASKPNALSVLIFIALALNGFGGMCMTFTSLTLPNMFGDLRSTFIALMIGSYASSAVTFPGIKLIYDAGVSFIVVLVVWAGCSGLVFLNCFFNWPLEPFPGPEDMDYSVKIKFSWLGFDHKITGKQFYKQVTTVGRRLSVGSSMRSAKEQVALQEGHKLCLSTVDLEVKCQPDAAVAPSFMHSVFSPILLLSLVTMCVTQLRLIFYMGAMNNILKFLVSGDQKTVGLYTSIFGVLQLLCLLTAPVIGYIMDWRLKECEDASEEPEEKDANQGEKKKKKRDRQIQKITNAMRAFAFTNLLLVGFGVTCLIPNLPLQILSFILHTIVRGFIHSAVGGLYAAVYPSTQFGSLTGLQSLISALFALLQQPLFLAMMGPLQGDPLWVNVGLLLLSLLGFCLPLYLICYRRQLERQLQQRQEDDKLFLKINGSSNQEAFV
>9JBT_E1 len=569 copies=1 longest=1
MAPTLATAHRRRWWMACTAVLENLLFSAVLLGWGSLLIMLKSEGFYSYLCTEPENVTNGTVGGTAEPGHEEVSWMNGWLSCQAQDEMLNLAFTVGSFLLSAITLPLGIVMDKYGPRKLRLLGSACFAVSCLLIAYGASKPNALSVLIFIALALNGFGGMCMTFTSLTLPNMFGDLRSTFIALMIGSYASSAVTFPGIKLIYDAGVSFIVVLVVWAGCSGLVFLNCFFNWPLEPFPGPEDMDYSVKIKFSWLGFDHKITGKQFYKQVTTVGRRLSVGSSMRSAKEQVALQEGHKLCLSTVDLEVKCQPDAAVAPSFMHSVFSPILLLSLVTMCVTQLRLIFYMGAMNNILKFLVSGDQKTVGLYTSIFGVLQLLCLLTAPVIGYIMDWRLKECEDASEEPEEKDANQGEKKKKKRDRQIQKITNAMRAFAFTNLLLVGFGVTCLIPNLPLQILSFILHTIVRGFIHSAVGGLYAAVYPSTQFGSLTGLQSLISALFALLQQPLFLAMMGPLQGDPLWVNVGLLLLSLLGFCLPLYLICYRRQLERQLQQRQEDDKLFLKINGSSNQEAFV
>9JBU_E1 len=569 copies=1 longest=1
MAPTLATAHRRRWWMACTAVLENLLFSAVLLGWGSLLIMLKSEGFYSYLCTEPENVTNGTVGGTAEPGHEEVSWMNGWLSCQAQDEMLNLAFTVGSFLLSAITLPLGIVMDKYGPRKLRLLGSACFAVSCLLIAYGASKPNALSVLIFIALALNGFGGMCMTFTSLTLPNMFGDLRSTFIALMIGSYASSAVTFPGIKLIYDAGVSFIVVLVVWAGCSGLVFLNCFFNWPLEPFPGPEDMDYSVKIKFSWLGFDHKITGKQFYKQVTTVGRRLSVGSSMRSAKEQVALQEGHKLCLSTVDLEVKCQPDAAVAPSFMHSVFSPILLLSLVTMCVTQLRLIFYMGAMNNILKFLVSGDQKTVGLYTSIFGVLQLLCLLTAPVIGYIMDWRLKECEDASEEPEEKDANQGEKKKKKRDRQIQKITNAMRAFAFTNLLLVGFGVTCLIPNLPLQILSFILHTIVRGFIHSAVGGLYAAVYPSTQFGSLTGLQSLISALFALLQQPLFLAMMGPLQGDPLWVNVGLLLLSLLGFCLPLYLICYRRQLERQLQQRQEDDKLFLKINGSSNQEAFV
>9JKH_E1 len=620 copies=1 longest=1
MSKSKCSVGLMSSVVAPAKEPNAVGPKEVELILVKEQNGVQLTSSTLTNPRQSPVEAQDRETWGKKIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV
>9JKI_E1 len=620 copies=1 longest=1
MSKSKCSVGLMSSVVAPAKEPNAVGPKEVELILVKEQNGVQLTSSTLTNPRQSPVEAQDRETWGKKIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV
>9JKJ_E1 len=620 copies=1 longest=1
MSKSKCSVGLMSSVVAPAKEPNAVGPKEVELILVKEQNGVQLTSSTLTNPRQSPVEAQDRETWGKKIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV
>9JKK_E1 len=620 copies=1 longest=1
MSKSKCSVGLMSSVVAPAKEPNAVGPKEVELILVKEQNGVQLTSSTLTNPRQSPVEAQDRETWGKKIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV
>9JKM_E1 len=620 copies=1 longest=1
MSKSKCSVGLMSSVVAPAKEPNAVGPKEVELILVKEQNGVQLTSSTLTNPRQSPVEAQDRETWGKKIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV
>9JZI_E1 len=393 copies=1 longest=1
MGSSHHHHHHSSGLVPRGSHMLEMPPKRRLVDDADAMEDQDLYEPPASLPKLPGKFLQYTVGGSDPHPGIGHEKDIRQNAVALLDQSRRDMFHTVTPSLVFLCLLIPGLHAAFVHGGVPRESYLSTPVTRGEQTVVKTAKFYGEKTTQRDLTELEISSIFSHCCSLLIGVVIGSSSKIKAGAEQIKKRFKTMMAALNRPSHGETATLLQMFNPHEAIDWINGQPWVGSFVLSLLTTDFESPGKEFMDQIKLVASYAQMTTYTTIKEYLAECMDATLTIPVVAYEIRDFLEVSAKLKEEHADLFPFLGAIRHPDAIKLAPRSFPNLASAAFYWSKKENPTMAGYRASTIQPGASVKETQLARYRRREISRGEDGAELSGEISAIMRMIGVTGLN
>9JZJ_E1 len=393 copies=1 longest=1
MGSSHHHHHHSSGLVPRGSHMLEMPPKRRLVDDADAMEDQDLYEPPASLPKLPGKFLQYTVGGSDPHPGIGHEKDIRQNAVALLDQSRRDMFHTVTPSLVFLCLLIPGLHAAFVHGGVPRESYLSTPVTRGEQTVVKTAKFYGEKTTQRDLTELEISSIFSHCCSLLIGVVIGSSSKIKAGAEQIKKRFKTMMAALNRPSHGETATLLQMFNPHEAIDWINGQPWVGSFVLSLLTTDFESPGKEFMDQIKLVASYAQMTTYTTIKEYLAECMDATLTIPVVAYEIRDFLEVSAKLKEEHADLFPFLGAIRHPDAIKLAPRSFPNLASAAFYWSKKENPTMAGYRASTIQPGASVKETQLARYRRREISRGEDGAELSGEISAIMRMIGVTGLN
>9JZN_E1 len=393 copies=1 longest=1
MGSSHHHHHHSSGLVPRGSHMLEMPPKRRLVDDADAMEDQDLYEPPASLPKLPGKFLQYTVGGSDPHPGIGHEKDIRQNAVALLDQSRRDMFHTVTPSLVFLCLLIPGLHAAFVHGGVPRESYLSTPVTRGEQTVVKTAKFYGEKTTQRDLTELEISSIFSHCCSLLIGVVIGSSSKIKAGAEQIKARFKTMMAALNRPSHGETATLLQMFNPHEAIDWINGQPWVGSFVLSLLTTDFESPGKEFMDQIKLVASYAQMTTYTTIKEYLAECMDATLTIPVVAYEIRDFLEVSAKLKEEHADLFPFLGAIRHPDAIKLAPRSFPNLASAAFYWSKKENPTMAGYRASTIQPGASVKETQLARYRRREISRGEDGAELSGEISAIMRMIGVTGLN
>9K7G_E1 len=413 copies=1 longest=1
MSTGKESSKAYADIESRGDYKDDGKKGSTLSSKQHFMLSLTFILIGLSSLNVWNTALGLNINFKYNTFQITGLVCSSIVALFVEIPKIMLPFLLGGLSILCAGFQISHSFFTDTQFDTYCLVAFIVIGVVAGLAQTIAFNIGSTMEDNMGGYMSAGIGISGVFIFVINLLLDQFVSPEKHYGVNKAKLLALYIICELCLILAIVFCVCNLDLTNKNNKKDEENKENNATLSYMELFKDSYKAILTMFLVNWLTLQLFPGVGHKKWQESHNISDYNVTIIVGMFQVFDFLSRYPPNLTHIKIFKNFTFSLNKLLVANSLRLLFIPWFILNACVDHPFFKNIVQQCVCMAMLAFTNGWFNTVPFLVFVKELKKAKKKKEIEIISTFLVIAMFVGLFCGIWTTYIYNLFNIVLPKP
>9K7S_E1 len=384 copies=1 longest=1
LSSKQHFMLSLTFILIGLSSLNVWNTALGLNINFKYNTFQITGLVCSSIVALFVEIPKIMLPFLLGGLSILCAGFQISHSFFTDTQFDTYCLVAFIVIGVVAGLAQTIAFNIGCTMEDNMGGYMSAGIGISGVFIFVINLLLDQFVSPEKHYGVNKAKLLYLYIICELCLILAIVFCVCNLDLTNKNNKKDEENKENNATLSYMELFKDSYKAILTMFLVNWLTLQLFPGVGHKKWQESHNISDYNVTIIVGMFQVFDFLSRYPPNLTHIKIFKNFTFSLNKLLVANSLRLLFIPWFILNACVDHPFFKNIVQQCVCMAMLAFTNGWFNTVPFLVFVKELKKAKKKKEIEIICTFLVIAMFVGLFCGIWTTYIYNLFNIVLPKP
>9K88_E1 len=384 copies=1 longest=1
LSSKQHFMLSLTFILIGLSSLNVWNTALGLNINFKYNTFQITGLVCSSIVALFVEIPKIMLPFLLGGLSILCAGFQISHSFFTDTQFDTYCLVAFIVIGVVAGLAQTIAFNIGSCMEDNMGGYMSAGIGISGVFIFVINLLLDQFVSPEKHYGVNKAKLLYLYIICELCLILAIVFCVCNLDLTNKNNKKDEENKENNATLSYMELFKDSYKAILTMFLVNWLTLQLFPGVGHKKWQESHNISDYNVTIIVGMFQVFDFLSRYPPNLTHIKIFKNFTFSLNKLLVANSLRLLFIPWFILNACVDHPFFKNIVQQCVCMAMLAFTNGWFNTVPFLVFVKELKKAKKCKEIEIISTFLVIAMFVGLFCGIWTTYIYNLFNIVLPKP
>9K8Y_E1 len=376 copies=1 longest=1
SSKQHFMLSLTFILIGLSSLNVWNTALGLRINFKYNTFQITGLVCSSIVALFVEIPKIMLPFLLGGLSILCAGFQISHSFFTDTQFDTYCLVAFIVIGVVAGLAQTIAFNIGSCMEDNMGGYMSAGIGISGVFIFVINLLLDQFVSPEKHYGVNKAKLLALYIICELCLILAIVFCVCNLDLTNKNNKKDEENKENNATLSYMELFKDSYKAILTMFLVNWLTLQLFPGVGHKKWRESHNISDYNVTIIVGMFQVFDFLSRYPPNLTHIKIFKNFTFSLNKLLVANSLRLLFIPWFILNACVDHPFFKNIVQQCVCMAMLAFTNGWFNTVPFLVFVKELKKAKKCKEIEIISTFLVIAMFVGLFCGIWTTYIYNLF
>9K90_E1 len=384 copies=1 longest=1
LSSKQHFMLSLTFILIGLSSLNVWNTALGLNINFKYNTFQITGLVCSSIVALFVEIPKIMLPFLLGGLSILCAGFQISHSFFTDTQFDTYCLVAFIVIGVVAGLAQTIAFNIGCTMEDNMGGYMSAGIGISGVFIFVINLLLDQFVSPEKHYGVNKAKLLYLYIICELCLILAIVFCVCNLDLTNKNNKKDEENKENNATLSYMELFKDSYKAILTMFLVNWLTLQLFPGVGHKKWQESHNISDYNVTIIVGMFQVFDFLSRYPPNLTHIKIFKNFTFSLNKLLVANSLRLLFIPWFILNACVDHPFFKNIVQQCVCMAMLAFTNGWFNTVPFLVFVKELKKAKKKKEIEIICTFLVIAMFVGLFCGIWTTYIYNLFNIVLPKP
>9KKN_E1 len=637 copies=1 longest=1
MADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLLQEPRRQRRLVLVIVCVALLLDNMLYMVIVPIVPDYIAHMRGGGEGPTRTPEVWEPTLPLPTPANASAYTANTSASPTAAWPAGSALRPRYPTESEDVKIGVLFASKAILQLLVNPLSGPFIDRMSYDVPLLIGLGVMFASTVLFAFAEDYATLFAARSLQGLGSAFADTSGIAMIADKYPEEPERSRALGVALAFISFGSLVAPPFGGILYEFAGKRVPFLVLAAVSLFDALLLLAVAKPFSAAARARANLPVGTPIHRLMLDPYIAVVAGALTTCNIPLAFLEPTIATWMKHTMAASEWEMGMAWLPAFVPHVLGVYLTVRLAARYPHLQWLYGALGLAVIGASSCIVPACRSFAPLVVSLCGLCFGIALVDTALLPTLAFLVDVRHVSVYGSVYAIADISYSVAYALGPIVAGHIVHSLGFEQLSLGMGLANLLYAPVLLLLRNVGLLTRSRSERDVLLDEPPQGLYDAVRLRERPVSGQDGEPRSPPGPFDACEDDYNYYYTRSHHHHHHHHHHGSVEDYKDDDDK
>9KKO_E1 len=637 copies=1 longest=1
MADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLLQEPRRQRRLVLVIVCVALLLDNMLYMVIVPIVPDYIAHMRGGGEGPTRTPEVWEPTLPLPTPANASAYTANTSASPTAAWPAGSALRPRYPTESEDVKIGVLFASKAILQLLVNPLSGPFIDRMSYDVPLLIGLGVMFASTVLFAFAEDYATLFAARSLQGLGSAFADTSGIAMIADKYPEEPERSRALGVALAFISFGSLVAPPFGGILYEFAGKRVPFLVLAAVSLFDALLLLAVAKPFSAAARARANLPVGTPIHRLMLDPYIAVVAGALTTCNIPLAFLEPTIATWMKHTMAASEWEMGMAWLPAFVPHVLGVYLTVRLAARYPHLQWLYGALGLAVIGASSCIVPACRSFAPLVVSLCGLCFGIALVDTALLPTLAFLVDVRHVSVYGSVYAIADISYSVAYALGPIVAGHIVHSLGFEQLSLGMGLANLLYAPVLLLLRNVGLLTRSRSERDVLLDEPPQGLYDAVRLRERPVSGQDGEPRSPPGPFDACEDDYNYYYTRSHHHHHHHHHHGSVEDYKDDDDK
>9KQ5_E1 len=637 copies=1 longest=1
MADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLLQEPRRQRRLVLVIVCVALLLDNMLYMVIVPIVPDYIAHMRGGGEGPTRTPEVWEPTLPLPTPANASAYTANTSASPTAAWPAGSALRPRYPTESEDVKIGVLFASKAILQLLVNPLSGPFIDRMSYDVPLLIGLGVMFASTVLFAFAEDYATLFAARSLQGLGSAFADTSGIAMIADKYPEEPERSRALGVALAFISFGSLVAPPFGGILYEFAGKRVPFLVLAAVSLFDALLLLAVAKPFSAAARARANLPVGTPIHRLMLDPYIAVVAGALTTCNIPLAFLEPTIATWMKHTMAASEWEMGMAWLPAFVPHVLGVYLTVRLAARYPHLQWLYGALGLAVIGASSCIVPACRSFAPLVVSLCGLCFGIALVDTALLPTLAFLVDVRHVSVYGSVYAIADISYSVAYALGPIVAGHIVHSLGFEQLSLGMGLANLLYAPVLLLLRNVGLLTRSRSERDVLLDEPPQGLYDAVRLRERPVSGQDGEPRSPPGPFDACEDDYNYYYTRSHHHHHHHHHHGSVEDYKDDDDK
>9KQM_E1 len=637 copies=1 longest=1
MADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLLQEPRRQRRLVLVIVCVALLLDNMLYMVIVPIVPDYIAHMRGGGEGPTRTPEVWEPTLPLPTPANASAYTANTSASPTAAWPAGSALRPRYPTESEDVKIGVLFASKAILQLLVNPLSGPFIDRMSYDVPLLIGLGVMFASTVLFAFAEDYATLFAARSLQGLGSAFADTSGIAMIADKYPEEPERSRALGVALAFISFGSLVAPPFGGILYEFAGKRVPFLVLAAVSLFDALLLLAVAKPFSAAARARANLPVGTPIHRLMLDPYIAVVAGALTTCNIPLAFLEPTIATWMKHTMAASEWEMGMAWLPAFVPHVLGVYLTVRLAARYPHLQWLYGALGLAVIGASSCIVPACRSFAPLVVSLCGLCFGIALVDTALLPTLAFLVDVRHVSVYGSVYAIADISYSVAYALGPIVAGHIVHSLGFEQLSLGMGLANLLYAPVLLLLRNVGLLTRSRSERDVLLDEPPQGLYDAVRLRERPVSGQDGEPRSPPGPFDACEDDYNYYYTRSHHHHHHHHHHGSVEDYKDDDDK
>9KSL_E1 len=315 copies=1 longest=1
MTSAQNESQALGDLAARQLANATKTVPQLSTITPRFLLHLLSWVPVEAGIYRVNRVVNPDRVAIHSEAGAGTEEPLPETYVDYETHPREYTLRSISTLLDVHTRVSDLYSSPHDQVTQQLRLTIETIKERQEYELVNNPEYGLLAQATPEQTIQTLAGAPTPDDLDALITKVWKTPAFFLTHPLGVAAFGRECTYRGVPPPTVSMYGAQFITWRGIPIVPSDKVPVEDGTTKFVLVRTGEERQGVVGLFQPGLVGEQAPGLSVRFTGINRSAIASYLVTLYTSLAVLTDDALAVLDGVAVDQFHEYQDYKDDDDK
>9KVS_E1 len=538 copies=1 longest=1
MGKTIIALSYIFCLVFADYKDDDDAANFTPVNGSSGNQSVRLVTSSSLEVLFQGPGSASFSAETNSTDLLSQPWNEPPVILSMVILSLTFLLGLPGNGLVLWVAGLKMQRTVNTIWFLHLTLADLLCCLSLPFSLAHLALQGQWPYGRFLCKLIPSIIVLNMFASVFLLTAISLDRCLVVFKPIWCQNHRNVGMACSICGCIWVVAFVMCIPVFVYREIFTTDNHNRCGYKFGLSSSLDYPDFYGDPLENRSLENIVQPPGEMNDRLDPSSFQTNDHPWTVPTVFQPQTFQRPSADSLPRGSARLTSQNLYSNVFKPADVVSPKIPSGFPIEDHETSPLDNSDAFLSTHLKLFPSASSNSFYESELPQGFQDYYNLGQFTDDDQVPTPLVAITITRLVVGFLLPSVIMIACYSFIVFRMQRGRFAKSQSKTFRVAVVVVAVFLVCWTPYHIFGVLSLLTDPETPLGKTLMSWDHVCIALASANSCFNPFLYALLGKDFRKKARQSIQGILEAAFSEELTRSTHCPSNNVISERNSTTV
>9L19_E1 len=620 copies=1 longest=1
MATKEKLQCLKDFHKDILKPSPGKSPGTRPEDEAEGKPPQREKWSSKIDFVLSVAGGFVGLGNVWRFPYLCYKNGGGAFLIPYFIFLFGSGLPVFFLEIIIGQYTSEGGITCWEKICPLFSGIGYASVVIVSLLNVYYIVILAWATYYLFQSFQKELPWAHCNHSWNTPHCMEDTMRKNKSVWITISSTNFTSPVIEFWERNVLSLSPGIDHPGSLKWDLALCLLLVWLVCFFCIWKGVRSTGKVVYFTATFPFAMLLVLLVRGLTLPGAGAGIKFYLYPDITRLEDPQVWIDAGTQIFFSYAICLGAMTSLGSYNKYKYNSYRDCMLLGCLNSGTSFVSGFAIFSILGFMAQEQGVDIADVAESGPGLAFIAYPKAVTMMPLPTFWSILFFIMLLLLGLDSQFVEVEGQITSLVDLYPSFLRKGYRREIFIAFVCSISYLLGLTMVTEGGMYVFQLFDYYAASGVCLLWVAFFECFVIAWIYGGDNLYDGIEDMIGYRPGPWMKYSWAVITPVLCVGCFIFSLVKYVPLTYNKTYVYPNWAIGLGWSLALSSMLCVPLVIVIRLCQTEGPFLVRVKYLLTPREPNRWAVEREGATPYNSRTVMNGALVKPTHIIVETMM
>9L1A_E1 len=620 copies=1 longest=1
MATKEKLQCLKDFHKDILKPSPGKSPGTRPEDEAEGKPPQREKWSSKIDFVLSVAGGFVGLGNVWRFPYLCYKNGGGAFLIPYFIFLFGSGLPVFFLEIIIGQYTSEGGITCWEKICPLFSGIGYASVVIVSLLNVYYIVILAWATYYLFQSFQKELPWAHCNHSWNTPHCMEDTMRKNKSVWITISSTNFTSPVIEFWERNVLSLSPGIDHPGSLKWDLALCLLLVWLVCFFCIWKGVRSTGKVVYFTATFPFAMLLVLLVRGLTLPGAGAGIKFYLYPDITRLEDPQVWIDAGTQIFFSYAICLGAMTSLGSYNKYKYNSYRDCMLLGCLNSGTSFVSGFAIFSILGFMAQEQGVDIADVAESGPGLAFIAYPKAVTMMPLPTFWSILFFIMLLLLGLDSQFVEVEGQITSLVDLYPSFLRKGYRREIFIAFVCSISYLLGLTMVTEGGMYVFQLFDYYAASGVCLLWVAFFECFVIAWIYGGDNLYDGIEDMIGYRPGPWMKYSWAVITPVLCVGCFIFSLVKYVPLTYNKTYVYPNWAIGLGWSLALSSMLCVPLVIVIRLCQTEGPFLVRVKYLLTPREPNRWAVEREGATPYNSRTVMNGALVKPTHIIVETMM
>9L1B_E1 len=620 copies=1 longest=1
MATKEKLQCLKDFHKDILKPSPGKSPGTRPEDEAEGKPPQREKWSSKIDFVLSVAGGFVGLGNVWRFPYLCYKNGGGAFLIPYFIFLFGSGLPVFFLEIIIGQYTSEGGITCWEKICPLFSGIGYASVVIVSLLNVYYIVILAWATYYLFQSFQKELPWAHCNHSWNTPHCMEDTMRKNKSVWITISSTNFTSPVIEFWERNVLSLSPGIDHPGSLKWDLALCLLLVWLVCFFCIWKGVRSTGKVVYFTATFPFAMLLVLLVRGLTLPGAGAGIKFYLYPDITRLEDPQVWIDAGTQIFFSYAICLGAMTSLGSYNKYKYNSYRDCMLLGCLNSGTSFVSGFAIFSILGFMAQEQGVDIADVAESGPGLAFIAYPKAVTMMPLPTFWSILFFIMLLLLGLDSQFVEVEGQITSLVDLYPSFLRKGYRREIFIAFVCSISYLLGLTMVTEGGMYVFQLFDYYAASGVCLLWVAFFECFVIAWIYGGDNLYDGIEDMIGYRPGPWMKYSWAVITPVLCVGCFIFSLVKYVPLTYNKTYVYPNWAIGLGWSLALSSMLCVPLVIVIRLCQTEGPFLVRVKYLLTPREPNRWAVEREGATPYNSRTVMNGALVKPTHIIVETMM
>9L1C_E1 len=620 copies=1 longest=1
MATKEKLQCLKDFHKDILKPSPGKSPGTRPEDEAEGKPPQREKWSSKIDFVLSVAGGFVGLGNVWRFPYLCYKNGGGAFLIPYFIFLFGSGLPVFFLEIIIGQYTSEGGITCWEKICPLFSGIGYASVVIVSLLNVYYIVILAWATYYLFQSFQKELPWAHCNHSWNTPHCMEDTMRKNKSVWITISSTNFTSPVIEFWERNVLSLSPGIDHPGSLKWDLALCLLLVWLVCFFCIWKGVRSTGKVVYFTATFPFAMLLVLLVRGLTLPGAGAGIKFYLYPDITRLEDPQVWIDAGTQIFFSYAICLGAMTSLGSYNKYKYNSYRDCMLLGCLNSGTSFVSGFAIFSILGFMAQEQGVDIADVAESGPGLAFIAYPKAVTMMPLPTFWSILFFIMLLLLGLDSQFVEVEGQITSLVDLYPSFLRKGYRREIFIAFVCSISYLLGLTMVTEGGMYVFQLFDYYAASGVCLLWVAFFECFVIAWIYGGDNLYDGIEDMIGYRPGPWMKYSWAVITPVLCVGCFIFSLVKYVPLTYNKTYVYPNWAIGLGWSLALSSMLCVPLVIVIRLCQTEGPFLVRVKYLLTPREPNRWAVEREGATPYNSRTVMNGALVKPTHIIVETMM
>9L1D_E1 len=620 copies=1 longest=1
MATKEKLQCLKDFHKDILKPSPGKSPGTRPEDEAEGKPPQREKWSSKIDFVLSVAGGFVGLGNVWRFPYLCYKNGGGAFLIPYFIFLFGSGLPVFFLEIIIGQYTSEGGITCWEKICPLFSGIGYASVVIVSLLNVYYIVILAWATYYLFQSFQKELPWAHCNHSWNTPHCMEDTMRKNKSVWITISSTNFTSPVIEFWERNVLSLSPGIDHPGSLKWDLALCLLLVWLVCFFCIWKGVRSTGKVVYFTATFPFAMLLVLLVRGLTLPGAGAGIKFYLYPDITRLEDPQVWIDAGTQIFFSYAICLGAMTSLGSYNKYKYNSYRDCMLLGCLNSGTSFVSGFAIFSILGFMAQEQGVDIADVAESGPGLAFIAYPKAVTMMPLPTFWSILFFIMLLLLGLDSQFVEVEGQITSLVDLYPSFLRKGYRREIFIAFVCSISYLLGLTMVTEGGMYVFQLFDYYAASGVCLLWVAFFECFVIAWIYGGDNLYDGIEDMIGYRPGPWMKYSWAVITPVLCVGCFIFSLVKYVPLTYNKTYVYPNWAIGLGWSLALSSMLCVPLVIVIRLCQTEGPFLVRVKYLLTPREPNRWAVEREGATPYNSRTVMNGALVKPTHIIVETMM
>9L1E_E1 len=620 copies=1 longest=1
MATKEKLQCLKDFHKDILKPSPGKSPGTRPEDEAEGKPPQREKWSSKIDFVLSVAGGFVGLGNVWRFPYLCYKNGGGAFLIPYFIFLFGSGLPVFFLEIIIGQYTSEGGITCWEKICPLFSGIGYASVVIVSLLNVYYIVILAWATYYLFQSFQKELPWAHCNHSWNTPHCMEDTMRKNKSVWITISSTNFTSPVIEFWERNVLSLSPGIDHPGSLKWDLALCLLLVWLVCFFCIWKGVRSTGKVVYFTATFPFAMLLVLLVRGLTLPGAGAGIKFYLYPDITRLEDPQVWIDAGTQIFFSYAICLGAMTSLGSYNKYKYNSYRDCMLLGCLNSGTSFVSGFAIFSILGFMAQEQGVDIADVAESGPGLAFIAYPKAVTMMPLPTFWSILFFIMLLLLGLDSQFVEVEGQITSLVDLYPSFLRKGYRREIFIAFVCSISYLLGLTMVTEGGMYVFQLFDYYAASGVCLLWVAFFECFVIAWIYGGDNLYDGIEDMIGYRPGPWMKYSWAVITPVLCVGCFIFSLVKYVPLTYNKTYVYPNWAIGLGWSLALSSMLCVPLVIVIRLCQTEGPFLVRVKYLLTPREPNRWAVEREGATPYNSRTVMNGALVKPTHIIVETMM
>9L38_E1 len=491 copies=1 longest=1
MAGQGLPLHVATLLTGLLECLGFAGVLFGWPSLVFVFKNEDYFKDLCGPDAGPIGNATGQADCKAQDERFSLIFTLGSFMNNFMTFPTGYIFDRFKTTVARLIAIFFYTTATLIIAFTSAGSAVLLFLAMPMLTIGGILFLITNLQIGNLFGQHRSTIITLYNGAFDSSSAVFLIIKLLYEKGISLRASFIFISVCSTWHVARTFLLMPRGHIPYPLPPNYSYGLCPGNGTTKEEKETAEHENRELQSKEFLSAKEETPGAGQKQELRSFWSYAFSRRFAWHLVWLSVIQLWHYLFIGTLNSLLTNMAGGDMARVSTYTNAFAFTQFGVLCAPWNGLLMDRLKQKYQKEARKTGSSTLAVALCSTVPSLALTSLLCLGFALCASVPILPLQYLTFILQVISRSFLYGSNAAFLTLAFPSEHFGKLFGLVMALSAVVSLLQFPIFTLIKGSLQNDPFYVNVMFMLAILLTFFHPFLVYRECRTWKESPSAIA
>9L8N_E1 len=761 copies=1 longest=1
GPQNNSTTPSPIITSSNSSVLVFEISSKMKMIEKKLEANTVHVLRLELDQSFILDLTKVAAEIVDSSKYSKEDGVILEVTVSNGRDSFLLKLPTVYPNLKLYTDGKLLNPLVEQDFGAHRKRHRIGDPHFHQNLIVTVQSRLNADIDYRLHVTHLDRAQYDFLKFKTGQTTKTLSNQKLTFVKPIGFFLNCSEQNISQFHVTLYSEDDICANLITVPANESIYDRSVISDKTHNRRVLSFTKRADIFFTETEISMFKSFRIFVFIAPDDSGCSTNTSRKSFNEKKKISFEFKKLENQSYAVPTALMMIFLTTPCLLFLPIVINIIKNSRKLAPSQSNLISFSPVPSEQRDMDLSHDEQQNTSSELENNGEIPAAENQIVEEITAENQETSVEEGNREIQVKIPLKQDSLSLHGQMLQYPVAIILPVLMHTAIEFHKWTTSTMANRDEMCFHNHACARPLGELRAWNNIITNIGYTLYGAIFIVLSICRRGRHEYSHVFGTYECTLLDVTIGVFMVLQSIASATYHICPSDVAFQFDTPCIQVICGLLMVRQWFVRHESPSPAYTNILLVGVVSLNFLISAFSKTSYVRFIIAVIHVIVVGSICLAKERSLGSEKLKTRFFIMAFSMGNFAAIVMYLTLSAFHLNQIATYCFIINCIMYLMYYGCMKVLHSERITSKAKLCGALSLLAWAVAGFFFFQDDTDWTRSAAASRALNKPCLLLGFFGSHDLWHIFGALAGLFTFIFVSFVDDDLINTRKTSINIF
>9LBW_E1 len=183 copies=2 longest=0
MDIDPYKEFGATVELLSFLPSDFFPSVRDLLDTASALYREALESPEHCSPHHTALRQAILCWGELMTLATWVGVNLEDPASRDLVVSYVNTNMGLKFRQLLWFHISCLTFGRETVIEYLVSFGVWIRTPPAYRPPNAPILSTLPETTVVRRRGRSPRRRTPSPRRRRSQSPRRRRSQSRESQC
>9LBW_E2 len=473 copies=1 longest=1
MGWSCIILFLVATATGVHSQLLLLQSGGGLVKPGGHMRLSCEGSGFTFGDYYMNWVRQAPGKGLEWIASMSPRETATYYADVVRGRFIISRDNAQQSTYLQMNFLRGDDSAVYYCARGVNSRSSNYVYGFEVWGQGTTVTVSSAKTTAPSVYPLAPVCGDTTGSSVTLGCLVKGYFPEPVTLTWNSGSLSSGVHTFPAVLQSDLYTLSSSVTVTSSTWPSQSITCNVAHPASSTKVDKKIEPRGPTIKPCPPCKCPAPNLLGGPSVFIFPPKIKDVLMISLSPIVTCVVVDVSEDDPDVQISWFVNNVEVHTAQTQTHREDYNSTLRVVSALPIQHQDWMSGKEFKCKVNNKDLPAPIERTISKPKGSVRAPQVYVLPPPEEEMTKKQVTLTCMVTDFMPEDIYVEWTNNGKTELNYKNTEPVLDSDGSYFMYSKLRVEKKNWVERNSYSCSVVHEGLHNHHTTKSFSRTPGK
>9LBW_E3 len=234 copies=1 longest=0
MGWSCIILFLVATATGVHSEVVLTQSPGTLSLFPGDRATLSCRASRTIDNTYLAWYQQKPGQAPRLLIYGASSRATGVPDRFSGGGSGTDFTLTIRRLEPEDYAVYYCQQYASPPYTFGRGTQLEIKRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKDINVKWKIDGSERQNGVLNSWTDQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC
>9LFA_E1 len=340 copies=1 longest=1
ASSWPPLELQSSNQSQLFPQNATACDNAPEAWDLLHRVLPTFIISICFFGLLGNLFVLLVFLLPRRQLNVAEIYLANLAASDLVFVLGLPFWAENIWNQFNWPFGALLCRVINGVIKANLFIKIFLVVAICQDRYRVLVHPMASRRQQRRRQARVTCVLIWVVGGLLSIPTFLLRSIQAVPDLNITACILLLPHEAWHFARIVELNILGFLLPLAAIVFFNYHILASLRTVRLLSGSREKDRNLRRITRLVLVVVVAFLVCWAPYHFFAFLEFLFQVQAVRGCFWEDFIDLGLQLANFFAFTNSSLNPVIYVFVGRLFRTKVWELYKQCTPKSLAPISSS
>9LFA_E2 len=171 copies=1 longest=0
MLLVNQSHQGFNKEHTSKMVSAIVLYVLLAAAAHSAFAMAQVQLQESGGGLVQAGESLRLSCAASGTIFRLYDMGWYRRVSGNQRELVASITSGGSTKYGDSVKGRFTISRDNAKNTVYLQMSSLKPEDTAVYYCNAEYRTGIWEELLDGWGQGTQVTVSSHHHHHHEPEA
>9LFC_E1 len=340 copies=1 longest=1
ASSWPPLELQSSNQSQLFPQNATACDNAPEAWDLLHRVLPTFIISICFFGLLGNLFVLLVFLLPRRQLNVAEIYLANLAASDLVFVLGLPFWAENIWNQFNWPFGALLCRVINGVIKANLFIKIFLVVAICQDRYRVLVHPMASRRQQRRRQARVTCVLIWVVGGLLSIPTFLLRSIQAVPDLNITACILLLPHEAWHFARIVELNILGFLLPLAAIVFFNYHILASLRTVRLLSGSREKDRNLRRITRLVLVVVVAFLVCWAPYHFFAFLEFLFQVQAVRGCFWEDFIDLGLQLANFFAFTNSSLNPVIYVFVGRLFRTKVWELYKQCTPKSLAPISSS
>9LFC_E2 len=171 copies=1 longest=0
MLLVNQSHQGFNKEHTSKMVSAIVLYVLLAAAAHSAFAMAQVQLQESGGGLVQAGESLRLSCAASGTIFRLYDMGWYRRVSGNQRELVASITSGGSTKYGDSVKGRFTISRDNAKNTVYLQMSSLKPEDTAVYYCNAEYRTGIWEELLDGWGQGTQVTVSSHHHHHHEPEA
>9LFC_E3 len=10 copies=1 longest=0
XKRPPGFSXI
>9LFD_E1 len=365 copies=1 longest=1
LNVTLQGPTLNGTFAQSKCPQVEWLGWLNTIQPPFLWVLFVLATLENIFVLSVFCLHKSSCTVAEIYLGNLAAADLILACGLPFWAITISNNFDWLFGETLCRVVNAIISMNLYSKIWFLMLVSIDRYLALVKTMSMGRMRGVRWAKLYSLVIWGCTLLLSSPMLVFRTMKEYSDEGHNVTACVISYPSLIWEVFTNMLLNVVGFLLPLSVITFCTMQIMQVLRSVRLLSGSREKDRNLRRITRLVLVVVAVFVICWLPFQISTFLDTLHRLGILSSCQDERIIDVITQIASFMAYSNSCLNPLVYVIVGKRFRKKSWEVYQGVCQKGGCRSEPIQMENSMGTLRTSISVERQIHKLQDWAGSRQ
>9LFD_E2 len=133 copies=1 longest=0
MAQVQLQESGGGLVQAGESLRLSCAASGTIFRLYDMGWYRRVSGNQRELVASITSGGSTKYGDSVKGRFTISRDNAKNTVYLQMSSLKPEDTAVYYCNAEYRTGIWEELLDGWGQGTQVTVSSHHHHHHEPEA
>9LKD_E1 len=317 copies=1 longest=1
MEGKNQTAPSEFIILGFDHLNELQYLLFTIFFLTYICTLGGNVFIIVVTIADSHLHTPMYYFLGNLALIDICYTTTNVPQMMVHLLSEKKIISYGGCVTQLFAFIFFVGSECLLLAAMAYDRYIAICKPLRYSFIMNKALCSWLAASCWTGGFLNSVLHTVLTFHLPFCGNNQINYFFCDIPPLLILSCGDTSLNELALLSIGILIGWTPFLCVILSYLYIISTILRIRSSEGRHKAFSTCASHLLIVILYYGSAIFTYVRPISSYSLEKDRLISVLYSVVTPMLNPVIYTLRNKDIKEAVKAIGRKWQPPVFSSDI
>9LLG_E1 len=516 copies=1 longest=1
RTLNTSAMDGTGLVVERDFSVRILTACFLSLLILSTLLGNTLVCAAVIRFRHLRSKVTNFFVISLAVSDLLVAVLVMPWKAVAEIAGFWPFGSFCNIWVAFDIMCSTASILNLCVISVDRYWAISSPFRYERKMTPKAAFILISVAWTLSVLISFIPVQLSWHKAKPTSPSDGNATSLAETIDNCDSSLSRTYAISSSVISFYIPVAIMIVTYTRIYRILKKELDALNDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRDAILKPFIEELKELKGKPFLIDAVIMGVFVCCWLPFFILNCILPFCGSGETQPFCIDSNTFDVFVWFGWANSSLNPIIYAFNADFRKAFSTLLGCYRLCPATNNAIETVSINNNGAAMFSSHHEPRGSISKECNLVYLIPHAVGSSEDLKKEEAAGIARPLEKLSPALSVILDYDTDVSLEKIQPITQNGQHPT
>9LLG_E2 len=95 copies=1 longest=0
PSPVPPPPPPLPIWKILLIIGTILYIVVFLYISIFLYRLLKTFVPKEERKKWFKFLGILFLIFLILLIYFVVYVIRVLFPPPPPPSPGPPPPPPP
>9LM0_E1 len=877 copies=2 longest=1
GLARAAGSVRLAGGLTLGGLFPVHARGAAGRACGQLKKEQGVHRLEAMLYALDRVNADPELLPGVRLGARLLDTCSRDTYALEQALSFVQALIRGRGDGDEVGVRCPGGVPPLRPAPPERVVAVVGASASSVSIMVANVLRLFAIPQISYASTAPELSDSTRYDFFSRVVPPDSYQAQAMVDIVRALGWNYVSTLASEGNYGESGVEAFVQISREAGGVCIAQSIKIPREPKPGEFSKVIRRLMETPNARGIIIFANEDDIRRVLEAARQANLTGHFLWVGSDSWGAKTSPILSLEDVAVGAITILPKRASIDGFDQYFMTRSLENNRRNIWFAEFWEENFNCKLTSSGTQSDDSTRKCTGEERIGRDSTYEQEGKVQFVIDAVYAIAHALHSMHQALCPGHTGLCPAMEPTDGRMLLQYIRAVRFNGSAGTPVMFNENGDAPGRYDIFQYQATNGSASSGGYQAVGQWAETLRLDVEALQWSGDPHEVPSSLCSLPCGPGERKKMVKGVPCCWHCEACDGYRFQVDEFTCEACPGDMRPTPNHTGCRPTPVVRLSWSSPWAAPPLLLAVLGIVATTTVVATFVRYNNTPIVRASGRELSYVLLTGIFLIYAITFLMVAEPGAAVCAARRLFLGLGTTLSYSALLTKTNRIYRIFEQGKRSVTPPPFISPTSQLVITFSLTSLQVVGMIAWLGARPPHSVIDYEEQRTVDPEQARGVLKCDMSDLSLIGCLGYSLLLMVTCTVYAIKARGVPETFNEAKPIGFTMYTTCIIWLAFVPIFFGTAQSAEKIYIQTTTLTVSLSLSASVSLGMLYVPKTYVILFHPEQNVQKRKRSLKATSTVAAPPKGEDAEAHKFESRASTVPRARDPPVATLEVLFQ
>9LNM_E1 len=614 copies=1 longest=1
MDGKVAVQECGPPAVSWVPEEGEKLDQEDEDQVKDRGQWTNKMEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFFIPYFIFFFVCGIPVFFLEVALGQYTSQGSVTAWRKICPLFQGIGLASVVIESYLNVYYIIILAWALFYLFSSFTSELPWTTCNNFWNTEHCTDFLNHSGAGTVTPFENFTSPVMEFWERRVLGITSGIHDLGSLRWELALCLLLAWVICYFCIWKGVKSTGKVVYFTATFPYLMLVILLIRGVTLPGAYQGIIYYLKPDLFRLKDPQVWMDAGTQIFFSFAICQGCLTALGSYNKYHNNCYKDCIALCFLNSATSFVAGFVVFSILGFMSQEQGVPISEVAESGPGLAFIAFPKAVTMMPLSQLWSCLFFIMLIFLGLDSQFVCVECLVTASIDMFPRQLRKSGRRELLILTIAVMCYLIGLFLVTEGGMYIFQLFDYYASSGICLLFLSLFEVVCISWVYGADRFYDNIEDMIGYRPWPLVKISWLFLTPGLCLATFLFSLSKYTPLKYNNVYVYPPWGYSIGWFLALSSMVCVPLFVVITLLKTRGPFRKRLRQLITPDSSLPQPKQHPCLDGSAGRNFGPSPTREGLIAGEKETHL
>9LNN_E1 len=614 copies=1 longest=1
MDGKVAVQECGPPAVSWVPEEGEKLDQEDEDQVKDRGQWTNKMEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFFIPYFIFFFVCGIPVFFLEVALGQYTSQGSVTAWRKICPLFQGIGLASVVIESYLNVYYIIILAWALFYLFSSFTSELPWTTCNNFWNTEHCTDFLNHSGAGTVTPFENFTSPVMEFWERRVLGITSGIHDLGSLRWELALCLLLAWVICYFCIWKGVKSTGKVVYFTATFPYLMLVILLIRGVTLPGAYQGIIYYLKPDLFRLKDPQVWMDAGTQIFFSFAICQGCLTALGSYNKYHNNCYKDCIALCFLNSATSFVAGFVVFSILGFMSQEQGVPISEVAESGPGLAFIAFPKAVTMMPLSQLWSCLFFIMLIFLGLDSQFVCVECLVTASIDMFPRQLRKSGRRELLILTIAVMCYLIGLFLVTEGGMYIFQLFDYYASSGICLLFLSLFEVVCISWVYGADRFYDNIEDMIGYRPWPLVKISWLFLTPGLCLATFLFSLSKYTPLKYNNVYVYPPWGYSIGWFLALSSMVCVPLFVVITLLKTRGPFRKRLRQLITPDSSLPQPKQHPCLDGSAGRNFGPSPTREGLIAGEKETHL
>9LNO_E1 len=614 copies=1 longest=1
MDGKVAVQECGPPAVSWVPEEGEKLDQEDEDQVKDRGQWTNKMEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFFIPYFIFFFVCGIPVFFLEVALGQYTSQGSVTAWRKICPLFQGIGLASVVIESYLNVYYIIILAWALFYLFSSFTSELPWTTCNNFWNTEHCTDFLNHSGAGTVTPFENFTSPVMEFWERRVLGITSGIHDLGSLRWELALCLLLAWVICYFCIWKGVKSTGKVVYFTATFPYLMLVILLIRGVTLPGAYQGIIYYLKPDLFRLKDPQVWMDAGTQIFFSFAICQGCLTALGSYNKYHNNCYKDCIALCFLNSATSFVAGFVVFSILGFMSQEQGVPISEVAESGPGLAFIAFPKAVTMMPLSQLWSCLFFIMLIFLGLDSQFVCVECLVTASIDMFPRQLRKSGRRELLILTIAVMCYLIGLFLVTEGGMYIFQLFDYYASSGICLLFLSLFEVVCISWVYGADRFYDNIEDMIGYRPWPLVKISWLFLTPGLCLATFLFSLSKYTPLKYNNVYVYPPWGYSIGWFLALSSMVCVPLFVVITLLKTRGPFRKRLRQLITPDSSLPQPKQHPCLDGSAGRNFGPSPTREGLIAGEKETHL
>9LOZ_E1 len=194 copies=1 longest=1
TNLCPFHEVFNATRFASVYAWNRTRISNCVADYSVLYNFAPFFAFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDSKHSGNYDYWYRSFRKSKLKPFERDISTEIYQAGNKPCKGKGPNCYFPLQSYGFRPTYGVGHQPYRVVVLSFELLHAPATVCGP
>9LOZ_E2 len=117 copies=1 longest=0
EVQLVESGGGLVQPGGSLRLSCEASEIIVNRNYMNWVRQAPGKGLEWVSIIYPGGSTFYADSVKGRFTISRDNSKNTMYLQMNSLRAEDTAVYYCARSYGDFYVDFWGQGTLVTVSS
>9LOZ_E3 len=107 copies=1 longest=0
VLTQSPGTLSLSPGERATLSCRATQSIPSTYLAWYQQKPGQAPRLLIYGASSRATGIPDRFSGSGSGTDFTLSINRLEPEDFAVYYCQHYGTSPFTFGPGTKVDIKR
>9LP1_E1 len=117 copies=1 longest=0
EVQLVESGGGLVQPGGSLRLSCEASEIIVNRNYMNWVRQAPGKGLEWVSIIYPGGSTFYADSVKGRFTISRDNSKNTMYLQMNSLRAEDTAVYYCARSYGDFYVDFWGQGTLVTVSS
>9LP1_E2 len=107 copies=1 longest=0
VLTQSPGTLSLSPGERATLSCRATQSIPSTYLAWYQQKPGQAPRLLIYGASSRATGIPDRFSGSGSGTDFTLSINRLEPEDFAVYYCQHYGTSPFTFGPGTKVDIKR
>9LP1_E3 len=194 copies=1 longest=1
NLCPFHEVFNATRFASVYAWNRTRISNCVADYSVLYNFAPFFAFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDSKHSGNYDYWYRSLRKSKLKPFERDISTEIYQAGNKPCKGKGPNCYFPLESYGFRPTYGVGHQPYRVVVLSFELLHAPATVCGPK
>9LQT_E1 len=380 copies=2 longest=1
MEEGGDFDNYYGADNQSECEYTDWKSSGALIPAIYMLVFLLGTTGNGLVLWTVFRSSREKRRSADIFIASLAVADLTFVVTLPLWATYTYRDYDWPFGTFFCKLSSYLIFVNMYASVFCLTGLSFDRYLAIVRPVANARLRLRVSGAVATAVLWVLAALLAMPVMVLRTTGDLENTTKVQCYMDYSMVATVSSEWAWEVGLGVSSTTVGFVVPFTIMLTCYFFIAQTIAGHFRKERIEGLRKRRRLLSIIVVLVVTFALCWMPYHLVKTLYMLGSLLHWPCDFDLFLMNIFPYCTCISYVNSCLNPFLYAFFDPRFRQACTSMLCCGQSRCAGTSHSSSGEKSASYSSGHSQGPGPNMGKGGEQMHEKSIPYSQETLVVD
>9LQY_E1 len=380 copies=2 longest=1
MEEGGDFDNYYGADNQSECEYTDWKSSGALIPAIYMLVFLLGTTGNGLVLWTVFRSSREKRRSADIFIASLAVADLTFVVTLPLWATYTYRDYDWPFGTFFCKLSSYLIFVNMYASVFCLTGLSFDRYLAIVRPVANARLRLRVSGAVATAVLWVLAALLAMPVMVLRTTGDLENTTKVQCYMDYSMVATVSSEWAWEVGLGVSSTTVGFVVPFTIMLTCYFFIAQTIAGHFRKERIEGLRKRRRLLSIIVVLVVTFALCWMPYHLVKTLYMLGSLLHWPCDFDLFLMNIFPYCTCISYVNSCLNPFLYAFFDPRFRQACTSMLCCGQSRCAGTSHSSSGEKSASYSSGHSQGPGPNMGKGGEQMHEKSIPYSQETLVVD
>9LQZ_E1 len=380 copies=2 longest=1
MEEGGDFDNYYGADNQSECEYTDWKSSGALIPAIYMLVFLLGTTGNGLVLWTVFRSSREKRRSADIFIASLAVADLTFVVTLPLWATYTYRDYDWPFGTFFCKLSSYLIFVNMYASVFCLTGLSFDRYLAIVRPVANARLRLRVSGAVATAVLWVLAALLAMPVMVLRTTGDLENTTKVQCYMDYSMVATVSSEWAWEVGLGVSSTTVGFVVPFTIMLTCYFFIAQTIAGHFRKERIEGLRKRRRLLSIIVVLVVTFALCWMPYHLVKTLYMLGSLLHWPCDFDLFLMNIFPYCTCISYVNSCLNPFLYAFFDPRFRQACTSMLCCGQSRCAGTSHSSSGEKSASYSSGHSQGPGPNMGKGGEQMHEKSIPYSQETLVVD
>9LV4_E1 len=430 copies=1 longest=1
AMDLILINSLPLVSDAETSLTCIASGWRPHEPITIGRDFEALMNQHQDPLEVTQDVTREWAKKVVWKREKASKINGAYFCEGRVRGEAIRIRTMKMRQQASFLPATLTMTVDKGDNVNISFKKVLIKEEDAVIYKNGSFIHSVPRHEVPDILEVHLPHAQPQDAGVYSARYIGGNLFTSAFTRLIVRRCEAQKWGPECNHLCTACMNNGVCHEDTGECICPPGFMGRTCEKACELHTFGRTCKERCSGQEGCKSYVFCLPDPYGCSCATGWKGLQCNEACHPGFYGPDCKLRCSCNNGEMCDRFQGCLCSPGWQGLQCEREGIPRMTPKIVDLPDHIEVNSGKFNPICKASGWPLPTNEEMTLVKPDGTVLHPKDFNHTDHFSVAIFTIHRILPPDSGVWVCSVNTVAGMVEKPFNISVKVLPKPLNAPN
>9LV4_E2 len=125 copies=1 longest=0
MSGSEEEQKLVEEAIEKLTKILEELAKKYGEKMKEPKEYYLRMSEKIKKNEQPEQDMAILIHNAGKEVLKLTGDEEAALELAKLSAKLFQYAGDTDGAVRALKQSGIGEEAEKIAEEIRKKAAGS
>9LW1_E1 len=297 copies=2 longest=1
MSRYSYQSLLDWLYGGVDPSFAGNGGPDCAAFLSWQQRLLESVVVLTLALLEILVALRHILRQTKEDGRGSPGSQPEQVTQRPEEGKESLSKNLLLVALCLTFGVEVGFKFATKTVIYLLNPCHLVTMMHIFLLACPPCRGAIVVFKLQMHMLNGALLALLFPVVNTRLLPFELEIYYIQHVMLYVVPIYLLWKGGAYTPEPLSSFRWALLSTGLMFFYHFSVLQILGLVTEVNLNNMLCPAISDPFYGPWYRIWASGHQTLMTMTHGKLVILFSYMAGPLCKYLLDLLRLPAKKID
>9LW3_E1 len=297 copies=2 longest=1
MSRYSYQSLLDWLYGGVDPSFAGNGGPDCAAFLSWQQRLLESVVVLTLALLEILVALRHILRQTKEDGRGSPGSQPEQVTQRPEEGKESLSKNLLLVALCLTFGVEVGFKFATKTVIYLLNPCHLVTMMHIFLLACPPCRGAIVVFKLQMHMLNGALLALLFPVVNTRLLPFELEIYYIQHVMLYVVPIYLLWKGGAYTPEPLSSFRWALLSTGLMFFYHFSVLQILGLVTEVNLNNMLCPAISDPFYGPWYRIWASGHQTLMTMTHGKLVILFSYMAGPLCKYLLDLLRLPAKKID
>9LY5_E1 len=223 copies=1 longest=0
QVQLVQSGAEVKKPGASVKVSCKASGYTFTEYIMHWVRQAPGQRLEWMGGINPGTGGTTYNQKFKDRVTITVDTSASTAYMELSSLRSEDTAVYYCTRRTPPWHFAVWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDK
>9LY5_E2 len=213 copies=1 longest=0
DIQLTQSPSFLSASVGDRVTITCSASSSVSYIHWYQQKPGKAPKRWIYDTSRLASGVPSRFSGSGSGTEFTLTISSLQPEDFATYYCQQWSSAPLTFGGGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9LY5_E3 len=448 copies=1 longest=1
LEVQVPEDPVVALVGTDATLCCSFSPEPGFSLAQLNLIWQLTDTKQLVHSFAEGQDQGSAYANRTALFPDLLAQGNASLRLQRVRVADEGSFTCFVSIRDFGSAAVSLQVAAPYSKPSMTLEPNKDLRPGDTVTITCSSYQGYPEAEVFWQDGQGVPLTGNVTTSQMANEQGLFDVHSILRVVLGANGTYSCLVRNPVLQQDAHSSVTITPQRSPTGAVEVQVPEDPVVALVGTDATLRCSFSPEPGFSLAQLNLIWQLTDTKQLVHSFTEGRDQGSAYANRTALFPDLLAQGNASLRLQRVRVADEGSFTCFVSIRDFGSAAVSLQVAAPYSKPSMTLEPNKDLRPGDTVTITCSSYRGYPEAEVFWQDGQGVPLTGNVTTSQMANEQGLFDVHSVLRVVLGANGTYSCLVRNPVLQQDAHGSVTITGQPMTGGGGSHHHHHHHHHH
>9M9M_E1 len=424 copies=2 longest=1
MAGRGFSWGPGHLNEDNARFLLLAALIVLYLLGGAAVFSALELAHERQAKQRWEERLANFSRGHNLSRDELRGFLRHYEEATRAGIRVDNVRPRWDFTGAFYFVGTVVSTIGFGMTTPATVGGKIFLIFYGLVGCSSTILFFNLFLERLITIIAYIMKSCHQRQLRRRGALPQESLKDAGQCEVDSLAGWKPSVYYVMLILCTASILISCCASAMYTPIEGWSYFDSLYFCFVAFSTIGFGDLVSSQNAHYESQGLYRFANFVFILMGVCCIYSLFNVISILIKQSLNWILRKMDSGCCPQCQRGLLRSRRNVVMPGSVRNRCNISIETDGVAESDTDGRRLSGEMISMKDLLAANKASLAILQKQLSEMANGCPHQTSTLARDNEFSGGVGAFAIMNNRLAETSGDRLEGGVEGGVEENLYFQ
>9MA0_E1 len=478 copies=1 longest=1
MEAALLVCQYTIQSLIHLTGEDPGFFNVEIPEFPFYPTCNVCTADVNVTINFDVGGKKHQLDLDFGQLTPHTKAVYQPRGAFGGSENATNLFLLELLGAGELALTMRSKKLPINVTTGEEQQVSLESVDVYFQDVFGTMWCHHAEMQNPVYLIPETVPYIKWDNCNSTNITAVVRAQGLDVTLPLSLPTSAQDSNFSVKTQMLGNEIDIECIMEDGEISQVLPGDNKFNITCSGYESHVPSGGILTSTSPVATPIPGTGYAYSLRLTPRPVSRFLGNNSILYVFYSGNGPKASGGDYCIQSNIVFSDEIPASQDMPTNTTDITYVGDNATYSVPMVTSEDANSPNVTVTAFWAWPNNTETDFKCKWTLTSGTPSGCENISGAFASNRTFDITVSGLGTAPKTLIITRTATNATTTTHKVIFSKAPESTTTSPTLNTTGFADPNTTTGLPSSTHVPTNLTAPASTGPTVSTGSHHHHHH
>9MA0_E2 len=248 copies=1 longest=0
MGWSCIILFLVATATGVHSQVQLVQSGAEVKKPGASVKVSCKASGYTFTGYYIHWVRQAPGQGLEWMGWMNPNSGATNYAQKFQGRVTMTRDTSINTAYMELSSLRSDDTAVYFCAKDPRYSSSWRHNWFDPWGQGTLVSVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCDK
>9MA0_E3 len=237 copies=1 longest=0
MGWSCIILFLVATATGSWADIVLTQSPLSLTVTPGEPASISCRSSQSLLHSSGHNFLSWYLQKPGQSPQLLIYLGSNRASGVPDRFSGSGTGADFTLKISRVEAEDVGVYYCLQTLRAPFTFGQGTRLEIKGQPKAAPSVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADSSPVKAGVETTTPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTVAPTECS
>9MAN_E1 len=500 copies=2 longest=0
GSHHHHHHHHMAKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELAKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTNAAAENLYFQGEFKTDSSFIMDSDPRRCMRHHYVDSISHPLYKCSSKMVLLARCEGHCSQASRSEPLVSFSTVLKQPFRSSCHCCRPQTSKLKALRLRCSGGMRLTATYRYILSCHCEECNS
>9MAN_E2 len=501 copies=2 longest=1
GSHHHHHHHHMAKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELAKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTNAAAENLYFQGEFYEQELMVPVQWSDMVTLKARMTNYGLPRYRWLTHAWNFFQREFKCCGVVYFTDWLEMTEMDWPPDSCCVREFPGCSKQAHQEDLSDLYQEGCGKKMYSFLRGTKQLQVLR
>9MKG_E1 len=222 copies=1 longest=0
QVQLVQSGAEVKKPGSSVKVSCKASGGIFSNHAISWVRQAPGQGLEWMGGIIPIFGTANYAQKFQGRVTITADESTSTAFMELSSLRSEDTAVYYCAREWADVFDIWGQGTMVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDK
>9MKG_E2 len=219 copies=1 longest=0
EIVLTQSPLSLPVTPGEPASISCRSSQSLLHSNGYNYLDWYLQKPGQSPQLLIYLGVNSVSGVPDRFSGSGSGTDFTLKISRVEAEDVGVYYCMQALQTPWTFGQGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9MKG_E3 len=277 copies=1 longest=1
GSMDPNFWLQVQESVTVQEGLCVLVPCTFFHPIPYYDKNSPVHGYWFREGAIISRDSPVATNKLDQEVQEETQGRFRLLGDPSRNNCSLSIVDARRRDNGSYFFRMERGSTKYSYKSPQLSVHVTDLTHRPKILIPGTLEPGHSKNLTCSVSWACEQGTPPIFSWLSAAPTSLGPRTTHSSVLIITPRPQDHGTNLTCQVKFAGAGVTTERTIQLNVTYVPQNPTTGIFPGDGSGKQETRAGVVHGNSGLNDIFEAQKIEWHENLYFQGDYKDDDDK
>9MM5_E1 len=69 copies=1 longest=0
DTNFELGVEYFMLGLQALVHGDYDNAIKYFNKAIEYFKKSSDKEKAAKYIALAQKYIDEAKKLKAEKEA
>9MM5_E2 len=149 copies=1 longest=0
MKTIIALSYIFCLVFADYKDDDDKHGSCQEANYGALLRELCLTQFQVDMEAVGETLWCDWGRTIRSYRELADCTWHMAEKLGCFWPNAEVDRFFLAVHGRYFRSCPISGRAVRDPPGSILYPFIVVPITVTLLVTALVVWQSKRTEGIV
>9MM5_E3 len=490 copies=1 longest=1
MKTIIALSYIFCLVFADYKDDDDLEVLFQGPAELEESPEDSIQLGVTRNKIMTAQYECYQKIMQDPIQQAEGVYCNRTWDGWLCWNDVAAGTESMQLCPDYFQDFDPSEKVTKICDQDGNWFRHPASNRTWTNYTQCNVNTHEKVKTALNLFYLTIIGHGLSIASLLISLGIFFYFKSLSCQRITLHKNLFFSFVCNSVVTIIHLTAVANNQALVATNPVSCKVSQFIHLYLMGCNYFWMLCEGIYLHTLIVVAVFAEKQHLMWYYFLGWGFPLIPACIHAIARSLYYNDNCWISSDTHLLYIIHGPICAALLVNLFFLLNIVRVLITKLKVTHQAESNLYMKAVRATLILVPLLGIEFVLIPWRPEGKIAEEVYDYIMHILMHFQGLLVSTIFCFFNGEVQAILRRNWNQYKIQFGNSFSNSEALRSASYTVSTISDGPGYSHDCPSEHLNGKSIHDIENVLLKPENLYNPAGLEVLFQGPHHHHHHHH
>9MNM_E1 len=231 copies=4 longest=1
PIVQNLQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEGATPQDLNTMLNTVGGHQAAMQMLKETINEEAAEWDRLHPVHAGPIAPGQMREPRGSDIAGTTSTLQEQIGWMTHNPPIPVGEIYKRWIILGLNKIVRMYSPTSILDIRQGPKEPFRDYVDRFYKTLRAEQASQEVKNWMTETLLVQNANPDCKTILKALGPGATLEEMMTACQGVGGPGHKARVL
>9MUH_E1 len=265 copies=2 longest=1
MKRQNVRTLALIVCTFTYLLVGAAVFDALESEPEMIERQRLELRQLELRARYNLSEGGYEELERVVLRLKPHKAGVQWRFAGSFYFAITVITTIGYGHAAPSTDGGKVFCMFYALLGIPLTLVMFQSLGERINTFVRYLLHRAKRGLGMRHAEVSMANMVLIGFVSCISTLCIGAAAFSYYERWTFFQAYYYCFITLTTIGFGDYVALQKDQALQTQPQYVAFSFVYILTGLTVIGAFLNLVVLRFMTMNAEDEKRSNSLEVLFQ
>9MUN_E1 len=561 copies=1 longest=1
MSPTISHKDSSRQRRPGNFSHSLDMKSGPLPPGGWDDSHLDSAGREGDREALLGDTGTGDFLKAPQSFRAELSSILLLLFLYVLQGIPLGLAGSIPLILQSKNVSYTDQAFFSFVFWPFSLKLLWAPLVDAVYVKNFGRRKSWLVPTQYILGLFMIYLSTQVDRLLGNTDDRTPDVIALTVAFFLFEFLAATQDIAVDGWALTMLSRENVGYASTCNSVGQTAGYFLGNVLFLALESADFCNKYLRFQPQPRGIVTLSDFLFFWGTVFLITTTLVALLKKENEVSVVKEETQGITDTYKLLFAIIKMPAVLTFCLLILTAKIGFSAADAVTGLKLVEEGVPKEHLALLAVPMVPLQIILPLIISKYTAGPQPLNTFYKAMPYRLLLGLEYALLVWWTPKVEGDGMVPPGPIYYYIVVLLSYALHQVTVYSMYVSIMAFNAKVSDPLIGGTYMTLLNTVSNLGGNWPSTVALWLVDPLTVKECVGASNQNCRTPDAVELCKKLGGSCVTALDGYYVESIICVFIGFGWWFFLGPKFKKLQDEGSSSWKCKRNNSNSLEVLFQ
>9NEN_E1 len=538 copies=1 longest=1
MEESVNQMQPLNEKQIANSQDGYVWQVTDMNRLHRFLCFGSEGGTYYIKEQKLGLENAEALIRLIEDGRGCEVIQEIKSFSQEGRTTKQEPMLFALAICSQCSDISTKQAAFKAVSEVCRIPTHLFTFIQFKKDLKESMKCGMWGRALRKAIADWYNEKGGMALALAVTKYKQRNGWSHKDLLRLSHLKPSSEGLAIVTKYITKGWKEVHELYKEKALSVETEKLLKYLEAVEKVKRTKDELEVIHLIEEHRLVREHLLTNHLKSKEVWKALLQEMPLTALLRNLGKMTANSVLEPGNSEVSLVCEKLCNEKLLKKARIHPFHILIALETYKTGHGLRGKLKWRPDEEILKALDAAFYKTFKTVEPTGKRFLLAVDVSASMNQRVLGSILNASTVAAAMCMVVTRTEKDSYVVAFSDEMVPCPVTTDMTLQQVLMAMSQIPAGGTDCSLPMIWAQKTNTPADVFIVFTDNETFAGGVHPAIALREYRKKMDIPAKLIVCGMTSNGFTIADPDDRGMLDMCGFDTGALDVIRNFTLDMI
>9NNZ_E1 len=514 copies=1 longest=1
QVQLVESGGGLVQTKTTTSVIDTTNDAQNLLTQAQTIVNTLKDYCPILIAKSSSSNGGTNNANTPSWQTAGGGKNSCATFGAEFSAASDMINNAQKIVQETQQLSANQPKNITQPHNLNLNSPSSLTALAQKMLKNAQSQAEILKLANQVESDFNKLSSGHLKDYIGKCDASAISSANMTMQNQKNNWGNGCAGVEETQSLLKTSAADFNNQTPQINQAQNLANTLIQELGNNTYEQLSRLLTNDNGTNSKTSAQAINQAVNNLNERAKTLAGGTTNSPAYQATLLALRSVLGLWNSMGYAVICGGYTKSPGENNQKDFHYTDENGNGTTINCGGSTNSNGTHSYNGTNTLKADKNVSLSIEQYEKIHEAYQILSKALKQAGLAPLNSKGEKLEAHVTTSKYGSLRLSCAASGFTFSRYVMNWVRQAPGKGLEWVSGISRDGSYTDYADSVKGRFTISRDNAKDTLYLQMNSLKPEDTAVYYCSKYNSVVTTPPGQGTQVTVSSHHHHHHEPEA
>9NNZ_E2 len=348 copies=1 longest=0
MNGTEGPNFYVPFSNKTGVVRSPFEAPQYYLAEPWQFSMLAAYMFLLIMLGFPINFLTLYVTVQHKKLRTPLNYILLNLAVADLFMVFGGFTTTLYTSLHGYFVFGPTGCNLEGFFATLGGEIALWSLVVLAIERYVVVCKPMSNFRFGENHAIMGVAFTWVMALACAAPPLVGWSRYIPEGMQCSCGIDYYTPHEETNNESFVIYMFVVHFIIPLIVIFFCYGQLVFTVKEAAAQQQESATTQKAEKEVTRMVIIMVIAFLICWLPYAGVAFYIFTHQGSDFGPIFMTIPAFFAKTSAVYNPVIYIMMNKQFRNCMVTTLCCGKNPLGDDEASTTVSKTETSQVAPA
>9NP0_E1 len=351 copies=1 longest=1
GPLGSMAMVSEFLKQAWFIENEEQEYVQTVKSSKGGPGSAVSPYPTFNPSSDVAALHKAIMVKGVDEATIIDILTKRNNAQRQQIKAAYLQETGKPLDETLKKALTGHLEEVVLALLKTPAQFDADELRAAMKGLGTDEDTLIEILASRTNKEIRDINRVYREELKRDLAKDITSDTSGDFRNALLSLAKGDRSEDFGVNEDLADSDARALYEAGERRKGTDVNVFNTILTTRSYPQLRRVFQKYTKYSKHDMNKVLDLELKGDIEKCLTAIVKCATSKPAFFAEKLHQAMKGVGTRHKALIRIMVSRSEIDMNDIKAFYQKMYGISLCQAILDETKGDYEKILVALCGGN
>9NP0_E2 len=225 copies=1 longest=0
EVKLVESGGGLVQPGGSLRLSCAISGFTFSDFYMEWVRQFPGKRLEWVAASKNKANDYTTEYSASVKGRFIVSRDTSQSILYLQMNALRAEDTAIYYCAAGGYDEGVGWYFDVWGAGTTVTVSSAAKTTPPSVYPLAPGSAAQTNSMVTLGCLVKGYFPEPVTVTWNSGSLSSGVHTFPAVLQSDLYTLSSSVTVPSSTWPSETVTCNVAHPASSTKVDKKIVPR
>9NP0_E3 len=212 copies=1 longest=0
DIVMTQSQKFMSTSVGDRVSVTCKASQNVGTNVAWFQQKPGLSPKPLIYSASYRYRGVPDRFTGSASGTDFTLTISSVQSEDLAEYFCQQYNNYPYTFGGGTRLEINRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKDINVKWKIDGSERQNGVLNSWTDQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRN
>9NU4_E1 len=614 copies=1 longest=1
MADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLLFSLAAVSIATMFSRVLGFARDAIVARIFGAGMATDAFFVAFKLPNLLRRIFAEGAFSQAFVPILAEYKSKQGEDATRVFVSYVSGLLTLALAVVTVAGMLAAPWVIMVTAPGFADTADKFALTSQLLKITFPYILLISLASLVGAILNTWNRFSIPAFAPTLLNISMIGFALFAAPYFNPPVLALAWAVTVGGVLQLVYQLPHLKKIGMLVLPRINFHDAGAMRVVKQMGPAILGVSVSQISLIINTIFASFLASGSVSWMYYADRLMEFPSGVLGVALGTILLPSLSKSFASGNHDEYNRLMDWGLRLCFLLALPSAVALGILSGPLTVSLFQYGKFTAFDALMTQRALIAYSVGLIGLIVVKVLAPGFYSRQDIKTPVKIAIVTLILTQLMNLAFIGPLKHAGLSLSIGLAACLNASLLYWQLRKQKIFTPQPGWMAFLLRLVVAVLVMSGVLLGMLHIMPEWSLGTMPWRLLRLMAVVLAGIAAYFAALAVLGSAWSHPQFEK
>9NU4_E2 len=66 copies=1 longest=0
MGHHHHHHGSGSGSGSGSGSGSGSGSGSGSKYIINLAFCVLLLVAGDSIAYRVSQYLAPLVDTFTK
>9O7G_E1 len=121 copies=1 longest=1
VQLVQSGSEVKKPGASVKVSCKASGYTFTNYGISWVRQAPGQGLEWMGWISAYNGNTNYAQNFQGRVTMTTDTSTKTAYMELRSLRSDDTAVYFCARSGYSGYDNNLGAFDIWGQGTMVTV
>9O7G_E2 len=77 copies=1 longest=0
DIPDYLCGKISFELMREPCITPSGITYDRKDIEEHLQRVGHFDPVTRSPLTQEQLIPNLAMKEVIDAFISENGWVED
>9O7G_E3 len=110 copies=1 longest=0
SVVTQPPSVSAAPGQKVTISCSGTSSNIGNSYVTWYQQLPGAAPKLLIYDNNKRPSGIPDRFSGSKSGTSATLGITGLQTGDEADYYCATWDSSLTAGRVFGGGTRLTVL
>9O7I_E1 len=72 copies=2 longest=0
SIEERRIHQESELHSYLSRLIAAERERELEECQRNHEGDEDDSHVRAQQACIEAKHDKYMADMDELFSQVDE
>9O7I_E2 len=214 copies=1 longest=0
LETTLTQSPGTLSLSPGERATLSCRASQSVSSTFLAWYQQKPGQAPRLLIYGASRRATGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQRYGATYTFGQGTKLEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGE
>9O7I_E3 len=224 copies=1 longest=1
AQVQLQQSGPGLMKGGQTLSLTCAISGDSVSSNSAAWNWIRQSPSRGLEWLGRTYYRSKWYNEYAVSVKSRITINPDTSKNQFSLQLNSVTPEDTAVYYCARAVAASFDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSC
>9OBT_E1 len=847 copies=2 longest=0
MSTMHLLTFALLFSCSFARAASDPKIVNIGAVLSTRKHEQMFREAVNQANKRHGSWKIQLQATSVTHKPNAIQMALSVCEDLISSQVYAILVSHPPTPNDHFTPTPVSYTAGFYRIPVLGLTTRMSIYSDKSIHLSFLRTVPPYSHQSSVWFEMMRVYNWNHIILLVSDDHEGRAAQKRLETLLEERESKAEKVLQFDPGTKNVTALLMEARELEARVIILSASEDDAATVYRAAAMLDMTGSGYVWLVGEREISGNALRYAPDGIIGLQLINGKNESAHISDAVGVVAQAVHELLEKENITDPPRGCVGNTNIWKTGPLFKRVLMSSKYADGVTGRVEFNEDGDRKFAQYSIMNLQNRKLVQVGIYNGTHVIPNDRKIIWPGGETEKPRGYQMSTRLKIVTIHQEPFVYVKPTMSDGTCKEEFTVNGDPVKKVICTGPNDTSPGSPRHTVPQCCYGFCIDLLIKLARTMQFTYEVHLVADGKFGTQERVQNSNKKEWNGMMGELLSGQADMIVAPLTINNERAQYIEFSKPFKYQGLTILVKKEIPRSTLDSFMQPFQSTLWLLVGLSVHVVAVMLYLLDRFSPFGRFKVNSEEEEEDALTLSSAMWFSWGVLLNSGIGEGAPRSFSARILGMVWAGFAMIIVASYTANLAAFLVLDRPEERITGINDPRLRNPSDKFIYATVKQSSVDIYFRRQVELSTMYRHMEKHNYESAAEAIQAVRDNKLHAFIWDSAVLEFEASQKCDLVTTGELFFRSGFGIGMRKDSPWKQQVSLSILKSHENGFMEDLDKTWVRYQECDSRSNAPATLTFENMAGVFMLVAGGIVAGIFLIFIEIAYKRHKDANGAQ
>9OBT_E2 len=862 copies=2 longest=1
GWSHPQFEKGGGSGGGSGGSAWSHPQFEKGALVPRGRSQKSPPSIGIAVILVGTSDEVAIKDAHEKDDFHHLSVVPRVELVAMNETDPKSIITRICDLMSDRKIQGVVFADDTDQEAIAQILDFISAQTLTPILGIHGGSSMIMADKDESSMFFQFGPSIEQQASVMLNIMEEYDWYIFSIVTTYFPGYQDFVNKIRSTIENSFVGWELEEVLLLDMSLDDGDSKIQNQLKKLQSPIILLYCTKEEATYIFEVANSVGLTGYGYTWIVPSLVAGDTDTVPSEFPTGLISVSYDEWDYGLPARVRDGIAIITTAASDMLSEHSFIPEPKSSCYNTHEKRIYQSNMLNRYLINVTFEGRNLSFSEDGYQMHPKLVIILLNKERKWERVGKWKDKSLQMKYYVWPRMCPETEEQEDDHLSIVTLEEAPFVIVESVDPLSGTCMRNTVPCQKRIISENKTDEEPGYIKKCCKGFCIDILKKISKSVKFTYDLYLVTNGKHGKKINGTWNGMIGEVVMKRAYMAVGSLTINEERSEVVDFSVPFIETGISVMVSRSNGTVSPSAFLEPFSADVWVMMFVMLLIVSAVAVFVFEYFSPVGYNRCLADGREPGGPSFTIGKAIWLLWGLVFNNSVPVQNPKGTTSKIMVSVWAFFAVIFLASYTANLAAFMIQEEYVDQVSGLSDKKFQRPNDFSPPFRFGTVPNGSTERNIRNNYAEMHAYMGKFNQRGVDDALLSLKTGKLDAFIYDAAVLNYMAGRDEGCKLVTIGSGKVFASTGYGIAIQKDSGWKRQVDLAILQLFGDGEMEELEALWLTGICHNEKNEVMSSQLDIDNMAGVFYMLGAAMALSLITFICEHLFYWQFRHSFMG
>9OBV_E1 len=847 copies=2 longest=0
MSTMHLLTFALLFSCSFARAASDPKIVNIGAVLSTRKHEQMFREAVNQANKRHGSWKIQLQATSVTHKPNAIQMALSVCEDLISSQVYAILVSHPPTPNDHFTPTPVSYTAGFYRIPVLGLTTRMSIYSDKSIHLSFLRTVPPYSHQSSVWFEMMRVYNWNHIILLVSDDHEGRAAQKRLETLLEERESKAEKVLQFDPGTKNVTALLMEARELEARVIILSASEDDAATVYRAAAMLDMTGSGYVWLVGEREISGNALRYAPDGIIGLQLINGKNESAHISDAVGVVAQAVHELLEKENITDPPRGCVGNTNIWKTGPLFKRVLMSSKYADGVTGRVEFNEDGDRKFAQYSIMNLQNRKLVQVGIYNGTHVIPNDRKIIWPGGETEKPRGYQMSTRLKIVTIHQEPFVYVKPTMSDGTCKEEFTVNGDPVKKVICTGPNDTSPGSPRHTVPQCCYGFCIDLLIKLARTMQFTYEVHLVADGKFGTQERVQNSNKKEWNGMMGELLSGQADMIVAPLTINNERAQYIEFSKPFKYQGLTILVKKEIPRSTLDSFMQPFQSTLWLLVGLSVHVVAVMLYLLDRFSPFGRFKVNSEEEEEDALTLSSAMWFSWGVLLNSGIGEGAPRSFSARILGMVWAGFAMIIVASYTANLAAFLVLDRPEERITGINDPRLRNPSDKFIYATVKQSSVDIYFRRQVELSTMYRHMEKHNYESAAEAIQAVRDNKLHAFIWDSAVLEFEASQKCDLVTTGELFFRSGFGIGMRKDSPWKQQVSLSILKSHENGFMEDLDKTWVRYQECDSRSNAPATLTFENMAGVFMLVAGGIVAGIFLIFIEIAYKRHKDANGAQ
>9OBV_E2 len=862 copies=2 longest=1
GWSHPQFEKGGGSGGGSGGSAWSHPQFEKGALVPRGRSQKSPPSIGIAVILVGTSDEVAIKDAHEKDDFHHLSVVPRVELVAMNETDPKSIITRICDLMSDRKIQGVVFADDTDQEAIAQILDFISAQTLTPILGIHGGSSMIMADKDESSMFFQFGPSIEQQASVMLNIMEEYDWYIFSIVTTYFPGYQDFVNKIRSTIENSFVGWELEEVLLLDMSLDDGDSKIQNQLKKLQSPIILLYCTKEEATYIFEVANSVGLTGYGYTWIVPSLVAGDTDTVPSEFPTGLISVSYDEWDYGLPARVRDGIAIITTAASDMLSEHSFIPEPKSSCYNTHEKRIYQSNMLNRYLINVTFEGRNLSFSEDGYQMHPKLVIILLNKERKWERVGKWKDKSLQMKYYVWPRMCPETEEQEDDHLSIVTLEEAPFVIVESVDPLSGTCMRNTVPCQKRIISENKTDEEPGYIKKCCKGFCIDILKKISKSVKFTYDLYLVTNGKHGKKINGTWNGMIGEVVMKRAYMAVGSLTINEERSEVVDFSVPFIETGISVMVSRSNGTVSPSAFLEPFSADVWVMMFVMLLIVSAVAVFVFEYFSPVGYNRCLADGREPGGPSFTIGKAIWLLWGLVFNNSVPVQNPKGTTSKIMVSVWAFFAVIFLASYTANLAAFMIQEEYVDQVSGLSDKKFQRPNDFSPPFRFGTVPNGSTERNIRNNYAEMHAYMGKFNQRGVDDALLSLKTGKLDAFIYDAAVLNYMAGRDEGCKLVTIGSGKVFASTGYGIAIQKDSGWKRQVDLAILQLFGDGEMEELEALWLTGICHNEKNEVMSSQLDIDNMAGVFYMLGAAMALSLITFICEHLFYWQFRHSFMG
>9OBW_E1 len=847 copies=2 longest=0
MSTMHLLTFALLFSCSFARAASDPKIVNIGAVLSTRKHEQMFREAVNQANKRHGSWKIQLQATSVTHKPNAIQMALSVCEDLISSQVYAILVSHPPTPNDHFTPTPVSYTAGFYRIPVLGLTTRMSIYSDKSIHLSFLRTVPPYSHQSSVWFEMMRVYNWNHIILLVSDDHEGRAAQKRLETLLEERESKAEKVLQFDPGTKNVTALLMEARELEARVIILSASEDDAATVYRAAAMLDMTGSGYVWLVGEREISGNALRYAPDGIIGLQLINGKNESAHISDAVGVVAQAVHELLEKENITDPPRGCVGNTNIWKTGPLFKRVLMSSKYADGVTGRVEFNEDGDRKFAQYSIMNLQNRKLVQVGIYNGTHVIPNDRKIIWPGGETEKPRGYQMSTRLKIVTIHQEPFVYVKPTMSDGTCKEEFTVNGDPVKKVICTGPNDTSPGSPRHTVPQCCYGFCIDLLIKLARTMQFTYEVHLVADGKFGTQERVQNSNKKEWNGMMGELLSGQADMIVAPLTINNERAQYIEFSKPFKYQGLTILVKKEIPRSTLDSFMQPFQSTLWLLVGLSVHVVAVMLYLLDRFSPFGRFKVNSEEEEEDALTLSSAMWFSWGVLLNSGIGEGAPRSFSARILGMVWAGFAMIIVASYTANLAAFLVLDRPEERITGINDPRLRNPSDKFIYATVKQSSVDIYFRRQVELSTMYRHMEKHNYESAAEAIQAVRDNKLHAFIWDSAVLEFEASQKCDLVTTGELFFRSGFGIGMRKDSPWKQQVSLSILKSHENGFMEDLDKTWVRYQECDSRSNAPATLTFENMAGVFMLVAGGIVAGIFLIFIEIAYKRHKDANGAQ
>9OBW_E2 len=862 copies=2 longest=1
GWSHPQFEKGGGSGGGSGGSAWSHPQFEKGALVPRGRSQKSPPSIGIAVILVGTSDEVAIKDAHEKDDFHHLSVVPRVELVAMNETDPKSIITRICDLMSDRKIQGVVFADDTDQEAIAQILDFISAQTLTPILGIHGGSSMIMADKDESSMFFQFGPSIEQQASVMLNIMEEYDWYIFSIVTTYFPGYQDFVNKIRSTIENSFVGWELEEVLLLDMSLDDGDSKIQNQLKKLQSPIILLYCTKEEATYIFEVANSVGLTGYGYTWIVPSLVAGDTDTVPSEFPTGLISVSYDEWDYGLPARVRDGIAIITTAASDMLSEHSFIPEPKSSCYNTHEKRIYQSNMLNRYLINVTFEGRNLSFSEDGYQMHPKLVIILLNKERKWERVGKWKDKSLQMKYYVWPRMCPETEEQEDDHLSIVTLEEAPFVIVESVDPLSGTCMRNTVPCQKRIISENKTDEEPGYIKKCCKGFCIDILKKISKSVKFTYDLYLVTNGKHGKKINGTWNGMIGEVVMKRAYMAVGSLTINEERSEVVDFSVPFIETGISVMVSRSNGTVSPSAFLEPFSADVWVMMFVMLLIVSAVAVFVFEYFSPVGYNRCLADGREPGGPSFTIGKAIWLLWGLVFNNSVPVQNPKGTTSKIMVSVWAFFAVIFLASYTANLAAFMIQEEYVDQVSGLSDKKFQRPNDFSPPFRFGTVPNGSTERNIRNNYAEMHAYMGKFNQRGVDDALLSLKTGKLDAFIYDAAVLNYMAGRDEGCKLVTIGSGKVFASTGYGIAIQKDSGWKRQVDLAILQLFGDGEMEELEALWLTGICHNEKNEVMSSQLDIDNMAGVFYMLGAAMALSLITFICEHLFYWQFRHSFMG
>9OBY_E1 len=847 copies=2 longest=0
MSTMHLLTFALLFSCSFARAASDPKIVNIGAVLSTRKHEQMFREAVNQANKRHGSWKIQLQATSVTHKPNAIQMALSVCEDLISSQVYAILVSHPPTPNDHFTPTPVSYTAGFYRIPVLGLTTRMSIYSDKSIHLSFLRTVPPYSHQSSVWFEMMRVYNWNHIILLVSDDHEGRAAQKRLETLLEERESKAEKVLQFDPGTKNVTALLMEARELEARVIILSASEDDAATVYRAAAMLDMTGSGYVWLVGEREISGNALRYAPDGIIGLQLINGKNESAHISDAVGVVAQAVHELLEKENITDPPRGCVGNTNIWKTGPLFKRVLMSSKYADGVTGRVEFNEDGDRKFAQYSIMNLQNRKLVQVGIYNGTHVIPNDRKIIWPGGETEKPRGYQMSTRLKIVTIHQEPFVYVKPTMSDGTCKEEFTVNGDPVKKVICTGPNDTSPGSPRHTVPQCCYGFCIDLLIKLARTMQFTYEVHLVADGKFGTQERVQNSNKKEWNGMMGELLSGQADMIVAPLTINNERAQYIEFSKPFKYQGLTILVKKEIPRSTLDSFMQPFQSTLWLLVGLSVHVVAVMLYLLDRFSPFGRFKVNSEEEEEDALTLSSAMWFSWGVLLNSGIGEGAPRSFSARILGMVWAGFAMIIVASYTANLAAFLVLDRPEERITGINDPRLRNPSDKFIYATVKQSSVDIYFRRQVELSTMYRHMEKHNYESAAEAIQAVRDNKLHAFIWDSAVLEFEASQKCDLVTTGELFFRSGFGIGMRKDSPWKQQVSLSILKSHENGFMEDLDKTWVRYQECDSRSNAPATLTFENMAGVFMLVAGGIVAGIFLIFIEIAYKRHKDANGAQ
>9OBY_E2 len=862 copies=2 longest=1
GWSHPQFEKGGGSGGGSGGSAWSHPQFEKGALVPRGRSQKSPPSIGIAVILVGTSDEVAIKDAHEKDDFHHLSVVPRVELVAMNETDPKSIITRICDLMSDRKIQGVVFADDTDQEAIAQILDFISAQTLTPILGIHGGSSMIMADKDESSMFFQFGPSIEQQASVMLNIMEEYDWYIFSIVTTYFPGYQDFVNKIRSTIENSFVGWELEEVLLLDMSLDDGDSKIQNQLKKLQSPIILLYCTKEEATYIFEVANSVGLTGYGYTWIVPSLVAGDTDTVPSEFPTGLISVSYDEWDYGLPARVRDGIAIITTAASDMLSEHSFIPEPKSSCYNTHEKRIYQSNMLNRYLINVTFEGRNLSFSEDGYQMHPKLVIILLNKERKWERVGKWKDKSLQMKYYVWPRMCPETEEQEDDHLSIVTLEEAPFVIVESVDPLSGTCMRNTVPCQKRIISENKTDEEPGYIKKCCKGFCIDILKKISKSVKFTYDLYLVTNGKHGKKINGTWNGMIGEVVMKRAYMAVGSLTINEERSEVVDFSVPFIETGISVMVSRSNGTVSPSAFLEPFSADVWVMMFVMLLIVSAVAVFVFEYFSPVGYNRCLADGREPGGPSFTIGKAIWLLWGLVFNNSVPVQNPKGTTSKIMVSVWAFFAVIFLASYTANLAAFMIQEEYVDQVSGLSDKKFQRPNDFSPPFRFGTVPNGSTERNIRNNYAEMHAYMGKFNQRGVDDALLSLKTGKLDAFIYDAAVLNYMAGRDEGCKLVTIGSGKVFASTGYGIAIQKDSGWKRQVDLAILQLFGDGEMEELEALWLTGICHNEKNEVMSSQLDIDNMAGVFYMLGAAMALSLITFICEHLFYWQFRHSFMG
>9OBZ_E1 len=847 copies=2 longest=0
MSTMHLLTFALLFSCSFARAASDPKIVNIGAVLSTRKHEQMFREAVNQANKRHGSWKIQLQATSVTHKPNAIQMALSVCEDLISSQVYAILVSHPPTPNDHFTPTPVSYTAGFYRIPVLGLTTRMSIYSDKSIHLSFLRTVPPYSHQSSVWFEMMRVYNWNHIILLVSDDHEGRAAQKRLETLLEERESKAEKVLQFDPGTKNVTALLMEARELEARVIILSASEDDAATVYRAAAMLDMTGSGYVWLVGEREISGNALRYAPDGIIGLQLINGKNESAHISDAVGVVAQAVHELLEKENITDPPRGCVGNTNIWKTGPLFKRVLMSSKYADGVTGRVEFNEDGDRKFAQYSIMNLQNRKLVQVGIYNGTHVIPNDRKIIWPGGETEKPRGYQMSTRLKIVTIHQEPFVYVKPTMSDGTCKEEFTVNGDPVKKVICTGPNDTSPGSPRHTVPQCCYGFCIDLLIKLARTMQFTYEVHLVADGKFGTQERVQNSNKKEWNGMMGELLSGQADMIVAPLTINNERAQYIEFSKPFKYQGLTILVKKEIPRSTLDSFMQPFQSTLWLLVGLSVHVVAVMLYLLDRFSPFGRFKVNSEEEEEDALTLSSAMWFSWGVLLNSGIGEGAPRSFSARILGMVWAGFAMIIVASYTANLAAFLVLDRPEERITGINDPRLRNPSDKFIYATVKQSSVDIYFRRQVELSTMYRHMEKHNYESAAEAIQAVRDNKLHAFIWDSAVLEFEASQKCDLVTTGELFFRSGFGIGMRKDSPWKQQVSLSILKSHENGFMEDLDKTWVRYQECDSRSNAPATLTFENMAGVFMLVAGGIVAGIFLIFIEIAYKRHKDANGAQ
>9OBZ_E2 len=862 copies=2 longest=1
GWSHPQFEKGGGSGGGSGGSAWSHPQFEKGALVPRGRSQKSPPSIGIAVILVGTSDEVAIKDAHEKDDFHHLSVVPRVELVAMNETDPKSIITRICDLMSDRKIQGVVFADDTDQEAIAQILDFISAQTLTPILGIHGGSSMIMADKDESSMFFQFGPSIEQQASVMLNIMEEYDWYIFSIVTTYFPGYQDFVNKIRSTIENSFVGWELEEVLLLDMSLDDGDSKIQNQLKKLQSPIILLYCTKEEATYIFEVANSVGLTGYGYTWIVPSLVAGDTDTVPSEFPTGLISVSYDEWDYGLPARVRDGIAIITTAASDMLSEHSFIPEPKSSCYNTHEKRIYQSNMLNRYLINVTFEGRNLSFSEDGYQMHPKLVIILLNKERKWERVGKWKDKSLQMKYYVWPRMCPETEEQEDDHLSIVTLEEAPFVIVESVDPLSGTCMRNTVPCQKRIISENKTDEEPGYIKKCCKGFCIDILKKISKSVKFTYDLYLVTNGKHGKKINGTWNGMIGEVVMKRAYMAVGSLTINEERSEVVDFSVPFIETGISVMVSRSNGTVSPSAFLEPFSADVWVMMFVMLLIVSAVAVFVFEYFSPVGYNRCLADGREPGGPSFTIGKAIWLLWGLVFNNSVPVQNPKGTTSKIMVSVWAFFAVIFLASYTANLAAFMIQEEYVDQVSGLSDKKFQRPNDFSPPFRFGTVPNGSTERNIRNNYAEMHAYMGKFNQRGVDDALLSLKTGKLDAFIYDAAVLNYMAGRDEGCKLVTIGSGKVFASTGYGIAIQKDSGWKRQVDLAILQLFGDGEMEELEALWLTGICHNEKNEVMSSQLDIDNMAGVFYMLGAAMALSLITFICEHLFYWQFRHSFMG
>9OC0_E1 len=847 copies=2 longest=0
MSTMHLLTFALLFSCSFARAASDPKIVNIGAVLSTRKHEQMFREAVNQANKRHGSWKIQLQATSVTHKPNAIQMALSVCEDLISSQVYAILVSHPPTPNDHFTPTPVSYTAGFYRIPVLGLTTRMSIYSDKSIHLSFLRTVPPYSHQSSVWFEMMRVYNWNHIILLVSDDHEGRAAQKRLETLLEERESKAEKVLQFDPGTKNVTALLMEARELEARVIILSASEDDAATVYRAAAMLDMTGSGYVWLVGEREISGNALRYAPDGIIGLQLINGKNESAHISDAVGVVAQAVHELLEKENITDPPRGCVGNTNIWKTGPLFKRVLMSSKYADGVTGRVEFNEDGDRKFAQYSIMNLQNRKLVQVGIYNGTHVIPNDRKIIWPGGETEKPRGYQMSTRLKIVTIHQEPFVYVKPTMSDGTCKEEFTVNGDPVKKVICTGPNDTSPGSPRHTVPQCCYGFCIDLLIKLARTMQFTYEVHLVADGKFGTQERVQNSNKKEWNGMMGELLSGQADMIVAPLTINNERAQYIEFSKPFKYQGLTILVKKEIPRSTLDSFMQPFQSTLWLLVGLSVHVVAVMLYLLDRFSPFGRFKVNSEEEEEDALTLSSAMWFSWGVLLNSGIGEGAPRSFSARILGMVWAGFAMIIVASYTANLAAFLVLDRPEERITGINDPRLRNPSDKFIYATVKQSSVDIYFRRQVELSTMYRHMEKHNYESAAEAIQAVRDNKLHAFIWDSAVLEFEASQKCDLVTTGELFFRSGFGIGMRKDSPWKQQVSLSILKSHENGFMEDLDKTWVRYQECDSRSNAPATLTFENMAGVFMLVAGGIVAGIFLIFIEIAYKRHKDANGAQ
>9OC0_E2 len=862 copies=2 longest=1
GWSHPQFEKGGGSGGGSGGSAWSHPQFEKGALVPRGRSQKSPPSIGIAVILVGTSDEVAIKDAHEKDDFHHLSVVPRVELVAMNETDPKSIITRICDLMSDRKIQGVVFADDTDQEAIAQILDFISAQTLTPILGIHGGSSMIMADKDESSMFFQFGPSIEQQASVMLNIMEEYDWYIFSIVTTYFPGYQDFVNKIRSTIENSFVGWELEEVLLLDMSLDDGDSKIQNQLKKLQSPIILLYCTKEEATYIFEVANSVGLTGYGYTWIVPSLVAGDTDTVPSEFPTGLISVSYDEWDYGLPARVRDGIAIITTAASDMLSEHSFIPEPKSSCYNTHEKRIYQSNMLNRYLINVTFEGRNLSFSEDGYQMHPKLVIILLNKERKWERVGKWKDKSLQMKYYVWPRMCPETEEQEDDHLSIVTLEEAPFVIVESVDPLSGTCMRNTVPCQKRIISENKTDEEPGYIKKCCKGFCIDILKKISKSVKFTYDLYLVTNGKHGKKINGTWNGMIGEVVMKRAYMAVGSLTINEERSEVVDFSVPFIETGISVMVSRSNGTVSPSAFLEPFSADVWVMMFVMLLIVSAVAVFVFEYFSPVGYNRCLADGREPGGPSFTIGKAIWLLWGLVFNNSVPVQNPKGTTSKIMVSVWAFFAVIFLASYTANLAAFMIQEEYVDQVSGLSDKKFQRPNDFSPPFRFGTVPNGSTERNIRNNYAEMHAYMGKFNQRGVDDALLSLKTGKLDAFIYDAAVLNYMAGRDEGCKLVTIGSGKVFASTGYGIAIQKDSGWKRQVDLAILQLFGDGEMEELEALWLTGICHNEKNEVMSSQLDIDNMAGVFYMLGAAMALSLITFICEHLFYWQFRHSFMG
>9OC1_E1 len=847 copies=2 longest=0
MSTMHLLTFALLFSCSFARAASDPKIVNIGAVLSTRKHEQMFREAVNQANKRHGSWKIQLQATSVTHKPNAIQMALSVCEDLISSQVYAILVSHPPTPNDHFTPTPVSYTAGFYRIPVLGLTTRMSIYSDKSIHLSFLRTVPPYSHQSSVWFEMMRVYNWNHIILLVSDDHEGRAAQKRLETLLEERESKAEKVLQFDPGTKNVTALLMEARELEARVIILSASEDDAATVYRAAAMLDMTGSGYVWLVGEREISGNALRYAPDGIIGLQLINGKNESAHISDAVGVVAQAVHELLEKENITDPPRGCVGNTNIWKTGPLFKRVLMSSKYADGVTGRVEFNEDGDRKFAQYSIMNLQNRKLVQVGIYNGTHVIPNDRKIIWPGGETEKPRGYQMSTRLKIVTIHQEPFVYVKPTMSDGTCKEEFTVNGDPVKKVICTGPNDTSPGSPRHTVPQCCYGFCIDLLIKLARTMQFTYEVHLVADGKFGTQERVQNSNKKEWNGMMGELLSGQADMIVAPLTINNERAQYIEFSKPFKYQGLTILVKKEIPRSTLDSFMQPFQSTLWLLVGLSVHVVAVMLYLLDRFSPFGRFKVNSEEEEEDALTLSSAMWFSWGVLLNSGIGEGAPRSFSARILGMVWAGFAMIIVASYTANLAAFLVLDRPEERITGINDPRLRNPSDKFIYATVKQSSVDIYFRRQVELSTMYRHMEKHNYESAAEAIQAVRDNKLHAFIWDSAVLEFEASQKCDLVTTGELFFRSGFGIGMRKDSPWKQQVSLSILKSHENGFMEDLDKTWVRYQECDSRSNAPATLTFENMAGVFMLVAGGIVAGIFLIFIEIAYKRHKDANGAQ
>9OC1_E2 len=862 copies=2 longest=1
GWSHPQFEKGGGSGGGSGGSAWSHPQFEKGALVPRGRSQKSPPSIGIAVILVGTSDEVAIKDAHEKDDFHHLSVVPRVELVAMNETDPKSIITRICDLMSDRKIQGVVFADDTDQEAIAQILDFISAQTLTPILGIHGGSSMIMADKDESSMFFQFGPSIEQQASVMLNIMEEYDWYIFSIVTTYFPGYQDFVNKIRSTIENSFVGWELEEVLLLDMSLDDGDSKIQNQLKKLQSPIILLYCTKEEATYIFEVANSVGLTGYGYTWIVPSLVAGDTDTVPSEFPTGLISVSYDEWDYGLPARVRDGIAIITTAASDMLSEHSFIPEPKSSCYNTHEKRIYQSNMLNRYLINVTFEGRNLSFSEDGYQMHPKLVIILLNKERKWERVGKWKDKSLQMKYYVWPRMCPETEEQEDDHLSIVTLEEAPFVIVESVDPLSGTCMRNTVPCQKRIISENKTDEEPGYIKKCCKGFCIDILKKISKSVKFTYDLYLVTNGKHGKKINGTWNGMIGEVVMKRAYMAVGSLTINEERSEVVDFSVPFIETGISVMVSRSNGTVSPSAFLEPFSADVWVMMFVMLLIVSAVAVFVFEYFSPVGYNRCLADGREPGGPSFTIGKAIWLLWGLVFNNSVPVQNPKGTTSKIMVSVWAFFAVIFLASYTANLAAFMIQEEYVDQVSGLSDKKFQRPNDFSPPFRFGTVPNGSTERNIRNNYAEMHAYMGKFNQRGVDDALLSLKTGKLDAFIYDAAVLNYMAGRDEGCKLVTIGSGKVFASTGYGIAIQKDSGWKRQVDLAILQLFGDGEMEELEALWLTGICHNEKNEVMSSQLDIDNMAGVFYMLGAAMALSLITFICEHLFYWQFRHSFMG
>9OC2_E1 len=847 copies=2 longest=0
MSTMHLLTFALLFSCSFARAASDPKIVNIGAVLSTRKHEQMFREAVNQANKRHGSWKIQLQATSVTHKPNAIQMALSVCEDLISSQVYAILVSHPPTPNDHFTPTPVSYTAGFYRIPVLGLTTRMSIYSDKSIHLSFLRTVPPYSHQSSVWFEMMRVYNWNHIILLVSDDHEGRAAQKRLETLLEERESKAEKVLQFDPGTKNVTALLMEARELEARVIILSASEDDAATVYRAAAMLDMTGSGYVWLVGEREISGNALRYAPDGIIGLQLINGKNESAHISDAVGVVAQAVHELLEKENITDPPRGCVGNTNIWKTGPLFKRVLMSSKYADGVTGRVEFNEDGDRKFAQYSIMNLQNRKLVQVGIYNGTHVIPNDRKIIWPGGETEKPRGYQMSTRLKIVTIHQEPFVYVKPTMSDGTCKEEFTVNGDPVKKVICTGPNDTSPGSPRHTVPQCCYGFCIDLLIKLARTMQFTYEVHLVADGKFGTQERVQNSNKKEWNGMMGELLSGQADMIVAPLTINNERAQYIEFSKPFKYQGLTILVKKEIPRSTLDSFMQPFQSTLWLLVGLSVHVVAVMLYLLDRFSPFGRFKVNSEEEEEDALTLSSAMWFSWGVLLNSGIGEGAPRSFSARILGMVWAGFAMIIVASYTANLAAFLVLDRPEERITGINDPRLRNPSDKFIYATVKQSSVDIYFRRQVELSTMYRHMEKHNYESAAEAIQAVRDNKLHAFIWDSAVLEFEASQKCDLVTTGELFFRSGFGIGMRKDSPWKQQVSLSILKSHENGFMEDLDKTWVRYQECDSRSNAPATLTFENMAGVFMLVAGGIVAGIFLIFIEIAYKRHKDANGAQ
>9OC2_E2 len=862 copies=2 longest=1
GWSHPQFEKGGGSGGGSGGSAWSHPQFEKGALVPRGRSQKSPPSIGIAVILVGTSDEVAIKDAHEKDDFHHLSVVPRVELVAMNETDPKSIITRICDLMSDRKIQGVVFADDTDQEAIAQILDFISAQTLTPILGIHGGSSMIMADKDESSMFFQFGPSIEQQASVMLNIMEEYDWYIFSIVTTYFPGYQDFVNKIRSTIENSFVGWELEEVLLLDMSLDDGDSKIQNQLKKLQSPIILLYCTKEEATYIFEVANSVGLTGYGYTWIVPSLVAGDTDTVPSEFPTGLISVSYDEWDYGLPARVRDGIAIITTAASDMLSEHSFIPEPKSSCYNTHEKRIYQSNMLNRYLINVTFEGRNLSFSEDGYQMHPKLVIILLNKERKWERVGKWKDKSLQMKYYVWPRMCPETEEQEDDHLSIVTLEEAPFVIVESVDPLSGTCMRNTVPCQKRIISENKTDEEPGYIKKCCKGFCIDILKKISKSVKFTYDLYLVTNGKHGKKINGTWNGMIGEVVMKRAYMAVGSLTINEERSEVVDFSVPFIETGISVMVSRSNGTVSPSAFLEPFSADVWVMMFVMLLIVSAVAVFVFEYFSPVGYNRCLADGREPGGPSFTIGKAIWLLWGLVFNNSVPVQNPKGTTSKIMVSVWAFFAVIFLASYTANLAAFMIQEEYVDQVSGLSDKKFQRPNDFSPPFRFGTVPNGSTERNIRNNYAEMHAYMGKFNQRGVDDALLSLKTGKLDAFIYDAAVLNYMAGRDEGCKLVTIGSGKVFASTGYGIAIQKDSGWKRQVDLAILQLFGDGEMEELEALWLTGICHNEKNEVMSSQLDIDNMAGVFYMLGAAMALSLITFICEHLFYWQFRHSFMG
>9ODH_E1 len=344 copies=1 longest=0
PGSSGSELDQLRQEAEQLKNQIRDARKACADATLSQITNNIDPVGRIQMRTRRTLRGHLAKIYAMHWGTDSRLLVSASQDGKLIIWDSYTTNKVHAIPLRSSWVMTCAYAPSGNYVACGGLDNICSIYNLKTREGNVRVSRELAGHTGYLSCCRFLDDNQIVTSSGDTTCALWDIETGQQTTTFTGHTGDVMSLSLAPDTRLFVSGACDASAKLWDVREGMCRQTFTGHESDINAICFFPNGNAFATGSDDATCRLFDLRADQELMTYSHDNIICGITSVSFSKSGRLLLAGYDDFNCNVWDALKADRAGVLAGHDNRVSCLGVTDDGMAVATGSWDSFLKIWN
>9ODH_E2 len=71 copies=1 longest=0
MASNNTASIAQARKLVEQLKMEANIDRIKVSKAAADLMAYCEAHAKEDPLLTPVPASENPFREKKFFCAIL
>9ODH_E3 len=354 copies=1 longest=1
MGCTLSAEDKAAVERSKMIDRNLREDGEKAAREVKLLLLGAGESGKSTIVKQMKIIHEAGYSEEECKQYKAVVYSNTIQSIIAIIRAMGRLKIDFGDSARADDARQLFVLAGAAEEGFMTAELAGVIKRLWKDSGVQACFNRSREYQLNDSAAYYLNDLDRIAQPNYIPTQQDVLRTRVKTTGIVETHFTFKDLHFKMFDVGGQRSERKKWIHCFEGVTAIIFCVALSDYDLVLAEDEEMNRMHESMKLFDSICNNKWFTDTSIILFLNKKDLFEEKIKKSPLTICYPEYAGSNTYEEAAAYIQCQFEDLNKRKDTKEIYTHFTCATDTKNVQFVFDAVTDVIIKNNLKDCGLF
>9ODI_E1 len=344 copies=1 longest=0
PGSSGSELDQLRQEAEQLKNQIRDARKACADATLSQITNNIDPVGRIQMRTRRTLRGHLAKIYAMHWGTDSRLLVSASQDGKLIIWDSYTTNKVHAIPLRSSWVMTCAYAPSGNYVACGGLDNICSIYNLKTREGNVRVSRELAGHTGYLSCCRFLDDNQIVTSSGDTTCALWDIETGQQTTTFTGHTGDVMSLSLAPDTRLFVSGACDASAKLWDVREGMCRQTFTGHESDINAICFFPNGNAFATGSDDATCRLFDLRADQELMTYSHDNIICGITSVSFSKSGRLLLAGYDDFNCNVWDALKADRAGVLAGHDNRVSCLGVTDDGMAVATGSWDSFLKIWN
>9ODI_E2 len=71 copies=1 longest=0
MASNNTASIAQARKLVEQLKMEANIDRIKVSKAAADLMAYCEAHAKEDPLLTPVPASENPFREKKFFCAIL
>9ODI_E3 len=354 copies=1 longest=1
MGCTLSAEDKAAVERSKMIDRNLREDGEKAAREVKLLLLGAGESGKSTIVKQMKIIHEAGYSEEECKQYKAVVYSNTIQSIIAIIRAMGRLKIDFGDSARADDARQLFVLAGAAEEGFMTAELAGVIKRLWKDSGVQACFNRSREYQLNDSAAYYLNDLDRIAQPNYIPTQQDVLRTRVKTTGIVETHFTFKDLHFKMFDVGGQRSERKKWIHCFEGVTAIIFCVALSDYDLVLAEDEEMNRMHESMKLFDSICNNKWFTDTSIILFLNKKDLFEEKIKKSPLTICYPEYAGSNTYEEAAAYIQCQFEDLNKRKDTKEIYTHFTCATDTKNVQFVFDAVTDVIIKNNLKDCGLF
>9OK7_E1 len=892 copies=1 longest=1
MDQPEAPCSSTGPRLAVARELLLAALEELSQEQLKRFRHKLRDVGPDGRSIPWGRLERADAVDLAEQLAQFYGPEPALEVARKTLKRADARDVAAQLQERRLQRLGLGSGTLLSVSEYKKKYREHVLQLHARVKERNARSVKITKRFTKLLIAPESAAPEEAMGPAEEPEPGRARRSDTHTFNRLFRRDEEGRRPLTVVLQGPAGIGKTMAAKKILYDWAAGKLYQGQVDFAFFMPCGELLERPGTRSLADLILDQCPDRGAPVPQMLAQPQRLLFILDGADELPALGGPEAAPCTDPFEAASGARVLGGLLSKALLPTALLLVTTRAAAPGRLQGRLCSPQCAEVRGFSDKDKKKYFYKYFRDERRAERAYRFVKENETLFALCFVPFVCWIVCTVLRQQLELGRDLSRTSKTTTSVYLLFITSVLSSAPVADGPRLQGDLRNLCRLAREGVLGRRAQFAEKELEQLELRGSKVQTLFLSKKELPGVLETEVTYQFIDQSFQEFLAALSYLLEDGGVPRTAAGGVGTLLRGDAQPHSHLVLTTRFLFGLLSAERMRDIERHFGCMVSERVKQEALRWVQGQGQGCPGVAPEVTEGAKGLEDTEEPEEEEEGEEPNYPLELLYCLYETQEDAFVRQALCRFPELALQRVRFCRMDVAVLSYCVRCCPAGQALRLISCRLVAAQEKKKKSLGKRLQASLGGGSSSQGTTKQLPASLLHPLFQAMTDPLCHLSSLTLSHCKLPDAVCRDLSEALRAAPALTELGLLHNRLSEAGLRMLSEGLAWPQCRVQTVRVQLPDPQRGLQYLVGMLRQSPALTTLDLSGCQLPAPMVTYLCAVLQHQGCGLQTLSLASVELSEQSLQELQAVKRAKPDLVITHPALDGHPQPPKELISTF
>9OK8_E1 len=892 copies=1 longest=1
MDQPEAPCSSTGPRLAVARELLLAALEELSQEQLKRFRHKLRDVGPDGRSIPWGRLERADAVDLAEQLAQFYGPEPALEVARKTLKRADARDVAAQLQERRLQRLGLGSGTLLSVSEYKKKYREHVLQLHARVKERNARSVKITKRFTKLLIAPESAAPEEAMGPAEEPEPGRARRSDTHTFNRLFRRDEEGRRPLTVVLQGPAGIGKTMAAKKILYDWAAGKLYQGQVDFAFFMPCGELLERPGTRSLADLILDQCPDRGAPVPQMLAQPQRLLFILDGADELPALGGPEAAPCTDPFEAASGARVLGGLLSKALLPTALLLVTTRAAAPGRLQGRLCSPQCAEVRGFSDKDKKKYFYKYFRDERRAERAYRFVKENETLFALCFVPFVCWIVCTVLRQQLELGRDLSRTSKTTTSVYLLFITSVLSSAPVADGPRLQGDLRNLCRLAREGVLGRRAQFAEKELEQLELRGSKVQTLFLSKKELPGVLETEVTYQFIDQSFQEFLAALSYLLEDGGVPRTAAGGVGTLLRGDAQPHSHLVLTTRFLFGLLSAERMRDIERHFGCMVSERVKQEALRWVQGQGQGCPGVAPEVTEGAKGLEDTEEPEEEEEGEEPNYPLELLYCLYETQEDAFVRQALCRFPELALQRVRFCRMDVAVLSYCVRCCPAGQALRLISCRLVAAQEKKKKSLGKRLQASLGGGSSSQGTTKQLPASLLHPLFQAMTDPLCHLSSLTLSHCKLPDAVCRDLSEALRAAPALTELGLLHNRLSEAGLRMLSEGLAWPQCRVQTVRVQLPDPQRGLQYLVGMLRQSPALTTLDLSGCQLPAPMVTYLCAVLQHQGCGLQTLSLASVELSEQSLQELQAVKRAKPDLVITHPALDGHPQPPKELISTF
>9OPQ_E1 len=376 copies=1 longest=0
MGILPSPGMPALLSLVSLLSVLLMGCVAETGTRIPDLPDCDIDKWLNNFNVPSPLNWERKIFSNCNFNLSTLLRLVHTDSFSCNNFDESKIYGSCFKSIVLDKFAIPNSRRSDLQLGSSGFLQSSNYKIDTTSSSCQLYYSLPAINVTINNYNPSSWNRRYGFNNFNLSSHSVVYSRYCFSVNNTFCPCAKPSFASSCKSHKPPSASCPIGTNYRSCESTTVIAGRSALEDLLGSGSGSPDHTDWCRCSCLPDPITAYDPRSCSQKKSLVGVGEHCAGFGVDEEKCGVLDGSYNVSCLCSTDAFLGWSYDTCVSNNRCNIFSNFILNGINSGTTCSNDLLQPNTGSSGGSGLNDIFEAQKIEWHEGGSHHHHHHHH
>9OPQ_E2 len=506 copies=1 longest=1
MTRLTVLALLAGLLASSRASMSDSEVNQEAKPEVKPEVKPETHINLKVSDGSSEIFFKIKKTTPLRRLMEAFAKRQGKEMDSLRFLYDGIRIQADQTPEDLDMEDNDIIEAHRENDDDDKTTGSKCSNSGIECDSSGTCINPSNWCDGVSHCPGGEDENRCVRLYGPNFILQVYSSQRKSWHPVCQDDWNENYGRAACRDMGYKNNFYSSQGIVDDSGSTSFMKLNTSAGNVDIYKKLYHSDACSSKAVVSLRCIACGVNLNSSDDDDKIVGGESALPGAWPWQVSLHVQNVHVCGGSIITPEWIVTAAHCVEKPLNNPWHWTAFAGILRQSFMFYGAGYQVEKVISHPNYDSKTKNNDIALMKLQKPLTFNDLVKPVCLPNPGMMLQPEQLCWISGWGATEEKGKTSEVLNAAKVLLIETQRCNSRYVYDNLITPAMICAGFLQGNVDSCQGDAGGPLVCSKNNIWWLIGDTSWGSGCAKAYRPGVYGNVMVFTDWIYRQMRADG
>9OTA_E1 len=270 copies=2 longest=1
GALLAGALAAYAAYLVLGALLVARLEGPHEARLRAELETLRAQLLQRSPCVAAPALDAFVERVLAAGRLGRVVLANASGSANASDPAWDFASALFFASTLITTVGYGYTTPLTDAGKAFSIAFALLGVPTTMLLLTASAQRLSLLLTHVPLSWLSMRWGWDPRRAACWHLVALLGVVVTVCFLVPAVIFAHLEEAWSFLDAFYFCFISLSTIGLGDYVPGEAPGQPYRALYKVLVTVYLFLGLVAMVLVLQTFRHVSDLHGLTELILLPP
>9OVK_E1 len=1344 copies=1 longest=1
VIGDLKCTSDNINDKDTGPPPISTDTVDVTNGLGTYYVLDRVYLNTTLFLNGYYPTSGSTYRNMALKGSVLLSRLWFKPPFLSDFINGIFAKVKNTKVIKDRVMYSEFPAITIGSTFVNTSYSVVVQPRTINSTQDGDNKLQGLLEVSVCQYNMCEYPQTICHPNLGNHRKELWHLDTGVVSCLYKRNFTYDVNADYLYFHFYQEGGTFYAYFTDTGVVTKFLFNVYLGMALSHYYVMPLTCNSKLTLEYWVTPLTSRQYLLAFNQDGIIFNAVDCMSDFMSEIKCKTQSIAPPTGVYELNGYTVQPIADVYRRKPNLPNCNIEAWLNDKSVPSPLNWERKTFSNCNFNMSSLMSFIQADSFTCNNIDAAKIYGMCFSSITIDKFAIPNGRKVDLQLGNLGYLQSFNYRIDTTATSCQLYYNLPAANVSVSRFNPSTWNKRFGFIEDSVFKPRPAGVLTNHDVVYAQHCFKAPKNFCPCKLNGSCVGSGPGKNNGIGTCPAGTNYLTCDNLCTPDPITFTGTYKCPQTKSLVGIGEHCSGLAVKSDYCGGNSCTCRPQAFLGWSADSCLQGDKCNIFANFILHDVNSGLTCSTDLQKANTDIILGVCVNYDLYGILGQGIFVEVNATYYNSWQNLLYDSNGNLYGFRDYITNRTFMIRSCYSGRVSAAFHANSSEPALLFRNIKCNYVFNNSLTRQLQPINYFDSYLGCVVNAYNSTAISVQTCDLTVGSGYCVDYSKNRRSRGAITTGYRFTNFEPFTVNSVNDSLEPVGGLYEIQIPSEFTIGNMVEFIQTSSPKVTIDCAAFVCGDYAACKSQLVEYGSFCDNINAILTEVNELLDTTQLQVANSLMNGVTLSTKLKDGVNFNVDDINFSPVLGCLGSECSKASSRSAIEDLLFDKVKLSDVGFVEAYNNCTGGAEIRDLICVQSYKGIKVLPPLLSENQISGYTLAATSASLFPPWTAAAGVPFYLNVQYRINGLGVTMDVLSQNQKLIANAFNNALYAIQEGFDAPNSALVKIQAVVNANAEALNNLLQQLSNRFGAISASLQEILSRLDPPEAEAQIDRLINGRLTALNAYVSQQLSDSTLVKFSAAQAMEKVNECVKSQSSRINFCGNGNHIISLVQNAPYGLYFIHFSYVPTKYVTARVSPGLCIAGDRGIAPKSGYFVNVNNTWMYTGSGYYYPEPITENNVVVMSTCAVNYTKAPYVMLNTSIPNLPDFKEELDQWFKNQTSVAPDLSLDYINVTFLDLQVEMNRLQEAIKVLNQSYINLKDIGTYEYYVKSGGYIPEAPRDGQAYVRKDGEWVLLSTFLNSGRAHHHHHHGAGGLNDIFEAQKIEWHEDTAAA
>9OVM_E1 len=1326 copies=1 longest=1
VIGDLNCTLDPRLKGSFNNRDTGPPSISIDTVDVTNGLGTYYVLDRVYLNTTLFLNGYYPTSGSTYRNMALKGTDLLSTLWFKPPFLSDFINGIFAKVKNTKVFKDGVMYSEFPAITIGSTFVNTSYSVVVQPRTINSTQDGVNKLQGLLEVSVCQYNMCEYPHTICHPNLGNHFKELWHYDTGVVSCLYKRNFTYDVNATYLYFHFYQEGGTFYAYFTDTGFVTKFLFNVYLGMALSHYYVMPLTCIRRPKDGFSLEYWVTPLTPRQYLLAFNQDGIIFNAVDCMSDFMSEIKCKTQSIAPPTGVYELNGYTVQPVADVYRRKPDLPNCNIEAWLNDKSVPSPLNWERKTFSNCNFNMSSLMSFIQADSFTCNNIDAAKIYGMCFSSITIDKFAIPNRRKVDLQLGNLGYLQSSNYRIDTTATSCQLYYNLPAANVSVSRFNPSTWNKRFGFIEDSVFVPQPTGVFTNHSVVYAQHCFKAPKNFCPCSSCPGKNNGIGTCPAGTNSLTCDNLCTLDPITLKAPDTYKCPQSKSLVGIGEHCSGLAVKSDYCGNNSCTCQPQAFLGWSADSCLQGDKCNIFANFILHDVNNGLTCSTDLQKANTEIELGVCVNYDLYGISGQGIFVEVNATYYNSWQNLLYDSNGNLYGFRDYITNRTFMIHSCYSGRVSAAYHANSSEPALLFRNIKCNYVFNNSLTRQLQPINYSFDSYLGCVVNAYNSTAISVQTCDLTVGSGYCVDYSKNRRSRRAITTGYRFTNFEPFTVNSVNDSLEPVGGLYEIQIPSEFTIGNMEEFIQTSSPKVTIDCAAFVCGDYAACKLQLVEYGSFCDNINAILTEVNELLDTTQLQVANSLMNGVTLSTKLKDGVNFNVDDINFSPVLGCLGSECSKASSRSAIEDLLFDKVKLSDVGFVEAYNNCTGGAEIRDLICVQSYKGIKVLPPLLSENQISGYTLAATSASLFPPWTAAAGVPFYLNVQYRINGLGVTMDVLSQNQKLIANAFNNALHAIQQGFDAPNSALVKIQAVVNANAEALNNLLQQLSNRFGAISASLQEILSRLDPPEAEAQIDRLINGRLTALNAYVSQQLSDSTLVKFSAAQAMEKVNECVKSQSSRINFCGNGNHIISLVQNAPYGLYFIHFNYVPTKYVTAKVSPGLCIAGNRGIAPKSGYFVNVNNTWMYTGSGYYYPEPITENNVVVMSTCAVNYTKAPYVMLNTSIPNLPDFKEELDQWFKNQTSVALDLSLDYINVTFLDLQVEMNRLQEAIKVLNHSYINLKDIGTYEYYVKSGGYIPEAPRDGQAYVRKDGEWVLLSTFLNSGRAHHHHHH
>9OVO_E1 len=1326 copies=1 longest=1
VIGDLNCTLDPRLKGSFNNRDTGPPSISIDTVDVTNGLGTYYVLDRVYLNTTLFLNGYYPTSGSTYRNMALKGTDLLSTLWFKPPFLSDFINGIFAKVKNTKVFKDGVMYSEFPAITIGSTFVNTSYSVVVQPRTINSTQDGVNKLQGLLEVSVCQYNMCEYPHTICHPNLGNHFKELWHYDTGVVSCLYKRNFTYDVNATYLYFHFYQEGGTFYAYFTDTGFVTKFLFNVYLGMALSHYYVMPLTCIRRPKDGFSLEYWVTPLTPRQYLLAFNQDGIIFNAVDCMSDFMSEIKCKTQSIAPPTGVYELNGYTVQPVADVYRRKPDLPNCNIEAWLNDKSVPSPLNWERKTFSNCNFNMSSLMSFIQADSFTCNNIDAAKIYGMCFSSITIDKFAIPNRRKVDLQLGNLGYLQSSNYRIDTTATSCQLYYNLPAANVSVSRFNPSTWNKRFGFIEDSVFVPQPTGVFTNHSVVYAQHCFKAPKNFCPCSSCPGKNNGIGTCPAGTNSLTCDNLCTLDPITLKAPDTYKCPQSKSLVGIGEHCSGLAVKSDYCGNNSCTCQPQAFLGWSADSCLQGDKCNIFANFILHDVNNGLTCSTDLQKANTEIELGVCVNYDLYGISGQGIFVEVNATYYNSWQNLLYDSNGNLYGFRDYITNRTFMIHSCYSGRVSAAYHANSSEPALLFRNIKCNYVFNNSLTRQLQPINYSFDSYLGCVVNAYNSTAISVQTCDLTVGSGYCVDYSKNRRSRRAITTGYRFTNFEPFTVNSVNDSLEPVGGLYEIQIPSEFTIGNMEEFIQTSSPKVTIDCAAFVCGDYAACKLQLVEYGSFCDNINAILTEVNELLDTTQLQVANSLMNGVTLSTKLKDGVNFNVDDINFSPVLGCLGSECSKASSRSAIEDLLFDKVKLSDVGFVEAYNNCTGGAEIRDLICVQSYKGIKVLPPLLSENQISGYTLAATSASLFPPWTAAAGVPFYLNVQYRINGLGVTMDVLSQNQKLIANAFNNALHAIQQGFDAPNSALVKIQAVVNANAEALNNLLQQLSNRFGAISASLQEILSRLDPPEAEAQIDRLINGRLTALNAYVSQQLSDSTLVKFSAAQAMEKVNECVKSQSSRINFCGNGNHIISLVQNAPYGLYFIHFNYVPTKYVTAKVSPGLCIAGNRGIAPKSGYFVNVNNTWMYTGSGYYYPEPITENNVVVMSTCAVNYTKAPYVMLNTSIPNLPDFKEELDQWFKNQTSVALDLSLDYINVTFLDLQVEMNRLQEAIKVLNHSYINLKDIGTYEYYVKSGGYIPEAPRDGQAYVRKDGEWVLLSTFLNSGRAHHHHHH
>9P1V_E1 len=838 copies=1 longest=1
MKINKKYLAGSVAVLALSVCSYELGRHQAGQVKKESNRVSYIDGDQAGQKAENLTPDEVSKREGINAEQIVIKITDQGYVTSHGDHYHYYNGKVPYDAIISEELLMKDPNYQLKDSDIVNEIKGGYVIKVDGKYYVYLKDAAHADNIRTKEEIKRQKQERSHNHNSRADNAVAAARAQGRYTTDDGYIFNASDIIEDTGDAYIVPHGDHYHYIPKNELSASELAAAEAYWNGKQGSRPSSSSSYNANPAQPRLSENHNLTVTPTYHQNQGENISSLLRELYAKPLSERHVESDGLIFDPAQITSRTARGVAVPHGNHYHFIPYEQMSELEKRIARIIPLRYRSNHWVPDSRPEQPSPQSTPEPSPSLQPAPNPQPAPSNPIDEKLVKEAVRKVGDGYVFEKNGVSRYIPAKDLSAETAAGIDSKLAKQESLSHKLGAKKTDLPSSDREFYNKTYDLLARIHQDLLDNKGRQVDFEALDNLLERLKDVSSDKVKLVDDILAFLAPIRHPERLGKPNAQITYTDDEIQVAKLAGKYTTEDGYIFDPRDITSDEGDAYVTPHMTHSHWIKKDSLSEAERAAAQAYAKEKGLTPPSTDHQDSGNTEAKGAEAIYNRVKAAKKVPLDRMPYNLQYTVEVKNGSLIIPHYDHYHNIKFEWFDEGLYEAPKGYTLEDLLATVKYYVEHPNERPHSDNGFGNASDHVRKNKADQDSKPDEDKGHDEVSEPTHPESDEKENHAGLNPSADNLYKPSTDTEETEEEAEDTTDEAEIPQVEHSVINAKIADAEALLEKVTDPSIRQNAMETLTGLKSSLLLGTKDNNTISAEVDSLLALLKKSQPAPIQ
>9P1V_E2 len=235 copies=1 longest=0
MVLQTQVFISLLLWISGAYGEIVLTQSPGTLSLSPGERATLSCRASQSLTDNYLAWYQQKPGQAPRLLIYAASTRATGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQQYQNSPFTFGGGKTVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9P1V_E3 len=474 copies=1 longest=0
MDWTWRILFLVAAATGTHAQVQLVQSGAEVKKPGSSVKVSCKASGAAFESFAFAWVRQAPGQGLEWMGRIIPILETRDYAQKFQGRVTITADESTSTAYMELSSLRSEDTAVYFCARDGHIMRTTLSDAALDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCDKTHTCPPCPAPELLGGPSVFLFPPKPKDTLMISRTPEVTCVVVDVSHEDPEVKFNWYVDGVEVHNAKTKPREEQYNSTYRVVSVLTVLHQDWLNGKEYKCKVSNKALPAPIEKTISKAKGQPREPQVYTLPPSREEMTKNQVSLTCLVKGFYPSDIAVEWESNGQPENNYKTTPPVLDSDGSFFLYSKLTVDKSRWQQGNVFSCSVMHEALHNHYTQKSLSLSPGK
>9P4L_E1 len=743 copies=1 longest=1
MAADGYLPDWLEDNLSEGIREWWALKPGAPQPKANQQHQDNARGLVLPGYKYLGPGNGLDKGEPVNAADAAALEHDKAYDQQLKAGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRLLEPLGLVEEAAKTAPGKKRPVEQSPQEPDSSAGIGKSGAQPAKKRLNFGQTGDTESVPDPQPIGEPPAAPSGVGSLTMASGGGAPVADNNEGADGVGSSSGNWHCDSQWLGDRVITTSTRTWALPTYNNHLYKQISNSTSGGSSNDNAYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTDNNGVKTIANNLTSTVQVFTDSDYQLPYVLGSAHEGCLPPFPADVFMIPQYGYLTLNDGSQAVGRSSFYCLEYFPSQMLRTGNNFQFSYEFENVPFHSSYAHSQSLDRLMNPLIDQYLYYLSKTIDGAATKNQTLKFSVAGPSNMAVQGRNYIPGPSYRQQRVSTTVTQNNNSEFAWPGASSWALNGRNSLMNPGPAMASHKEGEDRFFPLSGSLIFGKQGTGRDNVDADKVMITNEEEIKTTNPVATESYGQVATNHQSDGTLAVPFKAQAQTGWVQNQGILPGMVWQDRDVYLQGPIWAKIPHTDGNFHPSPLMGGFGMKHPPPQILIKNTPVPADPPTAFNKDKLNSFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYYKSNNVEFAVNTEGVYSEPRPIGTRYLTRNL
>9P4M_E1 len=736 copies=1 longest=1
MAADGYLPDWLEDNLSEGIREWWALKPGAPQPKANQQHQDNARGLVLPGYKYLGPGNGLDKGEPVNAADAAALEHDKAYDQQLKAGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRLLEPLGLVEEAAKTAPGKKRPVEQSPQEPDSSAGIGKSGAQPAKKRLNFGQTGDTESVPDPQPIGEPPAAPSGVGSLTMASGGGAPVADNNEGADGVGSSSGNWHCDSQWLGDRVITTSTRTWALPTYNNHLYKQISNSTSGGSSNDNAYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTDNNGVKTIANNLTSTVQVFTDSDYQLPYVLGSAHEGCLPPFPADVFMIPQYGYLTLNDGSQAVGRSSFYCLEYFPSQMLRTGNNFQFSYEFENVPFHSSYAHSQSLDRLMNPLIDQYLYYLSKTIDGAATKNQTLKFSVAGPSNMAVQGRNYIPGPSYRQQRVSTTVTQNNNSEFAWPGASSWALNGRNSLMNPGPAMASHKEGEDRFFPLSGSLIFGKQGTGRDNVDADKVMITNEEEIKTTNPVATESYGQVATNHQSAQAQAQTGWVQNQGILPGMVWQDRDVYLQGPIWAKIPHTDGNFHPSPLMGGFGMKHPPPQILIKNTPVPADPPTAFNKDKLNSFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYYKSNNVEFAVNTEGVYSEPRPIGTRYLTRNL
>9P4N_E1 len=743 copies=1 longest=1
MAADGYLPDWLEDNLSEGIREWWALKPGAPQPKANQQHQDNARGLVLPGYKYLGPGNGLDKGEPVNAADAAALEHDKAYDQQLKAGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRLLEPLGLVEEAAKTAPGKKRPVEQSPQEPDSSAGIGKSGAQPAKKRLNFGQTGDTESVPDPQPIGEPPAAPSGVGSLTMASGGGAPVADNNEGADGVGSSSGNWHCDSQWLGDRVITTSTRTWALPTYNNHLYKQISNSTSGGSSNDNAYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTDNNGVKTIANNLTSTVQVFTDSDYQLPYVLGSAHEGCLPPFPADVFMIPQYGYLTLNDGSQAVGRSSFYCLEYFPSQMLRTGNNFQFSYEFENVPFHSSYAHSQSLDRLMNPLIDQYLYYLSKTIDGAATKNQTLKFSVAGPSNMAVQGRNYIPGPSYRQQRVSTTVTQNNNSEFAWPGASSWALNGRNSLMNPGPAMASHKEGEDRFFPLSGSLIFGKQGTGRDNVDADKVMITNEEEIKTTNPVATESYGQVATNHQSAQGNNTRSVAQAQTGWVQNQGILPGMVWQDRDVYLQGPIWAKIPHTDGNFHPSPLMGGFGMKHPPPQILIKNTPVPADPPTAFNKDKLNSFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYYKSNNVEFAVNTEGVYSEPRPIGTRYLTRNL
>9P4N_E2 len=124 copies=1 longest=0
PRKNRPPVAVVSPQFQEISLPTTSTIIDGSQSTDDDKIVQYHWEELKGPLREEKISEDTAILKLSKLVPGNYTFSLTVVDSDGATNSTTASLTVNKAVDYPPGGSEQKLISEEDLGGSHHHHHH
>9P4O_E1 len=743 copies=1 longest=1
MAADGYLPDWLEDNLSEGIREWWALKPGAPQPKANQQHQDNARGLVLPGYKYLGPGNGLDKGEPVNAADAAALEHDKAYDQQLKAGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRLLEPLGLVEEAAKTAPGKKRPVEQSPQEPDSSAGIGKSGAQPAKKRLNFGQTGDTESVPDPQPIGEPPAAPSGVGSLTMASGGGAPVADNNEGADGVGSSSGNWHCDSQWLGDRVITTSTRTWALPTYNNHLYKQISNSTSGGSSNDNAYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTDNNGVKTIANNLTSTVQVFTDSDYQLPYVLGSAHEGCLPPFPADVFMIPQYGYLTLNDGSQAVGRSSFYCLEYFPSQMLRTGNNFQFSYEFENVPFHSSYAHSQSLDRLMNPLIDQYLYYLSKTINGAGTNQQTLKFSVAGPSNMAVQGRNYIPGPSYRQQRVSTTVTQNNNSEFAWPGASSWALNGRNSLMNPGPAMASHKEGEDRFFPLSGSLIFGKQGTGRDNVDADKVMITNEEEIKTTNPVATESYGQVATNHQSDGTLAVPFKAQAQTGWVQNQGILPGMVWQDRDVYLQGPIWAKIPHTDGNFHPSPLMGGFGMKHPPPQILIKNTPVPADPPTAFNKDKLNSFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYYKSNNVEFAVNTEGVYSEPRPIGTRYLTRNL
>9P4P_E1 len=743 copies=1 longest=1
MAADGYLPDWLEDNLSEGIREWWALKPGAPQPKANQQHQDNARGLVLPGYKYLGPGNGLDKGEPVNAADAAALEHDKAYDQQLKAGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRLLEPLGLVEEAAKTAPGKKRPVEQSPQEPDSSAGIGKSGAQPAKKRLNFGQTGDTESVPDPQPIGEPPAAPSGVGSLTMASGGGAPVADNNEGADGVGSSSGNWHCDSQWLGDRVITTSTRTWALPTYNNHLYKQISNSTSGGSSNDNAYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTDNNGVKTIANNLTSTVQVFTDSDYQLPYVLGSAHEGCLPPFPADVFMIPQYGYLTLNDGSQAVGRSSFYCLEYFPSQMLRTGNNFQFSYEFENVPFHSSYAHSQSLDRLMNPLIDQYLYYLSKTINGSGQNQQTLKFSVAGPSNMAVQGRNYIPGPSYRQQRVSTTVTQNNNSEFAWPGASSWALNGRNSLMNPGPAMASHKEGEDRFFPLSGSLIFGKQGTGRDNVDADKVMITNEEEIKTTNPVATESYGQVATNHQSAQGNNTRSVAQAQTGWVQNQGILPGMVWQDRDVYLQGPIWAKIPHTDGNFHPSPLMGGFGMKHPPPQILIKNTPVPADPPTAFNKDKLNSFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYYKSNNVEFAVNTEGVYSEPRPIGTRYLTRNL
>9P4Q_E1 len=743 copies=1 longest=1
MAADGYLPDWLEDNLSEGIREWWALKPGAPQPKANQQHQDNARGLVLPGYKYLGPGNGLDKGEPVNAADAAALEHDKAYDQQLKAGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRLLEPLGLVEEAAKTAPGKKRPVEQSPQEPDSSAGIGKSGAQPAKKRLNFGQTGDTESVPDPQPIGEPPAAPSGVGSLTMASGGGAPVADNNEGADGVGSSSGNWHCDSQWLGDRVITTSTRTWALPTYNNHLYKQISNSTSGGSSNDNAYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTDNNGVKTIANNLTSTVQVFTDSDYQLPYVLGSAHEGCLPPFPADVFMIPQYGYLTLNDGSQAVGRSSFYCLEYFPSQMLRTGNNFQFSYEFENVPFHSSYAHSQSLDRLMNPLIDQYLYYLSKTIDGAATKNQTLKFSVAGPSNMAVQGRNYIPGPSYRQQRVSTTVTQNNNSEFAWPGASSWALNGRNSLMNPGPAMASHKEGEDRFFPLSGSLIFGKQGTGRDNVDADKVMITNEEEIKTTNPVATESYGQVATNHQSDGTLAVPFKAQAQTGWVQNQGILPGMVWQDRDVYLQGPIWAKIPHTDGNFHPSPLMGGFGMKHPPPQILIKNTPVPADPPTAFNKDKLNSFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYYKSNNVEFAVNTEGVYSEPRPIGTRYLTRNL
>9P4Q_E2 len=124 copies=1 longest=0
PRKNRPPVAVVSPQFQEISLPTTSTIIDGSQSTDDDKIVQYHWEELKGPLREEKISEDTAILKLSKLVPGNYTFSLTVVDSDGATNSTTASLTVNKAVDYPPGGSEQKLISEEDLGGSHHHHHH
>9P4R_E1 len=743 copies=1 longest=1
MAADGYLPDWLEDNLSEGIREWWALKPGAPQPKANQQHQDNARGLVLPGYKYLGPGNGLDKGEPVNAADAAALEHDKAYDQQLKAGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRLLEPLGLVEEAAKTAPGKKRPVEQSPQEPDSSAGIGKSGAQPAKKRLNFGQTGDTESVPDPQPIGEPPAAPSGVGSLTMASGGGAPVADNNEGADGVGSSSGNWHCDSQWLGDRVITTSTRTWALPTYNNHLYKQISNSTSGGSSNDNAYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTDNNGVKTIANNLTSTVQVFTDSDYQLPYVLGSAHEGCLPPFPADVFMIPQYGYLTLNDGSQAVGRSSFYCLEYFPSQMLRTGNNFQFSYEFENVPFHSSYAHSQSLDRLMNPLIDQYLYYLSKTIDGAATKNQTLKFSVAGPSNMAVQGRNYIPGPSYRQQRVSTTVTQNNNSEFAWPGASSWALNGRNSLMNPGPAMASHKEGEDRFFPLSGSLIFGKQGTGRDNVDADKVMITNEEEIKTTNPVATESYGQVATNHQSAQGNNTRSVAQAQTGWVQNQGILPGMVWQDRDVYLQGPIWAKIPHTDGNFHPSPLMGGFGMKHPPPQILIKNTPVPADPPTAFNKDKLNSFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYYKSNNVEFAVNTEGVYSEPRPIGTRYLTRNL
>9PGD_E1 len=396 copies=1 longest=1
MTTLVLDNGAYNAKIGYSHENVSVIPNCQFRSKTARLKTFTANQIDEIKDPSGLFYILPFQKGYLVNWDVQRQVWDYLFGKEMYQVDFLDTNIIITEPYFNFTSIQESMNEILFEEYQFQAVLRVNAGALSAHRYFRDNPSELCCIIVDSGYSFTHIVPYCRSKKKKEAIIRINVGGKLLTNHLKEIISYRQLHVMDETHVINQVKEDVCYVSQDFYRDMDIAKLKGEENTVMIDYVLPDFSTIKKGFCKPREEMVLSGKYKSGEQILRLANERFAVPEILFNPSDIGIQEMGIPEAIVYSIQNLPEEMQPHFFKNIVLTGGNSLFPGFRDRVYSEVRCLTPTDYDVSVVLPENPITYAWEGGKLISENDDFEDMVVTREDYEENGHSVCEEKFDI
>9PGD_E2 len=154 copies=1 longest=0
MVEKKTSVRSQDPGQRRVLDRAARQRRINRQLEALENDNFQDDPHAGLPQLGKRLPQFDDDADTGKKKKKTRGDHFKLRFRKNFQALLEEQNLSVAEGPNYLTACAGPPSRPQRPFCAVCGFPSPYTCVSCGARYCTVRCLGTHQETRCLKWTV
>9PGD_E3 len=135 copies=1 longest=0
ARTKQTARKSTGGKAPRKQLATKAARKSAPATGGVKKPHRYRPGTVALREIRRYQKSTELLIRKLPFQRLVREIAQDFKTDLRFQSSAVMALQEASEAYLVALFEDTNLCAIHAKRVTIMPKDIQLARRIRGERA
>9PPQ_E1 len=334 copies=1 longest=1
LALAIAITALYSIVCVVGLFGNFLVMYVIVRYTKMKTATNIYIFNLALADALATSTLPFQSVNYLMGTWPFGTILCKIVISIDYYNMFTSIFTLCTMSVDRYIAVCHPVKALDFRTPRNAKIINVCNWILSSAIGLPVMFMATTKYRQGSIDCTLTFSHPTWYWENLLKICVFIFAFIMPVLIITVCYGLMILRLKSVRMLSGSKEKDRNLRRITRMVLVVVAVFIVCWTPIHIYVIIKALVTIPETTFQTVSWHFCIALGYTNSCLNPVLYAFLDENFKRCFREFCIPTSSNIEQQNSTRIRQNTRDHPSTANTVDRTNHQLENLEAETAPLP
>9PRZ_E1 len=231 copies=1 longest=1
PIVQNIQGQMVHQAISPRTLNAWVKVVEEKAFSPEVIPMFSALSEGATPQDLNTMLNTVGGHQAAMQMLKETINEEAAEWDRLHPVQAGPVAPGQMREPRGSDIAGTTSTLQEQIGWMTNNPPIPVGEIYKRWIILGLNKIVRMYSPSSILDIRQGPKESFRDYVDRFYKTLRAEQASQEVKNWMTETLLVQNANPDCKTILKALGPGATLEEMMTACQGVGGPGHKARVL
>9PV0_E1 len=495 copies=1 longest=1
MNKQIFVLYFNIFLIFLGIGLVIPVLPVYLKDLGLTGSDLGLLVAAFALSQMIISPFGGTLADKLGKKLIICIGLILFSVSEFMFAVGHNFSVLMLSRVIGGMSAGMVMPGVTGLIADISPSHQKAKNFGYMSAIINSGFILGPGIGGFMAEVSHRMPFYFAGALGILAFIMSIVLIHDPKKSTTSGFQKLEPQLLTKINWKVFITPVILTLVLSFGLSAFETLYSLYTADKVNYSPKDISIAITGGGIFGALFQIYFFDKFMKYFSELTFIAWSLLYSVVVLILLVFANDYWSIMLISFVVFIGFDMIRPAITNYFSNIAGERQGFAGGLNSTFTSMGNFIGPLIAGALFDVHIEAPIYMAIGVSLAGVVIVLIEKQHRAAAADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLENLYFQ
>9PVM_E1 len=580 copies=1 longest=1
MHHHHHHDYKDDDDKGSSGLEVLFQGPGSSGMSPTISHKDSSRQRRPGNFSHSLDMKSGPLPPGGWDDSHLDSAGREGDREALLGDTGTGDFLKAPQSFRAELSSILLLLFLYVLQGIPLGLAGSIPLILQSKNVSYTDQAFFSFVFWPFSLKLLWAPLVDAVYVKNFGRRKSWLVPTQYILGLFMIYLSTQVDRLLGNTDDRTPDVIALTVAFFLFEFLAATQDIAVDGWALTMLSRENVGYASTCNSVGQTAGYFLGNVLFLALESADFCNKYLRFQPQPRGIVTLSDFLFFWGTVFLITTTLVALLKKENEVSVVKEETQGITDTYKLLFAIIKMPAVLTFCLLILTAKIGFSAADAVTGLKLVEEGVPKEHLALLAVPMVPLQIILPLIISKYTAGPQPLNTFYKAMPYRLLLGLEYALLVWWTPKVEHQGGFPIYYYIVVLLSYALHQVTVYSMYVSIMAFNAKVSDPLIGGTYMTLLNTVSNLGGNWPSTVALWLVDPLTVKECVGASNQNCRTPDAVELCKKLGGSCVTALDGYYVESIICVFIGFGWWFFLGPKFKKLQDEGSSSWKCKRNN
>9PX8_E1 len=31 copies=15 longest=1
QLHQQQHQQQHLQHVQHLQQVQFHQHQQQLS
>9PXJ_E1 len=795 copies=1 longest=1
MKRFLLPALLSALMIAEVHAESFTVSDIRVNGLQRVSAGSVFAALPLNVGETIDDQALVQATRSLFKTGFFQDIQLGRDGNVLVVTVVERPSISSIEIEGNKAISKEDLLKGLKQSGLAEGEIFQRATLEGVRNELQRQYVAQGRYSAEINAEVIPQPRNRVALKININEGTVAAISHINVVGNTVFSEEDLTDLFELKTTNWLSFFKNDDKYAREKLSGDLERLRSYYLDRGYINMDIASTQVSITPDKKHVYITVNINEGEKYTIRDVKLTGDLKVPEEEVKRLLLVQKGQVFSRKVMTTTSDLITRRLGNEGYTFANVNGVPEAHDDDKTVSVTFVVDPGKRAYVNRINFRGNTKTEDEVLRREMRQMEGGWASTYLIDQSKARLERLGYFKEVNVETPAVPGTDDQVDVNYSVEEQPSGSITASVGFAQSAGLILGGSISQNNFLGTGNKVSIGLTRSEYQTRYNFGFVDPYWTVDGVSLGYNAFYRKTDYDELDVDVASYSVNSLGAGMSIGYPISETSRLTYGLSVQRDQIDTGRYTVDEIYDFLDKEGDNFTNFKASIGWSESTLNKGVLATRGHSQSLTLETTLPGSDLSFYKIDYRGQVFAPLTDNYTMRFHTELGYGDGYGSTERLPFYENYYAGGFNSVRGFKDSTLGPRSTPSIARDANGKQIPGPDSQGRYTDPDQDPEAFGGNILITGGAELLFPLPFVKDQRQLRTVLFWDVGSTFDTDCPTKTTTNCDGIKTDNLASSVGVGLTWITALGPLSFSLATPIKKPDNAETQVFQFSLGQTF
>9PXJ_E2 len=256 copies=1 longest=0
MASSLAPRQVIRDGQFITSPNGKYKLVMQADGNLVLYEDGTKPIWNTTPVGPGAKAVMEFNLNLYNKAGQMAWSSNVRTPYLFESFKDRAYLVLQDDGDFGIFLDQANWASLVLSEPEVGVKEKLIPTGTVMGPGTEYVNGNYRLVFQDDGNLVIYRINPLAVVWATYTMGADKAVVQEDGNFVIYKGTTALWQTRTATGVPAYLEFTDSGRLSLSQPNLQWTLKRGSLFVPPKVIPGQHGPLDTTPIWSWPHDYP
>9PZD_E1 len=31 copies=10 longest=1
QLHQQQHQQQHLQHVQHLQQVQFHQHQQQLS
>9Q33_E1 len=188 copies=1 longest=0
GSKMDMEDADMTLWTEAEFEEKCTYIVNDHPWDSGADGGTSVQAEASLPRNLLFKYATNSEEVIGVMSKEYIPKGTRFGPLIGEIYTNDTVPKNANRKYFWRIYSRGELHHFIDGFNEEKSNWMRYVNPAHSPREQNLAACQNGMNIYFYTIKPIPANQELLVWYCRDFAERLHYPYPGELTMMNLTQ
>9Q33_E2 len=442 copies=1 longest=1
MAGEGDQQDAAHNMGNHLPLLPAESEEEDEMEVEDQDSKEAKKPNIINFDTSLPTSHTYLGADMEEFHGRTLHDDDSCQVIPVLPQVMMILIPGQTLPLQLFHPQEVSMVRNLIQKDRTFAVLAYSNVQEREAQFGTTAEIYAYREEQDFGIEIVKVKAIGRQRFKVLELRTQSDGIQQAKVQILPECVLPSTMSAVQLESLNKCQIFPSKPVSREDQCSYKWWQKYQKRKFHCANLTSWPRWLYSLYDAETLMDRIKKQLREWDENLKDDSLPSNPIDFSYRVAACLPIDDVLRIQLLKIGSAIQRLRCELDIMNKCTSLCCKQCQETEITTKNEIFSLSLCGPMAAYVNPHGYVHETLTVYKACNLNLIGRPSTEHSWFPGYAWTVAQCKICASHIGWKFTATKKDMSPQKFWGLTRSALLPTIPDTEDEISPDKVILCL
>9Q37_E1 len=31 copies=15 longest=1
QLHQQQHQQQHLQHVQHLQQVQFHQHQQQLS
>9Q3L_E1 len=460 copies=2 longest=1
MKTIIALSYIFCLVFADYKDDDDAMGQPGNGSAFLLAPNRSHAPDHDVENLYFQGTQQRDEVWVVGMGIVMSLIVLAIVFGNVLVITAIAKFERLQTVTNYFITSLACADLVMGLAVVPFGAAHILTKTWTFGNFWCEFWTSIDVLCVTASIETLCVIAVDRYFAITSPFKYQSLLTKNKARVIILMVWIVSGLTSFLPIQMHWYRATHQEAINCYAEETCCDFFTNQAYAIASSIVSFYVPLVIMVFVYSRVFQEAKRQLQKIDKSEGRFHVQNLSQVEQDGRTGHGLRRSSKFCLKEHKALKTLGIIMGTFTLCWLPFFIVNIVHVIQDNLIRKEVYILLNWIGYVNSGFNPLIYCRSPDFRIAFQELLCLRRSSLKAYGNGYSSNGNTGEQSGLEVLFQGPYHVEQEKENKLLAEDLPGTEDFVGHQGTVPSDNIDSQGRNASTNDSLLETSQVAPA
>9Q7J_E1 len=31 copies=9 longest=1
QLHQQQHQQQHLQHVQHLQQVQFHQHQQQLS
>9QJJ_E1 len=435 copies=1 longest=1
SNAMLGNSAPGPATREAGSALLALQQTALQEDQENINPEKAAPVQQPRTRAALAVLKSGNPRGLAQQQRPKTRRVAPLKDLPVNDEHVTVPPWKANSKQPAFTIHVDEAEKEAQKKPAESQKIEREDALAFNSAISLPGPRKPLVPLDYPMDGSFESPHTMDMSIVLEDEKPVSVNEVPDYHEDIHTYLREMEVKCKPKVGYMKKQPDITNSMRAILVDWLVEVGEEYKLQNETLHLAVNYIDRFLSSMSVLRGKLQLVGTAAMLLASKFEEIYPPEVAEFVYITDDTYTKKQVLRMEHLVLKVLTFDLAAPTVNQFLTQYFLHQQPANCKVESLAMFLGELSLIDADPYLKYLPSVIAGAAFHLALYTVTGQSWPESLIRKTGYTLESLKPCLMDLHQTYLKAPQHAQQSIREKYKNSKYHGVSLLNPPETLNL
>9QJJ_E2 len=301 copies=1 longest=0
SNAMENFQKVEKIGEGTYGVVYKARNKLTGEVVALKKIRLDTETEGVPSTAIREISLLKELNHPNIVKLLDVIHTENKLYLVFEFLHQDLKKFMDASALTGIPLPLIKSYLFQLLQGLAFCHSHRVLHRDLKPQNLLINTEGAIKLADFGLARAFGVPVRTYTHEVVTLWYRAPEILLGCKYYSTAVDIWSLGCIFAEMVTRRALFPGDSEIDQLFRIFRTLGTPDEVVWPGVTSMPDYKPSFPKWARQDFSKVVPPLDEDGRSLLSQMLHYDPNKRISAKAALAHPFFQDVTKPVPHLRL
>9QJN_E1 len=93 copies=1 longest=0
SNAPVTFSTGIKMGQHISLAPIHKLEEALYEYQPLQIETYGPHVPELEMLGRLGYLNHVRAASPQDLAGGYTSSLACHRALQDAFSGLFWQPS
>9QJN_E2 len=324 copies=1 longest=0
XMYHNSSQKRHWTFSSEEQLARLRADANRKFRCKAVANGKVLPNDPVFLEPHEEMTLCKYYEKRLLEFCSVFKPAMPRSVVGTACMYFKRFYLNNSVMEYHPRIIMLTCAFLACKVDEFNVSSPQFVGNLRESPLGQEKALEQILEYELLLIQQLNFHLIVHNPYRPFEGFLIDLKTRYPILENPEILRKTADDFLNRIALTDAYLLYTPSQIALTAILSSASRAGITMESYLSESLMLKENRTCLSQLLDIMKSMRNLVKKYEPPRSEEVAVLKQKLERCHSAELALNVITKKRKGYEDDDYVSKKSKHEEEEWTDDDLVESL
>9QJN_E3 len=349 copies=1 longest=1
SNAMALDVKSRAKRYEKLDFLGEGQFATVYKARDKNTNQIVAIKKIKLGHRSEAKDGINRTALREIKLLQELSHPNIIGLLDAFGHKSNISLVFDFMETDLEVIIKDNSLVLTPSHIKAYMLMTLQGLEYLHQHWILHRDLKPNNLLLDENGVLKLADFGLAKSFGSPNRAYTHQVVTRWYRAPELLFGARMYGVGVDMWAVGCILAELLLRVPFLPGDSDLDQLTRIFETLGTPTEEQWPDMCSLPDYVTFKSFPGIPLHHIFSAAGDDLLDLIQGLFLFNPCARITATQALKMKYFSNRPGPTPGCQLPRPNCPVETLKEQSNPALAIKRKRTEALEQGGLPKKLIF
>9QL1_E1 len=311 copies=1 longest=0
SNAMAGKKNVLSSLAVYAEDSEPESDGEAGIEAVGSAAEEKGGLVSDAYGEDDFSRLGGDEDGYEEEEDENSRQSEDDDSETEKPEADDPKDNTEAEKRDPQELVASFSERVRNMSPDEIKIPPEPPGRCSNHLQDKIQKLYERKIKEGMDMNYIIQRKKEFRNPSIYEKLIQFCAIDELGTNYPKDMFDPHGWSEDSYYEALAKAQKIEMDKLEKAKKERTKIEFVTGTKKGTTTNATSTTTTTASTAVADAQKRKSKWDSAIPVTTIAQPTILTTTATLPAVVTVTTSASGSKTTVISAVGTIVKKAKQ
>9QL1_E2 len=322 copies=1 longest=0
SNAMAAAAAAAGAAGSAAPAAAAGAPGSGGAPSGSQGVLIGDRLYSGVLITLENCLLPDDKLRFTPSMSSGLDTDTETDLRVVGCELIQAAGILLRLPQVAMATGQVLFQRFFYTKSFVKHSMEHVSMACVHLASKIEEAPRRIRDVINVFHRLRQLRDKKKPVPLLLDQDYVNLKNQIIKAERRVLKELGFCVHVKHPHKIIVMYLQVLECERNQHLVQTSWNYMNDSLRTDVFVRFQPESIACACIYLAARTLEIPLPNRPHWFLLFGATEEEIQEICLKILQLYARKKVDLTHLEGEVEKRKHAIEEAKAQARGLLPGG
>9QL1_E3 len=442 copies=1 longest=1
SNAMSEDEERENENHLLVVPESRFDRDSGESEEAEEEVGEGTPQSSALTEGDYVPDSPALSPIELKQELPKYLPALQGCRSVEEFQCLNRIEEGTYGVVYRAKDKKTDEIVALKRLKMEKEKEGFPITSLREINTILKAQHPNIVTVREIVVGSNMDKIYIVMNYVEHDLKSLMETMKQPFLPGEVKTLMIQLLRGVKHLHDNWILHRDLKTSNLLLSHAGILKVGDFGLAREYGSPLKAYTPVVVTLWYRAPELLLGAKEYSTAVDMWSVGCIFGELLTQKPLFPGKSEIDQINKVFKDLGTPSEKIWPGYSELPAVKKMTFSEHPYNNLRKRFGALLSDQGFDLMNKFLTYFPGRRISAEDGLKHEYFRETPLPIDPSMFPTWPAKSEQQRVKRGTSPRPPEGGLGYSQLGDDDLKETGFHLTTTNQGASAAGPGFSLKF
>9QO8_E1 len=632 copies=1 longest=1
MTAEKALPLGNGKAAEEARESEAPGGGCSSGGAAPARHPRVKRDKAVHERGHWNNKVEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFLIPYVVFFICCGIPVFFLETALGQFTSEGGITCWRKVCPLFEGIGYATQVIEAHLNVYYIIILAWAIFYLSNCFTTELPWATCGHEWNTENCVEFQKLNVSNYSHVSLQNATSPVMEFWEHRVLAISDGIEHIGNLRWELALCLLAAWTICYFCIWKGTKSTGKVVYVTATFPYIMLLILLIRGVTLPGASEGIKFYLYPDLSRLSDPQVWVDAGTQIFFSYAICLGCLTALGSYNNYNNNCYRDCIMLCCLNSGTSFVAGFAIFSVLGFMAYEQGVPIAEVAESGPGLAFIAYPKAVTMMPLSPLWATLFFMMLIFLGLDSQFVCVESLVTAVVDMYPKVFRRGYRRELLILALSVISYFLGLVMLTEGGMYIFQLFDSYAASGMCLLFVAIFECICIGWVYGSNRFYDNIEDMIGYRPPSLIKWCWMIMTPGICAGIFIFFLIKYKPLKYNNIYTYPAWGYGIGWLMALSSMLCIPLWICITVWKTEGTLPEKLQKLTTPSTDLKMRGKLGVSPRMVTVNDCDAKLKSDGTIAAITEKETHF
>9QO9_E1 len=632 copies=1 longest=1
MTAEKALPLGNGKAAEEARESEAPGGGCSSGGAAPARHPRVKRDKAVHERGHWNNKVEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFLIPYVVFFICCGIPVFFLETALGQFTSEGGITCWRKVCPLFEGIGYATQVIEAHLNVYYIIILAWAIFYLSNCFTTELPWATCGHEWNTENCVEFQKLNVSNYSHVSLQNATSPVMEFWEHRVLAISDGIEHIGNLRWELALCLLAAWTICYFCIWKGTKSTGKVVYVTATFPYIMLLILLIRGVTLPGASEGIKFYLYPDLSRLSDPQVWVDAGTQIFFSYAICLGCLTALGSYNNYNNNCYRDCIMLCCLNSGTSFVAGFAIFSVLGFMAYEQGVPIAEVAESGPGLAFIAYPKAVTMMPLSPLWATLFFMMLIFLGLDSQFVCVESLVTAVVDMYPKVFRRGYRRELLILALSVISYFLGLVMLTEGGMYIFQLFDSYAASGMCLLFVAIFECICIGWVYGSNRFYDNIEDMIGYRPPSLIKWCWMIMTPGICAGIFIFFLIKYKPLKYNNIYTYPAWGYGIGWLMALSSMLCIPLWICITVWKTEGTLPEKLQKLTTPSTDLKMRGKLGVSPRMVTVNDCDAKLKSDGTIAAITEKETHF
>9QP4_E1 len=488 copies=1 longest=1
MSEEEFYLFKNISSVGPWDGPQYHIAPVWAFYLQAAFMGTVFLIGFPLNAMVLVATLRYKKLRQPLNYILVNVSFGGFLLCIFSVFPVFVASCNGYFVFGRHVCALEGFLGTVAGLVTGWSLAFLAFERYIVICKPFGNFRFSSKHALTVVLATWTIGIGVSIPPFFGWSRFIPEGLQCSCGPDWYTVGTKYRSESYTWFLFIFCFIVPLSLICFSYTQLLRALKAVAAQARRQLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLRSTLQESATTQKAEREVSRMVVVMVGSFCVCYVPYAAFAMYMVNNRNHGLDLRLVTIPSFFSKSACIYNPIIYCFMNKQFQACIMKMVCGKAMTDESDTCSSQKTEVSTVSSTQVGPNLEVLFQGPDYKDDDDKGAGATETSQVAPA
>9QP4_E2 len=215 copies=1 longest=0
MDIVLTQSPATLSVTPGDRVSLSCRASQSVSNYLHWYQQKSHESPRLLIKYASQSISGIPSRFSGSGSGTDFTLSINSVETEDFGMYFCQQSNSWPLTFGAGTKLELRRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKDINVKWKIDGSERQNGVLNSWTDQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC
>9QP4_E3 len=222 copies=1 longest=0
MQIQLQQSGPELVKPGASVKISCKASGYTFTDFYINWMKQRPGQGLEWIGWIFPGSGNTHYNEKFKGKATLIVDTSSSTAFMQLNSLTSEDSAVYFCTRPVSYYYDFDYWGQGTTLTVSSAKTTAPSVYPLAPVCGDTSGSSVTLGCLVKGYFPEPVTLTWNSGSLSSGVHTFPAVLQSDLYTLSSSVTVTSSTWPSQSITCNVAHPASSTKVDKKIEPRGC
>9QQS_E1 len=211 copies=1 longest=0
MDKLTIISGCLFLAADIFAIASIANPDWINTGESAGALTVGLVRQCQTIHGRDRTCIPPRLPPEWVTTLFFIIMGIISLTVTCGLLVASHWRREATKYARWIAFTGMILFCMAALIFPIGFYINEVGGQPYKLPNNTVVGSSYVLFVLSIFFTIVGLLFAGKVCLPGGTLEVLFQGPGGSGSAWSHPQFEKGGGSGGGSGGSAWSHPQFEK
>9QQS_E2 len=477 copies=1 longest=1
KCQCNGHADTCNEQDGTGCPCQNNTETGTCQGSSPSDRRDCYKYQCAKCRESFHGSPLGGQQCYRLISVEQECCLDPTSQTNCFHEPKRRALGPGRTVLFGVQPKFTNVDIRLTLDVTFGAVDLYVSTSYDTFVVRVAPDTGVHTVHIQPPPAPPPPPPPADGGPRGAGDPGGAGASSGPGAPAEPRVREVWPRGLITYVTVTEPSAVLVVRGVRDRLVITYPHEHHALKSSRFYLLLLGVGDPSGPGANGSADSQGLLFFRQDQAHIDLFVFFSVFFSCFFLFLSLCVLLWKAKQALDQRQEQRRHLQEMTKMASRPFAKVTVCFPPDPTAPASAWKPAGLPPPAFRRSEPFLAPLLLTGAGGPWGPMGGGCCPPAIPATTAGLRAGPITLEPTEDGMAGVATLLLQLPGGPHAPNGACLGSALVTLRHRLHEYCGGGGGAGGSGHGTGAGRKGLLSQDNLTSMSLGTETSQVAPA
>9QQS_E3 len=129 copies=1 longest=0
QVQLVESGGGLVQAGGSLRLSCAASGSIFSYDDMGWYRQAPGKQRELVATFTNVGSTNYVDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCHAYTVRRFQGMEYWGKGTQVTVSSHHHHHHEPEA
>9QTY_E1 len=211 copies=1 longest=0
MDKLTIISGCLFLAADIFAIASIANPDWINTGESAGALTVGLVRQCQTIHGRDRTCIPPRLPPEWVTTLFFIIMGIISLTVTCGLLVASHWRREATKYARWIAFTGMILFCMAALIFPIGFYINEVGGQPYKLPNNTVVGSSYVLFVLSIFFTIVGLLFAGKVCLPGGTLEVLFQGPGGSGSAWSHPQFEKGGGSGGGSGGSAWSHPQFEK
>9QTY_E2 len=129 copies=1 longest=0
QVQLVESGGGLVQAGGSLRLSCAASGSIFSYDDMGWYRQAPGKQRELVATFTNVGSTNYVDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCHAYTVRRFQGMEYWGKGTQVTVSSHHHHHHEPEA
>9QTY_E3 len=344 copies=1 longest=1
KCQCNGHADTCNEQDGTGCPCQNNTETGTCQGSSPSDRRDCYKYQCAKCRESFHGSPLGGQQCYRLISVEQECCLDPTSQTNCFHEPKRRALGPGRTVLFGVQPKFTNVDIRLTLDVTFGAVDLYVSTSYDTFVVRVAPDTGVHTVHIQPPPAPPPPPPPADGGPRGAGDPGGAGASSGPGAPAEPRVREVWPRGLITYVTVTEPSAVLVVRGVRDRLVITYPHEHHALKSSRFYLLLLGVGDPSGPGANGSADSQGLLFFRQDQAHIDLFVFFSVFFSCFFLFLSLCVLLWKAKQALDQRQEQRRHLQEMTKMASRPFAKVTVCFPPDPTAPAGTETSQVAPA
>9QZO_E1 len=663 copies=1 longest=1
MGLLLKPGARQGSGTSSVPDRRCPRSVFSNIKVFVLCHGLLQLCQLLYSAYFKSSLTTIEKRFGLSSSSSGLISSLNEISNATLIIFISYFGSRVNRPRMIGIGGLLLAAGAFVLTLPHFLSEPYQYTSTTDGNRSSFQTDLCQKHFGALPPSKCHSTVPDTHKETSSLWGLMVVAQLLAGIGTVPIQPFGISYVDDFAEPTNSPLYISILFAIAVFGPAFGYLLGSVMLRIFVDYGRVDTATVNLSPGDPRWIGAWWLGLLISSGFLIVTSLPFFFFPRAMSRGAERSVTAEETMQTEEDKSRGSLMDFIKRFPRIFLRLLMNPLFMLVVLSQCTFSSVIAGLSTFLNKFLEKQYGATAAYANFLIGAVNLPAAALGMLFGGILMKRFVFPLQTIPRVAATIITISMILCVPLFFMGCSTSAVAEVYPPSTSSSIHPQQPPACRRDCSCPDSFFHPVCGDNGVEYVSPCHAGCSSTNTSSEASKEPIYLNCSCVSGGSASAKTGSCPTSCAQLLLPSIFLISFAALIACISHNPLYMMVLRVVNQDEKSFAIGVQFLLMRLLAWLPAPSLYGLLIDSSCVRWNYLCSGRRGACAYYDNDALRNRYLGLQMVYKALGTLLLFFISWRMKKNREYSLQENTSGLIGLNDIFEAQKIEWHELEGG
>9R07_E1 len=663 copies=1 longest=1
MGLLLKPGARQGSGTSSVPDRRCPRSVFSNIKVFVLCHGLLQLCQLLYSAYFKSSLTTIEKRFGLSSSSSGLISSLNEISNATLIIFISYFGSRVNRPRMIGIGGLLLAAGAFVLTLPHFLSEPYQYTSTTDGNRSSFQTDLCQKHFGALPPSKCHSTVPDTHKETSSLWGLMVVAQLLAGIGTVPIQPFGISYVDDFAEPTNSPLYISILFAIAVFGPAFGYLLGSVMLRIFVDYGRVDTATVNLSPGDPRWIGAWWLGLLISSGFLIVTSLPFFFFPRAMSRGAERSVTAEETMQTEEDKSRGSLMDFIKRFPRIFLRLLMNPLFMLVVLSQCTFSSVIAGLSTFLNKFLEKQYGATAAYANFLIGAVNLPAAALGMLFGGILMKRFVFPLQTIPRVAATIITISMILCVPLFFMGCSTSAVAEVYPPSTSSSIHPQQPPACRRDCSCPDSFFHPVCGDNGVEYVSPCHAGCSSTNTSSEASKEPIYLNCSCVSGGSASAKTGSCPTSCAQLLLPSIFLISFAALIACISHNPLYMMVLRVVNQDEKSFAIGVQFLLMRLLAWLPAPSLYGLLIDSSCVRWNYLCSGRRGACAYYDNDALRNRYLGLQMVYKALGTLLLFFISWRMKKNREYSLQENTSGLIGLNDIFEAQKIEWHELEGG
>9R0A_E1 len=663 copies=1 longest=1
MGLLLKPGARQGSGTSSVPDRRCPRSVFSNIKVFVLCHGLLQLCQLLYSAYFKSSLTTIEKRFGLSSSSSGLISSLNEISNATLIIFISYFGSRVNRPRMIGIGGLLLAAGAFVLTLPHFLSEPYQYTSTTDGNRSSFQTDLCQKHFGALPPSKCHSTVPDTHKETSSLWGLMVVAQLLAGIGTVPIQPFGISYVDDFAEPTNSPLYISILFAIAVFGPAFGYLLGSVMLRIFVDYGRVDTATVNLSPGDPRWIGAWWLGLLISSGFLIVTSLPFFFFPRAMSRGAERSVTAEETMQTEEDKSRGSLMDFIKRFPRIFLRLLMNPLFMLVVLSQCTFSSVIAGLSTFLNKFLEKQYGATAAYANFLIGAVNLPAAALGMLFGGILMKRFVFPLQTIPRVAATIITISMILCVPLFFMGCSTSAVAEVYPPSTSSSIHPQQPPACRRDCSCPDSFFHPVCGDNGVEYVSPCHAGCSSTNTSSEASKEPIYLNCSCVSGGSASAKTGSCPTSCAQLLLPSIFLISFAALIACISHNPLYMMVLRVVNQDEKSFAIGVQFLLMRLLAWLPAPSLYGLLIDSSCVRWNYLCSGRRGACAYYDNDALRNRYLGLQMVYKALGTLLLFFISWRMKKNREYSLQENTSGLIGLNDIFEAQKIEWHELEGG
>9R0I_E1 len=663 copies=1 longest=1
MGLLLKPGARQGSGTSSVPDRRCPRSVFSNIKVFVLCHGLLQLCQLLYSAYFKSSLTTIEKRFGLSSSSSGLISSLNEISNATLIIFISYFGSRVNRPRMIGIGGLLLAAGAFVLTLPHFLSEPYQYTSTTDGNRSSFQTDLCQKHFGALPPSKCHSTVPDTHKETSSLWGLMVVAQLLAGIGTVPIQPFGISYVDDFAEPTNSPLYISILFAIAVFGPAFGYLLGSVMLRIFVDYGRVDTATVNLSPGDPRWIGAWWLGLLISSGFLIVTSLPFFFFPRAMSRGAERSVTAEETMQTEEDKSRGSLMDFIKRFPRIFLRLLMNPLFMLVVLSQCTFSSVIAGLSTFLNKFLEKQYGATAAYANFLIGAVNLPAAALGMLFGGILMKRFVFPLQTIPRVAATIITISMILCVPLFFMGCSTSAVAEVYPPSTSSSIHPQQPPACRRDCSCPDSFFHPVCGDNGVEYVSPCHAGCSSTNTSSEASKEPIYLNCSCVSGGSASAKTGSCPTSCAQLLLPSIFLISFAALIACISHNPLYMMVLRVVNQDEKSFAIGVQFLLMRLLAWLPAPSLYGLLIDSSCVRWNYLCSGRRGACAYYDNDALRNRYLGLQMVYKALGTLLLFFISWRMKKNREYSLQENTSGLIGLNDIFEAQKIEWHELEGG
>9R0M_E1 len=663 copies=1 longest=1
MGLLLKPGARQGSGTSSVPDRRCPRSVFSNIKVFVLCHGLLQLCQLLYSAYFKSSLTTIEKRFGLSSSSSGLISSLNEISNATLIIFISYFGSRVNRPRMIGIGGLLLAAGAFVLTLPHFLSEPYQYTSTTDGNRSSFQTDLCQKHFGALPPSKCHSTVPDTHKETSSLWGLMVVAQLLAGIGTVPIQPFGISYVDDFAEPTNSPLYISILFAIAVFGPAFGYLLGSVMLRIFVDYGRVDTATVNLSPGDPRWIGAWWLGLLISSGFLIVTSLPFFFFPRAMSRGAERSVTAEETMQTEEDKSRGSLMDFIKRFPRIFLRLLMNPLFMLVVLSQCTFSSVIAGLSTFLNKFLEKQYGATAAYANFLIGAVNLPAAALGMLFGGILMKRFVFPLQTIPRVAATIITISMILCVPLFFMGCSTSAVAEVYPPSTSSSIHPQQPPACRRDCSCPDSFFHPVCGDNGVEYVSPCHAGCSSTNTSSEASKEPIYLNCSCVSGGSASAKTGSCPTSCAQLLLPSIFLISFAALIACISHNPLYMMVLRVVNQDEKSFAIGVQFLLMRLLAWLPAPSLYGLLIDSSCVRWNYLCSGRRGACAYYDNDALRNRYLGLQMVYKALGTLLLFFISWRMKKNREYSLQENTSGLIGLNDIFEAQKIEWHELEGG
>9R0N_E1 len=663 copies=1 longest=1
MGLLLKPGARQGSGTSSVPDRRCPRSVFSNIKVFVLCHGLLQLCQLLYSAYFKSSLTTIEKRFGLSSSSSGLISSLNEISNATLIIFISYFGSRVNRPRMIGIGGLLLAAGAFVLTLPHFLSEPYQYTSTTDGNRSSFQTDLCQKHFGALPPSKCHSTVPDTHKETSSLWGLMVVAQLLAGIGTVPIQPFGISYVDDFAEPTNSPLYISILFAIAVFGPAFGYLLGSVMLRIFVDYGRVDTATVNLSPGDPRWIGAWWLGLLISSGFLIVTSLPFFFFPRAMSRGAERSVTAEETMQTEEDKSRGSLMDFIKRFPRIFLRLLMNPLFMLVVLSQCTFSSVIAGLSTFLNKFLEKQYGATAAYANFLIGAVNLPAAALGMLFGGILMKRFVFPLQTIPRVAATIITISMILCVPLFFMGCSTSAVAEVYPPSTSSSIHPQQPPACRRDCSCPDSFFHPVCGDNGVEYVSPCHAGCSSTNTSSEASKEPIYLNCSCVSGGSASAKTGSCPTSCAQLLLPSIFLISFAALIACISHNPLYMMVLRVVNQDEKSFAIGVQFLLMRLLAWLPAPSLYGLLIDSSCVRWNYLCSGRRGACAYYDNDALRNRYLGLQMVYKALGTLLLFFISWRMKKNREYSLQENTSGLIGLNDIFEAQKIEWHELEGG
>9R0O_E1 len=663 copies=1 longest=1
MGLLLKPGARQGSGTSSVPDRRCPRSVFSNIKVFVLCHGLLQLCQLLYSAYFKSSLTTIEKRFGLSSSSSGLISSLNEISNATLIIFISYFGSRVNRPRMIGIGGLLLAAGAFVLTLPHFLSEPYQYTSTTDGNRSSFQTDLCQKHFGALPPSKCHSTVPDTHKETSSLWGLMVVAQLLAGIGTVPIQPFGISYVDDFAEPTNSPLYISILFAIAVFGPAFGYLLGSVMLRIFVDYGRVDTATVNLSPGDPRWIGAWWLGLLISSGFLIVTSLPFFFFPRAMSRGAERSVTAEETMQTEEDKSRGSLMDFIKRFPRIFLRLLMNPLFMLVVLSQCTFSSVIAGLSTFLNKFLEKQYGATAAYANFLIGAVNLPAAALGMLFGGILMKRFVFPLQTIPRVAATIITISMILCVPLFFMGCSTSAVAEVYPPSTSSSIHPQQPPACRRDCSCPDSFFHPVCGDNGVEYVSPCHAGCSSTNTSSEASKEPIYLNCSCVSGGSASAKTGSCPTSCAQLLLPSIFLISFAALIACISHNPLYMMVLRVVNQDEKSFAIGVQFLLMRLLAWLPAPSLYGLLIDSSCVRWNYLCSGRRGACAYYDNDALRNRYLGLQMVYKALGTLLLFFISWRMKKNREYSLQENTSGLIGLNDIFEAQKIEWHELEGG
>9R1H_E1 len=613 copies=1 longest=1
MDENKARGNWSSKLDFILSMVGYAVGLGNVWRFPYLAFQNGGGAFLIPYLMMLALAGLPIFFLEVSLGQFASQGPVSVWKAIPALQGCGIAMLIISVLIAIYYNVIICYTLFYLFASFVSVLPWGSCNNPWNTPECKDKTKLLLDSCVISDHPKIQIKNSTFCMTAYPNVTMVNFTSQANKTFVSGSEEYFKYFVLKISAGIEYPGEIRWPLALCLFLAWVIVYASLAKGIKTSGKVVYFTATFPYVVLVILLIRGVTLPGAGAGIWYFITPKWEKLTDATVWKDAATQIFFSLSAAWGGLITLSSYNKFHNNCYRDTLIVTCTNSATSIFAGFVIFSVIGFMANERKVNIENVADQGPGIAFVVYPEALTRLPLSPFWAIIFFLMLLTLGLDTMFATIETIVTSISDEFPKYLRTHKPVFTLGCCICFFIMGFPMITQGGIYMFQLVDTYAASYALVIIAIFELVGISYVYGLQRFCEDIEMMIGFQPNIFWKVCWAFVTPTILTFILCFSFYQWEPMTYGSYRYPNWSMVLGWLMLACSVIWIPIMFVIKMHLAPGRFIERLKLVCSPQPDWGPFLAQHRGERYKNMIDPLGTSSLGLKLPVKDLELGTQC
>9R8P_E1 len=590 copies=1 longest=1
DAHKSEVAHRFKDLGEENFKALVLIAFAQYLQKCPFEELVKMVKEVTEFAKKCVADETAENCDKSLHTLFGDKLCQVATLRETYGEMADCCAKQDPERHECFLQHKDDNPNLPRLERPEPDVMCKAFHDNEETFLNKYLYEIARRHPYFYAPELLYFAKRYKKAFTECCQAEDKAACLLPKLDALREEGKESSAKQRHKCAILQKFGERAFKAWAIARLSQRFPKAPFEEISKLVTDLTKVHKECCHGDMLECMDDRAKLAKYICENQDSISSKLKECCEKPLLERSHCIAEVENDDKPEDLPPLAADFVESKDVCKNYAEAKDVFLARFLYEYARRHPDYSVTLLLRIAKTYEDTLEKCCKAEDPHECYAKVEEEFKKLVEETQNLIKQNCELFEKLGEYYFQNALLIRYTKKMPQVPTDTLVELTRNLGKVGSKCCKLPEEKRLPCIEDYLSIVLNQLCVLHEKTPVNDRVTKCCTESLVNRRHCFSALEVDETYVPKEFNAETFTFHADLCTLSEEERQIMKQKFLVELVKHKPKATEEQLKAVMDDFTAFVEKCCKAEDPETCFAEEGSKLIAKSQAALGHHHHHH
>9RH2_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RH2_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RH2_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RH3_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RH3_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RH3_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RH4_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RH4_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RH4_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RH5_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RH5_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RH5_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RH6_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RH6_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RH6_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RH7_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RH7_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RH7_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RH8_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RH8_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RH8_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RH9_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RH9_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RH9_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RHA_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RHA_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RHA_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RHB_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RHB_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RHB_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RHC_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RHC_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RHC_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RHD_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RHD_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RHD_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RHE_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RHE_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RHE_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RHF_E1 len=407 copies=1 longest=1
MKHLHRFFSSDASGGIILIIAAILAMIMANSGATSGWYHDFLETPVQLRVGSLEINKNMLLWINDALMAVFFLLVGLEVKRELMQGSLASLRQAAFPVIAAIGGMIVPALLYLAFNYADPITREGWAIPAATDIAFALGVLALLGSRVPLALKIFLMALAIIDDLGAIIIIALFYTNDLSMASLGVAAVAIAVLAVLNLCGARRTGVYILVGVVLWTAVLKSGVHATLAGVIVGFFIPLKEKHGRSPAKRLEHVLHPWVAYLILPLFAFANAGVSLQGVTLDGLTSILPLGIIAGLLIGKPLGISLFCWLALRLKLAHLPEGTTYQQIMVVGILCGIGFTMSIFIASLAFGSVDPELINWAKLGILVGSISSAVIGYSWLRVRLRPSVAAAIEGRIEGRLEHHHHHH
>9RHF_E2 len=127 copies=1 longest=0
EVKLHQSGAELVRPGASVKLSCKALGYTFSDYEMHWVKQTPVHGLEWIGAIHPGSGGTAYNQKFKGKATLTADKSSSTAYMELSSLTSEDPAVYYCTKEEYGNDFDYWGQGTTVTVSSAWRHPQFGG
>9RHF_E3 len=119 copies=1 longest=0
DIELTQTPSSLSASLGERVSLTCRASQEISGYLSWLQQKPDGTIKRLIYAASTLDSGVPKRFSGSRSGSDYSLTISSLESEDFADYYCLQYASYPFTFGSGTKLEIKREQKLISEEDLN
>9RQL_E1 len=729 copies=1 longest=1
MSPHGDGRGQAQGRAVRVGLRRSGGIRGGVAVFAAVAAVFTFTLPPSVPGGDSGELITAAHELGVAHPPGYPLFTLVAKLAITLFPFGSIAYRVNLLCGLFGAVAASLLFFTVFRLSGSSAGGILAAGVFSFSRLTWQWSIAAEVFSLNNLFVGLLMALTVHFEEAATAKERSKVAKIGAFCCGLSLCNQHTIILYVLCIIPWILFQLLKKKELSLGSLLKLSLYFSAGLLPYVHLPISSYLNHARWTWGDQTTLQGFLTHFLREEYGTFSLAKSEIGSSMSEILLSQVTNMRTELSFNIQALAVCANICLATKDRQNPSLVWLFTGMFCIYSLFFAWRANLDISKPLFMGVVERFWMQSNAVVAVLAGIGLAAVVSETNRVLNSNGLQCLEWLSATLFVVYQIYSNYSVCDQRTNYVIDKFAKNLLTSMPHDAIILLRGDLPGNSLRYMHYCEGLRPDISLVDQEMMTYEWYLPKMAKHLPGVNFPGNRWNPVEGILPSGMVTFNLYHFLEVNKQKETFVCIGIHEGDPTWKKNYSLWPWGSCDKLVPLEIVFNPEEWIKLTKSIYNWTEEYGRFDPSSWESVANEEMWQARMKTPFFIFNLAETAHMPSKVKAQLYAQAYDLYKEIVYLQKEHPVNWHKNYAIACERMLRLQARDADPEVLLSETIRHFRLYSQKAPNDPQQADILGALKHLRKELQSLRNRKNVDYKDHDGDYKDHDIDYKDDDDK
>9RQN_E1 len=729 copies=1 longest=1
MSPHGDGRGQAQGRAVRVGLRRSGGIRGGVAVFAAVAAVFTFTLPPSVPGGDSGELITAAHELGVAHPPGYPLFTLVAKLAITLFPFGSIAYRVNLLCGLFGAVAASLLFFTVFRLSGSSAGGILAAGVFSFSRLTWQWSIAAEVFSLNNLFVGLLMALTVHFEEAATAKERSKVAKIGAFCCGLSLCNQHTIILYVLCIIPWILFQLLKKKELSLGSLLKLSLYFSAGLLPYVHLPISSYLNHARWTWGDQTTLQGFLTHFLREEYGTFSLAKSEIGSSMSEILLSQVTNMRTELSFNIQALAVCANICLATKDRQNPSLVWLFTGMFCIYSLFFAWRANLDISKPLFMGVVERFWMQSNAVVAVLAGIGLAAVVSETNRVLNSNGLQCLEWLSATLFVVYQIYSNYSVCDQRTNYVIDKFAKNLLTSMPHDAIILLRGDLPGNSLRYMHYCEGLRPDISLVDQEMMTYEWYLPKMAKHLPGVNFPGNRWNPVEGILPSGMVTFNLYHFLEVNKQKETFVCIGIHEGDPTWKKNYSLWPWGSCDKLVPLEIVFNPEEWIKLTKSIYNWTEEYGRFDPSSWESVANEEMWQARMKTPFFIFNLAETAHMPSKVKAQLYAQAYDLYKEIVYLQKEHPVNWHKNYAIACERMLRLQARDADPEVLLSETIRHFRLYSQKAPNDPQQADILGALKHLRKELQSLRNRKNVDYKDHDGDYKDHDIDYKDDDDK
>9RRS_E1 len=397 copies=1 longest=0
EQNKENQNPNSDAPVIRSKTSARYMELVGWLVDKGITSEKQWIQEDQASYISFNAASNSRSQIKAALDNAGKIMSLTKTAPDYLVGQQPVEDISSNRIYKILELNGYDPQYAASVFLGWATKKFGKRNTIWLFGPATTGKTNIAEAIAHTVPFYGCVNWTNENFPFNDCVDKMVIWWEEGKMTAKVVESAKAILGGSKVRVDQKCKSSAQIDPTPVIVTSNTNMCAVIDGNSTTFEHQQPLQDRMFKFELTRRLDHDFGKVTKQEVKDFFRWAKDHVVEVEHEFYVKKGGAKKRPAPSDADISEPKRVRESVAQPSTSDAEASINYADRLARGHSLGSLVPRGSGGGGSGGGGSGGGGSMDYKDHDGDYKDHDIDYKDDDDKGSENLYFQSHHHHHH
>9RRS_E2 len=738 copies=1 longest=1
MAADGYLPDWLEDNLSEGIREWWALKPGAPKPKANQQKQDDGRGLVLPGYKYLGPFNGLDKGEPVNAADAAALEHDKAYDQQLQAGDNPYLRYNHADAEFQERLQEDTSFGGNLGRAVFQAKKRVLEPLGLVEEGAKTAPGKKRPVEPSPQRSPDSSTGIGKKGQQPARKRLNFGQTGDSESVPDPQPLGEPPAAPSGVGPNTMAAGGGAPMADNNEGADGVGSSSGNWHCDSTWLGDRVITTSTRTWALPTYNNHLYKQISNGTSGGATNDNTYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLSFKLFNIQVKEVTQNEGTKTIANNLTSTIQVFTDSEYQLPYVLGSAHQGCLPPFPADVFMIPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFQFTYTFEDVPFHSSYAHSQSLDRLMNPLIDQYLYYLSRTQTTGGTANTQTLGFSQGGPNTMANQAKNWLPGPCYRQQRVSTTTGQNNNSNFAWTAGTKYHLNGRNSLANPGIAMATHKDDEERFFPSNGILIFGKQNAARDNADYSDVMLTSEEEIKTTNPVATEEYGIVADNLQQQNTAPQIGTVNSQGALPGMVWQNRDVYLQGPIWAKIPHTDGNFHPSPLMGGFGLKHPPPQILIKNTPVPADPPTTFNQSKLNSFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYYKSTSVDFAVNTEGVYSEPRPIGTRYLTRNL
>9RWS_E1 len=551 copies=1 longest=1
MQPQPSATPDRKQDNKNKNYEKAGLLGFIGLLLVVAVYVAFTGNDVNWGALVFMFISYAIIFYIGSVTAGKKSDSASDMMVAGRSMPLWVAMFTMTATWVGGGYIAGTAETTFASGVVWAQAPWGYGLSLIIGGIFYARKMRRYEFTTMLDPLEVRFGKKVAGVLYLPALLGELFWSAAILVALGTTFGLILGLDFTTSIIISAIIAVAYTFVGGMWSVALTDVAQIIMIIIGMFLVVPFALSEVGGLGQAWGAYKEGMAGFTNLLPPLDGWNHPDWGNYYWNWWDYALLLIFGGIPWQVYFQRVLSAKNEKTAMWLSIAAGFLCIILAVPAVMIGVAGFSADWASYGIEGPGAASEILAYVVNYMSPYLVAIVALGAIAAAVMSSMDSSILSASSMAAWNIYRPLVKPKANGRDIKRTIRVSIILIGLAATIVALNIDSVYTLWYLCADLVYCMLFPQLTTALFDKKANTYGAIAGLSVSLFLRLGGGEPALGLPAFLPYPMIEDGVVLFPFRTLAMVSGLLTIIVVSRLTQKSCPPQPLRKLKTEMQDE
>9RY4_E1 len=280 copies=1 longest=1
MNDLMRDLAPISAKAWAEIETEARGTLTVTLAARKVVDFKGPLGWDASSVSLGRTEALAEEPKAAGSAAVVTVRKRAVQPLIELCVPFTLKRAELEAIARGASDADLDPVIEAARAIAIAEDRAVFHGFAAGGITGIGEASAEHALDLPADLADFPGVLVRALAVLRDRGVDGPYALVLGRTVYQQLMETTTPGGYPVLQHVRRLFEGPLIWAPGVDGAMLISQRGGDFELTVGRDFSIGYHDHDAQSVHLYLQESMTFRCLGPEAAVPLRGLSQAATKA
>9RZM_E1 len=280 copies=1 longest=1
MNDLMRDLAPISAKAWAEIETEARGTLTVTLAARKVVDFKGPLGWDASSVSLGRTEALAEEPKAAGSAAVVTVRKRAVQPLIELCVPFTLKRAELEAIARGASDADLDPVIEAARAIAIAEDRAVFHGFAAGGITGIGEASAEHALDLPADLADFPGVLVRALAVLRDRGVDGPYALVLGRTVYQQLMETTTPGGYPVLQHVRRLFEGPLIWAPGVDGAMLISQRGGDFELTVGRDFSIGYHDHDAQSVHLYLQESMTFRCLGPEAAVPLRGLSQAATKA
>9S15_E1 len=184 copies=1 longest=1
GLDNELSLVDGQDRTLTVQQWDTFLNGVFPLDRNRLTREWFHSGRAKYIVAGPGADEFEGTLELGYQIGFPWSLGVGINFSYTTPNILINNGNITAPPFGLNSVITPNLFPGVSISARLGNGPGIQEVATFSVRVSGAKGGVAVSNAHGTVTGAAGGVLLRPFARLIASTGDSVTTYGEPWNMN
>9S5J_E1 len=510 copies=1 longest=1
MKTTAAMLWEKLRDGSVESRAIEFAKTTLPYLMVDPMSGSRGVVEHDFQSAGALLVNNLAAKLARSLFPTGIPFFRSELTDAIRREADSRDTDITEVTAALARVDRKATQRLFQNASLAVLTQVIKLLIVTGNALLYRDSAAATVVAWSLRSYAVRRDATGRWMDIVLKQRYKSKDLDEEYKQDLMRAGRNLSGSGSVDLYTHVQRKKGTAMEYAELYHEIDGVRVGKEGRWPIHLCPYIVPTWNLAPGEHYGRGHVEDYIGDFAKLSLLSEKLGLYELESLEVLNLVDEAKGAVVDDYQDAEMGDYVPGGAEAVRAYERGDYNKMAAIQQSLQAVVVRLNQAFMYGANQRDAERVTAEEVRITAEEAENTLGGTYSLLAENLQSPLAYVCLSEVDDALLQGLITKQHKPAIETGLPALSRSAAVQSMLNASQVIAGLAPIAQLDPRISLPKMMDTIWAAFSVDTSQFYKSADELEAEAEQQRQQAAQAQAAQETLLEGASDMTNALAGV
>9S5J_E2 len=184 copies=1 longest=0
MLLLDAVNVILRKIGELPIPSMDETYPTMAIALPELEDQRIQLLTQGWWFNTWWRHKLTPDPTGRINLPKGTLAFYPDSPDLQWDGLGVRDANTGDDRIGKPVEGRLVLSREWDHIPEIAQRVIAHQAALAVYTHEIGPDETAQVIAQELQAYQNELSRMHTRSRPLNTQAKRSFSRWRRSLRT
>9S6R_E1 len=240 copies=1 longest=0
EISEVQLVESGGGLVQPGGSLRLSCAASGFNFSSSSIHWVRQAPGKGLEWVASISSSSGSTSYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARSQKMSYWWWYFYHYWPAIDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDKTHT
>9S6R_E2 len=215 copies=1 longest=0
SDIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQSSSSLITFGQGTKVEIKRTVAAPSVFIFPPSDSQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9S6R_E3 len=467 copies=1 longest=1
MEGEQSPQGEKSLQRKQFVRPPLDLWQDLKDGVRYVIFDCRANLIVMPLLILFESMLCKIIIKKVAYTEINYKAYMEQIEMIQLDGMLDYSQVSGGTGPLVYPAGHVLIYKMMYWLTEGMDHVERGQVFFRYLYLLTLALQMACYYLLHLPPWCVVLACLSKRLHSIYVLRLFNDCFTTLFMVVTVLGAIVASRCHQRPKLKKSLALVISATYSMAVSIKMNALLYFPAMMISLFILNDANVILTLLDLVAMIAWQVAVAVPFLRSFPQQYLHCAFNFGRKFMYQWSINWQMMDEEAFNDKRFHLALLISHLIALTTLFVTRYPRILPDLWSSLCHPLRKNAVLNANPAKTIPFVLIASNFIGVLFSRSLHYQFLSWYHWTLPILIFWSGMPFFVGPIWYVLHEWCWNSYPPNSQASTLLLALNTVLLLLLALTQLSGSVALAKSHLRTTSSMEKKLNVDALEVLFQ
>9SAS_E1 len=40 copies=2 longest=0
PKFGFNNYAEKLNGRAAMVGFLLILVIEYFTNQGVLAWLG
>9SAS_E2 len=295 copies=2 longest=1
IWKIRLQLMKPITWIPLIWGVVCGAASSGGYIWSVEDFLKALTCMLLSGPLMTGYTQTLNDFYDRDIDAINEPYRPIPSGAISVPQVVTQILILLVAGIGVAYGLDVWAQHDFPIMMVLTLGGAFVAYIYSAPPLKLKQNGWLGNYALGASYIALPWWAGHALFGTLNPTIMVLTLIYSLAGLGIAVVNDFKSVEGDRQLGLKSLPVMFGIGTAAWICVIMIDVFQAGIAGYLIYVHQQLYATIVLLLLIPQITFQDMYFLRNPLENDVKYQASAQPFLVFGMLATGLALGHAGI
>9SAU_E1 len=41 copies=2 longest=0
PKFGFNNYAEKLNGRAAMVGFLLILVIEYFTNQGVLAWLGL
>9SAU_E2 len=294 copies=2 longest=1
IWKIRLQLMKPITWIPLIWGVVCGAASSGGYIWSVEDFLKALTCMLLSGPLMTGYTQTLNDFYDRDIDAINEPYRPIPSGAISVPQVVTQILILLVAGIGVAYGLDVWAQHDFPIMMVLTLGGAFVAYIYSAPPLKLKQNGWLGNYALGASYIALPWWAGHALFGTLNPTIMVLTLIYSLAGLGIAVVNDFKSVEGDRQLGLKSLPVMFGIGTAAWICVIMIDVFQAGIAGYLIYVHQQLYATIVLLLLIPQITFQDMYFLRNPLENDVKYQASAQPFLVFGMLATGLALGHAG
>9SF5_E1 len=1284 copies=1 longest=1
MFVFLVLLPLVSSQCVMPLFNLITTTQSYTNSFTRGVYYPDKVFRSSVLHLTQDLFLPFFSNVTWFHAISGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVFIKVCEFQFCNDPFLDVYHKNNKSWMESESGVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPIIGRDFPQGFSALEPLVDLPIGINITRFQTLLALNRSYLTPGDSSSGWTAGAADYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNVTNLCPFHEVFNATRFASVYAWNRTRISNCVADYSVLYNFAPFFAFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDSKHSGNYDYWYRSLRKSKLKPFERDISTEIYQAGNKPCKGKGPNCYFPLESYGFRPTYGVGHQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTKSNKKFLPFQQFGRDIVDTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVSVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEYVNNSYECDIPIGAGICASYQTQTKSRGSASSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKYFGGFNFSQILPDPSKPSKRSFIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLFSTASALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFLTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQLELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGRSLEVLFQGPGHHHHHHHHGSAWSHPQFEKGGGSGGGSGGSAWSHPQFEK
>9SF5_E2 len=124 copies=1 longest=0
EVQLVESGGGLVQPGRSLRLSCTTSGFTFDDYAMSWVRQAPGKGLEWVGFIRSIPYGGTTEYAASVKGRFTISRDVSKSIAYLQMNSLKTEDTAVYYCTRYHYGPGRTDAFDIWGQGTMVTVSS
>9SF5_E3 len=106 copies=1 longest=0
SYELTQPPSVSVSPGQTASITCSGVKLGDKYASWYQQKSGQSPVLVIYHDTKRPSGIPERFSGSNSGNTATLTISGTQAMDEADYYCQAWDSNTAIFGTGTKVTVL
>9SF8_E1 len=562 copies=1 longest=1
IVKIGEIQGASHTSPLLKKAVTVEQVVVTYLDDSTHFYVQDLNGDGDLATSDGIRVFAKNAKVQVGDVLTISGEVEEFFGRGYEERKQTDLTITQIVAKAVTKTGTAQVPSPLVLGKDRIAPANIIDNDGLRVFDPEEDAIDYWESMEGMLVAVDDAKILGPMKNKEIYVLPGSSTRPLNNSGGVLLPANSYNTDVIPVLFKKGKQIIKAGDSYKGRLAGPVSYSYGNYKVFVDDSKNMPSLMDGHLKPEKTNLQKDLSKLSIASYNIENFSANPSSTKDEKVKRIAESFIHDLNAPDIIGLIEVQDNNGPTDDGTTDATQSAQRLIDAIKKLGGPTYRYVDIAPENNVDGGQPGGNIRTGFLYQPERVSLSDKPKGGARDALTWVNGELNLSVGRIDPTNAAWKDVRKSLAAEFIFQGRKVVVVANHLNSKRGDNALYGCVQPVTFKSEQRRHVLANMLAQFAKEGAKHQANIVMLGDFNDFEFTKTIQLIEEGDMVNLVSRHDISDRYSYFHQGNNQTLDNILVSRHLLDHYEFDMVHVNSPFMEAHGRASDHDPLLLQL
>9SP8_E1 len=114 copies=1 longest=0
QVQLVESGGGLVQAGGSLRLSCAASGFPVNMYWMHWYRQAPGKEREWVAAIQSYGQWTAYADSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAVGVGGYYLGQGTQVTVS
>9SP8_E2 len=600 copies=1 longest=1
MNSLVRRKQVDSVHLIKNDGPHQLAKKLSAVDLVAIGVGTTIGAGVYILVGTVAREHTGPALAVSFFIAGVAAALSACCYAELASRCPSAGSAYHYAYICLGEGIAWLVGWALVLDYTIGGSAIARGITPNLASFFGGLDNLPVFLARQTIPGVGIVVDPCAALLIMIVTILLCFGIKESSTVQAIVTSVNVCTLVFIIVVGGYLACKTGWVGYDLPSGYFPFGLNGILAGSAVVFFSYIGFDTVTSTAEEVKNPQRDLPLGIGIALLICCILYMLLSVVIVGLVPYYSLNPDTPISSAFGDSGMQWAAYILTTGAITALCASLLGSLLAQPRIFMAMARDGLLPAFFSEISPRTQVPVKSTIAIGVLAAALAFFMDVAQLSEMVSVGTLMAFTAVAVCVLVLRYVPPDGVPLSSSSQTLSDTDESRAETENFLVDAIESSDSPLLGNETARDEKYFGKRRKIAAWSIALVCIGVLGLASAASAERLPSFPRFTICGVSAVILLGSLITLGYIDEDEERHNFGHKGGFLCPFVPYLPVLCILINTYLIINIGAGTWIRVLIWLLIGSMIYIFYGRSHSLLNNAVYVPTMTCTRKTTDHLA
>9SR9_E1 len=392 copies=1 longest=1
MVGIQVQEVMTDRYAKIDINAPLSEAIGIIEKEDPDLILVFDDNVYKGVLTQDLIIRSHLKWDPTKAKVRDVYKPAPVVKPTDDLSHAAKLLLETDLRSLPVGENKAEILGVISDMALLERVVAEEFGKRKVEEFMTKDVITLGPDDTVAKALATMRDHGISRIPVVDEEGKLEGLVTLHDLIIRFIKPRFKAQYGELAGEKIPPFSMKLREAMIKGVITIMPEATIREAVSTMKDNNIDGLVVVDENNKVVGILTVKDLLLPISRMVEKEARFYLQLGGDASALSEFTRERIINDIKRFVDGYADLLGNEGIIYLYIRRFNEKFRGVHLYQARMRVVTDRGVFIARGETWGAIQAVHDAIRAIERQLLQKAELERDIRYAKRFIEKLELWR
>9SWM_E1 len=379 copies=2 longest=1
MPHSSLHPSIPCPRGHGAQKAALVLLSACLVTLWGLGEPPEHTLRYLVLHLASLQLGLLLNGVCSLAEELRHIHSRYRGSYWRTVRACLGCPLRRGALLLLSIYFYYSLPNAVGPPFTWMLALLGLSQALNILLGLKGLAPAEISAVCEKGNFNVAHGLAWSYYIGYLRLILPELQARIRTYNQHYNNLLRGAVSQRLYILLPLDCGVPDNLSMADPNIRFLDKLPQQTGDHAGIKDRVYSNSIYELLENGQRAGTCVLEYATPLQTLFAMSQYSQAGFSREDRLEQAKLFCRTLEDILADAPESQNNCRLIAYQEPADDSSFSLSQEVLRHLRQEEKEEVTVGSLKTSAVPSTSTMSQEPELLISGMEKPLPLRTDFS
>9SWQ_E1 len=379 copies=2 longest=1
MPHSSLHPSIPCPRGHGAQKAALVLLSACLVTLWGLGEPPEHTLRYLVLHLASLQLGLLLNGVCSLAEELRHIHSRYRGSYWRTVRACLGCPLRRGALLLLSIYFYYSLPNAVGPPFTWMLALLGLSQALNILLGLKGLAPAEISAVCEKGNFNVAHGLAWSYYHGYLRLILPELQARIRTYNQHYNNLLRGAVSQRLYILLPLDCGVPDNLSMADPNIRFLDKLPQQTGDHAGIKDRVYSNSIYELLENGQRAGTCVLEYATPLQTLFAMSQYSQAGFSREDRLEQAKLFCRTLEDILADAPESQNNCRLIAYQEPADDSSFSLSQEVLRHLRQEEKEEVTVGSLKTSAVPSTSTMSQEPELLISGMEKPLPLRTDFS
>9SZB_E1 len=314 copies=1 longest=1
MGIFKKREAAQPVPQTVNTINQVTIKDNTIVAAIDFNVRNEVAESVSSYLGAYSLSSDRLSKITNNTSFLHRLVKHVLKHEQTFIYNSETYGWVMTDTVVLNQTRVTMTIQLPNPYNSSIYLSVPLKDVGVIDTTMMNVDTEAANKMLEAAYEAVIKKLNNTGAIKAFISSNVDVGLNKMAEDANDKIKTMLKTASELAGYTFLNKGDEVTQMMPDYTTSNTADFAAMRAFAASQLSVSEKILDGSATDGEKVAVMFRFLDPILQQFKEYEPTLNYSIRDEMFVAFMTTGGLLNSNKIEYWGKEKTINDRASQV
>9SZB_E2 len=96 copies=1 longest=0
MVTILEQAARQARVSTSILESVIDNSIFRLKTAGVIVKADDEYLPNELLPIVVQFIAIDARITLLGDELWAPSVANMQQKRQGLINTIELLSKGNE
>9T3R_E1 len=216 copies=1 longest=0
QSVLTQPASVSGSPGQSITISCTGTSSDVGSYNLVSWYQQHPGKAPKLMIYTVSNRPSGVSNRFSGSKSGNTASLTISGLQAEDEADYYCSSYTSSSSLVFGGGTKLTVLGQPKAAPSVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADSSPVKAGVETTTPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTVAPTECS
>9T3R_E2 len=267 copies=1 longest=0
QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLEWIGWINAGNGNTKYSQKFQGRVTITRDTSASTAYMELSSLRSEDTAVYYCARDFSSQVATAAVDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDKTSGQAGQHHHHHHGAEQKLISEEDLGGLNDIFEAQKIEWHE
>9T3R_E3 len=636 copies=1 longest=1
TQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPINCTHSCVDLDDKGCPAEQRASPLTHHHHHH
>9T3S_E1 len=267 copies=1 longest=0
QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLEWIGWINAGNGNTKYSQKFQGRVTITRDTSASTAYMELSSLRSEDTAVYYCARDFSSQVATAAVDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDKTSGQAGQHHHHHHGAEQKLISEEDLGGLNDIFEAQKIEWHE
>9T3S_E2 len=216 copies=1 longest=0
QSVLTQPASVSGSPGQSITISCTGTSSDVGSYNLVSWYQQHPGKAPKLMIYEVSNRPSGVSNRFSGSKSGNTASLTISGLQAEDEADYYCSSYTSSSTLVFGGGTKLTVLGQPKAAPSVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADSSPVKAGVETTTPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTVAPTECS
>9T3S_E3 len=636 copies=1 longest=1
TQVCTGTDMKLRLPASPETHLDMLRHLYQGCQVVQGNLELTYLPTNASLSFLQDIQEVQGYVLIAHNQVRQVPLQRLRIVRGTQLFEDNYALAVLDNGDPLNNTTPVTGASPGGLRELQLRSLTEILKGGVLIQRNPQLCYQDTILWKDIFHKNNQLALTLIDTNRSRACHPCSPMCKGSRCWGESSEDCQSLTRTVCAGGCARCKGPLPTDCCHEQCAAGCTGPKHSDCLACLHFNHSGICELHCPALVTYNTDTFESMPNPEGRYTFGASCVTACPYNYLSTDVGSCTLVCPLHNQEVTAEDGTQRCEKCSKPCARVCYGLGMEHLREVRAVTSANIQEFAGCKKIFGSLAFLPESFDGDPASNTAPLQPEQLQVFETLEEITGYLYISAWPDSLPDLSVFQNLQVIRGRILHNGAYSLTLQGLGISWLGLRSLRELGSGLALIHHNTHLCFVHTVPWDQLFRNPHQALLHTANRPEDECVGEGLACHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPINCTHSCVDLDDKGCPAEQRASPLTHHHHHH
>9T5J_E1 len=614 copies=1 longest=1
GPMADFAGPSSAGRKAGAPRCSRKAAGTKQTSTLKQEDASKRKAELEAAVRKKIEFERKALHIVEQLLEENITEEFLMECGRFITPAHYSDVVDERSIVKLCGYPLCQKKLGIVPKQKYKISTKTNKVYDITERKSFCSNFCYQASKFFEAQIPKTPVWVREEERHPDFQLLKEEQSGHSGEEVQLCSKAIKTSDIDNPSHFEKQYESSSSSTHSDSSSDNEQDFVSSILPGNRPNSTNIRPQLHQKSIMKKKAGHKANSKHKDKEQTVVDVTEQLGDCKLDSQEKDATCELPLQKVNTQSSSNSTLPERLKASENSESEYSRSEITLVGISKKSAEHFKRKFAKSNQVSRSVSSSVQVCPEVGKRNLLKVLKETLIEWKTEETLRFLYGQNYASVCLKPEASLVKEELDEDDIISDPDSHFPAWRESQNSLDESLPFRGSGTAIKPLPSYENLKKETEKLNLRIREFYRGRYVLGEETTKSQDSEEHDSTFPLIDSSSQNQIRKRIVLEKLSKVLPGLLVPLQITLGDIYTQLKNLVRTFRLTNRNIIHKPAEWTLIAMVLLSLLTPILGIQKHSQEGMVFTRFLDTLLEELHLKNEDLESLTIIFRTSCLPE
>9T5J_E2 len=376 copies=1 longest=0
GPMAASAAAAELQASGGPRHPVCLLVLGMAGSGKTTFVQRLTGHLHAQGTPPYVINLDPAVHEVPFPANIDIRDTVKYKEVMKQYGLGPNGGIVTSLNLFATRFDQVMKFIEKAQNMSKYVLIDTPGQIEVFTWSASGTIITEALASSFPTVVIYVMDTSRSTNPVTFMSNMLYACSILYKTKLPFIVVMNKTDIIDHSFAVEWMQDFEAFQDALNQETTYVSNLTRSMSLVLDEFYSSLRVVGVSAVLGTGLDELFVQVTSAAEEYEREYRPEYERLKKSLANAESQQQREQLERLRKDMGSVALDAGTAKDSLSPVLHPSDLILTRGTLDEEDEEADSDTDDIDHRVTEESHEEPAFQNFMQESMAQYWKRNNK
>9T5J_E3 len=284 copies=1 longest=0
MPRYAQLVMGPAGSGKSTYCATMVQHCEALNRSVQVVNLDPAAEHFNYSVMADIRELIEVDDVMEDDSLRFGPNGGLVFCMEYFANNFDWLENCLGHVEDDYILFDCPGQIELYTHLPVMKQLVQQLEQWEFRVCGVFLVDSQFMVESFKFISGILAALSAMISLEIPQVNIMTKMDLLSKKAKKEIEKFLDPDMYSLLEDSTSDLRSKKFKKLTKAICGLIDDYSMVRFLPYDQSDEESMNIVLQHIDFAIQYGEDLEFKEPKEREDESSSMFDEYFQECQDE
>9T5K_E1 len=453 copies=1 longest=1
MHHHHHHGKPIPNPLLGLDSTENLYFQGIDPFTMFSTLKNETEKFREYQANLGRQGKPLASTATLTTKIIVYNPSNKSPALRWEITKFAMRLIWSPAASHSVKVGAALTLLSAHAENPGAMIRSLVNDPDIEVVITDISEFDHGVPRLERRGEKAEQQMDSYRRILDRAPQENLFYNPEVDDLEILDSGTFLFAIATVLAQVWILVAKAVTAPDTAEESENKRWAKYVQQKRVNPDYLVSNRWITAMRSLISIDLSVRKYMVEILIEVKKSGVARGRLNEMIADIGNYIEETGMAGFFLTIKYGLEMKFPVIVINEFQADLLTLQTLMRTYMDLGPRAPYMVLLEDSIQTKFAPGNYPLLWSFAMGVGTTLDRSMGALNINRSYLEPIYFKLGQNAARKNAGSIDRKLAEELGLTQEQANEIKEMMQEVTTQRHETNVQAREGKFNVAAGGIE
>9T9M_E1 len=218 copies=1 longest=0
DIQMTQSPSSLSASVGDRVTITCRSSQSLVHSNGIPYLHWYQQKPGKAPKLLIYRVSNRFSGVPSRFSGSGSGTDFTLTISSLQPEDFATYYCSQGTHVPPTFGQGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGE
>9T9M_E2 len=123 copies=1 longest=0
GSQVQLQESGGGLVQPGGSLRLSCAASGRTISRYAMSWFRQAPGKEREFVAVARRSGDGAFYADSVQGRFTVSRDDAKNTVYLQMNSLKPEDTAVYYCAIDSDTFYSGSYDYWGQGTQVTVSS
>9T9M_E3 len=220 copies=1 longest=1
EVQLVESGGGLVQPGGSLRLSCAASGYSFTDFYINWVRQAPGKGLEWVARIFPGGDNTYYNEKFKGRFTLSADTSKSTAYLQMNSLRAEDTAVYYCARRGLYYAMDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKS
>9THK_E1 len=293 copies=2 longest=1
MNPASDGGTSESIFDLDYASWGIRSTLMVAGFVFYLGVFVVCHQLSSSLNATYRSLVAREKVFWDLAATRAVFGVQSTAAGLWALLGDPVLHADKARGQQNWCWFHITTATGFFCFENVAVHLSNLIFRTFDLFLVIHHLFAFLGFLGCLVNLQAGHYLAMTTLLLEMSTPFTCVSWMLLKAGWSESLFWKLNQWLMIHMFHCRMVLTYHMWWVCFWHWDGLVSSLYLPHLTLFLVGLALLTLIINPYWTHKKTQQLLNPVDWNFAQPEAKSRPEGNGQLLRKKRPDAAIEGR
>9TIU_E1 len=136 copies=5 longest=1
MSIIKEFREFAMRGNVVDLAVGVIIGAAFGKIVSSLVADIIMPPLGLLIGGIDFKQFAVTLRDAQGDIPAVVMHYGVFIQNVFDFLIVAFAIFMAIKLINKLNRKKEEPAAAPAPTKEEVLLTEIRDLLKEQNNRS
>9TIV_E1 len=136 copies=5 longest=1
MSIIKEFREFAMRGNVVDLAVSVIIGAAFGKIVSSLVADIIMPPLGLLIGGIDFKQFAVTLRDAQGDIPAVVMHYGVFIQNVFDFLIVAFAIFMAIKLINKLNRKKEEPAAAPAPTKEEVLLTEIRDLLKEQNNRS
>9TK9_E1 len=293 copies=1 longest=1
MNPASDGGTSESIFDLDYASWGIRSTLMVAGFVFYLGVFVVCHQLSSSLNATYRSLVAREKVFWDLAATRAVFGVQSTAAGLWALLGDPVLHADKARGQQNWCWFHITTATGFFCFENVAVHLSNLIFRTFDLFLVIHHLFAFLGFLGCLVNLQAGHYLAMTTLLLEMSTPFTCVSWMLLKAGWSESLFWKLNQWLMIHMFHCRMVLTYHMWWVCFWHWDGLVSSLYLPHLTLFLVGLALLTLIINPYWTHKKTQQLLNPVDWNFAQPEAKSRPEGNGQLLRKKRPDAAIEGR
>9TKA_E1 len=293 copies=1 longest=1
MNPASDGGTSESIFDLDYASWGIRSTLMVAGFVFYLGVFVVCHQLSSSLNATYRSLVAREKVFWDLAATRAVFGVQSTAAGLWALLGDPVLHADKARGQQNWCWFHITTATGFFCFENVAVHLSNLIFRTFDLFLVIHHLFAFLGFLGCLVNLQAGHYLAMTTLLLEMSTPFTCVSWMLLKAGWSESLFWKLNQWLMIHMFHCRMVLTYHMWWVCFWHWDGLVSSLYLPHLTLFLVGLALLTLIINPYWTHKKTQQLLNPVDWNFAQPEAKSRPEGNGQLLRKKRPDAAIEGR
>9TKM_E1 len=278 copies=1 longest=1
GPVEESVERAMVRVADTVSSRPTNSESIPALTAAETGHTSQVVPSDTMQTRHVKNYHSRSESSIENFLCRAACVYYATYTNNSEKGYAEWVINTRQVAQLRRKLELFTYLRFDLELTFVITSAQQPSTATSVDAPVQTHQIMYVPPGGPVPTKVTDYAWQTSTNPSVFWTEGNAPPRMSIPFISIGNAYSCFYDGWTQFSRNGVYGINTLNNMGTLYMRHVNEAGQGPIKSTVRIYFKPKHVKAWVPRPPRLCQYEKQKNVNFTPTGVTTTRVGITTT
>9TKM_E2 len=263 copies=1 longest=0
SPSAEECGFSDRVRSITLGNSTITTQECANVVVGYGVWPEYLKDNEATAEDQPTQPDVATCRFYTLESVQWMKNSAGWWWKLPDALSQMGLFGQNMQYHYLGRTGYTIHVQCNASKFHQGCLLVVCVPEAEMGCSNLNNTPEFAELSGGDTARMFTDTQIGETNSKKVQTAVWNAGMGVGVGNLTIYPHQWINLRTNNSATIVMPYINSVPMDNMFRHNNLTLMIIPFVPLNYSEGSSPYVPITVTIAPMCAEYNGLRLASSQ
>9TKM_E3 len=238 copies=1 longest=0
GLPVMTTPGSTQFLTSDDFQSPSAMPQFDVTPEMQIPGRVNNLMEIAEVDSVVPVNNTEANVNSLKAYQIPVQSNSDNGKQVFGFPLQPGANGVLNRTLLGEILNYYTHWSGSIKLTFMFCGSAMATGKFLLAYSPPGAGVPKNRKDAMLGTHVIWDVGLQSSCVLCVPWISQTHYRYVVEDEYTAAGYITCWYQTNIVVPADVQSSCDILCFVSACNDFSVRMLKDTPFIRQDTFYQ
>9TUZ_E1 len=1109 copies=1 longest=1
MKLCILLAVVAFVGLSLGGTDLIEGIFYEKNEIDKLTFSLDHRVRDNLKTDLILNNNGENDYAYLNKYVYTILNRDSTEKIKTFFSHNKDMKSCDYFISKEYQSSDKTNQICYKKTFCGVVIPNSEEIKTNKITNDKLYCAHFQSTHIIIYYISQPLLLEPHVVYEETFFEKGKNDQINCQGMYISLRSVHVHTHNAILQQETLTYIKNLCDGKNNCKFDFDSIKYEQKSLTHYLFFINIQYQCISPLNLQENEMCDVYNDDTHKATCKYGFNKIELLKNVCEENYRCTQDICSVNQFCDGEQETCTCKTSLLPSAKNNCEYNDLCTVLNCPEQSTCEQIGNGKKAECKCENGKYYHNNKCYTKNDLELAIKIEPHKKEKFYKNNLYQGKALKPEYIFMQCENGFSIEVINAYVSCYRVSFNLNKLKYVTESLKKMCDGKTKCAYGNTIDPIDDLNHHNICNNFNTIFKYDYLCVFNNQQITSDKNSHLHSNIPSLYQSSILPDIQKSKFHLISRNSRTNQYPHNQISMLEIQNEISSHNSNQFSTDPHTNSNNINNMNIKKVEIFRSRFSSKLQCQGGKINIDKAILKGGEGCNDLLLTNSLKSYCNDLSECDIGLIYHFDTYCINDQYLFVSYSCSNLCNKCHQQSTCYGNRFNYDCFCDNPYISKYGNKLCERPNDCESVLCSQNQVCQILPNDKLICQCEEGYKNVKGKCVPDNKCDLSCPSNKVCVIENGKQTCKCSERFVLENGVCICANDYKMEDGINCIAKNKCKRKEYENICTNPNEMCAYNEETDIVKCECKECYYRSSRGECILNDYCKDINCKENEECSIVNCKPECVCKENLKKNNKGECIYENSCLINEGNCPKDSKCIYREYKPHECVCNKQGHVAVNGKCVLEDKCVHNKKCSENSICVNVMNKEPICVCTYNYYKKDGVCLIQNPCLKDNGGCSRNSECTFKYSKIQCTCKENYKNKDDSCVPNTNEYDESFTFQYNDDASIILGACGMIEFSYIYNQIIWKIQNSKESYVFYYDYPTAGNIEVQIKNEIFHTIIYLKKKIGNSVIYDDFQVDHQTCIYENVFYYSNQNGSASGLNDIFEAQKIEWHEEPEA
>9TUZ_E2 len=256 copies=1 longest=0
MGWSCIILFLVATATGVHSEVQLVEFGGGLVQPGGSLRLSCEGSGFIFSQYWMSWIRQAPGKGLEWVGNIKQDGTEKNYVDSVKGRFTISRDNTNNFLYLQMDSLRAEDTGVYFCARNTFYFDSSGYPLGQGALVAVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDKTHTCGKHHHHHH
>9TUZ_E3 len=233 copies=1 longest=0
MGWSCIILFLVATATGVHSAIQMTQSPSTLSASVGDSVTISCRASQSISSWLAWYQQKPGKAPNLLIYKASILESGVPSRFSGSGSGTEFTLTISSLQPDDFATYYCQKYDSYPYSFGPGTKLDIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9TZW_E1 len=142 copies=4 longest=1
SDTPDVFISYRRNSGSQLASLLKVHLQLHGFSVFIDVEKLEAGKFEDKLIQSVMGARNFVLVLSPGALDKCMQDHDCKDWVHKEIVTALSCGKNIVPIIDGFEWPEPQVLPEDMQAVLTFNGIKWSHEYQEATIEKIIRFLQ
>9TZY_E1 len=142 copies=4 longest=1
SDTPDVFISYRRNSGSQLASLLKVHLQLHGFSVFIDVEKLEAGKFEDKLIQSVMGARNFVLVLSPGALDKCMQDHDCKDWVHKEIVTALSCGKNIVPIIDGFEWPEPQVLPEDMQAVLTFNGIKWSHEYQEATIEKIIRFLQ
>9U4J_E1 len=288 copies=1 longest=1
DHLFPPSLYIFVIGVGLPTNCLALWAAYRQVQQRNELGVYLMNLSIADLLYICTLPLWVDYFLHHDNWIHGPGSCKLFGFIFYTNIYISIAFLCCISVDRYLAVAHPLRFARLRRVKTAVAVSSVVWATELGANSAPLFHDELFRDRYNHTFCFEKFPMEGWVAWMNLYRVFVGFLFPWALMLLSYRGILRAVRGSVSTERQEKAKIKRLALSLIAIVLVCFAPYHVLLLSRSAIYLGRPWDCGFEERVFSAYHSSLAFTSLNCVADPILYCLVNEGARRQLADLEDN
>9U7H_E1 len=613 copies=1 longest=1
APPGADAYFPEERWSPESPLQAPRVLIALLARNAAHALPTTLGALERLRHPRERTALWVATDHNMDNTSTVLREWLVAVKSLYHSVEWRPAEEPRSYPDEEGPKHWSDSRYEHVMKLRQAALKSARDMWADYILFVDADNLILNPDTLSLLIAENKTVVAPMLDSRAAYSNFWCGMTSQGYYKRTPAYIPIRKRDRRGCFAVPMVHSTFLIDLRKAASRNLAFYPPHPDYTWSFDDIIVFAFSCKQAEVQMYVCNKEEYGFLPVPLRAHSTLQDEAESFMHVQLEVMVKHPPAEPSRFISAPTKTPDKMGFDEVFMINLRRRQDRRERMLRALQAQEIECRLVEAVDGKAMNTSQVEALGIQMLPGYRDPYHGRPLTKGELGCFLSHYNIWKEVVDRGLQKSLVFEDDLRFEIFFKRRLMNLMRDVEREGLDWDLIYVGRKRMQVEHPEKAVPRVRNLVEADYSYWTLAYVISLQGARKLLAAEPLSKMLPVDEFLPVMFDKHPVSEYKAHFSLRNLHAFSVEPLLIYPTHYTGDDGYVSDTETSVVWNNEHVKTDWDRAKSQKMREQQALSREAKNSDVLQSPLDSAARGTLEVLFQGPLGSHHHHHHHHHH
>9U7H_E2 len=9 copies=1 longest=0
SGAKGEKGS
>9UBC_E1 len=196 copies=1 longest=1
MSPQDDLIVWLRGDRAGLGPFSADLVDQYWRWEQDPSVLIGYGRQTPDSLENRREGYGHQARGTDDQLRFTVYDLTGQDPVPVGTTAVLIDHHVRTGEFVIQLGAGHRGRGLGTEATRLTLDYAFHVSALACVHLAVLTPNTGAIAAYERAGFRRIGERRDSGFWLGRRVSETLMDAVPEDFPGPSVVRGFVEGGR
>9UBC_E2 len=15 copies=1 longest=0
GQGKGSAEDKRRAYN
>9UDC_E1 len=729 copies=1 longest=1
MNPFQKNESKETLFSPVSIEEVPPRPPSPPKKPSPTICGSNYPLSIAFIVVNEFCERFSYYGMKAVLILYFLYFLHWNEDTSTSIYHAFSSLCYFTPILGAAIADSWLGKFKTIIYLSLVYVLGHVIKSLGALPILGGQVVHTVLSLIGLSLIALGTGGIKPCVAAFGGDQFEEKHAEERTRYFSVFYLSINAGSLISTFITPMLRGDVQCFGEDCYALAFGVPGLLMVIALVVFAMGSKIYNKPPPEGNIVAQVFKCIWFAISNRFKNRSGDIPKRQHWLDWAAEKYPKQLIMDVKALTRVLFLYIPLPMFWALLDQQGSRWTLQAIRMNRNLGFFVLQPDQMQVLNPLLVLIFIPLFDFVIYRLVSKCGINFSSLRKMAVGMILACLAFAVAAAVEIKINEMAPAQPGPQEVFLQVLNLADDEVKVTVVGNENNSLLIESIKSFQKTPHYSKLHLKTKSQDFHFHLKYHNLSLYTEHSVQEKNWYSLVIREDGNSISSMMVKDTESRTTNGMTTVRFVNTLHKDVNISLSTDTSLNVGEDYGVSAYRTVQRGEYPAVHCRTEDKNFSLNLGLLDFGAAYLFVITNNTNQGLQAWKIEDIPANKMSIAWQLPQYALVTAGEVMFSVTGLEFSYSQAPSSMKSVLQAAWLLTIAVGNIIVLVVAQFSGLVQWAEFILFSCLLLVICLIFSIMGYYYVPVKTEDMRGPADKHIPHIQGNMIKLETKKTKL
>9UGL_E1 len=401 copies=1 longest=1
MLTIIALSYIFCLVFADYKDDDDGAPTSDGGRFKFPDGVQNWPALSIVIIIIMTIGGNILVIMAVSMEKKLHNATNYFLMSLAIADMLVGLLVMPLSLLAILYDYVWPLPRYLCPVWISLDVLFSTASIMHLCAISLDRYVAIRNPIEHSRFNSRTKAIMKIAIVWAISIGVSVPIPVIGLRDEEKVFVNNTTCVLNDPNFVLIGSFVAFFIPLTIMVITYCLTIYVLRRQALMLLHGHTEEPPGLSLDFLKCCKRNTAEEENSANPNQDQNARRRKKKERRPRGTMQAINNERKASKVLGIVFFVFLIMWCPFFITNILSVLCEKSCNQKLMEKLLNVFVWIGYVCSGINPLVYTLFNKIYRRAFSNYLRCNYKVEKKPGRPLEVLFQGPHHHHHHHHHH
>9UGU_E1 len=429 copies=1 longest=1
MAAQGYGYYRTVIFSAMFGGYSLYYFNRKTFSFVMPSLVEEIPLDKDDLGFITSSQSAAYAISKFVSGVLSDQMSARWLFSSGLLLVGLVNIFFAWSSTVPVFAALWFLNGLAQGLGWPPCGKVLRKWFEPSQFGTWWAILSTSMNLAGGLGPILATILAQSYSWRSTLALSGALCVVVSFLCLLLIHNEPADVGLRNLDPMPSEGKKGSLKEESTLQELLLSPYLWVLSTGYLVVFGVKTCCTDWGQFFLIQEKGQSALVGSSYMSALEVGGLVGSIAAGYLSDRAMAKAGLSNYGNPRHGLLLFMMAGMTVSMYLFRVTVTSDSPKLWILVLGAVFGFSSYGPIALFGVIANESAPPNLCGTSHAIVGLMANVGGFLAGLPFSTIAKHYSWSTAFWVAEVICAASTAAFFLLRNIRTKMGRVSKKAE
>9UGX_E1 len=429 copies=1 longest=1
MAAQGYGYYRTVIFSAMFGGYSLYYFNRKTFSFVMPSLVEEIPLDKDDLGFITSSQSAAYAISKFVSGVLSDQMSARWLFSSGLLLVGLVNIFFAWSSTVPVFAALWFLNGLAQGLGWPPCGKVLRKWFEPSQFGTWWAILSTSMNLAGGLGPILATILAQSYSWRSTLALSGALCVVVSFLCLLLIHNEPADVGLRNLDPMPSEGKKGSLKEESTLQELLLSPYLWVLSTGYLVVFGVKTCCTDWGQFFLIQEKGQSALVGSSYMSALEVGGLVGSIAAGYLSDRAMAKAGLSNYGNPRHGLLLFMMAGMTVSMYLFRVTVTSDSPKLWILVLGAVFGFSSYGPIALFGVIANESAPPNLCGTSHAIVGLMANVGGFLAGLPFSTIAKHYSWSTAFWVAEVICAASTAAFFLLRNIRTKMGRVSKKAE
>9UGY_E1 len=429 copies=1 longest=1
MAAQGYGYYRTVIFSAMFGGYSLYYFNRKTFSFVMPSLVEEIPLDKDDLGFITSSQSAAYAISKFVSGVLSDQMSARWLFSSGLLLVGLVNIFFAWSSTVPVFAALWFLNGLAQGLGWPPCGKVLRKWFEPSQFGTWWAILSTSMNLAGGLGPILATILAQSYSWRSTLALSGALCVVVSFLCLLLIHNEPADVGLRNLDPMPSEGKKGSLKEESTLQELLLSPYLWVLSTGYLVVFGVKTCCTDWGQFFLIQEKGQSALVGSSYMSALEVGGLVGSIAAGYLSDRAMAKAGLSNYGNPRHGLLLFMMAGMTVSMYLFRVTVTSDSPKLWILVLGAVFGFSSYGPIALFGVIANESAPPNLCGTSHAIVGLMANVGGFLAGLPFSTIAKHYSWSTAFWVAEVICAASTAAFFLLRNIRTKMGRVSKKAE
>9UHG_E1 len=198 copies=1 longest=1
VTNLCPFHEVFNATRFASVYAWNRTRISNCVADYSVLYNFAPFFAFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDSKHSGNYDYWYRSFRKSKLKPFERDISTEIYQAGNKPCKGKGPNCYFPLQSYGFRPTYGVGHQPYRVVVLSFELLHAPATVCGPKKS
>9UHG_E2 len=119 copies=1 longest=0
EVQLVESGGGLIQPGGSVRLSCTVSGFTVSSNYMSWVRQSPGRGLEWVSLIYPGGSSYYADSVKGRFTISRDNSKNTLYLQMNSLRGEDTAVYYCVRDMVGARAGMEVWGQGTTVTVSS
>9UHG_E3 len=107 copies=1 longest=0
DIQMTQSPSSLSASVGDRVTFTCQASQDINIYLNWYQQKTGKAPKLLIYDASNLETGVPSRFSGSGSGTEFTFTISSLQPEDIATYYCLQYDNLPRAFGQGTKVEIK
>9UI1_E1 len=636 copies=1 longest=1
MKSRVILNHRDRRDKNHNNNNTNHYTQVDTMERKPLEVEPSTTTTNTDVVDSFEEEQRKIVYRGWKVMPFIIGNETFEKLGIIGTLSNLLVYLTSVFNLKSYTAATIINAFSGTINFGTFIAAFLCDTYFGRYKTLSVAVIACFLGSFVILLTAAIPSLHPVACGNKISCEGPSVGQILFLLMGLGFLVVGAGGIRPCNLAFGADQFNPKSESGKKGINSFFNWYFFTFTFAQIISLTAVVYIQSNVSWTIGLIIPVALMFLACVIFFAGDRLYVKVKASGSPLAGIARVIAAAIKKRGLKPVKQPWVNLYNHIPSNYANTTLKYTDQFRFLDKAAIMTPEEKLNSDGTASDPWKLCTLQQVEEVKCIVRVIPIWFASTIYYLAITIQMTYPVFQALQSDRRLGSGGFRIPAATYVVFLMTGMTVFIIFYDRVLVPSLRRVTGLETGISLLQRIGAGFTFAIMSLLVSGFIEERRRNFALTKPTLGMAPRTGEISSMSALWLIPQLTLAGIAEAFAAIGQMEFYYKQFPENMKSFAGSIFYVGAGVSSYLASFLISTVHRTTAHSPSGNWLAEDLNKAKLDYFYFMLTGLMVVNMAYFLLMARWYRYKGGNDEDITEIETNEEETKQQQLQDKNSV
>9UI6_E1 len=636 copies=1 longest=1
MKSRVILNHRDRRDKNHNNNNTNHYTQVDTMERKPLEVEPSTTTTNTDVVDSFEEEQRKIVYRGWKVMPFIIGNETFEKLGIIGTLSNLLVYLTSVFNLKSYTAATIINAFSGTINFGTFIAAFLCDTYFGRYKTLSVAVIACFLGSFVILLTAAIPSLHPVACGNKISCEGPSVGQILFLLMGLGFLVVGAGGIRPCNLAFGADQFNPKSESGKKGINSFFNWYFFTFTFAQIISLTAVVYIQSNVSWTIGLIIPVALMFLACVIFFAGDRLYVKVKASGSPLAGIARVIAAAIKKRGLKPVKQPWVNLYNHIPSNYANTTLKYTDQFRFLDKAAIMTPEEKLNSDGTASDPWKLCTLQQVEEVKCIVRVIPIWFASTIYYLAITIQMTYPVFQALQSDRRLGSGGFRIPAATYVVFLMTGMTVFIIFYDRVLVPSLRRVTGLETGISLLQRIGAGFTFAIMSLLVSGFIEERRRNFALTKPTLGMAPRTGEISSMSALWLIPQLTLAGIAEAFAAIGQMEFYYKQFPENMKSFAGSIFYVGAGVSSYLASFLISTVHRTTAHSPSGNWLAEDLNKAKLDYFYFMLTGLMVVNMAYFLLMARWYRYKGGNDEDITEIETNEEETKQQQLQDKNSV
>9UIF_E1 len=636 copies=1 longest=1
MKSRVILNHRDRRDKNHNNNNTNHYTQVDTMERKPLEVEPSTTTTNTDVVDSFEEEQRKIVYRGWKVMPFIIGNETFEKLGIIGTLSNLLVYLTSVFNLKSYTAATIINAFSGTINFGTFIAAFLCDTYFGRYKTLSVAVIACFLGSFVILLTAAIPSLHPVACGNKISCEGPSVGQILFLLMGLGFLVVGAGGIRPCNLAFGADQFNPKSESGKKGINSFFNWYFFTFTFAQIISLTAVVYIQSNVSWTIGLIIPVALMFLACVIFFAGDRLYVKVKASGSPLAGIARVIAAAIKKRGLKPVKQPWVNLYNHIPSNYANTTLKYTDQFRFLDKAAIMTPEEKLNSDGTASDPWKLCTLQQVEEVKCIVRVIPIWFASTIYYLAITIQMTYPVFQALQSDRRLGSGGFRIPAATYVVFLMTGMTVFIIFYDRVLVPSLRRVTGLETGISLLQRIGAGFTFAIMSLLVSGFIEERRRNFALTKPTLGMAPRTGEISSMSALWLIPQLTLAGIAEAFAAIGQMEFYYKQFPENMKSFAGSIFYVGAGVSSYLASFLISTVHRTTAHSPSGNWLAEDLNKAKLDYFYFMLTGLMVVNMAYFLLMARWYRYKGGNDEDITEIETNEEETKQQQLQDKNSV
>9UIT_E1 len=636 copies=1 longest=1
MKSRVILNHRDRRDKNHNNNNTNHYTQVDTMERKPLEVEPSTTTTNTDVVDSFEEEQRKIVYRGWKVMPFIIGNETFEKLGIIGTLSNLLVYLTSVFNLKSYTAATIINAFSGTINFGTFIAAFLCDTYFGRYKTLSVAVIACFLGSFVILLTAAIPSLHPVACGNKISCEGPSVGQILFLLMGLGFLVVGAGGIRPCNLAFGADQFNPKSESGKKGINSFFNWYFFTFTFAQIISLTAVVYIQSNVSWTIGLIIPVALMFLACVIFFAGDRLYVKVKASGSPLAGIARVIAAAIKKRGLKPVKQPWVNLYNHIPSNYANTTLKYTDQFRFLDKAAIMTPEEKLNSDGTASDPWKLCTLQQVEEVKCIVRVIPIWFASTIYYLAITIQMTYPVFQALQSDRRLGSGGFRIPAATYVVFLMTGMTVFIIFYDRVLVPSLRRVTGLETGISLLQRIGAGFTFAIMSLLVSGFIEERRRNFALTKPTLGMAPRTGEISSMSALWLIPQLTLAGIAEAFAAIGQMEFYYKQFPENMKSFAGSIFYVGAGVSSYLASFLISTVHRTTAHSPSGNWLAEDLNKAKLDYFYFMLTGLMVVNMAYFLLMARWYRYKGGNDEDITEIETNEEETKQQQLQDKNSV
>9UJM_E1 len=389 copies=1 longest=1
MKTIIALSYIFCLVFADYKDDDDAKLQTLSLLHLQEKNWSALLTAVVIILTIAGNILVIMAVSLEKKLQNATNYFLMSLAIADMLLGFLVMPVSMLTILYGYRWPLPSKLCAVWIYLDVLFSTASIMHLCAISLDRYVAIQNPIHHSRFNSRTKAFLKIIAVWTISVGISMPIPVFGLQDDSKVFKEGSCLLADDNFVLIGSFVSFFIPATIMVITYFLTIKSLQKEATLCVSDLGTRAKLASFSFLPQSSLSSEKLFQRSIHREPGSYTGRRTMQSISNEQKACKVLGIVFFLFVVMWCPFFITNIMAVICKESCNEDVIGALLNVFVWIGYASSAVNPLVYTLFNKTYRSAFSRYIQCQYKENKKPGRPLEVLFQGPHHHHHHHHHH
>9UPM_E1 len=452 copies=1 longest=1
MAQQWSLQRLAGRHPQDSYEDSTQSSIFTYTNSNSTRGPFEGPNYHIAPRWVYHLTSVWMIFVVTASVFTNGLVLAATMKFKKLRHPLNWILVNLAVADLAETVIASTISVVNNVSGYFVLGHPMCVLEGFTVSLCGITGLWSLAIISWERWLVVCKPFGNVRFDAKLAIVGIAFSWIWSAVWTAPPIFGWSRYWPHGLKTSCGPDVFSGSSYPGVQSYMIVLMVTCCIIPLAIIVLCYLQVWLAIRAVAADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAGSGSGHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTINAYIQKYGSTQKAEKEVTRMVMVMIFAYCVCWGPYTFFACFAAANPGYAFHPLMAALPAYFAKSATIYNPIIYVFMNRQFRNCILQLFGKKVDDGSELSSASKTEVSSVSSVSPALEVLFQ
>9UPN_E1 len=434 copies=1 longest=1
MAQQWSLQRLAGRHPQDSYEDSTQSSIFTYTNSNSTRGPFEGPNYHIAPRWVYHLTSVWMIFVVIASVFTNGLVLAATMKFKKLRHPLNWILVNLAVADLAETVIASTISVVNNVYGYFVLGHPMCVLEGYTVSLCGITGLWSLAIISWERWMVVCKPFGNVRFDAKLAIVGIVFSWVWAAVWTAPPIFGWSRYWPHGLKTSCGPDVFSGSSYPGVQSYMIVLMVTCCITPLTIIVLCYLQVWLAIRAVAADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAGSGSGHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTINAYIQKYGSTQKAEKEVTRMVVVMFLAFCFCWGPYAFFACFAAANPGYAFHPLMAALPAYFAKSATIYNPIIYVFMNRQFRNCILQLFGKKVDDGSELEVLFQ
>9UPO_E1 len=434 copies=1 longest=1
MAQQWSLQRLAGRHPQDSYEDSTQSSIFTYTNSNSTRGPFEGPNYHIAPRWVYHLTSVWMIFVVIASVFTNGLVLAATMKFKKLRHPLNWILVNLAVADLAETVIASTISVVNQVYGYFVLGHPMCVLEGYTVSLCGITGLWSLAIISWERWMVVCKPFGNVRFDAKLAIVGIVFSWVWAAVWTAPPIFGWSRYWPHGLKTSCGPDVFSGSSYPGVQSYMIVLMVTCCITPLTIIVLCYLQVWLAIRAVAADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAGSGSGHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTINAYIQKYGSTQKAEKEVTRMVVVMFLAFCFCWGPYAFFACFAAANPGYAFHPLMAALPAYFAKSATIYNPIIYVFMNRQFRNCILQLFGKKVDDGSELEVLFQ
>9US8_E1 len=553 copies=1 longest=1
MAFSELLDLVGGLGRFQVLQTMALMVSIMWLCTQSMLENFSAAVPSHRCWAPLLDNSTAQASILGSLSPEALLAISIPPGPNQRPHQCRRFRQPQWQLLDPNATATSWSEADTEPCVDGWVYDRSIFTSTIVAKWNLVCDSHALKPMAQSIYLAGILVGAAVCGPASDRFGRRLVLTWSYLQMAVSGTAAAFAPTFPVYCLFRFLLAFAVAGVMMNTGTLLMEWTSARARPLVMTLNSLGFSFGHGLTAAVAYGVRDWTLLQLAVSVPFFLCFLYSWWLAESARWLLTTGRLDRGLQELRRVAAINGKRAVQDTLTPEVLLSAMREELSVGQAPASLGTLLRTPGLRLRTCISTLCWFAFGFTFFGLALDLQALGSNIFLLQVLIGVVDIPAKMGALLLLSRLGRRPTLAASLLLAGLCILANTLVPHEMGALRSALAVLGLGGVGAAFTCITIYSSELFPTVLRMTAVGLGQMAARGGAILGPLVRLLGVHGPWLPLLVYGTVPVLSGLAALLLPETQSLPLPDTIQDVQNQAVKKATHGTLGNSVLKSTQF
>9UUR_E1 len=403 copies=1 longest=1
MPKKKPTPIQLNPAPDGSAVNGTSSAETNLEALQKKLEELELDEQQRKRLEAFLTQKQKVGELKDDDFEKISELGAGNGGVVFKVSHKPSGLVMARKLIHLEIKPAIRNQIIRELQVLHECNSPYIVGFYGAFYSDGEISICMEHMDGGSLDQVLKKAGRIPEQILGKVSIAVIKGLTYLREKHKIMHRDVKPSNILVNSRGEIKLCDFGVSGQLIDSMANSFVGTRSYMSPERLQGTHYSVQSDIWSMGLSLVEMAVGRYPIPPPDAKELELMFGCQVEGDAAETPPRPRTPGRPLSSYGMDSRPPMAIFELLDYIVNEPPPKLPSGVFSLEFQDFVNKCLIKNPAERADLKQLMVHAFIKRSDAEEVDFAGWLCSTIGLNQPSTPTHAAGVLEHHHHHHHH
>9UUR_E2 len=388 copies=1 longest=0
HMAAAAAQGGGGGEPRRTEGVGPGVPGEVEMVKGQPFDVGPRYTQLQYIGEGAYGMVSSAYDHVRKTRVAIKKISPFEHQTYCQRTLREIQILLRFRHENVIGIRDILRASTLEAMRDVYIVQDLMETDLYKLLKSQQLSNDHICYFLYQILRGLKYIHSANVLHRDLKPSNLLINTTCDLKICDFGLARIADPEHDHTGFLTEYVATRWYRAPEIMLNSKGYTKSIDIWSVGCILAEMLSNRPIFPGKHYLDQLNHILGILGSPSQEDLNCIINMKARNYLQSLPSKTKVAWAKLFPKSDSKALDLLDRMLTFNPNKRITVEEALAHPYLEQYYDPTDEPVAEEPFTFAMELDDLPKERLKELIFQETARFQPGVLEAPLEHHHHHH
>9UUX_E1 len=403 copies=1 longest=1
MPKKKPTPIQLNPAPDGSAVNGTSSAETNLEALQKKLEELELDEQQRKRLEAFLTQKQKVGELKDDDFEKISELGAGNGGVVFKVSHKPSGLVMARKLIHLEIKPAIRNQIIRELQVLHECNSPYIVGFYGAFYSDGEISICMEHMDGGSLDQVLKKAGRIPEQILGKVSIAVIKGLTYLREKHKIMHRDVKPSNILVNSRGEIKLCDFGVSGQLIDSMANSFVGTRSYMSPERLQGTHYSVQSDIWSMGLSLVEMAVGRYPIPPPDAKELELMFGCQVEGDAAETPPRPRTPGRPLSSYGMDSRPPMAIFELLDYIVNEPPPKLPSGVFSLEFQDFVNKCLIKNPAERADLKQLMVHAFIKRSDAEEVDFAGWLCSTIGLNQPSTPTHAAGVLEHHHHHHHH
>9UUX_E2 len=388 copies=1 longest=0
HMAAAAAQGGGGGEPRRTEGVGPGVPGEVEMVKGQPFDVGPRYTQLQYIGEGAYGMVSSAYDHVRKTRVAIKKISPFEHQTYCQRTLREIQILLRFRHENVIGIRDILRASTLEAMRDVYIVQDLMETDLYKLLKSQQLSNDHICYFLYQILRGLKYIHSANVLHRDLKPSNLLINTTCDLKICDFGLARIADPEHDHTGFLTEYVATRWYRAPEIMLNSKGYTKSIDIWSVGCILAEMLSNRPIFPGKHYLDQLNHILGILGSPSQEDLNCIINMKARNYLQSLPSKTKVAWAKLFPKSDSKALDLLDRMLTFNPNKRITVEEALAHPYLEQYYDPTDEPVAEEPFTFAMELDDLPKERLKELIFQETARFQPGVLEAPLEHHHHHH
>9UVU_E1 len=10 copies=1 longest=0
RRPPGASXXR
>9UVU_E2 len=487 copies=1 longest=1
GPLSVREDSVPTTASFSADMLNVTLQGPTLNGTFAQSKCPQVEWLGWLNTIQPPFLWVLFVLATLENIFVLSVFCLHKSSCTVAEIYLGNLAAADLILACGLPFWAITISNNFDWLFGETLCRVVNAIISMNLYSSICFLMLVSIDRYLALVKTMSMGRMRGVRWAKLYSLVIWGCTLLLSSPMLVFRTMKEYSDEGHNVTACVISYPSLIWEVFTNMLLNVVGFLLPLSVITFCTMQIMQVLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLKEIQTERRATVLVLVVLLLFIICWLPFQISTFLDTLHRLGILSSCQDERIIDVITQIASFMAYSNSCLNPLVYVIVGKRFRKKSWEVYQGVCQKGGCRSEPIQMENSMGTLRTSISVERQIHKLQDWAGSRQLEVLFQ
>9UVW_E1 len=486 copies=1 longest=1
MKTIIALSYIFCLVFADYKDDDDAENLYFQGMLMASTTSAVPGHPSLPSLPSNSSQERPLDTRDPLLARAELALLSIVFVAVALSNGLVLAALARRGRRGHWAPIHVFIGHLCLADLAVALFQVLPQLAWKATDRFRGPDALCRAVKYLQMVGMYASSYMILAMTLDRHRAICRPMLAYRHGSGAHWNRPVLVAWAFSLLLSLPQLFIFAQRNVEGGSGVTDCWACFAEPWGRRTYVTWIALMVFVAPTLGIAACQVLIFREIHASLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLAVAKTVRMTLVIVVVYVLCWAPFFLVQLWAAWDPEAPLEGAPFVLLMLLASLNSCTNPWIYASFSSSVSSELRSLLCCARGRTPPSLGPQDESCTTASSSLAKDTSSLEVLFQ
>9UVX_E1 len=486 copies=1 longest=1
MKTIIALSYIFCLVFADYKDDDDAENLYFQGMLMASTTSAVPGHPSLPSLPSNSSQERPLDTRDPLLARAELALLSIVFVAVALSNGLVLAALARRGRRGHWAPIHVFIGHLCLADLAVALFQVLPQLAWKATDRFRGPDALCRAVKYLQMVGMYASSYMILAMTLDRHRAICRPMLAYRHGSGAHWNRPVLVAWAFSLLLSLPQLFIFAQRNVEGGSGVTDCWACFAEPWGRRTYVTWIALMVFVAPTLGIAACQVLIFREIHASLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLAVAKTVRMTLVIVVVYVLCWAPFFLVQLWAAWDPEAPLEGAPFVLLMLLASLNSCTNPWIYASFSSSVSSELRSLLCCARGRTPPSLGPQDESCTTASSSLAKDTSSLEVLFQ
>9UVY_E1 len=765 copies=1 longest=1
MKTIIALSYIFCLVFADYKDDDDAENLYFQGMLMASTTSAVPGHPSLPSLPSNSSQERPLDTRDPLLARAELALLSIVFVAVALSNGLVLAALARRGRRGHWAPIHVFIGHLCLADLAVALFQVLPQLAWKATDRFRGPDALCRAVKYLQMVGMYASSYMILAMTLDRHRAICRPMLAYRHGSGAHWNRPVLVAWAFSLLLSLPQLFIFAQRNVEGGSGVTDCWACFAEPWGRRTYVTWIALMVFVAPTLGIAACQVLIFREIHASLELRRLLREQGEKLLKLAEESEKKIKEAIEKGEDVVKVATEETHKLIKAIWESSFEVARLLGVPEEVLEKFRKELEKLLEESKKRIEEAIKKGEDVLKVVEEEAKKLREHSEKFLETARSLGVDPRPLERLARIVRVVEESLVRIVKAIKEGEDVEKVLLSESTLIKSSIILSSGLDPREAYEELLATYEETRDSELVKEFVELVRELLRLRGEPEELVRQLDKLIEAIDRMEKAVAEGEDTLPLFLELAREILKALSDVARILAEREGLDVELIDEIVDTASRLLEEVIKKAKEGEDLDEEEEKVKKELEELKKRTEERARALGQDVELVRTIVDSLSTLLSEIVEGIKRVKEGEDPLEVLARVVLTATLQILELVRTAERRRRERVAKTVRMTLVIVVVYVLCWAPFFLVQLWAAWDPEAPLEGAPFVLLMLLASLNSCTNPWIYASFSSSVSSELRSLLCCARGRTPPSLGPQDESCTTASSSLAKDTSSLEVLFQ
>9UVZ_E1 len=759 copies=1 longest=1
MKTIIALSYIFCLVFADYKDDDDAENLYFQGMGQCYYNETIGFFYNNSGKELSSHWRPKDVVVVALGLTVSVLVLLTNLLVIAAIASNRRFHQPIYYLLGNLAAADLFAGVAYLFLMFHTGPRTARLSLEGWFLRQGLLDTSLTASVATLLAIAVERHRSVMAVQLHSRLPRGRVVMLIVGVWVAALGLGLLPAHSWHCLCALDRCSRMAPLLSRSYLAVWALSSLLVFLLMVAVYTRIFFYVRRRVQRMLELRRLLREQGEKLLKLAEESEKKIKEAIEKGEDVVKVATEETHKLIKAIWESSFEVARLLGVPEEVLEKFRKELEKLLEESKKRIEEAIKKGEDVLKVVEEEAKKLREHSEKFLETARSLGVDPRPLERLARIVRVVEESLVRIVKAIKEGEDVEKVLLSESTLIKSSIILSSGLDPREAYEELLATYEETRDSELVKEFVELVRELLRLRGEPEELVRQLDKLIEAIDRMEKAVAEGEDTLPLFLELAREILKALSDVARILAEREGLDVELIDEIVDTASRLLEEVIKKAKEGEDLDEEEEKVKKELEELKKRTEERARALGQDVELVRTIVDSLSTLLSEIVEGIKRVKEGEDPLEVLARVVLTATLQILELVRTAERRRRERTLSLVKTVVIILGAFVVCWTPGQVVLLLDGLGCESCNVLAVEKYFLLLAEANSLVNAAVYSCRDAEMRRTFRRLLCCACLRQSTRESVHYTSSAQGGASTRIMLPENGHPLMDSTLLEVLFQ
>9UW3_E1 len=403 copies=1 longest=1
MPKKKPTPIQLNPAPDGSAVNGTSSAETNLEALQKKLEELELDEQQRKRLEAFLTQKQKVGELKDDDFEKISELGAGNGGVVFKVSHKPSGLVMARKLIHLEIKPAIRNQIIRELQVLHECNSPYIVGFYGAFYSDGEISICMEHMDGGSLDQVLKKAGRIPEQILGKVSIAVIKGLTYLREKHKIMHRDVKPSNILVNSRGEIKLCDFGVSGQLIDSMANSFVGTRSYMSPERLQGTHYSVQSDIWSMGLSLVEMAVGRYPIPPPDAKELELMFGCQVEGDAAETPPRPRTPGRPLSSYGMDSRPPMAIFELLDYIVNEPPPKLPSGVFSLEFQDFVNKCLIKNPAERADLKQLMVHAFIKRSDAEEVDFAGWLCSTIGLNQPSTPTHAAGVLEHHHHHHHH
>9UW3_E2 len=388 copies=1 longest=0
HMAAAAAQGGGGGEPRRTEGVGPGVPGEVEMVKGQPFDVGPRYTQLQYIGEGAYGMVSSAYDHVRKTRVAIKKISPFEHQTYCQRTLREIQILLRFRHENVIGIRDILRASTLEAMRDVYIVQDLMETDLYKLLKSQQLSNDHICYFLYQILRGLKYIHSANVLHRDLKPSNLLINTTCDLKICDFGLARIADPEHDHTGFLTEYVATRWYRAPEIMLNSKGYTKSIDIWSVGCILAEMLSNRPIFPGKHYLDQLNHILGILGSPSQEDLNCIINMKARNYLQSLPSKTKVAWAKLFPKSDSKALDLLDRMLTFNPNKRITVEEALAHPYLEQYYDPTDEPVAEEPFTFAMELDDLPKERLKELIFQETARFQPGVLEAPLEHHHHHH
>9UW4_E1 len=388 copies=1 longest=0
HMAAAAAQGGGGGEPRRTEGVGPGVPGEVEMVKGQPFDVGPRYTQLQYIGEGAYGMVSSAYDHVRKTRVAIKKISPFEHQTYCQRTLREIQILLRFRHENVIGIRDILRASTLEAMRDVYIVQDLMETDLYKLLKSQQLSNDHICYFLYQILRGLKYIHSANVLHRDLKPSNLLINTTCDLKICDFGLARIADPEHDHTGFLTEYVATRWYRAPEIMLNSKGYTKSIDIWSVGCILAEMLSNRPIFPGKHYLDQLNHILGILGSPSQEDLNCIINMKARNYLQSLPSKTKVAWAKLFPKSDSKALDLLDRMLTFNPNKRITVEEALAHPYLEQYYDPTDEPVAEEPFTFAMELDDLPKERLKELIFQETARFQPGVLEAPLEHHHHHH
>9UW4_E2 len=403 copies=1 longest=1
MPKKKPTPIQLNPAPDGSAVNGTSSAETNLEALQKKLEELELDEQQRKRLEAFLTQKQKVGELKDDDFEKISELGAGNGGVVFKVSHKPSGLVMARKLIHLEIKPAIRNQIIRELQVLHECNSPYIVGFYGAFYSDGEISICMEHMDGGSLDQVLKKAGRIPEQILGKVSIAVIKGLTYLREKHKIMHRDVKPSNILVNSRGEIKLCDFGVSGQLIDSMANSFVGTRSYMSPERLQGTHYSVQSDIWSMGLSLVEMAVGRYPIPPPDAKELELMFGCQVEGDAAETPPRPRTPGRPLSSYGMDSRPPMAIFELLDYIVNEPPPKLPSGVFSLEFQDFVNKCLIKNPAERADLKQLMVHAFIKRSDAEEVDFAGWLCSTIGLNQPSTPTHAAGVLEHHHHHHHH
>9UXN_E1 len=337 copies=1 longest=1
MNEPLDYLANASDFPDYAAAFGNCTDENIPLKMHYLPVIYGIIFLVGFPGNAVVISTYIFKMRPWKSSTIIMLNLACTDLLYLTSLPFLIHYYASGENWIFGDFMCKFIRFSFHFNLYSSILFLTCFSIFRYCVIIHPMSCFSIHKTRCAVVACAVVWIISLVAVIPMTFLITSTNRTNRSACLDLTSSDELNTIKWYNLILTATTFCLPLVIVTLCYTTIIHTLTHGLQTDSCLKQKARRLTILLLLAFYVCFLPFHILRVIRIESRLLSISCSIENQIHEAYIVSRPLAALNTFGNLLLYVVVSDNFQQAVCSTVRCKVSGNLEQAKKISYSNNP
>9UXO_E1 len=337 copies=1 longest=1
MNEPLDYLANASDFPDYAAAFGNCTDENIPLKMHYLPVIYGIIFLVGFPGNAVVISTYIFKMRPWKSSTIIMLNLACTDLLYLTSLPFLIHYYASGENWIFGDFMCKFIRFSFHFNLYSSILFLTCFSIFRYCVIIHPMSCFSIHKTRCAVVACAVVWIISLVAVIPMTFLITSTNRTNRSACLDLTSSDELNTIKWYNLILTATTFCLPLVIVTLCYTTIIHTLTHGLQTDSCLKQKARRLTILLLLAFYVCFLPFHILRVIRIESRLLSISCSIENQIHEAYIVSRPLAALNTFGNLLLYVVVSDNFQQAVCSTVRCKVSGNLEQAKKISYSNNP
>9UXP_E1 len=337 copies=1 longest=1
MNEPLDYLANASDFPDYAAAFGNCTDENIPLKMHYLPVIYGIIFLVGFPGNAVVISTYIFKMRPWKSSTIIMLNLACTDLLYLTSLPFLIHYYASGENWIFGDFMCKFIRFSFHFNLYSSILFLTCFSIFRYCVIIHPMSCFSIHKTRCAVVACAVVWIISLVAVIPMTFLITSTNRTNRSACLDLTSSDELNTIKWYNLILTATTFCLPLVIVTLCYTTIIHTLTHGLQTDSCLKQKARRLTILLLLAFYVCFLPFHILRVIRIESRLLSISCSIENQIHEAYIVSRPLAALNTFGNLLLYVVVSDNFQQAVCSTVRCKVSGNLEQAKKISYSNNP
>9UXQ_E1 len=337 copies=1 longest=1
MNEPLDYLANASDFPDYAAAFGNCTDENIPLKMHYLPVIYGIIFLVGFPGNAVVISTYIFKMRPWKSSTIIMLNLACTDLLYLTSLPFLIHYYASGENWIFGDFMCKFIRFSFHFNLYSSILFLTCFSIFRYCVIIHPMSCFSIHKTRCAVVACAVVWIISLVAVIPMTFLITSTNRTNRSACLDLTSSDELNTIKWYNLILTATTFCLPLVIVTLCYTTIIHTLTHGLQTDSCLKQKARRLTILLLLAFYVCFLPFHILRVIRIESRLLSISCSIENQIHEAYIVSRPLAALNTFGNLLLYVVVSDNFQQAVCSTVRCKVSGNLEQAKKISYSNNP
>9UXS_E1 len=137 copies=1 longest=0
PSVLCLPPKPKDTLMISRTPEVTCVVSDYNPHHIYKFKWYVDGVEVHNAQTKPREEYDYLNARVVDKLTVLHQDWLNGKEYKCKVSASDEYYYDSRGYYRGASDIWGPIECTISKAKGQAGHHHHHHGDYKDDDDKG
>9UXS_E2 len=1245 copies=1 longest=1
QCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHVISGTNGTKRFDNPVLPFNDGVYFASIEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFCNDPFLDHKNNKSWMESEFRVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPIIVREPEDLPQGFSALEPLVDLPIGINITRFQTLLALHRSYLTPGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNITNLCPFDEVFNATRFASVYAWNRKRISNCVADYSVLYNLAPFFTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDSKVSGNYNYLYRLFRKSNLKPFERDISTEIYQAGNKPCNGVAGFNCYFPLRSYSFRPTYGVGHQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLKGTGVLTESNKKFLPFQQFGRDIADTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVPVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEYVNNSYECDIPIGAGICASYQTQTKSRSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKYFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFKGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLSSTPSALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGRSLEVLFQGPGGGGGSGGGGSHHHHHH
>9V2U_E1 len=211 copies=1 longest=0
MAVTACQGLGFVVSLIGIAGIIAATCMDQWSTQDLYNNPVTAVFNYQGLWRSCVRESSGFTECRGYFTLLGLPAMLQAVRALMIVGIVLGAIGLLVSIFALKCIRIGSMEDSAKANMTLTSGIMFIVSGLCAIAGVSVFANMLVTNFWMSTANMYTGMGGMVQTVQTRYTFGAALFVGWVAGGLTLIGGVMMCIACRGLAPEETNYKAAAA
>9V2U_E2 len=249 copies=1 longest=1
QVQLVQSGAEVKKPGASVKVSCKASGYTFTGYNMNWVRQAPGQGLEWMGNIDPYYGGTSYNQKFKGRVTMTIDKSTSTVYMELSSLRSEDTAVYYCARMYHGNAFDYWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCAAAENLYFQGDYKDDDDKHHHHHHHHHH
>9V2U_E3 len=220 copies=1 longest=0
DIVMTQSPDSLAVSLGERATINCKSSQSLLNSGNLKNYLTWYQQKPGQPPKLLIYWASTRKSGVPDRFSGSGSGTDFTLTISSLQAEDVAVYYCQNDYSYPLTFGGGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9V35_E1 len=393 copies=1 longest=1
MGKTIIALSYIFCLVFADYKDDDDAANFTPVNGSSGNQSVRLVTSSSLEVLFQGPGSGNDSVSYEYGDYSDLSDRPVDCLDGACLAIDPLRVAPLPLYAAIFLVGVPGNAMVAWVAGKVARRRVGATWLLHLAVADLLCCLSLPILAVPIARGGHWPYGAVGCRALPSIILLTMYASVLLLAALSADLCFLALGPAWWSTVQRACGVQVACGAAWTLALLLTVPSAIYRRLHQEHFPARLQCVVDYGGSSSTENAVTAIRFLFGFLGPLVAVASCHSALLCWAARRCRPLGTAIVVGFFVCWAPYHLLGLVLTVAAPNSALLARALRAEPLIVGLALAHSCLNPMLFLYFGRAQLRRSLPAACHWALRESQGQDESVDSKKSTSHDLVSEMEV
>9V38_E1 len=393 copies=1 longest=1
MGKTIIALSYIFCLVFADYKDDDDAANFTPVNGSSGNQSVRLVTSSSLEVLFQGPGSGNDSVSYEYGDYSDLSDRPVDCLDGACLAIDPLRVAPLPLYAAIFLVGVPGNAMVAWVAGKVARRRVGATWLLHLAVADLLCCLSLPILAVPIARGGHWPYGAVGCRALPSIILLTMYASVLLLAALSADLCFLALGPAWWSTVQRACGVQVACGAAWTLALLLTVPSAIYRRLHQEHFPARLQCVVDYGGSSSTENAVTAIRFLFGFLGPLVAVASCHSALLCWAARRCRPLGTAIVVGFFVCWAPYHLLGLVLTVAAPNSALLARALRAEPLIVGLALAHSCLNPMLFLYFGRAQLRRSLPAACHWALRESQGQDESVDSKKSTSHDLVSEMEV
>9V38_E2 len=10 copies=1 longest=0
YSFKPMPLAR
>9V3C_E1 len=393 copies=1 longest=1
MGKTIIALSYIFCLVFADYKDDDDAANFTPVNGSSGNQSVRLVTSSSLEVLFQGPGSGNDSVSYEYGDYSDLSDRPVDCLDGACLAIDPLRVAPLPLYAAIFLVGVPGNAMVAWVAGKVARRRVGATWLLHLAVADLLCCLSLPILAVPIARGGHWPYGAVGCRALPSIILLTMYASVLLLAALSADLCFLALGPAWWSTVQRACGVQVACGAAWTLALLLTVPSAIYRRLHQEHFPARLQCVVDYGGSSSTENAVTAIRFLFGFLGPLVAVASCHSALLCWAARRCRPLGTAIVVGFFVCWAPYHLLGLVLTVAAPNSALLARALRAEPLIVGLALAHSCLNPMLFLYFGRAQLRRSLPAACHWALRESQGQDESVDSKKSTSHDLVSEMEV
>9V3C_E2 len=74 copies=1 longest=0
TLQKKIEEIAAKYKHSVVKKCCYDGACVNNDETCEQRAARISLGPRCIKAFTECCVVASQLRANISHKDMQLGR
>9V3T_E1 len=476 copies=1 longest=1
MSTQRLRNEDYHDYSSTDVSPEESPSEGLNNLSSPGSYQRFGQSNSTTWFQTLIHLLKGNIGTGLLGLPLAVKNAGIVMGPISLLIIGIVAVHCMGILVKCAHHFCRRLNKSFVDYGDTVMYGLESSPCSWLRNHAHWGRRVVDFFLIVTQLGFCCVYFVFLADNFKQVIEAANGTTNNCHNNETVILTPTMDSRLYMLSFLPFLVLLVFIRNLRALSIFSLLANITMLVSLVMIYQFIVQRIPDPSHLPLVAPWKTYPLFFGTAIFSFEGIGMVLPLENKMKDPRKFPLILYLGMVIVTILYISLGCLGYLQFGANIQGSITLNLPNCWLYQSVKLLYSIGIFFTYALQFYVPAEIIIPFFVSRAPEHCELVVDLFVRTVLVCLTCILAILIPRLDLVISLVGSVSSSALALIIPPLLEVTTFYSEGMSPLTIFKDALISILGFVGFVVGTYEALYELIQPSNAPIFINSTCAFI
>9V3V_E1 len=476 copies=1 longest=1
MSTQRLRNEDYHDYSSTDVSPEESPSEGLNNLSSPGSYQRFGQSNSTTWFQTLIHLLKGNIGTGLLGLPLAVKNAGIVMGPISLLIIGIVAVHCMGILVKCAHHFCRRLNKSFVDYGDTVMYGLESSPCSWLRNHAHWGRRVVDFFLIVTQLGFCCVYFVFLADNFKQVIEAANGTTNNCHNNETVILTPTMDSRLYMLSFLPFLVLLVFIRNLRALSIFSLLANITMLVSLVMIYQFIVQRIPDPSHLPLVAPWKTYPLFFGTAIFSFEGIGMVLPLENKMKDPRKFPLILYLGMVIVTILYISLGCLGYLQFGANIQGSITLNLPNCWLYQSVKLLYSIGIFFTYALQFYVPAEIIIPFFVSRAPEHCELVVDLFVRTVLVCLTCILAILIPRLDLVISLVGSVSSSALALIIPPLLEVTTFYSEGMSPLTIFKDALISILGFVGFVVGTYEALYELIQPSNAPIFINSTCAFI
>9V3X_E1 len=476 copies=1 longest=1
MSTQRLRNEDYHDYSSTDVSPEESPSEGLNNLSSPGSYQRFGQSNSTTWFQTLIHLLKGNIGTGLLGLPLAVKNAGIVMGPISLLIIGIVAVHCMGILVKCAHHFCRRLNKSFVDYGDTVMYGLESSPCSWLRNHAHWGRRVVDFFLIVTQLGFCCVYFVFLADNFKQVIEAANGTTNNCHNNETVILTPTMDSRLYMLSFLPFLVLLVFIRNLRALSIFSLLANITMLVSLVMIYQFIVQRIPDPSHLPLVAPWKTYPLFFGTAIFSFEGIGMVLPLENKMKDPRKFPLILYLGMVIVTILYISLGCLGYLQFGANIQGSITLNLPNCWLYQSVKLLYSIGIFFTYALQFYVPAEIIIPFFVSRAPEHCELVVDLFVRTVLVCLTCILAILIPRLDLVISLVGSVSSSALALIIPPLLEVTTFYSEGMSPLTIFKDALISILGFVGFVVGTYEALYELIQPSNAPIFINSTCAFI
>9V3Y_E1 len=6 copies=1 longest=0
FKPAAR
>9V3Y_E2 len=393 copies=1 longest=1
MGKTIIALSYIFCLVFADYKDDDDAANFTPVNGSSGNQSVRLVTSSSLEVLFQGPGSGNDSVSYEYGDYSDLSDRPVDCLDGACLAIDPLRVAPLPLYAAIFLVGVPGNAMVAWVAGKVARRRVGATWLLHLAVADLLCCLSLPILAVPIARGGHWPYGAVGCRALPSIILLTMYASVLLLAALSADLCFLALGPAWWSTVQRACGVQVACGAAWTLALLLTVPSAIYRRLHQEHFPARLQCVVDYGGSSSTENAVTAIRFLFGFLGPLVAVASCHSALLCWAARRCRPLGTAIVVGFFVCWAPYHLLGLVLTVAAPNSALLARALRAEPLIVGLALAHSCLNPMLFLYFGRAQLRRSLPAACHWALRESQGQDESVDSKKSTSHDLVSEMEV
>9V3Z_E1 len=476 copies=1 longest=1
MSTQRLRNEDYHDYSSTDVSPEESPSEGLNNLSSPGSYQRFGQSNSTTWFQTLIHLLKGNIGTGLLGLPLAVKNAGIVMGPISLLIIGIVAVHCMGILVKCAHHFCRRLNKSFVDYGDTVMYGLESSPCSWLRNHAHWGRRVVDFFLIVTQLGFCCVYFVFLADNFKQVIEAANGTTNNCHNNETVILTPTMDSRLYMLSFLPFLVLLVFIRNLRALSIFSLLANITMLVSLVMIYQFIVQRIPDPSHLPLVAPWKTYPLFFGTAIFSFEGIGMVLPLENKMKDPRKFPLILYLGMVIVTILYISLGCLGYLQFGANIQGSITLNLPNCWLYQSVKLLYSIGIFFTYALQFYVPAEIIIPFFVSRAPEHCELVVDLFVRTVLVCLTCILAILIPRLDLVISLVGSVSSSALALIIPPLLEVTTFYSEGMSPLTIFKDALISILGFVGFVVGTYEALYELIQPSNAPIFINSTCAFI
>9V4D_E1 len=393 copies=1 longest=1
MGKTIIALSYIFCLVFADYKDDDDAANFTPVNGSSGNQSVRLVTSSSLEVLFQGPGSGNDSVSYEYGDYSDLSDRPVDCLDGACLAIDPLRVAPLPLYAAIFLVGVPGNAMVAWVAGKVARRRVGATWLLHLAVADLLCCLSLPILAVPIARGGHWPYGAVGCRALPSIILLTMYASVLLLAALSADLCFLALGPAWWSTVQRACGVQVACGAAWTLALLLTVPSAIYRRLHQEHFPARLQCVVDYGGSSSTENAVTAIRFLFGFLGPLVAVASCHSALLCWAARRCRPLGTAIVVGFFVCWAPYHLLGLVLTVAAPNSALLARALRAEPLIVGLALAHSCLNPMLFLYFGRAQLRRSLPAACHWALRESQGQDESVDSKKSTSHDLVSEMEV
>9V4D_E2 len=9 copies=1 longest=0
XYKPLGLAY
>9V82_E1 len=472 copies=1 longest=1
DYKDDDDAKLQTMHHHHHHHHHHENLYFQGGTTMADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLGSTLEVLFQGPDPTVPVFGTKLTPINGREETPCYNQTLSFTVLTCIISLVGLTGNAVVLWLLGYRMRRNAVSIYILNLAAADFLFLSFQIIRSPLRLINISHLIRKILVSVMTFPYFTGLSMLSAISTERCLSVLWPIWYRCRRPTHLSAVVCVLLWGLSLLFSMLEWRFCDFLFSGADSSWCETSDFIPVAWLIFLCVVLCVSSLVLLVRILCGSRKMPLTRLYVTILLTVLVFLLCGLPFGILGALIYRMHLNLEVLYCHVYLVCMSLSSLNSSANPIIYFFVGSFRQRQNRQNLKLVLQRALQDKPEVDKGEGQLPEESLELSGSRLGP
>9V8W_E1 len=635 copies=1 longest=1
MAKKSAENGIYSVSGDEKKGPLIAPGPDGAPAKGDGPVGLGTPGGRLAVPPRETWTRQMDFIMSCVGFAVGLGNVWRFPYLCYKNGGGVFLIPYVLIALVGGIPIFFLEISLGQFMKAGSINVWNICPLFKGLGYASMVIVFYCNTYYIMVLAWGFYYLVKSFTTTLPWATCGHTWNTPDCVEIFRHEDCANASLANLTCDQLADRRSPVIEFWENKVLRLSGGLEVPGALNWEVTLCLLACWVLVYFCVWKGVKSTGKIVYFTATFPYVVLVVLLVRGVLLPGALDGIIYYLKPDWSKLGSPQVWIDAGTQIFFSYAIGLGALTALGSYNRFNNNCYKDAIILALINSGTSFFAGFVVFSILGFMAAEQGVHISKVAESGPGLAFIAYPRAVTLMPVAPLWAALFFFMLLLLGLDSQFVGVEGFITGLLDLLPASYYFRFQREISVALCCALCFVIDLSMVTDGGMYVFQLFDYYSASGTTLLWQAFWECVVVAWVYGADRFMDDIACMIGYRPCPWMKWCWSFFTPLVCMGIFIFNVVYYEPLVYNNTYVYPWWGEAMGWAFALSSMLCVPLHLLGCLLRAKGTMAERWQHLTQPIWGLHHLEYRAQDADVRGLTTLTPVSESSKVVVVESVM
>9V8X_E1 len=635 copies=1 longest=1
MAKKSAENGIYSVSGDEKKGPLIAPGPDGAPAKGDGPVGLGTPGGRLAVPPRETWTRQMDFIMSCVGFAVGLGNVWRFPYLCYKNGGGVFLIPYVLIALVGGIPIFFLEISLGQFMKAGSINVWNICPLFKGLGYASMVIVFYCNTYYIMVLAWGFYYLVKSFTTTLPWATCGHTWNTPDCVEIFRHEDCANASLANLTCDQLADRRSPVIEFWENKVLRLSGGLEVPGALNWEVTLCLLACWVLVYFCVWKGVKSTGKIVYFTATFPYVVLVVLLVRGVLLPGALDGIIYYLKPDWSKLGSPQVWIDAGTQIFFSYAIGLGALTALGSYNRFNNNCYKDAIILALINSGTSFFAGFVVFSILGFMAAEQGVHISKVAESGPGLAFIAYPRAVTLMPVAPLWAALFFFMLLLLGLDSQFVGVEGFITGLLDLLPASYYFRFQREISVALCCALCFVIDLSMVTDGGMYVFQLFDYYSASGTTLLWQAFWECVVVAWVYGADRFMDDIACMIGYRPCPWMKWCWSFFTPLVCMGIFIFNVVYYEPLVYNNTYVYPWWGEAMGWAFALSSMLCVPLHLLGCLLRAKGTMAERWQHLTQPIWGLHHLEYRAQDADVRGLTTLTPVSESSKVVVVESVM
>9V8Y_E1 len=635 copies=1 longest=1
MAKKSAENGIYSVSGDEKKGPLIAPGPDGAPAKGDGPVGLGTPGGRLAVPPRETWTRQMDFIMSCVGFAVGLGNVWRFPYLCYKNGGGVFLIPYVLIALVGGIPIFFLEISLGQFMKAGSINVWNICPLFKGLGYASMVIVFYCNTYYIMVLAWGFYYLVKSFTTTLPWATCGHTWNTPDCVEIFRHEDCANASLANLTCDQLADRRSPVIEFWENKVLRLSGGLEVPGALNWEVTLCLLACWVLVYFCVWKGVKSTGKIVYFTATFPYVVLVVLLVRGVLLPGALDGIIYYLKPDWSKLGSPQVWIDAGTQIFFSYAIGLGALTALGSYNRFNNNCYKDAIILALINSGTSFFAGFVVFSILGFMAAEQGVHISKVAESGPGLAFIAYPRAVTLMPVAPLWAALFFFMLLLLGLDSQFVGVEGFITGLLDLLPASYYFRFQREISVALCCALCFVIDLSMVTDGGMYVFQLFDYYSASGTTLLWQAFWECVVVAWVYGADRFMDDIACMIGYRPCPWMKWCWSFFTPLVCMGIFIFNVVYYEPLVYNNTYVYPWWGEAMGWAFALSSMLCVPLHLLGCLLRAKGTMAERWQHLTQPIWGLHHLEYRAQDADVRGLTTLTPVSESSKVVVVESVM
>9VBD_E1 len=893 copies=1 longest=1
LTNGLDASALNALKSEWTTPPDGWEGSDPCGTNWVGITCQNDRVVSISLGNLDLEGKLPADISFLSELRILDLSYNPKLSGPLPPNIGNLGKLRNLILVGCSFSGQIPESIGTLKELIYLSLNLNKFSGTIPPSIGLLSKLYWFDIADNQIEGELPVSNGTSAPGLDMLLQTKHFHFGKNKLSGNIPKELFSSNMSLIHVLFDGNQFTGEIPETLSLVKTLTVLRLDRNKLIGDIPSYLNNLTNLNELYLANNRFTGTLPNLTSLTSLYTLDVSNNTLDFSPIPSWISSLPSLSTLRMEGIQLNGPIPISFFSPPQLQTVILKRNSIVESLDFGTDVSSQLEFVDLQYNEITDYKPSANKVLQVILANNPVCLEAGNGPSYCSAIQHNTSFSTLPTNCSPCEPGMEASPTCRCAYPFMGTLYFRSPSFSGLFNSTNFSILQKAIADFFKKFNYPVDSVGVRNIRENPTDHQLLIDLLVFPLGRESFNQTGMSLVGFAFSNQTYKPPPIFGPYIFKADLYKQFSGSAMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQT
>9VBV_E1 len=893 copies=1 longest=1
LTNGLDASALNALKSEWTTPPDGWEGSDPCGTNWVGITCQNDRVVSISLGNLDLEGKLPADISFLSELRILDLSYNPKLSGPLPPNIGNLGKLRNLILVGCSFSGQIPESIGTLKELIYLSLNLNKFSGTIPPSIGLLSKLYWFDIADNQIEGELPVSNGTSAPGLDMLLQTKAFAFGKNKLSGNIPKELFSSNMSLIAVLFDGNQFTGEIPETLSLVKTLTVLRLDRNKLIGDIPSYLNNLTNLNELYLANNRFTGTLPNLTSLTSLYTLDVSNNTLDFSPIPSWISSLPSLSTLRMEGIQLNGPIPISFFSPPQLQTVILKRNSIVESLDFGTDVSSQLEFVDLQYNEITDYKPSANKVLQVILANNPVCLEAGNGPSYCSAIQHNTSFSTLPTNCSPCEPGMEASPTCRCAYPFMGTLYFRSPSFSGLFNSTNFSILQKAIADFFKKFNYPVDSVGVRNIRENPTDHQLLIDLLVFPLGRESFNQTGMSLVGFAFSNQTYKPPPIFGPYIFKADLYKQFSGSAMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQT
>9VCT_E1 len=760 copies=1 longest=1
MGSSHHHHHHSSGLVPRGSHMASSSASGGSKIPNTSLFVPLTVKPQGPSADGDVGAELTRPLDKNEVKKVLDKFYKRKEIQKLGADYGLDARLFHQAFISFRNYIMQSHSLDVDIHIVLNDICFGAAHADDLFPFFLRHAKQIFPVLDCKDDLRKISDLRIPPNWYPDARAMQRKIIFHSGPTNSGKTYHAIQKYFSAKSGVYCGPLKLLAHEIFEKSNAAGVPCDLVTGEERVTVQPNGKQASHVSCTVEMCSVTTPYEVAVIDEIQMIRDPARGWAWTRALLGLCAEEVHLCGEPAAIDLVMELMYTTGEEVEVRDYKRLTPISVLDHALESLDNLRPGDCIVCFSKNDIYSVSRQIEIRGLESAVIYGSLPPGTKLAQAKKFNDPNDPCKILVATDAIGMGLNLSIRRIIFYSLIKPSINEKGERELEPITTSQALQIAGRAGRFSSRFKEGEVTTMNHEDLSLLKEILKRPVDPIRAAGLHPTAEQIEMFAYHLPDATLSNLIDIFVDFSQVDGQYFVCNMDDFKFSAELIQHIPLSLRVRYVFCTAPINKKQPFVCSSLLQFARQYSRNEPLTFAWLRRYIKWPLLPPKNIKDLMDLEAVHDVLDLYLWLSYRFMDMFPDASLIRDLQKELDGIIQDGVHNITKLIKMSETHKLLNLEGFPSGSQSRLSGTLKSQARRTRGTKALGSKATEPPSPDAGELSLASRLVQQGLLTPDMLKQLEKEWMTQQTEHNKEKTESGTHPKGTRRKKKEPDSD
>9VH3_E1 len=835 copies=1 longest=1
MTRLPLPFLFFFLTSIPLSVFSQFNDQSTLLNLKRDLGDPPSLRLWNNTSSPCNWSEITCTAGNVTGINFKNQNFTGTVPTTICDLSNLNFLDLSFNYFAGEFPTVLYNCTKLQYLDLSQNLLNGSLPVDIDRLSPELDYLDLAANGFSGDIPKSLGRISKLKVLNLYQSEYDGTFPSEIGDLSELEELRLALNDKFTPAKIPIEFGKLKKLKYMWLEEMNLIGEISPVVFENMTDLEHVDLSVNNLTGRIPDVLFGLKNLTEFYLFANGLTGEIPKSISATNLVFLDLSANNLTGSIPVSIGNLTKLQVLNLFNNKLTGEIPPVIGKLPGLKEFKIFNNKLTGEIPAEIGVHSKLERFEVSENQLTGKLPENLCKGGKLQGVVVYSNNLTGEIPESLGDCGTLLTVQLQNNDFSGKFPSRIWNASSMYSLQVSNNSFTGELPENVAWNMSRIEIDNNRFSGEIPKKIGTWSSLVEFKAGNNQFSGEFPKELTSLSNLISIFLDENDLTGELPDEIISWKSLITLSLSKNKLSGEIPRALGLLPRLLNLDLSENQFSGGIPPEIGSLKLTTFNVSSNRLTGGIPEQLDNLAYERSFLNNSNLCADNPVLSLPDCRKQRRGSRGFPGKILAMILVIAVLLLTITLFVTFFVVRDYTRKQRRRGLETWKLTSFHRVDFAESDIVSNLMKHYVIGSGGSGKVYKIFVESSGQCVAVKRIWDSKKLDQKLEKEFIAEVEILGTIRHSNIVKLLCCISREDSKLLVYEYLEKRSLDQWLHGKKKGGTVEANNLTWSQRLNIAVGAAQGLCYMHHDCTPAIIHRDVKSSNILLDSEFNAKI
>9VH9_E1 len=643 copies=1 longest=1
MGLLPKLGASQGSDTSTSRAGRCARSVFGNIKVFVLCQGLLQLCQLLYSAYFKSSLTTIEKRFGLSSSSSGLISSLNEISNAILIIFVSYFGSRVHRPRLIGIGGLFLAAGAFILTLPHFLSEPYQYTLASTGNNSRLQAELCQKHWQDLPPSKCHSTTQNPQKETSSMWGLMVVAQLLAGIGTVPIQPFGISYVDDFSEPSNSPLYISILFAISVFGPAFGYLLGSVMLQIFVDYGRVNTAAVNLVPGDPRWIGAWWLGLLISSALLVLTSFPFFFFPRAMPIGAKRAPATADEARKLEEAKSRGSLVDFIKRFPCIFLRLLMNSLFVLVVLAQCTFSSVIAGLSTFLNKFLEKQYGTSAAYANFLIGAVNLPAAALGMLFGGILMKRFVFSLQAIPRIATTIITISMILCVPLFFMGCSTPTVAEVYPPSTSSSIHPQSPACRRDCSCPDSIFHPVCGDNGIEYLSPCHAGCSNINMSSATSKQLIYLNCSCVTGGSASAKTGSCPVPCAHFLLPAIFLISFVSLIACISHNPLYMMVLRVVNQEEKSFAIGVQFLLMRLLAWLPSPALYGLTIDHSCIRWNSLCLGRRGACAYYDNDALRDRYLGLQMGYKALGMLLLCFISWRVKKNKEYNVQKAAGLI
>9VI4_E1 len=516 copies=1 longest=1
VGNASGNWHCDSTWLGDRVITTSTRTWALPTYNNHLYKQISSASTGASNDNHYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTTNDGVTTIANNLTSTVQVFSDSEYQLPYVLGSAHQGCLPPFPADVFMIPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFTFSYTFEDVPFHSSYAHSQSLDRLMNPLIDQYLYYLNRTQNQSGSAQNKDLLFSRGSPAGMSVQPKNWLPGPCYRQQRVSKTKTDNNNSNFTWTGASKYNLNGRESIINPGTAMASHKDDKDKFFPMSGVMIFGKESAGASNTALDNVMITDEEEIKATNPVATERFGTVATNHQSVNYEEETGDVHVMGALPGMVWQDRDVYLQGPIWAKIPHTDGHFHPSPLMGGFGLKHPPPQILIKNTPVPANPPAEFSATKFASFITQYSTGQVSVEIEWELQKENSKRWNPEVQYTSNYAKSANVDFTVDNNGLYTEPRPIGTRYLTRPL
>9VJ3_E1 len=401 copies=1 longest=1
MANPFSRWFLSERPPNCHVADLETSLDPHQTLLKVQKYKPALSDWVHYIFLGSIMLFVFITNPAPWIFKILFYCFLGTLFIIPATSQFFFNALPILTWVALYFTSSYFPDDRRPPITVKVLPAVETILYGDNLSDILATSTNSFLDILAWLPYGLFHFGAPFVVAAILFVFGPPTVLQGYAFAFGYMNLFGVIMQNVFPAAPPWYKILYGLQSANYDMHGSPGGLARIDKLLGINMYTTAFSNSSVIFGAFPSLHSGCATMEALFFCYCFPKLKPLFIAYVCWLWWSTMYLTHHYFVDLMAGSVLSYVIFQYTKYTHLPIVDTSLFCRWSYTSIEKYDISKSDPLAADSNDIESVPLSNLELDFDLNMTDEPSVSPSLFDGSTSVSRSSATSITSLGVKRA
>9VJ3_E2 len=221 copies=1 longest=0
MRSSLLTLPKSFLGFMPLYLAVEIVLGISILNKCSGAYGILALFTGHPLDFMQWIAYLWSVFTLIVFSQGLYLIHKPNLLVFSQICVLYTIDTISTCFFTLWFTTQWFTLEDTANIDGNNALQSNPISTGKLTERGIDISKQSATESYEYTMTILITLVSLIFRFYFNFILASFVQELLHHPKYLVDRDDVEQNLKNKPIWKRLWAKSQKGCYKLCKNLLE
>9VJ4_E1 len=401 copies=1 longest=1
MANPFSRWFLSERPPNCHVADLETSLDPHQTLLKVQKYKPALSDWVHYIFLGSIMLFVFITNPAPWIFKILFYCFLGTLFIIPATSQFFFNALPILTWVALYFTSSYFPDDRRPPITVKVLPAVETILYGDNLSDILATSTNSFLDILAWLPYGLFHFGAPFVVAAILFVFGPPTVLQGYAFAFGYMNLFGVIMQNVFPAAPPWYKILYGLQSANYDMHGSPGGLARIDKLLGINMYTTAFSNSSVIFGAFPSLHSGCATMEALFFCYCFPKLKPLFIAYVCWLWWSTMYLTHHYFVDLMAGSVLSYVIFQYTKYTHLPIVDTSLFCRWSYTSIEKYDISKSDPLAADSNDIESVPLSNLELDFDLNMTDEPSVSPSLFDGSTSVSRSSATSITSLGVKRA
>9VJ4_E2 len=221 copies=1 longest=0
MRSSLLTLPKSFLGFMPLYLAVEIVLGISILNKCSGAYGILALFTGHPLDFMQWIAYLWSVFTLIVFSQGLYLIHKPNLLVFSQICVLYTIDTISTCFFTLWFTTQWFTLEDTANIDGNNALQSNPISTGKLTERGIDISKQSATESYEYTMTILITLVSLIFRFYFNFILASFVQELLHHPKYLVDRDDVEQNLKNKPIWKRLWAKSQKGCYKLCKNLLE
>9VND_E1 len=135 copies=2 longest=0
MELFSFLQVDSDLTAWFILDGQEYEMSHFDINFAQSVDHKGQPQDEVRGGIMSITLSQTLPENIYRWGMTSIPKNGSVIFKSKTTNPPLKINFINAYCIRFNRSIANEGGLESQLVISPDEMLINGISFDNHWVK
>9VND_E2 len=142 copies=4 longest=0
MFGHKSFLRIGALGDSSISGLYKDSYELENCHFSFNQGTNTDGKPQTDVRGGTLYLTYAGLPQDDMLRWILNSKKYEDGAIVICDDSDEPLEKILFEQAACTGLNIEYTQKGKAYIHTKIILQVRKIKVGETTFENRWTINE
>9VND_E3 len=288 copies=1 longest=1
MGKEIEIKKLPEFEAMVNAGNTTYTGLVEGIGYVYWTTETLYFIRTNPEQLWAIPKYQQIPFPYFQRKDAIIETKTLHTLHVLSKDELLKLDYDAYYAFYGIVEEMLKFIHRADAIKSYCEIPLPIIKSKGALKGNDARNGILSLGSQINDQIGAPLDAANMLMDNKHIGKIGDGLSLISIIDEVGNGEYWSAAGDILLFAAGKTKLSPYMTVISLGTWMYETDLMQWRLACINYSDYKKTLIKYRELQKKFESGDKSVEEKMNECHKILNSHYIEMQKNLGNLGVKF
>9VNE_E1 len=142 copies=4 longest=1
MFGHKSFLRIGALGDSSISGLYKDSYELENCHFSFNQGTNTDGKPQTDVRGGTLYLTYAGLPQDDMLRWILNSKKYEDGAIVICDDSDEPLEKILFEQAACTGLNIEYTQKGKAYIHTKIILQVRKIKVGETTFENRWTINE
>9VNE_E2 len=135 copies=2 longest=0
MELFSFLQVDSDLTAWFILDGQEYEMSHFDINFAQSVDHKGQPQDEVRGGIMSITLSQTLPENIYRWGMTSIPKNGSVIFKSKTTNPPLKINFINAYCIRFNRSIANEGGLESQLVISPDEMLINGISFDNHWVK
>9VPG_E1 len=582 copies=1 longest=1
MASWSHPQFEKGGGARGGSGGGSWSHPQFEKGFDYKDDDDKGTARKQNRNSKELGLVPLTDDTSHAGPPGPGRALLECDHLRSGVPGGRRRKDWSCSLLVASLAGAFGSSFLYGYNLSVVNAPTPYIKAFYNESWERRHGRPIDPDTLTLLWSVTVSIFAIGGLVGTLIVKMIGKVLGRKHTLLANNGFAISAALLMACSLQAGAFEMLIVGRFIMGIDGGVALSVLPMYLSEISPKEIRGSLGQVTAIFICIGVFTGQLLGLPELLGKESTWPYLFGVIVVPAVVQLLSLPFLPDSPRYLLLEKHNEARAVKAFQTFLGKADVSQEVEEVLAESRVQRSIRLVSVLELLRAPYVRWQVVTVIVTMACYQLCGLNAIWFYTNSIFGKAGIPPAKIPYVTLSTGGIETLAAVFSGLVIEHLGRRPLLIGGFGLMGLFFGTLTITLTLQDHAPWVPYLSIVGILAIIASFCSGPGGIPFILTGEFFQQSQRPAAFIIAGTVNWLSNFAVGLLFPFIQKSLDTYCFLVFATICITGAIYLYFVLPETKNRTYAEISQAFSKRNKAYPPEEKIDSAVTDGKINGRP
>9VUH_E1 len=332 copies=1 longest=0
MADYKDDDDKSGRGAMGALQGCRSVEEFQCLNRIEEGTYGVVYRAKDKKTDEIVALKRLKMEKEKEGFPITSLREINTILKAQHPNIVTVREIVVGSNMDKIYIVMNYVEHDLKSLMETMKQPFLPGEVKTLMIQLLRGVKHLHDNWILHRDLKTSNLLLSHAGILKVGDFGLAREYGSPLKAYTPVVVTLWYRAPELLLGAKEYSTAVDMWSVGCIFGELLTQKPLFPGKSEIDQINKVFKDLGTPSEKIWPGYSELPAVKKMTFSEHPYNNLRKRFGALLSDQGFDLMNKFLTYFPGRRISAEDGLKHEYFRETPLPIDPSMFPKLVEKY
>9VUH_E2 len=365 copies=1 longest=1
MASGPHSTATAAAAASSAAPSAGGSSSGTTTTTTTTTGGILIGDRLYSEVSLTIDHSLIPEERLSPTPSMQDGLDLPSETDLRILGCELIQAAGILLRLPQVAMATGQVLFHRFFYSKSFVKHSFEIVAMACINLASKIEEAPRRIRDVINVFHHLRQLRGKRTPSPLILDQNYINTKNQVIKAERRVLKELGFCVHVKHPHKIIVMYLQVLECERNQTLVQTAWNYMNDSLRTNVFVRFQPETIACACIYLAARALQIPLPTRPHWFLLFGTTEEEIQEICIETLRLYTRKKPNYELLEKEVEKRKVALQEAKLKAKGLNPDGRIPGLIWSHPQFEKGGGSGGGGGSGGGSGGSSAWSHPQFEK
>9VUH_E3 len=318 copies=1 longest=0
MGSENLYFQGMAGKKNVLSSLAVYAEDSEPESDGEAGIEAVGSTAEEKGGLVSDAYGEDDFSRLGGDEDGYEEEEDENSRQSEDDDSETEKPEADDPKDNTEAEKRDPQELVASFSERVRNMSPDEIKIPPEPPGRCSNHLQDKIQKLYERKIKEGMDMNYIIQRKKEFRNPSIYEKLIQFCAIDELGTNYPKDMFDPHGWSEDSYYEALAKAQKIEMDKLEKAKKERTKIEFVTGTKKGTTTNATSTTTTTASTAVADAQKRKSKWDSAIPVTTIAQPTILTTTATLPAVVTVTTSASGSKTTVISAVGTIVKNAKQ
>9VVM_E1 len=729 copies=1 longest=1
MNPFQKNESKETLFSPVSIEEVPPRPPSPPKKPSPTICGSNYPLSIAFIVVNEFCERFSYYGMKAVLILYFLYFLHWNEDTSTSIYHAFSSLCYFTPILGAAIADSWLGKFKTIIYLSLVYVLGHVIKSLGALPILGGQVVHTVLSLIGLSLIALGTGGIKPCVAAFGGDQFEEKHAEERTRYFSVFYLSINAGSLISTFITPMLRGDVQCFGEDCYALAFGVPGLLMVIALVVFAMGSKIYNKPPPEGNIVAQVFKCIWFAISNRFKNRSGDIPKRQHWLDWAAEKYPKQLIMDVKALTRVLFLYIPLPMFWALLDQQGSRWTLQAIRMNRNLGFFVLQPDQMQVLNPLLVLIFIPLFDFVIYRLVSKCGINFSSLRKMAVGMILACLAFAVAAAVEIKINEMAPAQPGPQEVFLQVLNLADDEVKVTVVGNENNSLLIESIKSFQKTPHYSKLHLKTKSQDFHFHLKYHNLSLYTEHSVQEKNWYSLVIREDGNSISSMMVKDTESRTTNGMTTVRFVNTLHKDVNISLSTDTSLNVGEDYGVSAYRTVQRGEYPAVHCRTEDKNFSLNLGLLDFGAAYLFVITNNTNQGLQAWKIEDIPANKMSIAWQLPQYALVTAGEVMFSVTGLEFSYSQAPSSMKSVLQAAWLLTIAVGNIIVLVVAQFSGLVQWAEFILFSCLLLVICLIFSIMGYYYVPVKTEDMRGPADKHIPHIQGNMIKLETKKTKL
>9VWR_E1 len=566 copies=1 longest=1
DAQPRETWGKKIDFLLSVVGFAVDLANVWRFPYLCYKNGGGAFLIPYTLFLIIAGMPLFYMELALGQYNREGAATVWKICPFFKGVGYAVILIALYVGFYYNVIIAWSLYYLFSSFTLNLPWTDCGHTWNSPNCTDPKLLNGSVLGNHTKYSKYKFTPAAEFYERGVLHLHESSGIHDIGLPQWQLLLCLMVVVIVLYFSLWKGVKTSGKVVWITATLPYFVLFVLLVHGVTLPGASNGINAYLHIDFYRLKEATVWIDAATQIFFSLGAGFGVLIAFASYNKFDNNCYRDALLTSSINCITSFVSGFAIFSILGYMAHEHKVNIEDVATEGAGLVFILYPEAISTLSGSTFWAVVFFVMLLALGLDSSMGGMEAVITGLADDFQVLKRHRKLFTFGVTFSTFLLALFCITKGGIYVLTLLDTFAAGTSILFAVLMEAIGVSWFYGVDRFSNDIQQMMGFRPGLYWRLCWKFVSPAFLLFVVVVSIINFKPLTYDDYIFPPWANWVGWGIALSSMVLVPIYVIYKFLSTQGSLWERLAYGITPENEHHLVAQRDIRQFQLQHWLAI
>9VWS_E1 len=630 copies=1 longest=1
METTPLNSQKQLSACEDGEDCQENGVLQKVVPTPGDKVESGQISNGYSAVPSPGAGDDTRHSIPATTTTLVAELHQGERETWGKKVDFLLSVIGYAVDLGNVWRFPYICYQNGGGAFLLPYTIMAIFGGIPLFYMELALGQYHRNGCISIWRKICPIFKGIGYAICIIAFYIASYYNTIMAWALYYLISSFTDQLPWTSCKNSWNTGNCTNYFSEDNITWTLHSTSPAEEFYTRHVLQIHRSKGLQDLGGISWQLALCIMLIFTVIYFSIWKGVKTSGKVVWVTATFPYIILSVLLVRGATLPGAWRGVLFYLKPNWQKLLETGVWIDAAAQIFFSLGPGFGVLLAFASYNKFNNNCYQDALVTSVVNCMTSFVSGFVIFTVLGYMAEMRNEDVSEVAKDAGPSLLFITYAEAIANMPASTFFAIIFFLMLITLGLDSTFAGLEGVITAVLDEFPHVWAKRRERFVLAVVITCFFGSLVTLTFGGAYVVKLLEEYATGPAVLTVALIEAVAVSWFYGITQFCRDVKEMLGFSPGWFWRICWVAISPLFLLFIICSFLMSPPQLRLFQYNYPYWSIILGYCIGTSSFICIPTYIAYRLIITPGTFKERIIKSITPETPTEIPCGDIRLNAV
>9VXT_E1 len=379 copies=2 longest=1
MPHSSLHPSIPCPRGHGAQKAALVLLSACLVTLWGLGEPPEHTLRYLVLHLASLQLGLLLNGVCSLAEELRHIHSRYRGSYWRTVRACLGCPLRRGALLLLSIYFYYSLPNAVGPPFTWMLALLGLSQALNILLGLKGLAPAEISAVCENGNFSVAHGLAWSYYIGYLRLILPELQARIRTYNQHYNNLLRGAVSQRLYILLPLDCGVPDNLSMADPNIRFLDKLPQQTGDRAGIKDRVYSNSIYELLENGQRAGTCVLEYATPLQTLFAMSQYSQAGFSREDRLEQAKLFCRTLEDILADAPESQNNCRLIAYQEPADDSSFSLSQEVLRHLRQEEKEEVTVGSLKTSAVPSTSTMSQEPELLISGMEKPLPLRTDFS
>9VZP_E1 len=620 copies=1 longest=1
MLSAIYTVLAGLLFLPLLVNLCCPYFFQDIGYFLKVAAVGRRVRSYGKRRPARTILRAFLEKARQTPHKPFLLFRDETLTYAQVDRRSNQVARALHDHLGLRQGDCVALLMGNEPAYVWLWLGLVKLGCAMACLNYNIRAKSLLHCFQCCGAKVLLVSPELQAAVEEILPSLKKDDVSIYYVSRTSNTDGIDSFLDKVDEVSTEPIPESWRSEVTFSTPALYIYTSGTTGLPKAAMITHQRIWYGTGLTFVSGLKADDVIYITLPFYHSAALLIGIHGCIVAGATLALRTKFSASQFWDDCRKYNVTVIQYIGELLRYLCNSPQKPNDRDHKVRLALGNGLRGDVWRQFVKRFGDICIYEFYAATEGNIGFMNYARKVGAVGRVNYLQKKIITYDLIKYDVEKDEPVRDENGYCVRVPKGEVGLLVCKITQLTPFNGYAGAKAQTEKKKLRDVFKKGDLYFNSGDLLMVDHENFIYFHDRVGDTFRWKGENVATTEVADTVGLVDFVQEVNVYGVHVPDHEGRIGMASIKMKENHEFDGKKLFQHIADYLPSYARPRFLRIQDTIEITGTFKHRKMTLVEEGFNPAVIKDALYFLDDTAKMYVPMTEDIYNAISAKTLKL
>9W14_E1 len=223 copies=1 longest=1
RVQPTESIVRFPNITNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNF
>9W14_E2 len=107 copies=1 longest=0
DIVLTQSPATLSVTPGDNVSLSCRASQIISNNLHWYQQKSHESPRLLIKYASQSISGIPSRFSGSGSGTDFTLSINSVETEDFGMYFCQQSNTWPLTCGSGTKLELN
>9W14_E3 len=119 copies=1 longest=0
QIQLVQSGPELKKPGETVKISCKASGYTFTDYGLNWVKQAPGKGLKWMGWINTYSGEPTYNDEFRGRFAFSLETSTITAYLKINNLKNEDTATYFCARGGNWDWYFDVWGAGTTVTVSS
>9W22_E1 len=358 copies=2 longest=1
DSGCVVSWKNKELKCGSGIFITDNVHTWTEQYKFQPESPSKLASAIQKAHEEGICGIRSVTRLENLMWKQITPELNHILTENEVKLTIMTGDIKGIMQAGKRSLRPQPTELKYSWKAWGKAKMLSTELHNHTFLIDGPETAECPNTNRAWNSLEVEDYGFGVFTTNIWLKLKERQDVFCDSKLMSAAIKDNRAVHADMGYWIESALNDTWKIEKASFIEVKSCHWPKSHTLWSNGVLESEMIIPKNFAGPVSQHNYRPGYHTQTAGPWHLGRLEMDFDFCEGTTVVVTEDCGNRGPSLRTTTASGKLITEWCCRSCTLPPLRYRGEDGCWYGMEIRPLKEKEENLVNSLVTAHHHHHH
>9W3F_E1 len=517 copies=1 longest=1
MKTIIALSYIFCLVFADYKDDDDAMGQPGNGSAFLLAPNRSHAPDHDVTQQRDEVWVVGMGIVMSLIVLAIVFGNVLVITAIAKFERLQTVTNYFITSLACADLVMGLAVVPFGAAHILMKMWTFGNFWCEFWTSIDVLCVTASIETLCVIAVDRYFAITSPFKYQSLLTKNKARVIILMVWIVSGLTSFLPIQMHWYRATHQEAINCYAEETCCDFFTNQAYAIASSIVSFYVPLVIMVFVYSRVFQEAKRQLNIFEMLRIDEGLRLKIYKDTEGYYTIGIGHLLTKSPSLNAAKSELDKAIGRNTNGVITKDEAEKLFNQDVDAAVRGILRNAKLKPVYDSLDAVRRAALINMVFQMGETGVAGFTNSLRMLQQKRWDEAAVNLAKSRWYNQTPNRAKRVITTFRTGTWDAYKFCLKEHKALKTLGIIMGTFTLCWLPFFIVNIVHVIQDNLIRKEVYILLNWIGYVNSGFNPLIYCRSPDFRIAFQELLCLRRSSLKAYGNGYSSNGNTGEQSG
>9W3F_E2 len=125 copies=1 longest=0
QVQLQESGGGLVQAGGSLRLSCAASGSIFSLNDMGWYRQAPGKLRELVAAITSGGSTKYADSVKGRFTISRDNAKNTVYLQMNSLKAEDTAVYYCNAKVAGTFSIYDYWGQGTQVTVSSHHHHHH
>9W43_E1 len=213 copies=1 longest=1
QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYWITWIRQAPGQGLEWIGRIAPGSGSIYYDEMFKDRVTMTVDPSTSTAYIELSSLRSEDTAVYYCARLGNYGWTMDYWGQGTTVTVSSASTKGPSVFPLAPSGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEP
>9W43_E2 len=211 copies=1 longest=0
DIVMTQSPKSLAVSLGERATINCKASQSVTNNVAWYQQKPGQPPKLLIYYVSNRYTGVPDRFSGSGYGTDFTLTISSLQAEDVAVYFCQQDYSSWTFGGGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRG
>9W43_E3 len=115 copies=1 longest=0
NPPTFSPALLVVTEGDNATFTCSFSNTSESFVLNWYRMSPSNQTDKLAAFPEDRSQPGQDCRFRVTQLPNGRDFHMSVVRARRNDSGTYLCGAISLAPKAQIKESLRAELRVTER
>9W4I_E1 len=297 copies=1 longest=0
IESIIKTATDTVKSEINAELGVVPSLNAVETGATSNTEPEEAIQTRTVINQHGVSETLVENFLSRAALVSKRSFEYKDHTSSTARADKNFFKWTINTRSFVQLRRKLELFTYLRFDAEITILTTVAVNGSGNNTYVGLPDLTLQAMFVPTGALTPEKQDSFHWQSGSNASVFFKISDPPARITIPFMCINSAYSVFYDGFAGFEKNGLYGINPADTIGNLCVRIVNEHQPVGFTVTVRVYMKPKHIKAWAPRPPRTLPYMSIANANYKGKERAPNALSAIIGNRDSVKTMPHNIVNT
>9W4I_E2 len=317 copies=1 longest=1
MGAQVTRQQTGTHENANIATNGSHITYNQINFYKDSYAASASKQDFSQDPSKFTEPVVEGLKAGAPVLKSPSAEACGYSDRVLQLKLGNSAIVTQEAANYCCAYGEWPNYLPDHEAVAIDKPTQPETATDRFYTLKSVKWETGSTGWWWKLPDALNNIGMFGQNVQHHYLYRSGFLIHVQCNATKFHQGALLVVAIPEHQRGAHNTNTSPGFDDIMKGEEGGTFNHPYVLDDGTSLACATIFPHQWINLRTNNSATIVLPWMNAAPMDFPLRHNQWTLAIIPVVPLGTRTTSSMVPITVSIAPMCCEFNGLRHAITQ
>9W4I_E3 len=247 copies=1 longest=0
GVPTYLLPGSGQFLTTDDHSSAPALPCFNPTPEMHIPGQVRNMLEVVQVESMMEINNTESAVGMERLKVDISALTDVDQLLFNIPLDIQLDGPLRNTLVGNISRYYTHWSGSLEMTFMFCGSFMAAGKLILCYTPPGGSCPTTRETAMLGTHIVWDFGLQSSVTLIIPWISGSHYRMFNNDAKSTNANVGYVTCFMQTNLIVPSESSDTCSLIGFIAAKDDFSLRLMRDSPDIGQLDHLHAAEAAYQ
>9W62_E1 len=914 copies=1 longest=1
MAAFSKYLTARNSSLAGAAFLLLCLLHKRRRALGLHGKKSGKPPLQNNEKEGKKERAVVDKVFFSRLIQILKIMVPRTFCKETGYLVLIAVMLVSRTYCDVWMIQNGTLIESGIIGRSRKDFKRYLLNFIAAMPLISLVNNFLKYGLNELKLCFRVRLTKYLYEEYLQAFTYYKMGNLDNRIANPDQLLTQDVEKFCNSVVDLYSNLSKPFLDIVLYIFKLTSAIGAQGPASMMAYLVVSGLFLTRLRRPIGKMTITEQKYEGEYRYVNSRLITNSEEIAFYNGNKREKQTVHSVFRKLVEHLHNFILFRFSMGFIDSIIAKYLATVVGYLVVSRPFLDLSHPRHLKSTHSELLEDYYQSGRMLLRMSQALGRIVLAGREMTRLAGFTARITELMQVLKDLNHGKYERTMVSQQEKGIEGVQVIPLIPGAGEIIIADNIIKFDHVPLATPNGDVLIRDLNFEVRSGANVLICGPNGCGKSSLFRVLGELWPLFGGRLTKPERGKLFYVPQRPYMTLGTLRDQVIYPDGREDQKRKGISDLVLKEYLDNVQLGHILEREGGWDSVQDWMDVLSGGEKQRMAMARLFYHKPQFAILDECTSAVSVDVEGYIYSHCRKVGITLFTVSHRKSLWKHHEYYLHMDGRGNYEFKQITEDTVEFGSLEVLFQGPAVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSKLSKDPNEKRDHMVLLEFVTAAGITLGMDELYKHHHHHHHH
>9W63_E1 len=914 copies=1 longest=1
MAAFSKYLTARNSSLAGAAFLLLCLLHKRRRALGLHGKKSGKPPLQNNEKEGKKERAVVDKVFFSRLIQILKIMVPRTFCKETGYLVLIAVMLVSRTYCDVWMIQNGTLIESGIIGRSRKDFKRYLLNFIAAMPLISLVNNFLKYGLNELKLCFRVRLTKYLYEEYLQAFTYYKMGNLDNRIANPDQLLTQDVEKFCNSVVDLYSNLSKPFLDIVLYIFKLTSAIGAQGPASMMAYLVVSGLFLTRLRRPIGKMTITEQKYEGEYRYVNSRLITNSEEIAFYNGNKREKQTVHSVFRKLVEHLHNFILFRFSMGFIDSIIAKYLATVVGYLVVSRPFLDLSHPRHLKSTHSELLEDYYQSGRMLLRMSQALGRIVLAGREMTRLAGFTARITELMQVLKDLNHGKYERTMVSQQEKGIEGVQVIPLIPGAGEIIIADNIIKFDHVPLATPNGDVLIRDLNFEVRSGANVLICGPNGCGKSSLFRVLGELWPLFGGRLTKPERGKLFYVPQRPYMTLGTLRDQVIYPDGREDQKRKGISDLVLKEYLDNVQLGHILEREGGWDSVQDWMDVLSGGEKQRMAMARLFYHKPQFAILDECTSAVSVDVEGYIYSHCRKVGITLFTVSHRKSLWKHHEYYLHMDGRGNYEFKQITEDTVEFGSLEVLFQGPAVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSKLSKDPNEKRDHMVLLEFVTAAGITLGMDELYKHHHHHHHH
>9W64_E1 len=914 copies=1 longest=1
MAAFSKYLTARNSSLAGAAFLLLCLLHKRRRALGLHGKKSGKPPLQNNEKEGKKERAVVDKVFFSRLIQILKIMVPRTFCKETGYLVLIAVMLVSRTYCDVWMIQNGTLIESGIIGRSRKDFKRYLLNFIAAMPLISLVNNFLKYGLNELKLCFRVRLTKYLYEEYLQAFTYYKMGNLDNRIANPDQLLTQDVEKFCNSVVDLYSNLSKPFLDIVLYIFKLTSAIGAQGPASMMAYLVVSGLFLTRLRRPIGKMTITEQKYEGEYRYVNSRLITNSEEIAFYNGNKREKQTVHSVFRKLVEHLHNFILFRFSMGFIDSIIAKYLATVVGYLVVSRPFLDLSHPRHLKSTHSELLEDYYQSGRMLLRMSQALGRIVLAGREMTRLAGFTARITELMQVLKDLNHGKYERTMVSQQEKGIEGVQVIPLIPGAGEIIIADNIIKFDHVPLATPNGDVLIRDLNFEVRSGANVLICGPNGCGKSSLFRVLGELWPLFGGRLTKPERGKLFYVPQRPYMTLGTLRDQVIYPDGREDQKRKGISDLVLKEYLDNVQLGHILEREGGWDSVQDWMDVLSGGEKQRMAMARLFYHKPQFAILDECTSAVSVDVEGYIYSHCRKVGITLFTVSHRKSLWKHHEYYLHMDGRGNYEFKQITEDTVEFGSLEVLFQGPAVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSKLSKDPNEKRDHMVLLEFVTAAGITLGMDELYKHHHHHHHH
>9W65_E1 len=914 copies=1 longest=1
MAAFSKYLTARNSSLAGAAFLLLCLLHKRRRALGLHGKKSGKPPLQNNEKEGKKERAVVDKVFFSRLIQILKIMVPRTFCKETGYLVLIAVMLVSRTYCDVWMIQNGTLIESGIIGRSRKDFKRYLLNFIAAMPLISLVNNFLKYGLNELKLCFRVRLTKYLYEEYLQAFTYYKMGNLDNRIANPDQLLTQDVEKFCNSVVDLYSNLSKPFLDIVLYIFKLTSAIGAQGPASMMAYLVVSGLFLTRLRRPIGKMTITEQKYEGEYRYVNSRLITNSEEIAFYNGNKREKQTVHSVFRKLVEHLHNFILFRFSMGFIDSIIAKYLATVVGYLVVSRPFLDLSHPRHLKSTHSELLEDYYQSGRMLLRMSQALGRIVLAGREMTRLAGFTARITELMQVLKDLNHGKYERTMVSQQEKGIEGVQVIPLIPGAGEIIIADNIIKFDHVPLATPNGDVLIRDLNFEVRSGANVLICGPNGCGKSSLFRVLGELWPLFGGRLTKPERGKLFYVPQRPYMTLGTLRDQVIYPDGREDQKRKGISDLVLKEYLDNVQLGHILEREGGWDSVQDWMDVLSGGEKQRMAMARLFYHKPQFAILDQCTSAVSVDVEGYIYSHCRKVGITLFTVSHRKSLWKHHEYYLHMDGRGNYEFKQITEDTVEFGSLEVLFQGPAVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSKLSKDPNEKRDHMVLLEFVTAAGITLGMDELYKHHHHHHHH
>9W66_E1 len=914 copies=1 longest=1
MAAFSKYLTARNSSLAGAAFLLLCLLHKRRRALGLHGKKSGKPPLQNNEKEGKKERAVVDKVFFSRLIQILKIMVPRTFCKETGYLVLIAVMLVSRTYCDVWMIQNGTLIESGIIGRSRKDFKRYLLNFIAAMPLISLVNNFLKYGLNELKLCFRVRLTKYLYEEYLQAFTYYKMGNLDNRIANPDQLLTQDVEKFCNSVVDLYSNLSKPFLDIVLYIFKLTSAIGAQGPASMMAYLVVSGLFLTRLRRPIGKMTITEQKYEGEYRYVNSRLITNSEEIAFYNGNKREKQTVHSVFRKLVEHLHNFILFRFSMGFIDSIIAKYLATVVGYLVVSRPFLDLSHPRHLKSTHSELLEDYYQSGRMLLRMSQALGRIVLAGREMTRLAGFTARITELMQVLKDLNHGKYERTMVSQQEKGIEGVQVIPLIPGAGEIIIADNIIKFDHVPLATPNGDVLIRDLNFEVRSGANVLICGPNGCGKSSLFRVLGELWPLFGGRLTKPERGKLFYVPQRPYMTLGTLRDQVIYPDGREDQKRKGISDLVLKEYLDNVQLGHILEREGGWDSVQDWMDVLSGGEKQRMAMARLFYHKPQFAILDQCTSAVSVDVEGYIYSHCRKVGITLFTVSHRKSLWKHHEYYLHMDGRGNYEFKQITEDTVEFGSLEVLFQGPAVSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVNRIELKGIDFKEDGNILGHKLEYNYNSHNVYIMADKQKNGIKVNFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSKLSKDPNEKRDHMVLLEFVTAAGITLGMDELYKHHHHHHHH
>9W8B_E1 len=514 copies=1 longest=1
MALSELALVRWLQESRRSRKLILFIVFLALLLDNMLLTVVVPIIPSYLYSIKHEKNATEIQTARPVHTASISDSFQSIFSYYDNSTMVTGNATRDLTLHQTATQHMVTNASAVPSDCPSEDKDLLNENVQVGLLFASKATVQLITNPFIGLLTNRIGYPIPIFAGFCIMFVSTIMFAFSSSYAFLLIARSLQGIGSSCSSVAGMGMLASVYTDDEERGNVMGIALGGLAMGVLVGPPFGSVLYEFVGKTAPFLVLAALVLLDGAIQLFVLQPSRVQPESQKGTPLTTLLKDPYILIAAGSICFANMGIAMLEPALPIWMMETMCSRKWQLGVAFLPASISYLIGTNIFGILAHKMGRWLCALLGMIIVGVSILCIPFAKNIYGLIAPNFGVGFAIGMVDSSMMPIMGYLVDLRHVSVYGSVYAIADVAFCMGYAIGPSAGGAIAKAIGFPWLMTIIGIIDILFAPLCFFLRSPPAKEEKMAILMDHNCPIKTKMYTQNNIQSYPIGEDEESESD
>9W8C_E1 len=514 copies=1 longest=1
MALSELALVRWLQESRRSRKLILFIVFLALLLDNMLLTVVVPIIPSYLYSIKHEKNATEIQTARPVHTASISDSFQSIFSYYDNSTMVTGNATRDLTLHQTATQHMVTNASAVPSDCPSEDKDLLNENVQVGLLFASKATVQLITNPFIGLLTNRIGYPIPIFAGFCIMFVSTIMFAFSSSYAFLLIARSLQGIGSSCSSVAGMGMLASVYTDDEERGNVMGIALGGLAMGVLVGPPFGSVLYEFVGKTAPFLVLAALVLLDGAIQLFVLQPSRVQPESQKGTPLTTLLKDPYILIAAGSICFANMGIAMLEPALPIWMMETMCSRKWQLGVAFLPASISYLIGTNIFGILAHKMGRWLCALLGMIIVGVSILCIPFAKNIYGLIAPNFGVGFAIGMVDSSMMPIMGYLVDLRHVSVYGSVYAIADVAFCMGYAIGPSAGGAIAKAIGFPWLMTIIGIIDILFAPLCFFLRSPPAKEEKMAILMDHNCPIKTKMYTQNNIQSYPIGEDEESESD
>9W8D_E1 len=514 copies=1 longest=1
MALSELALVRWLQESRRSRKLILFIVFLALLLDNMLLTVVVPIIPSYLYSIKHEKNATEIQTARPVHTASISDSFQSIFSYYDNSTMVTGNATRDLTLHQTATQHMVTNASAVPSDCPSEDKDLLNENVQVGLLFASKATVQLITNPFIGLLTNRIGYPIPIFAGFCIMFVSTIMFAFSSSYAFLLIARSLQGIGSSCSSVAGMGMLASVYTDDEERGNVMGIALGGLAMGVLVGPPFGSVLYEFVGKTAPFLVLAALVLLDGAIQLFVLQPSRVQPESQKGTPLTTLLKDPYILIAAGSICFANMGIAMLEPALPIWMMETMCSRKWQLGVAFLPASISYLIGTNIFGILAHKMGRWLCALLGMIIVGVSILCIPFAKNIYGLIAPNFGVGFAIGMVDSSMMPIMGYLVDLRHVSVYGSVYAIADVAFCMGYAIGPSAGGAIAKAIGFPWLMTIIGIIDILFAPLCFFLRSPPAKEEKMAILMDHNCPIKTKMYTQNNIQSYPIGEDEESESD
>9W8E_E1 len=474 copies=1 longest=1
MALSELALVRWLQESRRSRKLILFIVFLALLLDNMLLTVVVPIIPSYLYSIKHEKNATEIQTARPVHTASISDSFQSIFSYYDNSTMVTGNATRDLTLHQTATQHMVTNASAVPSDCPSEDKDLLNENVQVGLLFASKATVQLITNPFIGLLTNRIGYPIPIFAGFCIMFVSTIMFAFSSSYAFLLIARSLQGIGSSCSSVAGMGMLASVYTDDEERGNVMGIALGGLAMGVLVGPPFGSVLYEFVGKTAPFLVLAALVLLDGAIQLFVLQPSRVQPESQKGTPLTTLLKDPYILIAAGSICFANMGIAMLEPALPIWMMETMCSRKWQLGVAFLPASISYLIGTNIFGILAHKMGRWLCALLGMIIVGVSILCIPFAKNIYGLIAPNFGVGFAIGMVDSSMMPIMGYLVDLRHVSVYGSVYAIADVAFCMGYAIGPSAGGAIAKAIGFPWLMTIIGIIDILFAPLCFFLRSPP
>9W8F_E1 len=474 copies=1 longest=1
MALSELALVRWLQESRRSRKLILFIVFLALLLDNMLLTVVVPIIPSYLYSIKHEKNATEIQTARPVHTASISDSFQSIFSYYDNSTMVTGNATRDLTLHQTATQHMVTNASAVPSDCPSEDKDLLNENVQVGLLFASKATVQLITNPFIGLLTNRIGYPIPIFAGFCIMFVSTIMFAFSSSYAFLLIARSLQGIGSSCSSVAGMGMLASVYTDDEERGNVMGIALGGLAMGVLVGPPFGSVLYEFVGKTAPFLVLAALVLLDGAIQLFVLQPSRVQPESQKGTPLTTLLKDPYILIAAGSICFANMGIAMLEPALPIWMMETMCSRKWQLGVAFLPASISYLIGTNIFGILAHKMGRWLCALLGMIIVGVSILCIPFAKNIYGLIAPNFGVGFAIGMVDSSMMPIMGYLVDLRHVSVYGSVYAIADVAFCMGYAIGPSAGGAIAKAIGFPWLMTIIGIIDILFAPLCFFLRSPP
>9W8J_E1 len=595 copies=1 longest=1
QGDENKARGNWSSKLDFILSMVGYAVGLGNVWRFPYLAFQNGGGAFLIPYLMMLALAGLPIFFLEVSLGQFASQGPVSVWKAIPALQGCGIAMLIISVLIAIYYNVIICYTLFYLFASFVSVLPWGSCNNPWNTPECKDKTKLLLDSCVISDHPKIQIKNSTFCMTAYPNVTMVNFTSQANKTFVSGSEEYFKYFVLKISAGIEYPGEIRWPLALCLFLAWVIVYASLAKGIKTSGKVVYFTATFPYVVLVILLIRGVTLPGAGAGIWYFITPKWEKLTDATVWKDAATQIFFSLSAAWGGLITLSSYNKFHNNCYRDTLIVTCTNSATSIFAGFVIFSVIGFMANERKVNIENVADQGPGIAFVVYPEALTRLPLSPFWAIIFFLMLLTLGLDTMFATIETIVTSISDEFPKYLRTHKPVFTLGCCICFFIMGFPMITQGGIYMFQLVDTYAASYALVIIAIFELVGISYVYGLQRFCEDIEMMIGFQPNIFWKVCWAFVTPTILTFILCFSFYQWEPMTYGSYRYPNWSMVLGWLMLACSVIWIPIMFVIKMHLAPGRFIERLKLVCSPQPDWGPFLAQHRGERYKNMIDPLA
>9W8Q_E1 len=593 copies=1 longest=1
GDENKARGNWSSKLDFILSMVGYAVGLGNVWRFPYLAFQNGGGAFLIPYLMMLALAGLPIFFLEVSLGQFASQGPVSVWKAIPALQGCGIAMLIISVLIAIYYNVIICYTLFYLFASFVSVLPWGSCNNPWNTPECKDKTKLLLDSCVISDHPKIQIKNSTFCMTAYPNVTMVNFTSQANKTFVSGSEEYFKYFVLKISAGIEYPGEIRWPLALCLFLAWVIVYASLAKGIKTSGKVVYFTATFPYVVLVILLIRGVTLPGAGAGIWYFITPKWEKLTDATVWKDAATQIFFSLSAAWGGLITLSSYNKFHNNCYRDTLIVTCTNSATSIFAGFVIFSVIGFMANERKVNIENVADQGPGIAFVVYPEALTRLPLSPFWAIIFFLMLLTLGLDTMFATIETIVTSISDEFPKYLRTHKPVFTLGCCICFFIMGFPMITQGGIYMFQLVDTYAASYALVIIAIFELVGISYVYGLQRFCEDIEMMIGFQPNIFWKVCWAFVTPTILTFILCFSFYQWEPMTYGSYRYPNWSMVLGWLMLACSVIWIPIMFVIKMHLAPGRFIERLKLVCSPQPDWGPFLAQHRGERYKNMIDPL
>9W97_E1 len=1291 copies=1 longest=1
MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATIGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGKYKTRAVVKFEGDTLVNRIELKGTDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIKANFTVRHNVEGSGHHHHHHHHADRKAAVSHWQQSSGLVPRGSGWSHPQFEKGSGDYKDDDDKGSGWSHPQFEKGSGDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQTKLSKDPNEKRDHMVLHEYVNAAGITGSMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTGGLEVLFQGPEFMDGKVAVQERGPPAVSWVPEEGEKLDQEDEDQVKDRGQWTNKMEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFFIPYFIFFFVCGIPVFFLEVALGQYTSQGSVTAWRKICPLFQGIGLASVVIESYLNVYYIIILAWALFYLFSSFTSELPWTTCNNFWNTEHCTDFLNHSGAGTVTPFENFTSPVMEFWERRVLGITSGIHDLGSLRWELALCLLLAWVICYFCIWKGVKSTGKVVYFTATFPYLMLVILLIRGVTLPGAYQGIIYYLKPDLFRLKDPQVWMDAGTQIFFSFAICQGCLTALGSYNKYHNNCYKDCIALCFLNSATSFVAGFVVFSILGFMSQEQGVPISEVAESGPGLAFIAFPKAVTMMPLSQLWSCLFFIMLIFLGLDSQFVCVECLVTASIDMFPRQLRKSGRRELLILTIAVMCYLIGLFLVTEGGMYIFQLFDYYASSGICLLFLSLFEVVCISWVYGADRFYDNIEDMIGYRPWPLVKISWLFLTPGLCLATFLFSLSKYTPLKYNNVYVYPPWGYSIGWFLALSSMVCVPLFVVITLLKTRGPFRKRLRQLITPDSSLPQPKQHPCLDGSAGRNFGPSPTREGLIAGEKETHL
>9W98_E1 len=1291 copies=1 longest=1
MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATIGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGKYKTRAVVKFEGDTLVNRIELKGTDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIKANFTVRHNVEGSGHHHHHHHHADRKAAVSHWQQSSGLVPRGSGWSHPQFEKGSGDYKDDDDKGSGWSHPQFEKGSGDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQTKLSKDPNEKRDHMVLHEYVNAAGITGSMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTGGLEVLFQGPEFMDGKVAVQERGPPAVSWVPEEGEKLDQEDEDQVKDRGQWTNKMEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFFIPYFIFFFVCGIPVFFLEVALGQYTSQGSVTAWRKICPLFQGIGLASVVIESYLNVYYIIILAWALFYLFSSFTSELPWTTCNNFWNTEHCTDFLNHSGAGTVTPFENFTSPVMEFWERRVLGITSGIHDLGSLRWELALCLLLAWVICYFCIWKGVKSTGKVVYFTATFPYLMLVILLIRGVTLPGAYQGIIYYLKPDLFRLKDPQVWMDAGTQIFFSFAICQGCLTALGSYNKYHNNCYKDCIALCFLNSATSFVAGFVVFSILGFMSQEQGVPISEVAESGPGLAFIAFPKAVTMMPLSQLWSCLFFIMLIFLGLDSQFVCVECLVTASIDMFPRQLRKSGRRELLILTIAVMCYLIGLFLVTEGGMYIFQLFDYYASSGICLLFLSLFEVVCISWVYGADRFYDNIEDMIGYRPWPLVKISWLFLTPGLCLATFLFSLSKYTPLKYNNVYVYPPWGYSIGWFLALSSMVCVPLFVVITLLKTRGPFRKRLRQLITPDSSLPQPKQHPCLDGSAGRNFGPSPTREGLIAGEKETHL
>9W99_E1 len=1291 copies=1 longest=1
MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATIGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGKYKTRAVVKFEGDTLVNRIELKGTDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIKANFTVRHNVEGSGHHHHHHHHADRKAAVSHWQQSSGLVPRGSGWSHPQFEKGSGDYKDDDDKGSGWSHPQFEKGSGDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQTKLSKDPNEKRDHMVLHEYVNAAGITGSMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTGGLEVLFQGPEFMDGKVAVQERGPPAVSWVPEEGEKLDQEDEDQVKDRGQWTNKMEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFFIPYFIFFFVCGIPVFFLEVALGQYTSQGSVTAWRKICPLFQGIGLASVVIESYLNVYYIIILAWALFYLFSSFTSELPWTTCNNFWNTEHCTDFLNHSGAGTVTPFENFTSPVMEFWERRVLGITSGIHDLGSLRWELALCLLLAWVICYFCIWKGVKSTGKVVYFTATFPYLMLVILLIRGVTLPGAYQGIIYYLKPDLFRLKDPQVWMDAGTQIFFSFAICQGCLTALGSYNKYHNNCYKDCIALCFLNSATSFVAGFVVFSILGFMSQEQGVPISEVAESGPGLAFIAFPKAVTMMPLSQLWSCLFFIMLIFLGLDSQFVCVECLVTASIDMFPRQLRKSGRRELLILTIAVMCYLIGLFLVTEGGMYIFQLFDYYASSGICLLFLSLFEVVCISWVYGADRFYDNIEDMIGYRPWPLVKISWLFLTPGLCLATFLFSLSKYTPLKYNNVYVYPPWGYSIGWFLALSSMVCVPLFVVITLLKTRGPFRKRLRQLITPDSSLPQPKQHPCLDGSAGRNFGPSPTREGLIAGEKETHL
>9W9A_E1 len=1291 copies=1 longest=1
MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATIGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGKYKTRAVVKFEGDTLVNRIELKGTDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIKANFTVRHNVEGSGHHHHHHHHADRKAAVSHWQQSSGLVPRGSGWSHPQFEKGSGDYKDDDDKGSGWSHPQFEKGSGDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQTKLSKDPNEKRDHMVLHEYVNAAGITGSMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTGGLEVLFQGPEFMDGKVAVQERGPPAVSWVPEEGEKLDQEDEDQVKDRGQWTNKMEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFFIPYFIFFFVCGIPVFFLEVALGQYTSQGSVTAWRKICPLFQGIGLASVVIESYLNVYYIIILAWALFYLFSSFTSELPWTTCNNFWNTEHCTDFLNHSGAGTVTPFENFTSPVMEFWERRVLGITSGIHDLGSLRWELALCLLLAWVICYFCIWKGVKSTGKVVYFTATFPYLMLVILLIRGVTLPGAYQGIIYYLKPDLFRLKDPQVWMDAGTQIFFSFAICQGCLTALGSYNKYHNNCYKDCIALCFLNSATSFVAGFVVFSILGFMSQEQGVPISEVAESGPGLAFIAFPKAVTMMPLSQLWSCLFFIMLIFLGLDSQFVCVECLVTASIDMFPRQLRKSGRRELLILTIAVMCYLIGLFLVTEGGMYIFQLFDYYASSGICLLFLSLFEVVCISWVYGADRFYDNIEDMIGYRPWPLVKISWLFLTPGLCLATFLFSLSKYTPLKYNNVYVYPPWGYSIGWFLALSSMVCVPLFVVITLLKTRGPFRKRLRQLITPDSSLPQPKQHPCLDGSAGRNFGPSPTREGLIAGEKETHL
>9W9B_E1 len=1291 copies=1 longest=1
MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATIGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGKYKTRAVVKFEGDTLVNRIELKGTDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIKANFTVRHNVEGSGHHHHHHHHADRKAAVSHWQQSSGLVPRGSGWSHPQFEKGSGDYKDDDDKGSGWSHPQFEKGSGDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQTKLSKDPNEKRDHMVLHEYVNAAGITGSMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTGGLEVLFQGPEFMDGKVAVQERGPPAVSWVPEEGEKLDQEDEDQVKDRGQWTNKMEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFFIPYFIFFFVCGIPVFFLEVALGQYTSQGSVTAWRKICPLFQGIGLASVVIESYLNVYYIIILAWALFYLFSSFTSELPWTTCNNFWNTEHCTDFLNHSGAGTVTPFENFTSPVMEFWERRVLGITSGIHDLGSLRWELALCLLLAWVICYFCIWKGVKSTGKVVYFTATFPYLMLVILLIRGVTLPGAYQGIIYYLKPDLFRLKDPQVWMDAGTQIFFSFAICQGCLTALGSYNKYHNNCYKDCIALCFLNSATSFVAGFVVFSILGFMSQEQGVPISEVAESGPGLAFIAFPKAVTMMPLSQLWSCLFFIMLIFLGLDSQFVCVECLVTASIDMFPRQLRKSGRRELLILTIAVMCYLIGLFLVTEGGMYIFQLFDYYASSGICLLFLSLFEVVCISWVYGADRFYDNIEDMIGYRPWPLVKISWLFLTPGLCLATFLFSLSKYTPLKYNNVYVYPPWGYSIGWFLALSSMVCVPLFVVITLLKTRGPFRKRLRQLITPDSSLPQPKQHPCLDGSAGRNFGPSPTREGLIAGEKETHL
>9W9C_E1 len=1291 copies=1 longest=1
MSKGEELFTGVVPILVELDGDVNGHKFSVRGEGEGDATIGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKRHDFFKSAMPEGYVQERTISFKDDGKYKTRAVVKFEGDTLVNRIELKGTDFKEDGNILGHKLEYNFNSHNVYITADKQKNGIKANFTVRHNVEGSGHHHHHHHHADRKAAVSHWQQSSGLVPRGSGWSHPQFEKGSGDYKDDDDKGSGWSHPQFEKGSGDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQTKLSKDPNEKRDHMVLHEYVNAAGITGSMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTGGLEVLFQGPEFMDGKVAVQERGPPAVSWVPEEGEKLDQEDEDQVKDRGQWTNKMEFVLSVAGEIIGLGNVWRFPYLCYKNGGGAFFIPYFIFFFVCGIPVFFLEVALGQYTSQGSVTAWRKICPLFQGIGLASVVIESYLNVYYIIILAWALFYLFSSFTSELPWTTCNNFWNTEHCTDFLNHSGAGTVTPFENFTSPVMEFWERRVLGITSGIHDLGSLRWELALCLLLAWVICYFCIWKGVKSTGKVVYFTATFPYLMLVILLIRGVTLPGAYQGIIYYLKPDLFRLKDPQVWMDAGTQIFFSFAICQGCLTALGSYNKYHNNCYKDCIALCFLNSATSFVAGFVVFSILGFMSQEQGVPISEVAESGPGLAFIAFPKAVTMMPLSQLWSCLFFIMLIFLGLDSQFVCVECLVTASIDMFPRQLRKSGRRELLILTIAVMCYLIGLFLVTEGGMYIFQLFDYYASSGICLLFLSLFEVVCISWVYGADRFYDNIEDMIGYRPWPLVKISWLFLTPGLCLATFLFSLSKYTPLKYNNVYVYPPWGYSIGWFLALSSMVCVPLFVVITLLKTRGPFRKRLRQLITPDSSLPQPKQHPCLDGSAGRNFGPSPTREGLIAGEKETHL
>9WAG_E1 len=302 copies=1 longest=1
GLGQMLESMIDNTVRETVGAATSRDALPNTEASGPTHSKEIPALTAVETGATNPLVPSDTVQTRHVVQHRSRSESSIESFFARGACVTIMTVDNSASTTNKDKLFAVWKITYKDTVQLRRKLEFFTYSRFDMELTFVVTANFTETNNGHALNQVYQIMYVPPGAPVPEKWDDYTWQTSSNPSIFYTYGTAPARISVPYVGISNAYSHFYDGFSKVPLKDQSAALGDSLYGAASLNDFGILAVRVVNDHNPTKVTSKIRVYLKPKHIRVWCPRPPRAVAYYGPGVDYKDGTLTPLSTKDLTTY
>9WAG_E2 len=272 copies=1 longest=0
SPNIEACGYSDRVLQLTLGNSTITTQEAANSVVAYGRWPEYLRDSEANPVDQPTEPDVAACRFYTLDTVSWTKESRGWWWKLPDALRDMGLFGQNMYYHYLGRSGYTVHVQCNASKFHQGALGVFAVPEMCLAGDSNTTTMHTSYQNANPGEKGGTFTGTFTPDNNQTSPARRFCPVDYLLGNGTLLGNAFVFPHQIINLRTNNCATLVLPYVNSLSIDSMVKHNNWGIAILPLAPLNFASESSPEIPITLTIAPMCCEFNGLRNITLPRLQ
>9WAG_E3 len=238 copies=1 longest=0
GLPVMNTPGSNQYLTADNFQSPCALPEFDVTPPIDIPGEVKNMMELAEIDTMIPFDLSATKKNTMEMYRVRLSDKPHTDDPILCLSLSPASDPRLSHTMLGEILNYYTHWAGSLKFTFLFCGSMMATGKLLVSYAPPGADPPKKRKEAMLGTHVIWDIGLQSSCTMVVPWISNTTYRQTIDDSFTEGGYISVFYQTRIVVPLSTPREMDILGFVSACNDFSVRLLRDTTHIEQKALAQ
>9WBR_E1 len=797 copies=1 longest=1
MDCSAPKEMNKLPANSPEAAAAQGHPDGPCAPRTSPEQELPAAAAPPPPRVPRSASTGAQTFQSADARACEAERPGVGSCKLSSPRAQAASAALRDLREAQGAQASPPPGSSGPGNALHCKIPFLRGPEGDANVSVGKGTLERNNTPVVGWVNMSQSTVVLATDGITSVLPGSVATVATQEDEQGDENKARGNWSSKLDFILSMVGYAVGLGNVWRFPYLAFQNGGGAFLIPYLMMLALAGLPIFFLEVSLGQFASQGPVSVWKAIPALQGCGIAMLIISVLIAIYYNVIICYTLFYLFASFVSVLPWGSCNNPWNTPECKDKTKLLLDSCVISDHPKIQIKNSTFCMTAYPNVTMVNFTSQANKTFVSGSEEYFKYFVLKISAGIEYPGEIRWPLALCLFLAWVIVYASLAKGIKTSGKVVYFTATFPYVVLVILLIRGVTLPGAGAGIWYFITPKWEKLTDATVWKDAATQIFFSLSAAWGGLITLSSYNKFHNNCYRDTLIVTCTNSATSIFAGFVIFSVIGFMANERKVNIENVADQGPGIAFVVYPEALTRLPLSPFWAIIFFLMLLTLGLDTMFATIETIVTSISDEFPKYLRTHKPVFTLGCCICFFIMGFPMITQGGIYMFQLVDTYAASYALVIIAIFELVGISYVYGLQRFCEDIEMMIGFQPNIFWKVCWAFVTPTILTFILCFSFYQWEPMTYGSYRYPNWSMVLGWLMLACSVIWIPIMFVIKMHLAPGRFIERLKLVCSPQPDWGPFLAQHRGERYKNMIDPLGTSSLGLKLPVKDLELGTQC
>9WC6_E1 len=797 copies=1 longest=1
MDCSAPKEMNKLPANSPEAAAAQGHPDGPCAPRTSPEQELPAAAAPPPPRVPRSASTGAQTFQSADARACEAERPGVGSCKLSSPRAQAASAALRDLREAQGAQASPPPGSSGPGNALHCKIPFLRGPEGDANVSVGKGTLERNNTPVVGWVNMSQSTVVLATDGITSVLPGSVATVATQEDEQGDENKARGNWSSKLDFILSMVGYAVGLGNVWRFPYLAFQNGGGAFLIPYLMMLALAGLPIFFLEVSLGQFASQGPVSVWKAIPALQGCGIAMLIISVLIAIYYNVIICYTLFYLFASFVSVLPWGSCNNPWNTPECKDKTKLLLDSCVISDHPKIQIKNSTFCMTAYPNVTMVNFTSQANKTFVSGSEEYFKYFVLKISAGIEYPGEIRWPLALCLFLAWVIVYASLAKGIKTSGKVVYFTATFPYVVLVILLIRGVTLPGAGAGIWYFITPKWEKLTDATVWKDAATQIFFSLSAAWGGLITLSSYNKFHNNCYRDTLIVTCTNSATSIFAGFVIFSVIGFMANERKVNIENVADQGPGIAFVVYPEALTRLPLSPFWAIIFFLMLLTLGLDTMFATIETIVTSISDEFPKYLRTHKPVFTLGCCICFFIMGFPMITQGGIYMFQLVDTYAASYALVIIAIFELVGISYVYGLQRFCEDIEMMIGFQPNIFWKVCWAFVTPTILTFILCFSFYQWEPMTYGSYRYPNWSMVLGWLMLACSVIWIPIMFVIKMHLAPGRFIERLKLVCSPQPDWGPFLAQHRGERYKNMIDPLGTSSLGLKLPVKDLELGTQC
>9WF3_E1 len=697 copies=1 longest=1
SRIGKLLGFEWTDLSSWRRLVTLLNRPTDPASLAVFRFLFGFLMVLDIPQERGLSSLDRKYLDGLDVCRFPLLDALRPLPLDWMYLVYTIMFLGALGMMLGLCYRISCVLFLLPYWYVFLLDKTSWNNHSYLYGLLAFQLTFMDANHYWSVDGLLNAHRRNAHVPLWNYAVLRGQIFIVYFIAGVKKLDADWVEGYSMEYLSRHWLFSPFKLLLSEELTSLLVVHWGGLLLDLSAGFLLFFDVSRSIGLFFVSYFHCMNSQLFSIGMFSYVMLASSPLFCSPEWPRKLVSYCPRRLQQLLPLKAAPQPSVSCVYKRSRGKSGQKPGLRHQLGAAFTLLYLLEQLFLPYSHFLTQGYNNWTNGLYGYSWDMMVHSRSHQHVKITYRDGRTGELGYLNPGVFTQSRRWKDHADMLKQYATCLSRLLPKYNVTEPQIYFDIWVSINDRFQQRIFDPRVDIVQAAWSPFQRTSWVQPLLMDLSPWRAKLQEIKSSLDNHTEVVFIADFPGLHLENFVSEDLGNTSIQLLQGEVTVELVAEQKNQTLREGEKMQLPAGEYHKVYTTSPSPSCYMYVYVNTTELALEQDLAYLQELKEKVENGSETGPLPPELQPLLEGEVKGGPEPTPLVQTFLRRQQRLQEIERRRNTPFHERFFRFLLRKLYVFRRSFLMTCISLRNLILGRPSLEQLAQEVTYANLRPF
>9WF3_E2 len=28 copies=1 longest=0
TVFLDHENANKILNRPKRYNSGKLEEFV
>9WF9_E1 len=28 copies=1 longest=0
SLFIRREQANNILARVTRANSFLEEMKK
>9WF9_E2 len=697 copies=1 longest=1
SRIGKLLGFEWTDLSSWRRLVTLLNRPTDPASLAVFRFLFGFLMVLDIPQERGLSSLDRKYLDGLDVCRFPLLDALRPLPLDWMYLVYTIMFLGALGMMLGLCYRISCVLFLLPYWYVFLLDKTSWNNHSYLYGLLAFQLTFMDANHYWSVDGLLNAHRRNAHVPLWNYAVLRGQIFIVYFIAGVKKLDADWVEGYSMEYLSRHWLFSPFKLLLSEELTSLLVVHWGGLLLDLSAGFLLFFDVSRSIGLFFVSYFHCMNSQLFSIGMFSYVMLASSPLFCSPEWPRKLVSYCPRRLQQLLPLKAAPQPSVSCVYKRSRGKSGQKPGLRHQLGAAFTLLYLLEQLFLPYSHFLTQGYNNWTNGLYGYSWDMMVHSRSHQHVKITYRDGRTGELGYLNPGVFTQSRRWKDHADMLKQYATCLSRLLPKYNVTEPQIYFDIWVSINDRFQQRIFDPRVDIVQAAWSPFQRTSWVQPLLMDLSPWRAKLQEIKSSLDNHTEVVFIADFPGLHLENFVSEDLGNTSIQLLQGEVTVELVAEQKNQTLREGEKMQLPAGEYHKVYTTSPSPSCYMYVYVNTTELALEQDLAYLQELKEKVENGSETGPLPPELQPLLEGEVKGGPEPTPLVQTFLRRQQRLQEIERRRNTPFHERFFRFLLRKLYVFRRSFLMTCISLRNLILGRPSLEQLAQEVTYANLRPF
>9WFA_E1 len=291 copies=1 longest=1
MSISLITDFALLAQSNLPETVSKVDFLSSGQYNLVYNAFSFAIATMLAAALFFFNVRGQVGQKYQLALIVSALVVSIAGYHYYRIFGSWEAAYSLQNGNYVLTGAPFNDAYRYVDWLLTVPLLLVETVAVLALPGMEARPLLIKLVVAAILMIATGYPGEISTDLNTRIVWGAVSTVPFIYILYILWIELSKSLLRQPEGVQTLVKNMRLLLLFSWGVYPIAYLLPMLGISGATADVGVQIGYTIADVLAKPIFGLLVFAIALEKTRVDAGEQPTKESPMTMISSHHHHHH
>9WFC_E1 len=697 copies=1 longest=1
SRIGKLLGFEWTDLSSWRRLVTLLNRPTDPASLAVFRFLFGFLMVLDIPQERGLSSLDRKYLDGLDVCRFPLLDALRPLPLDWMYLVYTIMFLGALGMMLGLCYRISCVLFLLPYWYVFLLDKTSWNNHSYLYGLLAFQLTFMDANHYWSVDGLLNAHRRNAHVPLWNYAVLRGQIFIVYFIAGVKKLDADWVEGYSMEYLSRHWLFSPFKLLLSEELTSLLVVHWGGLLLDLSAGFLLFFDVSRSIGLFFVSYFHCMNSQLFSIGMFSYVMLASSPLFCSPEWPRKLVSYCPRRLQQLLPLKAAPQPSVSCVYKRSRGKSGQKPGLRHQLGAAFTLLYLLEQLFLPYSHFLTQGYNNWTNGLYGYSWDMMVHSRSHQHVKITYRDGRTGELGYLNPGVFTQSRRWKDHADMLKQYATCLSRLLPKYNVTEPQIYFDIWVSINDRFQQRIFDPRVDIVQAAWSPFQRTSWVQPLLMDLSPWRAKLQEIKSSLDNHTEVVFIADFPGLHLENFVSEDLGNTSIQLLQGEVTVELVAEQKNQTLREGEKMQLPAGEYHKVYTTSPSPSCYMYVYVNTTELALEQDLAYLQELKEKVENGSETGPLPPELQPLLEGEVKGGPEPTPLVQTFLRRQQRLQEIERRRNTPFHERFFRFLLRKLYVFRRSFLMTCISLRNLILGRPSLEQLAQEVTYANLRPF
>9WFC_E2 len=28 copies=1 longest=0
LCYESHESMSYELNPFINRRNANTFISP
>9WFL_E1 len=697 copies=1 longest=1
SRIGKLLGFEWTDLSSWRRLVTLLNRPTDPASLAVFRFLFGFLMVLDIPQERGLSSLDRKYLDGLDVCRFPLLDALRPLPLDWMYLVYTIMFLGALGMMLGLCYRISCVLFLLPYWYVFLLDKTSWNNHSYLYGLLAFQLTFMDANHYWSVDGLLNAHRRNAHVPLWNYAVLRGQIFIVYFIAGVKKLDADWVEGYSMEYLSRHWLFSPFKLLLSEELTSLLVVHWGGLLLDLSAGFLLFFDVSRSIGLFFVSYFHCMNSQLFSIGMFSYVMLASSPLFCSPEWPRKLVSYCPRRLQQLLPLKAAPQPSVSCVYKRSRGKSGQKPGLRHQLGAAFTLLYLLEQLFLPYSHFLTQGYNNWTNGLYGYSWDMMVHSRSHQHVKITYRDGRTGELGYLNPGVFTQSRRWKDHADMLKQYATCLSRLLPKYNVTEPQIYFDIWVSINDRFQQRIFDPRVDIVQAAWSPFQRTSWVQPLLMDLSPWRAKLQEIKSSLDNHTEVVFIADFPGLHLENFVSEDLGNTSIQLLQGEVTVELVAEQKNQTLREGEKMQLPAGEYHKVYTTSPSPSCYMYVYVNTTELALEQDLAYLQELKEKVENGSETGPLPPELQPLLEGEVKGGPEPTPLVQTFLRRQQRLQEIERRRNTPFHERFFRFLLRKLYVFRRSFLMTCISLRNLILGRPSLEQLAQEVTYANLRPF
>9WFL_E2 len=29 copies=1 longest=0
RVFLTGEKANSILKRYPRANGFFEEIRQG
>9WFN_E1 len=16 copies=1 longest=0
TVFLDHENANKILNRP
>9WFN_E2 len=696 copies=1 longest=1
SRIGKLLGFEWTDLSSWRRLVTLLNRPTDPASLAVFRFLFGFLMVLDIPQERGLSSLDRKYLDGLDVCRFPLLDALRPLPLDWMYLVYTIMFLGALGMMLGLCYRISCVLFLLPYWYVFLLDKTSWNNHSYLYGLLAFQLTFMDANHYWSVDGLLNAHRRNAHVPLWNYAVLRGQIFIVYFIAGVKKLDADWVEGYSMEYLSRHWLFSPFKLLLSEELTSLLVVHWGGLLLDLSAGFLLFFDVSRSIGLFFVSYFHCMNSQLFSIGMFSYVMLASSPLFCSPEWPRKLVSYCPRRLQQLLPLKAAPQPSVSCVYKRSRGKSGQKPGLRHQLGAAFTLLYLLEQLFLPYSHFLTQGYNNWTNGLYGYSWDMMVHSRSHQHVKITYRDGRTGELGYLNPGVFTQSRRWKDHADMLKQYATCLSRLLPKYNVTEPQIYFDIWVSINDRFQQRIFDPRVDIVQAAWSPFQRTSWVQPLLMDLSPWRAKLQEIKSSLDNHTEVVFIADFPGLHLENFVSEDLGNTSIQLLQGEVTVELVAEQKNQTLREGEKMQLPAGEYHKVYTTSPSPSCYMYVYVNTTELALEQDLAYLQELKEKVENGSETGPLPPELQPLLEGEVKGGPEPTPLVQTFLRRQQRLQEIERRRNTPFHERFFRFLLRKLYVFRRSFLMTCISLRNLILGRPSLEQLAQEVTYANLRP
>9WFO_E1 len=17 copies=1 longest=0
TVFLDHENANKILNRPK
>9WFO_E2 len=696 copies=1 longest=1
SRIGKLLGFEWTDLSSWRRLVTLLNRPTDPASLAVFRFLFGFLMVLDIPQERGLSSLDRKYLDGLDVCRFPLLDALRPLPLDWMYLVYTIMFLGALGMMLGLCYRISCVLFLLPYWYVFLLDKTSWNNHSYLYGLLAFQLTFMDANHYWSVDGLLNAHRRNAHVPLWNYAVLRGQIFIVYFIAGVKKLDADWVEGYSMEYLSRHWLFSPFKLLLSEELTSLLVVHWGGLLLDLSAGFLLFFDVSRSIGLFFVSYFHCMNSQLFSIGMFSYVMLASSPLFCSPEWPRKLVSYCPRRLQQLLPLKAAPQPSVSCVYKRSRGKSGQKPGLRHQLGAAFTLLYLLEQLFLPYSHFLTQGYNNWTNGLYGYSWDMMVHSRSHQHVKITYRDGRTGELGYLNPGVFTQSRRWKDHADMLKQYATCLSRLLPKYNVTEPQIYFDIWVSINDRFQQRIFDPRVDIVQAAWSPFQRTSWVQPLLMDLSPWRAKLQEIKSSLDNHTEVVFIADFPGLHLENFVSEDLGNTSIQLLQGEVTVELVAEQKNQTLREGEKMQLPAGEYHKVYTTSPSPSCYMYVYVNTTELALEQDLAYLQELKEKVENGSETGPLPPELQPLLEGEVKGGPEPTPLVQTFLRRQQRLQEIERRRNTPFHERFFRFLLRKLYVFRRSFLMTCISLRNLILGRPSLEQLAQEVTYANLRP
>9WIJ_E1 len=447 copies=1 longest=1
MKTIIALSYIFCLVFADYKDDDDKWNSSDANFSCYHESVLGYRYVAVSWGVVVAVTGTVGNVLTLLALAIQPKLRTRFNLLIANLTLADLLYCTLLQPFSVDTYLHLHWRTGATFCRVFGLLLFASNSVSILTLCLIALGRYLLIAHPKLFPQVFSAKGIVLALVSTWVVGVASFAPLWPIYILVPVVCTCSFDRIRGRPYTTILMGIYFVLGLSSVGIFYCLIHRQVKRAAQALDQYKLRQADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLERARSTLQKEVDSSSEFGKVTRMCFAVFLCFALSYIPFLLLNILDARVQAPRVVHMLAANLTWLNGCINPVLYAAMNRQFRQAYGSILKRGPRSFHRLH
>9WIX_E1 len=546 copies=1 longest=1
MPFKAFDTFKEKILKPGKEGVKNAVGDSLGILQRKIDGTNEEEDAIELNEEGRPVQTSRAHRPVCDCSCCGIPKRYICDCSCCGIPKRYIIAVMSGLGFCISFGIRCNLGVAIVEMVNNSTVYVDGKPEIQTAQFNWDPETVGLIHGSFFWGYIVTQIPGGFISNKFAASRVFGAAIFLTSTLNMFIPSAARVHYGCVMGVRILQGLVEGVTYPACHGMWSKWAPPLERSRLATTSFCGSYAGAVVAMPLAGVLVQYIGWASVFYIYGMFGIIWYMFWLLQAYECPAAHPTISNAERTYIETSIGEGANLASLSKFNTPWRRFFTSLPVYAIIVANFCRSWTFYLLLISQPAYFEEVFGFAISKVGLLSAVPHMVMTIVVPIGGQLADYLRSRKILTTTAVRKIMNCGGFGMEATLLLVVGFSHTKGVAISFLVLAVGFSGFAISGFNVNHLDIAPRYASILMGISNGVGTLSGMVCPLIVGAMTKHKTREEWQNVFLIAALVHYSGVIFYGVFASGEKQDWADPSRLEEELRRRTEPGSENLYFQ
>9WME_E1 len=453 copies=1 longest=0
MTPSISKDAPIKGSITISKKGATFTAYKLLDAIKSGDAYEYSVNSDLKDFFNNSNYGSYSQESIQKLNGEQVKEFAINLHKYILENKKSGQELKDGQKNTVDLGYYLVTETSSDSEGAAVASTPIIVSVPQVSGDSWNYDVTINPKDNTPILEKNIVKENQRVKTSSENIGDVVKYEVKASIPVYQKNAQNIMYKFTDTMSKGLTYDEKTGFKVTSGDKVFAKDTDYTVDVKKQEDGSTVITINFVYENIKAYAETGITLNYQATLNKDAVISNKENLGNTNNIQLDYTNNPHVKDSYKKLTDKVTTYTFGFGITKVDSELNSKLLQGAEFSVKDAGGKIVAKYTYDEKGQVVYLSGNGVTNSKGITTFLGLKEGKYFITEEVAPSGYSLLKNPVEVTITANKDESGNYTGAATIEISNGNKAGQIINDISEKDGNILFNVQIENHAHHHHHH
>9WME_E2 len=657 copies=1 longest=1
MEETKSVQNDGAVEITSTTFESNTVAKGISNNLRIDYKILNKDKLKDGDKIVISLPDIFKDIEPKCHDQHFKDFDVKDGVVTLTFNENVEKAVTGYMIIRFVGNSNIRKGVSYPVSIDLNGKPSTVYITGEEYSNSSSGAQYPLMYKTADLPTAATDKEQGREYYGEIVDRNKPIKYFVEINLGDGVNPNTRSYLSNADFFDNIPKGMALDVNSICIKRMGYYDERSSDVTKDFWESNRIKADTKHLEINFGDIRYERYTVIYETRITSTESGYLNDAKLYYDDKELPSKHYSKLSKDAGALNVYKYVDKTKVKNNLNDQKIKYDIKFDSYGYFFKDTLNIIDKLDPRLSDIKITATDQFITDFDENTKELVIKNSNGDIDAKKPAYITIEASMKNVGPGEVVKNIAYVNGNPTNEVSTRKNPIVEIIKVSKDDVESNLLEGAIFKLTTKDGKTVKDVYNKGVKTFTTSSEGSIRFELPNGDYKLEEIKAPNGYKLDQTPIEFTVNDESKIVKVVAKDDPLPTTCNLVINKINEKEIPILGAKFKLFEESNPKKVLKFSSQKNNYELNQNGVGKLTPSGKNASFKINNLHYGNYILKEVQAPKGYKLSDDIYITLGFEESFYRVGKQGEKIILNKNTETNTYNISVENVPRIILPET
>9WML_E1 len=636 copies=1 longest=1
MKKLQGAHLRKPVTPDLLMTPSDQGDVDLDVDFAAHRGNWTGKLDFLLSCIGYCVGLGNVWRFPYRAYTNGGGAFLVPYFLMLAICGIPLFFLELSLGQFSSLGPLAVWKISPLFKGAGAAMLLIVGLVAIYYNMIIAYVLFYLFASLTSDLPWEHCGNWWNTELCLEHRVSKDGNGALPLNLTCTVSPSEEYWSRYVLHIQGSQGIGSPGEIRWNLCLCLLLAWVIVFLCILKGVKSSGKVVYFTATFPYLILLMLLVRGVTLPGAWKGIQFYLTPQFHHLLSSKVWIEAALQIFYSLGVGFGGLLTFASYNTFHQNIYRDTFIVTLGNAITSILAGFAIFSVLGYMSQELGVPVDQVAKAGPGLAFVVYPQAMTMLPLSPFWSFLFFFMLLTLGLDSQFAFLETIVTAVTDEFPYYLRPKKAVFSGLICVAMYLMGLILTTDGGMYWLVLLDDYSASFGLMVVVITTCLAVTRVYGIQRFCRDIHMMLGFKPGLYFRACWLFLSPATLLALMVYSIVKYQPSEYGSYRFPPWAELLGILMGLLSCLMIPAGMLVAVLREEGSLWERLQQASRPAMDWGPSLEENRTGMYVATLAGSQSPKPLMVHMRKYGGITSFENTAIEVDREIAEEEESMM
>9WMM_E1 len=636 copies=1 longest=1
MKKLQGAHLRKPVTPDLLMTPSDQGDVDLDVDFAAHRGNWTGKLDFLLSCIGYCVGLGNVWRFPYRAYTNGGGAFLVPYFLMLAICGIPLFFLELSLGQFSSLGPLAVWKISPLFKGAGAAMLLIVGLVAIYYNMIIAYVLFYLFASLTSDLPWEHCGNWWNTELCLEHRVSKDGNGALPLNLTCTVSPSEEYWSRYVLHIQGSQGIGSPGEIRWNLCLCLLLAWVIVFLCILKGVKSSGKVVYFTATFPYLILLMLLVRGVTLPGAWKGIQFYLTPQFHHLLSSKVWIEAALQIFYSLGVGFGGLLTFASYNTFHQNIYRDTFIVTLGNAITSILAGFAIFSVLGYMSQELGVPVDQVAKAGPGLAFVVYPQAMTMLPLSPFWSFLFFFMLLTLGLDSQFAFLETIVTAVTDEFPYYLRPKKAVFSGLICVAMYLMGLILTTDGGMYWLVLLDDYSASFGLMVVVITTCLAVTRVYGIQRFCRDIHMMLGFKPGLYFRACWLFLSPATLLALMVYSIVKYQPSEYGSYRFPPWAELLGILMGLLSCLMIPAGMLVAVLREEGSLWERLQQASRPAMDWGPSLEENRTGMYVATLAGSQSPKPLMVHMRKYGGITSFENTAIEVDREIAEEEESMM
>9WMN_E1 len=636 copies=1 longest=1
MKKLQGAHLRKPVTPDLLMTPSDQGDVDLDVDFAAHRGNWTGKLDFLLSCIGYCVGLGNVWRFPYRAYTNGGGAFLVPYFLMLAICGIPLFFLELSLGQFSSLGPLAVWKISPLFKGAGAAMLLIVGLVAIYYNMIIAYVLFYLFASLTSDLPWEHCGNWWNTELCLEHRVSKDGNGALPLNLTCTVSPSEEYWSRYVLHIQGSQGIGSPGEIRWNLCLCLLLAWVIVFLCILKGVKSSGKVVYFTATFPYLILLMLLVRGVTLPGAWKGIQFYLTPQFHHLLSSKVWIEAALQIFYSLGVGFGGLLTFASYNTFHQNIYRDTFIVTLGNAITSILAGFAIFSVLGYMSQELGVPVDQVAKAGPGLAFVVYPQAMTMLPLSPFWSFLFFFMLLTLGLDSQFAFLETIVTAVTDEFPYYLRPKKAVFSGLICVAMYLMGLILTTDGGMYWLVLLDDYSASFGLMVVVITTCLAVTRVYGIQRFCRDIHMMLGFKPGLYFRACWLFLSPATLLALMVYSIVKYQPSEYGSYRFPPWAELLGILMGLLSCLMIPAGMLVAVLREEGSLWERLQQASRPAMDWGPSLEENRTGMYVATLAGSQSPKPLMVHMRKYGGITSFENTAIEVDREIAEEEESMM
>9WMO_E1 len=636 copies=1 longest=1
MKKLQGAHLRKPVTPDLLMTPSDQGDVDLDVDFAAHRGNWTGKLDFLLSCIGYCVGLGNVWRFPYRAYTNGGGAFLVPYFLMLAICGIPLFFLELSLGQFSSLGPLAVWKISPLFKGAGAAMLLIVGLVAIYYNMIIAYVLFYLFASLTSDLPWEHCGNWWNTELCLEHRVSKDGNGALPLNLTCTVSPSEEYWSRYVLHIQGSQGIGSPGEIRWNLCLCLLLAWVIVFLCILKGVKSSGKVVYFTATFPYLILLMLLVRGVTLPGAWKGIQFYLTPQFHHLLSSKVWIEAALQIFYSLGVGFGGLLTFASYNTFHQNIYRDTFIVTLGNAITSILAGFAIFSVLGYMSQELGVPVDQVAKAGPGLAFVVYPQAMTMLPLSPFWSFLFFFMLLTLGLDSQFAFLETIVTAVTDEFPYYLRPKKAVFSGLICVAMYLMGLILTTDGGMYWLVLLDDYSASFGLMVVVITTCLAVTRVYGIQRFCRDIHMMLGFKPGLYFRACWLFLSPATLLALMVYSIVKYQPSEYGSYRFPPWAELLGILMGLLSCLMIPAGMLVAVLREEGSLWERLQQASRPAMDWGPSLEENRTGMYVATLAGSQSPKPLMVHMRKYGGITSFENTAIEVDREIAEEEESMM
>9WMP_E1 len=636 copies=1 longest=1
MKKLQGAHLRKPVTPDLLMTPSDQGDVDLDVDFAAHRGNWTGKLDFLLSCIGYCVGLGNVWRFPYRAYTNGGGAFLVPYFLMLAICGIPLFFLELSLGQFSSLGPLAVWKISPLFKGAGAAMLLIVGLVAIYYNMIIAYVLFYLFASLTSDLPWEHCGNWWNTELCLEHRVSKDGNGALPLNLTCTVSPSEEYWSRYVLHIQGSQGIGSPGEIRWNLCLCLLLAWVIVFLCILKGVKSSGKVVYFTATFPYLILLMLLVRGVTLPGAWKGIQFYLTPQFHHLLSSKVWIEAALQIFYSLGVGFGGLLTFASYNTFHQNIYRDTFIVTLGNAITSILAGFAIFSVLGYMSQELGVPVDQVAKAGPGLAFVVYPQAMTMLPLSPFWSFLFFFMLLTLGLDSQFAFLETIVTAVTDEFPYYLRPKKAVFSGLICVAMYLMGLILTTDGGMYWLVLLDDYSASFGLMVVVITTCLAVTRVYGIQRFCRDIHMMLGFKPGLYFRACWLFLSPATLLALMVYSIVKYQPSEYGSYRFPPWAELLGILMGLLSCLMIPAGMLVAVLREEGSLWERLQQASRPAMDWGPSLEENRTGMYVATLAGSQSPKPLMVHMRKYGGITSFENTAIEVDREIAEEEESMM
>9WMQ_E1 len=182 copies=1 longest=1
PTAGTYYTKKYSTMNVISVGTPQDNKPWHANHFITRLNEWETAISFEYYKILKMKVTLSPVISPAQQKKTMFGHTAIDLDGAWTTNTWLQDDPYAESSTRKVMTSKKKHSRYFTPKPILAGTTSAHPGQSLFFFSRPTPWLNTYDPTVQWGALLWSIYVPEKTGMTDFYGTKEVWIRYKSVL
>9WMQ_E2 len=120 copies=1 longest=0
QAQLQQPGAELVRPGASVKLSCKASGYTFSSYWMNWVKQRPEQGLEWIGRIDFYDSENHYNQKFKDKAILTVDKSSSTAYMQLSSLTSEDSAVYYCAGESYGNYGRFERWGQGTTLTVSS
>9WMQ_E3 len=104 copies=1 longest=0
DIVMTQSQKFMSTSVGERVSVTCKASQNVGTNVAWYQQKPGQSPQPLIYSASYRYSGVPDRFTGSGSGTDFTLTISDVQSEDLAEYFCQQYNSYPLAFGGGTKL
>9WMR_E1 len=214 copies=1 longest=1
MVYRRRRRSSTGTTYGSTRRRRYARRRLFIRRPTAGTYYTKKYSTMNVISVGTPQDNKPWHANHFITRLNEWETAISFEYYKILKMKVTLSPVISPAQQKKTMFGHTAIDLDGAWTTNTWLQDDPYAESSTRKVMTSKKKHSRYFTPKPILAGTTSAHPGQSLFFFSRPTPWLNTYDPTVQWGALLWSIYVPEKTGMTDFYGTKEVWIRYKSVL
>9WNG_E1 len=236 copies=1 longest=0
MKKNIAFLLASMFVFSIATNAYADIVLIQSPAIMSASPGEKVTITCSASSSVSYMHWFQQKPGTSPKLWIYSTSNLASGVPARFSGSGSGTSYSLTISRMEAEDAATYYCQQRSSYPLTFGAGTKLEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSLPVTKSFNRGEC
>9WNG_E2 len=270 copies=1 longest=0
MKKNIAFLLASMFVFSIATNAYAEVQLQQSGPELVKPGASVKLSCKASGYTFTSSWMHWVKQRPGQGLEWIGNIYPSNGGTNYNERFKSKATLTVDRSSNTAYMQLSSLTSEDSAVYFCARFGSFITTILTTYYNPVDYWGQGTTLTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCENLYFQSGSHHHHHHHH
>9WNG_E3 len=307 copies=1 longest=1
GPGSGSGTESIPEEMKQIVEEQGNKLHWAALLILMVIIPTIGGNTLVILAVSLEKKLQYATNYFLMSLAVADLLVGLFVMPIALLTIMFEAMWPLPLVLCPAWLFLDVLFSTASIWHLCAISVDRYIAIKKPIQANQYNSRATAFIKITVVWLISIGIAIPVPIKGIETDVDNPNNITCVLTKERFGDFMLFGSLAAFFTPLAIMIVTYFLTIHALQKKRLLSGSRQTISNEQRASKVLGIVFFLFLLMWCPFFITNITLVLCDSCNQTTLQMLLEIFVWIGYVSSGVNPLVYTLFNKTFRDAFGRY
>9WQ6_E1 len=339 copies=1 longest=1
LSLLHLQEKNWSALLTAVVIILTIAGNILVIMAVSLEKKLQNATNYFLMSLAIADMLLGFLVMPVSMLTILYGYRWPLPSKLCAVWIYLDVLFSTASIMHLCAISLDRYVAIQNPIHHSRFNSRTKAFLKIIAVWTISVGISMPIPVFGLQDDSKVFKEGSCLLADDNFVLIGSFVSFFIPLTIMVITYFLTIKSLQKEATLCVSDLGTRAKLASFSFLPQSSLSSEKLFQRSIHREPGSYTGRRTMQSISNEQKACKVLGIVFFLFVVMWCPFFITNIMAVICKESCNEDVIGALLNVFVWIGYLSSAVNPLVYTLFNKTYRSAFSRYIQCQYKENKK
>9WQA_E1 len=318 copies=1 longest=1
GPGSGSGTESIPEEMKQIVEEQGNKLHWAALLILMVIIPTIGGNTLVILAVSLEKKLQYATNYFLMSLAVADLLVGLFVMPIALLTIMFEAMWPLPLVLCPAWLFLDVLFSTASIWHLCAISVDRYIAIKKPIQANQYNSRATAFIKITVVWLISIGIAIPVPIKGIETDVDNPNNITCVLTKERFGDFMLFGSLAAFFTPLAIMIVTYFLTIHALQKKRLLSGSRQTISNEQRASKVLGIVFFLFLLMWCPFFITNITLVLCDSCNQTTLQMLLEIFVWIGYVSSGVNPLVYTLFNKTFRDAFGRYITCNYRATKSV
>9WQA_E2 len=213 copies=1 longest=0
DIVLIQSPAIMSASPGEKVTITCSASSSVSYMHWFQQKPGTSPKLWIYSTSNLASGVPARFSGSGSGTSYSLTISRMEAEDAATYYCQQRSSYPLTFGAGTKLEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSLPVTKSFNRGEC
>9WQA_E3 len=247 copies=1 longest=0
EVQLQQSGPELVKPGASVKLSCKASGYTFTSSWMHWVKQRPGQGLEWIGNIYPSNGGTNYNERFKSKATLTVDRSSNTAYMQLSSLTSEDSAVYFCARFGSFITTILTTYYNPVDYWGQGTTLTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCENLYFQSGSHHHHHHHH
>9WQB_E1 len=318 copies=1 longest=1
GPGSGSGTESIPEEMKQIVEEQGNKLHWAALLILMVIIPTIGGNTLVILAVSLEKKLQYATNYFLMSLAVADLLVGLFVMPIALLTIMFEAMWPLPLVLCPAWLFLDVLFSTASIWHLCAISVDRYIAIKKPIQANQYNSRATAFIKITVVWLISIGIAIPVPIKGIETDVDNPNNITCVLTKERFGDFMLFGSLAAFFTPLAIMIVTYFLTIHALQKKRLLSGSRQTISNEQRASKVLGIVFFLFLLMWCPFFITNITLVLCDSCNQTTLQMLLEIFVWIGYVSSGVNPLVYTLFNKTFRDAFGRYITCNYRATKSV
>9WQB_E2 len=213 copies=1 longest=0
DIVLIQSPAIMSASPGEKVTITCSASSSVSYMHWFQQKPGTSPKLWIYSTSNLASGVPARFSGSGSGTSYSLTISRMEAEDAATYYCQQRSSYPLTFGAGTKLEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSLPVTKSFNRGEC
>9WQB_E3 len=247 copies=1 longest=0
EVQLQQSGPELVKPGASVKLSCKASGYTFTSSWMHWVKQRPGQGLEWIGNIYPSNGGTNYNERFKSKATLTVDRSSNTAYMQLSSLTSEDSAVYFCARFGSFITTILTTYYNPVDYWGQGTTLTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCENLYFQSGSHHHHHHHH
>9WS8_E1 len=429 copies=1 longest=1
MAAQGYGYYRTVIFSAMFGGYSLYYFNRKTFSFVMPSLVEEIPLDKDDLGFITSSQSAAYAISKFVSGVLSDQMSARWLFSSGLLLVGLVNIFFAWSSTVPVFAALWFLNGLAQGLGWPPCGKVLRKWFEPSQFGTWWAILSTSMNLAGGLGPILATILAQSYSWRSTLALSGALCVVVSFLCLLLIHNEPADVGLRNLDPMPSEGKKGSLKEESTLQELLLSPYLWVLSTGYLVVFGVKTCCTDWGQFFLIQEKGQSALVGSSYMSALEVGGLVGSIAAGYLSDRAMAKAGLSNYGNPRHGLLLFMMAGMTVSMYLFRVTVTSDSPKLWILVLGAVFGFSSYGPIALFGVIANESAPPNLCGTSHAIVGLMANVGGFLAGLPFSTIAKHYSWSTAFWVAEVICAASTAAFFLLRNIRTKMGRVSKKAE
>9WTX_E1 len=551 copies=1 longest=1
MRDYDEVIAFLGEWGPFQRLIFFLLSASIIPNGFNGMSVVFLAGTPEHRCRVPDAANLSSAWRNNSVPLRLRDGREVPHSCSRYRLATIANFSALGLEPGRDVDLGQLEQESCLDGWEFSQDVYLSTVVTEWNLVCEDNWKVPLTTSLFFVGVLLGSFVSGQLSDRFGRKNVLFATMAVQTGFSFLQIFSISWEMFTVLFVIVGMGQISNYVVAFILGTEILGKSVRIIFSTLGVCTFFAVGYMLLPLFAYFIRDWRMLLLALTVPGVLCVPLWWFIPESPRWLISQRRFREAEDIIQKAAKMNNIAVPAVIFDSVEELNPLKQQKAFILDLFRTRNIAIMTIMSLLLWMLTSVGYFALSLDAPNLHGDAYLNCFLSALIEIPAYITAWLLLRTLPRRYIIAAVLFWGGGVLLFIQLVPVDYYFLSIGLVMLGKFGITSAFSMLYVFTAELYPTLVRNMAVGVTSTASRVGSIIAPYFVYLGAYNRMLPYIVMGSLTVLIGILTLFFPESLGMTLPETLEQMQKVKWFRSGKKTRDSMETEENPKVLITAF
>9WTY_E1 len=551 copies=1 longest=1
MRDYDEVIAFLGEWGPFQRLIFFLLSASIIPNGFNGMSVVFLAGTPEHRCRVPDAANLSSAWRNNSVPLRLRDGREVPHSCSRYRLATIANFSALGLEPGRDVDLGQLEQESCLDGWEFSQDVYLSTVVTEWNLVCEDNWKVPLTTSLFFVGVLLGSFVSGQLSDRFGRKNVLFATMAVQTGFSFLQIFSISWEMFTVLFVIVGMGQISNYVVAFILGTEILGKSVRIIFSTLGVCTFFAVGYMLLPLFAYFIRDWRMLLLALTVPGVLCVPLWWFIPESPRWLISQRRFREAEDIIQKAAKMNNIAVPAVIFDSVEELNPLKQQKAFILDLFRTRNIAIMTIMSLLLWMLTSVGYFALSLDAPNLHGDAYLNCFLSALIEIPAYITAWLLLRTLPRRYIIAAVLFWGGGVLLFIQLVPVDYYFLSIGLVMLGKFGITSAFSMLYVFTAELYPTLVRNMAVGVTSTASRVGSIIAPYFVYLGAYNRMLPYIVMGSLTVLIGILTLFFPESLGMTLPETLEQMQKVKWFRSGKKTRDSMETEENPKVLITAF
>9WWG_E1 len=545 copies=1 longest=1
GSRPLVNFPEKISGDRFMSSFTHLDQKVHEGYAKEIQSLKDEVKNMLLKECKAVDKLYLIDNIERLGLSYHFEDEIDEQLQLLFDQNESIKDDRYSDLSTIALRFRLFRQHGYPISSDVFKQFTFGNGTFKTALQSDSRGILNLYDAAYVSTRNDVILDEAVRFTTDKLQRQSTRGAYIDKLVKHALEQSLHRGVPRLEARHFIDAYEHDEMKNVSLLKLAKLDFNFLQLLYKQELNESLRWWKEIDAATNVPYARNRLVESYFWALVMFYEPKYSKARLMVTKYLATGTFIDDTYDAYGSRDELEIFTSAIQRWDYSEADRLPGYMKKAYTIFLNTYVEFEEYANQQGRPYAFHYLQEGWKEYSRSSNMQSRWFFNPEELPLFDEYMVHGLITGAGRVNACVCALLMESGTEDAFQWLFQYPPILYAGNTIYRLVNDLAGYQVEKETGIPNSIDCYMKEFKVSEEEAKMRLTEMVENAWKDFNEEYMNRRKQTTPPIGEEIWTLLINQTRIMDVAYKNGVDGWTNSQDSLKHFLNGLLVDPIPI
>9WWI_E1 len=545 copies=1 longest=1
GSRPLVNFPEKISGDRFMSSFTHLDQKVHEGYAKEIQSLKDEVKNMLLKECKAVDKLYLIDNIERLGLSYHFEDEIDEQLQLLFDQNESIKDDRYSDLSTIALRFRLFRQHGYPISSDVFKQFTFGNGTFKTALQSDSRGILNLYDAAYVSTRNDVILDEAVRFTTDKLQRQSTRGAYIDKLVKHALEQSLHRGVPRLEARHFIDAYEHDEMKNVSLLKLAKLDFNFLQLLYKQELNESLRWWKEIDAATNVPYARNRLVESYFWALVMFYEPKYSKARLMVTKYLATGTFIDDTYDAYGSRDELEIFTSAIQRWDYSEADRLPGYMKKAYTIFLNTYVEFEEYANQQGRPYAFHYLQEGWKEYSRSSNMQSRWFFNPEELPLFDEYMVHGLITGAGRVNACVCALLMESGTEDAFQWLFQYPPILYAGNTIYRLVNDLAGYQVEKETGIPNSIDCYMKEFKVSEEEAKMRLTEMVENAWKDFNEEYMNRRKQTTPPIGEEIWTLLINQTRIMDVAYKNGVDGWTNSQDSLKHFLNGLLVDPIPI
>9WX7_E1 len=636 copies=1 longest=1
MKKLQGAHLRKPVTPDLLMTPSDQGDVDLDVDFAAHRGNWTGKLDFLLSCIGYCVGLGNVWRFPYRAYTNGGGAFLVPYFLMLAICGIPLFFLELSLGQFSSLGPLAVWKISPLFKGAGAAMLLIVGLVAIYYNMIIAYVLFYLFASLTSDLPWEHCGNWWNTELCLEHRVSKDGNGALPLNLTCTVSPSEEYWSRYVLHIQGSQGIGSPGEIRWNLCLCLLLAWVIVFLCILKGVKSSGKVVYFTATFPYLILLMLLVRGVTLPGAWKGIQFYLTPQFHHLLSSKVWIEAALQIFYSLGVGFGGLLTFASYNTFHQNIYRDTFIVTLGNAITSILAGFAIFSVLGYMSQELGVPVDQVAKAGPGLAFVVYPQAMTMLPLSPFWSFLFFFMLLTLGLDSQFAFLETIVTAVTDEFPYYLRPKKAVFSGLICVAMYLMGLILTTDGGMYWLVLLDDYSASFGLMVVVITTCLAVTRVYGIQRFCRDIHMMLGFKPGLYFRACWLFLSPATLLALMVYSIVKYQPSEYGSYRFPPWAELLGILMGLLSCLMIPAGMLVAVLREEGSLWERLQQASRPAMDWGPSLEENRTGMYVATLAGSQSPKPLMVHMRKYGGITSFENTAIEVDREIAEEEESMM
>9WX9_E1 len=636 copies=1 longest=1
MKKLQGAHLRKPVTPDLLMTPSDQGDVDLDVDFAAHRGNWTGKLDFLLSCIGYCVGLGNVWRFPYRAYTNGGGAFLVPYFLMLAICGIPLFFLELSLGQFSSLGPLAVWKISPLFKGAGAAMLLIVGLVAIYYNMIIAYVLFYLFASLTSDLPWEHCGNWWNTELCLEHRVSKDGNGALPLNLTCTVSPSEEYWSRYVLHIQGSQGIGSPGEIRWNLCLCLLLAWVIVFLCILKGVKSSGKVVYFTATFPYLILLMLLVRGVTLPGAWKGIQFYLTPQFHHLLSSKVWIEAALQIFYSLGVGFGGLLTFASYNTFHQNIYRDTFIVTLGNAITSILAGFAIFSVLGYMSQELGVPVDQVAKAGPGLAFVVYPQAMTMLPLSPFWSFLFFFMLLTLGLDSQFAFLETIVTAVTDEFPYYLRPKKAVFSGLICVAMYLMGLILTTDGGMYWLVLLDDYSASFGLMVVVITTCLAVTRVYGIQRFCRDIHMMLGFKPGLYFRACWLFLSPATLLALMVYSIVKYQPSEYGSYRFPPWAELLGILMGLLSCLMIPAGMLVAVLREEGSLWERLQQASRPAMDWGPSLEENRTGMYVATLAGSQSPKPLMVHMRKYGGITSFENTAIEVDREIAEEEESMM
>9X2N_E1 len=534 copies=1 longest=1
MANLLKSLAAVSSMTMFSRVLGFARDAIVARIFGAGMATDAFFVAFKLPNLLRRIFAEGAFSQAFVPILAEYKSKQGEDATRVFVSYVSGLLTLALAVVTVAGMLAAPWVIMVTAPGFADTADKFALTSQLLKITFPYILLISLASLVGAILNTWNRFSIPAFAPTLLNISMIGFALFAAPYFNPPVLALAWAVTVGGVLQLVYQLPHLKKIGMLVLPRINFHDAGAMRVVKQMGPAILGVSVSQISLIINTIFASFLASGSVSWMYYADRLMEFPSGVLGVALGTILLPSLSKSFASGNHDEYNRLMDWGLRLCFLLALPSAVALGILSGPLTVSLFQYGKFTAFDALMTQRALIAYSVGLIGLIVVKVLAPGFYSRQDIKTPVKIAIVTLILTQLMNLAFIGPLKHAGLSLSIGLAACLNASLLYWQLRKQKIFTPQPGWMAFLLRLVVAVLVMSGVLLGMLHIMPEWSLGTMPWRLLRLMAVVLAGIAAYFAALAVLGFKVKEFARRTVDYKDHDGDYKDHDIDYKDDDDK
>9X2N_E2 len=65 copies=1 longest=0
MHHHHHHHHHHSSTLCRWAVKALRCTRVYKEFIWKPLVALSYVTLYLLSSVFLSQLSYPIGSWAV
>9X35_E1 len=170 copies=1 longest=0
PKAVVFLEPQWYRVLEKDSVTLKCQGAYSPEDNSTQWFHNESLISSQASSYFIDAATVDDSGEYRCQTNLSTLSDPVQLEVHIGWLLLQAPRWVFKEEDPIHLRCHSWKNTALHKVTYLQNGKGRKYFHHNSDFYIPKATLKDSGSYFCRGLFGSKNVSSETVNITITQG
>9X35_E2 len=213 copies=1 longest=0
LGGPDVFLLPPKPKDTLMISRTPEVTCVVVDVSHEDPEVKFNWYVDGVEVHNAKTKPAEEQYNATLRVVSVLTVLHQDWLNGKEYKCKVSNKALPAPIAATISKAKGQPREPQVYTLPPSRDELTKNQVSLTCLVKGFYPSDIAVEWESNGQPENNYKTTPPVLDSDGSFFLYSKLTVDKSRWQQGNVFSCSVMHEALHNHYTQKSLSLSPGK
>9X35_E3 len=215 copies=1 longest=1
ELLGGPDVFLLPPKPKDTLMISRTPEVTCVVVDVSHEDPEVKFNWYVDGVEVHNAKTKPAEEQYNATLRVVSVLTVLHQDWLNGKEYKCKVSNKALPAPIAATISKAKGQPREPQVYTLPPSRDELTKNQVSLTCLVKGFYPSDIAVEWESNGQPENNYKTTPPVLDSDGSFFLYSKLTVDKSRWQQGNVFSCSVMHEALHNHYTQKSLSLSPGK
>9X3S_E1 len=163 copies=2 longest=0
MFVDDVTRAFESGDFARPNLFQVEISYLGQNFTFQCKATALPAGIVEKIPVGFMNRKINVAGDRTFDDWTVTVMNDEAHDARQKFVDWQSIAAGQGNEITGGKPAEYKKSAIVRQYARDAKTVTKEIEIKGLWPTNVGELQLDWDSNNEIQTFEVTLALDYWE
>9X3S_E2 len=321 copies=1 longest=0
MYSLEEFNNQAINADFQRNNMFSCVFATTPSTKSSSLISSISNFSYNNLGLNSDWLGLTQGDINQGITTLITAGTQKLIRKSGVSKYLIGAMSQRTVQSLLGSFTVGTYLIDFFNMAYNSSGLMIYSVKMPENRLSYETDWNYNSPNIRITGRELDPLVISFRMDSEACNYRAMQDWVNSVQDPVTGLRALPQDVEADIQVNLHSRNGLPHTAVMFTGCIPISVSAPELSYDGDNQITTFDVTFAYRVMQAGAVDRQRALEWLESAAINGIQSVLGNSGGVTGLSNSLSRLSRLGGTAGSISNINTMTGIVNSQSKILGAI
>9X3S_E3 len=364 copies=1 longest=1
MAIVKEITADLIKKSGEKISAGQSTKSEVGTKTYTAQFPTGRASGNDTTEDFQVTDLYKNGLLFTAYNMSSRDSGSLRSMRSNYSSSSSSILRTARNTISSTVSKLSNGLISNNNSGTISKSPIANILLPRSKSDVDTSSHRFNDVQESLISRGGGTATGVLSNIASTAVFGALESITQGIMADNNEQIYTTARSMYGGAENRTKVFTWDLTPRSTEDLMAIINIYQYFNYFSYGETGKSQYAAEIKGYLDDWYRSTLIEPLSPEDAAKNKTLFEKMTSSLTNVLVVSNPTVWMVKNFGATSKFDGKTEIFGPCQIQSIRFDKTPNGNFNGLAIAPNLPSTFTLEITMREIITLNRASLYAGTF
>9X3Y_E1 len=325 copies=1 longest=1
MDGSNVTSFVVEEPTNISTGRNASVGNAHRQIPIVHWVIMSISPVGFVENGILLWFLCFRMRRNPFTVYITHLSIADISLLFCIFILSIDYALDYELSSGHYYTIVTLSVTFLFGYNTGLYLLTAISVERCLSVLYPIWYRCHRPKYQSALVCALLWALSCLVTTMEYVMCIDREEESHSRNDCRAVIIFIAILSFLVFTPLMLVSSTILVVKIRKNTWASHSSKLYIVIMVTIIIFLIFAMPMRLLYLLYYEYWSTFGNLHHISLLFSTINSSANPFIYFFVGSSKKKRFKESLKVVLTRAFKDEMQPRRQKDNCNTVTVETVV
>9X3Y_E2 len=8 copies=1 longest=0
FLFQPQRF
>9X41_E1 len=325 copies=1 longest=1
MDGSNVTSFVVEEPTNISTGRNASVGNAHRQIPIVHWVIMSISPVGFVENGILLWFLCFRMRRNPFTVYITHLSIADISLLFCIFILSIDYALDYELSSGHYYTIVTLSVTFLFGYNTGLYLLTAISVERCLSVLYPIWYRCHRPKYQSALVCALLWALSCLVTTMEYVMCIDREEESHSRNDCRAVIIFIAILSFLVFTPLMLVSSTILVVKIRKNTWASHSSKLYIVIMVTIIIFLIFAMPMRLLYLLYYEYWSTFGNLHHISLLFSTINSSANPFIYFFVGSSKKKRFKESLKVVLTRAFKDEMQPRRQKDNCNTVTVETVV
>9XBS_E1 len=577 copies=1 longest=1
MNHKVHHHHHHIEGRHMTLSHETVLTPEQRARLAADPDLGGGNLLTKAIEANPHPELPFIHLGRPLTVPSGEQRTELSLLDLDELVQSWSVWYLKQGVRPRDRVAIYLHDSFAYSVHFYALAQIGAVAVLVNSKASRYIATELCRQTNPVGVYTDLDHLEILGEEFHLLPGLRWTQVAEELPAPPPAKLPQEARFRHVDEDPVSILHSSGTTGRPKPVIQTHRSCVAGPRFRLVDHHEQPGAIMMTALPQSHLGCIAYSTYAVLGGTPLVPWYDTSGPELAKAVEKYRPTTVMAFGHAYAELAAADLPAGAIDSVNVWISIGDAVHEKHIKTILGMRSADRAPASFFDRLGTTELGWGVLLKVRTLADERNDRCVGKPVGVAEVAVLRRDGTEADVNEVGLLGAKGPAITAGYWSDSDTTYRSKLSGFWLTGDMAYRDEAGNYFQVDRAVDAIETPTGTGYSVFMEELMLNELPEVLDVAVVAGIHRGRTAPVAVVTSSAARPDAQKLLNEANEALRAAGHPELTMLEVARSEEDFPVGVTGKVLKRRLREKYSSLSTYIREGGGKSIGTILNDVFV
>9XCB_E1 len=452 copies=1 longest=1
MDYKDDDDKASRATMAAQGYGYYRTVIFSAMFGGYSLYYFNRKTFSFVMPSLVEEIPLDKDDLGFITSSQSAAYAISKFVSGVLSDQMSARWLFSSGLLLVGLVNIFFAWSSTVPVFAALWFLNGLAQGLGWPPCGKVLRKWFEPSQFGTWWAILSTSMNLAGGLGPILATILAQSYSWRSTLALSGALCVVVSFLCLLLIHNEPADVGLRNLDPMPSEGKKGSLKEESTLQELLLSPYLWVLSTGYLVVFGVKTCCTDWGQFFLIQEKGQSALVGSSYMSALEVGGLVGSIAAGYLSDRAMAKAGLSNYGNPRHGLLLFMMAGMTVSMYLFRVTVTSDSPKLWILVLGAVFGFSSYGPIALFGVIANESAPPNLCGTSHAIVGLMANVGGFLAGLPFSTIAKHYSWSTAFWVAEVICAASTAAFFLLRNIRTKMGRVSKKAESNSLEVLFQ
>9XCH_E1 len=666 copies=1 longest=1
MQNLDEILKKLRKDLKKTSKETTEKLSVLSKKVAEDVVKVSNQTVQKINEVTKKFTVDTTVSTIDTQKILDDFMKNIRFSRFKQIWQAIEKTPKSKHILLKSLFLKGLLTHNKPLLEEIIRQIELIPYSSDLEMLGAFSQDKAHPLSPELLEFVQEMLANESEKVLLTILMNAFQSIPELSLDSKTGTLSMKTKKALLPLLIEKVDTSIAKSIMSQLTDISFDSVLPAFRPSIGDPSYQKTNINLNKYLSLVGNKTDISEFLILTIGLFTSLKIGDKGFLQELSADYLFVRYDDCLHKIIEKLKEKEVIEQLGEKEVQKILEASGNLKRFKPTSNNPRRFFETRLAQYINGLSHSEKTIEIDSIGEDKEPEDEELRDNTLKACNKILQFFLGDCGRVDTPREMEQKICIFANGYKDTSKFTKKEVEFITGVLHHAGAHKGRDRVERDKLTGIKERKFETDSEKVGSDVWNCGGGVMKGCSPVFYDAQRDPMRNMLVDSSTVAPDNKEGKQWFWNNNRQYSSYVGSISGHTCNIVGMLAKYMTEYKEDLDLQNDINLFLIQVIGVYAKRGFHAMLEVIDVLHDPYVQDIFKGYGVQVNLYSYFKENPELAGFLQHAMNDATTYTQALVNKKHVKEALKSNSFFSKKDEESGDKKNDHTPKDGTVYGC
>9XCH_E2 len=269 copies=1 longest=0
MIKMGRSEMKIASAELRELMKAVSEGHYETVNTILDKDPELVNQYAPPTYDSPLARVLNKKHIDYKMLDILVKHHVDFDYPINYHKETPIELACKNQDLQLFKYLVQHNAPISEQAPHFLLVNSTNIKYLTEDKIKNTCEIIKLMGGLEAVSSKCDAEGNRFGEQARKSQLINRFGGIVKYDYMQLLQSVYPIVDREVDAPTIHGSTEVLTNLLNKIRGQFSSKETYDQQNLKDSISLFFMTGGEIPPSRKVPESRFEEAGIDTPKNAL
>9XD0_E1 len=401 copies=1 longest=1
MANPFSRWFLSERPPNCHVADLETSLDPHQTLLKVQKYKPALSDWVHYIFLGSIMLFVFITNPAPWIFKILFYCFLGTLFIIPATSQFFFNALPILTWVALYFTSSYFPDDRRPPITVKVLPAVETILYGDNLSDILATSTNSFLDILAWLPYGLFHFGAPFVVAAILFVFGPPTVLQGYAFAFGYMNLFGVIMQNVFPAAPPWYKILYGLQSANYDMHGSPGGLARIDKLLGINMYTTAFSNSSVIFGAFPSLHSGCATMEALFFCYCFPKLKPLFIAYVCWLWWSTMYLTHHYFVDLMAGSVLSYVIFQYTKYTHLPIVDTSLFCRWSYTSIEKYDISKSDPLAADSNDIESVPLSNLELDFDLNMTDEPSVSPSLFDGSTSVSRSSATSITSLGVKRA
>9XD0_E2 len=221 copies=1 longest=0
MRSSLLTLPKSFLGFMPLYLAVEIVLGISILNKCSGAYGILALFTGHPLDFMQWIAYLWSVFTLIVFSQGLYLIHKPNLLVFSQICVLYTIDTISTCFFTLWFTTQWFTLEDTANIDGNNALQSNPISTGKLTERGIDISKQSATESYEYTMTILITLVSLIFRFYFNFILASFVQELLHHPKYLVDRDDVEQNLKNKPIWKRLWAKSQKGCYKLCKNLLE
>9XD0_E3 len=9 copies=1 longest=0
VLXXVFFPI
>9XFJ_E1 len=361 copies=1 longest=1
MSPECARAAGDAPLRSLEQANRTRFPFFSDVKGDHRLVLAAVETTVLVLIFAVSLLGNVCALVLVARRRRRGATACLVLNLFCADLLFISAIPLVLAVRWTEAWLLGPVACHLLFYVMTLSGSVTILTLAAVSLERMVCIVHLQRGVRGPGRRARAVLLALIWGYSAVAALPLCVFFRVVPQRLPGADQEISICTLIWPTIPGEISWDVSFVTLNFLVPGLVIVISYSKILQITKASRKRLTVSLAYSESHQIRVSQQDFRLFRTLFLLMVSFFIMWSPIIITILLILIQNFKQDLVIWPSLFFWVVAFTFANSALNPILYNMTLCRNEWKKIFCCFWFPEKGAILTDTSVKRNDLSIISG
>9XFJ_E2 len=354 copies=1 longest=0
MGCTLSAEDKAAVERSKMIDRNLREDGEKAAREVKLLLLGAGESGKSTIVKQMKIIHEAGYSEEECKQYKAVVYSNTIQSIIAIIRAMGRLKIDFGDSARADDARQLFVLAGAAEEGFMTAELAGVIKRLWKDSGVQACFNRSREYQLNDSAAYYLNDLDRIAQPNYIPTQQDVLRTRVKTTGIVETHFTFKDLHFKMFDVGAQRSERKKWIHCFEGVTAIIFCVALSDYDLVLAEDEEMNRMHESMKLFDSICNNKWFTDTSIILFLNKKDLFEEKIKKSPLTICYPEYAGSNTYEEAAAYIQCQFEDLNKRKDTKEIYTHFTCSTDTENIRFVFAAVKDTILQLNLKEYNLV
>9XGW_E1 len=127 copies=1 longest=0
QVQLVESGGGLVQPGRSLKLSCAASGFTFDNYAMHWVRQAPGKGLEWVSGSSWNSGTIGYADSVKGRFIISRDNAKNSLHLQMNRLRVEDTALYYCAKDLSGDEYDYDSSGLGFDYWGQGTLVTVSS
>9XGW_E2 len=109 copies=1 longest=0
EIVLTQSPGTLSLSPGERATLSCRASQSVSSSYLAWYQQKPGQAPRLLIYGASSRATGIPDRFSGSGSGTDFTLTISRLEPEDFAVYYCQQYGSSLLFTFGPGTKVEIK
>9XGW_E3 len=184 copies=1 longest=1
TNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFE
>9XGX_E1 len=117 copies=1 longest=0
EVQLVESGGGLIQPGGSLRLSCAASGFTVSSNYMSWVRQAPGKGLEWVSVIFAGGSTFYADSVKGRFTISRDNSKNTLFLQLNSLRAEDTAVYYCAREAILRGTFDWGQGTLVTVSS
>9XGX_E2 len=109 copies=1 longest=0
EIVLTQSPVTLSVSPGERATLSCRASQSVSSNLAWYQQKPGQAPRLLIYGASTRATGIPARFSGSGSGTEFTLTISSLQSEDFAVYYCQQYNNWPPGYTFGQGTKLEIK
>9XGX_E3 len=199 copies=1 longest=1
TNLCPFGEVFNATRFASVYAWNRKRISNCVADYSVLYNSASFSTFKCYGVSPTKLNDLCFTNVYADSFVIRGDEVRQIAPGQTGKIADYNYKLPDDFTGCVIAWNSNNLDSKVGGNYNYLYRLFRKSNLKPFERDISTEIYQAGSTPCNGVEGFNCYFPLQSYGFQPTNGVGYQPYRVVVLSFELLHAPATVCGPKKST
>9XI8_E1 len=228 copies=1 longest=0
QVQLVQSGAEVKKPGSSVKVSCKASGGSFTNHFISWVRQAPGQGLEWMGRIIPILGTANYAQNFQGRVMMTADKSTSTAYMELSSLRSEDTAVYYCARDSGYSGYGSTYYMDVWGKGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCD
>9XI8_E2 len=217 copies=1 longest=0
QSVLTQPPSVSGAPGQRVTISCTGSNSNIGAGYDVHWYQQLPGAAPKLLIYGNNNRPSGVPDRFSGSKSDTSASLAITGLQVEDEADYYCQSYDSSLSDSVFGSGTKVTVLGQPKAAPSVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADSSPVKAGVETTTPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTVAPTECS
>9XI8_E3 len=1256 copies=1 longest=1
MKILILAFLASLAKAQEGCGIISRKPQPKMAQVSSSRRGVYYNDDIFRSDVLHLTQDYFLPFDSNLTQYFSLNVDSDRYTYFDNPILDFGDGVYFAATEKSNVIRGWIFGSSFDNTTQSAVIVNNSTHIIIRVCNFNLCKEPMYTVSRGTQQNSWVYQSAFNCTYDRVEKSFQLDTAPKTGNFKDLREYVFKNRDGFLSVYQTYTAVNLPRGLPQGFSVLRPILKLPFGINITSYRVVMAMFSQSTSNFLPESAAYYVGNLKYSTFMLSFNENGTITDAIDCSQNPLAELKCTIKNFNVNKGIYQTSNFRVSPTQEVVRFPNITNRCPFDKVFNATRFPNVYAWERTKISDCVADYTVLYNSTSFSTFKCYGVSPSKLIDLCFTSVYADTFLIRSSEVRQVAPGETGVIADYNYKLPDDFTGCVIAWNTAKQDTGNYYYRSHRKTKLKPFERDLSSDDGNGVYTLSTYDFNPNVPVAYQATRVVVLSFELLNAPATVCGPKLSTQLVKNQCVNFNFNGLKGTGVLTPSLKRFQSFQQFGRDTSDFTDSVRDPQTLEILDISPCSFGGVSVITPGTNASSEVAVLYQDVNCTDVPTAIRADQLTPAWRVYSTGLNVFQTQAGCLIGAEHVNASYECDIPIGAGICASYHTASVLRSTGQKSIVAYTMSLGAENSIAYANNSIAIPTNFSISVTTEVMPVSMAKTSVDCTMYICGDSLECSNLLLQYGSFCTQLNRALTGIAIEQDKNTQEVFAQVKQMYKTPAIKDFGGFNFSQILPDPSKPTKRSFIEDLLFNKVTLADAGFMKQYGECLGDISARDLICAQKFNGLTVLPPLLTDEMIAAYTAALVSGTATAGWTFGAGAALQIPFAMQMAYRFNGIGVTQNVLYENQKQIANQFNKAISQIQESLTTTSTALGKLQDVVNQNAQALNTLVKQLSSNFGAISSVLNDILSRLDKVEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQAAPHGVVFLHVTYVPSQERNFTTAPAICHEGKAYFPREGVFVSNGTSWFITQRNFYSPQLITTDNTFVSGNCDVVIGIINNTVYDPLQPELDSFKDELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLLEVLFQGPGHHHHHHHHSAWSHPQFEKGGGSGGGGSGGSAWSHPQFEKSA
>9XJK_E1 len=188 copies=1 longest=0
HHHHHHGAAASIQTTVNTLSERISSKLEQEANASAQTKCDIEIGNFYIRQNHGCNITVKNMCSADADAQLDAVLSAATETYSGLTPEQKAYVPAMFTAALNIQTSVNTVVRDFENYVKQTCNSSAVVDNKLKIQNVIIDECYGAPGSPTNLEFINTGSSKGNCAIKALMQLTTKATTQIAPRQVAGTG
>9XJK_E2 len=225 copies=1 longest=1
EVQLVESGGGLAKPGGSLRLSCAASGFTLGGYAMHWVRQAPGKGLEWVSVIISGGTTYYADSVQGRFTISRDNSKNTLSLQMNSLRPEDTAVYYCTLRVAATESYYSGLDSWGQGVVVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSC
>9XJK_E3 len=218 copies=1 longest=0
QPVLTQPPSVSGAPGQRVTISCTGSSSNIGAGYGVQWYQQLPGTAPKLLIYENNKRPSGVSDRFSGSQSGTSASLTITGLQSEDEADYYCQSYDSSLSAHVLFGGGTRLTVLGQPKANPTVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADGSPVKAGVETTKPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTVAPTECS
>9XKO_E1 len=397 copies=1 longest=1
MKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQTNSSSNNNNNNNNNNLGIEGRISHMENLYFQ
>9XMM_E1 len=595 copies=1 longest=1
FNLDVDSPAEYSGPEGSYFGFAVDFFVPSASSRMFLLVGAPKANTTQPGIVEGGQVLKCDWSSTRRCQPIEFDATGNRDYAKDDPLEFKSHQWFGASVRSKQDKILACAPLYHWRTEMKQEREPVGTCFLQDGTKTVEYAPCRSQDIDADGQGFCQGGFSIDFTKADRVLLGGPGSFYWQGQLISDQVAEIVSKYDPNVYSIKYNNQLATRTAQAIFDDSYLGYSVAVGDFNGDGIDDFVSGVPRAARTLGMVYIYDGKNMSSLYNFTGEQMAAYFGFSVAATDINGDDYADVFIGAPLFMDRGSDGKLQEVGQVSVSLQRASGDFQTTKLNGFEVFARFGSAIAPLGDLDQDGFNDIAIAAPYGGEDKKGIVYIFNGRSTGLNAVPSQILEGQWAARSGCPPSFGYSMKGATDIDKNGYPDLIVGAFGVDRAILYRARPVITVNAGLEVYPSILNQDNKTCSLPGTALKVSCFNVRFCLKADGKGVLPRKLNFQVELLLDKLKQKGAIRRALFLYSRSPSHSKNMTISRGGLMQCEELIAYLRDESEFRDKLTPITIFMEYRLDYRTAADTTGLQPILNQFTPANISRQAHILL
>9XMM_E2 len=474 copies=1 longest=0
HVQGGCALGGAETCEDCLLIGPQCAWCAQENFTHPSGVGERCDTPANLLAKGCQLNFIENPVSQVEILKNKPLSVGRQKNSSDIVQIAPQSLILKLRPGGAQTLQVHVRQTEDYPVDLYYLMDLSASMDDDLNTIKELGSRLSKEMSKLTSNFRLGFGSFVEKPVSPFVKTTPEEIANPCSSIPYFCLPTFGFKHILPLTNDAERFNEIVKNQKISANIDTPEGGFDAIMQAAVCKEKIGWRNDSLHLLVFVSDADSHFGMDSKLAGIVCPNDGLCHLDSKNEYSMSTVLEYPTIGQLIDKLVQNNVLLIFAVTQEQVHLYENYAKLIPGATVGLLQKDSGNILQLIISAYEELRSEVELEVLGDTEGLNLSFTAICNNGTLFQHQKKCSHMKVGDTASFSVTVNIPHCERRSRHIIIKPVGLGDALELLVSPECNCDCQKEVEVNSSKCHHGNGSFQCGVCACHPGHMGPRCE
>9XMM_E3 len=9 copies=1 longest=0
AWRGDXCNP
>9XQB_E1 len=127 copies=1 longest=0
DIQLTQSPAVLSVSPGERVSFSCRASQTIGTSIHWYQQRTNGSPRLLVKFASESISGIPSRFSGSGSGTDFTLTINSVESEDIADYYCQQSNSWPYTFGGGTKLEIKRADAAPTVSKGEFQHTGGRY
>9XQB_E2 len=431 copies=1 longest=1
DYKDDDDKPIMGSSVYITVELAIAVLAILGNVLVCWAVWLNSNLQNVTNYFVVSLAAADIAVGVLAIPFAITISTGFCAACHGCLFIACFVLVLEQSRIFSLLAIAIDRYIAIRIPLRYNGLVTGTRAKGIIAICWVLSFAIGLTPMLGWNNCGQPKEGKQHSQGCGEGQVACLFEDVVPMNYMVYFNFFACVLVPLLLMLGVYLRIFLAARRQLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLERARSTLQKEVHAAKSLAIIVGLFALCWLPLHIINCFTFFCPDCSHAPLWLMYLAIVLSHTNSVVNPFIYAYRIREFRQTFRKIIRSHVLRQQEPFKAGGSGSGLEVLFQ
>9XQB_E3 len=136 copies=1 longest=0
EVKLVESGPELKKPGETVKISCKASGYSFTNYGINWMKQAPGEGLEWMGWINTYTGEATYDDDFKGRFAFSLETSASTAYLQIINLKNEDMATYFCSRRGAYYRYDGYFYTMDFWGQGTSVTVSSKGEFQHTGGRY
>9XQN_E1 len=422 copies=1 longest=1
ASGRQTVDEALKDAQTSQEGNSTSLQEGLQDLIHTATLVTCTFLLAVIFCLGSYGNFIVFLSFFDPAFRKFRTNFDFMILNLSFCDLFICGVTAPMFTFVLFFSSASSIPDAFCFTFHLTSSGFIIMSLKTVAVIALHRLRMVLGKQPNRTASFPCTVLLTLLLWATSFTLATLATLKTSKSHLCLPMSSLIAGKGKAILSLYVVDFTFCVAVVSVSYIMIAQTLRKNAERARSTLADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKASGSGSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLERARSTLAKDSKAVVTCVIIVLSVLVCCLPLGISLVQVVLSSNGSFILYQFELFGFTLIFFKSGLNPFIYSRNSAGLRRKVLWCLQS
>9XQN_E2 len=107 copies=1 longest=0
SDIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQYLYYSLVTFGQGTKVE
>9XQN_E3 len=121 copies=1 longest=0
VQLVESGGGLVQPGGSLRLSCAASGFNVVVFSIHWVRQAPGKGLEWVAYISSSSGSTSYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARWGYWPGEPWWKAFDYWGQGTLVTV
>9XR3_E1 len=485 copies=2 longest=1
MGRYSGKTCRLLFMLVLTVAFFVAELVSGYLGNSIALLSDSFNMLSDLISLCVGLSAGYIARRPTRGFSATYGYARAEVVGALSNAVFLTALCFTIFVEAVLRLARPERIDDPELVLIVGVLGLLVNVVGLLIFQDCAAWFACCLRGRSRRLQQRQQLAEGCVPGAFGGPQGAEDPRRAADPTAPGSDSAVTLRGTSVERKREKGATVFANVAGDSFNTQNEPEDMMKKEKKSEALNIRGVLLHVMGDALGSVVVVITAIIFYVLPLKSEDPCNWQCYIDPSLTVLMVIIILSSAFPLIKETAAILLQMVPKGVNMEELMSKLSAVPGISSVHEVHIWELVSGKIIATLHIKYPKDRGYQDASTKIREIFHHAGIHNVTIQFENVDLKEPLEQKDLLLLCNSPCISKGCAKQLCCPPGALPLAHVNGCAEHNGGPSLDTYGSDGLSRRDAREVAIEVSLDSCLSDHGQSLNKTQEDQCYVNRTHF
>9XR4_E1 len=485 copies=2 longest=1
MGRYSGKTCRLLFMLVLTVAFFVAELVSGYLGNSIALLSDSFNMLSDLISLCVGLSAGYIARRPTRGFSATYGYARAEVVGALSNAVFLTALCFTIFVEAVLRLARPERIDDPELVLIVGVLGLLVNVVGLLIFQDCAAWFACCLRGRSRRLQQRQQLAEGCVPGAFGGPQGAEDPRRAADPTAPGSDSAVTLRGTSVERKREKGATVFANVAGDSFNTQNEPEDMMKKEKKSEALNIRGVLLHVMGDALGSVVVVITAIIFYVLPLKSEDPCNWQCYIDPSLTVLMVIIILSSAFPLIKETAAILLQMVPKGVNMEELMSKLSAVPGISSVHEVHIWELVSGKIIATLHIKYPKDRGYQDASTKIREIFHHAGIHNVTIQFENVDLKEPLEQKDLLLLCNSPCISKGCAKQLCCPPGALPLAHVNGCAEHNGGPSLDTYGSDGLSRRDAREVAIEVSLDSCLSDHGQSLNKTQEDQCYVNRTHF
>9XZG_E1 len=443 copies=1 longest=1
SMAGEGDQQDAAHNMGNHLPLLPAESEEEDEMEVEDQDSKEAKKPNIINFDTSLPTSHTYLGADMEEFHGRTLHDDDSCQVIPVLPQVMMILIPGQTLPLQLFHPQEVSMVRNLIQKDRTFAVLAYSNVQEREAQFGTTAEIYAYREEQDFGIEIVKVKAIGRQRFKVLELRTQSDGIQQAKVQILPECVLPSTMSAVQLESLNKCQIFPSKPVSREDQCSYKWWQKYQKRKFHCANLTSWPRWLYSLYDAETLMDRIKKQLREWDENLKDDSLPSNPIDFSYRVAACLPIDDVLRIQLLKIGSAIQRLRCELDIMNKCTSLCCKQCQETEITTKNEIFSLSLCGPMAAYVNPHGYVHETLTVYKACNLNLIGRPSTEHSWFPGYAWTVAQCKICASHIGWKFTATKKDMSPQKFWGLTRSALLPTIPDTEDEISPDKVILCL
>9XZG_E2 len=323 copies=1 longest=0
GPGGSNPNYCFAGKTSSISDLKEVPRKNITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDELDFLMEALIISKFNHQNIVRCIGVSLQSLPRFILLELMAGGDLKSFLRETRPRPSQPSSLAMLDLLHVARDIACGCQYLEENHFIHRDIAARNCLLTCPGPGRVAKIGDFGMARDIYRASYYRKGGCAMLPVKWMPPEAFMEGIFTSKTDTWSFGVLLWEIFSLGYMPYPSKSNQEVLEFVTSGGRMDPPKNCPGPVYRIMTQCWQHQPEDRPNFAIILERIEYCTQDPDVINTALPIEYGPLVEEEEK
>9XZK_E1 len=776 copies=1 longest=1
MSSTRSQNPHGLKQIGLDQIWDDLRAGIQQVYTRQSMAKSRYMELYTHVYNYCTSVHQSNQARGAGVPPSKSKKGQTPGGAQFVGLELYKRLKEFLKNYLTNLLKDGEDLMDESVLKFYTQQWEDYRFSSKVLNGICAYLNRHWVRRECDEGRKGIYEIYSLALVTWRDCLFRPLNKQVTNAVLKLIEKERNGETINTRLISGVVQSYVELGLNEDDAFAKGPTLTVYKESFESQFLADTERFYTRESTEFLQQNPVTEYMKKAEARLLEEQRRVQVYLHESTQDELARKCEQVLIEKHLEIFHTEFQNLLDADKNEDLGRMYNLVSRIQDGLGELKKLLETHIHNQGLAAIEKCGEAALNDPKMYVQTVLDVHKKYNALVMSAFNNDAGFVAALDKACGRFINNNAVTKMAQSSSKSPELLARYCDSLLKKSSKNPEEAELEDTLNQVMVVFKYIEDKDVFQKFYAKMLAKRLVHQNSASDDAEASMISKLKQACGFEYTSKLQRMFQDIGVSKDLNEQFKKHLTNSEPLDLDFSIQVLSSGSWPFQQSCTFALPSELERSYQRFTAFYASRHSGRKLTWLYQLSKGELVTNCFKNRYTLQASTFQMAILLQYNTEDAYTVQQLTDSTQIKMDILAQVLQILLKSKLLVLEDENANVDEVELKPDTLIKLYLGYKNKKLRVNINVPMETEQKQEQETTHKNIEEDRKLLIQAAIVRIMKMRKVLKHQQLLGEVLTQLSSRFKPRVPVIKKCIDILIEKEYLERVDGEKDTYSYLA
>9XZK_E2 len=108 copies=1 longest=0
MAAAMDVDTPSGTNSGAGKKRFEVKKWNAVALWAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHCISRWLKTRQVCPLDNREWEFQKYGH
>9XZN_E1 len=594 copies=1 longest=1
MADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLLSYMGIRVLNIIFPILTGTYVARVLDRTDYGYFNSVDTILSFFLPFATYGVYNYGLRAISNVKDNKKDLNRTFSSLFYLCIACTILTTAVYILAYPLFFTDNPIVKKVYLVMGIQLIAQIFSIEWVNEALENYSFLFYKTAFIRILMLVSIFLFVKNEHDIVVYTLVMSLSTLINYLISYFWIKRDIKLVKIHLSDFKPLFLPLTAMLVFANANMLFTFLDRLFLVKTGIDVNVSYYTIAQRIVTVIAGVVTGAIGVSVPRLSYYLGKGDKEAYVSLVNRGSRIFNFFIIPLSFGLMVLGPNAILLYGSEKYIGGGILTSLFAFRTIILALDTILGSQILFTNGYEKRITVYTVFAGLLNLGLNSLLFFNHIVAPEYYLLTTMLSETSLLVFYIIFIHRKQLIHLGHIFSYTVRYSLFSLSFVAIYFLINFVYPVDMVINLPFLINTGLIVLLSAISYISLLVFTKDSIFYEFLNHVLALKNKFKKS
>9XZX_E1 len=236 copies=1 longest=1
PDPTPDELHKSSEFTGTMGNMKYLYDDHYVSATKVMSVDKFLAHDLIYNISDKKLKNYDKVKTELLNEDLAKKYKDEVVDVYGSNYYVNCYFSSKDNVGKVTGGKTCMYGGITKHEGNHFDNGNLQNVLIRVYENKRNTISFEVQTDKKSVTAQELDIKARNFLINKKNLYEFNSSPYETGYIKFIENNGNTFWYDMMPAPGDKFDQSKYLMMYNDNKTVDSKSVKIEVHLTTKNG
>9XZX_E2 len=116 copies=1 longest=0
QVQLVESGGGLVQPGGSLRLSCAPSGSTSGIYASGWYRQVPGKPQEWLADISRGGSTKYADSVKGRFTISRDNAKNTVYLQMNSLKPEDTATYYCRVIDTLGAEYRGQGTQVTVSS
>9XZX_E3 len=124 copies=1 longest=0
QVQLVESGGGLVQAGGSLRLSCTASERMYGIEMGWFRQAPGKERELVAAITWSGVVSHLQESVKGRFTISRDNPKKTVYLQMNSLKPEDTATYYCAGHRLQWASGYTRHDYGSWGQGTQVTVSS
>9Y1V_E1 len=31 copies=3 longest=1
QLHQQQHQQQHFQHIQHMQQMQFHQHQQQLS
>9Y61_E1 len=174 copies=1 longest=0
MAKKESCPNTGRDKDHLLKYNVGDLVWSKVSGYPWWPCMVSADPLLHSYTKLKGQKKSARQYHVQFFGDAPERAWIFEKSLVAFEGEGQFEKLCQESAKQAPTKAEKIKLLKPISGKLRAQWEMGIVQAEEAASMSVEERKAKFTFLYVGDQLHLNPQVAKEAGIAAEHHHHHH
>9Y61_E2 len=467 copies=1 longest=1
MSYYHHHHHHDYDIPTTGLVPRGSMMAGEGDQQDAAHNMGNHLPLLPAESEEEDEMEVEDQDSKEAKKPNIINFDTSLPTSHTYLGADMEEFHGRTLHDDDSCQVIPVLPQVMMILIPGQTLPLQLFHPQEVSMVRNLIQKDRTFAVLAYSNVQEREAQFGTTAEIYAYREEQDFGIEIVKVKAIGRQRFKVLELRTQSDGIQQAKVQILPECVLPSTMSAVQLESLNKCQIFPSKPVSREDQCSYKWWQKYQKRKFHCANLTSWPRWLYSLYDAETLMDRIKKQLREWDENLKDDSLPSNPIDFSYRVAACLPIDDVLRIQLLKIGSAIQRLRCELDIMNKCTSLCCKQCQETEITTKNEIFSLSLCGPMAAYVNPHGYVHETLTVYKACNLNLIGRPSTEHSWFPGYAWTVAQCKICASHIGWKFTATKKDMSPQKFWGLTRSALLPTIPDTEDEISPDKVILCL
>9Y63_E1 len=296 copies=3 longest=1
DYKDDDDKKTLREPGAGAQEVTLKVHISDASTHQPVADALIEIFTNQASIASGTSGTDGVAFIKFQYKLGSQLIVTASKHAYVPNSAPWKPIRLPVFSSLSLGLLPERSATLMVYEDVVQIVSGFQGARPQPRVHFQRRALRLPENTSYSDLTAFLTAASSPSEVDSFPYLRGLDGNGTGNSTRHDLTPVTAVSVHLLSSNGTPVLVDGPIYVTVPLATQSSLRHNAYVAAWRFDQKLGTWLKSGLGLVHQEGSQLTWTYIAPQLGYWVAAMSPPIPGPVVTQDITTYHTLEVLFQ
>9Y7L_E1 len=263 copies=1 longest=1
MHHHHHHMTMDEQQSQAVAPVYVGGFLARYDQSPDEAELLLPRDVVEHWLHAQGQGQPSLSVALPLNINHDDTAVVGHVAAMQSVRDGLFCLGCVTSPRFLEIVRRASEKSELVSRGPVSPLQPDKVVEFLSGSYAGLSLSSRRCDDVEVATSLSGSETTPFKHVALCSVGRRRGTLAVYGRDPEWVTQRFPDLTAADRDGLRAQWQRCGSTAVDVSGDPFRSDSYGLLGNSVDALYIRERLPKLRYDKQLVGVTERESYVKA
>9Y7L_E2 len=233 copies=1 longest=0
QVQLVQSGGGLVQPGRSLRLSCTASGFTFGDYAMSWFRQAPGKGLEWVGFIRSKAYGGTTEYAASVKGRFTISRDDSKSIAYLQMNSLKTEDTALYYCVVYNYYNYGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCAAAHHHHHH
>9Y7L_E3 len=220 copies=1 longest=0
LDVVMTQSPLSLPVTPGEPASISCRSSQSLLHSNGYNYLDWYLQKPGQSPQLLIYLGSNRASGVPDRFSGSGSGTDFTLKISRVEAEDVGVYYCMQALQTPITFGPGTKVDIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9Y7M_E1 len=233 copies=1 longest=0
QVQLVQSGGGLVQPGRSLRLSCTASGFTFGDYAMSWFRQAPGKGLEWVGFIRSKAYGGTTEYAASVKGRFTISRDDSKSIAYLQMNSLKTEDTALYYCVVYNYYNYGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCAAAHHHHHH
>9Y7M_E2 len=263 copies=1 longest=1
MHHHHHHMTMDEQQSQAVAPVYVGGFLARYDQSPDEAELLLPRDVVEHWLHAQGQGQPSLSVALPLNINHDDTAVVGHVAAMQSVRDGLFCLGCVTSPRFLEIVRRASEKSELVSRGPVSPLQPDKVVEFLSGSYAGLSLSSRRCDDVEVATSLSGSETTPFKHVALCSVGRRRGTLAVYGRDPEWVTQRFPDLTAADRDGLRAQWQRCGSTAVDVSGDPFRSDSYGLLGNSVDALYIRERLPKLRYDKQLVGVTERESYVKA
>9Y7M_E3 len=220 copies=1 longest=0
LDVVMTQSPLSLPVTPGEPASISCRSSQSLLHSNGYNYLDWYLQKPGQSPQLLIYLGSNRASGVPDRFSGSGSGTDFTLKISRVEAEDVGVYYCMQALQTPITFGPGTKVDIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9Y7N_E1 len=263 copies=1 longest=1
MHHHHHHMTMDEQQSQAVAPVYVGGFLARYDQSPDEAELLLPRDVVEHWLHAQGQGQPSLSVALPLNINHDDTAVVGHVAAMQSVRDGLFCLGCVTSPRFLEIVRRASEKSELVSRGPVSPLQPDKVVEFLSGSYAGLSLSSRRCDDVEVATSLSGSETTPFKHVALCSVGRRRGTLAVYGRDPEWVTQRFPDLTAADRDGLRAQWQRCGSTAVDVSGDPFRSDSYGLLGNSVDALYIRERLPKLRYDKQLVGVTERESYVKA
>9Y7N_E2 len=233 copies=1 longest=0
QVQLVQSGGGLVQPGRSLRLSCTASGFTFGDYAMSWFRQAPGKGLEWVGFIRSKAYGGTTEYAASVKGRFTISRDDSKSIAYLQMNSLKTEDTALYYCVVYNYYNYGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCAAAHHHHHH
>9Y7N_E3 len=220 copies=1 longest=0
LDVVMTQSPLSLPVTPGEPASISCRSSQSLLHSNGYNYLDWYLQKPGQSPQLLIYLGSNRASGVPDRFSGSGSGTDFTLKISRVEAEDVGVYYCMQALQTPITFGPGTKVDIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9Y9X_E1 len=31 copies=3 longest=1
QLHQQQHQQQHFQHIQHMQQMQFHQHQQQLS
>9YA9_E1 len=143 copies=2 longest=0
GSLDYKDDDDKENLYFQGADSCIQFTRHASDVLLNLNRLRSRDILTDVVIVVSREQFRAHKTVLMACSGLFYSIFTDQLKCNLSVINLDPEINPEGFCILLDFMYTSRLNLREGNIMAVMATAMYLQMEHVVDTCRKFIKASE
>9YA9_E2 len=467 copies=1 longest=1
MSYYHHHHHHDYDIPTTGLVPRGSMMAGEGDQQDAAHNMGNHLPLLPAESEEEDEMEVEDQDSKEAKKPNIINFDTSLPTSHTYLGADMEEFHGRTLHDDDSCQVIPVLPQVMMILIPGQTLPLQLFHPQEVSMVRNLIQKDRTFAVLAYSNVQEREAQFGTTAEIYAYREEQDFGIEIVKVKAIGRQRFKVLELRTQSDGIQQAKVQILPECVLPSTMSAVQLESLNKCQIFPSKPVSREDQCSYKWWQKYQKRKFHCANLTSWPRWLYSLYDAETLMDRIKKQLREWDENLKDDSLPSNPIDFSYRVAACLPIDDVLRIQLLKIGSAIQRLRCELDIMNKCTSLCCKQCQETEITTKNEIFSLSLCGPMAAYVNPHGYVHETLTVYKACNLNLIGRPSTEHSWFPGYAWTVAQCKICASHIGWKFTATKKDMSPQKFWGLTRSALLPTIPDTEDEISPDKVILCL
>9YC5_E1 len=283 copies=1 longest=1
LRCMQCKTNGDCRVEECALGQDLCRTTIVRLWEEGEELELVEKSCTHSEKTNRTLSYRTGLKITSLTEVVCGLDLCNQGNSGRAVTYSRSRYLECISCGSSDMSCERGRHQSLQCRSPEEQCLDVVTHWIQEGEEGRPKDDRHLRGCGYLPGCPGSNGFHNNDTFHFLKCCNTTKCNEGPILELENLPQNGRQCYSCKGNSTHGCSSEETFLIDCRGPMNQCLVATGTHEPKNQSYMVRGCATASMCQHAHLGDAFSMNHIDVSCCTKSGCNHPDLDVQYRSG
>9YC5_E2 len=221 copies=1 longest=0
EVQLQQSGSVLVRPGTSVNLSCKASGYTFTSYWMSWVKQRPGQGLEWIGEIFPKSGRINSNENFRGKATLTVDTSSSTAYINLSSLTSEDSAVYYCSPIYSGDYGFADWGQGTLVTVSAAKTTPPSVYPLAPGSAAQTNSMVTLGCLVKGYFPEPVTVTWNSGSLSSGVHTFPAVLQSDLYTLSSSVTVPSSTWPSETVTCNVAHPASSTKVDKKIVPRDC
>9YC5_E3 len=219 copies=1 longest=0
DIVMTQTPLSLPVSLGDQASISCRSSQSVVSNNGKTYLEWYLQKPGQSPKLLIYKVSNRFSGVPDRFSGSGSGTDFTLKISRVEAEDLGVYYCFQGSHVPWTFGGGTKLEVKRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKDINVKWKIDGSERQNGVLNSWTDQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC
>9YFW_E1 len=656 copies=1 longest=1
VNMTTFKIRFEVPKVLIEFYHLVGDCELSVVEILVLGLGAEIEIRTYDLKANAFLKEFCLKCPEYLDENKKPVYLVTTLDNTMEDLLTLEYVKAEKNVPDLKSTYNNVLQLIKVNFSSLDIHLHTEALLNTINYLHNILPQSEEKSAPVSTTETEDKGDVIKKLALKLSTNEDIITLQILAELSCLQIFIQDQKCNISEIKIEGLDSEMIMRPSETEINAKLRNIIVLDSDITAIYKKAVYITGKEVFSFKMVSYMDATAGSAYTDMNVVDIQVNLIVGCIEVVFVTKFLYSILAFIDNFQAAKQALAEATVQAAGMAATGVKELAQRSSRMALDINIKAPVVVIPQSPVSENVFVADFGLITMTNTFHMITESQSSPPPVIDLITIKLSEMRLYRSRFINDAYQEVLDLLLPLNLEVVVERNLCWEWYQEVPCFNVNAQLKPMEFILSQEDITTIFKTLHGNIWYEKDGSASPAVTKDQYSATSGVTTNASHHSGGATVVTAAVVEVHSRALLVKTTLNISFKTDDLTMVLYSPGPKQASFTDVRDPSLKLAEFKLENIISTLKMYTDGSTFSSFSLKNCILDDKRPHVKKATPRMIGLTVGFDKKDMMDIKYRKVRDGCVTDAVFQEMYICASVEFLQTVANVFLEAYTTGTAV
>9YGR_E1 len=1334 copies=1 longest=1
MFLIIFILPTTLAVIGDFNCTNSFINDYNKTIPRISEDVVDVSLGLGTYYVLNRVYLNTTLLFTGYFPKSGANFRDLALKGSIYLSTLWYKPPFLSDFNNGIFSKVKNTKLYVNNTLYSEFSTIVIGSVFVNTSYTIVVQPHNGILEITACQYTMCEYPHTVCKSKGSIRNESWHIDSSEPLCLFKKNFTYNVSADWLYFHFYQERGVFYAYYADVGMPTTFLFSLYLGTILSHYYVMPLTCNAISSNTDNETLEYWVTPLSRRQYLLNFDEHGVITNAVDCSSSFLSEIQCKTQSFAPNTGVYDLSGFTVKPVATVYRRIPNLPDCDIDNWLNNVSVPSPLNWERRIFSNCNFNLSTLLRLVHVDSFSCNNLDKSKIFGSCFNSITVDKFAIPNRRRDDLQLGSSGFLQSSNYKIDISSSSCQLYYSLPLVNVTINNFNPSSWNRRYGFGSFNLSSYDVVYSDHCFSVNSDFCPCADPSVVNSCAKSKPPSAICPAGTKYRHCDLDTTLYVKNWCRCSCLPDPISTYSPNTCPQKKVVVGIGEHCPGLGINEEKCGTQLNHSSCFCSPDAFLGWSFDSCISNNRCNIFSNFIFNGINSGTTCSNDLLYSNTEISTGVCVNYDLYGITGQGIFKEVSAAYYNNWQNLLYDSNGNIIGFKDFLTNKTYTILPCYSGRVSAAFYQNSSSPALLYRNLKCSYVLNNISFISQPFYFDSYLGCVLNAVNLTSYSVSSCDLRMGSGFCIDYALPSSGGSGSGISSPYRFVTFEPFNVSFVNDSVETVGGLFEIQIPTNFTIAGHEEFIQTSSPKVTIDCSAFVCSNYAACHDLLSEYGTFCDNINSILNEVNDLLDITQLQVANALMQGVTLSSNLNTNLHSDVDNIDFKSLLGCLGSQCGSSSRSLLEDLLFNKVKLSDVGFVEAYNNCTGGSEIRDLLCVQSFNGIKVLPPILSETQISGYTTAATVAAMFPPWSAAAGVPFSLNVQYRINGLGVTMDVLNKNQKLIANAFNKALLSIQNGFTATNSALAKIQSVVNANAQALNSLLQQLFNKFGAISSSLQEILSRLDPPEAQVQIDRLINGRLTALNAYVSQQLSDITLIKAGASRAIEKVNECVKSQSPRINFCGNGNHILSLVQNAPYGLLFIHFSYKPTSFKTVLVSPGLCLSGDRGIAPKQGYFIKQNDSWMFTGSSYYYPEPISDKNVVFMNSCSVNFTKAPFIYLNNSIPNLSDFEAELSLWFKNHTSIAPNLTFNSHINATFLDLYYEMNVIQESIKSLNSGRLEVLFQGPGGYIPEAPRDGQAYVRKDGEWVLLSTFLGHHHHHHHHSAWSHPQFEK
>9YGR_E2 len=228 copies=1 longest=0
QGTLKESGPMLVKPTQTLTLTCNFSGFSLSTSGVGVGWIRQPPGKALECLALIYWNDDQRYSPSLKTRLTITKDTSKNQVVLTMTNMDPVDTATYYCVYSMSRPVAGGVIDHWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKSCDK
>9YGR_E3 len=214 copies=1 longest=0
DIQLTQSPSSLSASEGDRVTITCWASQNIRGYLNWYQQKPGKAPKLLIYATSTLESGVPSRFSGSGSVTDYTLTISSLQPEDFATYSCQQSYSSPPTFGGGTKVDIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC
>9YI2_E1 len=590 copies=2 longest=1
MASWSHPQFEKGAENLYFQSNAGSTLGCHRSIPRDPSDLSHSRKFSAACNFSNILVNQERLNINTATEEELMTLPGVTRAVARSIVEYREYIGGFKKVEDLALVSGVGATKLEQVKFEICVSSKGSSAQHSPSSLRRDLLAEQQPHHLATAVPLTPRVNINTATPAQLMSVRGLSEKMALSIVDFRREHGPFRSVEDLVRMDGINAAFLDRIRHQVFAERSRPPSTHTNGGLTFTAKPHPSPTSLSLQSEDLDLPPGGPTQIISTRPSVEAFGGTRDGRPVLRLATWNLQGCSVEKANNPGVREVVCMTLLENSIKLLAVQELLDREALEKFCTELNQPTLPNIRKWKGPRGCWKAVVAEKPSNQLQKGAGYAGFLWDAAAGMELRDAGSQESSPSNGHGKLAGPSPYLGRFKVGSHDLTLVNLHLAALTLLGSENPSKNHSDGHRLASFAQTLQETLKGEKDVIILGDFGQGPDSNDYDILRKEKFHHLIPAHTFTNISTKNPQGSKSLDNIWISKSLKKVFTGHWAVVREGLTNPWIPDNWSWGGVASEHCPVLAEFYTEKDWSKKDAPRNGSGVALERSEANIKHER
>9YQL_E1 len=608 copies=1 longest=1
MAASKPVEAAVVAAAVPSSGSGVGGGGTAGPGTGGLPRWQLALAVGAPLLLGAGAIYLWSRQQRRREARGRGDASGLKRNSERKTPEGRASPAPGSGHPEGPGAHLDMNSLDRAQAAKNKGNKYFKAGKYEQAIQCYTEAISLCPTEKNVDLSTFYQNRAAAFEQLQKWKEVAQDCTKAVELNPKYVKALFRRAKAHEKLDNKKECLEDVTAVCILEGFQNQQSMLLADKVLKLLGKEKAKEKYKNREPLMPSPQFIKSYFSSFTDDIISQPMLKGEKSDEDKDKEGEALEVKENSGYLKAKQYMEEENYDKIISECSKEIDAEGKYMAEALLLRATFYLLIGNANAAKPDLDKVISLKEANVKLRANALIKRGSMYMQQQQPLLSTQDFNMAADIDPQNADVYHHRGQLKILLDQVEEAVADFDECIRLRPESALAQAQKCFALYRQAYTGNNSSQIQAAMKGFEEVIKKFPRCAEGYALYAQALTDQQQFGKADEMYDKCIDLEPDNATTYVHKGLLQLQWKQDLDRGLELISKAIEIDNKCDFAYETMGTIEVQRGNMEKAIDMFNKAINLAKSEMEMAHLYSLCDAAHAQTEVAKKYGLKPPTL
>9YR4_E1 len=607 copies=1 longest=1
MAASKPVEAAVVAAAVPSSGSGVGGGGTAGPGTGGLPRWQLALAVGAPLLLGAGAIYLWSRQQRRREARGRGDASGLKRNSERKTPEGRASPAPGSGHPEGPGAHLDMNSLDRAQAAKNKGNKYFKAGKYEQAIQCYTEAISLCPTEKNVDLSTFYQNRAAAFEQLQKWKEVAQDCTKAVELNPKYVKALFRRAKAHEKLDNKKECLEDVTAVCILEGFQNQQSMLLADKVLKLLGKEKAKEKYKNREPLMPSPQFIKSYFSSFTDDIISQPMLKGEKSDEDKDKEGEALEVKENSGYLKAKQYMEEENYDKIISECSKEIDAEGKYMAEALLLRATFYLLIGNANAAKPDLDKVISLKEANVKLRANALIKRGSMYMQQQQPLLSTQDFNMAADIDPQNADVYHHRGQLKILLDQVEEAVADFDECIRLRPESALAQAQKCFALYRQAYTGNNSSQIQAAMKGFEEVIKKFPRCAEGYALYAQALTDQQQFGKADEMYDKCIDLEPDNATTYVHKGLLQLQWKQDLDRGLELISKAIEIDNKCDFAYETMGTIEVQRGNMEKAIDMFNKAINLAKSEMEMAHLYSLCDAAHAQTEVAKKYGLKPPT
>9YR4_E2 len=10 copies=1 longest=0
YDTSRMEEVD
>9YW2_E1 len=821 copies=1 longest=0
MASGADSKGDDLSTAILKQKNRPNRLIVDEAINEDNSVVSLSQPKMDELQLFRGDTVLLKGKKRREAVCIVLSDDTCSDEKIRMNRVVRNNLRVRLGDVISIQPCPDVKYGKRIHVLPIDDTVEGITGNLFEVYLKPYFLEAYRPIRKGDIFLVRGGMRAVEFKVVETDPSPYCIVAPDTVIHCEGEPIKREDEEESLNEVGYDDIGGCRKQLAQIKEMVELPLRHPALFKAIGVKPPRGILLYGPPGTGKTLIARAVANETGAFFFLINGPEIMSKLAGESESNLRKAFEEAEKNAPAIIFIDELDAIAPKREKTHGEVERRIVSQLLTLMDGLKQRAHVIVMAATNRPNSIDPALRRFGRFDREVDIGIPDATGRLEILQIHTKNMKLADDVDLEQVANETHGHVGADLAALCSEAALQAIRKKMDLIDLEDETIDAEVMNSLAVTMDDFRWALSQSNPSALRETVVEVPQVTWEDIGGLEDVKRELQELVQYPVEHPDKFLKFGMTPSKGVLFYGPPGCGKTLLAKAIANECQANFISIKGPELLTMWFGESEANVREIFDKARQAAPCVLFFDELDSIAKARGGNIGDGGGAADRVINQILTEMDGMSTKKNVFIIGATNRPDIIDPAILRPGRLDQLIYIPLPDEKSRVAILKANLRKSPVAKDVDLEFLAKMTNGFSGADLTEICQRACKLAIRESIESEIRRERERQTNPSAMEVEEDDPVPEIRRDHFEEAMRFARRSVSDNDIRKYEMFAQTLQQSRGFGSFRFPSGNQGGAGPSQGSGGGTGGSVYTEDNDDDLYGVDKLAAALEHHHHHH
>9YW2_E2 len=880 copies=1 longest=1
MSPILGYWKIKGLVQPTRLLLEYLEEKYEEHLYERDEGDKWRNKKFELGLEFPNLPYYIDGDVKLTQSMAIIRYIADKHNMLGGCPKERAEISMLEGAVLDIRYGVSRIAYSKDFETLKVDFLSKLPEMLKMFEDRLCHKTYLNGDHVTHPDFMLYDALDVVLYMDPMCLDAFPKLVCFKKRIEAIPQIDKYLKSSKYIAWPLQGWQATFGGGDHPPKSDLEVLFQGPLGMASNMDREMILADFQACTGIENIDEAITLLEQNNWDLVAAINGVIPQENGILQSEYGGETIPGPAFNPASHPASAPTSSSSSAFRPVMPSRQIVERQPRMLDFRVEYRDRNVDVVLEDTCTVGEIKQILENELQIPVSKMLLKGWKTGDVEDSTVLKSLHLPKNNSLYVLTPDLPPPSSSSHAGALQESLNQNFMLIITHREVQREYNLNFSGSSTIQEVKRNVYDLTSIPVRHQLWEGWPTSATDDSMCLAESGLSYPCHRLTVGRRSSPAQTREQSEEQITDVHMVSDSDGDDFEDATEFGVDDGEVFGMASSALRKSPMMPENAENEGDALLQFTAEFSSRYGDCHPVFFIGSLEAAFQEAFYVKARDRKLLAIYLHHDESVLTNVFCSQMLCAESIVSYLSQNFITWAWDLTKDSNRARFLTMCNRHFGSVVAQTIRTQKTDQFPLFLIIMGKRSSNEVLNVIQGNTTVDELMMRLMAAMEIFTAQQQEDIKDEDEREARENVKREQDEAYRLSLEADRAKREAHEREMAEQFRLEQIRKEQEEEREAIRLSLEQALPPEPKEENAEPVSKLRIRTPSGEFLERRFLASNKLQIVFDFVASKGFPWDEYKLLSTFPRRDVTQLDPNKSLLEVKLFPQETLFLEAKE
>9YW2_E3 len=326 copies=1 longest=0
MSSHHHHHHSSGLVPRGSHMFSFNMFDHPIPRVFQNRFSTQYRCFSVSMLAGPNDRSDVEKGGKIIMPPSALDQLSRLNITYPMLFKLTNKNSDRMTHCGVLEFVADEGICYLPHWMMQNLLLEEGGLVQVESVNLQVATYSKFQPQSPDFLDITNPKAVLENALRNFACLTTGDVIAINYNEKIYELRVMETKPDKAVSIIECDMNVDFDAPLGYKEPERQVQHEESTEGEADHSGYAGELGFRAFSGSGNRLDGKKKGVEPSPSPIKPGDIKRGIPNYEFKLGKITFIRNSRPLVKKVEEDEAGGRFVAFSGEGQSLRKKGRKP
>9YWU_E1 len=456 copies=1 longest=1
MEALGDLEGPRAPGGDDPAGSAGETPGWLSREQVFVLISAASVNLGSMMCYSILGPFFPKEAEKKGASNTIIGMIFGCFALFELLASLVFGNYLVHIGAKFMFVAGMFVSGGVTILFGVLDRVPDGPVFIAMCFLVRVMDAVSFAAAMTASSSILAKAFPNNVATVLGSLETFSGLGLILGPPVGGFLYQSFGYEVPFIVLGCVVLLMVPLNMYILPNYESDPGEHSFWKLIALPKVGLIAFVINSLSSCFGFLDPTLSLFVLEKFNLPAGYVGLVFLGMALSYAISSPLFGLLSDKRPPLRKWLLVFGNLITAGCYMLLGPVPILHIKSQLWLLVLILVVSGLSAGMSIIPTFPEILSCAHENGFEEGLSTLGLVSGLFSAMWSIGAFMGPTLGGFLYEKIGFEWAAAIQGLWALISGLAMGLFYLLEYSRRKRSKSQNILSTEEERTTLLPNET
>9YWV_E1 len=456 copies=1 longest=1
MEALGDLEGPRAPGGDDPAGSAGETPGWLSREQVFVLISAASVNLGSMMCYSILGPFFPKEAEKKGASNTIIGMIFGCFALFELLASLVFGNYLVHIGAKFMFVAGMFVSGGVTILFGVLDRVPDGPVFIAMCFLVRVMDAVSFAAAMTASSSILAKAFPNNVATVLGSLETFSGLGLILGPPVGGFLYQSFGYEVPFIVLGCVVLLMVPLNMYILPNYESDPGEHSFWKLIALPKVGLIAFVINSLSSCFGFLDPTLSLFVLEKFNLPAGYVGLVFLGMALSYAISSPLFGLLSDKRPPLRKWLLVFGNLITAGCYMLLGPVPILHIKSQLWLLVLILVVSGLSAGMSIIPTFPEILSCAHENGFEEGLSTLGLVSGLFSAMWSIGAFMGPTLGGFLYEKIGFEWAAAIQGLWALISGLAMGLFYLLEYSRRKRSKSQNILSTEEERTTLLPNET
>9YX8_E1 len=1250 copies=3 longest=1
MFVFLVLLLLVSSQCVNLTTTTQLPLAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHVISGTNGTKRFDNPVLPFNDGVYFASIEKSNITRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFNNKSWMESESRVYSSAKNCTFEYVFQPFLMDLEGKQGNFTNLREFVFKNIDGYFKIYSKHTPIIVRDLPQGFSALEPLVDLPIGINITRFQTLLHRSYLTSGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESVVRFPNITNLCPFYEVFNATRFPSVYAWNRKRISNCVADYSVLYNFAPFFTFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVISWNSNRLDSKADGNYNYWYRLFRKSKLKPFERDISTEIYQAGNNPCNGVKGFNCYFPLQSYSFRPTYGVGYQPYRVVVLSFELLHAPATVCGPKNSTNLVKNKCVNFNFNGLTGTGVLTDSNKKFLPFQQFGRDIADTTDAVRDPQTLDILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVPVAIRADQLTPTWRVYSTGSKGFQTRAGCLIGAKYVNNSYECDIPIGAGICASYQTQTRSRGSASSVDSQSIIAYTMSLGAENLVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPITYFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICKQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLFSTPSALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSRDVDLGDISGINASVVNIQKEIERLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGRSLEVLFQGPGHHHHHHHHSAWSHPQFEK
>9YZ1_E1 len=268 copies=1 longest=1
SRRHARASRRNYRTRPLNRYRRRRQNRFKLFHLRLRRTLTADWPTAPVKPTNDPQTETPLLWNFDHLSFKLTDFLQASHGTGDFQHLPPFRFYKFKKVYIRARWINWPKTLMENVLGRTALDLDGEDQGRGNATRSHLDPGTVPGRLEPPKDPNKAPFIYDPLQDRSSSRSFNMATGFKRGLTPKPMFTQDITSPSATAPWLTRGTPWVSVIQGANMVWNGLSISLRQMKDMRPTTPDTSTSQIPQVQYDISAYIAFKEFDYETGRQL
>9Z1N_E1 len=1243 copies=1 longest=1
MFVFLVLLPLVSSQCVMPLFNLITTTQSYTNSFTRGVYYPDKVFRSSVLHLTQDLFLPFFSNVTWFHAISGTNGTKRFDNPVLPFNDGVYFASTEKSNIIRGWIFGTTLDSKTQSLLIVNNATNVFIKVCEFQFCNDPFLDVYHKNNKSWMESESGVYSSANNCTFEYVSQPFLMDLEGKQGNFKNLREFVFKNIDGYFKIYSKHTPIIGRDFPQGFSALEPLVDLPIGINITRFQTLLALNRSYLTPGDSSSGWTAGAADYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESIVRFPNVTNLCPFHEVFNATRFASVYAWNRTRISNCVADYSVLYNFAPFFAFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVIAWNSNKLDSKHSGNYDYWYRSLRKSKLKPFERDISTEIYQAGNKPCKGKGPNCYFPLESYGFRPTYGVGHQPYRVVVLSFELLHAPATVCGPKKSTNLVKNKCVNFNFNGLTGTGVLTKSNKKFLPFQQFGRDIVDTTDAVRDPQTLEILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVSVAIHADQLTPTWRVYSTGSNVFQTRAGCLIGAEYVNNSYECDIPIGAGICASYQTQTKSRAGARSVASQSIIAYTMSLGAENSVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPIKYFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICAQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLFSTPSALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFLTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQLELDSFKEELDKYFKNHTSPDVDLGDISGINASVVNIQKEIDRLNEVAKNLNESLIDLQELGKYEQYIKGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGHHHHHH
>9Z1N_E2 len=150 copies=1 longest=0
QVQLQESGGGLVQPGGSLRLSCTASGIALHSKATGWFRQAPGKEREGVSCISSGDGTTYYEDSVEGRFTISRDNAKNTVYLQMNSLKLEDTAVYYCAADPSGVCHSGSYYYTDDDFYYRGQGTQVTVSSGGQHHHHHHGAYPYDVPDYAS
>9Z1X_E1 len=1214 copies=1 longest=1
MVVPEKEQSWIPKIFKKKTCTTFIVDSTDPGGTLCQCGRPRTAHPAVAMEDAFGAAVVTVWDSDAHTTEKPTDAYGELDFTGAGRKHSNFLRLSDRTDPAAVYSLVTRTWGFRAPNLVVSVLGGSGGPVLQTWLQDLLRRGLVRAAQSTGAWIVTGGLHTGIGRHVGVAVRDHQMASTGGTKVVAMGVAPWGVVRNRDTLINPKGSFPARYRWRGDPEDGVQFPLDYNYSAFFLVDDGTHGCLGGENRFRLRLESYISQQKTGVGGTGIDIPVLLLLIDGDEKMLTRIENATQAQLPCLLVAGSGGAADCLAETLEDTLAPGSGGARQGEARDRIRRFFPKGDLEVLQAQVERIMTRKELLTVYSSEDGSEEFETIVLKALVKACGSSEASAYLDELRLAVAWNRVDIAQSELFRGDIQWRSFHLEASLMDALLNDRPEFVRLLISHGLSLGHFLTPMRLAQLYSAAPSNSLIRNLLDQASHSAGTKAPALKGGAAELRPPDVGHVLRMLLGKMCAPRYPSGGAWDPHPGQGFGESMYLLSDKATSPLSLDAGLGQAPWSDLLLWALLLNRAQMAMYFWEMGSNAVSSALGACLLLRVMARLEPDAEEAARRKDLAFKFEGMGVDLFGECYRSSEVRAARLLLRRCPLWGDATCLQLAMQADARAFFAQDGVQSLLTQKWWGDMASTTPIWALVLAFFCPPLIYTRLITFRKSEEEPTREELEFDMDSVINGEGPVGTADPAEKTPLGVPRQSGRPGCCGGRCGGRRCLRRWFHFWGAPVTIFMGNVVSYLLFLLLFSRVLLVDFQPAPPGSLELLLYFWAFTLLCEELRQGLSGGGGSLASGGPGPGHASLSQRLRLYLADSWNQCDLVALTCFLLGVGCRLTPGLYHLGRTVLCIDFMVFTVRLLHIFTVNKQLGPKIVIVSKMMKDVFFFLFFLGVWLVAYGVATEGLLRPRDSDFPSILRRVFYRPYLQIFGQIPQEDMDVALMEHSNCSSEPGFWAHPPGAQAGTCVSQYANWLVVLLLVIFLLVANILLVNLLIAMFSYTFGKVQGNSDLYWKAQRYRLIREFHSRPALAPPFIVISHLRLLLRQLCRRPRSPQPSSPALEHFRVYLSKEAERKLLTWESVHKENFLLARARDKRESDSERLKRTSQKVDLALKQLGHIREYEQRLKVLEREVQQCSRVLGWVAEALSRSALLPPGGPPPPDLPGSKD
>9Z1Y_E1 len=1214 copies=1 longest=1
MVVPEKEQSWIPKIFKKKTCTTFIVDSTDPGGTLCQCGRPRTAHPAVAMEDAFGAAVVTVWDSDAHTTEKPTDAYGELDFTGAGRKHSNFLRLSDRTDPAAVYSLVTRTWGFRAPNLVVSVLGGSGGPVLQTWLQDLLRRGLVRAAQSTGAWIVTGGLHTGIGRHVGVAVRDHQMASTGGTKVVAMGVAPWGVVRNRDTLINPKGSFPARYRWRGDPEDGVQFPLDYNYSAFFLVDDGTHGCLGGENRFRLRLESYISQQKTGVGGTGIDIPVLLLLIDGDEKMLTRIENATQAQLPCLLVAGSGGAADCLAETLEDTLAPGSGGARQGEARDRIRRFFPKGDLEVLQAQVERIMTRKELLTVYSSEDGSEEFETIVLKALVKACGSSEASAYLDELRLAVAWNRVDIAQSELFRGDIQWRSFHLEASLMDALLNDRPEFVRLLISHGLSLGHFLTPMRLAQLYSAAPSNSLIRNLLDQASHSAGTKAPALKGGAAELRPPDVGHVLRMLLGKMCAPRYPSGGAWDPHPGQGFGESMYLLSDKATSPLSLDAGLGQAPWSDLLLWALLLNRAQMAMYFWEMGSNAVSSALGACLLLRVMARLEPDAEEAARRKDLAFKFEGMGVDLFGECYRSSEVRAARLLLRRCPLWGDATCLQLAMQADARAFFAQDGVQSLLTQKWWGDMASTTPIWALVLAFFCPPLIYTRLITFRKSEEEPTREELEFDMDSVINGEGPVGTADPAEKTPLGVPRQSGRPGCCGGRCGGRRCLRRWFHFWGAPVTIFMGNVVSYLLFLLLFSRVLLVDFQPAPPGSLELLLYFWAFTLLCEELRQGLSGGGGSLASGGPGPGHASLSQRLRLYLADSWNQCDLVALTCFLLGVGCRLTPGLYHLGRTVLCIDFMVFTVRLLHIFTVNKQLGPKIVIVSKMMKDVFFFLFFLGVWLVAYGVATEGLLRPRDSDFPSILRRVFYRPYLQIFGQIPQEDMDVALMEHSNCSSEPGFWAHPPGAQAGTCVSQYANWLVVLLLVIFLLVANILLVNLLIAMFSYTFGKVQGNSDLYWKAQRYRLIREFHSRPALAPPFIVISHLRLLLRQLCRRPRSPQPSSPALEHFRVYLSKEAERKLLTWESVHKENFLLARARDKRESDSERLKRTSQKVDLALKQLGHIREYEQRLKVLEREVQQCSRVLGWVAEALSRSALLPPGGPPPPDLPGSKD
>9Z24_E1 len=1214 copies=1 longest=1
MVVPEKEQSWIPKIFKKKTCTTFIVDSTDPGGTLCQCGRPRTAHPAVAMEDAFGAAVVTVWDSDAHTTEKPTDAYGELDFTGAGRKHSNFLRLSDRTDPAAVYSLVTRTWGFRAPNLVVSVLGGSGGPVLQTWLQDLLRRGLVRAAQSTGAWIVTGGLHTGIGRHVGVAVRDHQMASTGGTKVVAMGVAPWGVVRNRDTLINPKGSFPARYRWRGDPEDGVQFPLDYNYSAFFLVDDGTHGCLGGENRFRLRLESYISQQKTGVGGTGIDIPVLLLLIDGDEKMLTRIENATQAQLPCLLVAGSGGAADCLAETLEDTLAPGSGGARQGEARDRIRRFFPKGDLEVLQAQVERIMTRKELLTVYSSEDGSEEFETIVLKALVKACGSSEASAYLDELRLAVAWNRVDIAQSELFRGDIQWRSFHLEASLMDALLNDRPEFVRLLISHGLSLGHFLTPMRLAQLYSAAPSNSLIRNLLDQASHSAGTKAPALKGGAAELRPPDVGHVLRMLLGKMCAPRYPSGGAWDPHPGQGFGESMYLLSDKATSPLSLDAGLGQAPWSDLLLWALLLNRAQMAMYFWEMGSNAVSSALGACLLLRVMARLEPDAEEAARRKDLAFKFEGMGVDLFGECYRSSEVRAARLLLRRCPLWGDATCLQLAMQADARAFFAQDGVQSLLTQKWWGDMASTTPIWALVLAFFCPPLIYTRLITFRKSEEEPTREELEFDMDSVINGEGPVGTADPAEKTPLGVPRQSGRPGCCGGRCGGRRCLRRWFHFWGAPVTIFMGNVVSYLLFLLLFSRVLLVDFQPAPPGSLELLLYFWAFTLLCEELRQGLSGGGGSLASGGPGPGHASLSQRLRLYLADSWNQCDLVALTCFLLGVGCRLTPGLYHLGRTVLCIDFMVFTVRLLHIFTVNKQLGPKIVIVSKMMKDVFFFLFFLGVWLVAYGVATEGLLRPRDSDFPSILRRVFYRPYLQIFGQIPQEDMDVALMEHSNCSSEPGFWAHPPGAQAGTCVSQYANWLVVLLLVIFLLVANILLVNLLIAMFSYTFGKVQGNSDLYWKAQRYRLIREFHSRPALAPPFIVISHLRLLLRQLCRRPRSPQPSSPALEHFRVYLSKEAERKLLTWESVHKENFLLARARDKRESDSERLKRTSQKVDLALKQLGHIREYEQRLKVLEREVQQCSRVLGWVAEALSRSALLPPGGPPPPDLPGSKD
>9Z26_E1 len=1214 copies=1 longest=1
MVVPEKEQSWIPKIFKKKTCTTFIVDSTDPGGTLCQCGRPRTAHPAVAMEDAFGAAVVTVWDSDAHTTEKPTDAYGELDFTGAGRKHSNFLRLSDRTDPAAVYSLVTRTWGFRAPNLVVSVLGGSGGPVLQTWLQDLLRRGLVRAAQSTGAWIVTGGLHTGIGRHVGVAVRDHQMASTGGTKVVAMGVAPWGVVRNRDTLINPKGSFPARYRWRGDPEDGVQFPLDYNYSAFFLVDDGTHGCLGGENRFRLRLESYISQQKTGVGGTGIDIPVLLLLIDGDEKMLTRIENATQAQLPCLLVAGSGGAADCLAETLEDTLAPGSGGARQGEARDRIRRFFPKGDLEVLQAQVERIMTRKELLTVYSSEDGSEEFETIVLKALVKACGSSEASAYLDELRLAVAWNRVDIAQSELFRGDIQWRSFHLEASLMDALLNDRPEFVRLLISHGLSLGHFLTPMRLAQLYSAAPSNSLIRNLLDQASHSAGTKAPALKGGAAELRPPDVGHVLRMLLGKMCAPRYPSGGAWDPHPGQGFGESMYLLSDKATSPLSLDAGLGQAPWSDLLLWALLLNRAQMAMYFWEMGSNAVSSALGACLLLRVMARLEPDAEEAARRKDLAFKFEGMGVDLFGECYRSSEVRAARLLLRRCPLWGDATCLQLAMQADARAFFAQDGVQSLLTQKWWGDMASTTPIWALVLAFFCPPLIYTRLITFRKSEEEPTREELEFDMDSVINGEGPVGTADPAEKTPLGVPRQSGRPGCCGGRCGGRRCLRRWFHFWGAPVTIFMGNVVSYLLFLLLFSRVLLVDFQPAPPGSLELLLYFWAFTLLCEELRQGLSGGGGSLASGGPGPGHASLSQRLRLYLADSWNQCDLVALTCFLLGVGCRLTPGLYHLGRTVLCIDFMVFTVRLLHIFTVNKQLGPKIVIVSKMMKDVFFFLFFLGVWLVAYGVATEGLLRPRDSDFPSILRRVFYRPYLQIFGQIPQEDMDVALMEHSNCSSEPGFWAHPPGAQAGTCVSQYANWLVVLLLVIFLLVANILLVNLLIAMFSYTFGKVQGNSDLYWKAQRYRLIREFHSRPALAPPFIVISHLRLLLRQLCRRPRSPQPSSPALEHFRVYLSKEAERKLLTWESVHKENFLLARARDKRESDSERLKRTSQKVDLALKQLGHIREYEQRLKVLEREVQQCSRVLGWVAEALSRSALLPPGGPPPPDLPGSKD
>9Z3Y_E1 len=1271 copies=1 longest=1
MFVFLVLLLLVSSQCVNLTTTTQLPLAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHVISGTNGTKRFDNPVLPFNDGVYFASIEKSNITRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFNNKSWMESESRVYSSAKNCTFEYVFQPFLMDLEGKQGNFTNLREFVFKNIDGYFKIYSKHTPIIVRDLPQGFSALEPLVDLPIGINITRFQTLLHRSYLTSGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESVVRFPNITNLCPFYEVFNATRFPSVYAWNRKRISNCVADYSVLYNFAPFFTFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVISWNSNRLDSKADGNYNYWYRLFRKSKLKPFERDISTEIYQAGNNPCNGVKGFNCYFPLQSYSFRPTYGVGYQPYRVVVLSFELLHAPATVCGPKNSTNLVKNKCVNFNFNGLTGTGVLTDSNKKFLPFQQFGRDIADTTDAVRDPQTLDILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVPVAIRADQLTPTWRVYSTGSKGFQTRAGCLIGAKYVNNSYECDIPIGAGICASYQTQTRSRGSASSVDSQSIIAYTMSLGAENLVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPITYFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICKQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLFSTPSALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSRDVDLGDISGINASVVNIQKEIERLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGRSLEVLFQGPGHHHHHHHHSAWSHPQFEKGGGSGGGGSGGSAWSHPQFEK
>9Z4I_E1 len=620 copies=1 longest=1
MAGRWLDPLWAPGFLCVALILETASGAGDLSTKAHGHIQFSARGVNQTAMADCRAVCSLNTSDRCDFVKRNPDCHSEGGYLDYLKGIFCYFPPNLLPLAITLYVFWLLYLFLILGVTAAKFFCPNLSAISTSLKLSHNVAGVTFLAFGNGAPDIFSALVAFSDPRTAGLAIGALFGAGVLVTTVVAGGITILRPFMAASRPFLRDITFYMVAVFLTFTALYLGRITLVWALGYLGLYVFYVVTVIICTWVYQRQRSRSLVHSISETPELLTDSEEDQMSSNTNSYDYGEEYRPLLLGEETTGQILLQALNPLDYRKWRTQSISCKLLKVAKLPVEFLLLLTVPVVDPDKDDRNWKRPLNCLQLVISPLVLVLTLQSGVYGIYEIGGLLPVWAVVVIVGTALASVTFFATSNSEPPRLHWLFAFLGFLTSALWINAAATEVVNILRSLGVVFRLSNTVLGLTLLAWGNSIGDAFSDFTLARQGYPRMAFSACFGGIIFNILVGVGLGCLLQIVRSHASEVKLEPDGLLVWVLASALGLSLVFSLVSVPLQCFQLSKAYGLCLLLFYICFIVVVLLTEFGVIHLKADAAAGGSWSHPQFEKGGGSGGGSGGGSAWSHPQFEK
>9Z4O_E1 len=575 copies=1 longest=1
MAGAIIENMSTKKLCIVGGILLVFQIIAFLVGGLIAPGPTTAVSYMSVKCVDARKNHHKTKWFVPWGPNHCDKIRDIEEAIPREIEANDIVFSVHIPLPHMEMSPWFQFMLFILQLDIAFKLNNQIRENAEVSMDVSLAYRDDAFAEWTEMAHERVPRKLKCTFTSPKTPEHEGRYYECDVLPFMEIGSVAHKFYLLNIRLPVNEKKKINVGIGEIKDIRLVGIHQNGGFTKVWFAMKTFLTPSIFIIMVWYWRRITMMSRPPVLLEKVIFALGISMTFINIPVEWFSIGFDWTWMLLFGDIRQGIFYAMLLSFWIIFCGEHMMDQHERNHIAGYWKQVGPIAVGSFCLFIFDMCERGVQLTNPFYSIWTTDIGTELAMAFIIVAGICLCLYFLFLCFMVFQVFRNISGKQSSLPAMSKVRRLHYEGLIFRFKFLMLITLACAAMTVIFFIVSQVTEGHWKWGGVTVQVNSAFFTGIYGMWNLYVFALMFLYAPSHKNYGEDQSNGDLGVHSGEELQLTTTITHVDGPTEIYKLTRKEAQEAAALEVLFQGPGAAEDQVDPRLIDGKHHHHHHHH
>9Z5I_E1 len=580 copies=1 longest=1
MHHHHHHDYKDDDDKGSSGLEVLFQGPGSSGMSPTISHKDSSRQRRPGNFSHSLDMKSGPLPPGGWDDSHLDSAGREGDREALLGDTGTGDFLKAPQSFRAELSSILLLLFLYVLQGIPLGLAGSIPLILQSKNVSYTDQAFFSFVFWPFSLKLLWAPLVDAVYVKNFGRRKSWLVPTQYILGLFMIYLSTQVDRLLGNTDDRTPDVIALTVAFFLFEFLAATQDIAVDGWALTMLSRENVGYASTCNSVGQTAGYFLGNVLFLALESADFCNKYLRFQPQPRGIVTLSDFLFFWGTVFLITTTLVALLKKENEVSVVKEETQGITDTYKLLFAIIKMPAVLTFCLLILTAKIGFSAADAVTGLKLVEEGVPKEHLALLAVPMVPLQIILPLIISKYTAGPQPLNTFYKAMPYRLLLGLEYALLVWWTPKVEHQGGFPIYYYIVVLLSYALHQVTVYSMYVSIMAFNAKVSDPLIGGTYMTLLNTVSNLGGNWPSTVALWLVDPLTVKECVGASNQNCRTPDAVELCKKLGGSCVTALDGYYVESIICVFIGFGWWFFLGPKFKKLQDEGSSSWKCKRNN
>9Z5U_E1 len=589 copies=1 longest=1
YQVRNSTGLYHVTNDCPNSSIVYETADAILHTPGCVPCVREGNASRCWVPMTCTVATRDGKLPATQLRRGGSGGSGGSPRRHWTTQDCNCSIYPGHVTGHRMAWDMMMNPSPTTALVVAQLLRIPQAILDMIAPGGRIARLEEKVKTLKAQNSELASTANMLREQVAQLKQKVMNYRRRRRRQTYVTGGTAARATSGLANFFSPGAKQDVQLINTNGSWHINRTALNCNTSLETGWIAGLIYLNKFNSSGCPERMASCRPLADFAQGWGPISYANGSGPDHRPYCWHYPPKPCGIVSAKSVCGPVYCFTPSPVVVGTTNKLGAPTYSWGENETDVFVLNNTRPPLGNWFGCTWMNSTGFTKVCGAPPCAIGGVGNNTLHCPTDCFRKHPEATYSRCGSGPWITPRCLVDYPYRLWHYPCTANYTRFKVRMYIGGVEHRLDAACNWTRGERCDLEDRDRSELSPLLLSTTQWQVLPCSFTTPPALSCGLIHLHQNIVDVQYLYGVGSSIVSWAIPGGLTDTLQAETDQLEDKKSALQTEIANLLKEKEKLEFILAAYGGSAWSHPQFEKGGGSGGGSGGSSAWSHPQFEK
>9Z5U_E2 len=229 copies=1 longest=0
QVQLVQSGAEVKKPGSSVKVSCKASGGTFSGYGITWVRQAPGQGLEWMGTIIPVSATETYAQRLQGRVTISADEHSTTSYMEVSSLKSEDTALYYCARHDYFWGTPLDIWGQGTMVIVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCHHHHHH
>9Z5U_E3 len=214 copies=1 longest=0
EIVLTQSPDFQSVTPKEKVTITCRASQSIGSNLHWYQQKPGQSPKLLIKYASQSFSGVPSRFSGSGSGTDFTLTINSLEAEDAATYFCHQSYNLPRTFGGGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTRSFNRGEC
>9Z60_E1 len=646 copies=1 longest=1
GPEFATMAPRPKHEHESDEKQPETLDGARKKSLSMAPAVSTAALIEDDPWEMMELQDTGVKWADLDTKKKVLRVFTTAAKLIMLLGLLYMFVCSLDVLSSAFQLVGGKAAGDIFQENKVLSNPLAGLVIGMLVTLLVQSSSTSSSIVVSMVSSGMLEVATAVPIIMGTNIGTSVTNTLVAIAQVGDRNKFRRAFAGATVHDFFNWLSVLVLLPLEVASGYLEKVTSLIVRSFNIESGEKAPALLNVITDPLTHSIIQLDESVMSGIAVGDPEARNKSLIKVWCHTASNTTVQNVTTTNCTSPSLCWTDGIQNWTMKNVTEIINIKKCSHIFVNTSLSDLAVGLILLAGSLLILCTCLICIVKLLNSMLKGQVAVVIKKIVNTDFPFPFAWLTGYIAILVGAGMTFIVQSSSVFTSAITPLVGIGVISIERAYPLSLGSNIGTTTTAILAAMASPGETLGNSLQIALVHFFFNLSGILLWYPIPITRIPIRLAKGLGETTAQYRWFAAFYIILCFFGLPLLVFGLSMAGWQVLMGVLVPIAVILIFAIIVNILQKHKPQWLPSALRSWDFLPLWAHSLDPWDRVVTVIAARCCCCCKCCNSNEEDEKAKLENLANGIEINDNTMTTVEIIEPKKTVDSCEILKATSL
>9Z60_E2 len=228 copies=1 longest=0
QIQLVQSGPELKKPGETVKISCRASGYTFTNCGMNWVKQAPGKGLKWMGWINTYTGEPTYADDFKGRFAFSLETSANTAYLQISNLKNEDTATYFCARNYYYGSTYRGFDYWGQGTTLTVSSAKTTPPSVYPLAPGCGDTTGSSVTLGCLVKGYFPESVTVTWNSGSLSSSVHTFPALLQSGLYTMSSSVTVPSSTWPSQTVTCSVAHPASSTTVDKKLEPSGPISTI
>9Z60_E3 len=218 copies=1 longest=0
DIVLTQSPASLAVSLGQRATISCRASESVDNFGISFMHWYHQKPGQPPKLLIYRASNLESGIPARFSGSGSGTDFTLTINPVEADDVATYFCQQSNEDPYTFGGGTKLEIKRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKDINVKWKIDGSERQNGVLNSWTDQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC
>9Z62_E1 len=646 copies=1 longest=1
GPEFATMAPRPKHEHESDEKQPETLDGARKKSLSMAPAVSTAALIEDDPWEMMELQDTGVKWADLDTKKKVLRVFTTAAKLIMLLGLLYMFVCSLDVLSSAFQLVGGKAAGDIFQENKVLSNPLAGLVIGMLVTLLVQSSSTSSSIVVSMVSSGMLEVATAVPIIMGTNIGTSVTNTLVAIAQVGDRNKFRRAFAGATVHDFFNWLSVLVLLPLEVASGYLEKVTSLIVRSFNIESGEKAPALLNVITDPLTHSIIQLDESVMSGIAVGDPEARNKSLIKVWCHTASNTTVQNVTTTNCTSPSLCWTDGIQNWTMKNVTEIINIKKCSHIFVNTSLSDLAVGLILLAGSLLILCTCLICIVKLLNSMLKGQVAVVIKKIVNTDFPFPFAWLTGYIAILVGAGMTFIVQSSSVFTSAITPLVGIGVISIERAYPLSLGSNIGTTTTAILAAMASPGETLGNSLQIALVHFFFNLSGILLWYPIPITRIPIRLAKGLGETTAQYRWFAAFYIILCFFGLPLLVFGLSMAGWQVLMGVLVPIAVILIFAIIVNILQKHKPQWLPSALRSWDFLPLWAHSLDPWDRVVTVIAARCCCCCKCCNSNEEDEKAKLENLANGIEINDNTMTTVEIIEPKKTVDSCEILKATSL
>9Z62_E2 len=228 copies=1 longest=0
QIQLVQSGPELKKPGETVKISCRASGYTFTNCGMNWVKQAPGKGLKWMGWINTYTGEPTYADDFKGRFAFSLETSANTAYLQISNLKNEDTATYFCARNYYYGSTYRGFDYWGQGTTLTVSSAKTTPPSVYPLAPGCGDTTGSSVTLGCLVKGYFPESVTVTWNSGSLSSSVHTFPALLQSGLYTMSSSVTVPSSTWPSQTVTCSVAHPASSTTVDKKLEPSGPISTI
>9Z62_E3 len=218 copies=1 longest=0
DIVLTQSPASLAVSLGQRATISCRASESVDNFGISFMHWYHQKPGQPPKLLIYRASNLESGIPARFSGSGSGTDFTLTINPVEADDVATYFCQQSNEDPYTFGGGTKLEIKRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKDINVKWKIDGSERQNGVLNSWTDQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC
>9Z64_E1 len=646 copies=1 longest=1
GPEFATMAPRPKHEHESDEKQPETLDGARKKSLSMAPAVSTAALIEDDPWEMMELQDTGVKWADLDTKKKVLRVFTTAAKLIMLLGLLYMFVCSLDVLSSAFQLVGGKAAGDIFQENKVLSNPLAGLVIGMLVTLLVQSSCTSSSIVVSMVSSGMLEVATAVPIIMGTNIGTSVTNTLVAIAQVGDRNKFRRAFAGATVHDFFNWLSVLVLLPLEVASGYLEKVTSLIVRSFNIESGEKAPALLNVITDPLTHSIIQLDESVMSGIAVGDPEARNKSLIKVWCHTASNTTVQNVTTTNCTSPSLCWTDGIQNWTMKNVTEIINIKKCSHIFVNTSLSDLAVGLILLAGSLLILCTCLICIVKLLNSMLKGQVAVVIKKIVNTDFPFPFAWLTGYIAILVGAGMTFIVQSSSVFTSAITPLVGIGVISIERAYPLSLGSNIGTTTTAILAAMASPGETLGNSLQIALVHFFFNLSGILLWYPIPITRIPIRLAKGLGETTAQYRWFAAFYIILCFFGLPLLVFGLSMAGWQVLMGVLVPIAVILIFAIIVNILQKHKPQWLPSALRSWDFLPLWAHSLDPWDRVVTVIAARCCCCCKCCNSNEEDEKAKLENLANGIEINDNTMTTVEIIEPKKTVDSCEILKATSL
>9Z64_E2 len=228 copies=1 longest=0
QIQLVQSGPELKKPGETVKISCRASGYTFTNCGMNWVKQAPGKGLKWMGWINTYTGEPTYADDFKGRFAFSLETSANTAYLQISNLKNEDTATYFCARNYYYGSTYRGFDYWGQGTTLTVSSAKTTPPSVYPLAPGCGDTTGSSVTLGCLVKGYFPESVTVTWNSGSLSSSVHTFPALLQSGLYTMSSSVTVPSSTWPSQTVTCSVAHPASSTTVDKKLEPSGPISTI
>9Z64_E3 len=218 copies=1 longest=0
DIVLTQSPASLAVSLGQRATISCRASESVDNFGISFMHWYHQKPGQPPKLLIYRASNLESGIPARFSGSGSGTDFTLTINPVEADDVATYFCQQSNEDPYTFGGGTKLEIKRADAAPTVSIFPPSSEQLTSGGASVVCFLNNFYPKDINVKWKIDGSERQNGVLNSWTDQDSKDSTYSMSSTLTLTKDEYERHNSYTCEATHKTSTSPIVKSFNRNEC
>9Z6N_E1 len=392 copies=2 longest=1
MSQGRGKYDFYIGLGLAMSSSIFIGGSFILKKKGLLRLARKGSMRAGQGGHAYLKEWLWWAGLLSMGAGEVANFAAYAFAPATLVTPLGALSVLVSAILSSYFLNERLNLHGKIGCLLSILGSTVMVIHAPKEEEIETLNEMSHKLGDPGFVVFATLVVIVALILIFVVGPRHGQTNILVYITICSVIGAFSVSCVKGLGIAIKELFAGKPVLRHPLAWILLLSLIVCVSTQINYLNRALDIFNTSIVTPIYYVFFTTSVLTCSAILFKEWQDMPVDDVIGTLSGFFTIIVGIFLLHAFKDVSFSLASLPVSFRKDEKAMNGNLSNMYEVLNNNEESLTCGIEQHTGENVSRRNGNLTAFSNSLEVLFQGPAADYKDDDDKAHHHHHHHHHH
>9Z7R_E1 len=392 copies=2 longest=1
MSQGRGKYDFYIGLGLAMSSSIFIGGSFILKKKGLLRLARKGSMRAGQGGHAYLKEWLWWAGLLSMGAGEVANFAAYAFAPATLVTPLGALSVLVSAILSSYFLNERLNLHGKIGCLLSILGSTVMVIHAPKEEEIETLNEMSHKLGDPGFVVFATLVVIVALILIFVVGPRHGQTNILVYITICSVIGAFSVSCVKGLGIAIKELFAGKPVLRHPLAWILLLSLIVCVSTQINYLNRALDIFNTSIVTPIYYVFFTTSVLTCSAILFKEWQDMPVDDVIGTLSGFFTIIVGIFLLHAFKDVSFSLASLPVSFRKDEKAMNGNLSNMYEVLNNNEESLTCGIEQHTGENVSRRNGNLTAFSNSLEVLFQGPAADYKDDDDKAHHHHHHHHHH
>9Z7S_E1 len=392 copies=2 longest=1
MSQGRGKYDFYIGLGLAMSSSIFIGGSFILKKKGLLRLARKGSMRAGQGGHAYLKEWLWWAGLLSMGAGEVANFTAYAFAPATLVTPLGALSVLVSAILSSYFLNERLNLHGKIGCLLSILGSTVMVIHAPKEEEIETLNEMSHKLGDPGFVVFATLVVIVALILIFVVGPRHGQTNILVYITICSVIGAFSVSCVKGLGIAIKELFAGKPVLRHPLAWILLLSLIVCVSTQINYLNRALDIFNTSIVTPIYYVFFTTSVLTCSAILFKEWQDMPVDDVIGTLSGFFTIIVGIFLLHAFKDVSFSLASLPVSFRKDEKAMNGNLSNMYEVLNNNEESLTCGIEQHTGENVSRRNGNLTAFSNSLEVLFQGPAADYKDDDDKAHHHHHHHHHH
>9Z7T_E1 len=392 copies=2 longest=1
MSQGRGKYDFYIGLGLAMSSSIFIGGSFILKKKGLLRLARKGSMRAGQGGHAYLKEWLWWAGLLSMGAGEVANFAAYAFAPATLVTPLGALSVLVSAILSSYFLNERLNLHGKIGCLLSILGSTVMVIHAPKEEEIETLNEMSHKLGDPGFVVFATLVVIVALILIFVVGPRHGQTNILVYITICSVIGAFSVSCVKGLGIAIKELFAGKPVLRHPLAWILLLSLIVCVSTQINYLNRALDIFNTSIVTPIYYVFFTTSVLTCSAILFKEWQDMPVDDVIGTLSGFFTIIVGIFLLHAFKDVSFSLASLPVSFRKDEKAMNGNLSNMYEVLNNNEESLTCGIEQHTGENVSRRNGNLTAFSNSLEVLFQGPAADYKDDDDKAHHHHHHHHHH
>9ZBV_E1 len=134 copies=4 longest=1
MHHHHHHGPTGTGESKAPLMVKVLDAVRGSPAINVAVHVFRKAADDTWEPFASGKTSESGELHGLTTEEEFVEGIYKVEIDTKSYWKALGISPFHEHAEVVFTANDSGPRRYTIAALLSPYSYSTTAVVTNPKE
>9ZDY_E1 len=134 copies=4 longest=1
MHHHHHHGPTGTGESKCPLMVKVLDAVRGSPAINVAVHVFRKAADDTWEPFASGKTSESGELHGLTTEEEFVEGIYKVEIDTKSYWKALGISPFHEHAEVVFTANDSGPRRYTIAALLSPYSYSTTAVVTNPKE
>9ZK8_E1 len=208 copies=1 longest=1
DFVPCPVDTVQIVLPPHLERIREKLAENIHELWALTRIEQGWTYGPVRDDNKRLHPCLVNFHSLPEPERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPKTYMMSNGYKPAPLDLSHVRLTPAQTTLVDRLAENGHNVWARDRVAQGWSYSAVQDIPARRNPRLVPYRLLDEATKRSNRDSLCQAVRTLLGYGYNIEPPD
>9ZKC_E1 len=208 copies=1 longest=1
DFVPCPVDTVQIVLPPHLERIREKLAENIHELWALTRIEQGWTYGPVRDDNKRLHPCLVNFHSLPEPERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPKTYMMSNGYKPAPLDLSHVRLTPAQTTLVDRLAENGHNVWARDRVAQGWSYSAVQDIPARRNPRLVPYRLLDEATKRSNRDSLCQAVRTLLGYGYNIEPPD
>9ZKD_E1 len=208 copies=1 longest=1
DFVPCPVDTVQIVLPPHLERIREKLAENIHELWALTRIEQGWTYGPVRDDNKRLHPCLVNFHSLPEPERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPKTYMMSNGYKPAPLDLSHVRLTPAQTTLVDRLAENGHNVWARDRVAQGWSYSAVQDIPARRNPRLVPYRLLDEATKRSNRDSLCQAVRTLLGYGYNIEPPD
>9ZL7_E1 len=208 copies=1 longest=1
DFVPCPVDTVQIVLPPHLERIREKLAENIHELWALTRIEQGWTYGPVRDDNKRLHPCLVNFHSLPEPERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPKTYMMSNGYKPAPLDLSHVRLTPAQTTLVDRLAENGHNVWARDRVAQGWSYSAVQDIPARRNPRLVPYRLLDEATKRSNRDSLCQAVRTLLGYGYNIEPPD
>9ZLC_E1 len=208 copies=1 longest=1
DFVPCPVDTVQIVLPPHLERIREKLAENIHELWALTRIEQGWTYGPVRDDNKRLHPCLVNFHSLPEPERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPKTYMMSNGYKPAPLDLSHVRLTPAQTTLVDRLAENGHNVWARDRVAQGWSYSAVQDIPARRNPRLVPYRLLDEATKRSNRDSLCQAVRTLLGYGYNIEPPD
>9ZLF_E1 len=206 copies=1 longest=1
DFVPCPVDTVQIVLPPHLERIREKLAENIHELWALTRIEQGWTYGPVRDDNKRLHPCLVNFHSLPEPERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPKTYMMSNGYKPAPLDLSHVRLTPAQTTLVDRLAENGHNVWARDRVAQGWSYSAVQDIPARRNPRLVPYRLLDEATKRSNRDSLCQAVRTLLGYGYNIEP
>9ZLH_E1 len=206 copies=1 longest=1
DFVPCPVDTVQIVLPPHLERIREKLAENIHELWALTRIEQGWTYGPVRDDNKRLHPCLVNFHSLPEPERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPKTYMMSNGYKPAPLDLSHVRLTPAQTTLVDRLAENGHNVWARDRVAQGWSYSAVQDIPARRNPRLVPYRLLDEATKRSNRDSLCQAVRTLLGYGYNIEP
>9ZNO_E1 len=365 copies=2 longest=1
HHHHHHMIYHLAILLREHFFAFNVLKYITFRSFTAILLAFFITLILSPTFMKKFAKIQRLFGGYVREYTPEHHESKKYTPTMGGVVIVTVILITSVLLMRLDIRYTWVLVFSTLSFALIGFVDDWIKLKNKKGLSIKAKLAFQMSFALAVSLLIFYWVGLETKLYFPFFKELTVDLGWLYIPFSMFIIVGTANAVNLTDGLDGLAIGPSMTTATAFGVIAYVVGHSKIAQYLGVPHVPYAGEITVFCFAIIGAGLGFLWFNTYPAQVFMGDVGALGLGAALATVSIMTKSEFLLAVAGGVFVFETVTVILQIIYFRATGGKRLFRKAPFHHHLEEKGLDEPKIVVRMWIVSALLAIVSVAMLKLR
>9ZR7_E1 len=304 copies=1 longest=1
MKHHHHHHSSGLNDIFEAQKIEWHEENLYFQGSMTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETSLLDILDTAGKEEYSAMRDQYMRTGEGFLLVFAINNSKSFADINLYREQIKRVKDSDDVPMVLVGNKSDLPTRTVDTKQAHELAKSYGIPFIETSAKTRQGVEDAFYTLVREIRQYRMADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQ
>9ZR7_E2 len=228 copies=1 longest=0
EISEVQLVESGGGLVQPGGSLRLSCAASGFNVVDFSLHWVRQAPGKGLEWVAYISSSSGSTSYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARWGYWPGEPWWKAFDYWGQGTLVTVFNQIKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEPKS
>9ZR7_E3 len=213 copies=1 longest=0
DIQMTQSPSSLSASVGDRVTITCRASQSVSSAVAWYQQKPGKAPKLLIYSASSLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQYLYYSLVTFGQGTKVEIKRTVAAPSVFIFPPSDSQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRG
>9ZRT_E1 len=357 copies=1 longest=1
MHHHHHHGLNDIFEAQKIEWHEGSGSNITTLTNSISTSSFSPNNTNGSSTETVNSDIKTTTSSHPVSSLTMLNDTLHNIRTTNQALKKELSQKTLTKTSLEEIALHSSQISMDVNKSAQLLDILSRNEYPINKDARELLHSAPKEAELDGDQMISHRELWAKIANSINDINEQYLKVYEHAVSSYTQMYQDFSAVLSSLAGWISPGGNDGNSVKLQVNSLKKALEELKEKYKDKPLYPANNTVSQEQANKWLTELGGTIGKVSQKNGGYVVSINMTPIDNMLKSLDNLGGNGEVVLDNAKYQAWNAGFSAEDETMKNNLQTLVQKYSNANSIFDNLVKVLSSTISSCTDTDKLFLHF
>9ZRT_E2 len=232 copies=1 longest=0
EVQLAESGPGLVKPLETLSLTCAVSGDYIRNAHYWSWIRQVPGKGPEWMGGIDGRSSSTDYNPSLESRVTVSTDTSNNQLSLKLTSVTAADTAVYYCARDARTGVIFMPTNWFELWGLGVLVTVSSASTKGPSVFPLAPSSRSTSESTAALGCLVKDYFPEPVTVSWNSGSLTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYVCNVNHKPSNTKVDKRVEIKTCGGG
>9ZRT_E3 len=214 copies=1 longest=0
DIVMTQSPSSLSACVGDRVTITCRASQDIGAFLNWYQQKPGKAPKFLVLDRNRLEGGVPSRFSGSGSGTEFTLTISSLQPEDFATYYCQQHKNFPFTFGPGTKVEIKRTVAAPSVFIFPPSEDQVKSGTVSVVCLLNNFYPREASVKWKVDGALKTGNSQESVTEQDSKDNTYSLSSTLTLSSTEYQSHKVYACEVTHQGLSSPVTKSFNRGEC


In [ ]:
%%writefile data/pool/pool_ids_all_cryoem.txt
10AC
10AD
10AL
10AM
10AW
10AY
10BE
10BU
10DB
10DK
10DN
10DP
10DQ
10DW
10EC
10ED
10EG
10EH
10EN
10EO
10EP
10EQ
10ER
10ES
10ET
10EU
10EV
10EX
10EY
10EZ
10FA
10FB
10FC
10FD
10FE
10FF
10FI
10FJ
10FL
10FM
10FN
10FO
10FT
10FZ
10GH
10GJ
10GK
10GL
10GM
10GN
10GO
10GX
10IC
10ID
10IE
10IF
10IG
10IH
10IJ
10IK
10JA
10JB
10JC
10JF
10JG
10JH
10JI
10JJ
10JK
10KF
10KR
10LH
10LJ
10LK
10LL
10LM
10LN
10MA
10MB
10MC
10MD
10ME
10MF
10MG
10MI
10MJ
10MK
10ML
10MR
10MS
10MT
10MU
10MY
10MZ
10NA
10NB
10NK
10NL
10NM
10NO
10NP
10NQ
10NR
10NT
10OG
10OP
10OR
10OY
10OZ
10PB
10PC
10PD
10PF
10PL
10PM
10PP
10QM
10QQ
10QR
10QT
10QX
10QY
10RI
10RJ
10RK
10RN
10RR
10RX
10SD
10SE
10SF
10SG
10SH
10SJ
10SM
10SS
10TL
10TM
10TP
10TQ
10TU
10UA
10UJ
10UP
10VM
10WC
10WP
10XU
10XW
10XZ
10YA
10YB
10YC
10YD
10YE
10YF
10YG
10YH
10YI
10ZK
10ZL
10ZM
10ZN
10ZP
10ZT
10ZU
10ZY
10ZZ
11AA
11AG
11AH
11AI
11AK
11AM
11BB
11BC
11BE
11BP
11BQ
11BR
11BT
11CJ
11CK
11CL
11CN
11CO
11CQ
11CV
11CZ
11DA
11DB
11DC
11DD
11DE
11DF
11DG
11DR
11DV
11EC
11ED
11EE
11EH
11EI
11EL
11EN
11EO
11EP
11EQ
11EU
11EV
11FG
11FH
11FS
11FV
11FW
11GG
11GI
11GJ
11GK
11GO
11GP
11GQ
11HC
11HD
11HE
11HF
11HG
11HK
11HL
11HM
11HN
11HO
11HR
11HT
11HU
11HV
11HW
11HX
11HZ
11IA
11IB
11IC
11ID
11IE
11IM
11IQ
11IS
11IT
11IW
11IY
11JB
11JC
11JD
11JF
11JJ
11KC
11KH
11KM
11LC
11LL
11LP
11LQ
11LR
11LS
11LT
11LX
11ME
11MF
11MG
11MH
11MK
11ML
11MN
11MP
11MR
11MS
11MT
11MU
11MV
11MX
11MY
11MZ
11NA
11NB
11NC
11ND
11NE
11NF
11NH
11NI
11NJ
11NK
11NL
11NM
11NN
11NO
11NP
11NR
11NT
11NU
11NW
11NX
11NZ
11OA
11OB
11OC
11OH
11OI
11OK
11OL
11OO
11OQ
11OR
11OU
11OV
11PB
11QC
11RL
11RM
11RN
11SM
11SO
11SQ
11SR
11SS
11ST
11SU
11SV
11SX
11SY
11SZ
11TA
11TD
11TF
11TH
11TK
11TL
11UF
11UG
11UH
11UI
11UY
11VE
11VQ
11VV
11VW
11VX
11VY
11VZ
11WA
11WB
11WC
11WD
11WE
11WN
11WO
11WP
11WQ
11WR
11WZ
11XB
11XC
11XU
11XX
11XY
11XZ
11YA
11YB
11YE
11YJ
11YZ
11ZC
11ZJ
11ZK
11ZS
11ZV
11ZW
12AD
12AG
12AH
12AI
12AP
12AQ
12AR
12AT
12BI
12BJ
12BK
12BL
12BM
12BN
12BP
12BW
12BX
12BY
12BZ
12DK
12DL
12DR
12DS
12DT
12DU
12DV
12DW
12EB
12EC
12ED
12EE
12EF
12EG
12EH
12EI
12EQ
12ER
12ES
12ID
12IE
12IF
12IU
12IV
12IY
12IZ
12JA
12JP
12JT
12JV
12JZ
12KE
12OF
12OG
12OY
12OZ
12PB
12QJ
12RI
12RJ
12RP
12RT
12RU
12RV
12SL
12SN
12US
12UT
12UU
12UV
12ZE
12ZN
13AJ
13BD
13BF
13BR
13BS
13BV
13CM
13CO
13CP
13DE
13DJ
13DM
13DU
13DV
13ER
13ES
13EU
13FM
13FN
13FO
13FU
13FZ
13GH
13GJ
13GM
13GQ
13HE
13HQ
13HR
13HS
13IB
13IO
13JV
13KK
13KW
13LJ
13LK
13LL
13LM
13LN
13LO
13LP
13LQ
13LR
13LS
13LT
13LU
13MF
20YC
20YJ
20YR
20YS
20YV
20YW
20YY
20ZC
20ZD
20ZG
20ZH
20ZI
20ZJ
20ZK
20ZL
20ZN
20ZX
21AG
21AH
21AK
21AL
21AM
21AO
21AQ
21AU
21AV
21BH
21BO
21BP
21BQ
21CB
21CN
21CO
21CQ
21DI
21DU
21DX
21DZ
21EA
21EI
21EN
21EO
21ES
21FB
21FF
21FG
21FH
21FI
21FJ
21FK
21FY
21GF
21GG
21GL
21GM
21GQ
21GR
21GT
21IM
21IQ
21IR
21JU
21JV
21KL
21KR
21KS
21KT
21LT
21MW
21NP
21NQ
21NR
21NS
21NT
21NW
21NX
21NY
21NZ
21OR
21OU
21RH
21RJ
21RK
21RN
21RO
21RP
21RS
21RT
21TP
21TQ
21TV
21TW
21TY
21UY
21VL
21VM
21VQ
21VV
21WA
21WC
21WD
21WE
21WG
21WH
21WI
21WJ
21WL
21WO
21WV
21WY
21XJ
21XK
21XL
21XM
21XO
21ZH
21ZT
22AG
22AJ
22AK
22AV
22AX
22AY
22AZ
22BA
22BC
22BD
22BE
22BF
22BG
22BH
22BI
22BJ
22BK
22BU
22BV
22CR
22DX
22DY
22DZ
22EA
22EB
22EC
22EM
22ES
22ET
22EU
22FN
22FR
22FX
22FY
22GK
22GL
22GM
22HG
22HP
22IB
22IX
22JW
22JX
22JY
22JZ
22KG
22KK
22KQ
22LL
22MM
22OA
22OF
22OK
22OV
22PD
22PL
22PS
22QS
22QT
22QU
22QV
22QW
22RD
22RJ
22RQ
22RX
22RY
22SF
22SK
22TE
22TH
22TR
22TU
22VL
22VT
22WP
22WR
22WT
22XC
22XK
22XM
22XO
22ZO
23AS
23AT
23AY
23BB
23BD
23BG
23BT
23BX
23CP
23DJ
23EW
23EX
23EY
23FA
23FB
23FC
23FD
23FE
23FJ
23FU
23GD
23GF
23GX
23IC
23IF
23IG
23IL
23IT
23IU
23IV
23IW
23NX
23OI
23OR
23PG
23PH
23PI
23PS
23PX
23PY
23PZ
23RS
23SF
23SG
23SH
23SO
23SP
23SQ
23SR
23TT
23TU
23TV
23TX
23TY
23VU
23VV
23VW
23WI
23WJ
23WM
23WN
23WO
23WQ
23XI
23XK
23XM
23XO
24EW
24GL
24GT
24ID
24IP
24KR
24MC
24MQ
24NA
24NB
24NC
24ND
24QH
24QP
24QR
24QS
24QT
24RC
24RT
24SV
24SW
24TB
24TI
24UM
24VC
24VD
24VE
24VI
24VJ
24VK
24VL
24VM
24VN
24VO
24VP
24VQ
24VR
24VS
24VT
24VU
24VV
24VW
24VX
24VY
24VZ
24WA
24XB
24XX
24XY
24XZ
24YE
24YW
25EX
25FQ
25GE
25GF
25GN
25HN
25HO
25HR
25HS
25HW
25IF
25IH
25II
25IJ
25IK
25IL
25IN
25IO
25IP
25JN
25JP
25JX
25LE
25LV
25MI
25MJ
25MK
25NV
25NX
25PV
25QK
25QL
25QP
25RG
25RV
25RW
25RY
25RZ
25SA
25SB
25ST
25SU
25SV
25SW
25SX
25SY
25TV
25TW
25TX
25TY
25UJ
25VD
25VE
25VF
25VJ
25VK
25WF
25WG
25WZ
25XL
25XM
25XN
25XO
26CI
26CW
26CZ
26DA
26DE
26DK
26FB
26FE
26FV
26GA
26GP
26GS
26HH
26JZ
26KA
26KB
26KD
26LA
26LK
26NL
26PC
26PF
26QF
26QG
26QH
26QI
26QJ
26QM
26QS
26QT
26SG
26VA
26VB
26VC
26VD
26VE
26WC
26XH
26ZN
27EM
27KE
27KF
27OA
27PT
27PY
27PZ
27QA
27QB
27TM
27UR
27UT
27WM
27WR
27XK
27ZD
27ZV
28EP
28IK
28IZ
28JC
28JL
28JM
28JQ
28JR
28JS
28JU
28JV
28JW
28KE
28LJ
28LN
28LO
28LP
28LQ
28LU
28LW
28MC
28ME
28MJ
28MK
28MO
28MP
28MQ
28NO
28NT
28OA
28OB
28OC
28OE
28OP
28PN
28PP
28PQ
28PR
28QL
28QM
28QN
28RF
28RO
28SD
28TO
28UH
28UI
28UJ
28UK
28UL
28UM
28UN
28VJ
28VY
28WY
28WZ
28XA
28XB
28XG
28XH
28YC
28YE
28YJ
28YM
28YN
28YO
28YP
28YQ
28YR
28ZU
28ZV
28ZX
28ZY
29CP
29EX
29GV
29HB
29HG
29HH
29HI
29HP
29HQ
29IN
29IW
29IX
29IY
29IZ
29JF
29JR
29KM
29KN
29OS
29OU
29RF
29RT
29TC
29TE
29TI
29TK
29TX
29UB
29VF
29WD
29XZ
29YA
29YB
29YC
29YD
29YE
30BT
30BU
30DR
30EP
30ES
30FF
30FH
30FM
30GA
30GB
30GD
30GF
30GH
30GT
30HD
30HI
30HU
30IE
30JE
30JV
30JW
30JZ
30KH
30KW
30KZ
30LA
30LD
30MN
30OD
30OE
30OF
30OG
30QM
30QN
30QU
30QX
30QZ
30RA
30RB
30SJ
30UC
30US
30UT
30UU
30UV
30WC
30XT
30YS
30YT
30YU
30YV
30YW
30YX
30YY
30YZ
30ZA
31AK
31BG
31CJ
31CK
31EP
31ES
31EU
31EV
31EW
31EY
31EZ
31FW
31FX
31FY
31FZ
31HX
31HY
31IB
31IC
31JO
31LE
31LF
31MR
31OX
31SY
31VX
31ZN
32FD
32FE
32FF
32FI
32GD
32GE
32KS
32NJ
32PZ
32QA
32QB
32TO
32UL
32UM
32UN
32YY
32ZB
33AR
33AS
33AT
33AU
33AV
33AW
33BF
33CG
33CH
33CI
33HM
34AD
35SE
35SF
35SG
35SY
35TE
35TL
35TR
35TW
35TX
35ZR
35ZV
35ZW
36AX
36AZ
36BB
36BD
36BL
36BM
36BT
36CG
36CP
36CQ
36CR
36CW
36DK
36DN
36HT
36HU
36HW
36IS
36IU
36IV
36JA
36JB
36JK
36JL
36PB
36PC
36PD
36PE
36PF
36PG
36PJ
36PK
36PM
36PO
36PP
36PR
36PY
36QD
36QU
36QV
36QW
36QX
36RO
36RP
36RS
36UK
36UP
36YD
36ZA
37BM
37BO
37BP
37BQ
37BR
37BS
37CY
37HT
37IB
37ID
37IX
37KX
37LU
37LV
37SO
37UN
37UO
37XD
37YA
38FU
38FV
38FW
38FX
38FY
38HM
38JC
38JD
38JE
38JL
38KL
38KM
38KN
38KO
38KP
38KQ
38KY
38KZ
38LA
38LB
38LP
38OA
38OK
38PA
38PM
38RH
38RI
38RK
38RR
38RX
38TD
43DY
43FX
43IB
43IE
43IF
43IG
43JF
43JG
43JH
43KN
43KO
43LS
43MI
43MU
43SN
43YY
44AB
44BB
44BV
44BW
44FH
44FI
44KH
44NX
44NY
44ZG
45IL
45IM
45IO
45IP
45IQ
45IR
46KE
8QKP
8SUP
8U9S
8UBI
8UBJ
8UBK
8UBL
8UX4
8V9T
8Y71
8YRQ
9BID
9BIE
9BJ8
9BJ9
9BJI
9BJJ
9CDQ
9CIN
9CK4
9CK6
9CKI
9CLA
9CLM
9CLT
9CLV
9CMG
9CMM
9CO4
9COC
9COE
9COF
9COG
9COH
9CON
9CTY
9CWP
9CWQ
9CWR
9CZB
9CZC
9D72
9D73
9D74
9DC4
9DEI
9DHX
9DOV
9DP7
9DPD
9DPI
9DSL
9DWT
9DX0
9DX1
9DX2
9DX3
9DX4
9DX5
9DXD
9DXF
9DXG
9DXH
9DXJ
9DXK
9DZ4
9DZF
9DZG
9DZH
9DZJ
9DZK
9E00
9E05
9E09
9E0F
9E0G
9E0L
9E0R
9E16
9E1S
9E2D
9E2F
9E2Q
9E2R
9E3D
9E3I
9E3R
9E3U
9E49
9E4A
9E4B
9E4C
9E4D
9E4E
9E4F
9E4G
9E4H
9E4I
9E4J
9E4K
9E5N
9E5U
9E5V
9E5X
9E6G
9E6J
9E7I
9E93
9E94
9E9D
9E9E
9E9V
9EB9
9EBA
9EBB
9EBC
9EBT
9EC0
9EC3
9EC7
9EDR
9EDS
9EDT
9EEB
9EEK
9EEL
9EEM
9EEN
9EEO
9EEP
9EEQ
9EER
9EES
9EEU
9EFH
9EFS
9EFT
9EGG
9EHO
9EJ9
9EJA
9EJB
9EKA
9ELS
9EVT
9FQZ
9FTE
9FY1
9FY2
9FY3
9G2F
9G2H
9G41
9G42
9G44
9G83
9G9Y
9G9Z
9GAE
9GJ5
9GJ6
9GJA
9GJB
9GJC
9GJD
9GJE
9GJF
9GJG
9GJH
9GJI
9GJJ
9GJK
9GJL
9GJM
9H1H
9H1V
9H21
9H2G
9H2M
9H2Q
9H3G
9H4J
9H4K
9H4L
9H4M
9H50
9H51
9H52
9H54
9H55
9H5I
9H6I
9H6T
9H72
9H7U
9H8N
9H8O
9HBP
9HES
9HFK
9HIG
9HJQ
9HJX
9HKT
9HMW
9HN1
9HN2
9HNI
9HP8
9HP9
9HTW
9HU4
9HU5
9HU6
9HUE
9HUF
9HUG
9HUM
9HUQ
9HUS
9HUX
9HUY
9HUZ
9HV0
9HV4
9HV5
9HV6
9HVY
9HWJ
9HWK
9HWL
9HXB
9HXD
9HXE
9HXF
9HXG
9HXH
9HXI
9HXJ
9HXL
9HXM
9HXN
9HY6
9HYF
9HYI
9HYM
9HYU
9HYW
9HZG
9I06
9I07
9I14
9I1Z
9I28
9I2D
9I2E
9I2F
9I2G
9I3L
9I3P
9I3R
9I3S
9I3T
9I3V
9I4T
9I5C
9I5N
9I5Z
9I60
9I61
9I66
9I6D
9I7M
9I7V
9I7W
9I7X
9I82
9I83
9I88
9I89
9I8C
9I8U
9I9Y
9IAN
9IAO
9IAR
9IAS
9IAX
9IAZ
9IB0
9IB1
9IB2
9IB3
9IBC
9IBD
9IBL
9IBS
9IBW
9IF5
9IF7
9IF8
9IFC
9IFM
9IFO
9IG8
9IGJ
9IGS
9IGW
9IGX
9IHH
9IHN
9INZ
9IP5
9IPM
9IPN
9IPO
9IRD
9IRF
9IRG
9IRI
9IRS
9IRU
9IU2
9IU3
9IU4
9IU5
9IU6
9IU7
9IU8
9IU9
9IUA
9IUB
9IVJ
9IWR
9IX6
9IY6
9IYC
9J4C
9J5I
9J5K
9J5L
9J5N
9J5O
9J5R
9J6R
9J6S
9J6T
9J6U
9J6V
9J87
9J92
9J93
9J94
9J95
9J9E
9J9X
9JB7
9JBS
9JBT
9JBU
9JC0
9JC8
9JCS
9JHB
9JHC
9JHD
9JKH
9JKI
9JKJ
9JKK
9JKM
9JKR
9JL3
9JLP
9JN1
9JN2
9JPF
9JPG
9JPH
9JPM
9JPO
9JPP
9JQ2
9JQO
9JRD
9JRE
9JRF
9JRG
9JRH
9JRJ
9JY3
9JY4
9JZI
9JZJ
9JZK
9JZL
9JZN
9JZR
9K11
9K12
9K13
9K14
9K15
9K16
9K17
9K18
9K19
9K1E
9K2Z
9K30
9K31
9K32
9K3S
9K3X
9K3Y
9K4A
9K4B
9K4J
9K4K
9K4L
9K4M
9K4O
9K4P
9K4R
9K4S
9K4T
9K4U
9K4V
9K4W
9K4X
9K4Y
9K4Z
9K50
9K51
9K53
9K54
9K55
9K56
9K57
9K58
9K59
9K5A
9K5B
9K5C
9K5D
9K5E
9K6X
9K7G
9K7S
9K7V
9K88
9K8Y
9K90
9K91
9K9G
9K9H
9K9J
9K9K
9KAF
9KAG
9KBY
9KBZ
9KC4
9KCF
9KDT
9KDU
9KDV
9KDW
9KH6
9KKN
9KKO
9KOF
9KOG
9KOH
9KPZ
9KQ0
9KQ5
9KQM
9KSL
9KU1
9KVS
9KWY
9KZG
9KZH
9L06
9L0A
9L19
9L1A
9L1B
9L1C
9L1D
9L1E
9L1L
9L2O
9L2T
9L36
9L38
9L3I
9L3Q
9L58
9L5D
9L5I
9L6I
9L6O
9L6P
9L76
9L8M
9L8N
9L8O
9L8W
9L94
9L95
9L96
9L98
9L9C
9L9H
9L9J
9L9K
9L9O
9L9V
9LAA
9LAB
9LAH
9LBQ
9LBW
9LCK
9LCO
9LCP
9LCQ
9LD2
9LDJ
9LDV
9LDW
9LDX
9LDZ
9LE0
9LE1
9LE2
9LEB
9LEG
9LET
9LEU
9LEV
9LEX
9LEY
9LEZ
9LFA
9LFC
9LFD
9LGT
9LHZ
9LI1
9LIH
9LIQ
9LIR
9LIS
9LIV
9LIW
9LIX
9LIY
9LJ0
9LJ4
9LJA
9LJB
9LJC
9LJE
9LJF
9LJI
9LK2
9LK6
9LKB
9LKD
9LKJ
9LKZ
9LL7
9LL8
9LL9
9LLA
9LLB
9LLC
9LLE
9LLF
9LLG
9LLH
9LLI
9LLJ
9LLY
9LLZ
9LM0
9LM5
9LM6
9LN4
9LN6
9LN7
9LN9
9LNA
9LNM
9LNN
9LNO
9LO8
9LOE
9LOF
9LOG
9LOU
9LOV
9LOX
9LOY
9LOZ
9LP0
9LP1
9LPK
9LPW
9LPZ
9LQJ
9LQO
9LQP
9LQQ
9LQT
9LQU
9LQV
9LQW
9LQX
9LQY
9LQZ
9LR0
9LR1
9LR2
9LR3
9LRK
9LRM
9LRS
9LRV
9LRW
9LS3
9LSY
9LSZ
9LT3
9LTJ
9LTL
9LTM
9LTO
9LTR
9LTT
9LTW
9LTZ
9LU1
9LU4
9LU5
9LU6
9LU7
9LU8
9LUJ
9LUL
9LUT
9LUU
9LV4
9LV5
9LV6
9LV7
9LV9
9LVI
9LVS
9LW1
9LW3
9LW5
9LWC
9LWI
9LWJ
9LWK
9LWL
9LWN
9LXD
9LXE
9LXF
9LXP
9LY2
9LY3
9LY5
9LY6
9LYI
9LZ0
9LZ1
9LZ2
9LZH
9LZY
9LZZ
9M0J
9M0N
9M0P
9M0X
9M0Y
9M0Z
9M1P
9M1R
9M1S
9M1U
9M2O
9M2S
9M2X
9M3Q
9M3S
9M40
9M42
9M48
9M49
9M4C
9M4D
9M4E
9M5A
9M5C
9M5H
9M5K
9M5L
9M5T
9M5Z
9M6W
9M72
9M76
9M77
9M7B
9M7G
9M7H
9M7K
9M7Q
9M7R
9M7S
9M7U
9M7W
9M80
9M8D
9M8E
9M8F
9M8U
9M98
9M9D
9M9E
9M9F
9M9M
9M9P
9M9Q
9M9W
9M9X
9MA0
9MA5
9MA7
9MA8
9MAE
9MAG
9MAI
9MAK
9MAN
9MAO
9MAV
9MAZ
9MB0
9MB1
9MB2
9MB5
9MB6
9MB7
9MB8
9MBO
9MBP
9MBQ
9MBZ
9MC2
9MC4
9MC5
9MC6
9MC7
9MC9
9MCB
9MCG
9MCR
9MCS
9MCT
9MCU
9MCV
9MI8
9MIX
9MJA
9MJO
9MK6
9MKF
9MKG
9MKT
9MKU
9MKV
9MKW
9MKX
9MLC
9MM5
9MMY
9MMZ
9MN0
9MN1
9MN3
9MNB
9MNC
9MNM
9MOF
9MOG
9MOH
9MOY
9MPZ
9MQ0
9MQM
9MT7
9MU7
9MUH
9MUN
9MUR
9MVC
9MVD
9MVE
9MVF
9MVS
9MX4
9MX6
9MX7
9MXU
9MYS
9MZE
9MZF
9MZG
9MZI
9MZJ
9MZK
9MZL
9MZM
9MZN
9MZO
9MZP
9MZR
9MZS
9N00
9N01
9N04
9N05
9N09
9N0E
9N0L
9N1P
9N1Q
9N29
9N2I
9N37
9N38
9N3B
9N3I
9N3J
9N3V
9N3Y
9N3Z
9N5J
9N5N
9N6D
9N7F
9N7Q
9N7S
9N7T
9N7V
9N84
9N8F
9N8J
9N8Y
9N8Z
9N96
9N9E
9N9F
9N9G
9N9H
9N9I
9N9W
9NA1
9NA7
9NAH
9NAK
9NAL
9NAM
9NAP
9NAS
9NAU
9NAV
9NB1
9NB3
9NBC
9NBH
9NBT
9NBY
9NC0
9NC3
9NC6
9NC8
9NC9
9NCJ
9NCK
9NCN
9NCO
9NDD
9NDV
9NDW
9NDX
9NEF
9NEJ
9NEN
9NEP
9NEQ
9NET
9NEY
9NF7
9NF8
9NFA
9NFN
9NFT
9NFV
9NFW
9NFX
9NFY
9NFZ
9NG0
9NGK
9NGO
9NGZ
9NH5
9NH6
9NHQ
9NHX
9NHZ
9NI3
9NI4
9NI5
9NI6
9NI7
9NI8
9NID
9NIE
9NIF
9NJ2
9NJ3
9NJ6
9NJ9
9NJA
9NJS
9NLC
9NMS
9NMT
9NN1
9NNC
9NNL
9NNN
9NNZ
9NO8
9NO9
9NOC
9NON
9NOO
9NOP
9NOQ
9NOZ
9NP0
9NPM
9NPW
9NQ1
9NQ6
9NQL
9NQY
9NR1
9NR4
9NRD
9NRE
9NRF
9NRG
9NRH
9NSI
9NSP
9NT7
9NT8
9NT9
9NTT
9NTU
9NTV
9NTW
9NU4
9NU5
9NU8
9NUC
9NUF
9NUH
9NUI
9NUJ
9NUK
9NUL
9NUM
9NUN
9NUO
9NUP
9NUQ
9NUR
9NUS
9NUT
9NUU
9NUV
9NUW
9NUX
9NUY
9NVG
9NVN
9NVO
9NVP
9NVQ
9NVR
9NVS
9NVV
9NVW
9NVX
9NVY
9NVZ
9NW0
9NW1
9NWR
9NWS
9NWT
9NWV
9NX0
9NX1
9NX2
9NXZ
9NY0
9NYP
9NYR
9NYX
9NZ0
9NZ2
9NZO
9NZP
9NZQ
9NZR
9NZS
9NZT
9NZU
9O04
9O0G
9O0L
9O1I
9O1L
9O1M
9O1N
9O1O
9O1P
9O1Q
9O2X
9O2Y
9O36
9O3G
9O3O
9O3V
9O3W
9O3Y
9O3Z
9O49
9O4C
9O4E
9O4G
9O4H
9O4R
9O59
9O5A
9O5D
9O5E
9O5L
9O5Q
9O65
9O6A
9O6G
9O6V
9O6W
9O6X
9O6Y
9O6Z
9O76
9O79
9O7B
9O7C
9O7D
9O7E
9O7F
9O7G
9O7H
9O7I
9O7M
9O7O
9O7T
9O8D
9O8E
9O8H
9O8M
9O8P
9O8Q
9O8R
9O8S
9O8T
9O8Z
9O95
9O96
9O97
9O98
9O9J
9O9K
9O9L
9O9M
9OA3
9OA9
9OAA
9OAB
9OAC
9OAD
9OAE
9OAJ
9OB1
9OBK
9OBS
9OBT
9OBU
9OBV
9OBW
9OBX
9OBY
9OBZ
9OC0
9OC1
9OC2
9OD2
9ODE
9ODF
9ODG
9ODH
9ODI
9ODJ
9ODK
9ODL
9ODM
9ODN
9ODO
9ODP
9ODT
9ODU
9ODY
9ODZ
9OE0
9OE1
9OE2
9OE3
9OE4
9OE5
9OE6
9OE7
9OE8
9OE9
9OED
9OEF
9OEG
9OEH
9OEK
9OEM
9OEN
9OEO
9OFA
9OFI
9OFM
9OFN
9OFW
9OFY
9OFZ
9OGH
9OGY
9OH4
9OHC
9OHD
9OHE
9OHF
9OHK
9OHM
9OHN
9OI2
9OI3
9OI9
9OID
9OIF
9OIG
9OII
9OIP
9OIR
9OJ2
9OJJ
9OJK
9OJM
9OK7
9OK8
9OKB
9OKD
9OKE
9OKK
9OKL
9OKO
9OKP
9OKQ
9OKR
9OL4
9OLF
9OLK
9OLL
9OLM
9OLX
9OM0
9OM2
9OM3
9OM5
9OM7
9OME
9OMR
9OMY
9OMZ
9ON2
9ON3
9ON5
9ON6
9ONC
9ONJ
9ONK
9OOX
9OP9
9OPJ
9OPQ
9OPR
9OPU
9OQ7
9OQD
9OR5
9ORP
9OS2
9OS7
9OSB
9OSF
9OSG
9OSI
9OSS
9OST
9OSU
9OTA
9OTH
9OTK
9OUM
9OUN
9OUO
9OUP
9OUQ
9OUR
9OUS
9OUZ
9OV0
9OV1
9OV3
9OV4
9OV5
9OV6
9OV7
9OV8
9OV9
9OVB
9OVC
9OVD
9OVE
9OVK
9OVL
9OVM
9OVN
9OVO
9OVP
9OVT
9OVU
9OVV
9OVW
9OVY
9OW3
9OW4
9OW5
9OW6
9OWE
9OWJ
9OWK
9OWL
9OWM
9OWN
9OWO
9OWP
9OWQ
9OWS
9OWT
9OWU
9OWV
9OWW
9OWX
9OWY
9OX7
9OX8
9OXA
9OXB
9OXH
9OXI
9OXL
9OXM
9OY2
9OY3
9OY4
9OY8
9OYA
9OYB
9OYC
9OYD
9OYE
9OYF
9OYG
9OYH
9OYI
9OYJ
9OYK
9OYL
9OYM
9OYN
9OYV
9OYZ
9OZ1
9OZ6
9OZ8
9OZ9
9OZC
9OZH
9OZI
9OZN
9OZZ
9P0E
9P0J
9P0K
9P0L
9P0M
9P0N
9P0O
9P1U
9P1V
9P1X
9P24
9P35
9P3N
9P3O
9P3P
9P3Q
9P3R
9P3S
9P3T
9P3U
9P3V
9P3W
9P4F
9P4L
9P4M
9P4N
9P4O
9P4P
9P4Q
9P4R
9P52
9P53
9P54
9P6E
9P6G
9P6S
9P6U
9P6V
9P6X
9P7M
9P7P
9P7S
9P7Z
9P80
9P81
9P82
9P83
9P84
9P85
9P86
9P87
9P88
9P89
9P8A
9P8Y
9P90
9P91
9P9A
9P9B
9P9C
9P9D
9P9E
9P9F
9P9G
9P9L
9P9M
9P9Q
9P9R
9P9S
9P9T
9P9W
9PA1
9PA2
9PA3
9PA4
9PA5
9PA6
9PA8
9PAR
9PAW
9PB3
9PB5
9PB6
9PB7
9PB8
9PBD
9PBI
9PBS
9PBT
9PC7
9PC8
9PCK
9PCP
9PD3
9PD4
9PD5
9PD7
9PDP
9PDS
9PDT
9PDV
9PDW
9PE0
9PE1
9PE2
9PE3
9PE4
9PE5
9PEA
9PEF
9PEG
9PEH
9PEI
9PEJ
9PEK
9PEL
9PEW
9PEX
9PFJ
9PFK
9PFQ
9PFR
9PFS
9PFT
9PFU
9PFV
9PFZ
9PG8
9PGA
9PGC
9PGD
9PGF
9PGG
9PGH
9PGI
9PGL
9PGM
9PHC
9PHF
9PI7
9PI8
9PIE
9PIF
9PIG
9PIH
9PII
9PIJ
9PIK
9PIL
9PIM
9PIN
9PIT
9PIV
9PJ5
9PJ6
9PJ7
9PJ8
9PJ9
9PJA
9PJC
9PKB
9PKD
9PKO
9PKP
9PKW
9PKX
9PKY
9PKZ
9PL9
9PLQ
9PLR
9PM0
9PM6
9PMB
9PMJ
9PMO
9PMQ
9PMS
9PN5
9PN8
9PN9
9PNA
9PNB
9PNS
9PPQ
9PPW
9PPX
9PPY
9PPZ
9PQ0
9PQJ
9PQM
9PQO
9PQQ
9PQR
9PQT
9PR3
9PR4
9PR5
9PR6
9PR7
9PRA
9PRC
9PRD
9PRO
9PRQ
9PRT
9PRX
9PRY
9PRZ
9PS0
9PS5
9PS7
9PSI
9PSM
9PT4
9PT5
9PT8
9PT9
9PTA
9PTB
9PTC
9PTR
9PTS
9PTU
9PU5
9PUD
9PUM
9PUP
9PUR
9PUV
9PUW
9PV0
9PVA
9PVB
9PVC
9PVH
9PVM
9PVO
9PVP
9PVY
9PVZ
9PW3
9PW5
9PWA
9PWB
9PWC
9PWE
9PWF
9PWG
9PWH
9PWP
9PWQ
9PX6
9PX7
9PX8
9PX9
9PXF
9PXG
9PXI
9PXJ
9PXK
9PXL
9PXM
9PXN
9PXT
9PY5
9PYC
9PYD
9PYH
9PYK
9PZB
9PZC
9PZD
9PZM
9PZN
9PZO
9Q05
9Q06
9Q09
9Q0E
9Q0F
9Q0G
9Q0M
9Q0N
9Q0O
9Q0R
9Q0U
9Q0V
9Q0X
9Q0Y
9Q0Z
9Q10
9Q11
9Q12
9Q14
9Q15
9Q16
9Q17
9Q18
9Q19
9Q1E
9Q1Q
9Q1S
9Q1X
9Q1Z
9Q20
9Q23
9Q24
9Q25
9Q26
9Q28
9Q29
9Q2A
9Q2B
9Q2C
9Q2E
9Q2F
9Q2H
9Q2I
9Q2L
9Q2M
9Q2O
9Q2P
9Q2T
9Q33
9Q37
9Q3A
9Q3E
9Q3L
9Q3Q
9Q3R
9Q3S
9Q3U
9Q3V
9Q3X
9Q40
9Q42
9Q43
9Q44
9Q50
9Q57
9Q59
9Q5A
9Q5L
9Q5M
9Q5V
9Q5W
9Q60
9Q69
9Q6B
9Q6F
9Q6H
9Q6N
9Q6O
9Q6Y
9Q6Z
9Q71
9Q7A
9Q7B
9Q7C
9Q7J
9Q7K
9Q7L
9Q7M
9Q7N
9Q7O
9Q7Q
9Q7U
9Q7Y
9Q80
9Q85
9Q8V
9Q8X
9Q9F
9QAW
9QC6
9QCR
9QCS
9QDL
9QDM
9QEB
9QEF
9QEJ
9QER
9QET
9QF0
9QFM
9QFP
9QFX
9QG7
9QGK
9QGQ
9QGU
9QGY
9QH1
9QH4
9QH6
9QH7
9QH8
9QH9
9QHD
9QHE
9QHF
9QHG
9QHJ
9QHK
9QHM
9QHN
9QHO
9QHP
9QHQ
9QHS
9QHV
9QHW
9QHX
9QID
9QIK
9QIN
9QJC
9QJJ
9QJN
9QJP
9QJQ
9QJR
9QJS
9QJV
9QJX
9QJY
9QK0
9QK1
9QKT
9QKZ
9QL1
9QLH
9QLI
9QLO
9QLP
9QLQ
9QLV
9QMS
9QNO
9QO0
9QO1
9QO2
9QO3
9QO4
9QO5
9QO6
9QO8
9QO9
9QOH
9QON
9QOO
9QOP
9QP4
9QPE
9QPO
9QPP
9QQE
9QQF
9QQL
9QQP
9QQQ
9QQS
9QQW
9QR2
9QR6
9QRU
9QS6
9QSA
9QSH
9QSJ
9QSX
9QSY
9QSZ
9QT4
9QTJ
9QTN
9QTO
9QTY
9QU6
9QU7
9QU9
9QVX
9QW9
9QWQ
9QWT
9QWW
9QXL
9QY3
9QZO
9QZP
9R01
9R03
9R07
9R0A
9R0I
9R0M
9R0N
9R0O
9R1D
9R1H
9R1Q
9R1R
9R1T
9R1U
9R1V
9R2G
9R2I
9R2N
9R3A
9R3D
9R3S
9R41
9R42
9R4I
9R4J
9R4X
9R4Y
9R5K
9R5S
9R5W
9R60
9R61
9R62
9R63
9R64
9R65
9R66
9R67
9R68
9R69
9R6A
9R6B
9R6C
9R6S
9R6T
9R7L
9R7M
9R7N
9R7Q
9R7R
9R7S
9R7T
9R7U
9R7V
9R7W
9R7X
9R7Y
9R7Z
9R80
9R81
9R82
9R85
9R88
9R89
9R8A
9R8D
9R8G
9R8J
9R8K
9R8L
9R8M
9R8N
9R8P
9R8T
9R8V
9R8W
9R90
9R91
9R94
9R9A
9R9H
9R9I
9R9N
9R9W
9R9X
9R9Y
9R9Z
9RA0
9RA1
9RA2
9RAD
9RAE
9RAF
9RAG
9RAH
9RB3
9RB6
9RB7
9RB8
9RB9
9RBA
9RBB
9RBF
9RBO
9RBQ
9RBR
9RBU
9RC1
9RCB
9RCD
9RCE
9RCH
9RD0
9RD7
9RD8
9RD9
9RDK
9RF3
9RFV
9RFW
9RG7
9RGD
9RGE
9RGM
9RGN
9RGO
9RGP
9RH1
9RH2
9RH3
9RH4
9RH5
9RH6
9RH7
9RH8
9RH9
9RHA
9RHB
9RHC
9RHD
9RHE
9RHF
9RHI
9RHJ
9RHL
9RHM
9RHU
9RIG
9RIH
9RII
9RIJ
9RIK
9RIL
9RIM
9RIN
9RIO
9RIP
9RIQ
9RIR
9RIV
9RIW
9RJA
9RJB
9RJC
9RKD
9RKW
9RKX
9RKZ
9RL3
9RL4
9RL5
9RLA
9RLC
9RLD
9RLE
9RLF
9RLG
9RLH
9RLI
9RLJ
9RLK
9RLT
9RLZ
9RM0
9RM1
9RM5
9RM9
9RMA
9RMC
9RMI
9RMK
9RN1
9RN2
9RNJ
9RO9
9ROA
9ROD
9ROE
9ROF
9ROG
9ROH
9ROI
9ROJ
9ROK
9ROL
9ROM
9RON
9ROO
9ROP
9ROQ
9ROR
9ROT
9ROV
9ROW
9ROX
9ROZ
9RPB
9RPF
9RQJ
9RQL
9RQM
9RQN
9RQT
9RRF
9RRM
9RRN
9RRO
9RRQ
9RRS
9RRU
9RSC
9RSD
9RSG
9RSH
9RSM
9RSQ
9RSS
9RST
9RSU
9RT2
9RT4
9RTW
9RTX
9RTY
9RTZ
9RU6
9RU7
9RU8
9RU9
9RUA
9RUC
9RUF
9RUG
9RUH
9RUX
9RV1
9RVC
9RVM
9RVN
9RVU
9RVV
9RVY
9RVZ
9RW0
9RW1
9RW2
9RW3
9RW4
9RW5
9RW6
9RW7
9RW8
9RW9
9RWA
9RWB
9RWG
9RWS
9RWT
9RWZ
9RX1
9RX4
9RX5
9RX6
9RX7
9RX8
9RX9
9RXA
9RXB
9RY4
9RY7
9RYB
9RYE
9RYF
9RYI
9RYJ
9RYK
9RYM
9RYN
9RYO
9RYP
9RYQ
9RYU
9RYV
9RYW
9RYX
9RZE
9RZM
9RZU
9RZV
9S05
9S0N
9S0R
9S0S
9S0Y
9S0Z
9S10
9S11
9S15
9S16
9S1E
9S1F
9S1H
9S1L
9S1M
9S28
9S29
9S2C
9S2E
9S2F
9S2G
9S2H
9S2R
9S2S
9S34
9S3B
9S3C
9S3D
9S3E
9S3F
9S3M
9S3Q
9S3R
9S3T
9S3U
9S3V
9S47
9S4L
9S4N
9S4Q
9S4S
9S4T
9S4U
9S53
9S54
9S55
9S5A
9S5C
9S5E
9S5F
9S5J
9S5N
9S5T
9S5U
9S5W
9S5Y
9S5Z
9S63
9S67
9S6E
9S6F
9S6P
9S6Q
9S6R
9S6S
9S6T
9S6U
9S6Y
9S72
9S7B
9S7C
9S7D
9S7E
9S7G
9S7J
9S7O
9S7P
9S7Q
9S7S
9S7V
9S86
9S8B
9S8C
9S8D
9S8E
9S8G
9S8K
9S8L
9S97
9S98
9S99
9S9B
9S9H
9SA8
9SAS
9SAU
9SAY
9SAZ
9SB0
9SB1
9SB2
9SB3
9SB4
9SB5
9SBL
9SBM
9SBN
9SBO
9SBP
9SBQ
9SBR
9SBS
9SBT
9SBU
9SBV
9SBW
9SBX
9SBY
9SBZ
9SC0
9SC1
9SC2
9SC3
9SC4
9SCU
9SD9
9SDA
9SDB
9SDG
9SDH
9SDL
9SDO
9SDP
9SDQ
9SDT
9SE4
9SE6
9SE7
9SEC
9SED
9SEF
9SEH
9SEJ
9SF4
9SF5
9SF8
9SF9
9SFD
9SFF
9SFH
9SFI
9SFJ
9SFN
9SFP
9SFV
9SFW
9SGK
9SGL
9SGM
9SGN
9SGO
9SGP
9SGQ
9SGR
9SGS
9SGT
9SGY
9SH1
9SH2
9SH6
9SHE
9SHM
9SHN
9SHO
9SHP
9SHQ
9SHV
9SHW
9SHX
9SHY
9SHZ
9SI0
9SI8
9SIE
9SII
9SIO
9SIQ
9SIT
9SIU
9SJ2
9SJ7
9SJ9
9SJB
9SJC
9SJD
9SJE
9SJL
9SJM
9SJN
9SJO
9SJR
9SJS
9SJT
9SJU
9SJV
9SK2
9SK3
9SK4
9SK5
9SK6
9SK7
9SK8
9SK9
9SKA
9SKB
9SKC
9SKD
9SKE
9SKF
9SKI
9SKJ
9SKK
9SKL
9SKR
9SKS
9SKT
9SKU
9SKV
9SKW
9SKX
9SKY
9SKZ
9SL1
9SL3
9SL4
9SL5
9SL6
9SL7
9SLC
9SLD
9SLF
9SLG
9SLL
9SLM
9SLN
9SLP
9SM2
9SM4
9SM8
9SMD
9SMF
9SMG
9SMH
9SMI
9SMJ
9SMK
9SML
9SMN
9SMP
9SMR
9SMS
9SMX
9SN2
9SNA
9SNK
9SO9
9SOW
9SOX
9SP8
9SPE
9SPF
9SPG
9SPI
9SPJ
9SPU
9SPV
9SPW
9SQ0
9SQ4
9SQ5
9SQ6
9SQH
9SQM
9SQN
9SQO
9SQP
9SQQ
9SQR
9SQV
9SQW
9SQY
9SR9
9SRA
9SRB
9SRC
9SRD
9SRE
9SRF
9SRI
9SRO
9SRQ
9SRR
9SRS
9SRT
9SRU
9SRV
9SRW
9SRX
9SRZ
9SS0
9SS1
9SS2
9SS4
9SS5
9SS6
9SSC
9SSL
9ST9
9STA
9SUL
9SUM
9SV6
9SVK
9SW2
9SW3
9SW4
9SW5
9SW6
9SWA
9SWK
9SWM
9SWN
9SWO
9SWQ
9SWR
9SWS
9SX2
9SX6
9SY5
9SYH
9SYJ
9SYS
9SYT
9SYV
9SZ7
9SZ9
9SZA
9SZB
9SZC
9SZD
9SZE
9SZW
9T0C
9T0E
9T0F
9T0Y
9T0Z
9T17
9T19
9T1A
9T1B
9T1C
9T1D
9T1E
9T1F
9T1G
9T1H
9T1I
9T1J
9T1S
9T2J
9T2K
9T2L
9T2M
9T2N
9T2O
9T2P
9T2Q
9T2R
9T2S
9T33
9T34
9T35
9T3R
9T3S
9T3X
9T3Y
9T4R
9T4U
9T4W
9T4X
9T53
9T56
9T5C
9T5H
9T5J
9T5K
9T5L
9T5M
9T5N
9T5O
9T5P
9T5T
9T5U
9T5V
9T5W
9T5X
9T5Z
9T61
9T63
9T64
9T65
9T66
9T67
9T68
9T69
9T6C
9T6D
9T6H
9T6I
9T6L
9T6M
9T6N
9T6Z
9T74
9T7H
9T7J
9T7L
9T7P
9T7Q
9T7T
9T7V
9T88
9T8C
9T8D
9T8H
9T8I
9T8K
9T8L
9T8M
9T8N
9T8O
9T8P
9T8Q
9T8R
9T8S
9T8T
9T92
9T93
9T94
9T97
9T9M
9T9P
9T9R
9T9U
9T9V
9TA3
9TAJ
9TAK
9TAL
9TAM
9TAN
9TAO
9TAW
9TAX
9TAY
9TBH
9TBI
9TBJ
9TBY
9TCC
9TCM
9TCN
9TD2
9TDM
9TDS
9TDT
9TEH
9TEN
9TEO
9TEP
9TEQ
9TER
9TEW
9TEX
9TEY
9TEZ
9TF0
9TF1
9TF2
9TFV
9TG3
9TG4
9TG6
9TG8
9TG9
9TGA
9TGC
9TGD
9TGM
9TGN
9TGO
9TGW
9TH8
9THI
9THK
9THM
9THN
9THP
9TI4
9TI5
9TI6
9TI7
9TI8
9TI9
9TIC
9TID
9TIE
9TIF
9TIG
9TIH
9TII
9TIJ
9TIK
9TIL
9TIM
9TIN
9TIO
9TIP
9TIR
9TIS
9TIT
9TIU
9TIV
9TIW
9TK9
9TKA
9TKE
9TKF
9TKG
9TKH
9TKI
9TKJ
9TKK
9TKL
9TKM
9TKN
9TKO
9TKP
9TKQ
9TKR
9TKZ
9TL2
9TL6
9TL7
9TL8
9TL9
9TLA
9TLB
9TLC
9TLE
9TLG
9TLH
9TLK
9TMI
9TMN
9TMS
9TNB
9TNL
9TNS
9TNZ
9TO0
9TOI
9TOZ
9TPA
9TPJ
9TPK
9TPL
9TPQ
9TPS
9TPT
9TPW
9TQ4
9TQ7
9TQ8
9TQA
9TQB
9TQC
9TQD
9TQG
9TQJ
9TQK
9TRA
9TRC
9TRD
9TRE
9TRL
9TRM
9TT7
9TU7
9TU8
9TU9
9TUA
9TUB
9TUC
9TUD
9TUZ
9TV0
9TV2
9TVU
9TW2
9TW3
9TX4
9TX5
9TX6
9TX7
9TX8
9TX9
9TXA
9TXB
9TXI
9TXJ
9TXK
9TXL
9TYG
9TYH
9TYI
9TYV
9TYY
9TZ3
9TZ5
9TZ9
9TZE
9TZW
9TZX
9TZY
9U02
9U04
9U05
9U06
9U2S
9U2T
9U32
9U34
9U35
9U36
9U37
9U38
9U3A
9U3K
9U3L
9U40
9U43
9U4A
9U4B
9U4C
9U4D
9U4J
9U4M
9U4O
9U4R
9U4Z
9U56
9U57
9U58
9U5E
9U5F
9U5P
9U5Q
9U5R
9U5U
9U5Z
9U60
9U61
9U62
9U65
9U66
9U67
9U68
9U69
9U6R
9U6S
9U7H
9U7I
9U7J
9U7L
9U7R
9U7T
9U7Y
9U8B
9U8K
9U9H
9U9I
9U9J
9U9R
9U9S
9U9Z
9UA0
9UA1
9UA3
9UA5
9UAM
9UAN
9UAQ
9UAR
9UAS
9UAT
9UAX
9UB1
9UB7
9UB8
9UBC
9UBL
9UBM
9UBO
9UBR
9UBU
9UCL
9UCO
9UDC
9UDK
9UE3
9UED
9UEJ
9UEK
9UEP
9UER
9UF6
9UFB
9UFD
9UFE
9UFF
9UFL
9UFN
9UFS
9UFT
9UFV
9UFW
9UFX
9UFY
9UFZ
9UG0
9UG1
9UG2
9UGG
9UGJ
9UGL
9UGO
9UGU
9UGX
9UGY
9UH3
9UH4
9UH7
9UH9
9UHA
9UHF
9UHG
9UHH
9UHQ
9UI1
9UI6
9UIF
9UIN
9UIQ
9UIT
9UJ4
9UJE
9UJF
9UJM
9UJQ
9UJS
9UJT
9UJV
9UJW
9UJX
9UK2
9UK9
9UKA
9UKB
9UKC
9UKD
9UKE
9UKF
9UKG
9UKH
9UKI
9UKJ
9UKK
9UKL
9UKZ
9ULO
9ULZ
9UMG
9UMT
9UMU
9UN2
9UN3
9UNJ
9UNK
9UNM
9UNN
9UNO
9UNP
9UNQ
9UNR
9UNU
9UNW
9UOE
9UOF
9UOV
9UP2
9UPB
9UPC
9UPE
9UPM
9UPN
9UPO
9UPR
9UPU
9UPV
9UPY
9UQ0
9URI
9URJ
9URM
9URZ
9US8
9USU
9USV
9USW
9USX
9USY
9USZ
9UT0
9UT1
9UT3
9UT4
9UT5
9UT6
9UT7
9UTG
9UUH
9UUM
9UUP
9UUQ
9UUR
9UUX
9UUY
9UV2
9UV3
9UV4
9UV7
9UV9
9UVA
9UVB
9UVC
9UVT
9UVU
9UVV
9UVW
9UVX
9UVY
9UVZ
9UW3
9UW4
9UWG
9UWI
9UWJ
9UWL
9UWM
9UWQ
9UWV
9UWW
9UWX
9UWY
9UX1
9UX2
9UX3
9UX4
9UX7
9UX8
9UX9
9UXA
9UXB
9UXC
9UXH
9UXI
9UXK
9UXL
9UXN
9UXO
9UXP
9UXQ
9UXS
9UXT
9UXU
9UXW
9UXX
9UY9
9UYD
9UYO
9UYP
9UYQ
9UZ7
9UZ9
9UZK
9UZL
9UZM
9V09
9V0A
9V0B
9V0C
9V0E
9V0F
9V0X
9V10
9V1H
9V2S
9V2U
9V2V
9V2W
9V31
9V32
9V33
9V35
9V36
9V38
9V3C
9V3E
9V3F
9V3S
9V3T
9V3V
9V3X
9V3Y
9V3Z
9V42
9V43
9V4D
9V5P
9V65
9V66
9V6L
9V6Q
9V6S
9V6T
9V6U
9V6X
9V75
9V7C
9V7N
9V7O
9V7P
9V7Q
9V7S
9V7T
9V7U
9V7X
9V7Z
9V81
9V82
9V8A
9V8L
9V8W
9V8X
9V8Y
9V93
9V95
9V96
9V9C
9V9E
9V9G
9V9H
9V9I
9V9P
9V9Q
9V9R
9V9S
9V9T
9V9U
9V9V
9V9W
9V9X
9V9Y
9V9Z
9VA1
9VA2
9VAM
9VAP
9VAT
9VAU
9VAV
9VAW
9VAX
9VAY
9VAZ
9VB0
9VB1
9VB2
9VB3
9VB4
9VB5
9VB6
9VB7
9VBA
9VBD
9VBL
9VBM
9VBN
9VBO
9VBP
9VBT
9VBV
9VBX
9VBY
9VBZ
9VC0
9VC6
9VC8
9VC9
9VCB
9VCC
9VCF
9VCP
9VCS
9VCT
9VCU
9VCV
9VCW
9VD0
9VD1
9VE1
9VE9
9VED
9VEE
9VEF
9VEG
9VEJ
9VEK
9VF4
9VF5
9VFI
9VFV
9VFX
9VGD
9VGF
9VGU
9VH2
9VH3
9VH4
9VH5
9VH9
9VHD
9VHL
9VI0
9VI1
9VI4
9VIN
9VJ3
9VJ4
9VJ8
9VJ9
9VJA
9VJB
9VJD
9VJH
9VJI
9VJN
9VJO
9VJP
9VJQ
9VJR
9VJS
9VJT
9VKO
9VKP
9VKQ
9VKR
9VKS
9VKY
9VKZ
9VL3
9VL5
9VLG
9VLN
9VLQ
9VLS
9VLT
9VLW
9VLZ
9VM2
9VM3
9VM4
9VM5
9VM6
9VM7
9VM8
9VMD
9VMG
9VMH
9VMN
9VMO
9VMP
9VN9
9VNA
9VNB
9VNC
9VND
9VNE
9VNY
9VNZ
9VO0
9VO1
9VO2
9VOS
9VPB
9VPC
9VPD
9VPG
9VPM
9VPN
9VQO
9VQP
9VQQ
9VQR
9VQS
9VQT
9VQU
9VQV
9VQW
9VQX
9VQY
9VRB
9VRC
9VRD
9VRE
9VS3
9VS4
9VS5
9VSU
9VSV
9VSW
9VSZ
9VT1
9VT2
9VT4
9VTG
9VTH
9VTI
9VTM
9VTN
9VTO
9VTP
9VTQ
9VTR
9VTS
9VTW
9VTX
9VTY
9VTZ
9VU0
9VU1
9VU9
9VUA
9VUB
9VUC
9VUE
9VUF
9VUG
9VUH
9VUI
9VUJ
9VUK
9VUL
9VUM
9VUQ
9VUS
9VV0
9VV6
9VVG
9VVI
9VVJ
9VVK
9VVM
9VVO
9VW8
9VWA
9VWC
9VWE
9VWF
9VWG
9VWI
9VWJ
9VWM
9VWR
9VWS
9VWZ
9VX1
9VX6
9VXE
9VXM
9VXN
9VXP
9VXT
9VXU
9VXV
9VXX
9VXY
9VXZ
9VY0
9VY1
9VZ2
9VZ3
9VZ8
9VZP
9VZX
9W01
9W02
9W03
9W04
9W06
9W07
9W08
9W09
9W0B
9W0D
9W0E
9W0G
9W0I
9W0J
9W0K
9W0M
9W0N
9W0O
9W0P
9W0Q
9W0R
9W0S
9W0T
9W0U
9W0W
9W0X
9W0Z
9W14
9W1B
9W1E
9W1F
9W1G
9W1H
9W1I
9W1J
9W1T
9W1X
9W1Y
9W1Z
9W20
9W21
9W22
9W23
9W24
9W25
9W26
9W2B
9W2D
9W2F
9W2G
9W2H
9W2I
9W2J
9W2M
9W2R
9W2S
9W2T
9W2U
9W2V
9W2W
9W2X
9W2Y
9W2Z
9W32
9W33
9W34
9W35
9W36
9W37
9W39
9W3D
9W3E
9W3F
9W3G
9W3K
9W3L
9W3P
9W3T
9W3U
9W3V
9W3W
9W3Z
9W40
9W41
9W43
9W45
9W4A
9W4C
9W4D
9W4E
9W4I
9W4J
9W4K
9W4L
9W4O
9W4Q
9W4Y
9W50
9W51
9W5B
9W5I
9W62
9W63
9W64
9W65
9W66
9W6E
9W6F
9W6G
9W6H
9W6I
9W6J
9W6K
9W6L
9W6M
9W6N
9W6O
9W6P
9W6Q
9W6T
9W6U
9W71
9W72
9W73
9W7D
9W7J
9W7K
9W7X
9W7Y
9W81
9W89
9W8A
9W8B
9W8C
9W8D
9W8E
9W8F
9W8J
9W8Q
9W8R
9W8S
9W8T
9W8U
9W8V
9W8W
9W8X
9W8Y
9W8Z
9W90
9W97
9W98
9W99
9W9A
9W9B
9W9C
9W9E
9W9F
9W9G
9W9J
9W9K
9W9L
9W9N
9W9O
9W9P
9W9Q
9W9R
9W9S
9W9T
9W9U
9W9V
9W9W
9W9X
9W9Y
9W9Z
9WA0
9WA1
9WA2
9WAC
9WAD
9WAE
9WAF
9WAG
9WAH
9WAI
9WAJ
9WAK
9WAO
9WAP
9WB3
9WB4
9WBG
9WBJ
9WBK
9WBL
9WBR
9WBV
9WBW
9WBZ
9WC0
9WC1
9WC4
9WC6
9WCA
9WCB
9WCW
9WCX
9WCY
9WD0
9WD1
9WD4
9WD5
9WD7
9WD9
9WDA
9WDB
9WDC
9WDI
9WDO
9WDP
9WDQ
9WDR
9WDS
9WDT
9WDU
9WDV
9WDW
9WDX
9WDY
9WE7
9WET
9WEU
9WEY
9WEZ
9WF0
9WF2
9WF3
9WF9
9WFA
9WFC
9WFF
9WFG
9WFL
9WFN
9WFO
9WFR
9WFU
9WFZ
9WG0
9WG2
9WG4
9WG6
9WG7
9WG8
9WGB
9WH1
9WHK
9WHU
9WHX
9WHY
9WID
9WIE
9WIF
9WIG
9WIJ
9WIK
9WIT
9WIX
9WIZ
9WJ0
9WJS
9WJY
9WK1
9WKR
9WKZ
9WL5
9WL6
9WL7
9WL8
9WL9
9WLA
9WLB
9WLR
9WLS
9WMD
9WME
9WMF
9WMH
9WMI
9WMJ
9WML
9WMM
9WMN
9WMO
9WMP
9WMQ
9WMR
9WN8
9WNB
9WND
9WNF
9WNG
9WNS
9WNU
9WOP
9WP6
9WPE
9WPF
9WPG
9WPH
9WPL
9WPM
9WPQ
9WPX
9WPY
9WPZ
9WQ0
9WQ2
9WQ3
9WQ5
9WQ6
9WQ7
9WQ8
9WQ9
9WQA
9WQB
9WQK
9WQL
9WQM
9WQN
9WQO
9WQQ
9WQR
9WQV
9WRE
9WRK
9WRL
9WRM
9WS4
9WS8
9WSD
9WSM
9WSN
9WSO
9WSP
9WSQ
9WSR
9WSZ
9WT0
9WT1
9WT2
9WTD
9WTO
9WTW
9WTX
9WTY
9WTZ
9WUC
9WUD
9WUE
9WUF
9WUG
9WUK
9WUM
9WUP
9WUS
9WUT
9WUU
9WUV
9WUW
9WUY
9WV1
9WV2
9WV3
9WV5
9WV6
9WV7
9WV9
9WVC
9WVX
9WW4
9WW5
9WW6
9WW7
9WW9
9WWA
9WWB
9WWC
9WWG
9WWI
9WWL
9WWS
9WWU
9WWV
9WWW
9WX4
9WX6
9WX7
9WX9
9WXB
9WXD
9WXF
9WXH
9WXI
9WXJ
9WXP
9WXR
9WXV
9WY1
9WY8
9WYK
9WYL
9WYM
9WYP
9WYT
9WYV
9WYX
9WZ3
9WZE
9WZR
9WZS
9WZT
9WZU
9WZV
9WZX
9X04
9X05
9X0B
9X0C
9X0F
9X0J
9X0K
9X0L
9X0Q
9X0R
9X0W
9X0X
9X0Y
9X1A
9X1B
9X1C
9X1D
9X1L
9X1M
9X1N
9X1O
9X1P
9X1V
9X20
9X21
9X22
9X23
9X28
9X29
9X2A
9X2B
9X2C
9X2F
9X2M
9X2N
9X35
9X3J
9X3K
9X3O
9X3P
9X3Q
9X3R
9X3S
9X3Y
9X3Z
9X40
9X41
9X46
9X47
9X48
9X49
9X4A
9X4B
9X4C
9X4D
9X4E
9X4G
9X4H
9X4I
9X57
9X58
9X5F
9X5G
9X5J
9X5R
9X5W
9X5X
9X5Y
9X5Z
9X60
9X61
9X65
9X67
9X69
9X6B
9X6D
9X6S
9X6Z
9X74
9X8X
9X94
9X95
9X97
9X9B
9X9J
9X9K
9X9T
9XA3
9XA7
9XA8
9XA9
9XAA
9XAB
9XAC
9XAD
9XAE
9XAF
9XAG
9XAH
9XAI
9XAJ
9XAK
9XB1
9XB9
9XBS
9XBY
9XC6
9XCA
9XCB
9XCF
9XCG
9XCH
9XCM
9XD0
9XD4
9XD5
9XD6
9XDA
9XDB
9XDC
9XDD
9XDE
9XDQ
9XDR
9XE6
9XE7
9XE9
9XEC
9XED
9XF4
9XFG
9XFH
9XFI
9XFJ
9XFK
9XFL
9XFO
9XFV
9XGB
9XGC
9XGK
9XGV
9XGW
9XGX
9XH2
9XHH
9XHI
9XHY
9XHZ
9XI0
9XI1
9XI2
9XI3
9XI4
9XI5
9XI6
9XI7
9XI8
9XI9
9XIP
9XIQ
9XIR
9XIX
9XIY
9XJ1
9XJ5
9XJ7
9XJ8
9XJ9
9XJK
9XJL
9XJM
9XJP
9XJW
9XK3
9XK4
9XK6
9XK7
9XK8
9XK9
9XKA
9XKL
9XKO
9XMG
9XMI
9XMM
9XN2
9XN3
9XN4
9XNC
9XNH
9XNO
9XNP
9XNX
9XNY
9XNZ
9XO0
9XO1
9XO2
9XO3
9XO4
9XO5
9XOU
9XPE
9XPF
9XPG
9XPH
9XPS
9XQ9
9XQB
9XQC
9XQD
9XQF
9XQH
9XQM
9XQN
9XQS
9XQU
9XR3
9XR4
9XR5
9XRL
9XRO
9XSF
9XSG
9XSM
9XTB
9XTC
9XTD
9XTE
9XTI
9XTJ
9XTT
9XTU
9XU3
9XUK
9XUL
9XUM
9XUO
9XUR
9XUS
9XVC
9XVE
9XVG
9XVI
9XVP
9XVR
9XVV
9XW5
9XW7
9XW8
9XWA
9XWG
9XWM
9XWN
9XWO
9XXT
9XY4
9XYC
9XYE
9XYU
9XZG
9XZJ
9XZK
9XZL
9XZM
9XZN
9XZQ
9XZR
9XZS
9XZX
9XZY
9Y03
9Y04
9Y05
9Y06
9Y07
9Y08
9Y09
9Y0B
9Y0C
9Y0H
9Y1M
9Y1N
9Y1Q
9Y1V
9Y2A
9Y2B
9Y2C
9Y2F
9Y3D
9Y3E
9Y3F
9Y3G
9Y3H
9Y3P
9Y3Q
9Y3T
9Y3U
9Y3V
9Y42
9Y44
9Y49
9Y4A
9Y4B
9Y4C
9Y4D
9Y4E
9Y4F
9Y4G
9Y4H
9Y4J
9Y4L
9Y4P
9Y4T
9Y4W
9Y52
9Y54
9Y5I
9Y5J
9Y5K
9Y5L
9Y5M
9Y5N
9Y5O
9Y5R
9Y5S
9Y5T
9Y5U
9Y61
9Y63
9Y66
9Y67
9Y68
9Y69
9Y6A
9Y6F
9Y6G
9Y6H
9Y6N
9Y6O
9Y6P
9Y6T
9Y6V
9Y70
9Y71
9Y72
9Y73
9Y75
9Y77
9Y78
9Y79
9Y7A
9Y7B
9Y7C
9Y7H
9Y7I
9Y7J
9Y7L
9Y7M
9Y7N
9Y7O
9Y7P
9Y7V
9Y7W
9Y7X
9Y7Z
9Y80
9Y8Z
9Y90
9Y93
9Y94
9Y95
9Y96
9Y9D
9Y9F
9Y9G
9Y9J
9Y9L
9Y9M
9Y9P
9Y9V
9Y9X
9YA5
9YA7
9YA8
9YA9
9YAB
9YAE
9YAI
9YAO
9YAP
9YAQ
9YAW
9YAX
9YAZ
9YB0
9YB1
9YB2
9YB3
9YB4
9YBL
9YBM
9YBN
9YBP
9YBY
9YC5
9YC6
9YC8
9YCD
9YCF
9YCH
9YCI
9YCJ
9YCX
9YCY
9YCZ
9YD0
9YD5
9YDB
9YDC
9YDD
9YDE
9YDO
9YDS
9YDT
9YDW
9YDX
9YE0
9YE6
9YE7
9YE8
9YEC
9YED
9YEE
9YEH
9YEI
9YEJ
9YEK
9YEL
9YEM
9YEO
9YER
9YET
9YEU
9YEX
9YF0
9YF4
9YF9
9YFD
9YFG
9YFT
9YFU
9YFV
9YFW
9YG4
9YG5
9YG8
9YG9
9YGA
9YGB
9YGC
9YGD
9YGF
9YGG
9YGH
9YGI
9YGJ
9YGL
9YGM
9YGN
9YGO
9YGP
9YGQ
9YGR
9YGT
9YGU
9YGV
9YGZ
9YH0
9YH1
9YH6
9YH7
9YHA
9YHB
9YHO
9YHQ
9YHR
9YHS
9YHT
9YI2
9YI4
9YIB
9YID
9YIE
9YIF
9YIG
9YIH
9YII
9YIJ
9YIK
9YIL
9YIM
9YIW
9YIX
9YJ0
9YJ1
9YJ2
9YJ3
9YJP
9YJV
9YJW
9YJZ
9YK0
9YK9
9YKK
9YKN
9YKQ
9YKV
9YKW
9YKX
9YKY
9YL3
9YLB
9YLE
9YLF
9YLI
9YLU
9YLV
9YLX
9YLY
9YM8
9YM9
9YMA
9YMB
9YMC
9YMD
9YMF
9YMJ
9YMK
9YML
9YMU
9YMV
9YMW
9YMX
9YMY
9YMZ
9YN0
9YNB
9YNC
9YND
9YNE
9YNF
9YNG
9YNH
9YNK
9YNO
9YNP
9YNQ
9YNR
9YNX
9YOK
9YOP
9YOR
9YOS
9YOT
9YOU
9YOV
9YP4
9YP6
9YP8
9YP9
9YPB
9YPE
9YPF
9YPR
9YQ6
9YQ8
9YQK
9YQL
9YQO
9YQP
9YQQ
9YR4
9YR7
9YRB
9YRC
9YRG
9YRH
9YRK
9YRL
9YRM
9YRN
9YRO
9YRP
9YRQ
9YS3
9YSG
9YSJ
9YSK
9YSR
9YSS
9YT4
9YT6
9YT7
9YTA
9YTC
9YTQ
9YTS
9YUQ
9YUY
9YUZ
9YVG
9YVH
9YVI
9YVK
9YVL
9YVM
9YVN
9YVO
9YVP
9YW0
9YW2
9YWQ
9YWS
9YWU
9YWV
9YX6
9YX7
9YX8
9YX9
9YXA
9YXB
9YXM
9YXN
9YXQ
9YXV
9YXX
9YYD
9YYG
9YYH
9YYU
9YYV
9YZ1
9YZ4
9Z03
9Z04
9Z05
9Z06
9Z07
9Z0E
9Z0H
9Z0P
9Z12
9Z18
9Z1M
9Z1N
9Z1W
9Z1X
9Z1Y
9Z1Z
9Z20
9Z21
9Z22
9Z23
9Z24
9Z25
9Z26
9Z27
9Z2D
9Z2F
9Z2M
9Z2N
9Z2O
9Z2P
9Z2Q
9Z2R
9Z2T
9Z2U
9Z2V
9Z2W
9Z32
9Z3D
9Z3J
9Z3K
9Z3W
9Z3Y
9Z3Z
9Z40
9Z49
9Z4A
9Z4H
9Z4I
9Z4J
9Z4O
9Z4Q
9Z4W
9Z5I
9Z5N
9Z5U
9Z60
9Z61
9Z62
9Z63
9Z64
9Z66
9Z67
9Z6I
9Z6N
9Z6T
9Z6U
9Z6V
9Z6W
9Z6Y
9Z6Z
9Z76
9Z77
9Z78
9Z79
9Z7A
9Z7B
9Z7P
9Z7Q
9Z7R
9Z7S
9Z7T
9Z7U
9Z7V
9Z7W
9Z7X
9Z80
9Z81
9Z82
9Z8F
9Z8L
9Z8M
9Z91
9Z9B
9Z9G
9Z9H
9Z9K
9Z9L
9Z9S
9Z9W
9Z9X
9Z9Y
9ZA1
9ZA2
9ZA3
9ZA4
9ZAK
9ZAW
9ZB4
9ZB5
9ZB9
9ZBA
9ZBB
9ZBC
9ZBD
9ZBI
9ZBJ
9ZBK
9ZBL
9ZBP
9ZBS
9ZBT
9ZBU
9ZBV
9ZC4
9ZC5
9ZCD
9ZCN
9ZCO
9ZCP
9ZCQ
9ZCR
9ZCU
9ZCV
9ZCW
9ZCY
9ZCZ
9ZD0
9ZD1
9ZD2
9ZD3
9ZD4
9ZD5
9ZD7
9ZD8
9ZDK
9ZDQ
9ZDV
9ZDW
9ZDX
9ZDY
9ZE0
9ZE2
9ZE3
9ZE4
9ZEB
9ZEC
9ZED
9ZEN
9ZEO
9ZEZ
9ZF0
9ZF2
9ZF3
9ZF5
9ZF9
9ZFC
9ZFD
9ZFO
9ZFY
9ZFZ
9ZGF
9ZGM
9ZGN
9ZGO
9ZGP
9ZGQ
9ZGR
9ZGS
9ZGT
9ZGU
9ZGV
9ZGW
9ZGX
9ZGY
9ZGZ
9ZH0
9ZH1
9ZH2
9ZH5
9ZH7
9ZH9
9ZHA
9ZHB
9ZHC
9ZHD
9ZHE
9ZHF
9ZHG
9ZHH
9ZHI
9ZHJ
9ZHK
9ZHL
9ZHO
9ZHP
9ZHQ
9ZHR
9ZHS
9ZHT
9ZHU
9ZHV
9ZHW
9ZHX
9ZHZ
9ZI0
9ZI2
9ZI3
9ZI4
9ZI5
9ZI6
9ZI7
9ZI8
9ZI9
9ZIB
9ZIC
9ZID
9ZIE
9ZIF
9ZIG
9ZIH
9ZII
9ZIJ
9ZIK
9ZIL
9ZIM
9ZIN
9ZIU
9ZIV
9ZIW
9ZIX
9ZIY
9ZIZ
9ZJN
9ZJR
9ZJS
9ZJT
9ZJU
9ZJV
9ZJW
9ZJZ
9ZK8
9ZK9
9ZKA
9ZKC
9ZKD
9ZKE
9ZKM
9ZKN
9ZKO
9ZKP
9ZKQ
9ZKR
9ZKS
9ZKT
9ZKU
9ZKV
9ZKW
9ZL7
9ZLA
9ZLC
9ZLD
9ZLF
9ZLH
9ZLK
9ZLY
9ZM2
9ZMK
9ZMM
9ZN2
9ZN5
9ZN6
9ZN7
9ZN8
9ZN9
9ZNA
9ZNC
9ZND
9ZNF
9ZNG
9ZNI
9ZNN
9ZNO
9ZNP
9ZO4
9ZO5
9ZO8
9ZOB
9ZOE
9ZOF
9ZOH
9ZP0
9ZP1
9ZP2
9ZP3
9ZP4
9ZP9
9ZPK
9ZPL
9ZPO
9ZPT
9ZPV
9ZPW
9ZPX
9ZPY
9ZPZ
9ZQ0
9ZQ1
9ZQ6
9ZQ7
9ZQ9
9ZQA
9ZQB
9ZQC
9ZQR
9ZQS
9ZQT
9ZQU
9ZQY
9ZR7
9ZRG
9ZRJ
9ZRK
9ZRL
9ZRQ
9ZRR
9ZRS
9ZRT
9ZRX
9ZRY
9ZRZ
9ZS2
9ZS4
9ZS7
9ZT5
9ZT6
9ZT7
9ZT8
9ZTC
9ZTJ
9ZTK
9ZTL
9ZUB
9ZUN
9ZV0
9ZV1
9ZV4
9ZV9
9ZVH
9ZVL
9ZVM
9ZVV
9ZVX
9ZW3
9ZW4
9ZWE
9ZWK
9ZWL
9ZWM
9ZWN
9ZWO
9ZXC
9ZXD
9ZXX
9ZY0
9ZY1
9ZY3
9ZY5
9ZY7
9ZYR
9ZYS
9ZZ6
9ZZI
9ZZJ
9ZZK
9ZZM
9ZZO
9ZZR
9ZZY


In [ ]:
%%writefile data/selection/candidate_clusters.csv
pdb_id,cluster_id,cluster_size,longest_seq_len
10ED,10ED,1,606
10FJ,9LBW,15,238
10OY,10OY,3,344
10TL,10TL,5,388
10TM,10TL,5,388
10VM,10VM,1,274
10XW,10XW,1,241
11CV,11CV,1,326
11HL,11HL,1,605
11HM,9OVK,25,1258
11IT,9OVK,25,1258
11JF,11JF,1,1724
11KM,11KM,5,676
11OR,9LBW,15,217
11OU,9LBW,15,219
11TD,9LLG,10,475
11TF,9LLG,10,475
11TH,9LLG,10,475
11TK,9LLG,10,475
11TL,9LLG,10,475
11WN,11KM,5,676
11WO,11KM,5,676
11WP,11KM,5,676
11WQ,11KM,5,676
11XX,11XX,1,553
11ZV,11ZV,1,176
12IY,12IY,1,374
12OY,12OY,3,304
12OZ,12OY,3,304
12PB,12OY,3,304
12QJ,12QJ,1,394
12RP,9OVK,25,1271
13BD,9OVK,25,1271
13BF,9LBW,15,221
13BR,13BR,1,284
13BS,9LBW,15,226
13KK,13KK,2,102
13KW,13KK,2,102
20ZN,20ZN,2,620
21AH,9LLG,10,366
21AO,21AO,2,321
21AQ,21AO,2,321
21DI,21DI,1,134
21FH,21FH,4,527
21FI,21FH,4,502
21FJ,21FH,4,527
21FK,21FH,4,504
21NP,9XCB,8,429
21NQ,9XCB,8,429
21NX,9LBW,15,263
21OR,9XCB,8,429
21TW,21TW,1,363
21TY,9UVY,5,748
21VL,21VL,1,522
21WL,21WL,1,468
22FX,22FX,1,182
22JY,22JY,1,51
22OF,22OF,1,309
22PD,9OVK,25,200
22PL,22PL,1,311
22RD,22RD,1,333
22WP,9W3F,11,336
22WR,9W3F,11,336
22WT,9W3F,11,280
23IC,23IC,3,499
23IF,23IC,3,499
23IG,23IC,3,499
23WM,23WM,3,1552
23WN,23WM,3,1513
23WO,23WM,3,1503
25IF,25IF,6,400
25PV,25PV,1,427
25RV,25RV,2,724
25RW,25RV,2,724
26CW,26CW,1,119
26DE,26DE,1,376
26DK,26DK,2,197
26GP,26GP,1,620
26PF,26PF,2,864
26QF,26QF,8,394
26QG,26QF,8,394
26QH,26QF,8,394
26QI,26QF,8,394
26QJ,26QF,8,394
26QM,26QF,8,394
26QS,26QF,8,391
26QT,26QF,8,394
26VA,26VA,3,624
26VB,26VA,3,624
26VE,26VA,3,624
26WC,26WC,1,187
28IZ,10OY,3,267
28QL,9W97,39,227
28QM,9W97,39,373
28QN,9W97,39,374
28VJ,9PV0,2,489
28WZ,9W97,39,687
31CJ,31CJ,2,534
31CK,31CJ,2,534
31ZN,31ZN,3,442
32PZ,32PZ,2,549
32QB,32PZ,2,549
33AS,33AS,3,488
33AU,33AS,3,488
33AV,33AS,3,488
33AW,33AW,1,482
35TL,35TL,1,290
36BB,9LBW,15,472
36CQ,36CQ,1,427
43JF,43JF,1,750
43JG,43JG,1,164
43JH,43JH,1,171
43MU,9WAG,3,299
43YY,43YY,1,487
46KE,26DK,2,197
9CWP,9OVK,25,193
9CWQ,9OVK,25,304
9E94,9E94,2,313
9FTE,9FTE,1,426
9H2Q,9W97,39,674
9HJQ,9HJQ,1,401
9I3T,9I3T,8,492
9I5N,9I5N,1,143
9I8C,9I8C,1,338
9IB1,9OVK,25,1288
9IB3,9OVK,25,1288
9IBW,9I3T,8,488
9J6R,9W97,39,562
9J6S,9W97,39,562
9J6T,9W97,39,555
9J6U,9W97,39,555
9J6V,9W97,39,555
9JBS,9JBS,4,569
9JBT,9JBS,4,569
9JBU,9JBS,4,569
9JKH,9W97,39,620
9JKI,9W97,39,620
9JKJ,9W97,39,620
9JKK,9W97,39,620
9JKM,9W97,39,620
9JZI,9JZI,3,393
9JZJ,9JZI,3,393
9JZN,9JZI,3,393
9K7G,9K7G,5,413
9K7S,9K7G,5,384
9K88,9K7G,5,384
9K8Y,9K7G,5,376
9K90,9K7G,5,384
9KKN,9KKN,9,637
9KKO,9KKN,9,637
9KQ5,9KKN,9,637
9KQM,9KKN,9,637
9KSL,9KSL,1,315
9KVS,9KVS,1,538
9L19,9W97,39,620
9L1A,9W97,39,620
9L1B,9W97,39,620
9L1C,9W97,39,620
9L1D,9W97,39,620
9L1E,9W97,39,620
9L38,9JBS,4,491
9L8N,9L8N,1,761
9LBW,9LBW,15,473
9LFA,10TL,5,340
9LFC,10TL,5,340
9LFD,9I3T,8,365
9LKD,9LKD,1,317
9LLG,9LLG,10,516
9LM0,9LM0,1,877
9LNM,9W97,39,614
9LNN,9W97,39,614
9LNO,9W97,39,614
9LOZ,9OVK,25,194
9LP1,9OVK,25,194
9LQT,9LQT,3,380
9LQY,9LQT,3,380
9LQZ,9LQT,3,380
9LV4,9LV4,1,430
9LW1,9LW1,2,297
9LW3,9LW1,2,297
9LY5,9LY5,1,448
9M9M,9M9M,1,424
9MA0,26PF,2,478
9MAN,9W97,39,501
9MKG,9MKG,1,277
9MM5,9MM5,1,490
9MNM,9MNM,2,231
9MUH,10OY,3,265
9MUN,9PVM,3,561
9NEN,9NEN,1,538
9NNZ,9NNZ,1,514
9NP0,9NP0,1,351
9NU4,9XZN,3,614
9O7G,9LBW,15,121
9O7I,9LBW,15,224
9OBT,9OBT,8,862
9OBV,9OBT,8,862
9OBW,9OBT,8,862
9OBY,9OBT,8,862
9OBZ,9OBT,8,862
9OC0,9OBT,8,862
9OC1,9OBT,8,862
9OC2,9OBT,8,862
9ODH,9ODH,2,354
9ODI,9ODH,2,354
9OK7,9OK7,2,892
9OK8,9OK7,2,892
9OPQ,9OPQ,1,506
9OTA,9E94,2,270
9OVK,9OVK,25,1344
9OVM,9OVK,25,1326
9OVO,9OVK,25,1326
9P1V,9P1V,1,838
9P4L,9P4L,9,743
9P4M,9P4L,9,736
9P4N,9P4L,9,743
9P4O,9P4L,9,743
9P4P,9P4L,9,743
9P4Q,9P4L,9,743
9P4R,9P4L,9,743
9PGD,9PGD,1,396
9PPQ,10TL,5,334
9PRZ,9MNM,2,231
9PV0,9PV0,2,495
9PVM,9PVM,3,580
9PX8,9PX8,6,31
9PXJ,9PXJ,1,795
9PZD,9PX8,6,31
9Q33,9Y61,4,442
9Q37,9PX8,6,31
9Q3L,9W3F,11,460
9Q7J,9PX8,6,31
9QJJ,9QJJ,1,435
9QJN,31ZN,3,349
9QL1,31ZN,3,442
9QO8,9W97,39,632
9QO9,9W97,39,632
9QP4,9I3T,8,488
9QQS,9QQS,2,477
9QTY,9QQS,2,344
9QZO,9QZO,8,663
9R07,9QZO,8,663
9R0A,9QZO,8,663
9R0I,9QZO,8,663
9R0M,9QZO,8,663
9R0N,9QZO,8,663
9R0O,9QZO,8,663
9R1H,9WBR,12,613
9R8P,9R8P,1,590
9RH2,9RH2,14,407
9RH3,9RH2,14,407
9RH4,9RH2,14,407
9RH5,9RH2,14,407
9RH6,9RH2,14,407
9RH7,9RH2,14,407
9RH8,9RH2,14,407
9RH9,9RH2,14,407
9RHA,9RH2,14,407
9RHB,9RH2,14,407
9RHC,9RH2,14,407
9RHD,9RH2,14,407
9RHE,9RH2,14,407
9RHF,9RH2,14,407
9RQL,9RQL,2,729
9RQN,9RQL,2,729
9RRS,9P4L,9,738
9RWS,9RWS,1,551
9RY4,9RY4,2,280
9RZM,9RY4,2,280
9S15,9S15,1,184
9S5J,9S5J,1,510
9S6R,9S6R,1,467
9SAS,9SAS,2,295
9SAU,9SAS,2,294
9SF5,9OVK,25,1284
9SF8,9SF8,1,562
9SP8,9SP8,1,600
9SR9,9SR9,1,392
9SWM,9SWM,3,379
9SWQ,9SWM,3,379
9SZB,9SZB,1,314
9T3R,9T3R,2,636
9T3S,9T3R,2,636
9T5J,9T5J,1,614
9T5K,9T5K,1,453
9T9M,9LBW,15,220
9THK,9THK,3,293
9TIU,9TIU,2,136
9TIV,9TIU,2,136
9TK9,9THK,3,293
9TKA,9THK,3,293
9TKM,9WAG,3,278
9TUZ,9TUZ,1,1109
9TZW,9TZW,2,142
9TZY,9TZW,2,142
9U4J,9U4J,1,288
9U7H,9U7H,1,613
9UBC,9UBC,1,196
9UDC,9UDC,2,729
9UGL,9W3F,11,401
9UGU,9XCB,8,429
9UGX,9XCB,8,429
9UGY,9XCB,8,429
9UHG,9OVK,25,198
9UI1,9UI1,4,636
9UI6,9UI1,4,636
9UIF,9UI1,4,636
9UIT,9UI1,4,636
9UJM,9W3F,11,389
9UPM,9I3T,8,452
9UPN,9I3T,8,434
9UPO,9I3T,8,434
9US8,9US8,3,553
9UUR,9UUR,4,403
9UUX,9UUR,4,403
9UVU,9I3T,8,487
9UVW,9UVY,5,486
9UVX,9UVY,5,486
9UVY,9UVY,5,765
9UVZ,9UVY,5,759
9UW3,9UUR,4,403
9UW4,9UUR,4,403
9UXN,9UXN,4,337
9UXO,9UXN,4,337
9UXP,9UXN,4,337
9UXQ,9UXN,4,337
9UXS,9OVK,25,1245
9V2U,9LBW,15,249
9V35,25IF,6,393
9V38,25IF,6,393
9V3C,25IF,6,393
9V3T,9V3T,4,476
9V3V,9V3T,4,476
9V3X,9V3T,4,476
9V3Y,25IF,6,393
9V3Z,9V3T,4,476
9V4D,25IF,6,393
9V82,9V82,3,472
9V8W,9W97,39,635
9V8X,9W97,39,635
9V8Y,9W97,39,635
9VBD,9VBD,2,893
9VBV,9VBD,2,893
9VCT,9VCT,1,760
9VH3,9VH3,1,835
9VH9,9QZO,8,643
9VI4,9P4L,9,516
9VJ3,9VJ3,3,401
9VJ4,9VJ3,3,401
9VND,9VND,1,288
9VNE,9VNE,1,142
9VPG,9VPG,1,582
9VUH,9VUH,1,365
9VVM,9UDC,2,729
9VWR,9W97,39,566
9VWS,9W97,39,630
9VXT,9SWM,3,379
9VZP,20ZN,2,620
9W14,9OVK,25,223
9W22,9W22,1,358
9W3F,9W3F,11,517
9W43,9LBW,15,213
9W4I,9W4I,1,317
9W62,9W62,5,914
9W63,9W62,5,914
9W64,9W62,5,914
9W65,9W62,5,914
9W66,9W62,5,914
9W8B,9KKN,9,514
9W8C,9KKN,9,514
9W8D,9KKN,9,514
9W8E,9KKN,9,474
9W8F,9KKN,9,474
9W8J,9WBR,12,595
9W8Q,9WBR,12,593
9W97,9W97,39,1291
9W98,9W97,39,1291
9W99,9W97,39,1291
9W9A,9W97,39,1291
9W9B,9W97,39,1291
9W9C,9W97,39,1291
9WAG,9WAG,3,302
9WBR,9WBR,12,797
9WC6,9WBR,12,797
9WF3,9WF3,6,697
9WF9,9WF3,6,697
9WFA,9WFA,1,291
9WFC,9WF3,6,697
9WFL,9WF3,6,697
9WFN,9WF3,6,696
9WFO,9WF3,6,696
9WIJ,9LLG,10,447
9WIX,9WIX,1,546
9WME,9WME,1,657
9WML,9WBR,12,636
9WMM,9WBR,12,636
9WMN,9WBR,12,636
9WMO,9WBR,12,636
9WMP,9WBR,12,636
9WMQ,9WMR,2,182
9WMR,9WMR,2,214
9WNG,9W3F,11,307
9WQ6,9W3F,11,339
9WQA,9W3F,11,318
9WQB,9W3F,11,318
9WS8,9XCB,8,429
9WTX,9US8,3,551
9WTY,9US8,3,551
9WWG,9WWG,2,545
9WWI,9WWG,2,545
9WX7,9WBR,12,636
9WX9,9WBR,12,636
9X2N,9XZN,3,534
9X35,9LBW,15,215
9X3S,9X3S,1,364
9X3Y,9V82,3,325
9X41,9V82,3,325
9XBS,9XBS,1,577
9XCB,9XCB,8,452
9XCH,9XCH,1,666
9XD0,9VJ3,3,401
9XFJ,9XFJ,1,361
9XGW,9OVK,25,184
9XGX,9OVK,25,199
9XI8,9OVK,25,1256
9XJK,9LBW,15,225
9XKO,9W97,39,397
9XMM,9XMM,1,595
9XQB,9LLG,10,431
9XQN,9LLG,10,422
9XR3,9XR3,2,485
9XR4,9XR3,2,485
9XZG,9Y61,4,443
9XZK,9XZK,1,776
9XZN,9XZN,3,594
9XZX,9XZX,1,236
9Y1V,9PX8,6,31
9Y61,9Y61,4,467
9Y63,9Y63,1,296
9Y7L,9Y7L,3,263
9Y7M,9Y7L,3,263
9Y7N,9Y7L,3,263
9Y9X,9PX8,6,31
9YA9,9Y61,4,467
9YC5,9YC5,1,283
9YFW,9YFW,1,656
9YGR,9OVK,25,1334
9YI2,9YI2,1,590
9YQL,9YQL,2,608
9YR4,9YQL,2,607
9YW2,9YW2,1,880
9YWU,9YWU,2,456
9YWV,9YWU,2,456
9YX8,9OVK,25,1250
9YZ1,9YZ1,1,268
9Z1N,9OVK,25,1243
9Z1X,9Z1X,4,1214
9Z1Y,9Z1X,4,1214
9Z24,9Z1X,4,1214
9Z26,9Z1X,4,1214
9Z3Y,9OVK,25,1271
9Z4I,9Z4I,1,620
9Z4O,9Z4O,1,575
9Z5I,9PVM,3,580
9Z5U,9Z5U,1,589
9Z60,9Z60,3,646
9Z62,9Z60,3,646
9Z64,9Z60,3,646
9Z6N,9Z6N,4,392
9Z7R,9Z6N,4,392
9Z7S,9Z6N,4,392
9Z7T,9Z6N,4,392
9ZBV,9ZBV,2,134
9ZDY,9ZBV,2,134
9ZK8,9ZK8,7,208
9ZKC,9ZK8,7,208
9ZKD,9ZK8,7,208
9ZL7,9ZK8,7,208
9ZLC,9ZK8,7,208
9ZLF,9ZK8,7,206
9ZLH,9ZK8,7,206
9ZNO,9ZNO,1,365
9ZR7,9ZR7,1,304
9ZRT,9ZRT,1,357


In [ ]:
EXPECTED = {'select_entries.py': 'cd77d03a2ee62837a61e5aab482c0d747fe5d5532150c86d86abcadfde00c9f2', 'compute_cc_mask.py': '44d3e43e0c1dd7f3e460570da550f2a78072f96052cf52e73e6af17a8da70fac', 'novelty_check.py': '3819d7792235f7b026a27847747e253dd7625b69c2118a1d4332011dbe37e406', 'data/pool/pool_candidates.csv': '4bba9e1e1ab497a7ffdf6713fff6aed697245730f76a1637b19d3acf050737c3', 'data/pool/pool_candidate_sequences.fasta': 'c914ce06feb39e07f3e60cdedc6a422d586c378097c2b938cb306494b7819795', 'data/pool/pool_ids_all_cryoem.txt': 'dca391ef5d0db171e4ded15d00a73472dc0b24e48d986f5f6dd639698aabc86c', 'data/selection/candidate_clusters.csv': '7e73cb8dfae5bced1c8c187613719bbfee45abdcccf8cc86ed9d1d7bb83bb53f'}
bad = [p for p, h in EXPECTED.items() if not os.path.exists(p) or sha256_file(p) != h]
if bad:
    fail("these files were not written as expected: %s. Use Runtime -> Run all again; if it happens again, send me this message." % ", ".join(bad))
else:
    log("all %d files written and checked (hashes match)" % len(EXPECTED))


## Step 3. Install Phenix

Needed once per session (about 15 to 25 minutes). Two boxes ask for the user name and the password from the Phenix email.
If Phenix is already installed in this session, this step is skipped.

In [ ]:
PHENIX_URL = "https://phenix-online.org/download/phenix/release/send_octet_stream.cgi?version=2.2.1-6174&file=Phenix-2.2.1-6174-Linux-x86_64.sh"
PHENIX_VERSION = "2.2.1-6174"
INSTALLER = "Phenix-2.2.1-6174-Linux-x86_64.sh"
PREFIX = "/content/phenix-" + PHENIX_VERSION if IN_COLAB else os.path.join(WORK, "phenix-" + PHENIX_VERSION)
MIN_INSTALLER_MB = 50


def find_phenix_env():
    for pattern in ("phenix_env.sh", "*/phenix_env.sh", "*/*/phenix_env.sh"):
        hits = sorted(glob.glob(os.path.join(PREFIX, pattern)))
        if hits:
            return hits[0]
    return None


def phenix_works(env_script):
    r = subprocess.run(["bash", "-c", 'source "$0" && phenix.version', env_script], capture_output=True, text=True, errors="replace")
    lines = [ln.strip() for ln in (r.stdout + r.stderr).splitlines() if ln.strip()]
    text = " | ".join(lines[:3])[:200]
    if r.returncode == 0:
        return True, text
    r2 = subprocess.run(["bash", "-c", 'source "$0" && command -v phenix.map_model_cc', env_script], capture_output=True, text=True, errors="replace")
    if r2.returncode == 0:
        return True, "phenix.map_model_cc found at %s (phenix.version said: %s)" % (r2.stdout.strip(), text or "nothing")
    return False, text or "phenix.version did not run"


def download_installer():
    import requests
    if os.path.exists(INSTALLER) and os.path.getsize(INSTALLER) >= MIN_INSTALLER_MB * 1e6:
        log("%s is already here (%d MB)" % (INSTALLER, os.path.getsize(INSTALLER) // 1000000))
        return True
    user = secret("PHENIX_USER", "Phenix user name (from the email), then Enter: ", hidden=False)
    password = secret("PHENIX_PASSWORD", "Phenix password (from the email), then Enter: ")
    if len(password) >= 6:
        SECRETS.append(password)
    last = ""
    for attempt in range(1, 4):
        try:
            with requests.get(PHENIX_URL, auth=(user, password), stream=True, timeout=(30, 120)) as r:
                if r.status_code == 401:
                    fail("the Phenix server did not accept the user name and password (HTTP 401). The password changes on the 1st of every "
                         "month, so use the one from the newest email, and type both exactly as they appear. Then use Runtime -> Run all again.")
                    return False
                if r.status_code != 200:
                    last = "HTTP %d" % r.status_code
                    raise IOError(last)
                total = int(r.headers.get("Content-Length") or 0)
                done, shown = 0, -1
                with open(INSTALLER + ".part", "wb") as f:
                    for block in r.iter_content(1 << 22):
                        f.write(block)
                        done += len(block)
                        mark = done * 10 // total if total else done // 250000000
                        if mark != shown:
                            shown = mark
                            log("  downloaded %d MB%s" % (done // 1000000, " of %d MB" % (total // 1000000) if total else ""))
            if total and done != total:
                last = "the file is incomplete (%d of %d bytes)" % (done, total)
                raise IOError(last)
            with open(INSTALLER + ".part", "rb") as f:
                head = f.read(2)
            if head != b"#!" or done < MIN_INSTALLER_MB * 1e6:
                fail("the Phenix server sent %d bytes that do not look like the installer (it is probably a web page, not the file). Check the "
                     "user name and password. As a last resort, download the file in your browser from the same link and upload it into the "
                     "folder 'work' with the folder icon on the left, then use Runtime -> Run all again." % done)
                os.remove(INSTALLER + ".part")
                return False
            os.replace(INSTALLER + ".part", INSTALLER)
            return True
        except (requests.RequestException, IOError) as e:
            last = last or str(e)[:150]
            log("download attempt %d of 3 failed: %s" % (attempt, last))
            time.sleep(10 * attempt)
            last = ""
    fail("the Phenix installer could not be downloaded after 3 attempts. Use Runtime -> Run all again in a few minutes.")
    return False


def install_phenix():
    env = find_phenix_env()
    if env:
        ok, text = phenix_works(env)
        if ok:
            log("Phenix is already installed: %s (%s)" % (text, env))
            STATE.update(phenix_env=env, phenix_version=text)
            return
    if not download_installer():
        return
    STATE["installer_sha256"] = sha256_file(INSTALLER)
    log("installer sha256: %s" % STATE["installer_sha256"])
    tmp = os.path.join(os.path.dirname(PREFIX), "tmp_phenix_install")
    os.makedirs(tmp, exist_ok=True)
    shutil.rmtree(PREFIX, ignore_errors=True)
    log("installing Phenix (15 to 25 minutes; a line is printed now and then) ...")
    t0 = time.time()
    rc, last = run(["bash", INSTALLER, "-b", "-p", PREFIX], env={"TMPDIR": tmp}, every=60)
    shutil.rmtree(tmp, ignore_errors=True)
    if rc != 0:
        log("\n".join(last[-25:]))
        fail("the Phenix installer stopped with exit code %d (the last lines are above). If they say 'No space left', disconnect and delete the "
             "runtime and start again; otherwise send me the zip." % rc)
        return
    env = find_phenix_env()
    ok, text = phenix_works(env) if env else (False, "phenix_env.sh was not found under %s" % PREFIX)
    if not ok:
        fail("Phenix was installed but does not start: %s" % text)
        return
    STATE.update(phenix_env=env, phenix_version=text)
    log("Phenix installed in %d minutes: %s" % ((time.time() - t0) // 60, text))
    os.remove(INSTALLER)


if STATE["ok"]:
    install_phenix()
else:
    log("skipped: an earlier step stopped")


## Step 4. Test Phenix on two entries

Before the long run, Phenix computes CC_mask for the first two entries and shows what it printed, so a problem shows up in minutes, not hours.
Phenix 2.2.1 has retired `phenix.map_model_cc`, so the script tries the ways of calling Phenix one after the other and keeps the first one that works
for all later entries. If none gives a number, the run stops here and the zip then also holds Phenix's own help texts.

In [ ]:
SEL = "sel_out"
BATCH = 16
MAX_ROUNDS = 12


def cc_rows():
    import csv
    path = os.path.join(SEL, "cc_mask.csv")
    if not os.path.exists(path):
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def read_todo():
    import csv
    with open(os.path.join(SEL, "cc_todo.csv"), newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def select_round():
    cmd = [sys.executable, "-u", "select_entries.py", "select", "--pool", "data/pool", "--out", SEL,
           "--clusters", "data/selection/candidate_clusters.csv", "--entries", SEL + "/entries_2026.csv", "--batch", str(BATCH)]
    if os.path.exists(os.path.join(SEL, "cc_mask.csv")):
        cmd += ["--cc", SEL + "/cc_mask.csv"]
    return run(cmd)[0]


def compute(limit=0, show=False):
    cmd = [sys.executable, "-u", "compute_cc_mask.py", "--todo", SEL + "/cc_todo.csv", "--out", SEL + "/cc_mask.csv",
           "--phenix-env", STATE["phenix_env"], "--logs", SEL + "/cc_logs", "--work", "cc_work"]
    if limit:
        cmd += ["--limit", str(limit)]
    if show:
        cmd.append("--show-output")
    return run(cmd)[0]


def diagnose():
    run([sys.executable, "-u", "compute_cc_mask.py", "--diagnose", "--out", SEL + "/cc_mask.csv", "--phenix-env", STATE["phenix_env"], "--work", "cc_work"])


def drop_rows(ids):
    """Takes rows out of cc_mask.csv and keeps them in cc_mask_FAILED_TEST.csv: a failure of Phenix itself is not a result of an entry."""
    import csv
    path = os.path.join(SEL, "cc_mask.csv")
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fields, rows = reader.fieldnames, list(reader)
    keep = [r for r in rows if r["pdb_id"] not in ids]
    with open(os.path.join(SEL, "cc_mask_FAILED_TEST.csv"), "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, lineterminator="\n")
        if f.tell() == 0:
            w.writeheader()
        w.writerows([r for r in rows if r["pdb_id"] in ids])
    if not keep:
        os.remove(path)
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, lineterminator="\n")
        w.writeheader()
        w.writerows(keep)


def test_two_entries():
    log("=== first round of the selection rule ===")
    rc = select_round()
    if rc == 0:
        log("the list is already final")
        return
    if rc != 3:
        fail("select_entries.py stopped with exit code %d (see the lines above)" % rc)
        return
    done_before = {r["pdb_id"] for r in cc_rows()}
    log("\n=== test: CC_mask for the first two entries, with Phenix's own output ===")
    rc = compute(limit=2, show=True)
    if rc == 3:
        fail("Phenix or the disk stopped the test (see the lines above)")
        return
    new = [r for r in cc_rows() if r["pdb_id"] not in done_before]
    if not new:
        fail("the two test entries could not be downloaded (see the lines above). Use Runtime -> Run all again in a few minutes.")
        return
    if not any(r["status"] == "ok" for r in new):
        drop_rows({r["pdb_id"] for r in new if not r["note"].startswith("download:")})
        diagnose()
        fail("Phenix gave no CC_mask for either of the two test entries (reasons: %s). Send me the zip: its folder cc_logs has the full Phenix output, "
             "and diagnostics.txt has Phenix's help texts." % "; ".join(dict.fromkeys(r["note"][:160] for r in new)))
        return
    log("\nThe test worked: %s" % ", ".join("%s CC_mask %s [%s]" % (r["pdb_id"], r["cc_mask"] or r["status"], r["variant"]) for r in new))


if STATE["ok"]:
    test_two_entries()
else:
    log("skipped: an earlier step stopped")


## Step 5. Compute CC_mask and select, round by round

Each round applies the selection rule to the results so far, asks for the CC_mask of the next entries, and computes them.
It repeats until the rule needs nothing more. The result does not depend on how many rounds it takes. This is the long step.

In [ ]:
def selection_loop():
    for round_no in range(1, MAX_ROUNDS + 1):
        log("\n=== round %d: apply the selection rule ===" % round_no)
        rc = select_round()
        STATE["rounds"] = round_no
        if rc == 0:
            STATE["final"] = True
            return
        if rc != 3:
            fail("select_entries.py stopped with exit code %d (see the lines above)" % rc)
            return
        have = {r["pdb_id"] for r in cc_rows()}
        todo = [r for r in read_todo() if r["pdb_id"] not in have]
        log("%d entries need a CC_mask now" % len(todo))
        before = len(have)
        rc = compute()
        if rc == 3:
            fail("Phenix or the disk stopped the CC_mask computation (see the lines above)")
            return
        log("elapsed since the start: %d minutes" % ((time.time() - STATE["t0"]) // 60))
        if len(cc_rows()) == before:
            fail("no CC_mask could be computed in this round (download problems?). Use Runtime -> Run all again in a few minutes: it continues where it stopped.")
            return
    fail("the selection did not finish within %d rounds" % MAX_ROUNDS)


def show_result():
    import csv
    path = os.path.join(SEL, "entries_2026.csv")
    with open(path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    log("\nSELECTED ENTRIES (%d)" % len(rows))
    log("%-5s %-10s %-12s %5s %9s %7s %9s %7s" % ("PDB", "EMDB", "stratum", "res A", "residues", "CC_mask", "clus.size", "map MB"))
    for r in rows:
        log("%-5s %-10s %-12s %5s %9s %7s %9s %7s" % (r["pdb_id"], r["emdb_id"], r["stratum"], r["resolution_A"], r["modeled_residues"], r["cc_mask"],
                                                     r["cluster_size"], r["map_size_mb"]))
    rows_cc = cc_rows()
    high = sum(1 for r in rows_cc if r["status"] == "ok" and float(r["cc_mask"]) >= 0.70)
    low = sum(1 for r in rows_cc if r["status"] == "ok" and float(r["cc_mask"]) < 0.70)
    log("\nCC_mask was computed for %d entries: %d at 0.70 or more, %d below 0.70, %d without a number (Phenix failed)"
        % (len(rows_cc), high, low, len(rows_cc) - high - low))


if STATE["ok"]:
    selection_loop()
    if STATE["final"]:
        show_result()
else:
    log("skipped: an earlier step stopped")


## Step 6. Novelty check

For each chosen entry, how similar is its protein to proteins in the PDB before 2026? (MMseqs2, decision 9.) This step needs a few minutes.
If it fails, the selection above is still valid and the zip is still made.

In [ ]:
MMSEQS_VERSION = "8cc5ce367b5638c4306c2d7cfc652dd099a4643f"
MMSEQS_URL = "https://github.com/soedinglab/MMseqs2/releases/download/18-8cc5c/mmseqs-linux-@ARCH@.tar.gz"
MMSEQS_SHA256 = {'avx2': 'bd9b0234da5949ad528d5b5f9ea4cda9c1e23dce14b46c0791d4d919a76e61ce', 'sse41': '0ece6a6af8f5d198bccabc98cb5fddfc9212ce04bb332e486bfdcc468c0d7b08'}
NOVELTY_EXTRA = []


def get_mmseqs():
    import tarfile, urllib.request
    arch = "avx2"
    try:
        with open("/proc/cpuinfo") as f:
            arch = "avx2" if " avx2" in f.read() else "sse41"
    except OSError:
        pass
    exe = os.path.abspath("mmseqs/bin/mmseqs")
    if not os.path.exists(exe):
        url, tgz = MMSEQS_URL.replace("@ARCH@", arch), "mmseqs.tar.gz"
        urllib.request.urlretrieve(url, tgz)
        if sha256_file(tgz) != MMSEQS_SHA256[arch]:
            raise RuntimeError("the MMseqs2 download has an unexpected hash (%s)" % sha256_file(tgz))
        with tarfile.open(tgz) as t:
            t.extractall(".", filter="data") if sys.version_info >= (3, 12) else t.extractall(".")
        os.remove(tgz)
    r = subprocess.run([exe, "version"], capture_output=True, text=True)
    version = (r.stdout or r.stderr).strip()
    if version != MMSEQS_VERSION:
        raise RuntimeError("MMseqs2 reports version %s, expected %s" % (version, MMSEQS_VERSION))
    STATE["mmseqs_version"] = version
    return exe


def novelty():
    exe = get_mmseqs()
    cmd = [sys.executable, "-u", "novelty_check.py", "--entries", SEL + "/entries_2026.csv", "--fasta", "data/pool/pool_candidate_sequences.fasta",
           "--cryoem-ids", "data/pool/pool_ids_all_cryoem.txt", "--out", SEL + "/novelty", "--mmseqs", exe,
           "--threads", str(max(1, min(4, os.cpu_count() or 1)))] + NOVELTY_EXTRA
    rc, last = run(cmd)
    if rc != 0:
        raise RuntimeError("novelty_check.py stopped with exit code %d (see the lines above)" % rc)


if STATE["ok"] and STATE["final"]:
    try:
        novelty()
        STATE["novelty"] = "done"
    except Exception as e:
        STATE["novelty"] = "failed: %s" % e
        log("\nNOVELTY CHECK DID NOT FINISH: %s\nThe selection is not affected. Send me the zip anyway; I will run this step another way." % e)
else:
    log("skipped: the selection is not final")


## Step 7. Pack the results

Always run this last cell, also after a problem or after stopping early. It packs what exists into one zip and downloads it.
The Phenix password is never written to the zip; the cell checks every file for it before packing.

In [ ]:
def pack():
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M")
    name = "selection_result_%s%s" % (stamp, "" if STATE["final"] and STATE["ok"] else "_INCOMPLETE")
    info = {"notebook": "select_colab", "started_utc": STATE["started"], "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "elapsed_minutes": round((time.time() - STATE["t0"]) / 60), "final": STATE["final"], "rounds": STATE["rounds"], "problem": STATE["problem"],
            "python": platform.python_version(), "platform": platform.platform(), "in_colab": IN_COLAB, "phenix_installer": INSTALLER,
            "phenix_installer_sha256": STATE["installer_sha256"], "phenix_version_output": STATE["phenix_version"],
            "mmseqs_version": STATE["mmseqs_version"], "novelty": STATE["novelty"], "batch": BATCH, "input_sha256": EXPECTED,
            "cc_mask_rows": len(cc_rows()), "cc_mask_ok": sum(1 for r in cc_rows() if r["status"] == "ok")}
    with open("run_info.json", "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)
    wanted = ["run.log", "run_info.json"]
    for f in ("entries_2026.csv", "selection_log.csv", "selection_info.json", "cc_todo.csv", "cc_mask.csv", "cc_mask_FAILED_TEST.csv", "diagnostics.txt"):
        wanted.append(os.path.join(SEL, f))
    for root in (os.path.join(SEL, "cc_logs"), os.path.join(SEL, "novelty")):
        for dirpath, _, files in os.walk(root):
            wanted += [os.path.join(dirpath, f) for f in sorted(files)]
    zip_path = name + ".zip"
    left_out = []
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for path in wanted:
            if not os.path.isfile(path):
                continue
            with open(path, "rb") as f:
                data = f.read()
            if any(s and s.encode() in data for s in SECRETS):
                left_out.append(path)
                continue
            arc = os.path.relpath(path, SEL) if path.startswith(SEL + os.sep) else path
            z.write(path, name + "/" + arc)
        n = len(z.namelist())
    log("zip written: %s (%d files)" % (zip_path, n))
    if left_out:
        log("left out of the zip because they contained the password: %s" % ", ".join(left_out))
    return zip_path


zip_path = pack()
if STATE["ok"] and STATE["final"]:
    log("\nRESULT: OK. The entry list is final. Send me %s." % zip_path)
elif STATE["ok"]:
    log("\nRESULT: NOT FINISHED. Send me %s anyway; it shows how far the run got." % zip_path)
else:
    log("\nRESULT: STOPPED (%s). Send me %s." % (STATE["problem"], zip_path))
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    log("(This is not Colab, or the download did not start. Download %s from the folder 'work' on the left.)" % zip_path)
